<a href="https://colab.research.google.com/github/May-ysaa/InvoiceFlow-AI/blob/main/notebooks/01_data_collection_and_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


**Cell 1 — Markdown: Identitas notebook**

# InvoiceFlow AI — Data Collection and Audit

**Notebook ID:** `01_data_collection_and_audit.ipynb`  
**Project:** InvoiceFlow AI – Bulk Invoice Intelligence  
**Repository:** `May-ysaa/InvoiceFlow-AI`  
**Stage:** Data Collection, Governance, and Quality Audit  
**Environment:** Google Colab  
**Input:** PDF, JPG, JPEG, dan PNG  
**Supported Languages:** Indonesia dan Inggris  

## Purpose

Notebook ini digunakan untuk mengumpulkan, mencatat, dan mengaudit dataset
invoice sebelum dokumen digunakan dalam pipeline OCR dan field extraction.

Audit mencakup:

1. sumber dan izin penggunaan dokumen;
2. format serta integritas file;
3. ukuran dan keterbacaan dokumen;
4. jenis invoice digital atau hasil pemindaian;
5. bahasa dan variasi layout;
6. keberadaan file identik;
7. kelengkapan field utama;
8. keamanan dan privasi dokumen.

> Notebook ini tidak melakukan OCR, preprocessing, training model, atau
> ekstraksi field. Invoice asli hanya dibaca untuk audit dan tidak dimodifikasi.

**Cell 2 — Markdown: Tujuan dan kriteria keberhasilan**

## 1. Objective

Tujuan utama notebook ini adalah menghasilkan inventaris dataset invoice yang
dapat dipercaya sebelum masuk ke tahap preprocessing dan OCR.

Dataset dinyatakan siap apabila:

- setiap file mempunyai identitas unik;
- sumber dan izin penggunaannya tercatat;
- file dapat dibuka dan tidak rusak;
- format file termasuk PDF, JPG, JPEG, atau PNG;
- invoice asli tetap tersimpan tanpa perubahan;
- file duplikat identik dapat dikenali;
- bahasa dan jenis dokumen dapat diidentifikasi;
- variasi layout dan kualitas visual terdokumentasi;
- kelengkapan field utama dapat dinilai;
- dokumen yang mengandung informasi sensitif ditandai;
- file bermasalah dipisahkan dari file yang layak diproses.

## Expected Outputs

Notebook ini akan menghasilkan:

1. `invoice_inventory.csv` — inventaris seluruh dokumen;
2. `duplicate_report.csv` — daftar file identik;
3. `audit_summary.json` — ringkasan hasil audit;
4. daftar invoice yang diterima, perlu ditinjau, atau ditolak.

## Audit Status

Setiap dokumen akan memperoleh salah satu status berikut:

| Status | Arti |
|---|---|
| `ACCEPTED` | Layak digunakan pada tahap berikutnya |
| `REVIEW` | Memerlukan pemeriksaan manual |
| `REJECTED` | Rusak, tidak didukung, atau bukan invoice |

**Cell 3 — Markdown: Pemeriksaan environment**

## 2. Environment

Tahap ini memvalidasi lingkungan kerja sebelum dataset dibaca.

Pemeriksaan mencakup:

- versi Python;
- ketersediaan Google Colab;
- koneksi Google Drive;
- lokasi repository dan data;
- library yang diperlukan untuk audit;
- keberadaan environment hasil Notebook 00.

Notebook akan menggunakan prinsip **fail-fast**: proses dihentikan dengan pesan
yang jelas apabila komponen penting belum tersedia. Hal ini mencegah audit
dijalankan pada environment yang salah atau tidak lengkap.

**Cell 4 — Code: Validasi runtime dasar**

In [ ]:
from __future__ import annotations

import importlib.util
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path


SUPPORTED_PYTHON_MIN = (3, 10)
SUPPORTED_PYTHON_MAX = (3, 13)

CURRENT_PYTHON = sys.version_info[:2]
IN_COLAB = importlib.util.find_spec("google.colab") is not None
IS_64_BIT = sys.maxsize > 2**32
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

if not SUPPORTED_PYTHON_MIN <= CURRENT_PYTHON <= SUPPORTED_PYTHON_MAX:
    raise RuntimeError(
        "Versi Python tidak didukung. "
        f"Ditemukan {CURRENT_PYTHON}; diperlukan "
        f"{SUPPORTED_PYTHON_MIN}–{SUPPORTED_PYTHON_MAX}."
    )

if not IS_64_BIT:
    raise RuntimeError(
        "InvoiceFlow AI membutuhkan runtime Python 64-bit."
    )

if not IN_COLAB:
    raise RuntimeError(
        "Notebook ini dirancang untuk dijalankan di Google Colab."
    )

runtime_information = {
    "python_version": sys.version.split()[0],
    "operating_system": platform.system(),
    "architecture": platform.machine(),
    "is_64_bit": IS_64_BIT,
    "is_google_colab": IN_COLAB,
    "run_id": RUN_ID,
}

for key, value in runtime_information.items():
    print(f"{key:20}: {value}")

print("\n✅ Runtime dasar memenuhi persyaratan.")

**Cell 5 — Code: Konfigurasi pusat audit**

In [ ]:
PROJECT_NAME = "InvoiceFlow-AI"
NOTEBOOK_ID = "01_data_collection_and_audit"

GITHUB_OWNER = "May-ysaa"
GITHUB_REPOSITORY = "InvoiceFlow-AI"

REPOSITORY_ROOT = Path("/content") / GITHUB_REPOSITORY
DATA_ROOT = (
    Path("/content/drive")
    / "MyDrive"
    / "InvoiceFlow-AI-Data"
)

RAW_DIR = DATA_ROOT / "raw"
INTERIM_DIR = DATA_ROOT / "interim"
PROCESSED_DIR = DATA_ROOT / "processed"
SAMPLES_DIR = DATA_ROOT / "samples"
AUDIT_OUTPUT_DIR = PROCESSED_DIR / "audit"

INVENTORY_OUTPUT_PATH = (
    AUDIT_OUTPUT_DIR / "invoice_inventory.csv"
)
DUPLICATE_OUTPUT_PATH = (
    AUDIT_OUTPUT_DIR / "duplicate_report.csv"
)
AUDIT_SUMMARY_PATH = (
    AUDIT_OUTPUT_DIR / "audit_summary.json"
)

SUPPORTED_EXTENSIONS = {
    ".pdf",
    ".jpg",
    ".jpeg",
    ".png",
}

SUPPORTED_LANGUAGES = {
    "indonesian",
    "english",
}

AUDIT_STATUSES = {
    "ACCEPTED",
    "REVIEW",
    "REJECTED",
}

EXPECTED_FIELDS = [
    "vendor",
    "invoice_number",
    "invoice_date",
    "due_date",
    "currency",
    "subtotal",
    "tax",
    "discount",
    "total",
]

# Digunakan sebagai tanda review, bukan batas penolakan otomatis.
LARGE_FILE_THRESHOLD_MB = 25
LARGE_FILE_THRESHOLD_BYTES = (
    LARGE_FILE_THRESHOLD_MB * 1024 * 1024
)

configuration_summary = {
    "project": PROJECT_NAME,
    "notebook": NOTEBOOK_ID,
    "repository_root": str(REPOSITORY_ROOT),
    "data_root": str(DATA_ROOT),
    "raw_directory": str(RAW_DIR),
    "audit_output_directory": str(AUDIT_OUTPUT_DIR),
    "supported_extensions": sorted(SUPPORTED_EXTENSIONS),
    "supported_languages": sorted(SUPPORTED_LANGUAGES),
    "expected_fields": len(EXPECTED_FIELDS),
    "large_file_threshold_mb": LARGE_FILE_THRESHOLD_MB,
}

for key, value in configuration_summary.items():
    print(f"{key:27}: {value}")

print("\n✅ Konfigurasi audit berhasil ditetapkan.")

**Cell 6 — Code: Menghubungkan dan memvalidasi Google Drive**

In [ ]:
from google.colab import drive


DRIVE_MOUNT_POINT = Path("/content/drive")
MY_DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"

drive.mount(
    str(DRIVE_MOUNT_POINT),
    force_remount=False,
)

if not DRIVE_MOUNT_POINT.exists():
    raise RuntimeError(
        "Google Drive gagal dipasang pada runtime Colab."
    )

if not MY_DRIVE_ROOT.exists():
    raise RuntimeError(
        "Folder MyDrive tidak ditemukan setelah proses mount."
    )

if not MY_DRIVE_ROOT.is_dir():
    raise RuntimeError(
        f"Path MyDrive bukan direktori: {MY_DRIVE_ROOT}"
    )

print(f"Drive mount : {DRIVE_MOUNT_POINT}")
print(f"MyDrive     : {MY_DRIVE_ROOT}")
print(f"Data root   : {DATA_ROOT}")
print("\n✅ Google Drive berhasil dihubungkan dan divalidasi.")

**Cell 7 — Code: Menyiapkan command runner untuk Git**

In [ ]:
import subprocess
from collections.abc import Sequence


def run_command(
    command: Sequence[str],
    working_directory: Path | None = None,
    check: bool = True,
) -> subprocess.CompletedProcess:
    """
    Menjalankan perintah sistem secara terkontrol.

    Parameters
    ----------
    command:
        Daftar command dan argumennya.
    working_directory:
        Direktori tempat command dijalankan.
    check:
        Jika True, hentikan proses ketika command gagal.
    """

    result = subprocess.run(
        list(command),
        cwd=(
            str(working_directory)
            if working_directory
            else None
        ),
        capture_output=True,
        text=True,
        check=False,
    )

    if result.stdout.strip():
        print(result.stdout.strip())

    if result.returncode != 0 and check:
        error_detail = (
            result.stderr.strip()
            or result.stdout.strip()
            or "Unknown command error"
        )

        raise RuntimeError(
            f"Command gagal: {' '.join(command)}\n"
            f"Exit code: {result.returncode}\n"
            f"Detail: {error_detail[-4000:]}"
        )

    return result


git_version_result = run_command(["git", "--version"])

print("\n✅ Command runner siap dan Git berhasil ditemukan.")

**Cell 8 — Code: Menghubungkan repository GitHub**

In [ ]:
GITHUB_URL = (
    f"https://github.com/"
    f"{GITHUB_OWNER}/{GITHUB_REPOSITORY}.git"
)

GIT_DIRECTORY = REPOSITORY_ROOT / ".git"

if REPOSITORY_ROOT.exists() and not GIT_DIRECTORY.exists():
    if any(REPOSITORY_ROOT.iterdir()):
        raise RuntimeError(
            f"Folder {REPOSITORY_ROOT} sudah berisi file, "
            "tetapi bukan repository Git."
        )

if not GIT_DIRECTORY.exists():
    print("Repository belum tersedia. Memulai clone...")

    run_command(
        [
            "git",
            "clone",
            GITHUB_URL,
            str(REPOSITORY_ROOT),
        ]
    )
else:
    print("Repository sudah tersedia. Memeriksa remote...")

    remote_url_result = run_command(
        ["git", "remote", "get-url", "origin"],
        working_directory=REPOSITORY_ROOT,
    )

    actual_remote_url = remote_url_result.stdout.strip()

    valid_remote_urls = {
        GITHUB_URL,
        GITHUB_URL.removesuffix(".git"),
    }

    if actual_remote_url not in valid_remote_urls:
        raise RuntimeError(
            "Remote repository tidak sesuai.\n"
            f"Diharapkan : {GITHUB_URL}\n"
            f"Ditemukan  : {actual_remote_url}"
        )

    local_changes_result = run_command(
        ["git", "status", "--porcelain"],
        working_directory=REPOSITORY_ROOT,
        check=False,
    )

    has_local_changes = bool(
        local_changes_result.stdout.strip()
    )

    run_command(
        ["git", "fetch", "origin", "--prune"],
        working_directory=REPOSITORY_ROOT,
    )

    remote_branch_result = run_command(
        [
            "git",
            "rev-parse",
            "--verify",
            f"origin/{DEFAULT_BRANCH}",
        ],
        working_directory=REPOSITORY_ROOT,
        check=False,
    )

    if remote_branch_result.returncode == 0:
        current_branch_result = run_command(
            ["git", "branch", "--show-current"],
            working_directory=REPOSITORY_ROOT,
            check=False,
        )

        current_branch = (
            current_branch_result.stdout.strip()
        )

        if (
            current_branch == DEFAULT_BRANCH
            and not has_local_changes
        ):
            run_command(
                [
                    "git",
                    "pull",
                    "--ff-only",
                    "origin",
                    DEFAULT_BRANCH,
                ],
                working_directory=REPOSITORY_ROOT,
            )
        elif has_local_changes:
            print(
                "ℹ️ Ada perubahan lokal. Pull otomatis dilewati "
                "untuk mencegah konflik."
            )
        else:
            print(
                f"ℹ️ Branch aktif adalah '{current_branch}'. "
                "Pull otomatis dilewati."
            )
    else:
        print(
            "ℹ️ Repository belum mempunyai commit pada "
            f"branch {DEFAULT_BRANCH}."
        )

if not GIT_DIRECTORY.exists():
    raise RuntimeError(
        "Repository GitHub gagal dihubungkan."
    )

final_remote = run_command(
    ["git", "remote", "get-url", "origin"],
    working_directory=REPOSITORY_ROOT,
).stdout.strip()

print(f"\nRepository : {GITHUB_OWNER}/{GITHUB_REPOSITORY}")
print(f"Remote     : {final_remote}")
print(f"Local path : {REPOSITORY_ROOT}")
print("\n✅ Repository GitHub berhasil dihubungkan.")

**Cell 9 — Code: Memvalidasi hasil Notebook 00**

In [ ]:
import json


ENVIRONMENT_MANIFEST_PATH = (
    DATA_ROOT / "environment_manifest.json"
)

if not ENVIRONMENT_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "Environment manifest tidak ditemukan:\n"
        f"{ENVIRONMENT_MANIFEST_PATH}\n\n"
        "Pastikan 00_environment_setup.ipynb telah dijalankan "
        "sampai selesai pada Google Drive yang sama."
    )

try:
    environment_manifest = json.loads(
        ENVIRONMENT_MANIFEST_PATH.read_text(
            encoding="utf-8"
        )
    )
except json.JSONDecodeError as error:
    raise RuntimeError(
        "Environment manifest ditemukan, tetapi format JSON rusak."
    ) from error

manifest_project = environment_manifest.get(
    "project",
    {},
)

manifest_readiness = environment_manifest.get(
    "readiness_checks",
    {},
)

manifest_project_name = manifest_project.get("name")
failed_previous_checks = [
    check_name
    for check_name, check_result
    in manifest_readiness.items()
    if check_result is not True
]

if manifest_project_name != PROJECT_NAME:
    raise RuntimeError(
        "Manifest berasal dari proyek yang berbeda.\n"
        f"Diharapkan : {PROJECT_NAME}\n"
        f"Ditemukan  : {manifest_project_name}"
    )

if not manifest_readiness:
    raise RuntimeError(
        "Manifest tidak memiliki readiness checks."
    )

if failed_previous_checks:
    raise RuntimeError(
        "Notebook 00 memiliki pemeriksaan yang belum berhasil: "
        f"{failed_previous_checks}"
    )

print(f"Manifest path : {ENVIRONMENT_MANIFEST_PATH}")
print(f"Project       : {manifest_project_name}")
print(
    "Created UTC   : "
    f"{environment_manifest.get('created_at_utc', 'UNKNOWN')}"
)
print(
    "Paddle device : "
    f"{environment_manifest.get('hardware', {}).get('paddle_device', 'UNKNOWN')}"
)
print(
    "Packages      : "
    f"{len(environment_manifest.get('packages', {}))}"
)

print("\n✅ Environment manifest Notebook 00 valid.")

**Cell 10 — Code: Memvalidasi dependensi runtime saat ini**

In [ ]:
from importlib.metadata import (
    PackageNotFoundError,
    version,
)


manifest_packages = environment_manifest.get(
    "packages",
    {},
)

if not manifest_packages:
    raise RuntimeError(
        "Daftar package tidak ditemukan dalam environment manifest."
    )

current_package_versions = {}
missing_packages = []
version_differences = []

for package_name, expected_version in manifest_packages.items():
    try:
        current_version = version(package_name)
        current_package_versions[package_name] = current_version

        if current_version != expected_version:
            version_differences.append(
                {
                    "package": package_name,
                    "expected": expected_version,
                    "current": current_version,
                }
            )

    except PackageNotFoundError:
        current_package_versions[package_name] = "NOT FOUND"
        missing_packages.append(package_name)

print(
    f"{'Package':30} "
    f"{'Expected':15} "
    f"{'Current':15} "
    f"{'Status'}"
)
print("-" * 78)

for package_name, expected_version in manifest_packages.items():
    current_version = current_package_versions[package_name]

    if current_version == "NOT FOUND":
        status = "MISSING"
    elif current_version == expected_version:
        status = "MATCH"
    else:
        status = "DIFFERENT"

    print(
        f"{package_name:30} "
        f"{expected_version:15} "
        f"{current_version:15} "
        f"{status}"
    )

if missing_packages:
    raise RuntimeError(
        "Package berikut tidak tersedia pada runtime saat ini: "
        f"{missing_packages}. "
        "Jalankan kembali bagian instalasi pada Notebook 00."
    )

if version_differences:
    print(
        "\n⚠️ Terdapat perbedaan versi package. "
        "Notebook dapat dilanjutkan, tetapi perbedaan ini "
        "akan dicatat pada audit manifest."
    )
else:
    print(
        "\n✅ Seluruh versi package sesuai dengan "
        "environment manifest."
    )

print("✅ Dependensi runtime tersedia.")

**Cell 10A — Code: Menyinkronkan environment**

In [ ]:
import importlib


required_versions = environment_manifest["packages"]

paddle_version = required_versions["paddlepaddle"]

print(
    f"Memasang PaddlePaddle {paddle_version} "
    "dari repository resmi..."
)

run_command(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        f"paddlepaddle=={paddle_version}",
        "-i",
        "https://www.paddlepaddle.org.cn/packages/stable/cpu/",
    ]
)

packages_to_synchronize = [
    f"paddleocr=={required_versions['paddleocr']}",
    f"PyMuPDF=={required_versions['PyMuPDF']}",
    f"streamlit=={required_versions['streamlit']}",
    (
        "opencv-python-headless=="
        f"{required_versions['opencv-python-headless']}"
    ),
]

print("Menyinkronkan package yang hilang atau berbeda...")

run_command(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "--upgrade",
        "--upgrade-strategy",
        "only-if-needed",
        *packages_to_synchronize,
    ]
)

importlib.invalidate_caches()

print("\nPackage yang disinkronkan:")

for package_specification in packages_to_synchronize:
    print(f"  ✅ {package_specification}")

print(f"  ✅ paddlepaddle=={paddle_version}")
print("\n✅ Sinkronisasi environment selesai.")

**Cell 11 — Code: Mengimpor library audit**

In [ ]:
import hashlib
import logging
import mimetypes
import os
import re
import warnings
from collections import Counter
from dataclasses import asdict, dataclass
from typing import Any

import cv2
import numpy as np
import pandas as pd
import pymupdf
from PIL import (
    Image,
    UnidentifiedImageError,
)


# Konfigurasi tampilan tabel audit.
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

# Logging terstruktur.
logging.basicConfig(
    level=logging.INFO,
    format=(
        "%(asctime)s | %(levelname)s | "
        "%(name)s | %(message)s"
    ),
    force=True,
)

logger = logging.getLogger(NOTEBOOK_ID)

logger.info("Library audit berhasil dimuat.")
logger.info("Pandas version: %s", pd.__version__)
logger.info("NumPy version: %s", np.__version__)
logger.info("OpenCV version: %s", cv2.__version__)
logger.info("PyMuPDF version: %s", pymupdf.__version__)
logger.info("Pillow version: %s", Image.__version__)

print("\n✅ Seluruh library audit berhasil diimpor.")

**Cell 12 — Code: Memvalidasi struktur penyimpanan data**

In [ ]:
DATA_DIRECTORIES = {
    "raw": RAW_DIR,
    "interim": INTERIM_DIR,
    "processed": PROCESSED_DIR,
    "samples": SAMPLES_DIR,
    "audit_output": AUDIT_OUTPUT_DIR,
}

for directory_name, directory_path in DATA_DIRECTORIES.items():
    directory_path.mkdir(
        parents=True,
        exist_ok=True,
    )

    if not directory_path.exists():
        raise RuntimeError(
            f"Direktori gagal dibuat: {directory_path}"
        )

    if not directory_path.is_dir():
        raise RuntimeError(
            f"Path bukan direktori: {directory_path}"
        )

resolved_repository_root = REPOSITORY_ROOT.resolve()
resolved_data_root = DATA_ROOT.resolve()

if resolved_data_root.is_relative_to(
    resolved_repository_root
):
    raise RuntimeError(
        "DATA_ROOT berada di dalam repository GitHub. "
        "Kondisi ini dapat membocorkan invoice privat."
    )

directory_records = []

for directory_name, directory_path in DATA_DIRECTORIES.items():
    directory_records.append(
        {
            "directory": directory_name,
            "path": str(directory_path),
            "exists": directory_path.exists(),
            "is_directory": directory_path.is_dir(),
        }
    )

directory_table = pd.DataFrame(directory_records)

display(directory_table)

print(f"Repository root : {resolved_repository_root}")
print(f"Private data    : {resolved_data_root}")
print(
    "\n✅ Struktur penyimpanan valid dan data privat "
    "terpisah dari repository."
)

**Cell 13 — Markdown: Kebijakan data dan governance**

## 3. Data Governance and Collection Policy

InvoiceFlow AI hanya boleh menggunakan dokumen yang sumber dan hak
penggunaannya dapat dipertanggungjawabkan.

### Sumber yang Diizinkan

- invoice sintetis yang dibuat khusus untuk pengembangan;
- dataset publik dengan lisensi yang mengizinkan penggunaan;
- invoice milik sendiri;
- invoice dari organisasi atau klien yang memberikan izin tertulis;
- dokumen yang telah dianonimkan secara sah.

### Sumber yang Dilarang

- invoice yang diperoleh tanpa izin;
- dokumen hasil kebocoran data;
- invoice pelanggan yang digunakan di luar tujuan persetujuan;
- dokumen berbayar yang didistribusikan kembali tanpa hak;
- data dengan lisensi yang tidak mengizinkan penggunaan;
- dokumen yang sumbernya tidak dapat diverifikasi.

### Privacy Classification

Setiap sumber data harus diberi salah satu klasifikasi berikut:

| Classification | Description |
|---|---|
| `PUBLIC` | Dataset publik dengan lisensi yang jelas |
| `SYNTHETIC` | Dokumen buatan yang tidak mewakili individu nyata |
| `INTERNAL` | Data internal yang penggunaannya telah diizinkan |
| `CONFIDENTIAL` | Data sensitif dengan akses terbatas |

### Ground Rules

1. File dalam folder `raw/` bersifat **immutable** dan tidak boleh diubah.
2. Hasil transformasi disimpan ke `interim/` atau `processed/`.
3. Source code tidak boleh berisi invoice privat.
4. Token, password, dan kredensial tidak boleh ditulis di notebook.
5. Setiap sumber harus mempunyai catatan asal, lisensi, dan izin penggunaan.
6. Dokumen sensitif tidak boleh ditampilkan pada screenshot, README, atau demo.
7. Sampel produk publik harus menggunakan data sintetis atau data berlisensi.
8. Status lisensi yang belum jelas harus diberi `REVIEW`, bukan dianggap aman.

**Cell 14 — Code: Membuat source registry**

In [ ]:
SOURCE_REGISTRY_PATH = (
    DATA_ROOT / "source_registry.csv"
)

SOURCE_REGISTRY_COLUMNS = [
    "source_id",
    "source_name",
    "source_type",
    "source_location",
    "owner_or_provider",
    "license_or_permission",
    "license_url",
    "privacy_classification",
    "ml_usage_allowed",
    "commercial_use_allowed",
    "attribution_required",
    "collection_date_utc",
    "declared_document_count",
    "review_status",
    "notes",
]

ALLOWED_SOURCE_TYPES = {
    "PUBLIC_DATASET",
    "SYNTHETIC",
    "INTERNAL",
    "CLIENT_AUTHORIZED",
}

ALLOWED_PRIVACY_CLASSIFICATIONS = {
    "PUBLIC",
    "SYNTHETIC",
    "INTERNAL",
    "CONFIDENTIAL",
}

ALLOWED_SOURCE_REVIEW_STATUSES = {
    "APPROVED",
    "REVIEW",
    "REJECTED",
}

if not SOURCE_REGISTRY_PATH.exists():
    empty_source_registry = pd.DataFrame(
        columns=SOURCE_REGISTRY_COLUMNS
    )

    empty_source_registry.to_csv(
        SOURCE_REGISTRY_PATH,
        index=False,
        encoding="utf-8",
    )

    print("Source registry baru berhasil dibuat.")
else:
    print("Source registry sudah tersedia dan tidak ditimpa.")

source_registry = pd.read_csv(
    SOURCE_REGISTRY_PATH,
    dtype="string",
    keep_default_na=False,
)

missing_registry_columns = [
    column
    for column in SOURCE_REGISTRY_COLUMNS
    if column not in source_registry.columns
]

unexpected_registry_columns = [
    column
    for column in source_registry.columns
    if column not in SOURCE_REGISTRY_COLUMNS
]

if missing_registry_columns:
    raise RuntimeError(
        "Source registry tidak memiliki kolom wajib: "
        f"{missing_registry_columns}"
    )

if unexpected_registry_columns:
    logger.warning(
        "Source registry memiliki kolom tambahan: %s",
        unexpected_registry_columns,
    )

print(f"Registry path : {SOURCE_REGISTRY_PATH}")
print(f"Total sources : {len(source_registry):,}")

display(source_registry.head())

print("\n✅ Struktur source registry valid.")

**Cell 15 — Code: Mendaftarkan sumber sintetis utama**

In [ ]:
SYNTHETIC_SOURCE_ID = "SYNTHETIC-INVOICE-V1"

synthetic_source_record = {
    "source_id": SYNTHETIC_SOURCE_ID,
    "source_name": (
        "InvoiceFlow AI Synthetic Invoice Dataset V1"
    ),
    "source_type": "SYNTHETIC",
    "source_location": "raw/synthetic_v1",
    "owner_or_provider": (
        "InvoiceFlow AI / Muhammad Mayyosa"
    ),
    "license_or_permission": (
        "Project-owned synthetic data; external dataset "
        "license will be defined before distribution"
    ),
    "license_url": "",
    "privacy_classification": "SYNTHETIC",
    "ml_usage_allowed": "true",
    "commercial_use_allowed": "true",
    "attribution_required": "false",
    # Dikosongkan sampai proses pembuatan dataset selesai.
    "collection_date_utc": "",
    "declared_document_count": "0",
    "review_status": "APPROVED",
    "notes": (
        "Dataset akan dibuat oleh proyek menggunakan template "
        "orisinal dan identitas fiktif. Target awal: 200 invoice, "
        "100 Indonesia dan 100 Inggris."
    ),
}

if (
    synthetic_source_record["source_type"]
    not in ALLOWED_SOURCE_TYPES
):
    raise ValueError(
        "source_type tidak termasuk nilai yang diizinkan."
    )

if (
    synthetic_source_record["privacy_classification"]
    not in ALLOWED_PRIVACY_CLASSIFICATIONS
):
    raise ValueError(
        "privacy_classification tidak valid."
    )

if (
    synthetic_source_record["review_status"]
    not in ALLOWED_SOURCE_REVIEW_STATUSES
):
    raise ValueError(
        "review_status tidak valid."
    )

source_already_registered = (
    source_registry["source_id"]
    .eq(SYNTHETIC_SOURCE_ID)
    .any()
)

if source_already_registered:
    print(
        f"ℹ️ Sumber {SYNTHETIC_SOURCE_ID} "
        "sudah terdaftar dan tidak ditambahkan ulang."
    )
else:
    new_source_dataframe = pd.DataFrame(
        [synthetic_source_record],
        columns=SOURCE_REGISTRY_COLUMNS,
    )

    updated_source_registry = pd.concat(
        [
            source_registry,
            new_source_dataframe,
        ],
        ignore_index=True,
    )

    temporary_registry_path = (
        SOURCE_REGISTRY_PATH.with_suffix(".tmp")
    )

    updated_source_registry.to_csv(
        temporary_registry_path,
        index=False,
        encoding="utf-8",
    )

    temporary_registry_path.replace(
        SOURCE_REGISTRY_PATH
    )

    source_registry = updated_source_registry

    print(
        f"✅ Sumber {SYNTHETIC_SOURCE_ID} "
        "berhasil didaftarkan."
    )

display(
    source_registry.loc[
        source_registry["source_id"].eq(
            SYNTHETIC_SOURCE_ID
        )
    ]
)

print(f"Total sources : {len(source_registry):,}")
print("✅ Source registry berhasil diperbarui.")

**Cell 16 — Markdown: Spesifikasi dataset sintetis V1**

## 4. Synthetic Dataset Specification

### Dataset Identity

| Attribute | Value |
|---|---|
| Dataset ID | `SYNTHETIC-INVOICE-V1` |
| Dataset name | InvoiceFlow AI Synthetic Invoice Dataset V1 |
| Document type | Vendor/Purchase Invoice |
| Target size | 200 dokumen unik |
| Languages | Indonesia dan Inggris |
| Random seed | `42` |
| Ownership | InvoiceFlow AI |
| Privacy | Seluruh identitas bersifat fiktif |
| Ground truth | JSON terstruktur untuk setiap invoice |

### Language Distribution

| Language | Documents | Percentage |
|---|---:|---:|
| Indonesia | 100 | 50% |
| Inggris | 100 | 50% |
| **Total** | **200** | **100%** |

### File Format Distribution

| Format | Documents | Percentage |
|---|---:|---:|
| PDF | 100 | 50% |
| PNG | 50 | 25% |
| JPG | 50 | 25% |
| **Total** | **200** | **100%** |

Setiap invoice dasar hanya dirender ke satu format agar tidak menciptakan
duplikat yang tidak disengaja.

### Document Quality Distribution

| Quality profile | Documents | Description |
|---|---:|---|
| Digital clean | 140 | Dokumen digital dengan teks dan layout bersih |
| Scan simulation | 60 | Simulasi blur, noise, rotasi, atau kontras rendah |
| **Total** | **200** |  |

Degradasi visual harus terkontrol dan parameter transformasinya dicatat dalam
ground truth.

### Layout Distribution

Dataset menggunakan 10 template orisinal dengan 20 invoice per template.

Template harus memiliki variasi:

- posisi logo dan identitas vendor;
- susunan alamat vendor dan pembeli;
- posisi nomor serta tanggal invoice;
- bentuk tabel;
- posisi subtotal, pajak, diskon, dan total;
- penggunaan warna yang tetap terbaca;
- orientasi dan kepadatan konten;
- satu atau beberapa halaman jika diperlukan.

### Currency Distribution

| Currency | Documents |
|---|---:|
| IDR | 80 |
| USD | 70 |
| EUR | 30 |
| GBP | 20 |
| **Total** | **200** |

Currency digunakan untuk menguji normalisasi simbol, kode mata uang, separator
ribuan, dan separator desimal. Dataset ini tidak digunakan sebagai mesin
kepatuhan pajak atau kurs valuta asing.

### Required Ground-Truth Fields

Setiap dokumen harus mempunyai ground truth untuk:

- `vendor`;
- `invoice_number`;
- `invoice_date`;
- `due_date`;
- `currency`;
- `subtotal`;
- `tax`;
- `discount`;
- `total`.

Nilai field boleh kosong hanya jika status field tersebut secara eksplisit
ditandai sebagai opsional pada ground truth.

### Financial Integrity

Setiap invoice normal harus memenuhi:

```text
total = subtotal + tax - discount

**Cell 17 — Markdown: Professional Dataset Protocol**

## 5. Professional Dataset Protocol

Dokumen ini menetapkan aturan teknis agar Synthetic Invoice Dataset V1 dapat
digunakan untuk pengembangan yang dapat diaudit dan direproduksi.

### 5.1 Dataset Classification

Dataset V1 diklasifikasikan sebagai:

```text
Engineering and Benchmark Dataset
```

Dataset ini digunakan untuk:

- membangun pipeline awal;
- menguji preprocessing dan OCR;
- mengembangkan field extraction;
- menguji normalisasi serta validasi finansial;
- mengembangkan duplicate detection;
- mengembangkan anomaly detection;
- membuat automated test.

Dataset ini belum boleh digunakan sebagai satu-satunya bukti bahwa sistem siap
dipakai pada invoice nyata.

---

### 5.2 Dataset Components

Dataset terdiri dari tiga komponen terpisah.

| Component | Purpose |
|---|---|
| `canonical` | 200 invoice unik dan valid sesuai spesifikasi Cell 16 |
| `duplicate_challenge` | Pasangan positif dan negatif untuk duplicate detection |
| `anomaly_challenge` | Invoice normal dan anomali terkontrol |

Jumlah 200 pada Cell 16 merujuk kepada dokumen `canonical`. Dokumen turunan
dalam challenge set tidak dihitung sebagai canonical invoice baru.

---

### 5.3 Canonical Dataset Split

| Split | Documents | Percentage | Purpose |
|---|---:|---:|---|
| Development | 120 | 60% | Pengembangan pipeline dan rules |
| Validation | 40 | 20% | Pemilihan threshold dan validasi |
| Test | 40 | 20% | Evaluasi akhir yang tidak digunakan untuk tuning |
| **Total** | **200** | **100%** |  |

Pembagian tidak dilakukan secara acak per file. Pemisahan dilakukan berdasarkan
kelompok template dan vendor.

```text
Development : Template 01–06
Validation  : Template 07–08
Test        : Template 09–10
```

Dengan aturan ini, test set mengandung layout yang tidak pernah digunakan
selama pengembangan.

---

### 5.4 Anti-Leakage Rules

Hal berikut tidak boleh muncul pada lebih dari satu split:

- `canonical_invoice_id`;
- `invoice_number`;
- `vendor_id`;
- template yang sama;
- file hasil render dari invoice dasar yang sama;
- exact duplicate;
- near duplicate;
- variasi visual dari dokumen yang sama.

Duplicate challenge hanya boleh dibuat dari kelompok test yang telah dikunci dan
tidak boleh digunakan untuk menyusun rules atau menentukan threshold.

---

### 5.5 Ground-Truth Schema

Setiap dokumen harus mempunyai satu file JSON dengan schema version yang jelas.

```json
{
  "schema_version": "1.0.0",
  "dataset_id": "SYNTHETIC-INVOICE-V1",
  "generator_version": "1.0.0",
  "document_id": "INV-SYN-000001",
  "canonical_invoice_id": "CANON-000001",
  "split": "development",
  "language": "id",
  "currency": "IDR",
  "template_id": "TPL-01",
  "document_profile": "digital_clean",
  "file": {
    "name": "INV-SYN-000001.pdf",
    "format": "pdf",
    "sha256": "DOCUMENT_SHA256",
    "size_bytes": 0,
    "page_count": 1
  },
  "fields": {
    "vendor": {
      "raw_text": "PT Contoh Nusantara",
      "normalized_value": "PT Contoh Nusantara",
      "data_type": "string",
      "present": true,
      "optional": false,
      "page_index": 0,
      "bbox": [0.10, 0.08, 0.45, 0.13]
    },
    "invoice_number": {
      "raw_text": "INV-2026-000001",
      "normalized_value": "INV-2026-000001",
      "data_type": "string",
      "present": true,
      "optional": false,
      "page_index": 0,
      "bbox": [0.68, 0.08, 0.90, 0.12]
    }
  },
  "financial_validation": {
    "subtotal": "100000.00",
    "tax": "11000.00",
    "discount": "5000.00",
    "total": "106000.00",
    "equation_valid": true,
    "rounding_mode": "ROUND_HALF_UP",
    "decimal_places": 2
  },
  "pages": [
    {
      "page_index": 0,
      "full_text": "FULL PAGE TRANSCRIPTION"
    }
  ],
  "generation": {
    "seed": 42,
    "created_at_utc": "ISO-8601 TIMESTAMP",
    "degradation_parameters": {}
  }
}
```

Contoh tersebut menunjukkan struktur, bukan nilai tetap yang harus disalin ke
semua invoice.

---

### 5.6 Annotation Rules

- Bounding box menggunakan format `[x_min, y_min, x_max, y_max]`.
- Koordinat dinormalisasi pada rentang `0.0–1.0`.
- `page_index` dimulai dari `0`.
- Tanggal hasil normalisasi menggunakan ISO `YYYY-MM-DD`.
- Nilai uang disimpan sebagai string desimal.
- Perhitungan finansial menggunakan `Decimal`, bukan `float`.
- `raw_text` harus sama dengan teks yang terlihat pada dokumen.
- `normalized_value` berisi nilai setelah normalisasi.
- Field yang tidak tampil harus menggunakan `present: false`.
- Transkripsi halaman diperlukan untuk evaluasi CER dan WER.
- Setiap perubahan schema harus menaikkan `schema_version`.

---

### 5.7 Duplicate Challenge Set

Duplicate challenge harus berisi:

| Category | Positive/Negative | Minimum Pairs |
|---|---|---:|
| Byte-identical copy | Positive | 10 |
| Re-encoded or resized copy | Positive | 10 |
| Same invoice rendered differently | Positive | 10 |
| Same vendor, different invoice | Negative | 10 |
| Same total, different vendor | Negative | 10 |
| Similar invoice number, different document | Negative | 10 |

Setiap pasangan harus memiliki:

- `pair_id`;
- `document_id_a`;
- `document_id_b`;
- `duplicate_type`;
- `is_duplicate`;
- alasan ground truth.

Exact duplicate diperbolehkan hanya di challenge set dan harus ditandai secara
eksplisit.

---

### 5.8 Anomaly Challenge Set

Anomaly challenge harus mencakup dokumen normal sebagai kontrol dan kasus:

- subtotal, pajak, diskon, dan total tidak konsisten;
- invoice number kosong atau memiliki pola tidak biasa;
- due date lebih awal daripada invoice date;
- nominal ekstrem dibandingkan pola vendor;
- currency tidak konsisten;
- invoice yang sengaja memiliki field penting hilang;
- kombinasi vendor, nomor, tanggal, dan total yang mencurigakan.

Label yang digunakan:

```text
NORMAL
POTENTIAL_ANOMALY
REVIEW_RECOMMENDED
```

Dataset tidak boleh menggunakan label:

```text
FRAUD
FRAUD_DETECTED
```

karena sistem tidak membuktikan tindakan fraud secara hukum.

---

### 5.9 Build and Publication Process

Dataset tidak dibuat langsung di folder final.

```text
interim/synthetic_v1_build/
        ↓
automated validation
        ↓
manual quality review
        ↓
raw/synthetic_v1/
```

Folder `raw/synthetic_v1/` hanya menerima dataset yang sudah lulus quality gate.
Setelah dipublikasikan, file canonical tidak boleh diedit. Perubahan harus
menghasilkan versi dataset baru.

---

### 5.10 Quality Gates

Dataset hanya dipublikasikan jika:

- 100% dokumen dapat dibuka;
- 100% dokumen mempunyai ground truth;
- 100% ID bersifat unik;
- 100% pasangan dokumen dan JSON cocok;
- 100% checksum dapat diverifikasi;
- 100% invoice normal memenuhi persamaan finansial;
- distribusi bahasa, format, currency, template, dan kualitas sesuai spesifikasi;
- tidak ada kebocoran antar-split;
- tidak ada exact duplicate pada canonical dataset;
- semua bounding box berada pada rentang valid;
- semua tanggal dan nilai uang dapat dinormalisasi;
- tidak ditemukan credential atau data pribadi nyata;
- seluruh kegagalan audit terdokumentasi.

Selain automated validation:

- minimal 10% dokumen dari setiap template diperiksa secara visual;
- seluruh dokumen yang gagal automated check diperiksa manual;
- test set dikunci setelah dipublikasikan.

---

### 5.11 Required Dataset Artifacts

Setiap versi dataset harus memiliki:

```text
synthetic_v1/
├── documents/
│   ├── development/
│   ├── validation/
│   └── test/
├── ground_truth/
│   ├── development/
│   ├── validation/
│   └── test/
├── challenges/
│   ├── duplicates/
│   └── anomalies/
├── manifests/
│   ├── document_manifest.csv
│   ├── duplicate_pairs.csv
│   ├── anomaly_labels.csv
│   └── checksums.sha256
├── dataset_card.md
├── quality_report.json
└── changelog.md
```

---

### 5.12 Dataset Versioning

Gunakan semantic versioning:

```text
1.0.0
│ │ │
│ │ └── Perbaikan metadata tanpa mengubah isi utama
│ └──── Penambahan dokumen atau annotation yang kompatibel
└────── Perubahan schema atau struktur yang tidak kompatibel
```

Setiap versi harus mencatat:

- perubahan jumlah dokumen;
- perubahan schema;
- generator version;
- alasan perubahan;
- hasil quality gate;
- checksum manifest;
- waktu publikasi.

---

### 5.13 Real-World Validation Gate

Sebelum membuat klaim production-ready, sistem harus diuji secara terpisah
menggunakan invoice nyata yang:

- diperoleh dengan izin;
- dianonimkan;
- tidak digunakan untuk menyusun rules;
- berasal dari vendor dan layout yang tidak ada pada data sintetis;
- disimpan secara privat;
- tidak didistribusikan bersama source code.

Target pilot validation adalah sedikitnya 200 invoice nyata berizin dari
berbagai kelompok vendor dan layout. Hasil sintetis dan hasil real-world harus
dilaporkan secara terpisah.

---

### 5.14 Reporting Integrity

Laporan performa harus menyebutkan secara jelas:

- jenis dataset yang digunakan;
- jumlah data;
- proporsi data sintetis dan nyata;
- pembagian development, validation, dan test;
- known limitations;
- kegagalan sistem;
- confidence interval jika memungkinkan;
- bahwa hasil pada data sintetis tidak otomatis mewakili performa produksi.

**Cell 18 — Code: Audit instalasi OpenCV**

In [ ]:
from importlib.metadata import (
    PackageNotFoundError,
    version,
)


OPENCV_DISTRIBUTIONS = [
    "opencv-python",
    "opencv-python-headless",
    "opencv-contrib-python",
    "opencv-contrib-python-headless",
]

opencv_installation_records = []

for distribution_name in OPENCV_DISTRIBUTIONS:
    try:
        distribution_version = version(
            distribution_name
        )
        installation_status = "INSTALLED"
    except PackageNotFoundError:
        distribution_version = None
        installation_status = "NOT INSTALLED"

    opencv_installation_records.append(
        {
            "distribution": distribution_name,
            "distribution_version": distribution_version,
            "status": installation_status,
        }
    )

opencv_installation_table = pd.DataFrame(
    opencv_installation_records
)

display(opencv_installation_table)

installed_opencv_distributions = [
    record
    for record in opencv_installation_records
    if record["status"] == "INSTALLED"
]

active_cv2_version = cv2.__version__

print(f"Active cv2 version : {active_cv2_version}")
print(
    "Installed OpenCV distributions : "
    f"{len(installed_opencv_distributions)}"
)

if len(installed_opencv_distributions) == 0:
    raise RuntimeError(
        "Tidak ada distribusi OpenCV yang terdeteksi."
    )

if len(installed_opencv_distributions) > 1:
    print(
        "\n⚠️ Lebih dari satu distribusi OpenCV terpasang. "
        "Kondisi ini dapat menimbulkan konflik modul cv2."
    )
else:
    print(
        "\n✅ Hanya satu distribusi OpenCV yang terpasang."
    )

print(
    "✅ Audit OpenCV selesai. "
    "Jangan menghapus package sebelum hasilnya diperiksa."
)

**Cell 19 — Code: Menormalkan instalasi OpenCV**

In [ ]:
import importlib


OPENCV_TARGET_PACKAGE = "opencv-contrib-python"
OPENCV_TARGET_VERSION = "4.10.0.84"

OPENCV_PACKAGES_TO_REMOVE = [
    "opencv-python",
    "opencv-python-headless",
    "opencv-contrib-python",
    "opencv-contrib-python-headless",
]

print("Menghapus distribusi OpenCV yang saling bertabrakan...")

run_command(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "--yes",
        *OPENCV_PACKAGES_TO_REMOVE,
    ]
)

print(
    "\nMemasang satu distribusi OpenCV "
    "yang menjadi standar proyek..."
)

run_command(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "--no-deps",
        "--force-reinstall",
        (
            f"{OPENCV_TARGET_PACKAGE}"
            f"=={OPENCV_TARGET_VERSION}"
        ),
    ]
)

importlib.invalidate_caches()

print(
    "\n✅ Normalisasi OpenCV selesai."
)
print(
    "Package aktif yang ditargetkan: "
    f"{OPENCV_TARGET_PACKAGE}=={OPENCV_TARGET_VERSION}"
)
print(
    "Jalankan ulang Cell 18 untuk verifikasi."
)

**Cell 20 — Code: Memperbarui environment manifest**

In [ ]:
import copy
import json
from importlib.metadata import version


OPENCV_MANIFEST_CHANGE_ID = (
    "opencv-distribution-normalization-v1"
)
OPENCV_STANDARD_PACKAGE = "opencv-contrib-python"
OPENCV_STANDARD_VERSION = version(
    OPENCV_STANDARD_PACKAGE
)

OPENCV_OBSOLETE_PACKAGES = {
    "opencv-python",
    "opencv-python-headless",
    "opencv-contrib-python-headless",
}

updated_environment_manifest = copy.deepcopy(
    environment_manifest
)

updated_manifest_packages = dict(
    updated_environment_manifest.get(
        "packages",
        {},
    )
)

removed_manifest_packages = {}

for package_name in OPENCV_OBSOLETE_PACKAGES:
    if package_name in updated_manifest_packages:
        removed_manifest_packages[package_name] = (
            updated_manifest_packages.pop(package_name)
        )

updated_manifest_packages[
    OPENCV_STANDARD_PACKAGE
] = OPENCV_STANDARD_VERSION

updated_environment_manifest["packages"] = (
    updated_manifest_packages
)

maintenance_history = (
    updated_environment_manifest.setdefault(
        "maintenance_history",
        [],
    )
)

change_already_recorded = any(
    maintenance_record.get("change_id")
    == OPENCV_MANIFEST_CHANGE_ID
    for maintenance_record in maintenance_history
)

if not change_already_recorded:
    maintenance_history.append(
        {
            "change_id": OPENCV_MANIFEST_CHANGE_ID,
            "changed_at_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "reason": (
                "Removed conflicting OpenCV distributions "
                "and standardized the runtime on "
                "opencv-contrib-python."
            ),
            "removed_packages": (
                removed_manifest_packages
            ),
            "standard_package": (
                OPENCV_STANDARD_PACKAGE
            ),
            "standard_version": (
                OPENCV_STANDARD_VERSION
            ),
        }
    )

updated_environment_manifest[
    "updated_at_utc"
] = datetime.now(timezone.utc).isoformat()

temporary_manifest_path = (
    ENVIRONMENT_MANIFEST_PATH.with_suffix(".tmp")
)

temporary_manifest_path.write_text(
    json.dumps(
        updated_environment_manifest,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

temporary_manifest_path.replace(
    ENVIRONMENT_MANIFEST_PATH
)

# Baca kembali untuk memastikan hasil tersimpan dengan benar.
environment_manifest = json.loads(
    ENVIRONMENT_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

verified_packages = environment_manifest["packages"]

if "opencv-python-headless" in verified_packages:
    raise RuntimeError(
        "Package OpenCV lama masih terdapat dalam manifest."
    )

if (
    verified_packages.get(OPENCV_STANDARD_PACKAGE)
    != OPENCV_STANDARD_VERSION
):
    raise RuntimeError(
        "Package OpenCV standar gagal dicatat dalam manifest."
    )

print("Package OpenCV yang dihapus dari manifest:")

if removed_manifest_packages:
    for package_name, package_version in (
        removed_manifest_packages.items()
    ):
        print(f"  - {package_name}=={package_version}")
else:
    print("  - Tidak ada; perubahan sudah pernah diterapkan.")

print(
    "\nPackage OpenCV standar:"
    f"\n  + {OPENCV_STANDARD_PACKAGE}"
    f"=={OPENCV_STANDARD_VERSION}"
)

print(
    f"\nManifest path : {ENVIRONMENT_MANIFEST_PATH}"
)
print(
    "Total packages: "
    f"{len(environment_manifest['packages'])}"
)
print(
    "\n✅ Environment manifest berhasil diperbarui."
)

**Cell 21 — Code: Menyinkronkan requirements.txt**

In [ ]:
REQUIREMENTS_PATH = (
    REPOSITORY_ROOT / "requirements.txt"
)

DIRECT_DEPENDENCY_ORDER = [
    "paddlepaddle",
    "paddleocr",
    "PyMuPDF",
    "Pillow",
    "opencv-contrib-python",
    "pandas",
    "numpy",
    "scikit-learn",
    "sentence-transformers",
    "streamlit",
    "plotly",
    "openpyxl",
    "pytest",
]

manifest_packages = environment_manifest.get(
    "packages",
    {},
)

missing_direct_dependencies = [
    package_name
    for package_name in DIRECT_DEPENDENCY_ORDER
    if package_name not in manifest_packages
]

if missing_direct_dependencies:
    raise RuntimeError(
        "Manifest belum memiliki dependency berikut: "
        f"{missing_direct_dependencies}"
    )

requirement_lines = [
    (
        f"{package_name}"
        f"=={manifest_packages[package_name]}"
    )
    for package_name in DIRECT_DEPENDENCY_ORDER
]

requirements_content = (
    "# InvoiceFlow AI — Direct dependencies\n"
    "# Generated from the validated environment manifest.\n"
    "# Do not add more than one OpenCV distribution.\n"
    "# PaddlePaddle must be installed from its official index.\n\n"
    + "\n".join(requirement_lines)
    + "\n"
)

existing_requirements_content = ""

if REQUIREMENTS_PATH.exists():
    existing_requirements_content = (
        REQUIREMENTS_PATH.read_text(
            encoding="utf-8"
        )
    )

if (
    existing_requirements_content
    and existing_requirements_content
    != requirements_content
):
    requirements_backup_path = (
        AUDIT_OUTPUT_DIR
        / f"requirements_backup_{RUN_ID}.txt"
    )

    requirements_backup_path.write_text(
        existing_requirements_content,
        encoding="utf-8",
    )

    print(
        "Backup requirements lama:"
        f"\n  {requirements_backup_path}"
    )

temporary_requirements_path = (
    REQUIREMENTS_PATH.with_suffix(".tmp")
)

temporary_requirements_path.write_text(
    requirements_content,
    encoding="utf-8",
)

temporary_requirements_path.replace(
    REQUIREMENTS_PATH
)

saved_requirements_content = (
    REQUIREMENTS_PATH.read_text(
        encoding="utf-8"
    )
)

if saved_requirements_content != requirements_content:
    raise RuntimeError(
        "requirements.txt gagal diverifikasi setelah disimpan."
    )

opencv_requirement_lines = [
    line
    for line in saved_requirements_content.splitlines()
    if line.lower().startswith("opencv")
]

if opencv_requirement_lines != [
    "opencv-contrib-python==4.10.0.84"
]:
    raise RuntimeError(
        "Konfigurasi OpenCV dalam requirements.txt tidak valid: "
        f"{opencv_requirement_lines}"
    )

print(f"\nRequirements path: {REQUIREMENTS_PATH}")
print("\nIsi requirements.txt:\n")
print(saved_requirements_content)
print("✅ requirements.txt berhasil disinkronkan.")

**Cell 22 — Code: Konfigurasi dataset yang dapat divalidasi mesin**

In [ ]:
DATASET_ID = "SYNTHETIC-INVOICE-V1"
DATASET_VERSION = "1.0.0"
GENERATOR_VERSION = "1.0.0"

CANONICAL_DOCUMENT_COUNT = 200
DOCUMENTS_PER_TEMPLATE = 20
RANDOM_SEED = 42

SPLIT_TEMPLATE_MAPPING = {
    "development": [
        "TPL-01",
        "TPL-02",
        "TPL-03",
        "TPL-04",
        "TPL-05",
        "TPL-06",
    ],
    "validation": [
        "TPL-07",
        "TPL-08",
    ],
    "test": [
        "TPL-09",
        "TPL-10",
    ],
}

EXPECTED_SPLIT_DISTRIBUTION = {
    "development": 120,
    "validation": 40,
    "test": 40,
}

# Setiap template berisi 10 invoice Indonesia
# dan 10 invoice Inggris.
LANGUAGE_PER_TEMPLATE = {
    "id": 10,
    "en": 10,
}

# Setiap template menghasilkan distribusi format yang sama.
FORMAT_PER_TEMPLATE = {
    "pdf": 10,
    "png": 5,
    "jpg": 5,
}

# Setiap template menghasilkan 14 dokumen bersih
# dan 6 dokumen dengan simulasi scan.
QUALITY_PER_TEMPLATE = {
    "digital_clean": 14,
    "scan_simulation": 6,
}

# Distribusi currency dikaitkan dengan bahasa/locale.
CURRENCY_PER_LANGUAGE_PER_TEMPLATE = {
    "id": {
        "IDR": 8,
        "USD": 2,
    },
    "en": {
        "USD": 5,
        "EUR": 3,
        "GBP": 2,
    },
}

EXPECTED_GLOBAL_DISTRIBUTIONS = {
    "language": {
        "id": 100,
        "en": 100,
    },
    "format": {
        "pdf": 100,
        "png": 50,
        "jpg": 50,
    },
    "quality": {
        "digital_clean": 140,
        "scan_simulation": 60,
    },
    "currency": {
        "IDR": 80,
        "USD": 70,
        "EUR": 30,
        "GBP": 20,
    },
}

CHALLENGE_SPECIFICATION = {
    "duplicate_positive_pairs": {
        "byte_identical": 10,
        "reencoded_or_resized": 10,
        "same_invoice_different_render": 10,
    },
    "duplicate_negative_pairs": {
        "same_vendor_different_invoice": 10,
        "same_total_different_vendor": 10,
        "similar_number_different_invoice": 10,
    },
    "anomaly_categories": [
        "financial_mismatch",
        "missing_critical_field",
        "invalid_date_sequence",
        "currency_inconsistency",
        "vendor_amount_outlier",
    ],
}

SYNTHETIC_DATASET_CONFIGURATION = {
    "dataset_id": DATASET_ID,
    "dataset_version": DATASET_VERSION,
    "generator_version": GENERATOR_VERSION,
    "random_seed": RANDOM_SEED,
    "canonical_document_count": CANONICAL_DOCUMENT_COUNT,
    "documents_per_template": DOCUMENTS_PER_TEMPLATE,
    "split_template_mapping": SPLIT_TEMPLATE_MAPPING,
    "expected_split_distribution": (
        EXPECTED_SPLIT_DISTRIBUTION
    ),
    "language_per_template": LANGUAGE_PER_TEMPLATE,
    "format_per_template": FORMAT_PER_TEMPLATE,
    "quality_per_template": QUALITY_PER_TEMPLATE,
    "currency_per_language_per_template": (
        CURRENCY_PER_LANGUAGE_PER_TEMPLATE
    ),
    "expected_global_distributions": (
        EXPECTED_GLOBAL_DISTRIBUTIONS
    ),
    "challenge_specification": CHALLENGE_SPECIFICATION,
}

# ---------------------------------------------------------
# Automated configuration validation
# ---------------------------------------------------------

all_template_ids = [
    template_id
    for template_ids in SPLIT_TEMPLATE_MAPPING.values()
    for template_id in template_ids
]

if len(all_template_ids) != len(set(all_template_ids)):
    raise ValueError(
        "Satu template ditemukan pada lebih dari satu split."
    )

if len(all_template_ids) != 10:
    raise ValueError(
        "Dataset harus mempunyai tepat 10 template."
    )

if sum(LANGUAGE_PER_TEMPLATE.values()) != DOCUMENTS_PER_TEMPLATE:
    raise ValueError(
        "Distribusi bahasa per template tidak berjumlah 20."
    )

if sum(FORMAT_PER_TEMPLATE.values()) != DOCUMENTS_PER_TEMPLATE:
    raise ValueError(
        "Distribusi format per template tidak berjumlah 20."
    )

if sum(QUALITY_PER_TEMPLATE.values()) != DOCUMENTS_PER_TEMPLATE:
    raise ValueError(
        "Distribusi kualitas per template tidak berjumlah 20."
    )

for language, currency_distribution in (
    CURRENCY_PER_LANGUAGE_PER_TEMPLATE.items()
):
    if sum(currency_distribution.values()) != (
        LANGUAGE_PER_TEMPLATE[language]
    ):
        raise ValueError(
            "Distribusi currency tidak cocok untuk bahasa "
            f"'{language}'."
        )

calculated_split_distribution = {
    split_name: (
        len(template_ids) * DOCUMENTS_PER_TEMPLATE
    )
    for split_name, template_ids
    in SPLIT_TEMPLATE_MAPPING.items()
}

if calculated_split_distribution != (
    EXPECTED_SPLIT_DISTRIBUTION
):
    raise ValueError(
        "Distribusi split tidak sesuai spesifikasi."
    )

if sum(calculated_split_distribution.values()) != (
    CANONICAL_DOCUMENT_COUNT
):
    raise ValueError(
        "Jumlah dokumen canonical tidak sama dengan 200."
    )

for distribution_name, distribution_values in (
    EXPECTED_GLOBAL_DISTRIBUTIONS.items()
):
    if sum(distribution_values.values()) != (
        CANONICAL_DOCUMENT_COUNT
    ):
        raise ValueError(
            f"Distribusi '{distribution_name}' "
            "tidak berjumlah 200."
        )

registered_source = source_registry.loc[
    source_registry["source_id"].eq(DATASET_ID)
]

if registered_source.empty:
    raise RuntimeError(
        f"Sumber {DATASET_ID} belum terdaftar."
    )

if (
    registered_source.iloc[0]["review_status"]
    != "APPROVED"
):
    raise RuntimeError(
        f"Sumber {DATASET_ID} belum berstatus APPROVED."
    )

split_table = pd.DataFrame(
    [
        {
            "split": split_name,
            "templates": len(template_ids),
            "template_ids": ", ".join(template_ids),
            "documents": calculated_split_distribution[
                split_name
            ],
        }
        for split_name, template_ids
        in SPLIT_TEMPLATE_MAPPING.items()
    ]
)

distribution_table = pd.DataFrame(
    [
        {
            "dimension": dimension_name,
            "category": category,
            "documents": document_count,
        }
        for dimension_name, distribution
        in EXPECTED_GLOBAL_DISTRIBUTIONS.items()
        for category, document_count
        in distribution.items()
    ]
)

display(split_table)
display(distribution_table)

print(
    f"Dataset ID          : {DATASET_ID}"
)
print(
    f"Canonical documents : {CANONICAL_DOCUMENT_COUNT}"
)
print(
    f"Total templates     : {len(all_template_ids)}"
)
print(
    f"Configuration seed  : {RANDOM_SEED}"
)
print(
    "\n✅ Konfigurasi dataset valid dan konsisten."
)

**Cell 23 — Code: Membuat staging area datase**

In [ ]:
BUILD_BASE_ROOT = (
    INTERIM_DIR / "synthetic_v1_build"
)

BUILD_RUN_ROOT = (
    BUILD_BASE_ROOT / RUN_ID
)

PUBLISH_ROOT = (
    RAW_DIR / "synthetic_v1"
)

BUILD_DOCUMENTS_ROOT = (
    BUILD_RUN_ROOT / "documents"
)

BUILD_GROUND_TRUTH_ROOT = (
    BUILD_RUN_ROOT / "ground_truth"
)

BUILD_CHALLENGES_ROOT = (
    BUILD_RUN_ROOT / "challenges"
)

BUILD_MANIFESTS_ROOT = (
    BUILD_RUN_ROOT / "manifests"
)

BUILD_CONFIGURATION_ROOT = (
    BUILD_RUN_ROOT / "configuration"
)

BUILD_LOGS_ROOT = (
    BUILD_RUN_ROOT / "logs"
)

BUILD_DIRECTORY_REGISTRY = {
    "build_run": BUILD_RUN_ROOT,
    "documents": BUILD_DOCUMENTS_ROOT,
    "ground_truth": BUILD_GROUND_TRUTH_ROOT,
    "duplicate_challenge": (
        BUILD_CHALLENGES_ROOT / "duplicates"
    ),
    "anomaly_challenge": (
        BUILD_CHALLENGES_ROOT / "anomalies"
    ),
    "manifests": BUILD_MANIFESTS_ROOT,
    "configuration": BUILD_CONFIGURATION_ROOT,
    "logs": BUILD_LOGS_ROOT,
}

for split_name in EXPECTED_SPLIT_DISTRIBUTION:
    BUILD_DIRECTORY_REGISTRY[
        f"documents_{split_name}"
    ] = BUILD_DOCUMENTS_ROOT / split_name

    BUILD_DIRECTORY_REGISTRY[
        f"ground_truth_{split_name}"
    ] = BUILD_GROUND_TRUTH_ROOT / split_name

for directory_path in BUILD_DIRECTORY_REGISTRY.values():
    directory_path.mkdir(
        parents=True,
        exist_ok=True,
    )

resolved_build_root = BUILD_RUN_ROOT.resolve()
resolved_interim_root = INTERIM_DIR.resolve()
resolved_publish_root = PUBLISH_ROOT.resolve()
resolved_raw_root = RAW_DIR.resolve()

if not resolved_build_root.is_relative_to(
    resolved_interim_root
):
    raise RuntimeError(
        "Build directory harus berada di dalam interim/."
    )

if not resolved_publish_root.is_relative_to(
    resolved_raw_root
):
    raise RuntimeError(
        "Publish directory harus berada di dalam raw/."
    )

invalid_directories = [
    directory_name
    for directory_name, directory_path
    in BUILD_DIRECTORY_REGISTRY.items()
    if (
        not directory_path.exists()
        or not directory_path.is_dir()
        or directory_path.is_symlink()
    )
]

if invalid_directories:
    raise RuntimeError(
        "Direktori build tidak valid: "
        f"{invalid_directories}"
    )

build_directory_table = pd.DataFrame(
    [
        {
            "directory": directory_name,
            "relative_path": str(
                directory_path.relative_to(
                    BUILD_RUN_ROOT
                )
            ),
            "exists": directory_path.exists(),
        }
        for directory_name, directory_path
        in BUILD_DIRECTORY_REGISTRY.items()
    ]
)

display(build_directory_table)

print(f"Build run     : {BUILD_RUN_ROOT}")
print(f"Future publish: {PUBLISH_ROOT}")
print(
    "Publish exists: "
    f"{PUBLISH_ROOT.exists()}"
)

print(
    "\n✅ Staging area berhasil dibuat. "
    "Folder final belum dipublikasikan."
)

Cell 24 — Code: Menyimpan konfigurasi build secara immutable

In [ ]:
BUILD_CONFIGURATION_PATH = (
    BUILD_CONFIGURATION_ROOT
    / "dataset_configuration.json"
)

BUILD_CONFIGURATION_HASH_PATH = (
    BUILD_CONFIGURATION_ROOT
    / "dataset_configuration.sha256"
)

source_record_for_build = (
    source_registry.loc[
        source_registry["source_id"].eq(
            DATASET_ID
        )
    ]
    .iloc[0]
    .to_dict()
)

build_configuration = {
    "configuration_schema_version": "1.0.0",
    "dataset": SYNTHETIC_DATASET_CONFIGURATION,
    "build": {
        "run_id": RUN_ID,
        "created_at_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "status": "INITIALIZED",
        "publication_status": "NOT_PUBLISHED",
    },
    "source": source_record_for_build,
    "paths": {
        "documents": "documents",
        "ground_truth": "ground_truth",
        "duplicate_challenge": (
            "challenges/duplicates"
        ),
        "anomaly_challenge": (
            "challenges/anomalies"
        ),
        "manifests": "manifests",
        "configuration": "configuration",
        "logs": "logs",
    },
}

serialized_build_configuration = json.dumps(
    build_configuration,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
)

configuration_bytes = (
    serialized_build_configuration.encode("utf-8")
)

configuration_sha256 = hashlib.sha256(
    configuration_bytes
).hexdigest()

temporary_configuration_path = (
    BUILD_CONFIGURATION_PATH.with_suffix(".tmp")
)

temporary_configuration_path.write_bytes(
    configuration_bytes
)

temporary_configuration_path.replace(
    BUILD_CONFIGURATION_PATH
)

hash_file_content = (
    f"{configuration_sha256}  "
    f"{BUILD_CONFIGURATION_PATH.name}\n"
)

temporary_hash_path = (
    BUILD_CONFIGURATION_HASH_PATH.with_suffix(".tmp")
)

temporary_hash_path.write_text(
    hash_file_content,
    encoding="utf-8",
)

temporary_hash_path.replace(
    BUILD_CONFIGURATION_HASH_PATH
)

# ---------------------------------------------------------
# Read-back verification
# ---------------------------------------------------------

saved_configuration_bytes = (
    BUILD_CONFIGURATION_PATH.read_bytes()
)

saved_configuration_sha256 = hashlib.sha256(
    saved_configuration_bytes
).hexdigest()

if saved_configuration_sha256 != configuration_sha256:
    raise RuntimeError(
        "Checksum konfigurasi berubah setelah disimpan."
    )

loaded_build_configuration = json.loads(
    saved_configuration_bytes.decode("utf-8")
)

if (
    loaded_build_configuration["dataset"]["dataset_id"]
    != DATASET_ID
):
    raise RuntimeError(
        "Dataset ID pada konfigurasi tersimpan tidak valid."
    )

if (
    loaded_build_configuration["build"]["run_id"]
    != RUN_ID
):
    raise RuntimeError(
        "Run ID pada konfigurasi tersimpan tidak valid."
    )

if (
    loaded_build_configuration["build"][
        "publication_status"
    ]
    != "NOT_PUBLISHED"
):
    raise RuntimeError(
        "Status publikasi awal tidak valid."
    )

print(
    f"Configuration : {BUILD_CONFIGURATION_PATH}"
)
print(
    f"Checksum file : {BUILD_CONFIGURATION_HASH_PATH}"
)
print(
    f"SHA-256       : {configuration_sha256}"
)
print(
    "Build status : "
    f"{loaded_build_configuration['build']['status']}"
)
print(
    "Publication  : "
    f"{loaded_build_configuration['build']['publication_status']}"
)

print(
    "\n✅ Konfigurasi build berhasil disimpan "
    "dan diverifikasi."
)

**Cell 25 — Code: Generator capability preflight**

In [ ]:
import tempfile


GENERATOR_FONT_REGULAR = "helv"
GENERATOR_FONT_BOLD = "hebo"
GENERATOR_PAGE_WIDTH = 595
GENERATOR_PAGE_HEIGHT = 842
GENERATOR_RENDER_DPI = 200

generator_capability_results = {
    "pdf_creation": False,
    "pdf_reopening": False,
    "pdf_text_extraction": False,
    "pdf_rasterization": False,
    "pillow_image_read": False,
    "opencv_image_read": False,
}

with tempfile.TemporaryDirectory(
    dir=str(BUILD_RUN_ROOT),
    prefix="generator_preflight_",
) as temporary_directory:
    temporary_root = Path(temporary_directory)

    test_pdf_path = (
        temporary_root / "generator_preflight.pdf"
    )

    test_png_path = (
        temporary_root / "generator_preflight.png"
    )

    expected_test_text = (
        "INVOICEFLOW AI GENERATOR PREFLIGHT"
    )

    # -----------------------------------------------------
    # Create a temporary PDF
    # -----------------------------------------------------

    test_document = pymupdf.open()

    test_page = test_document.new_page(
        width=GENERATOR_PAGE_WIDTH,
        height=GENERATOR_PAGE_HEIGHT,
    )

    test_page.draw_rect(
        pymupdf.Rect(40, 40, 555, 180),
        color=(0.12, 0.36, 0.24),
        fill=(0.95, 0.98, 0.96),
        width=1,
    )

    test_page.insert_text(
        pymupdf.Point(60, 80),
        expected_test_text,
        fontname=GENERATOR_FONT_BOLD,
        fontsize=16,
        color=(0.08, 0.20, 0.14),
    )

    test_page.insert_text(
        pymupdf.Point(60, 115),
        "Vendor: Example Nusantara Ltd.",
        fontname=GENERATOR_FONT_REGULAR,
        fontsize=11,
        color=(0, 0, 0),
    )

    test_page.insert_text(
        pymupdf.Point(60, 140),
        "Total: IDR 1,000,000.00",
        fontname=GENERATOR_FONT_REGULAR,
        fontsize=11,
        color=(0, 0, 0),
    )

    test_document.set_metadata(
        {
            "title": "InvoiceFlow AI Generator Preflight",
            "author": "InvoiceFlow AI",
            "subject": "Environment capability validation",
            "keywords": (
                "invoiceflow, synthetic invoice, preflight"
            ),
        }
    )

    test_document.save(
        test_pdf_path,
        garbage=4,
        deflate=True,
    )

    test_document.close()

    generator_capability_results[
        "pdf_creation"
    ] = (
        test_pdf_path.exists()
        and test_pdf_path.stat().st_size > 0
    )

    # -----------------------------------------------------
    # Reopen and inspect the generated PDF
    # -----------------------------------------------------

    with pymupdf.open(test_pdf_path) as reopened_document:
        generator_capability_results[
            "pdf_reopening"
        ] = reopened_document.page_count == 1

        extracted_text = (
            reopened_document[0].get_text("text")
        )

        generator_capability_results[
            "pdf_text_extraction"
        ] = expected_test_text in extracted_text

        render_scale = (
            GENERATOR_RENDER_DPI / 72
        )

        render_matrix = pymupdf.Matrix(
            render_scale,
            render_scale,
        )

        pixmap = reopened_document[0].get_pixmap(
            matrix=render_matrix,
            alpha=False,
        )

        pixmap.save(test_png_path)

    generator_capability_results[
        "pdf_rasterization"
    ] = (
        test_png_path.exists()
        and test_png_path.stat().st_size > 0
    )

    # -----------------------------------------------------
    # Validate the raster image
    # -----------------------------------------------------

    with Image.open(test_png_path) as pillow_image:
        pillow_image.verify()

    with Image.open(test_png_path) as pillow_image:
        generator_capability_results[
            "pillow_image_read"
        ] = (
            pillow_image.width > 0
            and pillow_image.height > 0
        )

        rendered_image_size = (
            pillow_image.width,
            pillow_image.height,
        )

    opencv_image = cv2.imread(
        str(test_png_path),
        cv2.IMREAD_COLOR,
    )

    generator_capability_results[
        "opencv_image_read"
    ] = (
        opencv_image is not None
        and opencv_image.size > 0
    )

preflight_table = pd.DataFrame(
    [
        {
            "capability": capability_name,
            "result": capability_result,
            "status": (
                "READY"
                if capability_result
                else "FAILED"
            ),
        }
        for capability_name, capability_result
        in generator_capability_results.items()
    ]
)

display(preflight_table)

failed_capabilities = [
    capability_name
    for capability_name, capability_result
    in generator_capability_results.items()
    if not capability_result
]

if failed_capabilities:
    raise RuntimeError(
        "Generator preflight gagal pada: "
        f"{failed_capabilities}"
    )

print(
    f"Rendered image size : {rendered_image_size}"
)
print(
    f"Render DPI          : {GENERATOR_RENDER_DPI}"
)
print(
    "Temporary files    : deleted automatically"
)

print(
    "\n✅ Generator PDF dan image siap digunakan."
)

**Cell 26 — Code: Model data invoice dan validasi finansial**

In [ ]:
from typing import Any
from dataclasses import dataclass
from datetime import date
from decimal import (
    Decimal,
    ROUND_HALF_UP,
)


CURRENCY_DECIMAL_PLACES = {
    "IDR": 0,
    "USD": 2,
    "EUR": 2,
    "GBP": 2,
}


def money_quantum(currency: str) -> Decimal:
    """
    Mengembalikan unit pembulatan terkecil untuk currency.
    """

    if currency not in CURRENCY_DECIMAL_PLACES:
        raise ValueError(
            f"Currency tidak didukung: {currency}"
        )

    decimal_places = CURRENCY_DECIMAL_PLACES[currency]

    return Decimal(1).scaleb(-decimal_places)


def quantize_money(
    value: Decimal,
    currency: str,
) -> Decimal:
    """
    Membulatkan nilai uang secara deterministik.
    """

    return value.quantize(
        money_quantum(currency),
        rounding=ROUND_HALF_UP,
    )


def decimal_to_string(
    value: Decimal,
    currency: str,
) -> str:
    """
    Mengubah Decimal menjadi string tanpa scientific notation.
    """

    normalized_value = quantize_money(
        value,
        currency,
    )

    decimal_places = CURRENCY_DECIMAL_PLACES[
        currency
    ]

    return format(
        normalized_value,
        f".{decimal_places}f",
    )


@dataclass(frozen=True, slots=True)
class SyntheticParty:
    """
    Identitas organisasi sintetis.
    """

    party_id: str
    name: str
    address_lines: tuple[str, ...]
    email: str
    phone: str
    tax_identifier: str

    def __post_init__(self) -> None:
        if not re.fullmatch(
            r"[A-Z]+-[A-Z0-9-]+",
            self.party_id,
        ):
            raise ValueError(
                f"party_id tidak valid: {self.party_id}"
            )

        if not self.name.strip():
            raise ValueError(
                "Nama organisasi tidak boleh kosong."
            )

        if not self.address_lines:
            raise ValueError(
                "Alamat sintetis tidak boleh kosong."
            )

        if not self.email.lower().endswith(
            "@example.com"
        ):
            raise ValueError(
                "Email sintetis wajib menggunakan example.com."
            )

        if not self.tax_identifier.startswith(
            "SYN-"
        ):
            raise ValueError(
                "Tax identifier sintetis wajib diawali SYN-."
            )


@dataclass(frozen=True, slots=True)
class SyntheticLineItem:
    """
    Satu baris item invoice.
    """

    description: str
    quantity: Decimal
    unit_price: Decimal

    def __post_init__(self) -> None:
        if not self.description.strip():
            raise ValueError(
                "Deskripsi item tidak boleh kosong."
            )

        if self.quantity <= 0:
            raise ValueError(
                "Quantity harus lebih besar dari nol."
            )

        if self.unit_price < 0:
            raise ValueError(
                "Unit price tidak boleh negatif."
            )

    def calculate_total(
        self,
        currency: str,
    ) -> Decimal:
        return quantize_money(
            self.quantity * self.unit_price,
            currency,
        )


@dataclass(frozen=True, slots=True)
class SyntheticInvoice:
    """
    Representasi canonical invoice sebelum rendering.
    """

    document_id: str
    canonical_invoice_id: str
    split: str
    language: str
    currency: str
    template_id: str
    vendor: SyntheticParty
    buyer: SyntheticParty
    invoice_number: str
    invoice_date: date
    due_date: date
    items: tuple[SyntheticLineItem, ...]
    tax_rate_percent: Decimal
    discount_amount: Decimal

    def __post_init__(self) -> None:
        if not re.fullmatch(
            r"INV-SYN-\d{6}",
            self.document_id,
        ):
            raise ValueError(
                f"document_id tidak valid: {self.document_id}"
            )

        if not re.fullmatch(
            r"CANON-\d{6}",
            self.canonical_invoice_id,
        ):
            raise ValueError(
                "canonical_invoice_id tidak valid: "
                f"{self.canonical_invoice_id}"
            )

        if self.split not in SPLIT_TEMPLATE_MAPPING:
            raise ValueError(
                f"Split tidak valid: {self.split}"
            )

        if self.template_id not in (
            SPLIT_TEMPLATE_MAPPING[self.split]
        ):
            raise ValueError(
                f"Template {self.template_id} "
                f"tidak diizinkan untuk split {self.split}."
            )

        if self.language not in LANGUAGE_PER_TEMPLATE:
            raise ValueError(
                f"Language tidak valid: {self.language}"
            )

        allowed_currencies = (
            CURRENCY_PER_LANGUAGE_PER_TEMPLATE[
                self.language
            ]
        )

        if self.currency not in allowed_currencies:
            raise ValueError(
                f"Currency {self.currency} tidak sesuai "
                f"dengan language {self.language}."
            )

        if not self.invoice_number.strip():
            raise ValueError(
                "Invoice number tidak boleh kosong."
            )

        if self.due_date < self.invoice_date:
            raise ValueError(
                "Due date tidak boleh lebih awal "
                "daripada invoice date."
            )

        if not 1 <= len(self.items) <= 12:
            raise ValueError(
                "Canonical invoice harus mempunyai 1–12 item."
            )

        if not Decimal("0") <= (
            self.tax_rate_percent
        ) <= Decimal("25"):
            raise ValueError(
                "Tax rate harus berada pada rentang 0–25%."
            )

        if self.discount_amount < 0:
            raise ValueError(
                "Discount tidak boleh negatif."
            )

        if self.discount_amount > (
            self.subtotal + self.tax_amount
        ):
            raise ValueError(
                "Discount tidak boleh melebihi "
                "subtotal ditambah tax."
            )

    @property
    def subtotal(self) -> Decimal:
        return quantize_money(
            sum(
                (
                    item.calculate_total(self.currency)
                    for item in self.items
                ),
                Decimal("0"),
            ),
            self.currency,
        )

    @property
    def tax_amount(self) -> Decimal:
        return quantize_money(
            (
                self.subtotal
                * self.tax_rate_percent
                / Decimal("100")
            ),
            self.currency,
        )

    @property
    def total(self) -> Decimal:
        return quantize_money(
            (
                self.subtotal
                + self.tax_amount
                - self.discount_amount
            ),
            self.currency,
        )


def serialize_party(
    party: SyntheticParty,
) -> dict[str, Any]:
    return {
        "party_id": party.party_id,
        "name": party.name,
        "address_lines": list(
            party.address_lines
        ),
        "email": party.email,
        "phone": party.phone,
        "tax_identifier": party.tax_identifier,
    }


def serialize_invoice(
    invoice: SyntheticInvoice,
) -> dict[str, Any]:
    """
    Mengubah invoice menjadi object yang aman disimpan ke JSON.
    """

    return {
        "document_id": invoice.document_id,
        "canonical_invoice_id": (
            invoice.canonical_invoice_id
        ),
        "split": invoice.split,
        "language": invoice.language,
        "currency": invoice.currency,
        "template_id": invoice.template_id,
        "vendor": serialize_party(invoice.vendor),
        "buyer": serialize_party(invoice.buyer),
        "invoice_number": invoice.invoice_number,
        "invoice_date": invoice.invoice_date.isoformat(),
        "due_date": invoice.due_date.isoformat(),
        "items": [
            {
                "description": item.description,
                "quantity": format(
                    item.quantity,
                    "f",
                ),
                "unit_price": decimal_to_string(
                    item.unit_price,
                    invoice.currency,
                ),
                "line_total": decimal_to_string(
                    item.calculate_total(
                        invoice.currency
                    ),
                    invoice.currency,
                ),
            }
            for item in invoice.items
        ],
        "financials": {
            "subtotal": decimal_to_string(
                invoice.subtotal,
                invoice.currency,
            ),
            "tax_rate_percent": format(
                invoice.tax_rate_percent,
                "f",
            ),
            "tax": decimal_to_string(
                invoice.tax_amount,
                invoice.currency,
            ),
            "discount": decimal_to_string(
                invoice.discount_amount,
                invoice.currency,
            ),
            "total": decimal_to_string(
                invoice.total,
                invoice.currency,
            ),
            "rounding_mode": "ROUND_HALF_UP",
            "decimal_places": (
                CURRENCY_DECIMAL_PLACES[
                    invoice.currency
                ]
            ),
            "equation_valid": True,
        },
    }


print(
    "Supported currencies : "
    f"{sorted(CURRENCY_DECIMAL_PLACES)}"
)
print(
    "Data models          : "
    "SyntheticParty, SyntheticLineItem, SyntheticInvoice"
)
print(
    "Monetary arithmetic  : Decimal + ROUND_HALF_UP"
)
print(
    "\n✅ Model data canonical invoice berhasil didefinisikan."
)

**Cell 27 — Code: Generator identitas sintetis yang aman**

In [ ]:
import random
import unicodedata
import random

RANDOM_SEED = 42


INDONESIAN_COMPANY_PREFIXES = [
    "PT",
    "CV",
]

INDONESIAN_BRAND_WORDS = [
    "Arunika",
    "Cakrawana",
    "Kiranusa",
    "Lenterasa",
    "Nusakarsa",
    "Pijaraya",
    "Ruangcipta",
    "Swaranusa",
    "Terasena",
    "Widyakara",
]

INDONESIAN_BUSINESS_WORDS = [
    "Distribusi Contoh",
    "Kreasi Simulasi",
    "Logistik Uji",
    "Niaga Contoh",
    "Solusi Simulasi",
    "Teknologi Uji",
]

ENGLISH_BRAND_WORDS = [
    "Asterwyn",
    "Brightmere",
    "Cinderlane",
    "Evermont",
    "Frostwell",
    "Lumenridge",
    "Northvale",
    "Silvergrove",
    "Westhaven",
    "Willowcrest",
]

ENGLISH_BUSINESS_WORDS = [
    "Example Analytics",
    "Demo Distribution",
    "Example Logistics",
    "Simulation Services",
    "Test Supplies",
    "Example Technologies",
]

ENGLISH_COMPANY_SUFFIXES = [
    "Ltd.",
    "LLC",
    "Inc.",
    "PLC",
]

INDONESIAN_ADDRESS_LINES = [
    "Jalan Contoh",
    "Kawasan Simulasi",
    "Kompleks Data Uji",
    "Koridor Fiktif",
]

ENGLISH_ADDRESS_LINES = [
    "Example Avenue",
    "Simulation Road",
    "Test Market Street",
    "Fictional Commerce Lane",
]


def normalize_identifier_text(value: str) -> str:
    """
    Mengubah teks menjadi token identifier ASCII.
    """

    normalized = unicodedata.normalize(
        "NFKD",
        value,
    )

    ascii_value = normalized.encode(
        "ascii",
        "ignore",
    ).decode("ascii")

    identifier = re.sub(
        r"[^A-Za-z0-9]+",
        "-",
        ascii_value,
    ).strip("-")

    return identifier.upper()


def generate_synthetic_party(
    role: str,
    language: str,
    index: int,
    rng: random.Random,
) -> SyntheticParty:
    """
    Membuat organisasi fiktif secara deterministik.

    role harus berupa 'vendor' atau 'buyer'.
    """

    if role not in {"vendor", "buyer"}:
        raise ValueError(
            f"Role tidak valid: {role}"
        )

    if language not in {"id", "en"}:
        raise ValueError(
            f"Language tidak valid: {language}"
        )

    if index < 1:
        raise ValueError(
            "Index harus dimulai dari 1."
        )

    role_code = (
        "VEN"
        if role == "vendor"
        else "BUY"
    )

    language_code = language.upper()

    party_id = (
        f"{role_code}-{language_code}-{index:04d}"
    )

    if language == "id":
        company_name = " ".join(
            [
                rng.choice(
                    INDONESIAN_COMPANY_PREFIXES
                ),
                rng.choice(
                    INDONESIAN_BRAND_WORDS
                ),
                rng.choice(
                    INDONESIAN_BUSINESS_WORDS
                ),
            ]
        )

        street_number = rng.randint(10, 999)
        block_letter = chr(
            ord("A") + rng.randint(0, 20)
        )
        block_number = rng.randint(1, 99)
        postal_code = 90000 + index

        address_lines = (
            (
                f"{rng.choice(INDONESIAN_ADDRESS_LINES)} "
                f"No. {street_number}"
            ),
            (
                f"Blok {block_letter}-{block_number}, "
                "Kota Uji"
            ),
            f"Kode Pos {postal_code}",
        )

        phone = (
            f"+62-000-{index:04d}-{rng.randint(1000, 9999)}"
        )

    else:
        company_name = " ".join(
            [
                rng.choice(
                    ENGLISH_BRAND_WORDS
                ),
                rng.choice(
                    ENGLISH_BUSINESS_WORDS
                ),
                rng.choice(
                    ENGLISH_COMPANY_SUFFIXES
                ),
            ]
        )

        street_number = rng.randint(10, 9999)
        postal_code = 90000 + index

        address_lines = (
            (
                f"{street_number} "
                f"{rng.choice(ENGLISH_ADDRESS_LINES)}"
            ),
            "Simulation District",
            f"Test City, ZZ {postal_code}",
        )

        phone = (
            f"+1-555-01{index % 100:02d}"
        )

    email_local_part = (
        f"{role}.{language}.{index:04d}"
    )

    email = (
        f"{email_local_part}@example.com"
    )

    tax_identifier = (
        f"SYN-{language_code}-{role_code}-{index:06d}"
    )

    party = SyntheticParty(
        party_id=party_id,
        name=company_name,
        address_lines=address_lines,
        email=email,
        phone=phone,
        tax_identifier=tax_identifier,
    )

    return party


# ---------------------------------------------------------
# Determinism verification
# ---------------------------------------------------------

determinism_rng_a = random.Random(RANDOM_SEED)
determinism_rng_b = random.Random(RANDOM_SEED)

party_test_a = generate_synthetic_party(
    role="vendor",
    language="id",
    index=1,
    rng=determinism_rng_a,
)

party_test_b = generate_synthetic_party(
    role="vendor",
    language="id",
    index=1,
    rng=determinism_rng_b,
)

if party_test_a != party_test_b:
    raise RuntimeError(
        "Generator identitas tidak deterministik."
    )

english_party_test = generate_synthetic_party(
    role="buyer",
    language="en",
    index=1,
    rng=random.Random(RANDOM_SEED + 1),
)

party_preview = pd.DataFrame(
    [
        {
            "party_id": party_test_a.party_id,
            "language": "id",
            "name": party_test_a.name,
            "email": party_test_a.email,
            "tax_identifier": (
                party_test_a.tax_identifier
            ),
        },
        {
            "party_id": english_party_test.party_id,
            "language": "en",
            "name": english_party_test.name,
            "email": english_party_test.email,
            "tax_identifier": (
                english_party_test.tax_identifier
            ),
        },
    ]
)

display(party_preview)

print(
    "Deterministic test : PASSED"
)
print(
    "External data used : NO"
)
print(
    "Safe email domain  : example.com"
)
print(
    "\n✅ Generator identitas sintetis siap digunakan."
)

**Cell 28 — Code: Locale dan formatting engine**

In [ ]:
LOCALE_LABELS = {
    "id": {
        "invoice_title": "FAKTUR",
        "vendor": "PEMASOK",
        "bill_to": "DITAGIHKAN KEPADA",
        "invoice_number": "Nomor Faktur",
        "invoice_date": "Tanggal Faktur",
        "due_date": "Jatuh Tempo",
        "currency": "Mata Uang",
        "description": "Deskripsi",
        "quantity": "Kuantitas",
        "unit_price": "Harga Satuan",
        "line_total": "Total Baris",
        "subtotal": "Subtotal",
        "tax": "Pajak",
        "discount": "Diskon",
        "total": "TOTAL",
        "email": "Email",
        "phone": "Telepon",
        "tax_identifier": "ID Pajak Sintetis",
        "payment_terms": "Ketentuan Pembayaran",
        "thank_you": (
            "Terima kasih atas kerja sama Anda."
        ),
    },
    "en": {
        "invoice_title": "INVOICE",
        "vendor": "VENDOR",
        "bill_to": "BILL TO",
        "invoice_number": "Invoice Number",
        "invoice_date": "Invoice Date",
        "due_date": "Due Date",
        "currency": "Currency",
        "description": "Description",
        "quantity": "Quantity",
        "unit_price": "Unit Price",
        "line_total": "Line Total",
        "subtotal": "Subtotal",
        "tax": "Tax",
        "discount": "Discount",
        "total": "TOTAL",
        "email": "Email",
        "phone": "Phone",
        "tax_identifier": "Synthetic Tax ID",
        "payment_terms": "Payment Terms",
        "thank_you": (
            "Thank you for your business."
        ),
    },
}

ITEM_CATALOG = {
    "id": [
        "Perlengkapan administrasi",
        "Perangkat kantor",
        "Paket dokumentasi digital",
        "Layanan pemeliharaan sistem",
        "Konsultasi proses bisnis",
        "Jasa konfigurasi jaringan",
        "Pelatihan penggunaan sistem",
        "Layanan dukungan teknis",
        "Penyimpanan arsip digital",
        "Lisensi perangkat lunak contoh",
        "Paket analisis data",
        "Layanan integrasi sistem",
    ],
    "en": [
        "Administrative supplies",
        "Office equipment",
        "Digital documentation package",
        "System maintenance service",
        "Business process consulting",
        "Network configuration service",
        "System usage training",
        "Technical support service",
        "Digital archive storage",
        "Example software license",
        "Data analysis package",
        "System integration service",
    ],
}

PAYMENT_TERMS = {
    "id": [
        "Pembayaran 14 hari",
        "Pembayaran 30 hari",
        "Pembayaran 45 hari",
        "Pembayaran saat diterima",
    ],
    "en": [
        "Net 14",
        "Net 30",
        "Net 45",
        "Due on receipt",
    ],
}

TAX_RATE_OPTIONS = [
    Decimal("0"),
    Decimal("5"),
    Decimal("10"),
    Decimal("11"),
    Decimal("12"),
    Decimal("20"),
]

DISCOUNT_RATE_OPTIONS = [
    Decimal("0"),
    Decimal("0"),
    Decimal("0"),
    Decimal("2.5"),
    Decimal("5"),
    Decimal("10"),
]

CURRENCY_DISPLAY_CODES = {
    "IDR": "Rp",
    "USD": "USD",
    "EUR": "EUR",
    "GBP": "GBP",
}


def format_date_for_display(
    value: date,
    language: str,
) -> str:
    """
    Memformat tanggal sesuai locale dokumen.
    Ground truth tetap menggunakan ISO YYYY-MM-DD.
    """

    if language == "id":
        return value.strftime("%d/%m/%Y")

    if language == "en":
        return value.strftime("%m/%d/%Y")

    raise ValueError(
        f"Language tidak didukung: {language}"
    )


def format_number_for_display(
    value: Decimal,
    currency: str,
    language: str,
) -> str:
    """
    Memformat angka dengan separator locale.
    """

    normalized_value = quantize_money(
        value,
        currency,
    )

    decimal_places = (
        CURRENCY_DECIMAL_PLACES[currency]
    )

    formatted_value = format(
        normalized_value,
        f",.{decimal_places}f",
    )

    if language == "id":
        formatted_value = (
            formatted_value
            .replace(",", "__THOUSAND__")
            .replace(".", ",")
            .replace("__THOUSAND__", ".")
        )
    elif language != "en":
        raise ValueError(
            f"Language tidak didukung: {language}"
        )

    return formatted_value


def format_money_for_display(
    value: Decimal,
    currency: str,
    language: str,
) -> str:
    """
    Menggabungkan currency marker dan nilai terformat.
    """

    if currency not in CURRENCY_DISPLAY_CODES:
        raise ValueError(
            f"Currency tidak didukung: {currency}"
        )

    currency_marker = (
        CURRENCY_DISPLAY_CODES[currency]
    )

    formatted_number = format_number_for_display(
        value=value,
        currency=currency,
        language=language,
    )

    return f"{currency_marker} {formatted_number}"


# ---------------------------------------------------------
# Locale validation
# ---------------------------------------------------------

required_label_keys = set(
    LOCALE_LABELS["id"].keys()
)

for language, labels in LOCALE_LABELS.items():
    if set(labels.keys()) != required_label_keys:
        raise RuntimeError(
            f"Label locale {language} tidak lengkap."
        )

    if len(ITEM_CATALOG[language]) < 10:
        raise RuntimeError(
            f"Katalog item {language} terlalu sedikit."
        )

formatting_preview = pd.DataFrame(
    [
        {
            "language": "id",
            "date": format_date_for_display(
                date(2026, 9, 4),
                "id",
            ),
            "IDR": format_money_for_display(
                Decimal("1250000"),
                "IDR",
                "id",
            ),
            "USD": format_money_for_display(
                Decimal("1250.75"),
                "USD",
                "id",
            ),
        },
        {
            "language": "en",
            "date": format_date_for_display(
                date(2026, 9, 4),
                "en",
            ),
            "IDR": format_money_for_display(
                Decimal("1250000"),
                "IDR",
                "en",
            ),
            "USD": format_money_for_display(
                Decimal("1250.75"),
                "USD",
                "en",
            ),
        },
    ]
)

display(formatting_preview)

print(
    "Supported locales : "
    f"{sorted(LOCALE_LABELS)}"
)
print(
    "Items per locale  : "
    f"{ {key: len(value) for key, value in ITEM_CATALOG.items()} }"
)
print(
    "\n✅ Locale dan formatting engine siap digunakan."
)

**Cell 29 — Code: Generator canonical invoice record**

In [ ]:
from datetime import timedelta


UNIT_PRICE_RANGES = {
    "IDR": {
        "minimum": Decimal("50000"),
        "maximum": Decimal("5000000"),
        "increment": Decimal("10000"),
    },
    "USD": {
        "minimum": Decimal("10"),
        "maximum": Decimal("5000"),
        "increment": Decimal("0.01"),
    },
    "EUR": {
        "minimum": Decimal("10"),
        "maximum": Decimal("4500"),
        "increment": Decimal("0.01"),
    },
    "GBP": {
        "minimum": Decimal("10"),
        "maximum": Decimal("4000"),
        "increment": Decimal("0.01"),
    },
}

PAYMENT_TERM_DAYS = [
    0,
    14,
    30,
    45,
]


def create_deterministic_rng(
    *seed_components: Any,
) -> random.Random:
    """
    Membuat RNG stabil berdasarkan SHA-256.

    Hasil tidak bergantung pada urutan eksekusi
    atau Python hash randomization.
    """

    seed_payload = "|".join(
        str(component)
        for component in (
            RANDOM_SEED,
            *seed_components,
        )
    ).encode("utf-8")

    seed_digest = hashlib.sha256(
        seed_payload
    ).digest()

    derived_seed = int.from_bytes(
        seed_digest[:8],
        byteorder="big",
        signed=False,
    )

    return random.Random(derived_seed)


def generate_unit_price(
    currency: str,
    rng: random.Random,
) -> Decimal:
    """
    Menghasilkan harga satuan sesuai skala currency.
    """

    price_configuration = UNIT_PRICE_RANGES[
        currency
    ]

    minimum = price_configuration["minimum"]
    maximum = price_configuration["maximum"]
    increment = price_configuration["increment"]

    step_count = int(
        (maximum - minimum) / increment
    )

    selected_step = rng.randint(
        0,
        step_count,
    )

    generated_price = (
        minimum
        + increment * selected_step
    )

    return quantize_money(
        generated_price,
        currency,
    )


def generate_invoice_record(
    document_number: int,
    local_template_index: int,
    template_id: str,
    split: str,
    language: str,
    currency: str,
) -> SyntheticInvoice:
    """
    Menghasilkan satu canonical invoice yang valid.
    """

    if document_number < 1:
        raise ValueError(
            "document_number harus dimulai dari 1."
        )

    if not 1 <= local_template_index <= (
        DOCUMENTS_PER_TEMPLATE
    ):
        raise ValueError(
            "local_template_index harus berada "
            "pada rentang 1–20."
        )

    document_rng = create_deterministic_rng(
        "invoice",
        document_number,
        template_id,
        split,
        language,
        currency,
    )

    template_number_match = re.fullmatch(
        r"TPL-(\d{2})",
        template_id,
    )

    if not template_number_match:
        raise ValueError(
            f"Template ID tidak valid: {template_id}"
        )

    template_number = int(
        template_number_match.group(1)
    )

    # Empat vendor dan empat buyer per template/language.
    vendor_index = (
        (template_number - 1) * 4
        + ((local_template_index - 1) % 4)
        + 1
    )

    buyer_index = (
        (template_number - 1) * 4
        + ((local_template_index + 1) % 4)
        + 1
    )

    vendor = generate_synthetic_party(
        role="vendor",
        language=language,
        index=vendor_index,
        rng=create_deterministic_rng(
            "party",
            "vendor",
            language,
            vendor_index,
        ),
    )

    buyer = generate_synthetic_party(
        role="buyer",
        language=language,
        index=buyer_index,
        rng=create_deterministic_rng(
            "party",
            "buyer",
            language,
            buyer_index,
        ),
    )

    base_date = date(2026, 1, 1)

    invoice_date = (
        base_date
        + timedelta(
            days=document_rng.randint(0, 364)
        )
    )

    payment_term_days = document_rng.choice(
        PAYMENT_TERM_DAYS
    )

    due_date = (
        invoice_date
        + timedelta(days=payment_term_days)
    )

    item_count = document_rng.randint(2, 8)

    selected_descriptions = document_rng.sample(
        ITEM_CATALOG[language],
        k=item_count,
    )

    items = tuple(
        SyntheticLineItem(
            description=description,
            quantity=Decimal(
                document_rng.randint(1, 10)
            ),
            unit_price=generate_unit_price(
                currency,
                document_rng,
            ),
        )
        for description in selected_descriptions
    )

    tax_rate_percent = document_rng.choice(
        TAX_RATE_OPTIONS
    )

    discount_rate_percent = document_rng.choice(
        DISCOUNT_RATE_OPTIONS
    )

    preliminary_subtotal = quantize_money(
        sum(
            (
                item.calculate_total(currency)
                for item in items
            ),
            Decimal("0"),
        ),
        currency,
    )

    discount_amount = quantize_money(
        (
            preliminary_subtotal
            * discount_rate_percent
            / Decimal("100")
        ),
        currency,
    )

    invoice_prefix = (
        "FTR"
        if language == "id"
        else "INV"
    )

    invoice_number = (
        f"{invoice_prefix}-"
        f"{invoice_date.year}-"
        f"{template_number:02d}-"
        f"{document_number:06d}"
    )

    invoice = SyntheticInvoice(
        document_id=(
            f"INV-SYN-{document_number:06d}"
        ),
        canonical_invoice_id=(
            f"CANON-{document_number:06d}"
        ),
        split=split,
        language=language,
        currency=currency,
        template_id=template_id,
        vendor=vendor,
        buyer=buyer,
        invoice_number=invoice_number,
        invoice_date=invoice_date,
        due_date=due_date,
        items=items,
        tax_rate_percent=tax_rate_percent,
        discount_amount=discount_amount,
    )

    return invoice


# ---------------------------------------------------------
# Determinism and financial validation test
# ---------------------------------------------------------

invoice_test_a = generate_invoice_record(
    document_number=1,
    local_template_index=1,
    template_id="TPL-01",
    split="development",
    language="id",
    currency="IDR",
)

invoice_test_b = generate_invoice_record(
    document_number=1,
    local_template_index=1,
    template_id="TPL-01",
    split="development",
    language="id",
    currency="IDR",
)

serialized_test_a = serialize_invoice(
    invoice_test_a
)

serialized_test_b = serialize_invoice(
    invoice_test_b
)

if serialized_test_a != serialized_test_b:
    raise RuntimeError(
        "Canonical invoice generator tidak deterministik."
    )

if invoice_test_a.total != (
    invoice_test_a.subtotal
    + invoice_test_a.tax_amount
    - invoice_test_a.discount_amount
):
    raise RuntimeError(
        "Validasi finansial invoice uji gagal."
    )

invoice_test_preview = pd.DataFrame(
    [
        {
            "document_id": invoice_test_a.document_id,
            "split": invoice_test_a.split,
            "language": invoice_test_a.language,
            "currency": invoice_test_a.currency,
            "template_id": invoice_test_a.template_id,
            "vendor_id": invoice_test_a.vendor.party_id,
            "invoice_number": (
                invoice_test_a.invoice_number
            ),
            "invoice_date": (
                invoice_test_a.invoice_date.isoformat()
            ),
            "due_date": (
                invoice_test_a.due_date.isoformat()
            ),
            "items": len(invoice_test_a.items),
            "subtotal": decimal_to_string(
                invoice_test_a.subtotal,
                invoice_test_a.currency,
            ),
            "tax": decimal_to_string(
                invoice_test_a.tax_amount,
                invoice_test_a.currency,
            ),
            "discount": decimal_to_string(
                invoice_test_a.discount_amount,
                invoice_test_a.currency,
            ),
            "total": decimal_to_string(
                invoice_test_a.total,
                invoice_test_a.currency,
            ),
        }
    ]
)

display(invoice_test_preview)

print("Deterministic generation : PASSED")
print("Financial validation    : PASSED")
print(
    "\n✅ Canonical invoice record generator siap."
)

**Cell 30 — Code: Membuat generation plan**

In [ ]:
GENERATION_PLAN_PATH = (
    BUILD_CONFIGURATION_ROOT
    / "generation_plan.csv"
)


def shuffled_copy(
    values: list[str],
    *seed_components: Any,
) -> list[str]:
    """
    Mengacak salinan list secara deterministik.
    """

    shuffled_values = list(values)

    shuffle_rng = create_deterministic_rng(
        "generation-plan",
        *seed_components,
    )

    shuffle_rng.shuffle(shuffled_values)

    return shuffled_values


def find_split_for_template(
    template_id: str,
) -> str:
    """
    Menemukan split berdasarkan template ID.
    """

    matching_splits = [
        split_name
        for split_name, template_ids
        in SPLIT_TEMPLATE_MAPPING.items()
        if template_id in template_ids
    ]

    if len(matching_splits) != 1:
        raise RuntimeError(
            f"Template {template_id} harus berada "
            "pada tepat satu split."
        )

    return matching_splits[0]


LANGUAGE_LEVEL_DESIGN = {
    "id": {
        "formats": (
            ["pdf"] * 5
            + ["png"] * 3
            + ["jpg"] * 2
        ),
        "qualities": (
            ["digital_clean"] * 7
            + ["scan_simulation"] * 3
        ),
        "currencies": (
            ["IDR"] * 8
            + ["USD"] * 2
        ),
    },
    "en": {
        "formats": (
            ["pdf"] * 5
            + ["png"] * 2
            + ["jpg"] * 3
        ),
        "qualities": (
            ["digital_clean"] * 7
            + ["scan_simulation"] * 3
        ),
        "currencies": (
            ["USD"] * 5
            + ["EUR"] * 3
            + ["GBP"] * 2
        ),
    },
}

generation_plan_records = []
document_number = 0

for template_id in sorted(all_template_ids):
    split_name = find_split_for_template(
        template_id
    )

    local_template_index = 0

    for language in ["id", "en"]:
        language_design = (
            LANGUAGE_LEVEL_DESIGN[language]
        )

        format_assignments = shuffled_copy(
            language_design["formats"],
            template_id,
            language,
            "format",
        )

        quality_assignments = shuffled_copy(
            language_design["qualities"],
            template_id,
            language,
            "quality",
        )

        currency_assignments = shuffled_copy(
            language_design["currencies"],
            template_id,
            language,
            "currency",
        )

        language_document_count = (
            LANGUAGE_PER_TEMPLATE[language]
        )

        for language_index in range(
            language_document_count
        ):
            document_number += 1
            local_template_index += 1

            document_id = (
                f"INV-SYN-{document_number:06d}"
            )

            canonical_invoice_id = (
                f"CANON-{document_number:06d}"
            )

            file_format = format_assignments[
                language_index
            ]

            quality_profile = quality_assignments[
                language_index
            ]

            currency = currency_assignments[
                language_index
            ]

            seed_payload = "|".join(
                [
                    str(RANDOM_SEED),
                    document_id,
                    template_id,
                    split_name,
                    language,
                    currency,
                ]
            ).encode("utf-8")

            document_seed_token = hashlib.sha256(
                seed_payload
            ).hexdigest()[:16]

            generation_plan_records.append(
                {
                    "generation_sequence": (
                        document_number
                    ),
                    "document_id": document_id,
                    "canonical_invoice_id": (
                        canonical_invoice_id
                    ),
                    "split": split_name,
                    "template_id": template_id,
                    "local_template_index": (
                        local_template_index
                    ),
                    "language": language,
                    "currency": currency,
                    "file_format": file_format,
                    "quality_profile": (
                        quality_profile
                    ),
                    "document_seed_token": (
                        document_seed_token
                    ),
                    "document_relative_path": (
                        f"documents/{split_name}/"
                        f"{document_id}.{file_format}"
                    ),
                    "ground_truth_relative_path": (
                        f"ground_truth/{split_name}/"
                        f"{document_id}.json"
                    ),
                }
            )

generation_plan = pd.DataFrame(
    generation_plan_records
)

# ---------------------------------------------------------
# Generation plan validation
# ---------------------------------------------------------

if len(generation_plan) != (
    CANONICAL_DOCUMENT_COUNT
):
    raise RuntimeError(
        "Generation plan tidak berjumlah 200."
    )

unique_columns = [
    "generation_sequence",
    "document_id",
    "canonical_invoice_id",
    "document_relative_path",
    "ground_truth_relative_path",
    "document_seed_token",
]

for column_name in unique_columns:
    if not generation_plan[
        column_name
    ].is_unique:
        raise RuntimeError(
            f"Kolom {column_name} tidak unik."
        )

actual_split_distribution = (
    generation_plan["split"]
    .value_counts()
    .to_dict()
)

actual_language_distribution = (
    generation_plan["language"]
    .value_counts()
    .to_dict()
)

actual_format_distribution = (
    generation_plan["file_format"]
    .value_counts()
    .to_dict()
)

actual_quality_distribution = (
    generation_plan["quality_profile"]
    .value_counts()
    .to_dict()
)

actual_currency_distribution = (
    generation_plan["currency"]
    .value_counts()
    .to_dict()
)

distribution_validations = {
    "split": (
        actual_split_distribution
        == EXPECTED_SPLIT_DISTRIBUTION
    ),
    "language": (
        actual_language_distribution
        == EXPECTED_GLOBAL_DISTRIBUTIONS[
            "language"
        ]
    ),
    "format": (
        actual_format_distribution
        == EXPECTED_GLOBAL_DISTRIBUTIONS[
            "format"
        ]
    ),
    "quality": (
        actual_quality_distribution
        == EXPECTED_GLOBAL_DISTRIBUTIONS[
            "quality"
        ]
    ),
    "currency": (
        actual_currency_distribution
        == EXPECTED_GLOBAL_DISTRIBUTIONS[
            "currency"
        ]
    ),
}

template_document_counts = (
    generation_plan["template_id"]
    .value_counts()
)

if not template_document_counts.eq(
    DOCUMENTS_PER_TEMPLATE
).all():
    raise RuntimeError(
        "Tidak semua template mempunyai 20 dokumen."
    )

failed_distribution_checks = [
    dimension
    for dimension, is_valid
    in distribution_validations.items()
    if not is_valid
]

if failed_distribution_checks:
    raise RuntimeError(
        "Distribusi generation plan tidak valid: "
        f"{failed_distribution_checks}"
    )

# ---------------------------------------------------------
# Save generation plan atomically
# ---------------------------------------------------------

temporary_generation_plan_path = (
    GENERATION_PLAN_PATH.with_suffix(".tmp")
)

generation_plan.to_csv(
    temporary_generation_plan_path,
    index=False,
    encoding="utf-8",
)

temporary_generation_plan_path.replace(
    GENERATION_PLAN_PATH
)

saved_generation_plan = pd.read_csv(
    GENERATION_PLAN_PATH,
    dtype={
        "document_id": "string",
        "canonical_invoice_id": "string",
        "document_seed_token": "string",
    },
)

if len(saved_generation_plan) != len(
    generation_plan
):
    raise RuntimeError(
        "Generation plan berubah setelah disimpan."
    )

display(generation_plan.head(10))

validation_table = pd.DataFrame(
    [
        {
            "dimension": dimension,
            "status": (
                "VALID"
                if is_valid
                else "INVALID"
            ),
        }
        for dimension, is_valid
        in distribution_validations.items()
    ]
)

display(validation_table)

print(f"Generation plan : {GENERATION_PLAN_PATH}")
print(f"Total documents : {len(generation_plan):,}")
print(
    "Templates valid: "
    f"{template_document_counts.eq(DOCUMENTS_PER_TEMPLATE).all()}"
)
print(
    "\n✅ Generation plan berhasil dibuat "
    "dan divalidasi."
)

**Cell 31- Code: Membuat dan memvalidasi canonical invoice**

In [ ]:
# Membuat seluruh canonical invoice berdasarkan generation plan.
# Tahap ini belum merender dokumen dan belum memublikasikan dataset.

APPROVED_CANONICAL_DOCUMENT_COUNT = 200

expected_document_count = len(
    generation_plan
)

if (
    expected_document_count
    != APPROVED_CANONICAL_DOCUMENT_COUNT
):
    raise RuntimeError(
        "Generation plan tidak memiliki jumlah dokumen "
        "yang telah disetujui. "
        f"Expected: {APPROVED_CANONICAL_DOCUMENT_COUNT}, "
        f"actual: {expected_document_count}"
    )

canonical_invoices = []
canonical_invoice_payloads = {}
canonical_audit_rows = []

for plan_row in generation_plan.itertuples(index=False):
    invoice = generate_invoice_record(
        document_number=int(
            plan_row.generation_sequence
        ),
        local_template_index=int(
            plan_row.local_template_index
        ),
        template_id=str(plan_row.template_id),
        split=str(plan_row.split),
        language=str(plan_row.language),
        currency=str(plan_row.currency),
    )

    # --------------------------------------------------------------
    # Memastikan identitas invoice sesuai generation plan
    # --------------------------------------------------------------

    identity_checks = {
        "document_id": (
            invoice.document_id,
            str(plan_row.document_id),
        ),
        "split": (
            invoice.split,
            str(plan_row.split),
        ),
        "template_id": (
            invoice.template_id,
            str(plan_row.template_id),
        ),
        "language": (
            invoice.language,
            str(plan_row.language),
        ),
        "currency": (
            invoice.currency,
            str(plan_row.currency),
        ),
    }

    identity_errors = {
        field_name: {
            "actual": actual_value,
            "expected": expected_value,
        }
        for field_name, (
            actual_value,
            expected_value,
        ) in identity_checks.items()
        if actual_value != expected_value
    }

    if identity_errors:
        raise RuntimeError(
            f"Identitas invoice "
            f"{plan_row.document_id} tidak sesuai "
            f"generation plan: {identity_errors}"
        )

    # --------------------------------------------------------------
    # Memvalidasi persamaan finansial
    # --------------------------------------------------------------

    recalculated_total = (
        invoice.subtotal
        + invoice.tax_amount
        - invoice.discount_amount
    )

    financial_valid = (
        invoice.total == recalculated_total
    )

    if not financial_valid:
        raise RuntimeError(
            f"Persamaan finansial tidak valid pada "
            f"{invoice.document_id}: "
            f"{invoice.subtotal} "
            f"+ {invoice.tax_amount} "
            f"- {invoice.discount_amount} "
            f"!= {invoice.total}"
        )

    # --------------------------------------------------------------
    # Memvalidasi urutan tanggal
    # --------------------------------------------------------------

    if invoice.due_date < invoice.invoice_date:
        raise RuntimeError(
            f"Due date lebih awal dari invoice date "
            f"pada {invoice.document_id}."
        )

    # Memastikan record dapat diserialisasi.
    serialized_invoice = serialize_invoice(
        invoice
    )

    canonical_invoices.append(
        invoice
    )

    canonical_invoice_payloads[
        str(plan_row.canonical_invoice_id)
    ] = serialized_invoice

    canonical_audit_rows.append(
        {
            "generation_sequence": int(
                plan_row.generation_sequence
            ),
            "document_id": invoice.document_id,
            "canonical_invoice_id": str(
                plan_row.canonical_invoice_id
            ),
            "split": invoice.split,
            "template_id": invoice.template_id,
            "language": invoice.language,
            "currency": invoice.currency,
            "vendor_id": invoice.vendor.party_id,
            "buyer_id": invoice.buyer.party_id,
            "invoice_number": (
                invoice.invoice_number
            ),
            "invoice_date": (
                invoice.invoice_date.isoformat()
            ),
            "due_date": (
                invoice.due_date.isoformat()
            ),
            "line_item_count": len(
                invoice.items
            ),
            "subtotal": decimal_to_string(
                invoice.subtotal,
                invoice.currency,
            ),
            "tax_amount": decimal_to_string(
                invoice.tax_amount,
                invoice.currency,
            ),
            "discount_amount": decimal_to_string(
                invoice.discount_amount,
                invoice.currency,
            ),
            "total": decimal_to_string(
                invoice.total,
                invoice.currency,
            ),
            "financial_validation": "VALID",
        }
    )

canonical_audit = pd.DataFrame(
    canonical_audit_rows
)

# ------------------------------------------------------------------
# Dataset-level integrity controls
# ------------------------------------------------------------------

document_count_valid = (
    len(canonical_invoices)
    == expected_document_count
)

unique_document_ids_valid = (
    canonical_audit[
        "document_id"
    ].nunique()
    == len(canonical_audit)
)

unique_canonical_ids_valid = (
    canonical_audit[
        "canonical_invoice_id"
    ].nunique()
    == len(canonical_audit)
)

unique_invoice_numbers_valid = (
    canonical_audit[
        "invoice_number"
    ].nunique()
    == len(canonical_audit)
)

all_financial_records_valid = (
    canonical_audit[
        "financial_validation"
    ]
    .eq("VALID")
    .all()
)

# ------------------------------------------------------------------
# Anti-leakage control: template
# ------------------------------------------------------------------

template_split_counts = (
    canonical_audit
    .groupby("template_id")["split"]
    .nunique()
)

template_leakage_count = int(
    template_split_counts.gt(1).sum()
)

# ------------------------------------------------------------------
# Anti-leakage control: vendor
# ------------------------------------------------------------------

vendor_split_counts = (
    canonical_audit
    .groupby("vendor_id")["split"]
    .nunique()
)

vendor_leakage_count = int(
    vendor_split_counts.gt(1).sum()
)

# ------------------------------------------------------------------
# Membuat tabel hasil kontrol
# ------------------------------------------------------------------

audit_controls = pd.DataFrame(
    [
        {
            "control": "generation_plan_count",
            "expected": (
                APPROVED_CANONICAL_DOCUMENT_COUNT
            ),
            "actual": expected_document_count,
            "status": (
                "VALID"
                if expected_document_count
                == APPROVED_CANONICAL_DOCUMENT_COUNT
                else "INVALID"
            ),
        },
        {
            "control": "canonical_document_count",
            "expected": expected_document_count,
            "actual": len(canonical_invoices),
            "status": (
                "VALID"
                if document_count_valid
                else "INVALID"
            ),
        },
        {
            "control": "unique_document_id",
            "expected": len(canonical_audit),
            "actual": canonical_audit[
                "document_id"
            ].nunique(),
            "status": (
                "VALID"
                if unique_document_ids_valid
                else "INVALID"
            ),
        },
        {
            "control": "unique_canonical_invoice_id",
            "expected": len(canonical_audit),
            "actual": canonical_audit[
                "canonical_invoice_id"
            ].nunique(),
            "status": (
                "VALID"
                if unique_canonical_ids_valid
                else "INVALID"
            ),
        },
        {
            "control": "unique_invoice_number",
            "expected": len(canonical_audit),
            "actual": canonical_audit[
                "invoice_number"
            ].nunique(),
            "status": (
                "VALID"
                if unique_invoice_numbers_valid
                else "INVALID"
            ),
        },
        {
            "control": "financial_validation",
            "expected": len(canonical_audit),
            "actual": int(
                canonical_audit[
                    "financial_validation"
                ]
                .eq("VALID")
                .sum()
            ),
            "status": (
                "VALID"
                if all_financial_records_valid
                else "INVALID"
            ),
        },
        {
            "control": "template_split_leakage",
            "expected": 0,
            "actual": template_leakage_count,
            "status": (
                "VALID"
                if template_leakage_count == 0
                else "INVALID"
            ),
        },
        {
            "control": "vendor_split_leakage",
            "expected": 0,
            "actual": vendor_leakage_count,
            "status": (
                "VALID"
                if vendor_leakage_count == 0
                else "INVALID"
            ),
        },
    ]
)

invalid_controls = audit_controls.loc[
    audit_controls["status"] != "VALID",
    "control",
].tolist()

if invalid_controls:
    display(audit_controls)

    raise RuntimeError(
        "Canonical invoice audit gagal pada "
        f"kontrol: {invalid_controls}"
    )

# ------------------------------------------------------------------
# Menampilkan hasil
# ------------------------------------------------------------------

display(
    canonical_audit.head(10)
)

display(
    audit_controls
)

print(
    f"Generation plan  : "
    f"{expected_document_count}"
)
print(
    f"Canonical records: "
    f"{len(canonical_invoices)}"
)
print(
    f"Unique vendors   : "
    f"{canonical_audit['vendor_id'].nunique()}"
)
print(
    f"Unique invoices  : "
    f"{canonical_audit['invoice_number'].nunique()}"
)
print(
    f"Template leakage : "
    f"{template_leakage_count}"
)
print(
    f"Vendor leakage   : "
    f"{vendor_leakage_count}"
)
print()
print(
    "✅ 200 canonical invoice records berhasil "
    "dibuat dan divalidasi."
)

**Cell 32 — Code: Mendefinisikan registry desain 10 template invoice**

In [ ]:
# Registry desain menentukan variasi visual dan struktur layout.
# Pada tahap ini dokumen belum dirender.

@dataclass(frozen=True)
class InvoiceTemplateSpec:
    template_id: str
    split: str
    layout_family: str
    page_size: str
    header_layout: str
    party_layout: str
    table_style: str
    totals_position: str
    accent_position: str
    density: str
    primary_color: str
    accent_color: str


TEMPLATE_REGISTRY = {
    "TPL-01": InvoiceTemplateSpec(
        template_id="TPL-01",
        split="development",
        layout_family="classic_corporate",
        page_size="A4",
        header_layout="full_width_band",
        party_layout="two_columns",
        table_style="full_grid",
        totals_position="bottom_right",
        accent_position="top",
        density="regular",
        primary_color="#16324F",
        accent_color="#2F80ED",
    ),
    "TPL-02": InvoiceTemplateSpec(
        template_id="TPL-02",
        split="development",
        layout_family="modern_minimal",
        page_size="A4",
        header_layout="open_left",
        party_layout="stacked",
        table_style="horizontal_rules",
        totals_position="bottom_right",
        accent_position="left",
        density="spacious",
        primary_color="#243B53",
        accent_color="#00A896",
    ),
    "TPL-03": InvoiceTemplateSpec(
        template_id="TPL-03",
        split="development",
        layout_family="split_header",
        page_size="LETTER",
        header_layout="split_title_metadata",
        party_layout="two_columns",
        table_style="striped",
        totals_position="bottom_card",
        accent_position="header",
        density="regular",
        primary_color="#3D405B",
        accent_color="#E07A5F",
    ),
    "TPL-04": InvoiceTemplateSpec(
        template_id="TPL-04",
        split="development",
        layout_family="left_sidebar",
        page_size="A4",
        header_layout="sidebar_identity",
        party_layout="vendor_sidebar",
        table_style="minimal",
        totals_position="bottom_right",
        accent_position="left",
        density="compact",
        primary_color="#264653",
        accent_color="#2A9D8F",
    ),
    "TPL-05": InvoiceTemplateSpec(
        template_id="TPL-05",
        split="development",
        layout_family="centered_editorial",
        page_size="A4",
        header_layout="centered_title",
        party_layout="horizontal_cards",
        table_style="open_rows",
        totals_position="full_width",
        accent_position="bottom",
        density="spacious",
        primary_color="#4A4E69",
        accent_color="#9A8C98",
    ),
    "TPL-06": InvoiceTemplateSpec(
        template_id="TPL-06",
        split="development",
        layout_family="boxed_enterprise",
        page_size="LETTER",
        header_layout="boxed_right",
        party_layout="two_columns_boxed",
        table_style="full_grid",
        totals_position="side_panel",
        accent_position="right",
        density="compact",
        primary_color="#1D3557",
        accent_color="#457B9D",
    ),
    "TPL-07": InvoiceTemplateSpec(
        template_id="TPL-07",
        split="validation",
        layout_family="top_stripe",
        page_size="A4",
        header_layout="title_with_top_stripe",
        party_layout="stacked_split",
        table_style="striped",
        totals_position="bottom_card",
        accent_position="top",
        density="regular",
        primary_color="#283618",
        accent_color="#DDA15E",
    ),
    "TPL-08": InvoiceTemplateSpec(
        template_id="TPL-08",
        split="validation",
        layout_family="modular_cards",
        page_size="LETTER",
        header_layout="metadata_cards",
        party_layout="horizontal_cards",
        table_style="boxed_rows",
        totals_position="side_panel",
        accent_position="header",
        density="spacious",
        primary_color="#003049",
        accent_color="#F77F00",
    ),
    "TPL-09": InvoiceTemplateSpec(
        template_id="TPL-09",
        split="test",
        layout_family="compact_ledger",
        page_size="A4",
        header_layout="compact_metadata",
        party_layout="two_columns",
        table_style="ledger",
        totals_position="inline",
        accent_position="bottom",
        density="compact",
        primary_color="#2B2D42",
        accent_color="#8D99AE",
    ),
    "TPL-10": InvoiceTemplateSpec(
        template_id="TPL-10",
        split="test",
        layout_family="international_clean",
        page_size="LETTER",
        header_layout="open_right",
        party_layout="stacked",
        table_style="horizontal_rules",
        totals_position="full_width",
        accent_position="right",
        density="regular",
        primary_color="#14213D",
        accent_color="#FCA311",
    ),
}

# ------------------------------------------------------------------
# Validasi registry
# ------------------------------------------------------------------

expected_template_ids = {
    f"TPL-{template_number:02d}"
    for template_number in range(1, 11)
}

actual_template_ids = set(
    TEMPLATE_REGISTRY
)

if actual_template_ids != expected_template_ids:
    missing_templates = sorted(
        expected_template_ids - actual_template_ids
    )
    unexpected_templates = sorted(
        actual_template_ids - expected_template_ids
    )

    raise RuntimeError(
        "Template registry tidak lengkap. "
        f"Missing: {missing_templates}; "
        f"unexpected: {unexpected_templates}"
    )

allowed_page_sizes = {
    "A4",
    "LETTER",
}

allowed_densities = {
    "compact",
    "regular",
    "spacious",
}

template_audit_rows = []

for template_id, template_spec in sorted(
    TEMPLATE_REGISTRY.items()
):
    if template_spec.template_id != template_id:
        raise RuntimeError(
            f"Template key tidak konsisten: "
            f"{template_id}"
        )

    if template_spec.page_size not in allowed_page_sizes:
        raise RuntimeError(
            f"Page size tidak didukung pada "
            f"{template_id}: "
            f"{template_spec.page_size}"
        )

    if template_spec.density not in allowed_densities:
        raise RuntimeError(
            f"Density tidak didukung pada "
            f"{template_id}: "
            f"{template_spec.density}"
        )

    for color_name, color_value in {
        "primary_color": template_spec.primary_color,
        "accent_color": template_spec.accent_color,
    }.items():
        if not re.fullmatch(
            r"#[0-9A-F]{6}",
            color_value,
        ):
            raise RuntimeError(
                f"{color_name} tidak valid pada "
                f"{template_id}: {color_value}"
            )

    plan_rows = generation_plan.loc[
        generation_plan["template_id"]
        == template_id
    ]

    plan_splits = sorted(
        plan_rows["split"].unique().tolist()
    )

    if plan_splits != [template_spec.split]:
        raise RuntimeError(
            f"Split template {template_id} tidak "
            "sesuai generation plan. "
            f"Registry: {template_spec.split}; "
            f"plan: {plan_splits}"
        )

    template_audit_rows.append(
        {
            "template_id": template_id,
            "split": template_spec.split,
            "layout_family": (
                template_spec.layout_family
            ),
            "page_size": template_spec.page_size,
            "header_layout": (
                template_spec.header_layout
            ),
            "party_layout": (
                template_spec.party_layout
            ),
            "table_style": (
                template_spec.table_style
            ),
            "totals_position": (
                template_spec.totals_position
            ),
            "density": template_spec.density,
            "documents": len(plan_rows),
            "status": "VALID",
        }
    )

template_registry_audit = pd.DataFrame(
    template_audit_rows
)

# Setiap template harus memiliki kombinasi layout berbeda.
design_fingerprints = {
    (
        template_spec.page_size,
        template_spec.header_layout,
        template_spec.party_layout,
        template_spec.table_style,
        template_spec.totals_position,
        template_spec.accent_position,
        template_spec.density,
    )
    for template_spec in TEMPLATE_REGISTRY.values()
}

if len(design_fingerprints) != len(TEMPLATE_REGISTRY):
    raise RuntimeError(
        "Ditemukan template dengan fingerprint "
        "desain yang sama."
    )

if not template_registry_audit[
    "documents"
].eq(20).all():
    raise RuntimeError(
        "Setiap template harus digunakan oleh "
        "tepat 20 dokumen."
    )

display(template_registry_audit)

print(
    f"Registered templates : "
    f"{len(TEMPLATE_REGISTRY)}"
)
print(
    f"Unique designs       : "
    f"{len(design_fingerprints)}"
)
print(
    f"Covered documents    : "
    f"{template_registry_audit['documents'].sum()}"
)
print()
print(
    "✅ Registry 10 desain template invoice "
    "berhasil dibuat dan divalidasi."
)

**Cell 33 — Code: Fondasi rendering dan anotasi bidang**

In [ ]:
# Fondasi rendering:
# - ukuran halaman;
# - validasi warna;
# - penyisipan teks;
# - pencatatan bounding box;
# - normalisasi koordinat ground truth.

PAGE_SIZE_POINTS = {
    "A4": (595.276, 841.890),
    "LETTER": (612.000, 792.000),
}

TEXT_ALIGNMENTS = {
    "left": pymupdf.TEXT_ALIGN_LEFT,
    "center": pymupdf.TEXT_ALIGN_CENTER,
    "right": pymupdf.TEXT_ALIGN_RIGHT,
    "justify": pymupdf.TEXT_ALIGN_JUSTIFY,
}


@dataclass(frozen=True)
class FieldAnnotation:
    field_name: str
    text: str
    page_number: int
    bbox_points: tuple[
        float,
        float,
        float,
        float,
    ]
    bbox_normalized: tuple[
        float,
        float,
        float,
        float,
    ]
    annotation_type: str = "field_region"


def hex_to_rgb(
    hex_color: str,
) -> tuple[float, float, float]:
    """Mengubah warna hexadecimal menjadi RGB PyMuPDF."""

    if not re.fullmatch(
        r"#[0-9A-Fa-f]{6}",
        hex_color,
    ):
        raise ValueError(
            f"Format warna tidak valid: {hex_color}"
        )

    return tuple(
        int(
            hex_color[index:index + 2],
            16,
        ) / 255.0
        for index in (1, 3, 5)
    )


def get_page_dimensions(
    page_size: str,
) -> tuple[float, float]:
    """Mengambil ukuran halaman dalam satuan point."""

    if page_size not in PAGE_SIZE_POINTS:
        raise ValueError(
            f"Page size tidak didukung: {page_size}"
        )

    return PAGE_SIZE_POINTS[page_size]


def validate_rectangle(
    page: pymupdf.Page,
    rectangle: pymupdf.Rect,
) -> None:
    """Memastikan rectangle valid dan berada di dalam halaman."""

    if rectangle.is_empty or rectangle.is_infinite:
        raise ValueError(
            f"Rectangle tidak valid: {rectangle}"
        )

    page_rectangle = page.rect
    tolerance = 0.01

    outside_page = (
        rectangle.x0 < page_rectangle.x0 - tolerance
        or rectangle.y0 < page_rectangle.y0 - tolerance
        or rectangle.x1 > page_rectangle.x1 + tolerance
        or rectangle.y1 > page_rectangle.y1 + tolerance
    )

    if outside_page:
        raise ValueError(
            "Rectangle berada di luar halaman. "
            f"Rectangle: {rectangle}; "
            f"page: {page_rectangle}"
        )


def normalize_rectangle(
    rectangle: pymupdf.Rect,
    page_rectangle: pymupdf.Rect,
) -> tuple[float, float, float, float]:
    """Mengubah bbox point menjadi koordinat relatif 0–1."""

    normalized = (
        rectangle.x0 / page_rectangle.width,
        rectangle.y0 / page_rectangle.height,
        rectangle.x1 / page_rectangle.width,
        rectangle.y1 / page_rectangle.height,
    )

    if not all(
        0.0 <= coordinate <= 1.0
        for coordinate in normalized
    ):
        raise ValueError(
            "Bounding box hasil normalisasi "
            f"berada di luar rentang 0–1: {normalized}"
        )

    return tuple(
        round(coordinate, 6)
        for coordinate in normalized
    )


def wrap_text_to_width(
    text: str,
    max_width: float,
    font_name: str = "helv",
    font_size: float = 9.0,
) -> str:
    """Membungkus teks berdasarkan lebar aktual font."""

    clean_text = " ".join(
        str(text).split()
    )

    if not clean_text:
        return ""

    words = clean_text.split(" ")
    lines = []
    current_line = ""

    for word in words:
        candidate_line = (
            word
            if not current_line
            else f"{current_line} {word}"
        )

        candidate_width = pymupdf.get_text_length(
            candidate_line,
            fontname=font_name,
            fontsize=font_size,
        )

        if candidate_width <= max_width:
            current_line = candidate_line
            continue

        if current_line:
            lines.append(current_line)
            current_line = word
        else:
            # Menangani token tunggal yang sangat panjang.
            partial_word = ""

            for character in word:
                candidate_word = (
                    partial_word + character
                )

                candidate_word_width = (
                    pymupdf.get_text_length(
                        candidate_word,
                        fontname=font_name,
                        fontsize=font_size,
                    )
                )

                if (
                    candidate_word_width
                    <= max_width
                ):
                    partial_word = candidate_word
                else:
                    if partial_word:
                        lines.append(partial_word)

                    partial_word = character

            current_line = partial_word

    if current_line:
        lines.append(current_line)

    return "\n".join(lines)


def draw_panel(
    page: pymupdf.Page,
    rectangle: pymupdf.Rect,
    border_color: tuple[
        float,
        float,
        float,
    ] | None = None,
    fill_color: tuple[
        float,
        float,
        float,
    ] | None = None,
    border_width: float = 0.8,
    corner_radius: float = 0.0,
) -> None:
    """Menggambar panel visual tanpa membuat anotasi data."""

    validate_rectangle(
        page,
        rectangle,
    )

    if corner_radius > 0:
        page.draw_rect(
            rectangle,
            color=border_color,
            fill=fill_color,
            width=border_width,
            radius=corner_radius,
            overlay=True,
        )
    else:
        page.draw_rect(
            rectangle,
            color=border_color,
            fill=fill_color,
            width=border_width,
            overlay=True,
        )


def insert_field_text(
    page: pymupdf.Page,
    rectangle: pymupdf.Rect,
    text: str,
    field_name: str,
    font_name: str = "helv",
    font_size: float = 9.0,
    font_color: tuple[
        float,
        float,
        float,
    ] = (0.0, 0.0, 0.0),
    alignment: str = "left",
) -> FieldAnnotation:
    """
    Menyisipkan teks dan menghasilkan anotasi field-region.

    Bounding box menunjukkan area bidang, bukan batas
    setiap karakter atau kata.
    """

    if not re.fullmatch(
        r"[a-z][a-z0-9_.\[\]-]*",
        field_name,
    ):
        raise ValueError(
            f"Nama field tidak valid: {field_name}"
        )

    if alignment not in TEXT_ALIGNMENTS:
        raise ValueError(
            f"Alignment tidak didukung: {alignment}"
        )

    validate_rectangle(
        page,
        rectangle,
    )

    clean_text = str(text).strip()

    if not clean_text:
        raise ValueError(
            f"Nilai kosong untuk field: {field_name}"
        )

    remaining_height = page.insert_textbox(
        rectangle,
        clean_text,
        fontname=font_name,
        fontsize=font_size,
        color=font_color,
        align=TEXT_ALIGNMENTS[alignment],
        overlay=True,
    )

    if remaining_height < 0:
        raise RuntimeError(
            f"Teks field '{field_name}' tidak muat "
            f"pada rectangle {rectangle}. "
            f"Overflow: {abs(remaining_height):.2f} point."
        )

    bbox_points = tuple(
        round(coordinate, 3)
        for coordinate in (
            rectangle.x0,
            rectangle.y0,
            rectangle.x1,
            rectangle.y1,
        )
    )

    bbox_normalized = normalize_rectangle(
        rectangle,
        page.rect,
    )

    return FieldAnnotation(
        field_name=field_name,
        text=clean_text,
        page_number=page.number + 1,
        bbox_points=bbox_points,
        bbox_normalized=bbox_normalized,
    )


# ------------------------------------------------------------------
# Preflight test
# ------------------------------------------------------------------

preflight_document = pymupdf.open()

try:
    page_width, page_height = (
        get_page_dimensions("A4")
    )

    preflight_page = preflight_document.new_page(
        width=page_width,
        height=page_height,
    )

    primary_rgb = hex_to_rgb("#16324F")
    accent_rgb = hex_to_rgb("#2F80ED")

    draw_panel(
        page=preflight_page,
        rectangle=pymupdf.Rect(
            36,
            36,
            page_width - 36,
            110,
        ),
        border_color=primary_rgb,
        fill_color=(0.96, 0.97, 0.99),
        border_width=1.0,
    )

    title_annotation = insert_field_text(
        page=preflight_page,
        rectangle=pymupdf.Rect(
            48,
            52,
            330,
            82,
        ),
        text="INVOICE RENDERING PREFLIGHT",
        field_name="preflight.title",
        font_name="hebo",
        font_size=14.0,
        font_color=primary_rgb,
        alignment="left",
    )

    value_annotation = insert_field_text(
        page=preflight_page,
        rectangle=pymupdf.Rect(
            360,
            52,
            page_width - 48,
            82,
        ),
        text="READY",
        field_name="preflight.status",
        font_name="hebo",
        font_size=12.0,
        font_color=accent_rgb,
        alignment="right",
    )

    preflight_annotations = [
        title_annotation,
        value_annotation,
    ]

    if len(preflight_annotations) != 2:
        raise RuntimeError(
            "Jumlah anotasi preflight tidak sesuai."
        )

    if not all(
        annotation.annotation_type
        == "field_region"
        for annotation in preflight_annotations
    ):
        raise RuntimeError(
            "Annotation type preflight tidak valid."
        )

    annotation_preview = pd.DataFrame(
        [
            {
                "field_name": annotation.field_name,
                "text": annotation.text,
                "page_number": annotation.page_number,
                "bbox_points": (
                    annotation.bbox_points
                ),
                "bbox_normalized": (
                    annotation.bbox_normalized
                ),
                "annotation_type": (
                    annotation.annotation_type
                ),
                "status": "VALID",
            }
            for annotation in preflight_annotations
        ]
    )

finally:
    preflight_document.close()

display(annotation_preview)

print(
    f"Supported page sizes : "
    f"{sorted(PAGE_SIZE_POINTS)}"
)
print(
    f"Annotation type      : field_region"
)
print(
    "Coordinate systems  : points + normalized"
)
print()
print(
    "✅ Fondasi rendering dan anotasi "
    "berhasil divalidasi."
)

**Cell 34 — Code: Audit schema payload canonical invoice**

In [ ]:
# Memeriksa konsistensi struktur serialized payload
# dari seluruh canonical invoice.

import json


def collect_schema_signature(
    value,
    path: str = "$",
) -> set[tuple[str, str]]:
    """Menghasilkan pasangan path dan tipe data."""

    schema_entries = set()

    if isinstance(value, dict):
        schema_entries.add(
            (path, "object")
        )

        for key in sorted(value):
            child_path = (
                f"{path}.{key}"
            )

            schema_entries.update(
                collect_schema_signature(
                    value[key],
                    child_path,
                )
            )

    elif isinstance(value, list):
        schema_entries.add(
            (path, "array")
        )

        if not value:
            schema_entries.add(
                (f"{path}[]", "empty")
            )
        else:
            for item in value:
                schema_entries.update(
                    collect_schema_signature(
                        item,
                        f"{path}[]",
                    )
                )

    elif value is None:
        schema_entries.add(
            (path, "null")
        )

    elif isinstance(value, bool):
        schema_entries.add(
            (path, "boolean")
        )

    elif isinstance(value, int):
        schema_entries.add(
            (path, "integer")
        )

    elif isinstance(value, float):
        schema_entries.add(
            (path, "number")
        )

    elif isinstance(value, str):
        schema_entries.add(
            (path, "string")
        )

    else:
        schema_entries.add(
            (
                path,
                type(value).__name__,
            )
        )

    return schema_entries


expected_payload_ids = set(
    canonical_audit[
        "canonical_invoice_id"
    ].astype(str)
)

actual_payload_ids = set(
    canonical_invoice_payloads
)

missing_payload_ids = sorted(
    expected_payload_ids
    - actual_payload_ids
)

unexpected_payload_ids = sorted(
    actual_payload_ids
    - expected_payload_ids
)

payload_key_alignment_valid = (
    not missing_payload_ids
    and not unexpected_payload_ids
)

if not payload_key_alignment_valid:
    raise RuntimeError(
        "Canonical payload IDs tidak konsisten. "
        f"Missing: {missing_payload_ids[:10]}; "
        f"unexpected: {unexpected_payload_ids[:10]}"
    )

if len(canonical_invoice_payloads) != 200:
    raise RuntimeError(
        "Jumlah canonical payload tidak sesuai. "
        f"Expected: 200; "
        f"actual: {len(canonical_invoice_payloads)}"
    )

reference_payload_id = sorted(
    canonical_invoice_payloads
)[0]

reference_payload = (
    canonical_invoice_payloads[
        reference_payload_id
    ]
)

reference_schema = (
    collect_schema_signature(
        reference_payload
    )
)

schema_mismatch_ids = []
serialization_errors = []

for canonical_id, payload in sorted(
    canonical_invoice_payloads.items()
):
    payload_schema = (
        collect_schema_signature(payload)
    )

    if payload_schema != reference_schema:
        schema_mismatch_ids.append(
            canonical_id
        )

    try:
        json.dumps(
            payload,
            ensure_ascii=False,
            allow_nan=False,
        )
    except (
        TypeError,
        ValueError,
    ) as error:
        serialization_errors.append(
            {
                "canonical_invoice_id": (
                    canonical_id
                ),
                "error": str(error),
            }
        )

schema_consistency_valid = (
    len(schema_mismatch_ids) == 0
)

json_serialization_valid = (
    len(serialization_errors) == 0
)

schema_audit = pd.DataFrame(
    [
        {
            "control": "payload_count",
            "expected": 200,
            "actual": len(
                canonical_invoice_payloads
            ),
            "status": "VALID",
        },
        {
            "control": "payload_key_alignment",
            "expected": 200,
            "actual": len(
                actual_payload_ids
                & expected_payload_ids
            ),
            "status": (
                "VALID"
                if payload_key_alignment_valid
                else "INVALID"
            ),
        },
        {
            "control": "schema_consistency",
            "expected": 0,
            "actual": len(
                schema_mismatch_ids
            ),
            "status": (
                "VALID"
                if schema_consistency_valid
                else "INVALID"
            ),
        },
        {
            "control": "json_serialization_errors",
            "expected": 0,
            "actual": len(
                serialization_errors
            ),
            "status": (
                "VALID"
                if json_serialization_valid
                else "INVALID"
            ),
        },
    ]
)

schema_preview = pd.DataFrame(
    [
        {
            "field_path": field_path,
            "data_type": data_type,
        }
        for field_path, data_type in sorted(
            reference_schema
        )
    ]
)

invalid_controls = schema_audit.loc[
    schema_audit["status"] != "VALID",
    "control",
].tolist()

display(schema_preview)
display(schema_audit)

print(
    f"Reference payload : {reference_payload_id}"
)
print(
    f"Schema paths      : {len(reference_schema)}"
)
print(
    f"Payloads audited  : "
    f"{len(canonical_invoice_payloads)}"
)
print()
print("Reference payload preview:")
print(
    json.dumps(
        reference_payload,
        ensure_ascii=False,
        indent=2,
    )
)

if invalid_controls:
    raise RuntimeError(
        "Canonical payload schema audit gagal: "
        f"{invalid_controls}"
    )

print()
print(
    "✅ Schema seluruh canonical invoice payload "
    "konsisten dan JSON-serializable."
)

**Cell 35 — Code: Membuat presentation payload bilingu**

In [ ]:
from datetime import date
from decimal import Decimal, ROUND_HALF_UP


RENDER_LABELS = {
    "id": {
        "document_title": "INVOICE",
        "vendor": "Dari",
        "buyer": "Ditagihkan Kepada",
        "invoice_number": "Nomor Invoice",
        "invoice_date": "Tanggal Invoice",
        "due_date": "Jatuh Tempo",
        "description": "Deskripsi",
        "quantity": "Kuantitas",
        "unit_price": "Harga Satuan",
        "line_total": "Jumlah",
        "subtotal": "Subtotal",
        "tax": "Pajak",
        "discount": "Diskon",
        "total": "Total",
        "tax_identifier": "ID Pajak",
        "page": "Halaman",
        "synthetic_notice": (
            "DATA SINTETIS — BUKAN DOKUMEN "
            "TRANSAKSI NYATA"
        ),
    },
    "en": {
        "document_title": "INVOICE",
        "vendor": "From",
        "buyer": "Bill To",
        "invoice_number": "Invoice Number",
        "invoice_date": "Invoice Date",
        "due_date": "Due Date",
        "description": "Description",
        "quantity": "Quantity",
        "unit_price": "Unit Price",
        "line_total": "Amount",
        "subtotal": "Subtotal",
        "tax": "Tax",
        "discount": "Discount",
        "total": "Total",
        "tax_identifier": "Tax ID",
        "page": "Page",
        "synthetic_notice": (
            "SYNTHETIC DATA — NOT A REAL "
            "TRANSACTION DOCUMENT"
        ),
    },
}

MONTH_NAMES = {
    "id": (
        "",
        "Januari",
        "Februari",
        "Maret",
        "April",
        "Mei",
        "Juni",
        "Juli",
        "Agustus",
        "September",
        "Oktober",
        "November",
        "Desember",
    ),
    "en": (
        "",
        "January",
        "February",
        "March",
        "April",
        "May",
        "June",
        "July",
        "August",
        "September",
        "October",
        "November",
        "December",
    ),
}


def require_mapping_keys(
    mapping: dict,
    required_keys: set[str],
    mapping_name: str,
) -> None:
    """Memastikan seluruh key wajib tersedia."""

    missing_keys = sorted(
        required_keys - set(mapping)
    )

    if missing_keys:
        raise KeyError(
            f"Key wajib tidak tersedia pada "
            f"{mapping_name}: {missing_keys}"
        )


def format_render_date(
    iso_date: str,
    language: str,
) -> str:
    """Menghasilkan tanggal yang mudah dibaca."""

    if language not in MONTH_NAMES:
        raise ValueError(
            f"Bahasa tidak didukung: {language}"
        )

    parsed_date = date.fromisoformat(
        iso_date
    )

    month_name = MONTH_NAMES[
        language
    ][parsed_date.month]

    if language == "id":
        return (
            f"{parsed_date.day} "
            f"{month_name} "
            f"{parsed_date.year}"
        )

    return (
        f"{month_name} "
        f"{parsed_date.day}, "
        f"{parsed_date.year}"
    )


def format_render_money(
    raw_amount: str,
    currency: str,
    language: str,
) -> str:
    """Memformat nilai uang tanpa mengubah nilai canonical."""

    decimal_places = (
        0 if currency == "IDR" else 2
    )

    quantum = (
        Decimal("1")
        if decimal_places == 0
        else Decimal("0.01")
    )

    amount = Decimal(
        str(raw_amount)
    ).quantize(
        quantum,
        rounding=ROUND_HALF_UP,
    )

    formatted_amount = (
        f"{amount:,.{decimal_places}f}"
    )

    if language == "id":
        formatted_amount = (
            formatted_amount
            .replace(",", "__THOUSAND__")
            .replace(".", ",")
            .replace("__THOUSAND__", ".")
        )
    elif language != "en":
        raise ValueError(
            f"Bahasa tidak didukung: {language}"
        )

    return (
        f"{currency} {formatted_amount}"
    )


def build_party_render_payload(
    party_payload: dict,
    language: str,
) -> dict:
    """Membentuk informasi pihak untuk ditampilkan."""

    require_mapping_keys(
        party_payload,
        {
            "party_id",
            "name",
            "address_lines",
            "email",
            "phone",
            "tax_identifier",
        },
        "party",
    )

    if not isinstance(
        party_payload["address_lines"],
        list,
    ):
        raise TypeError(
            "address_lines harus berupa list."
        )

    if not party_payload["address_lines"]:
        raise ValueError(
            "address_lines tidak boleh kosong."
        )

    labels = RENDER_LABELS[language]

    return {
        "party_id": str(
            party_payload["party_id"]
        ),
        "name": str(
            party_payload["name"]
        ),
        "address_lines": [
            str(address_line)
            for address_line
            in party_payload["address_lines"]
        ],
        "email": str(
            party_payload["email"]
        ),
        "phone": str(
            party_payload["phone"]
        ),
        "tax_identifier": str(
            party_payload["tax_identifier"]
        ),
        "tax_identifier_display": (
            f"{labels['tax_identifier']}: "
            f"{party_payload['tax_identifier']}"
        ),
    }


def build_invoice_render_payload(
    canonical_payload: dict,
) -> dict:
    """Mengubah canonical payload menjadi presentation payload."""

    require_mapping_keys(
        canonical_payload,
        {
            "document_id",
            "canonical_invoice_id",
            "split",
            "language",
            "currency",
            "template_id",
            "vendor",
            "buyer",
            "invoice_number",
            "invoice_date",
            "due_date",
            "items",
            "financials",
        },
        "canonical invoice",
    )

    language = str(
        canonical_payload["language"]
    )

    currency = str(
        canonical_payload["currency"]
    )

    if language not in RENDER_LABELS:
        raise ValueError(
            f"Bahasa tidak didukung: {language}"
        )

    labels = RENDER_LABELS[language]

    financials = canonical_payload[
        "financials"
    ]

    require_mapping_keys(
        financials,
        {
            "subtotal",
            "tax_rate_percent",
            "tax",
            "discount",
            "total",
            "rounding_mode",
            "decimal_places",
            "equation_valid",
        },
        "financials",
    )

    render_items = []

    for item_index, item in enumerate(
        canonical_payload["items"],
        start=1,
    ):
        require_mapping_keys(
            item,
            {
                "description",
                "quantity",
                "unit_price",
                "line_total",
            },
            f"items[{item_index - 1}]",
        )

        render_items.append(
            {
                "item_number": item_index,
                "description": str(
                    item["description"]
                ),
                "quantity": str(
                    item["quantity"]
                ),
                "unit_price": (
                    format_render_money(
                        item["unit_price"],
                        currency,
                        language,
                    )
                ),
                "line_total": (
                    format_render_money(
                        item["line_total"],
                        currency,
                        language,
                    )
                ),
                "raw": {
                    "quantity": str(
                        item["quantity"]
                    ),
                    "unit_price": str(
                        item["unit_price"]
                    ),
                    "line_total": str(
                        item["line_total"]
                    ),
                },
            }
        )

    if not 1 <= len(render_items) <= 12:
        raise ValueError(
            "Jumlah line item berada di luar "
            "batas yang diizinkan: "
            f"{len(render_items)}"
        )

    return {
        "document_id": str(
            canonical_payload["document_id"]
        ),
        "canonical_invoice_id": str(
            canonical_payload[
                "canonical_invoice_id"
            ]
        ),
        "split": str(
            canonical_payload["split"]
        ),
        "template_id": str(
            canonical_payload["template_id"]
        ),
        "language": language,
        "currency": currency,
        "title": labels["document_title"],
        "synthetic_notice": (
            labels["synthetic_notice"]
        ),
        "labels": dict(labels),
        "vendor": build_party_render_payload(
            canonical_payload["vendor"],
            language,
        ),
        "buyer": build_party_render_payload(
            canonical_payload["buyer"],
            language,
        ),
        "metadata": {
            "invoice_number": {
                "label": labels[
                    "invoice_number"
                ],
                "raw_value": str(
                    canonical_payload[
                        "invoice_number"
                    ]
                ),
                "display_value": str(
                    canonical_payload[
                        "invoice_number"
                    ]
                ),
            },
            "invoice_date": {
                "label": labels[
                    "invoice_date"
                ],
                "raw_value": str(
                    canonical_payload[
                        "invoice_date"
                    ]
                ),
                "display_value": (
                    format_render_date(
                        canonical_payload[
                            "invoice_date"
                        ],
                        language,
                    )
                ),
            },
            "due_date": {
                "label": labels[
                    "due_date"
                ],
                "raw_value": str(
                    canonical_payload[
                        "due_date"
                    ]
                ),
                "display_value": (
                    format_render_date(
                        canonical_payload[
                            "due_date"
                        ],
                        language,
                    )
                ),
            },
        },
        "items": render_items,
        "financials": {
            "subtotal": {
                "label": labels["subtotal"],
                "raw_value": str(
                    financials["subtotal"]
                ),
                "display_value": (
                    format_render_money(
                        financials["subtotal"],
                        currency,
                        language,
                    )
                ),
            },
            "tax": {
                "label": (
                    f"{labels['tax']} "
                    f"({financials['tax_rate_percent']}%)"
                ),
                "raw_value": str(
                    financials["tax"]
                ),
                "display_value": (
                    format_render_money(
                        financials["tax"],
                        currency,
                        language,
                    )
                ),
            },
            "discount": {
                "label": labels["discount"],
                "raw_value": str(
                    financials["discount"]
                ),
                "display_value": (
                    format_render_money(
                        financials["discount"],
                        currency,
                        language,
                    )
                ),
            },
            "total": {
                "label": labels["total"],
                "raw_value": str(
                    financials["total"]
                ),
                "display_value": (
                    format_render_money(
                        financials["total"],
                        currency,
                        language,
                    )
                ),
            },
        },
        "footer": {
            "document_reference": (
                f"{canonical_payload['document_id']} · "
                f"{canonical_payload['template_id']}"
            ),
            "synthetic_notice": (
                labels["synthetic_notice"]
            ),
        },
    }


# ------------------------------------------------------------------
# Membuat presentation payload untuk seluruh invoice
# ------------------------------------------------------------------

invoice_render_payloads = {
    canonical_id: build_invoice_render_payload(
        canonical_payload
    )
    for canonical_id, canonical_payload
    in sorted(
        canonical_invoice_payloads.items()
    )
}

if len(invoice_render_payloads) != 200:
    raise RuntimeError(
        "Jumlah presentation payload tidak sesuai. "
        f"Expected: 200; "
        f"actual: {len(invoice_render_payloads)}"
    )

reference_render_id = sorted(
    invoice_render_payloads
)[0]

reference_render_payload = (
    invoice_render_payloads[
        reference_render_id
    ]
)

reference_render_schema = (
    collect_schema_signature(
        reference_render_payload
    )
)

render_schema_mismatch_ids = []

for canonical_id, render_payload in (
    invoice_render_payloads.items()
):
    current_schema = (
        collect_schema_signature(
            render_payload
        )
    )

    if current_schema != reference_render_schema:
        render_schema_mismatch_ids.append(
            canonical_id
        )

    json.dumps(
        render_payload,
        ensure_ascii=False,
        allow_nan=False,
    )

if render_schema_mismatch_ids:
    raise RuntimeError(
        "Presentation payload mempunyai schema "
        "yang tidak konsisten: "
        f"{render_schema_mismatch_ids[:10]}"
    )

# ------------------------------------------------------------------
# Preview presentation payload pertama
# ------------------------------------------------------------------

metadata_preview = pd.DataFrame(
    [
        {
            "field": field_name,
            "label": field_payload["label"],
            "raw_value": (
                field_payload["raw_value"]
            ),
            "display_value": (
                field_payload["display_value"]
            ),
        }
        for field_name, field_payload
        in reference_render_payload[
            "metadata"
        ].items()
    ]
)

items_preview = pd.DataFrame(
    reference_render_payload["items"]
).drop(
    columns=["raw"]
)

financial_preview = pd.DataFrame(
    [
        {
            "field": field_name,
            "label": field_payload["label"],
            "raw_value": (
                field_payload["raw_value"]
            ),
            "display_value": (
                field_payload["display_value"]
            ),
        }
        for field_name, field_payload
        in reference_render_payload[
            "financials"
        ].items()
    ]
)

display(metadata_preview)
display(items_preview)
display(financial_preview)

print(
    f"Reference invoice : {reference_render_id}"
)
print(
    f"Language          : "
    f"{reference_render_payload['language']}"
)
print(
    f"Currency          : "
    f"{reference_render_payload['currency']}"
)
print(
    f"Render payloads   : "
    f"{len(invoice_render_payloads)}"
)
print(
    f"Schema paths      : "
    f"{len(reference_render_schema)}"
)
print(
    "Safety notice     : "
    f"{reference_render_payload['synthetic_notice']}"
)
print()
print(
    "✅ Presentation payload bilingual "
    "berhasil dibuat dan divalidasi."
)

**Cell 36 — Code: Renderer prototype TPL-01**

In [ ]:
from pathlib import Path
from PIL import Image as PILImage
import json
import os
import re


PROTOTYPE_ROOT = (
    BUILD_RUN_ROOT
    / "prototypes"
)

PROTOTYPE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ==================================================================
# Text fitting helpers
# ==================================================================

def insert_textbox_with_fit(
    page: pymupdf.Page,
    rectangle: pymupdf.Rect,
    text: str,
    font_name: str,
    font_size: float,
    minimum_font_size: float,
    font_color: tuple[
        float,
        float,
        float,
    ],
    alignment: str,
) -> float:
    """
    Menulis teks dengan penyesuaian font bertahap.

    PyMuPDF tidak menulis teks apabila insert_textbox()
    mengembalikan nilai negatif.
    """

    validate_rectangle(
        page,
        rectangle,
    )

    if alignment not in TEXT_ALIGNMENTS:
        raise ValueError(
            f"Alignment tidak didukung: {alignment}"
        )

    clean_text = str(text).strip()

    if not clean_text:
        raise ValueError(
            "Teks tidak boleh kosong."
        )

    current_font_size = float(
        font_size
    )

    while (
        current_font_size
        >= minimum_font_size
    ):
        remaining_height = page.insert_textbox(
            rectangle,
            clean_text,
            fontname=font_name,
            fontsize=current_font_size,
            color=font_color,
            align=TEXT_ALIGNMENTS[alignment],
            overlay=True,
        )

        if remaining_height >= 0:
            return round(
                current_font_size,
                2,
            )

        current_font_size = round(
            current_font_size - 0.25,
            2,
        )

    raise RuntimeError(
        f"Teks tidak dapat dimuat dalam rectangle "
        f"{rectangle}, bahkan pada ukuran font "
        f"minimum {minimum_font_size}: {clean_text!r}"
    )


def insert_static_text(
    page: pymupdf.Page,
    rectangle: pymupdf.Rect,
    text: str,
    font_name: str = "helv",
    font_size: float = 8.0,
    minimum_font_size: float = 6.0,
    font_color: tuple[
        float,
        float,
        float,
    ] = (0.0, 0.0, 0.0),
    alignment: str = "left",
) -> float:
    """Menulis label visual tanpa anotasi ground truth."""

    return insert_textbox_with_fit(
        page=page,
        rectangle=rectangle,
        text=text,
        font_name=font_name,
        font_size=font_size,
        minimum_font_size=minimum_font_size,
        font_color=font_color,
        alignment=alignment,
    )


def insert_fitted_field_text(
    page: pymupdf.Page,
    rectangle: pymupdf.Rect,
    text: str,
    field_name: str,
    font_name: str = "helv",
    font_size: float = 9.0,
    minimum_font_size: float = 6.5,
    font_color: tuple[
        float,
        float,
        float,
    ] = (0.0, 0.0, 0.0),
    alignment: str = "left",
) -> tuple[FieldAnnotation, float]:
    """
    Menulis nilai field dan menghasilkan anotasi bbox.

    Font hanya boleh diperkecil sampai batas minimum
    agar dokumen tetap terbaca.
    """

    if not re.fullmatch(
        r"[a-z][a-z0-9_.\[\]-]*",
        field_name,
    ):
        raise ValueError(
            f"Nama field tidak valid: {field_name}"
        )

    clean_text = str(text).strip()

    if not clean_text:
        raise ValueError(
            f"Nilai kosong untuk field: {field_name}"
        )

    used_font_size = insert_textbox_with_fit(
        page=page,
        rectangle=rectangle,
        text=clean_text,
        font_name=font_name,
        font_size=font_size,
        minimum_font_size=minimum_font_size,
        font_color=font_color,
        alignment=alignment,
    )

    bbox_points = tuple(
        round(coordinate, 3)
        for coordinate in (
            rectangle.x0,
            rectangle.y0,
            rectangle.x1,
            rectangle.y1,
        )
    )

    bbox_normalized = normalize_rectangle(
        rectangle,
        page.rect,
    )

    annotation = FieldAnnotation(
        field_name=field_name,
        text=clean_text,
        page_number=page.number + 1,
        bbox_points=bbox_points,
        bbox_normalized=bbox_normalized,
        annotation_type="field_region",
    )

    return (
        annotation,
        used_font_size,
    )


def annotation_to_dict(
    annotation: FieldAnnotation,
) -> dict:
    """Mengubah anotasi menjadi dictionary JSON."""

    return {
        "field_name": annotation.field_name,
        "text": annotation.text,
        "page_number": annotation.page_number,
        "bbox_points": list(
            annotation.bbox_points
        ),
        "bbox_normalized": list(
            annotation.bbox_normalized
        ),
        "annotation_type": (
            annotation.annotation_type
        ),
    }


# ==================================================================
# TPL-01 renderer
# ==================================================================

def render_template_01(
    render_payload: dict,
    output_pdf_path: Path,
) -> tuple[list[FieldAnnotation], dict]:
    """Merender satu invoice dengan desain TPL-01."""

    if render_payload["template_id"] != "TPL-01":
        raise ValueError(
            "render_template_01 hanya menerima TPL-01."
        )

    if not 1 <= len(
        render_payload["items"]
    ) <= 8:
        raise ValueError(
            "TPL-01 hanya mendukung 1–8 line item "
            "dalam satu halaman."
        )

    output_pdf_path = Path(
        output_pdf_path
    )

    output_pdf_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    template_spec = TEMPLATE_REGISTRY[
        "TPL-01"
    ]

    page_width, page_height = (
        get_page_dimensions(
            template_spec.page_size
        )
    )

    primary_color = hex_to_rgb(
        template_spec.primary_color
    )

    accent_color = hex_to_rgb(
        template_spec.accent_color
    )

    dark_text = (
        0.12,
        0.15,
        0.18,
    )

    muted_text = (
        0.38,
        0.42,
        0.46,
    )

    light_fill = (
        0.965,
        0.975,
        0.985,
    )

    border_color = (
        0.80,
        0.83,
        0.86,
    )

    white = (
        1.0,
        1.0,
        1.0,
    )

    annotations = []
    field_font_audit = []

    document = pymupdf.open()

    temporary_pdf_path = (
        output_pdf_path.parent
        / f".{output_pdf_path.name}.tmp"
    )

    def add_field(
        page: pymupdf.Page,
        rectangle: pymupdf.Rect,
        text: str,
        field_name: str,
        font_name: str = "helv",
        font_size: float = 9.0,
        minimum_font_size: float = 6.5,
        font_color: tuple[
            float,
            float,
            float,
        ] = dark_text,
        alignment: str = "left",
    ) -> None:
        """Menambahkan field dan mencatat font aktual."""

        (
            annotation,
            used_font_size,
        ) = insert_fitted_field_text(
            page=page,
            rectangle=rectangle,
            text=text,
            field_name=field_name,
            font_name=font_name,
            font_size=font_size,
            minimum_font_size=minimum_font_size,
            font_color=font_color,
            alignment=alignment,
        )

        annotations.append(
            annotation
        )

        field_font_audit.append(
            {
                "field_name": field_name,
                "requested_font_size": (
                    font_size
                ),
                "used_font_size": (
                    used_font_size
                ),
                "adjusted": (
                    used_font_size
                    < font_size
                ),
            }
        )

    try:
        page = document.new_page(
            width=page_width,
            height=page_height,
        )

        # ==========================================================
        # Header
        # ==========================================================

        page.draw_rect(
            pymupdf.Rect(
                0,
                0,
                page_width,
                12,
            ),
            color=accent_color,
            fill=accent_color,
            width=0,
            overlay=True,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                42,
                34,
                310,
                55,
            ),
            text="SYNTHETIC COMMERCE DOCUMENT",
            font_name="hebo",
            font_size=9.0,
            minimum_font_size=7.0,
            font_color=primary_color,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                42,
                58,
                310,
                78,
            ),
            text=render_payload[
                "footer"
            ]["document_reference"],
            font_name="helv",
            font_size=8.0,
            minimum_font_size=7.0,
            font_color=muted_text,
        )

        # Area judul diperbesar dan font dibuat lebih aman.
        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                330,
                27,
                page_width - 42,
                86,
            ),
            text=render_payload["title"],
            font_name="hebo",
            font_size=24.0,
            minimum_font_size=18.0,
            font_color=primary_color,
            alignment="right",
        )

        page.draw_line(
            pymupdf.Point(
                42,
                92,
            ),
            pymupdf.Point(
                page_width - 42,
                92,
            ),
            color=primary_color,
            width=1.2,
            overlay=True,
        )

        # ==========================================================
        # Vendor dan buyer
        # ==========================================================

        party_panels = [
            (
                "vendor",
                render_payload[
                    "labels"
                ]["vendor"],
                pymupdf.Rect(
                    42,
                    112,
                    288,
                    250,
                ),
            ),
            (
                "buyer",
                render_payload[
                    "labels"
                ]["buyer"],
                pymupdf.Rect(
                    306,
                    112,
                    page_width - 42,
                    250,
                ),
            ),
        ]

        for (
            party_type,
            party_label,
            panel_rectangle,
        ) in party_panels:
            party = render_payload[
                party_type
            ]

            draw_panel(
                page=page,
                rectangle=panel_rectangle,
                border_color=border_color,
                fill_color=light_fill,
                border_width=0.8,
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    panel_rectangle.x0 + 10,
                    122,
                    panel_rectangle.x1 - 10,
                    137,
                ),
                text=party_label.upper(),
                font_name="hebo",
                font_size=7.5,
                minimum_font_size=6.5,
                font_color=accent_color,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    panel_rectangle.x0 + 10,
                    141,
                    panel_rectangle.x1 - 10,
                    158,
                ),
                text=party["name"],
                field_name=(
                    f"{party_type}.name"
                ),
                font_name="hebo",
                font_size=9.5,
                minimum_font_size=7.5,
                font_color=dark_text,
            )

            address_text = "\n".join(
                party["address_lines"]
            )

            # Area alamat diperbesar.
            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    panel_rectangle.x0 + 10,
                    160,
                    panel_rectangle.x1 - 10,
                    196,
                ),
                text=address_text,
                field_name=(
                    f"{party_type}.address_lines"
                ),
                font_name="helv",
                font_size=7.2,
                minimum_font_size=6.5,
                font_color=dark_text,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    panel_rectangle.x0 + 10,
                    198,
                    panel_rectangle.x1 - 10,
                    212,
                ),
                text=party["email"],
                field_name=(
                    f"{party_type}.email"
                ),
                font_name="helv",
                font_size=7.3,
                minimum_font_size=6.5,
                font_color=dark_text,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    panel_rectangle.x0 + 10,
                    214,
                    panel_rectangle.x1 - 10,
                    228,
                ),
                text=party["phone"],
                field_name=(
                    f"{party_type}.phone"
                ),
                font_name="helv",
                font_size=7.3,
                minimum_font_size=6.5,
                font_color=dark_text,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    panel_rectangle.x0 + 10,
                    230,
                    panel_rectangle.x1 - 10,
                    245,
                ),
                text=party[
                    "tax_identifier_display"
                ],
                field_name=(
                    f"{party_type}.tax_identifier"
                ),
                font_name="helv",
                font_size=7.3,
                minimum_font_size=6.5,
                font_color=dark_text,
            )

        # ==========================================================
        # Invoice metadata
        # ==========================================================

        metadata_fields = [
            (
                "invoice_number",
                42,
                198,
            ),
            (
                "invoice_date",
                208,
                354,
            ),
            (
                "due_date",
                364,
                480,
            ),
        ]

        for (
            field_name,
            x0,
            x1,
        ) in metadata_fields:
            field_payload = (
                render_payload[
                    "metadata"
                ][field_name]
            )

            draw_panel(
                page=page,
                rectangle=pymupdf.Rect(
                    x0,
                    266,
                    x1,
                    324,
                ),
                border_color=border_color,
                fill_color=white,
                border_width=0.7,
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    x0 + 8,
                    275,
                    x1 - 8,
                    289,
                ),
                text=field_payload["label"],
                font_name="hebo",
                font_size=7.0,
                minimum_font_size=6.0,
                font_color=muted_text,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    x0 + 8,
                    294,
                    x1 - 8,
                    316,
                ),
                text=field_payload[
                    "display_value"
                ],
                field_name=field_name,
                font_name="hebo",
                font_size=8.5,
                minimum_font_size=7.0,
                font_color=dark_text,
            )

        currency_x0 = 490
        currency_x1 = (
            page_width - 42
        )

        draw_panel(
            page=page,
            rectangle=pymupdf.Rect(
                currency_x0,
                266,
                currency_x1,
                324,
            ),
            border_color=border_color,
            fill_color=white,
            border_width=0.7,
        )

        currency_label = (
            "Mata Uang"
            if render_payload["language"] == "id"
            else "Currency"
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                currency_x0 + 8,
                275,
                currency_x1 - 8,
                289,
            ),
            text=currency_label,
            font_name="hebo",
            font_size=7.0,
            minimum_font_size=6.0,
            font_color=muted_text,
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                currency_x0 + 8,
                294,
                currency_x1 - 8,
                316,
            ),
            text=render_payload["currency"],
            field_name="currency",
            font_name="hebo",
            font_size=9.0,
            minimum_font_size=7.0,
            font_color=dark_text,
        )

        # ==========================================================
        # Line-item table
        # ==========================================================

        table_x_positions = [
            42,
            70,
            302,
            352,
            451,
            page_width - 42,
        ]

        table_header_y0 = 340
        table_header_y1 = 369
        row_height = 26

        page.draw_rect(
            pymupdf.Rect(
                table_x_positions[0],
                table_header_y0,
                table_x_positions[-1],
                table_header_y1,
            ),
            color=primary_color,
            fill=primary_color,
            width=0.8,
            overlay=True,
        )

        table_headers = [
            "#",
            render_payload[
                "labels"
            ]["description"],
            render_payload[
                "labels"
            ]["quantity"],
            render_payload[
                "labels"
            ]["unit_price"],
            render_payload[
                "labels"
            ]["line_total"],
        ]

        for column_index, header_text in enumerate(
            table_headers
        ):
            header_alignment = (
                "left"
                if column_index == 1
                else "center"
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    table_x_positions[
                        column_index
                    ] + 4,
                    table_header_y0 + 7,
                    table_x_positions[
                        column_index + 1
                    ] - 4,
                    table_header_y1 - 3,
                ),
                text=header_text,
                font_name="hebo",
                font_size=7.4,
                minimum_font_size=6.0,
                font_color=white,
                alignment=header_alignment,
            )

        for item_index, item in enumerate(
            render_payload["items"]
        ):
            row_y0 = (
                table_header_y1
                + item_index * row_height
            )

            row_y1 = (
                row_y0 + row_height
            )

            row_fill = (
                light_fill
                if item_index % 2 == 0
                else white
            )

            page.draw_rect(
                pymupdf.Rect(
                    table_x_positions[0],
                    row_y0,
                    table_x_positions[-1],
                    row_y1,
                ),
                color=border_color,
                fill=row_fill,
                width=0.5,
                overlay=True,
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    table_x_positions[0] + 3,
                    row_y0 + 6,
                    table_x_positions[1] - 3,
                    row_y1 - 3,
                ),
                text=str(
                    item["item_number"]
                ),
                font_name="helv",
                font_size=7.8,
                minimum_font_size=6.5,
                font_color=dark_text,
                alignment="center",
            )

            item_fields = [
                (
                    "description",
                    1,
                    "left",
                ),
                (
                    "quantity",
                    2,
                    "center",
                ),
                (
                    "unit_price",
                    3,
                    "right",
                ),
                (
                    "line_total",
                    4,
                    "right",
                ),
            ]

            for (
                item_field,
                column_index,
                field_alignment,
            ) in item_fields:
                add_field(
                    page=page,
                    rectangle=pymupdf.Rect(
                        table_x_positions[
                            column_index
                        ] + 4,
                        row_y0 + 6,
                        table_x_positions[
                            column_index + 1
                        ] - 4,
                        row_y1 - 3,
                    ),
                    text=item[item_field],
                    field_name=(
                        f"items[{item_index}]"
                        f".{item_field}"
                    ),
                    font_name="helv",
                    font_size=7.6,
                    minimum_font_size=6.5,
                    font_color=dark_text,
                    alignment=field_alignment,
                )

        # ==========================================================
        # Financial summary
        # ==========================================================

        totals_panel = pymupdf.Rect(
            330,
            598,
            page_width - 42,
            706,
        )

        draw_panel(
            page=page,
            rectangle=totals_panel,
            border_color=border_color,
            fill_color=light_fill,
            border_width=0.8,
        )

        financial_order = [
            "subtotal",
            "tax",
            "discount",
            "total",
        ]

        for row_index, field_name in enumerate(
            financial_order
        ):
            field_payload = (
                render_payload[
                    "financials"
                ][field_name]
            )

            row_y0 = (
                607 + row_index * 23
            )

            is_total = (
                field_name == "total"
            )

            if is_total:
                page.draw_line(
                    pymupdf.Point(
                        342,
                        row_y0 - 4,
                    ),
                    pymupdf.Point(
                        page_width - 54,
                        row_y0 - 4,
                    ),
                    color=accent_color,
                    width=1.0,
                    overlay=True,
                )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    342,
                    row_y0,
                    426,
                    row_y0 + 18,
                ),
                text=field_payload["label"],
                font_name=(
                    "hebo"
                    if is_total
                    else "helv"
                ),
                font_size=(
                    8.5
                    if is_total
                    else 8.0
                ),
                minimum_font_size=6.5,
                font_color=(
                    primary_color
                    if is_total
                    else dark_text
                ),
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    430,
                    row_y0,
                    page_width - 54,
                    row_y0 + 18,
                ),
                text=field_payload[
                    "display_value"
                ],
                field_name=(
                    f"financials.{field_name}"
                ),
                font_name=(
                    "hebo"
                    if is_total
                    else "helv"
                ),
                font_size=(
                    8.8
                    if is_total
                    else 8.0
                ),
                minimum_font_size=6.5,
                font_color=(
                    primary_color
                    if is_total
                    else dark_text
                ),
                alignment="right",
            )

        # ==========================================================
        # Footer
        # ==========================================================

        page.draw_line(
            pymupdf.Point(
                42,
                758,
            ),
            pymupdf.Point(
                page_width - 42,
                758,
            ),
            color=border_color,
            width=0.7,
            overlay=True,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                42,
                767,
                page_width - 42,
                782,
            ),
            text=render_payload[
                "footer"
            ]["document_reference"],
            font_name="helv",
            font_size=7.0,
            minimum_font_size=6.0,
            font_color=muted_text,
            alignment="center",
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                42,
                787,
                page_width - 42,
                807,
            ),
            text=render_payload[
                "synthetic_notice"
            ],
            field_name=(
                "document.synthetic_notice"
            ),
            font_name="hebo",
            font_size=7.2,
            minimum_font_size=6.5,
            font_color=accent_color,
            alignment="center",
        )

        # ==========================================================
        # Pre-save validation
        # ==========================================================

        annotation_names = [
            annotation.field_name
            for annotation in annotations
        ]

        if (
            len(annotation_names)
            != len(set(annotation_names))
        ):
            raise RuntimeError(
                "Ditemukan field annotation duplikat."
            )

        minimum_used_font_size = min(
            font_record["used_font_size"]
            for font_record in field_font_audit
        )

        if minimum_used_font_size < 6.5:
            raise RuntimeError(
                "Ukuran font field berada di bawah "
                "batas keterbacaan."
            )

        if temporary_pdf_path.exists():
            temporary_pdf_path.unlink()

        document.save(
            str(temporary_pdf_path),
            garbage=4,
            deflate=True,
        )

    except Exception:
        if temporary_pdf_path.exists():
            temporary_pdf_path.unlink()

        raise

    finally:
        document.close()

    if not temporary_pdf_path.is_file():
        raise RuntimeError(
            "File PDF sementara tidak berhasil dibuat."
        )

    os.replace(
        temporary_pdf_path,
        output_pdf_path,
    )

    font_adjustment_count = sum(
        font_record["adjusted"]
        for font_record in field_font_audit
    )

    rendering_metadata = {
        "renderer_version": "1.1.0",
        "template_id": "TPL-01",
        "page_size": template_spec.page_size,
        "page_width_points": round(
            page_width,
            3,
        ),
        "page_height_points": round(
            page_height,
            3,
        ),
        "page_count": 1,
        "annotation_type": "field_region",
        "minimum_field_font_size": min(
            font_record["used_font_size"]
            for font_record in field_font_audit
        ),
        "font_adjustment_count": int(
            font_adjustment_count
        ),
    }

    return (
        annotations,
        rendering_metadata,
    )


# ==================================================================
# Render satu prototype
# ==================================================================

prototype_canonical_id = (
    "CANON-000001"
)

prototype_render_payload = (
    invoice_render_payloads[
        prototype_canonical_id
    ]
)

prototype_pdf_path = (
    PROTOTYPE_ROOT
    / "INV-SYN-000001_TPL-01_prototype.pdf"
)

prototype_png_path = (
    PROTOTYPE_ROOT
    / "INV-SYN-000001_TPL-01_preview.png"
)

prototype_ground_truth_path = (
    PROTOTYPE_ROOT
    / "INV-SYN-000001_TPL-01_ground_truth.json"
)

(
    prototype_annotations,
    prototype_rendering_metadata,
) = render_template_01(
    render_payload=prototype_render_payload,
    output_pdf_path=prototype_pdf_path,
)


# ==================================================================
# Simpan prototype ground truth secara atomik
# ==================================================================

prototype_ground_truth = {
    "schema_version": "1.0.0",
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "document": {
        "document_id": (
            prototype_render_payload[
                "document_id"
            ]
        ),
        "canonical_invoice_id": (
            prototype_canonical_id
        ),
        "split": (
            prototype_render_payload[
                "split"
            ]
        ),
        "template_id": (
            prototype_render_payload[
                "template_id"
            ]
        ),
        "language": (
            prototype_render_payload[
                "language"
            ]
        ),
        "currency": (
            prototype_render_payload[
                "currency"
            ]
        ),
    },
    "canonical": (
        canonical_invoice_payloads[
            prototype_canonical_id
        ]
    ),
    "rendering": (
        prototype_rendering_metadata
    ),
    "annotations": [
        annotation_to_dict(annotation)
        for annotation
        in prototype_annotations
    ],
}

temporary_ground_truth_path = (
    prototype_ground_truth_path.parent
    / (
        f".{prototype_ground_truth_path.name}"
        ".tmp"
    )
)

with temporary_ground_truth_path.open(
    mode="w",
    encoding="utf-8",
    newline="\n",
) as ground_truth_file:
    json.dump(
        prototype_ground_truth,
        ground_truth_file,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )

    ground_truth_file.write("\n")

os.replace(
    temporary_ground_truth_path,
    prototype_ground_truth_path,
)


# ==================================================================
# Reopen PDF, extract text, and create preview
# ==================================================================

temporary_png_path = (
    prototype_png_path.parent
    / f".{prototype_png_path.name}.tmp.png"
)

with pymupdf.open(
    str(prototype_pdf_path)
) as prototype_document:
    if prototype_document.page_count != 1:
        raise RuntimeError(
            "Prototype harus memiliki tepat "
            "satu halaman."
        )

    prototype_page = (
        prototype_document[0]
    )

    extracted_text = (
        prototype_page.get_text("text")
    )

    render_matrix = pymupdf.Matrix(
        150 / 72,
        150 / 72,
    )

    prototype_pixmap = (
        prototype_page.get_pixmap(
            matrix=render_matrix,
            alpha=False,
        )
    )

    prototype_pixmap.save(
        str(temporary_png_path)
    )

os.replace(
    temporary_png_path,
    prototype_png_path,
)


# ==================================================================
# Post-render validation
# ==================================================================

# CHECKPOINT CLEANUP:
# Fragmen validasi lama yang salah indentasi dihapus.
# Validasi TPL-01 dilakukan kembali oleh renderer final,
# technical QA, dan final integrity audit.

annotation_names = [
    annotation.field_name
    for annotation
    in prototype_annotations
]

required_annotation_names = {
    "vendor.name",
    "buyer.name",
    "invoice_number",
    "invoice_date",
    "due_date",
    "currency",
    "financials.subtotal",
    "financials.tax",
    "financials.discount",
    "financials.total",
    "document.synthetic_notice",
}

missing_annotations = sorted(
    required_annotation_names
    - set(annotation_names)
)

if missing_annotations:
    raise RuntimeError(
        "Anotasi wajib tidak tersedia: "
        f"{missing_annotations}"
    )

if not prototype_pdf_path.is_file():
    raise RuntimeError(
        "Prototype PDF tidak ditemukan."
    )

if not prototype_png_path.is_file():
    raise RuntimeError(
        "Prototype preview tidak ditemukan."
    )

if not prototype_ground_truth_path.is_file():
    raise RuntimeError(
        "Prototype ground truth tidak ditemukan."
    )

annotation_preview = pd.DataFrame(
    [
        annotation_to_dict(annotation)
        for annotation
        in prototype_annotations
    ]
)

font_audit_summary = pd.DataFrame(
    [
        {
            "control": "annotation_count",
            "value": len(
                prototype_annotations
            ),
            "status": "VALID",
        },
        {
            "control": "minimum_field_font_size",
            "value": (
                prototype_rendering_metadata[
                    "minimum_field_font_size"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "font_adjustment_count",
            "value": (
                prototype_rendering_metadata[
                    "font_adjustment_count"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "required_text_extraction",
            "value": len(
                required_text_fragments
            ),
            "status": "VALID",
        },
    ]
)

display(
    annotation_preview.head(15)
)

display(
    font_audit_summary
)

with PILImage.open(
    prototype_png_path
) as preview_image:
    display(
        preview_image.copy()
    )

print(
    f"Prototype PDF       : "
    f"{prototype_pdf_path}"
)
print(
    f"Prototype preview   : "
    f"{prototype_png_path}"
)
print(
    f"Prototype GT        : "
    f"{prototype_ground_truth_path}"
)
print(
    f"Annotations         : "
    f"{len(prototype_annotations)}"
)
print(
    f"Extracted characters: "
    f"{len(extracted_text)}"
)
print(
    f"Preview size        : "
    f"{prototype_pixmap.width} × "
    f"{prototype_pixmap.height}"
)
print(
    f"Minimum field font  : "
    f"{prototype_rendering_metadata['minimum_field_font_size']}"
)
print(
    f"Adjusted fields     : "
    f"{prototype_rendering_metadata['font_adjustment_count']}"
)
print()
print(
    "✅ Prototype TPL-01 berhasil dirender "
    "dan lolos pemeriksaan teknis."
)

In [ ]:
from pathlib import Path
from google.colab import drive


# ================================================================
# 1. Mount Google Drive jika belum terhubung
# ================================================================

DRIVE_ROOT = Path(
    "/content/drive/MyDrive"
)

if not DRIVE_ROOT.is_dir():
    print(
        "Google Drive belum terhubung. "
        "Memulai proses mount..."
    )

    drive.mount(
        "/content/drive"
    )
else:
    print(
        "✅ Google Drive sudah terhubung."
    )


# ================================================================
# 2. Menetapkan build yang akan dilanjutkan
# ================================================================

PROJECT_DATA_ROOT = (
    DRIVE_ROOT
    / "InvoiceFlow-AI-Data"
)

RESUME_BUILD_ID = (
    "20260904T150025Z"
)

BUILD_BASE_ROOT = (
    PROJECT_DATA_ROOT
    / "interim"
    / "synthetic_v1_build"
)

BUILD_RUN_ROOT = (
    BUILD_BASE_ROOT
    / RESUME_BUILD_ID
)

BUILD_CONFIGURATION_ROOT = (
    BUILD_RUN_ROOT
    / "configuration"
)

PROTOTYPE_ROOT = (
    BUILD_RUN_ROOT
    / "prototypes"
)


# ================================================================
# 3. Menetapkan artifact checkpoint
# ================================================================

ENVIRONMENT_MANIFEST_PATH = (
    PROJECT_DATA_ROOT
    / "environment_manifest.json"
)

SOURCE_REGISTRY_PATH = (
    PROJECT_DATA_ROOT
    / "source_registry.csv"
)

DATASET_CONFIGURATION_PATH = (
    BUILD_CONFIGURATION_ROOT
    / "dataset_configuration.json"
)

GENERATION_PLAN_PATH = (
    BUILD_CONFIGURATION_ROOT
    / "generation_plan.csv"
)

prototype_pdf_path = (
    PROTOTYPE_ROOT
    / "INV-SYN-000001_TPL-01_prototype.pdf"
)

prototype_png_path = (
    PROTOTYPE_ROOT
    / "INV-SYN-000001_TPL-01_preview.png"
)

prototype_ground_truth_path = (
    PROTOTYPE_ROOT
    / "INV-SYN-000001_TPL-01_ground_truth.json"
)


# ================================================================
# 4. Memvalidasi folder dan seluruh artifact wajib
# ================================================================

required_directories = {
    "project_data_root": PROJECT_DATA_ROOT,
    "build_run_root": BUILD_RUN_ROOT,
    "configuration": BUILD_CONFIGURATION_ROOT,
    "prototypes": PROTOTYPE_ROOT,
}

required_files = {
    "environment_manifest": (
        ENVIRONMENT_MANIFEST_PATH
    ),
    "source_registry": (
        SOURCE_REGISTRY_PATH
    ),
    "dataset_configuration": (
        DATASET_CONFIGURATION_PATH
    ),
    "generation_plan": (
        GENERATION_PLAN_PATH
    ),
    "prototype_pdf": (
        prototype_pdf_path
    ),
    "prototype_preview": (
        prototype_png_path
    ),
    "prototype_ground_truth": (
        prototype_ground_truth_path
    ),
}

missing_directories = [
    directory_name
    for directory_name, directory_path
    in required_directories.items()
    if not directory_path.is_dir()
]

missing_files = [
    file_name
    for file_name, file_path
    in required_files.items()
    if (
        not file_path.is_file()
        or file_path.stat().st_size == 0
    )
]

if missing_directories:
    raise FileNotFoundError(
        "Folder checkpoint tidak lengkap: "
        f"{missing_directories}"
    )

if missing_files:
    raise FileNotFoundError(
        "File checkpoint tidak lengkap atau kosong: "
        f"{missing_files}"
    )


# ================================================================
# 5. Menampilkan hasil
# ================================================================

print()
print(
    f"Project data : {PROJECT_DATA_ROOT}"
)
print(
    f"Resume build : {RESUME_BUILD_ID}"
)
print(
    f"Build path   : {BUILD_RUN_ROOT}"
)
print()
print("Artifact checkpoint:")

for file_name, file_path in (
    required_files.items()
):
    file_size_kb = (
        file_path.stat().st_size
        / 1024
    )

    print(
        f"  ✅ {file_name:<24} "
        f"{file_size_kb:>10.2f} KB"
    )

print()
print(
    "✅ Checkpoint Cell 36 tersedia dan siap "
    "dipulihkan. Build baru tidak dibuat."
)

In [ ]:
import sys
import subprocess
from importlib import metadata


# ================================================================
# Package minimum untuk audit dan rendering Notebook 01
# ================================================================

AUDIT_RUNTIME_PACKAGES = {
    "numpy": "2.1.3",
    "pandas": "2.2.3",
    "PyMuPDF": "1.28.2",
    "Pillow": "11.3.0",
    "opencv-contrib-python": "4.10.0.84",
}

OPENCV_DISTRIBUTIONS = (
    "opencv-python",
    "opencv-python-headless",
    "opencv-contrib-python",
    "opencv-contrib-python-headless",
)


def get_distribution_version(
    distribution_name: str,
) -> str | None:
    """Mengambil versi package yang terpasang."""

    try:
        return metadata.version(
            distribution_name
        )
    except metadata.PackageNotFoundError:
        return None


def run_pip_command(
    arguments: list[str],
) -> None:
    """Menjalankan pip dengan error handling."""

    command = [
        sys.executable,
        "-m",
        "pip",
        "--disable-pip-version-check",
        *arguments,
    ]

    subprocess.run(
        command,
        check=True,
    )


# ================================================================
# Normalisasi OpenCV
# ================================================================

installed_opencv = {
    distribution_name: (
        get_distribution_version(
            distribution_name
        )
    )
    for distribution_name
    in OPENCV_DISTRIBUTIONS
}

opencv_conflicts = [
    distribution_name
    for distribution_name, installed_version
    in installed_opencv.items()
    if (
        installed_version is not None
        and distribution_name
        != "opencv-contrib-python"
    )
]

opencv_standard_version = (
    installed_opencv[
        "opencv-contrib-python"
    ]
)

opencv_requires_normalization = (
    bool(opencv_conflicts)
    or opencv_standard_version
    != AUDIT_RUNTIME_PACKAGES[
        "opencv-contrib-python"
    ]
)

if opencv_requires_normalization:
    installed_opencv_names = [
        distribution_name
        for distribution_name, installed_version
        in installed_opencv.items()
        if installed_version is not None
    ]

    if installed_opencv_names:
        print(
            "Menormalkan distribusi OpenCV..."
        )

        run_pip_command(
            [
                "uninstall",
                "-y",
                *installed_opencv_names,
            ]
        )

    print(
        "Memasang OpenCV standar proyek..."
    )

    run_pip_command(
        [
            "install",
            "--quiet",
            (
                "opencv-contrib-python"
                "==4.10.0.84"
            ),
        ]
    )
else:
    print(
        "✅ Distribusi OpenCV sudah sesuai."
    )


# ================================================================
# Sinkronisasi package audit lainnya
# ================================================================

packages_to_install = []

for (
    distribution_name,
    expected_version,
) in AUDIT_RUNTIME_PACKAGES.items():
    if (
        distribution_name
        == "opencv-contrib-python"
    ):
        continue

    current_version = (
        get_distribution_version(
            distribution_name
        )
    )

    if current_version != expected_version:
        packages_to_install.append(
            f"{distribution_name}"
            f"=={expected_version}"
        )

if packages_to_install:
    print(
        "Menyinkronkan package audit..."
    )

    run_pip_command(
        [
            "install",
            "--quiet",
            "--upgrade",
            *packages_to_install,
        ]
    )
else:
    print(
        "✅ Package audit sudah sesuai."
    )


# ================================================================
# Import ulang library runtime
# ================================================================

import cv2
import numpy as np
import pandas as pd
import pymupdf

from PIL import Image as PILImage


# ================================================================
# Validasi versi akhir
# ================================================================

runtime_audit_rows = []

for (
    distribution_name,
    expected_version,
) in AUDIT_RUNTIME_PACKAGES.items():
    current_version = (
        get_distribution_version(
            distribution_name
        )
    )

    runtime_audit_rows.append(
        {
            "distribution": (
                distribution_name
            ),
            "expected_version": (
                expected_version
            ),
            "current_version": (
                current_version
            ),
            "status": (
                "MATCH"
                if current_version
                == expected_version
                else "DIFFERENT"
            ),
        }
    )

runtime_audit = pd.DataFrame(
    runtime_audit_rows
)

remaining_opencv_distributions = {
    distribution_name: (
        get_distribution_version(
            distribution_name
        )
    )
    for distribution_name
    in OPENCV_DISTRIBUTIONS
    if get_distribution_version(
        distribution_name
    ) is not None
}

active_cv2_version = (
    cv2.__version__
)

invalid_runtime_packages = (
    runtime_audit.loc[
        runtime_audit["status"] != "MATCH",
        "distribution",
    ].tolist()
)

if invalid_runtime_packages:
    display(runtime_audit)

    raise RuntimeError(
        "Versi package audit belum sesuai: "
        f"{invalid_runtime_packages}"
    )

if list(
    remaining_opencv_distributions
) != ["opencv-contrib-python"]:
    raise RuntimeError(
        "Distribusi OpenCV belum bersih: "
        f"{remaining_opencv_distributions}"
    )

if active_cv2_version != "4.10.0":
    raise RuntimeError(
        "Versi modul cv2 aktif tidak sesuai. "
        f"Expected: 4.10.0; "
        f"actual: {active_cv2_version}"
    )

display(runtime_audit)

print(
    f"Active cv2       : {active_cv2_version}"
)
print(
    "OpenCV packages : "
    f"{remaining_opencv_distributions}"
)
print()
print(
    "✅ Runtime audit berhasil dipulihkan "
    "dan siap untuk Cell 37."
)

In [ ]:
from dataclasses import dataclass
import json


# ================================================================
# Kontrak anotasi
# ================================================================

@dataclass(frozen=True)
class FieldAnnotation:
    field_name: str
    text: str
    page_number: int
    bbox_points: tuple[
        float,
        float,
        float,
        float,
    ]
    bbox_normalized: tuple[
        float,
        float,
        float,
        float,
    ]
    annotation_type: str = "field_region"


def normalize_rectangle(
    rectangle: pymupdf.Rect,
    page_rectangle: pymupdf.Rect,
) -> tuple[float, float, float, float]:
    """Mengubah bbox point menjadi koordinat 0–1."""

    if (
        rectangle.is_empty
        or rectangle.is_infinite
    ):
        raise ValueError(
            f"Rectangle tidak valid: {rectangle}"
        )

    normalized_coordinates = (
        rectangle.x0 / page_rectangle.width,
        rectangle.y0 / page_rectangle.height,
        rectangle.x1 / page_rectangle.width,
        rectangle.y1 / page_rectangle.height,
    )

    if not all(
        0.0 <= coordinate <= 1.0
        for coordinate
        in normalized_coordinates
    ):
        raise ValueError(
            "Koordinat normalisasi berada di luar "
            f"rentang 0–1: {normalized_coordinates}"
        )

    return tuple(
        round(coordinate, 6)
        for coordinate
        in normalized_coordinates
    )


# ================================================================
# Memuat ground truth prototype
# ================================================================

with prototype_ground_truth_path.open(
    mode="r",
    encoding="utf-8",
) as ground_truth_file:
    restored_ground_truth = json.load(
        ground_truth_file
    )

required_root_keys = {
    "schema_version",
    "dataset_id",
    "document",
    "canonical",
    "rendering",
    "annotations",
}

missing_root_keys = sorted(
    required_root_keys
    - set(restored_ground_truth)
)

if missing_root_keys:
    raise KeyError(
        "Ground truth prototype tidak lengkap: "
        f"{missing_root_keys}"
    )

restored_document = (
    restored_ground_truth["document"]
)

restored_canonical = (
    restored_ground_truth["canonical"]
)

restored_annotation_rows = (
    restored_ground_truth["annotations"]
)

prototype_rendering_metadata = (
    restored_ground_truth["rendering"]
)

if not restored_annotation_rows:
    raise RuntimeError(
        "Ground truth prototype tidak memiliki "
        "anotasi."
    )


# ================================================================
# Memulihkan object FieldAnnotation
# ================================================================

prototype_annotations = []

for annotation_row in (
    restored_annotation_rows
):
    required_annotation_keys = {
        "field_name",
        "text",
        "page_number",
        "bbox_points",
        "bbox_normalized",
        "annotation_type",
    }

    missing_annotation_keys = sorted(
        required_annotation_keys
        - set(annotation_row)
    )

    if missing_annotation_keys:
        raise KeyError(
            "Struktur anotasi tidak lengkap pada "
            f"{annotation_row.get('field_name')}: "
            f"{missing_annotation_keys}"
        )

    bbox_points = tuple(
        float(coordinate)
        for coordinate
        in annotation_row["bbox_points"]
    )

    bbox_normalized = tuple(
        float(coordinate)
        for coordinate
        in annotation_row["bbox_normalized"]
    )

    if (
        len(bbox_points) != 4
        or len(bbox_normalized) != 4
    ):
        raise ValueError(
            "Bounding box harus memiliki tepat "
            "empat koordinat."
        )

    prototype_annotations.append(
        FieldAnnotation(
            field_name=str(
                annotation_row["field_name"]
            ),
            text=str(
                annotation_row["text"]
            ),
            page_number=int(
                annotation_row["page_number"]
            ),
            bbox_points=bbox_points,
            bbox_normalized=bbox_normalized,
            annotation_type=str(
                annotation_row[
                    "annotation_type"
                ]
            ),
        )
    )


# ================================================================
# Membuat lookup anotasi
# ================================================================

annotation_lookup = {
    annotation.field_name: annotation
    for annotation
    in prototype_annotations
}

if (
    len(annotation_lookup)
    != len(prototype_annotations)
):
    raise RuntimeError(
        "Ditemukan nama anotasi duplikat."
    )

required_restored_fields = {
    "vendor.name",
    "buyer.name",
    "invoice_number",
    "invoice_date",
    "due_date",
    "currency",
    "financials.subtotal",
    "financials.tax",
    "financials.discount",
    "financials.total",
    "document.synthetic_notice",
}

missing_restored_fields = sorted(
    required_restored_fields
    - set(annotation_lookup)
)

if missing_restored_fields:
    raise RuntimeError(
        "Anotasi wajib tidak berhasil dipulihkan: "
        f"{missing_restored_fields}"
    )


# ================================================================
# Memulihkan reference render payload untuk Cell 37
# ================================================================

prototype_canonical_id = str(
    restored_document[
        "canonical_invoice_id"
    ]
)

prototype_render_payload = {
    "document_id": str(
        restored_document["document_id"]
    ),
    "canonical_invoice_id": (
        prototype_canonical_id
    ),
    "template_id": str(
        restored_document["template_id"]
    ),
    "language": str(
        restored_document["language"]
    ),
    "currency": str(
        restored_document["currency"]
    ),
    "vendor": {
        "name": annotation_lookup[
            "vendor.name"
        ].text,
    },
    "buyer": {
        "name": annotation_lookup[
            "buyer.name"
        ].text,
    },
    "metadata": {
        "invoice_number": {
            "display_value": (
                annotation_lookup[
                    "invoice_number"
                ].text
            ),
        },
        "invoice_date": {
            "display_value": (
                annotation_lookup[
                    "invoice_date"
                ].text
            ),
        },
        "due_date": {
            "display_value": (
                annotation_lookup[
                    "due_date"
                ].text
            ),
        },
    },
    "financials": {
        "subtotal": {
            "display_value": (
                annotation_lookup[
                    "financials.subtotal"
                ].text
            ),
        },
        "tax": {
            "display_value": (
                annotation_lookup[
                    "financials.tax"
                ].text
            ),
        },
        "discount": {
            "display_value": (
                annotation_lookup[
                    "financials.discount"
                ].text
            ),
        },
        "total": {
            "display_value": (
                annotation_lookup[
                    "financials.total"
                ].text
            ),
        },
    },
}


# ================================================================
# Validasi hubungan PDF, ground truth, dan state
# ================================================================

with pymupdf.open(
    str(prototype_pdf_path)
) as restored_pdf:
    restored_page_count = (
        restored_pdf.page_count
    )

    if restored_page_count != 1:
        raise RuntimeError(
            "Prototype PDF harus memiliki tepat "
            "satu halaman."
        )

    restored_page = restored_pdf[0]
    restored_page_rectangle = (
        restored_page.rect
    )

    normalization_errors = []

    for annotation in (
        prototype_annotations
    ):
        annotation_rectangle = (
            pymupdf.Rect(
                *annotation.bbox_points
            )
        )

        recalculated_normalized = (
            normalize_rectangle(
                annotation_rectangle,
                restored_page_rectangle,
            )
        )

        maximum_difference = max(
            abs(
                stored_coordinate
                - calculated_coordinate
            )
            for stored_coordinate,
            calculated_coordinate
            in zip(
                annotation.bbox_normalized,
                recalculated_normalized,
            )
        )

        if maximum_difference > 1e-5:
            normalization_errors.append(
                annotation.field_name
            )

if normalization_errors:
    raise RuntimeError(
        "Koordinat normalisasi tidak konsisten: "
        f"{normalization_errors}"
    )


# ================================================================
# Ringkasan pemulihan
# ================================================================

restore_audit = pd.DataFrame(
    [
        {
            "control": "ground_truth_loaded",
            "actual": (
                prototype_ground_truth_path.is_file()
            ),
            "status": "VALID",
        },
        {
            "control": "pdf_page_count",
            "actual": restored_page_count,
            "status": (
                "VALID"
                if restored_page_count == 1
                else "INVALID"
            ),
        },
        {
            "control": "annotation_count",
            "actual": len(
                prototype_annotations
            ),
            "status": (
                "VALID"
                if len(
                    prototype_annotations
                ) == 27
                else "INVALID"
            ),
        },
        {
            "control": "unique_annotation_names",
            "actual": len(
                annotation_lookup
            ),
            "status": (
                "VALID"
                if len(annotation_lookup)
                == len(prototype_annotations)
                else "INVALID"
            ),
        },
        {
            "control": "normalization_errors",
            "actual": len(
                normalization_errors
            ),
            "status": (
                "VALID"
                if not normalization_errors
                else "INVALID"
            ),
        },
    ]
)

invalid_restore_controls = (
    restore_audit.loc[
        restore_audit["status"]
        != "VALID",
        "control",
    ].tolist()
)

display(restore_audit)

print(
    f"Document ID       : "
    f"{prototype_render_payload['document_id']}"
)
print(
    f"Canonical ID      : "
    f"{prototype_canonical_id}"
)
print(
    f"Template          : "
    f"{prototype_render_payload['template_id']}"
)
print(
    f"Annotations       : "
    f"{len(prototype_annotations)}"
)
print(
    f"Renderer version  : "
    f"{prototype_rendering_metadata['renderer_version']}"
)

if invalid_restore_controls:
    raise RuntimeError(
        "Pemulihan state prototype gagal pada: "
        f"{invalid_restore_controls}"
    )

print()
print(
    "✅ State prototype Cell 36 berhasil "
    "dipulihkan. Siap menjalankan Cell 37."
)

**Cell 37 — Code: Audit kualitas prototype TPL-01**

In [ ]:
from pathlib import Path
import hashlib
import json
import os


PROTOTYPE_QA_REPORT_PATH = (
    PROTOTYPE_ROOT
    / "INV-SYN-000001_TPL-01_qa_report.json"
)


def calculate_file_sha256(
    file_path: Path,
) -> str:
    """Menghitung checksum SHA-256 sebuah file."""

    sha256 = hashlib.sha256()

    with Path(file_path).open(
        mode="rb"
    ) as source_file:
        for chunk in iter(
            lambda: source_file.read(
                1024 * 1024
            ),
            b"",
        ):
            sha256.update(chunk)

    return sha256.hexdigest()


# ================================================================
# Memastikan artifact tersedia
# ================================================================

required_prototype_files = {
    "pdf": prototype_pdf_path,
    "preview": prototype_png_path,
    "ground_truth": (
        prototype_ground_truth_path
    ),
}

missing_prototype_files = [
    file_name
    for file_name, file_path
    in required_prototype_files.items()
    if not Path(file_path).is_file()
]

if missing_prototype_files:
    raise FileNotFoundError(
        "Artifact prototype tidak lengkap: "
        f"{missing_prototype_files}"
    )


# ================================================================
# Membaca ground truth
# ================================================================

with prototype_ground_truth_path.open(
    mode="r",
    encoding="utf-8",
) as ground_truth_file:
    loaded_ground_truth = json.load(
        ground_truth_file
    )

ground_truth_annotations = (
    loaded_ground_truth["annotations"]
)

if not ground_truth_annotations:
    raise RuntimeError(
        "Ground truth tidak memiliki anotasi."
    )


# ================================================================
# Audit PDF dan bounding box
# ================================================================

invalid_bbox_fields = []
normalization_mismatch_fields = []
annotation_without_words = []
severe_overlap_pairs = []

with pymupdf.open(
    str(prototype_pdf_path)
) as audit_document:
    pdf_page_count = (
        audit_document.page_count
    )

    if pdf_page_count != 1:
        raise RuntimeError(
            "Prototype harus memiliki satu halaman."
        )

    audit_page = audit_document[0]
    page_rectangle = audit_page.rect

    extracted_text = audit_page.get_text(
        "text"
    )

    extracted_words = audit_page.get_text(
        "words"
    )

    for annotation in ground_truth_annotations:
        field_name = annotation[
            "field_name"
        ]

        bbox_points = annotation[
            "bbox_points"
        ]

        bbox_normalized = annotation[
            "bbox_normalized"
        ]

        if (
            not isinstance(bbox_points, list)
            or len(bbox_points) != 4
        ):
            invalid_bbox_fields.append(
                field_name
            )
            continue

        annotation_rectangle = pymupdf.Rect(
            *bbox_points
        )

        bbox_inside_page = (
            not annotation_rectangle.is_empty
            and not annotation_rectangle.is_infinite
            and annotation_rectangle.x0
            >= page_rectangle.x0
            and annotation_rectangle.y0
            >= page_rectangle.y0
            and annotation_rectangle.x1
            <= page_rectangle.x1
            and annotation_rectangle.y1
            <= page_rectangle.y1
        )

        if not bbox_inside_page:
            invalid_bbox_fields.append(
                field_name
            )
            continue

        recalculated_normalized = (
            normalize_rectangle(
                annotation_rectangle,
                page_rectangle,
            )
        )

        normalization_difference = max(
            abs(
                float(stored_coordinate)
                - float(recalculated_coordinate)
            )
            for (
                stored_coordinate,
                recalculated_coordinate,
            ) in zip(
                bbox_normalized,
                recalculated_normalized,
            )
        )

        if normalization_difference > 1e-5:
            normalization_mismatch_fields.append(
                field_name
            )

        words_inside_bbox = []

        for word in extracted_words:
            (
                word_x0,
                word_y0,
                word_x1,
                word_y1,
                word_text,
                *_,
            ) = word

            word_center = pymupdf.Point(
                (
                    word_x0 + word_x1
                ) / 2,
                (
                    word_y0 + word_y1
                ) / 2,
            )

            if annotation_rectangle.contains(
                word_center
            ):
                words_inside_bbox.append(
                    word_text
                )

        if not words_inside_bbox:
            annotation_without_words.append(
                field_name
            )

    # --------------------------------------------------------------
    # Mendeteksi overlap berat antar-anotasi
    # --------------------------------------------------------------

    for first_index in range(
        len(ground_truth_annotations)
    ):
        first_annotation = (
            ground_truth_annotations[
                first_index
            ]
        )

        first_rectangle = pymupdf.Rect(
            *first_annotation[
                "bbox_points"
            ]
        )

        for second_index in range(
            first_index + 1,
            len(ground_truth_annotations),
        ):
            second_annotation = (
                ground_truth_annotations[
                    second_index
                ]
            )

            second_rectangle = pymupdf.Rect(
                *second_annotation[
                    "bbox_points"
                ]
            )

            intersection = (
                first_rectangle
                & second_rectangle
            )

            if intersection.is_empty:
                continue

            smaller_area = min(
                first_rectangle.get_area(),
                second_rectangle.get_area(),
            )

            if smaller_area <= 0:
                continue

            overlap_ratio = (
                intersection.get_area()
                / smaller_area
            )

            if overlap_ratio > 0.10:
                severe_overlap_pairs.append(
                    {
                        "first_field": (
                            first_annotation[
                                "field_name"
                            ]
                        ),
                        "second_field": (
                            second_annotation[
                                "field_name"
                            ]
                        ),
                        "overlap_ratio": round(
                            overlap_ratio,
                            4,
                        ),
                    }
                )


# ================================================================
# Audit raster preview
# ================================================================

prototype_grayscale = cv2.imread(
    str(prototype_png_path),
    cv2.IMREAD_GRAYSCALE,
)

if prototype_grayscale is None:
    raise RuntimeError(
        "Preview PNG tidak dapat dibaca OpenCV."
    )

image_height, image_width = (
    prototype_grayscale.shape
)

ink_pixel_ratio = float(
    np.mean(
        prototype_grayscale < 245
    )
)

white_pixel_ratio = float(
    np.mean(
        prototype_grayscale >= 250
    )
)

grayscale_standard_deviation = float(
    np.std(prototype_grayscale)
)

raster_content_valid = (
    0.01 <= ink_pixel_ratio <= 0.40
)

raster_contrast_valid = (
    grayscale_standard_deviation >= 10.0
)


# ================================================================
# Integrity controls
# ================================================================

annotation_field_names = [
    annotation["field_name"]
    for annotation
    in ground_truth_annotations
]

annotation_names_unique = (
    len(annotation_field_names)
    == len(set(annotation_field_names))
)

annotation_count_matches = (
    len(ground_truth_annotations)
    == len(prototype_annotations)
)

required_extracted_values = [
    prototype_render_payload[
        "metadata"
    ]["invoice_number"]["display_value"],
    prototype_render_payload[
        "vendor"
    ]["name"],
    prototype_render_payload[
        "buyer"
    ]["name"],
    prototype_render_payload[
        "financials"
    ]["total"]["display_value"],
]

missing_extracted_values = [
    expected_value
    for expected_value
    in required_extracted_values
    if expected_value not in extracted_text
]

qa_controls = pd.DataFrame(
    [
        {
            "control": "pdf_page_count",
            "expected": 1,
            "actual": pdf_page_count,
            "status": (
                "VALID"
                if pdf_page_count == 1
                else "INVALID"
            ),
        },
        {
            "control": "annotation_count_match",
            "expected": len(
                prototype_annotations
            ),
            "actual": len(
                ground_truth_annotations
            ),
            "status": (
                "VALID"
                if annotation_count_matches
                else "INVALID"
            ),
        },
        {
            "control": "unique_annotation_names",
            "expected": len(
                ground_truth_annotations
            ),
            "actual": len(
                set(annotation_field_names)
            ),
            "status": (
                "VALID"
                if annotation_names_unique
                else "INVALID"
            ),
        },
        {
            "control": "invalid_bounding_boxes",
            "expected": 0,
            "actual": len(
                invalid_bbox_fields
            ),
            "status": (
                "VALID"
                if not invalid_bbox_fields
                else "INVALID"
            ),
        },
        {
            "control": "normalization_mismatches",
            "expected": 0,
            "actual": len(
                normalization_mismatch_fields
            ),
            "status": (
                "VALID"
                if not normalization_mismatch_fields
                else "INVALID"
            ),
        },
        {
            "control": "annotations_without_words",
            "expected": 0,
            "actual": len(
                annotation_without_words
            ),
            "status": (
                "VALID"
                if not annotation_without_words
                else "INVALID"
            ),
        },
        {
            "control": "severe_annotation_overlaps",
            "expected": 0,
            "actual": len(
                severe_overlap_pairs
            ),
            "status": (
                "VALID"
                if not severe_overlap_pairs
                else "INVALID"
            ),
        },
        {
            "control": "missing_extracted_values",
            "expected": 0,
            "actual": len(
                missing_extracted_values
            ),
            "status": (
                "VALID"
                if not missing_extracted_values
                else "INVALID"
            ),
        },
        {
            "control": "raster_content_ratio",
            "expected": "0.01–0.40",
            "actual": round(
                ink_pixel_ratio,
                4,
            ),
            "status": (
                "VALID"
                if raster_content_valid
                else "INVALID"
            ),
        },
        {
            "control": "raster_contrast",
            "expected": ">=10.0",
            "actual": round(
                grayscale_standard_deviation,
                4,
            ),
            "status": (
                "VALID"
                if raster_contrast_valid
                else "INVALID"
            ),
        },
    ]
)

invalid_qa_controls = qa_controls.loc[
    qa_controls["status"] != "VALID",
    "control",
].tolist()


# ================================================================
# Checksums dan QA report
# ================================================================

artifact_checksums = {
    artifact_name: calculate_file_sha256(
        artifact_path
    )
    for artifact_name, artifact_path
    in required_prototype_files.items()
}

prototype_qa_report = {
    "schema_version": "1.0.0",
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "document_id": (
        prototype_render_payload[
            "document_id"
        ]
    ),
    "canonical_invoice_id": (
        prototype_canonical_id
    ),
    "template_id": "TPL-01",
    "status": (
        "PASSED"
        if not invalid_qa_controls
        else "FAILED"
    ),
    "metrics": {
        "page_count": pdf_page_count,
        "annotation_count": len(
            ground_truth_annotations
        ),
        "invalid_bbox_count": len(
            invalid_bbox_fields
        ),
        "normalization_mismatch_count": len(
            normalization_mismatch_fields
        ),
        "annotation_without_words_count": len(
            annotation_without_words
        ),
        "severe_overlap_count": len(
            severe_overlap_pairs
        ),
        "ink_pixel_ratio": round(
            ink_pixel_ratio,
            6,
        ),
        "white_pixel_ratio": round(
            white_pixel_ratio,
            6,
        ),
        "grayscale_standard_deviation": round(
            grayscale_standard_deviation,
            6,
        ),
        "image_width_pixels": int(
            image_width
        ),
        "image_height_pixels": int(
            image_height
        ),
    },
    "checksums_sha256": (
        artifact_checksums
    ),
    "failures": {
        "invalid_bbox_fields": (
            invalid_bbox_fields
        ),
        "normalization_mismatch_fields": (
            normalization_mismatch_fields
        ),
        "annotation_without_words": (
            annotation_without_words
        ),
        "severe_overlap_pairs": (
            severe_overlap_pairs
        ),
        "missing_extracted_values": (
            missing_extracted_values
        ),
    },
}

temporary_qa_report_path = (
    PROTOTYPE_QA_REPORT_PATH.parent
    / f".{PROTOTYPE_QA_REPORT_PATH.name}.tmp"
)

with temporary_qa_report_path.open(
    mode="w",
    encoding="utf-8",
    newline="\n",
) as qa_report_file:
    json.dump(
        prototype_qa_report,
        qa_report_file,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )

    qa_report_file.write("\n")

os.replace(
    temporary_qa_report_path,
    PROTOTYPE_QA_REPORT_PATH,
)


# ================================================================
# Output
# ================================================================

display(qa_controls)

print(
    f"QA report         : "
    f"{PROTOTYPE_QA_REPORT_PATH}"
)
print(
    f"Image size        : "
    f"{image_width} × {image_height}"
)
print(
    f"Ink pixel ratio   : "
    f"{ink_pixel_ratio:.4f}"
)
print(
    f"White pixel ratio : "
    f"{white_pixel_ratio:.4f}"
)
print(
    f"Raster contrast   : "
    f"{grayscale_standard_deviation:.4f}"
)
print(
    f"QA status         : "
    f"{prototype_qa_report['status']}"
)

if invalid_qa_controls:
    raise RuntimeError(
        "Prototype quality audit gagal pada "
        f"kontrol: {invalid_qa_controls}"
    )

print()
print(
    "✅ Prototype TPL-01 lulus audit geometri, "
    "teks, raster, dan integritas artifact."
)

**Cell 38 — Code: Menyimpan build checkpoint dan resume pointe**

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os


# ================================================================
# Lokasi checkpoint
# ================================================================

BUILD_MANIFESTS_ROOT = (
    BUILD_RUN_ROOT
    / "manifests"
)

BUILD_MANIFESTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

BUILD_STATE_PATH = (
    BUILD_MANIFESTS_ROOT
    / "build_state.json"
)

ACTIVE_BUILD_POINTER_PATH = (
    BUILD_BASE_ROOT
    / "active_build.json"
)


# ================================================================
# Atomic JSON writer
# ================================================================

def write_json_atomically(
    destination_path: Path,
    payload: dict,
) -> None:
    """Menulis JSON melalui temporary file."""

    destination_path = Path(
        destination_path
    )

    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        destination_path.parent
        / f".{destination_path.name}.tmp"
    )

    try:
        with temporary_path.open(
            mode="w",
            encoding="utf-8",
            newline="\n",
        ) as destination_file:
            json.dump(
                payload,
                destination_file,
                ensure_ascii=False,
                indent=2,
                allow_nan=False,
            )

            destination_file.write("\n")

        os.replace(
            temporary_path,
            destination_path,
        )

    except Exception:
        if temporary_path.exists():
            temporary_path.unlink()

        raise


def checkpoint_sha256(
    file_path: Path,
) -> str:
    """Menghitung SHA-256 artifact checkpoint."""

    digest = hashlib.sha256()

    with Path(file_path).open(
        mode="rb"
    ) as source_file:
        for chunk in iter(
            lambda: source_file.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ================================================================
# Memvalidasi QA report sebelum checkpoint
# ================================================================

if not PROTOTYPE_QA_REPORT_PATH.is_file():
    raise FileNotFoundError(
        "Prototype QA report tidak ditemukan: "
        f"{PROTOTYPE_QA_REPORT_PATH}"
    )

with PROTOTYPE_QA_REPORT_PATH.open(
    mode="r",
    encoding="utf-8",
) as qa_report_file:
    validated_qa_report = json.load(
        qa_report_file
    )

if (
    validated_qa_report.get("status")
    != "PASSED"
):
    raise RuntimeError(
        "Build tidak boleh diberi checkpoint "
        "karena prototype QA belum PASSED."
    )

if (
    validated_qa_report.get("template_id")
    != "TPL-01"
):
    raise RuntimeError(
        "QA report bukan milik TPL-01."
    )


# ================================================================
# Artifact yang menjadi bagian checkpoint
# ================================================================

checkpoint_artifacts = {
    "dataset_configuration": (
        DATASET_CONFIGURATION_PATH
    ),
    "generation_plan": (
        GENERATION_PLAN_PATH
    ),
    "prototype_pdf": (
        prototype_pdf_path
    ),
    "prototype_preview": (
        prototype_png_path
    ),
    "prototype_ground_truth": (
        prototype_ground_truth_path
    ),
    "prototype_qa_report": (
        PROTOTYPE_QA_REPORT_PATH
    ),
}

missing_checkpoint_artifacts = [
    artifact_name
    for artifact_name, artifact_path
    in checkpoint_artifacts.items()
    if (
        not Path(artifact_path).is_file()
        or Path(artifact_path).stat().st_size == 0
    )
]

if missing_checkpoint_artifacts:
    raise FileNotFoundError(
        "Artifact checkpoint tidak lengkap: "
        f"{missing_checkpoint_artifacts}"
    )

artifact_manifest = {}

for (
    artifact_name,
    artifact_path,
) in checkpoint_artifacts.items():
    artifact_path = Path(
        artifact_path
    )

    artifact_manifest[
        artifact_name
    ] = {
        "relative_path": str(
            artifact_path.relative_to(
                PROJECT_DATA_ROOT
            )
        ),
        "size_bytes": int(
            artifact_path.stat().st_size
        ),
        "sha256": checkpoint_sha256(
            artifact_path
        ),
    }


# ================================================================
# Menulis build state
# ================================================================

checkpoint_timestamp = (
    datetime.now(
        timezone.utc
    ).isoformat()
)

build_state = {
    "schema_version": "1.0.0",
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "dataset_version": "1.0.0",
    "build_id": BUILD_RUN_ROOT.name,
    "build_status": (
        "PROTOTYPE_TPL_01_VALIDATED"
    ),
    "publication_status": "NOT_PUBLISHED",
    "last_completed_notebook": (
        "01_data_collection_and_audit"
    ),
    "last_completed_cell": 37,
    "last_completed_stage": (
        "prototype_tpl_01_quality_audit"
    ),
    "next_stage": (
        "template_renderer_expansion"
    ),
    "updated_utc": checkpoint_timestamp,
    "quality_gates": {
        "generation_plan": "PASSED",
        "canonical_records": "PASSED",
        "payload_schema": "PASSED",
        "presentation_payload": "PASSED",
        "prototype_tpl_01_render": "PASSED",
        "prototype_tpl_01_qa": "PASSED",
    },
    "prototype": {
        "document_id": (
            validated_qa_report[
                "document_id"
            ]
        ),
        "canonical_invoice_id": (
            validated_qa_report[
                "canonical_invoice_id"
            ]
        ),
        "template_id": (
            validated_qa_report[
                "template_id"
            ]
        ),
        "qa_status": (
            validated_qa_report["status"]
        ),
    },
    "artifacts": artifact_manifest,
}

write_json_atomically(
    BUILD_STATE_PATH,
    build_state,
)

build_state_checksum = (
    checkpoint_sha256(
        BUILD_STATE_PATH
    )
)


# ================================================================
# Menulis pointer ke active build
# ================================================================

active_build_pointer = {
    "schema_version": "1.0.0",
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "active_build_id": (
        BUILD_RUN_ROOT.name
    ),
    "active_build_relative_path": str(
        BUILD_RUN_ROOT.relative_to(
            PROJECT_DATA_ROOT
        )
    ),
    "build_state_relative_path": str(
        BUILD_STATE_PATH.relative_to(
            PROJECT_DATA_ROOT
        )
    ),
    "build_state_sha256": (
        build_state_checksum
    ),
    "build_status": (
        build_state["build_status"]
    ),
    "publication_status": (
        build_state["publication_status"]
    ),
    "updated_utc": checkpoint_timestamp,
}

write_json_atomically(
    ACTIVE_BUILD_POINTER_PATH,
    active_build_pointer,
)


# ================================================================
# Membuka ulang dan memverifikasi checkpoint
# ================================================================

with BUILD_STATE_PATH.open(
    mode="r",
    encoding="utf-8",
) as build_state_file:
    verified_build_state = json.load(
        build_state_file
    )

with ACTIVE_BUILD_POINTER_PATH.open(
    mode="r",
    encoding="utf-8",
) as pointer_file:
    verified_active_pointer = json.load(
        pointer_file
    )

verified_build_state_checksum = (
    checkpoint_sha256(
        BUILD_STATE_PATH
    )
)

checkpoint_valid = (
    verified_build_state[
        "build_id"
    ]
    == BUILD_RUN_ROOT.name
    and verified_build_state[
        "build_status"
    ]
    == "PROTOTYPE_TPL_01_VALIDATED"
    and verified_active_pointer[
        "active_build_id"
    ]
    == BUILD_RUN_ROOT.name
    and verified_active_pointer[
        "build_state_sha256"
    ]
    == verified_build_state_checksum
)

if not checkpoint_valid:
    raise RuntimeError(
        "Verifikasi build checkpoint gagal."
    )


# ================================================================
# Output
# ================================================================

checkpoint_summary = pd.DataFrame(
    [
        {
            "property": "active_build_id",
            "value": BUILD_RUN_ROOT.name,
            "status": "VALID",
        },
        {
            "property": "build_status",
            "value": verified_build_state[
                "build_status"
            ],
            "status": "VALID",
        },
        {
            "property": "publication_status",
            "value": verified_build_state[
                "publication_status"
            ],
            "status": "VALID",
        },
        {
            "property": "prototype_qa",
            "value": verified_build_state[
                "quality_gates"
            ]["prototype_tpl_01_qa"],
            "status": "VALID",
        },
        {
            "property": "artifact_count",
            "value": len(
                artifact_manifest
            ),
            "status": "VALID",
        },
        {
            "property": "next_stage",
            "value": verified_build_state[
                "next_stage"
            ],
            "status": "VALID",
        },
    ]
)

display(checkpoint_summary)

print(
    f"Build state   : {BUILD_STATE_PATH}"
)
print(
    f"Active pointer: "
    f"{ACTIVE_BUILD_POINTER_PATH}"
)
print(
    f"State SHA-256 : "
    f"{verified_build_state_checksum}"
)
print()
print(
    "✅ Build checkpoint berhasil disimpan. "
    "Proses dapat dilanjutkan setelah runtime restart."
)

Cell 39 — Code: Menyimpan template registry sebagai **artifact**

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import re


# ================================================================
# Memuat generation plan dari checkpoint
# ================================================================

generation_plan = pd.read_csv(
    GENERATION_PLAN_PATH,
    dtype={
        "document_id": "string",
        "canonical_invoice_id": "string",
        "split": "string",
        "template_id": "string",
        "language": "string",
        "currency": "string",
        "file_format": "string",
        "quality_profile": "string",
        "document_seed_token": "string",
        "document_relative_path": "string",
        "ground_truth_relative_path": "string",
    },
)

if len(generation_plan) != 200:
    raise RuntimeError(
        "Generation plan checkpoint tidak valid. "
        f"Expected: 200; actual: {len(generation_plan)}"
    )


# ================================================================
# Registry desain sebagai data serializable
# ================================================================

TEMPLATE_REGISTRY_DATA = {
    "TPL-01": {
        "template_id": "TPL-01",
        "split": "development",
        "layout_family": "classic_corporate",
        "page_size": "A4",
        "header_layout": "full_width_band",
        "party_layout": "two_columns",
        "table_style": "full_grid",
        "totals_position": "bottom_right",
        "accent_position": "top",
        "density": "regular",
        "primary_color": "#16324F",
        "accent_color": "#2F80ED",
    },
    "TPL-02": {
        "template_id": "TPL-02",
        "split": "development",
        "layout_family": "modern_minimal",
        "page_size": "A4",
        "header_layout": "open_left",
        "party_layout": "stacked",
        "table_style": "horizontal_rules",
        "totals_position": "bottom_right",
        "accent_position": "left",
        "density": "spacious",
        "primary_color": "#243B53",
        "accent_color": "#00A896",
    },
    "TPL-03": {
        "template_id": "TPL-03",
        "split": "development",
        "layout_family": "split_header",
        "page_size": "LETTER",
        "header_layout": "split_title_metadata",
        "party_layout": "two_columns",
        "table_style": "striped",
        "totals_position": "bottom_card",
        "accent_position": "header",
        "density": "regular",
        "primary_color": "#3D405B",
        "accent_color": "#E07A5F",
    },
    "TPL-04": {
        "template_id": "TPL-04",
        "split": "development",
        "layout_family": "left_sidebar",
        "page_size": "A4",
        "header_layout": "sidebar_identity",
        "party_layout": "vendor_sidebar",
        "table_style": "minimal",
        "totals_position": "bottom_right",
        "accent_position": "left",
        "density": "compact",
        "primary_color": "#264653",
        "accent_color": "#2A9D8F",
    },
    "TPL-05": {
        "template_id": "TPL-05",
        "split": "development",
        "layout_family": "centered_editorial",
        "page_size": "A4",
        "header_layout": "centered_title",
        "party_layout": "horizontal_cards",
        "table_style": "open_rows",
        "totals_position": "full_width",
        "accent_position": "bottom",
        "density": "spacious",
        "primary_color": "#4A4E69",
        "accent_color": "#9A8C98",
    },
    "TPL-06": {
        "template_id": "TPL-06",
        "split": "development",
        "layout_family": "boxed_enterprise",
        "page_size": "LETTER",
        "header_layout": "boxed_right",
        "party_layout": "two_columns_boxed",
        "table_style": "full_grid",
        "totals_position": "side_panel",
        "accent_position": "right",
        "density": "compact",
        "primary_color": "#1D3557",
        "accent_color": "#457B9D",
    },
    "TPL-07": {
        "template_id": "TPL-07",
        "split": "validation",
        "layout_family": "top_stripe",
        "page_size": "A4",
        "header_layout": "title_with_top_stripe",
        "party_layout": "stacked_split",
        "table_style": "striped",
        "totals_position": "bottom_card",
        "accent_position": "top",
        "density": "regular",
        "primary_color": "#283618",
        "accent_color": "#DDA15E",
    },
    "TPL-08": {
        "template_id": "TPL-08",
        "split": "validation",
        "layout_family": "modular_cards",
        "page_size": "LETTER",
        "header_layout": "metadata_cards",
        "party_layout": "horizontal_cards",
        "table_style": "boxed_rows",
        "totals_position": "side_panel",
        "accent_position": "header",
        "density": "spacious",
        "primary_color": "#003049",
        "accent_color": "#F77F00",
    },
    "TPL-09": {
        "template_id": "TPL-09",
        "split": "test",
        "layout_family": "compact_ledger",
        "page_size": "A4",
        "header_layout": "compact_metadata",
        "party_layout": "two_columns",
        "table_style": "ledger",
        "totals_position": "inline",
        "accent_position": "bottom",
        "density": "compact",
        "primary_color": "#2B2D42",
        "accent_color": "#8D99AE",
    },
    "TPL-10": {
        "template_id": "TPL-10",
        "split": "test",
        "layout_family": "international_clean",
        "page_size": "LETTER",
        "header_layout": "open_right",
        "party_layout": "stacked",
        "table_style": "horizontal_rules",
        "totals_position": "full_width",
        "accent_position": "right",
        "density": "regular",
        "primary_color": "#14213D",
        "accent_color": "#FCA311",
    },
}


# ================================================================
# Validasi registry terhadap generation plan
# ================================================================

expected_template_ids = {
    f"TPL-{template_number:02d}"
    for template_number in range(1, 11)
}

if set(TEMPLATE_REGISTRY_DATA) != expected_template_ids:
    raise RuntimeError(
        "Template registry tidak memiliki "
        "TPL-01 sampai TPL-10 secara lengkap."
    )

template_registry_rows = []
design_fingerprints = set()

for template_id, specification in sorted(
    TEMPLATE_REGISTRY_DATA.items()
):
    template_plan = generation_plan.loc[
        generation_plan["template_id"]
        == template_id
    ]

    document_count = len(
        template_plan
    )

    plan_splits = sorted(
        template_plan[
            "split"
        ].dropna().unique().tolist()
    )

    expected_split = specification[
        "split"
    ]

    split_valid = (
        plan_splits == [expected_split]
    )

    document_count_valid = (
        document_count == 20
    )

    color_values_valid = all(
        re.fullmatch(
            r"#[0-9A-F]{6}",
            specification[color_field],
        )
        is not None
        for color_field in (
            "primary_color",
            "accent_color",
        )
    )

    design_fingerprint = (
        specification["page_size"],
        specification["header_layout"],
        specification["party_layout"],
        specification["table_style"],
        specification["totals_position"],
        specification["accent_position"],
        specification["density"],
    )

    if design_fingerprint in design_fingerprints:
        raise RuntimeError(
            "Fingerprint desain duplikat pada "
            f"{template_id}."
        )

    design_fingerprints.add(
        design_fingerprint
    )

    template_status = (
        "VALID"
        if (
            split_valid
            and document_count_valid
            and color_values_valid
        )
        else "INVALID"
    )

    template_registry_rows.append(
        {
            "template_id": template_id,
            "split": expected_split,
            "layout_family": specification[
                "layout_family"
            ],
            "page_size": specification[
                "page_size"
            ],
            "documents": document_count,
            "plan_split": ", ".join(
                plan_splits
            ),
            "status": template_status,
        }
    )

template_registry_audit = pd.DataFrame(
    template_registry_rows
)

invalid_templates = (
    template_registry_audit.loc[
        template_registry_audit["status"]
        != "VALID",
        "template_id",
    ].tolist()
)

if invalid_templates:
    display(template_registry_audit)

    raise RuntimeError(
        "Template registry gagal divalidasi: "
        f"{invalid_templates}"
    )


# ================================================================
# Menyimpan registry dan checksum
# ================================================================

TEMPLATE_REGISTRY_PATH = (
    BUILD_CONFIGURATION_ROOT
    / "template_registry.json"
)

TEMPLATE_REGISTRY_CHECKSUM_PATH = (
    BUILD_CONFIGURATION_ROOT
    / "template_registry.sha256"
)

template_registry_artifact = {
    "schema_version": "1.0.0",
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "dataset_version": "1.0.0",
    "registry_version": "1.0.0",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "template_count": len(
        TEMPLATE_REGISTRY_DATA
    ),
    "documents_per_template": 20,
    "validated_against": str(
        GENERATION_PLAN_PATH.relative_to(
            PROJECT_DATA_ROOT
        )
    ),
    "templates": [
        TEMPLATE_REGISTRY_DATA[
            template_id
        ]
        for template_id in sorted(
            TEMPLATE_REGISTRY_DATA
        )
    ],
}

write_json_atomically(
    TEMPLATE_REGISTRY_PATH,
    template_registry_artifact,
)

template_registry_checksum = (
    checkpoint_sha256(
        TEMPLATE_REGISTRY_PATH
    )
)

temporary_checksum_path = (
    TEMPLATE_REGISTRY_CHECKSUM_PATH.parent
    / (
        f".{TEMPLATE_REGISTRY_CHECKSUM_PATH.name}"
        ".tmp"
    )
)

with temporary_checksum_path.open(
    mode="w",
    encoding="utf-8",
    newline="\n",
) as checksum_file:
    checksum_file.write(
        f"{template_registry_checksum}  "
        f"{TEMPLATE_REGISTRY_PATH.name}\n"
    )

os.replace(
    temporary_checksum_path,
    TEMPLATE_REGISTRY_CHECKSUM_PATH,
)


# ================================================================
# Membuka ulang dan memverifikasi artifact
# ================================================================

with TEMPLATE_REGISTRY_PATH.open(
    mode="r",
    encoding="utf-8",
) as registry_file:
    verified_template_registry = (
        json.load(registry_file)
    )

verified_checksum = checkpoint_sha256(
    TEMPLATE_REGISTRY_PATH
)

registry_valid = (
    verified_template_registry[
        "template_count"
    ] == 10
    and len(
        verified_template_registry[
            "templates"
        ]
    ) == 10
    and verified_checksum
    == template_registry_checksum
)

if not registry_valid:
    raise RuntimeError(
        "Verifikasi template registry gagal."
    )


# ================================================================
# Output
# ================================================================

display(template_registry_audit)

print(
    f"Template registry : "
    f"{TEMPLATE_REGISTRY_PATH}"
)
print(
    f"Checksum file     : "
    f"{TEMPLATE_REGISTRY_CHECKSUM_PATH}"
)
print(
    f"SHA-256           : "
    f"{template_registry_checksum}"
)
print(
    f"Templates         : "
    f"{len(TEMPLATE_REGISTRY_DATA)}"
)
print(
    f"Covered documents : "
    f"{template_registry_audit['documents'].sum()}"
)
print()
print(
    "✅ Template registry berhasil disimpan "
    "dan diverifikasi."
)

**Cell 40 — Code: Menyimpan canonical payload secara permanen**

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os


# ================================================================
# Lokasi artifact canonical
# ================================================================

CANONICAL_RECORDS_JSONL_PATH = (
    BUILD_MANIFESTS_ROOT
    / "canonical_invoices.jsonl"
)

CANONICAL_AUDIT_CSV_PATH = (
    BUILD_MANIFESTS_ROOT
    / "canonical_invoice_audit.csv"
)

CANONICAL_MANIFEST_PATH = (
    BUILD_MANIFESTS_ROOT
    / "canonical_invoice_manifest.json"
)

CANONICAL_MANIFEST_CHECKSUM_PATH = (
    BUILD_MANIFESTS_ROOT
    / "canonical_invoice_manifest.sha256"
)


# ================================================================
# Validasi state sebelum penyimpanan
# ================================================================

if len(canonical_invoice_payloads) != 200:
    raise RuntimeError(
        "Canonical payload belum lengkap. "
        f"Expected: 200; "
        f"actual: {len(canonical_invoice_payloads)}"
    )

if len(canonical_audit) != 200:
    raise RuntimeError(
        "Canonical audit belum lengkap. "
        f"Expected: 200; "
        f"actual: {len(canonical_audit)}"
    )

expected_canonical_ids = {
    f"CANON-{record_number:06d}"
    for record_number in range(1, 201)
}

actual_canonical_ids = set(
    canonical_invoice_payloads
)

if actual_canonical_ids != expected_canonical_ids:
    raise RuntimeError(
        "Canonical invoice ID tidak lengkap "
        "atau tidak berurutan."
    )


# ================================================================
# Menulis JSONL secara atomik
# ================================================================

temporary_jsonl_path = (
    CANONICAL_RECORDS_JSONL_PATH.parent
    / f".{CANONICAL_RECORDS_JSONL_PATH.name}.tmp"
)

try:
    with temporary_jsonl_path.open(
        mode="w",
        encoding="utf-8",
        newline="\n",
    ) as jsonl_file:
        for canonical_id in sorted(
            canonical_invoice_payloads
        ):
            payload = (
                canonical_invoice_payloads[
                    canonical_id
                ]
            )

            json_line = json.dumps(
                payload,
                ensure_ascii=False,
                separators=(",", ":"),
                allow_nan=False,
            )

            jsonl_file.write(
                json_line + "\n"
            )

    os.replace(
        temporary_jsonl_path,
        CANONICAL_RECORDS_JSONL_PATH,
    )

except Exception:
    if temporary_jsonl_path.exists():
        temporary_jsonl_path.unlink()

    raise


# ================================================================
# Menulis canonical audit CSV secara atomik
# ================================================================

canonical_audit_to_save = (
    canonical_audit
    .sort_values(
        "generation_sequence"
    )
    .reset_index(drop=True)
)

temporary_audit_path = (
    CANONICAL_AUDIT_CSV_PATH.parent
    / f".{CANONICAL_AUDIT_CSV_PATH.name}.tmp"
)

try:
    canonical_audit_to_save.to_csv(
        temporary_audit_path,
        index=False,
        encoding="utf-8",
        lineterminator="\n",
    )

    os.replace(
        temporary_audit_path,
        CANONICAL_AUDIT_CSV_PATH,
    )

except Exception:
    if temporary_audit_path.exists():
        temporary_audit_path.unlink()

    raise


# ================================================================
# Checksums artifact utama
# ================================================================

canonical_jsonl_checksum = (
    checkpoint_sha256(
        CANONICAL_RECORDS_JSONL_PATH
    )
)

canonical_audit_checksum = (
    checkpoint_sha256(
        CANONICAL_AUDIT_CSV_PATH
    )
)


# ================================================================
# Membuat canonical manifest
# ================================================================

canonical_manifest = {
    "schema_version": "1.0.0",
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "dataset_version": "1.0.0",
    "build_id": BUILD_RUN_ROOT.name,
    "artifact_type": (
        "canonical_invoice_collection"
    ),
    "record_format": "JSON Lines",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "record_count": len(
        canonical_invoice_payloads
    ),
    "unique_document_count": int(
        canonical_audit[
            "document_id"
        ].nunique()
    ),
    "unique_invoice_number_count": int(
        canonical_audit[
            "invoice_number"
        ].nunique()
    ),
    "unique_vendor_count": int(
        canonical_audit[
            "vendor_id"
        ].nunique()
    ),
    "distributions": {
        "split": {
            str(key): int(value)
            for key, value
            in canonical_audit[
                "split"
            ].value_counts().sort_index().items()
        },
        "language": {
            str(key): int(value)
            for key, value
            in canonical_audit[
                "language"
            ].value_counts().sort_index().items()
        },
        "currency": {
            str(key): int(value)
            for key, value
            in canonical_audit[
                "currency"
            ].value_counts().sort_index().items()
        },
        "template": {
            str(key): int(value)
            for key, value
            in canonical_audit[
                "template_id"
            ].value_counts().sort_index().items()
        },
    },
    "artifacts": {
        "canonical_jsonl": {
            "relative_path": str(
                CANONICAL_RECORDS_JSONL_PATH.relative_to(
                    PROJECT_DATA_ROOT
                )
            ),
            "size_bytes": int(
                CANONICAL_RECORDS_JSONL_PATH
                .stat()
                .st_size
            ),
            "sha256": (
                canonical_jsonl_checksum
            ),
        },
        "canonical_audit_csv": {
            "relative_path": str(
                CANONICAL_AUDIT_CSV_PATH.relative_to(
                    PROJECT_DATA_ROOT
                )
            ),
            "size_bytes": int(
                CANONICAL_AUDIT_CSV_PATH
                .stat()
                .st_size
            ),
            "sha256": (
                canonical_audit_checksum
            ),
        },
    },
    "quality_status": "PASSED",
}

write_json_atomically(
    CANONICAL_MANIFEST_PATH,
    canonical_manifest,
)

canonical_manifest_checksum = (
    checkpoint_sha256(
        CANONICAL_MANIFEST_PATH
    )
)


# ================================================================
# Menulis checksum manifest
# ================================================================

temporary_manifest_checksum_path = (
    CANONICAL_MANIFEST_CHECKSUM_PATH.parent
    / (
        f".{CANONICAL_MANIFEST_CHECKSUM_PATH.name}"
        ".tmp"
    )
)

with temporary_manifest_checksum_path.open(
    mode="w",
    encoding="utf-8",
    newline="\n",
) as checksum_file:
    checksum_file.write(
        f"{canonical_manifest_checksum}  "
        f"{CANONICAL_MANIFEST_PATH.name}\n"
    )

os.replace(
    temporary_manifest_checksum_path,
    CANONICAL_MANIFEST_CHECKSUM_PATH,
)


# ================================================================
# Membuka ulang JSONL dan memverifikasi isinya
# ================================================================

reloaded_canonical_payloads = {}

with CANONICAL_RECORDS_JSONL_PATH.open(
    mode="r",
    encoding="utf-8",
) as jsonl_file:
    for line_number, line in enumerate(
        jsonl_file,
        start=1,
    ):
        clean_line = line.strip()

        if not clean_line:
            continue

        try:
            payload = json.loads(
                clean_line
            )
        except json.JSONDecodeError as error:
            raise RuntimeError(
                "JSONL tidak valid pada baris "
                f"{line_number}: {error}"
            ) from error

        canonical_id = str(
            payload[
                "canonical_invoice_id"
            ]
        )

        if canonical_id in (
            reloaded_canonical_payloads
        ):
            raise RuntimeError(
                "Canonical ID duplikat dalam JSONL: "
                f"{canonical_id}"
            )

        reloaded_canonical_payloads[
            canonical_id
        ] = payload

reloaded_audit = pd.read_csv(
    CANONICAL_AUDIT_CSV_PATH,
    dtype={
        "document_id": "string",
        "canonical_invoice_id": "string",
        "split": "string",
        "template_id": "string",
        "language": "string",
        "currency": "string",
        "vendor_id": "string",
        "buyer_id": "string",
        "invoice_number": "string",
    },
)

payload_count_valid = (
    len(reloaded_canonical_payloads) == 200
)

audit_count_valid = (
    len(reloaded_audit) == 200
)

payload_equality_valid = (
    reloaded_canonical_payloads
    == canonical_invoice_payloads
)

checksum_valid = (
    checkpoint_sha256(
        CANONICAL_RECORDS_JSONL_PATH
    )
    == canonical_jsonl_checksum
    and checkpoint_sha256(
        CANONICAL_AUDIT_CSV_PATH
    )
    == canonical_audit_checksum
    and checkpoint_sha256(
        CANONICAL_MANIFEST_PATH
    )
    == canonical_manifest_checksum
)


# ================================================================
# Memperbarui build state dan active pointer
# ================================================================

with BUILD_STATE_PATH.open(
    mode="r",
    encoding="utf-8",
) as build_state_file:
    updated_build_state = json.load(
        build_state_file
    )

persistent_artifacts = {
    "canonical_invoices_jsonl": (
        CANONICAL_RECORDS_JSONL_PATH
    ),
    "canonical_invoice_audit_csv": (
        CANONICAL_AUDIT_CSV_PATH
    ),
    "canonical_invoice_manifest": (
        CANONICAL_MANIFEST_PATH
    ),
    "canonical_invoice_manifest_checksum": (
        CANONICAL_MANIFEST_CHECKSUM_PATH
    ),
}

for (
    artifact_name,
    artifact_path,
) in persistent_artifacts.items():
    updated_build_state.setdefault(
        "artifacts",
        {},
    )[artifact_name] = {
        "relative_path": str(
            artifact_path.relative_to(
                PROJECT_DATA_ROOT
            )
        ),
        "size_bytes": int(
            artifact_path.stat().st_size
        ),
        "sha256": checkpoint_sha256(
            artifact_path
        ),
    }

updated_timestamp = datetime.now(
    timezone.utc
).isoformat()

updated_build_state[
    "build_status"
] = "CANONICAL_PAYLOADS_PERSISTED"

updated_build_state[
    "last_completed_cell"
] = 40

updated_build_state[
    "last_completed_stage"
] = "canonical_payload_persistence"

updated_build_state[
    "next_stage"
] = "presentation_payload_preparation"

updated_build_state[
    "updated_utc"
] = updated_timestamp

updated_build_state.setdefault(
    "quality_gates",
    {},
)[
    "canonical_payload_persistence"
] = "PASSED"

write_json_atomically(
    BUILD_STATE_PATH,
    updated_build_state,
)

updated_build_state_checksum = (
    checkpoint_sha256(
        BUILD_STATE_PATH
    )
)

with ACTIVE_BUILD_POINTER_PATH.open(
    mode="r",
    encoding="utf-8",
) as pointer_file:
    updated_active_pointer = json.load(
        pointer_file
    )

updated_active_pointer[
    "build_state_sha256"
] = updated_build_state_checksum

updated_active_pointer[
    "build_status"
] = (
    updated_build_state["build_status"]
)

updated_active_pointer[
    "updated_utc"
] = updated_timestamp

write_json_atomically(
    ACTIVE_BUILD_POINTER_PATH,
    updated_active_pointer,
)


# ================================================================
# Final validation
# ================================================================

persistence_controls = pd.DataFrame(
    [
        {
            "control": "jsonl_record_count",
            "expected": 200,
            "actual": len(
                reloaded_canonical_payloads
            ),
            "status": (
                "VALID"
                if payload_count_valid
                else "INVALID"
            ),
        },
        {
            "control": "audit_record_count",
            "expected": 200,
            "actual": len(
                reloaded_audit
            ),
            "status": (
                "VALID"
                if audit_count_valid
                else "INVALID"
            ),
        },
        {
            "control": "payload_round_trip",
            "expected": True,
            "actual": (
                payload_equality_valid
            ),
            "status": (
                "VALID"
                if payload_equality_valid
                else "INVALID"
            ),
        },
        {
            "control": "artifact_checksums",
            "expected": True,
            "actual": checksum_valid,
            "status": (
                "VALID"
                if checksum_valid
                else "INVALID"
            ),
        },
    ]
)

invalid_persistence_controls = (
    persistence_controls.loc[
        persistence_controls["status"]
        != "VALID",
        "control",
    ].tolist()
)

display(persistence_controls)

print(
    f"Canonical JSONL  : "
    f"{CANONICAL_RECORDS_JSONL_PATH}"
)
print(
    f"Canonical audit  : "
    f"{CANONICAL_AUDIT_CSV_PATH}"
)
print(
    f"Manifest         : "
    f"{CANONICAL_MANIFEST_PATH}"
)
print(
    f"Records persisted: "
    f"{len(reloaded_canonical_payloads)}"
)
print(
    f"JSONL SHA-256    : "
    f"{canonical_jsonl_checksum}"
)
print(
    f"Build status     : "
    f"{updated_build_state['build_status']}"
)

if invalid_persistence_controls:
    raise RuntimeError(
        "Canonical payload persistence gagal: "
        f"{invalid_persistence_controls}"
    )

print()
print(
    "✅ 200 canonical payload berhasil disimpan, "
    "dibuka ulang, dan diverifikasi."
)

**Cell 41 — Code: Menyimpan presentation payload**

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os


# ================================================================
# Lokasi artifact presentation payload
# ================================================================

PRESENTATION_JSONL_PATH = (
    BUILD_MANIFESTS_ROOT
    / "invoice_render_payloads.jsonl"
)

PRESENTATION_MANIFEST_PATH = (
    BUILD_MANIFESTS_ROOT
    / "invoice_render_payload_manifest.json"
)

PRESENTATION_MANIFEST_CHECKSUM_PATH = (
    BUILD_MANIFESTS_ROOT
    / "invoice_render_payload_manifest.sha256"
)


# ================================================================
# Validasi state sebelum penyimpanan
# ================================================================

if len(invoice_render_payloads) != 200:
    raise RuntimeError(
        "Presentation payload belum lengkap. "
        f"Expected: 200; "
        f"actual: {len(invoice_render_payloads)}"
    )

expected_render_ids = {
    f"CANON-{record_number:06d}"
    for record_number in range(1, 201)
}

actual_render_ids = set(
    invoice_render_payloads
)

if actual_render_ids != expected_render_ids:
    raise RuntimeError(
        "Canonical ID presentation payload "
        "tidak lengkap atau tidak berurutan."
    )


# ================================================================
# Menulis presentation JSONL secara atomik
# ================================================================

temporary_presentation_path = (
    PRESENTATION_JSONL_PATH.parent
    / f".{PRESENTATION_JSONL_PATH.name}.tmp"
)

try:
    with temporary_presentation_path.open(
        mode="w",
        encoding="utf-8",
        newline="\n",
    ) as presentation_file:
        for canonical_id in sorted(
            invoice_render_payloads
        ):
            render_payload = (
                invoice_render_payloads[
                    canonical_id
                ]
            )

            presentation_file.write(
                json.dumps(
                    render_payload,
                    ensure_ascii=False,
                    separators=(",", ":"),
                    allow_nan=False,
                )
                + "\n"
            )

    os.replace(
        temporary_presentation_path,
        PRESENTATION_JSONL_PATH,
    )

except Exception:
    if temporary_presentation_path.exists():
        temporary_presentation_path.unlink()

    raise


# ================================================================
# Checksum presentation JSONL
# ================================================================

presentation_jsonl_checksum = (
    checkpoint_sha256(
        PRESENTATION_JSONL_PATH
    )
)


# ================================================================
# Membuka ulang dan memverifikasi round-trip
# ================================================================

reloaded_render_payloads = {}

with PRESENTATION_JSONL_PATH.open(
    mode="r",
    encoding="utf-8",
) as presentation_file:
    for line_number, line in enumerate(
        presentation_file,
        start=1,
    ):
        clean_line = line.strip()

        if not clean_line:
            continue

        try:
            render_payload = json.loads(
                clean_line
            )
        except json.JSONDecodeError as error:
            raise RuntimeError(
                "Presentation JSONL tidak valid "
                f"pada baris {line_number}: {error}"
            ) from error

        canonical_id = str(
            render_payload[
                "canonical_invoice_id"
            ]
        )

        if canonical_id in reloaded_render_payloads:
            raise RuntimeError(
                "Canonical ID duplikat dalam "
                f"presentation JSONL: {canonical_id}"
            )

        reloaded_render_payloads[
            canonical_id
        ] = render_payload

render_count_valid = (
    len(reloaded_render_payloads) == 200
)

render_round_trip_valid = (
    reloaded_render_payloads
    == invoice_render_payloads
)

reloaded_schema_mismatch_ids = []

for canonical_id, render_payload in (
    reloaded_render_payloads.items()
):
    current_schema = (
        collect_schema_signature(
            render_payload
        )
    )

    if current_schema != reference_render_schema:
        reloaded_schema_mismatch_ids.append(
            canonical_id
        )

render_schema_valid = (
    len(reloaded_schema_mismatch_ids) == 0
)


# ================================================================
# Membuat manifest presentation payload
# ================================================================

presentation_manifest = {
    "schema_version": "1.0.0",
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "dataset_version": "1.0.0",
    "build_id": BUILD_RUN_ROOT.name,
    "artifact_type": (
        "invoice_render_payload_collection"
    ),
    "record_format": "JSON Lines",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "record_count": len(
        reloaded_render_payloads
    ),
    "schema_path_count": len(
        reference_render_schema
    ),
    "supported_languages": sorted(
        {
            payload["language"]
            for payload
            in reloaded_render_payloads.values()
        }
    ),
    "supported_currencies": sorted(
        {
            payload["currency"]
            for payload
            in reloaded_render_payloads.values()
        }
    ),
    "template_ids": sorted(
        {
            payload["template_id"]
            for payload
            in reloaded_render_payloads.values()
        }
    ),
    "safety_notice_required": True,
    "lineage": {
        "canonical_jsonl_relative_path": str(
            CANONICAL_RECORDS_JSONL_PATH.relative_to(
                PROJECT_DATA_ROOT
            )
        ),
        "canonical_jsonl_sha256": (
            checkpoint_sha256(
                CANONICAL_RECORDS_JSONL_PATH
            )
        ),
    },
    "artifact": {
        "relative_path": str(
            PRESENTATION_JSONL_PATH.relative_to(
                PROJECT_DATA_ROOT
            )
        ),
        "size_bytes": int(
            PRESENTATION_JSONL_PATH
            .stat()
            .st_size
        ),
        "sha256": (
            presentation_jsonl_checksum
        ),
    },
    "quality_controls": {
        "record_count": (
            "PASSED"
            if render_count_valid
            else "FAILED"
        ),
        "round_trip_equality": (
            "PASSED"
            if render_round_trip_valid
            else "FAILED"
        ),
        "schema_consistency": (
            "PASSED"
            if render_schema_valid
            else "FAILED"
        ),
    },
    "quality_status": (
        "PASSED"
        if (
            render_count_valid
            and render_round_trip_valid
            and render_schema_valid
        )
        else "FAILED"
    ),
}

write_json_atomically(
    PRESENTATION_MANIFEST_PATH,
    presentation_manifest,
)

presentation_manifest_checksum = (
    checkpoint_sha256(
        PRESENTATION_MANIFEST_PATH
    )
)


# ================================================================
# Menulis checksum manifest
# ================================================================

temporary_checksum_path = (
    PRESENTATION_MANIFEST_CHECKSUM_PATH.parent
    / (
        f".{PRESENTATION_MANIFEST_CHECKSUM_PATH.name}"
        ".tmp"
    )
)

with temporary_checksum_path.open(
    mode="w",
    encoding="utf-8",
    newline="\n",
) as checksum_file:
    checksum_file.write(
        f"{presentation_manifest_checksum}  "
        f"{PRESENTATION_MANIFEST_PATH.name}\n"
    )

os.replace(
    temporary_checksum_path,
    PRESENTATION_MANIFEST_CHECKSUM_PATH,
)


# ================================================================
# Memperbarui build state
# ================================================================

with BUILD_STATE_PATH.open(
    mode="r",
    encoding="utf-8",
) as build_state_file:
    updated_build_state = json.load(
        build_state_file
    )

presentation_artifacts = {
    "invoice_render_payloads_jsonl": (
        PRESENTATION_JSONL_PATH
    ),
    "invoice_render_payload_manifest": (
        PRESENTATION_MANIFEST_PATH
    ),
    "invoice_render_payload_manifest_checksum": (
        PRESENTATION_MANIFEST_CHECKSUM_PATH
    ),
}

for (
    artifact_name,
    artifact_path,
) in presentation_artifacts.items():
    updated_build_state.setdefault(
        "artifacts",
        {},
    )[artifact_name] = {
        "relative_path": str(
            artifact_path.relative_to(
                PROJECT_DATA_ROOT
            )
        ),
        "size_bytes": int(
            artifact_path.stat().st_size
        ),
        "sha256": checkpoint_sha256(
            artifact_path
        ),
    }

updated_timestamp = datetime.now(
    timezone.utc
).isoformat()

updated_build_state[
    "build_status"
] = "PRESENTATION_PAYLOADS_PERSISTED"

updated_build_state[
    "last_completed_cell"
] = 41

updated_build_state[
    "last_completed_stage"
] = "presentation_payload_persistence"

updated_build_state[
    "next_stage"
] = "template_renderer_expansion"

updated_build_state[
    "updated_utc"
] = updated_timestamp

updated_build_state.setdefault(
    "quality_gates",
    {},
)[
    "presentation_payload_persistence"
] = "PASSED"

write_json_atomically(
    BUILD_STATE_PATH,
    updated_build_state,
)

updated_build_state_checksum = (
    checkpoint_sha256(
        BUILD_STATE_PATH
    )
)


# ================================================================
# Memperbarui active build pointer
# ================================================================

with ACTIVE_BUILD_POINTER_PATH.open(
    mode="r",
    encoding="utf-8",
) as pointer_file:
    updated_active_pointer = json.load(
        pointer_file
    )

updated_active_pointer[
    "build_state_sha256"
] = updated_build_state_checksum

updated_active_pointer[
    "build_status"
] = updated_build_state[
    "build_status"
]

updated_active_pointer[
    "updated_utc"
] = updated_timestamp

write_json_atomically(
    ACTIVE_BUILD_POINTER_PATH,
    updated_active_pointer,
)


# ================================================================
# Final controls
# ================================================================

presentation_controls = pd.DataFrame(
    [
        {
            "control": "render_payload_count",
            "expected": 200,
            "actual": len(
                reloaded_render_payloads
            ),
            "status": (
                "VALID"
                if render_count_valid
                else "INVALID"
            ),
        },
        {
            "control": "round_trip_equality",
            "expected": True,
            "actual": (
                render_round_trip_valid
            ),
            "status": (
                "VALID"
                if render_round_trip_valid
                else "INVALID"
            ),
        },
        {
            "control": "schema_consistency",
            "expected": 0,
            "actual": len(
                reloaded_schema_mismatch_ids
            ),
            "status": (
                "VALID"
                if render_schema_valid
                else "INVALID"
            ),
        },
        {
            "control": "jsonl_checksum",
            "expected": (
                presentation_jsonl_checksum
            ),
            "actual": checkpoint_sha256(
                PRESENTATION_JSONL_PATH
            ),
            "status": (
                "VALID"
                if checkpoint_sha256(
                    PRESENTATION_JSONL_PATH
                )
                == presentation_jsonl_checksum
                else "INVALID"
            ),
        },
    ]
)

invalid_presentation_controls = (
    presentation_controls.loc[
        presentation_controls["status"]
        != "VALID",
        "control",
    ].tolist()
)

display(presentation_controls)

print(
    f"Presentation JSONL : "
    f"{PRESENTATION_JSONL_PATH}"
)
print(
    f"Manifest          : "
    f"{PRESENTATION_MANIFEST_PATH}"
)
print(
    f"Records persisted : "
    f"{len(reloaded_render_payloads)}"
)
print(
    f"Schema paths      : "
    f"{len(reference_render_schema)}"
)
print(
    f"JSONL SHA-256     : "
    f"{presentation_jsonl_checksum}"
)
print(
    f"Build status      : "
    f"{updated_build_state['build_status']}"
)

if invalid_presentation_controls:
    raise RuntimeError(
        "Presentation payload persistence gagal: "
        f"{invalid_presentation_controls}"
    )

print()
print(
    "✅ 200 presentation payload berhasil "
    "disimpan, dibuka ulang, dan diverifikasi."
)

**Cell 42 — Code: Memuat dan memvalidasi renderer foundation**

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import json
import re


RENDER_ENGINE_VERSION = "2.0.0"

PAGE_SIZE_POINTS = {
    "A4": (
        595.276,
        841.890,
    ),
    "LETTER": (
        612.000,
        792.000,
    ),
}

TEXT_ALIGNMENTS = {
    "left": pymupdf.TEXT_ALIGN_LEFT,
    "center": pymupdf.TEXT_ALIGN_CENTER,
    "right": pymupdf.TEXT_ALIGN_RIGHT,
    "justify": pymupdf.TEXT_ALIGN_JUSTIFY,
}


# ================================================================
# Kontrak anotasi renderer
# ================================================================

@dataclass(frozen=True)
class FieldAnnotation:
    field_name: str
    text: str
    page_number: int
    bbox_points: tuple[
        float,
        float,
        float,
        float,
    ]
    bbox_normalized: tuple[
        float,
        float,
        float,
        float,
    ]
    annotation_type: str = "field_region"


# ================================================================
# Helper geometri dan warna
# ================================================================

def hex_to_rgb(
    hex_color: str,
) -> tuple[float, float, float]:
    """Mengubah hexadecimal menjadi RGB 0–1."""

    if not re.fullmatch(
        r"#[0-9A-Fa-f]{6}",
        hex_color,
    ):
        raise ValueError(
            f"Warna tidak valid: {hex_color}"
        )

    return tuple(
        int(
            hex_color[index:index + 2],
            16,
        ) / 255.0
        for index in (
            1,
            3,
            5,
        )
    )


def get_page_dimensions(
    page_size: str,
) -> tuple[float, float]:
    """Mengambil ukuran halaman dalam point."""

    if page_size not in PAGE_SIZE_POINTS:
        raise ValueError(
            f"Page size tidak didukung: {page_size}"
        )

    return PAGE_SIZE_POINTS[
        page_size
    ]


def validate_rectangle(
    page: pymupdf.Page,
    rectangle: pymupdf.Rect,
) -> None:
    """Memastikan rectangle valid dan di dalam halaman."""

    if (
        rectangle.is_empty
        or rectangle.is_infinite
    ):
        raise ValueError(
            f"Rectangle tidak valid: {rectangle}"
        )

    page_rectangle = page.rect
    tolerance = 0.01

    rectangle_outside_page = (
        rectangle.x0
        < page_rectangle.x0 - tolerance
        or rectangle.y0
        < page_rectangle.y0 - tolerance
        or rectangle.x1
        > page_rectangle.x1 + tolerance
        or rectangle.y1
        > page_rectangle.y1 + tolerance
    )

    if rectangle_outside_page:
        raise ValueError(
            "Rectangle berada di luar halaman. "
            f"Rectangle: {rectangle}; "
            f"page: {page_rectangle}"
        )


def normalize_rectangle(
    rectangle: pymupdf.Rect,
    page_rectangle: pymupdf.Rect,
) -> tuple[float, float, float, float]:
    """Mengubah bbox point menjadi koordinat 0–1."""

    normalized_coordinates = (
        rectangle.x0 / page_rectangle.width,
        rectangle.y0 / page_rectangle.height,
        rectangle.x1 / page_rectangle.width,
        rectangle.y1 / page_rectangle.height,
    )

    if not all(
        0.0 <= coordinate <= 1.0
        for coordinate
        in normalized_coordinates
    ):
        raise ValueError(
            "Koordinat normalisasi di luar "
            f"rentang 0–1: {normalized_coordinates}"
        )

    return tuple(
        round(coordinate, 6)
        for coordinate
        in normalized_coordinates
    )


# ================================================================
# Helper text fitting
# ================================================================

def insert_textbox_with_fit(
    page: pymupdf.Page,
    rectangle: pymupdf.Rect,
    text: str,
    font_name: str = "helv",
    font_size: float = 9.0,
    minimum_font_size: float = 6.5,
    font_color: tuple[
        float,
        float,
        float,
    ] = (0.0, 0.0, 0.0),
    alignment: str = "left",
) -> float:
    """Menulis teks dengan font fitting terkontrol."""

    validate_rectangle(
        page,
        rectangle,
    )

    if alignment not in TEXT_ALIGNMENTS:
        raise ValueError(
            f"Alignment tidak didukung: {alignment}"
        )

    clean_text = str(text).strip()

    if not clean_text:
        raise ValueError(
            "Teks tidak boleh kosong."
        )

    current_font_size = float(
        font_size
    )

    while (
        current_font_size
        >= minimum_font_size
    ):
        remaining_height = page.insert_textbox(
            rectangle,
            clean_text,
            fontname=font_name,
            fontsize=current_font_size,
            color=font_color,
            align=TEXT_ALIGNMENTS[
                alignment
            ],
            overlay=True,
        )

        if remaining_height >= 0:
            return round(
                current_font_size,
                2,
            )

        current_font_size = round(
            current_font_size - 0.25,
            2,
        )

    raise RuntimeError(
        "Teks tidak muat pada batas font minimum. "
        f"Rectangle: {rectangle}; "
        f"text: {clean_text!r}"
    )


def insert_static_text(
    page: pymupdf.Page,
    rectangle: pymupdf.Rect,
    text: str,
    font_name: str = "helv",
    font_size: float = 8.0,
    minimum_font_size: float = 6.0,
    font_color: tuple[
        float,
        float,
        float,
    ] = (0.0, 0.0, 0.0),
    alignment: str = "left",
) -> float:
    """Menulis label dekoratif tanpa anotasi."""

    return insert_textbox_with_fit(
        page=page,
        rectangle=rectangle,
        text=text,
        font_name=font_name,
        font_size=font_size,
        minimum_font_size=minimum_font_size,
        font_color=font_color,
        alignment=alignment,
    )


def insert_annotated_text(
    page: pymupdf.Page,
    rectangle: pymupdf.Rect,
    text: str,
    field_name: str,
    font_name: str = "helv",
    font_size: float = 9.0,
    minimum_font_size: float = 6.5,
    font_color: tuple[
        float,
        float,
        float,
    ] = (0.0, 0.0, 0.0),
    alignment: str = "left",
) -> tuple[FieldAnnotation, float]:
    """Menulis nilai dan menghasilkan field-region bbox."""

    if not re.fullmatch(
        r"[a-z][a-z0-9_.\[\]-]*",
        field_name,
    ):
        raise ValueError(
            f"Nama field tidak valid: {field_name}"
        )

    clean_text = str(text).strip()

    used_font_size = insert_textbox_with_fit(
        page=page,
        rectangle=rectangle,
        text=clean_text,
        font_name=font_name,
        font_size=font_size,
        minimum_font_size=minimum_font_size,
        font_color=font_color,
        alignment=alignment,
    )

    bbox_points = tuple(
        round(coordinate, 3)
        for coordinate in (
            rectangle.x0,
            rectangle.y0,
            rectangle.x1,
            rectangle.y1,
        )
    )

    bbox_normalized = normalize_rectangle(
        rectangle,
        page.rect,
    )

    annotation = FieldAnnotation(
        field_name=field_name,
        text=clean_text,
        page_number=page.number + 1,
        bbox_points=bbox_points,
        bbox_normalized=bbox_normalized,
        annotation_type="field_region",
    )

    return (
        annotation,
        used_font_size,
    )


def draw_panel(
    page: pymupdf.Page,
    rectangle: pymupdf.Rect,
    border_color: tuple[
        float,
        float,
        float,
    ] | None = None,
    fill_color: tuple[
        float,
        float,
        float,
    ] | None = None,
    border_width: float = 0.8,
) -> None:
    """Menggambar panel visual."""

    validate_rectangle(
        page,
        rectangle,
    )

    page.draw_rect(
        rectangle,
        color=border_color,
        fill=fill_color,
        width=border_width,
        overlay=True,
    )


def annotation_to_dict(
    annotation: FieldAnnotation,
) -> dict:
    """Mengubah anotasi menjadi dictionary JSON."""

    return {
        "field_name": annotation.field_name,
        "text": annotation.text,
        "page_number": annotation.page_number,
        "bbox_points": list(
            annotation.bbox_points
        ),
        "bbox_normalized": list(
            annotation.bbox_normalized
        ),
        "annotation_type": (
            annotation.annotation_type
        ),
    }


# ================================================================
# Memuat dan memverifikasi template registry
# ================================================================

if not TEMPLATE_REGISTRY_PATH.is_file():
    raise FileNotFoundError(
        "Template registry tidak ditemukan: "
        f"{TEMPLATE_REGISTRY_PATH}"
    )

if not TEMPLATE_REGISTRY_CHECKSUM_PATH.is_file():
    raise FileNotFoundError(
        "Checksum template registry tidak ditemukan."
    )

with TEMPLATE_REGISTRY_PATH.open(
    mode="r",
    encoding="utf-8",
) as registry_file:
    loaded_template_artifact = json.load(
        registry_file
    )

stored_registry_checksum = (
    TEMPLATE_REGISTRY_CHECKSUM_PATH
    .read_text(
        encoding="utf-8"
    )
    .strip()
    .split()[0]
)

actual_registry_checksum = (
    checkpoint_sha256(
        TEMPLATE_REGISTRY_PATH
    )
)

if (
    stored_registry_checksum
    != actual_registry_checksum
):
    raise RuntimeError(
        "Checksum template registry tidak valid."
    )

template_registry = {
    template_specification[
        "template_id"
    ]: template_specification
    for template_specification
    in loaded_template_artifact[
        "templates"
    ]
}

if len(template_registry) != 10:
    raise RuntimeError(
        "Template registry harus berisi "
        "tepat 10 template."
    )


# ================================================================
# Memuat presentation payload dari checkpoint
# ================================================================

if not PRESENTATION_JSONL_PATH.is_file():
    raise FileNotFoundError(
        "Presentation JSONL tidak ditemukan: "
        f"{PRESENTATION_JSONL_PATH}"
    )

invoice_render_payloads = {}

with PRESENTATION_JSONL_PATH.open(
    mode="r",
    encoding="utf-8",
) as presentation_file:
    for line_number, line in enumerate(
        presentation_file,
        start=1,
    ):
        clean_line = line.strip()

        if not clean_line:
            continue

        try:
            render_payload = json.loads(
                clean_line
            )
        except json.JSONDecodeError as error:
            raise RuntimeError(
                "Presentation JSONL tidak valid "
                f"pada baris {line_number}."
            ) from error

        canonical_id = str(
            render_payload[
                "canonical_invoice_id"
            ]
        )

        if canonical_id in invoice_render_payloads:
            raise RuntimeError(
                "Canonical ID duplikat: "
                f"{canonical_id}"
            )

        invoice_render_payloads[
            canonical_id
        ] = render_payload

if len(invoice_render_payloads) != 200:
    raise RuntimeError(
        "Presentation payload tidak lengkap. "
        f"Expected: 200; "
        f"actual: {len(invoice_render_payloads)}"
    )


# ================================================================
# Validasi hubungan template dan payload
# ================================================================

renderer_foundation_rows = []

for template_id in sorted(
    template_registry
):
    specification = template_registry[
        template_id
    ]

    template_payload_count = sum(
        payload["template_id"]
        == template_id
        for payload
        in invoice_render_payloads.values()
    )

    page_size_valid = (
        specification["page_size"]
        in PAGE_SIZE_POINTS
    )

    colors_valid = all(
        re.fullmatch(
            r"#[0-9A-F]{6}",
            specification[color_name],
        )
        is not None
        for color_name in (
            "primary_color",
            "accent_color",
        )
    )

    template_valid = (
        template_payload_count == 20
        and page_size_valid
        and colors_valid
    )

    renderer_foundation_rows.append(
        {
            "template_id": template_id,
            "layout_family": specification[
                "layout_family"
            ],
            "page_size": specification[
                "page_size"
            ],
            "payloads": (
                template_payload_count
            ),
            "status": (
                "VALID"
                if template_valid
                else "INVALID"
            ),
        }
    )

renderer_foundation_audit = pd.DataFrame(
    renderer_foundation_rows
)

invalid_foundation_templates = (
    renderer_foundation_audit.loc[
        renderer_foundation_audit[
            "status"
        ] != "VALID",
        "template_id",
    ].tolist()
)

if invalid_foundation_templates:
    display(
        renderer_foundation_audit
    )

    raise RuntimeError(
        "Renderer foundation gagal untuk: "
        f"{invalid_foundation_templates}"
    )


# ================================================================
# Preflight A4 dan Letter
# ================================================================

preflight_document = pymupdf.open()
preflight_annotations = []

try:
    for page_size in (
        "A4",
        "LETTER",
    ):
        page_width, page_height = (
            get_page_dimensions(
                page_size
            )
        )

        preflight_page = (
            preflight_document.new_page(
                width=page_width,
                height=page_height,
            )
        )

        preflight_annotation, used_size = (
            insert_annotated_text(
                page=preflight_page,
                rectangle=pymupdf.Rect(
                    40,
                    40,
                    page_width - 40,
                    72,
                ),
                text=(
                    f"Renderer {page_size} READY"
                ),
                field_name=(
                    f"preflight.{page_size.lower()}"
                ),
                font_name="hebo",
                font_size=12.0,
                minimum_font_size=9.0,
                font_color=(0.1, 0.2, 0.3),
            )
        )

        preflight_annotations.append(
            {
                "page_size": page_size,
                "field_name": (
                    preflight_annotation.field_name
                ),
                "used_font_size": used_size,
                "bbox_normalized": (
                    preflight_annotation
                    .bbox_normalized
                ),
                "status": "VALID",
            }
        )

finally:
    preflight_document.close()

preflight_audit = pd.DataFrame(
    preflight_annotations
)

display(renderer_foundation_audit)
display(preflight_audit)

print(
    f"Render engine version : "
    f"{RENDER_ENGINE_VERSION}"
)
print(
    f"Templates loaded      : "
    f"{len(template_registry)}"
)
print(
    f"Payloads loaded       : "
    f"{len(invoice_render_payloads)}"
)
print(
    f"Registry SHA-256      : "
    f"{actual_registry_checksum}"
)
print()
print(
    "✅ Renderer foundation berhasil dimuat "
    "dan divalidasi."
)

**Cell 43 — Code: Renderer prototype TPL-02**

In [ ]:
# ================================================================
# CELL 43 — FINAL
# Prototype renderer TPL-02: Modern Minimal
# ================================================================

from pathlib import Path
from typing import Any
from PIL import Image as PILImage

import json
import os

import pandas as pd
import pymupdf


# ================================================================
# Validasi dependency runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "PROTOTYPE_ROOT",
    "CANONICAL_RECORDS_JSONL_PATH",
    "FieldAnnotation",
    "template_registry",
    "invoice_render_payloads",
    "get_page_dimensions",
    "hex_to_rgb",
    "insert_annotated_text",
    "insert_static_text",
    "annotation_to_dict",
    "write_json_atomically",
    "RENDER_ENGINE_VERSION",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency runtime Cell 43 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali cell pemulihan runtime dan Cell 42."
    )


# ================================================================
# Konfigurasi prototype TPL-02
# ================================================================

TPL02_TEMPLATE_ID = "TPL-02"
TPL02_RENDERER_VERSION = "1.1.0"
TPL02_MAX_ITEMS_PER_PAGE = 8

TPL02_PROTOTYPE_ROOT = (
    Path(PROTOTYPE_ROOT)
    / TPL02_TEMPLATE_ID
)

TPL02_PROTOTYPE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ================================================================
# Helper pemuatan canonical payload
# ================================================================

def load_canonical_payload_by_id(
    canonical_id: str,
) -> dict[str, Any]:
    """Memuat satu canonical invoice dari file JSONL."""

    if not isinstance(canonical_id, str) or not canonical_id.strip():
        raise ValueError(
            "canonical_id wajib berupa string yang tidak kosong."
        )

    canonical_path = Path(
        CANONICAL_RECORDS_JSONL_PATH
    )

    if not canonical_path.is_file():
        raise FileNotFoundError(
            "Canonical JSONL tidak ditemukan: "
            f"{canonical_path}"
        )

    with canonical_path.open(
        mode="r",
        encoding="utf-8",
    ) as canonical_file:
        for line_number, line in enumerate(
            canonical_file,
            start=1,
        ):
            clean_line = line.strip()

            if not clean_line:
                continue

            try:
                payload = json.loads(
                    clean_line
                )
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    "Canonical JSONL tidak valid pada "
                    f"baris {line_number}."
                ) from error

            if (
                payload.get("canonical_invoice_id")
                == canonical_id
            ):
                return payload

    raise KeyError(
        "Canonical payload tidak ditemukan: "
        f"{canonical_id}"
    )


# ================================================================
# Renderer TPL-02
# ================================================================

def render_template_02(
    render_payload: dict[str, Any],
    output_pdf_path: Path,
) -> tuple[list[FieldAnnotation], dict[str, Any]]:
    """Merender invoice satu halaman dengan desain TPL-02."""

    if not isinstance(render_payload, dict):
        raise TypeError(
            "render_payload wajib berupa dictionary."
        )

    if render_payload.get("template_id") != TPL02_TEMPLATE_ID:
        raise ValueError(
            "render_template_02 hanya menerima payload TPL-02."
        )

    required_payload_keys = {
        "canonical_invoice_id",
        "document_id",
        "template_id",
        "language",
        "currency",
        "title",
        "labels",
        "metadata",
        "vendor",
        "buyer",
        "items",
        "financials",
        "footer",
        "synthetic_notice",
    }

    missing_payload_keys = sorted(
        required_payload_keys
        - set(render_payload)
    )

    if missing_payload_keys:
        raise KeyError(
            "Render payload TPL-02 belum lengkap: "
            f"{missing_payload_keys}"
        )

    item_count = len(
        render_payload["items"]
    )

    if not 1 <= item_count <= TPL02_MAX_ITEMS_PER_PAGE:
        raise ValueError(
            "TPL-02 hanya mendukung 1–8 item "
            "dalam satu halaman."
        )

    language = render_payload["language"]

    if language == "id":
        actual_quantity_label = (
            render_payload["labels"]["quantity"]
        )

        if actual_quantity_label != "Kuantitas":
            raise RuntimeError(
                "Label kuantitas bahasa Indonesia belum benar. "
                f"Expected: 'Kuantitas'; "
                f"actual: {actual_quantity_label!r}. "
                "Jalankan kembali Cell 35 dan Cell 41."
            )

    template_specification = (
        template_registry[TPL02_TEMPLATE_ID]
    )

    page_width, page_height = (
        get_page_dimensions(
            template_specification["page_size"]
        )
    )

    primary_color = hex_to_rgb(
        template_specification["primary_color"]
    )

    accent_color = hex_to_rgb(
        template_specification["accent_color"]
    )

    dark_text = (
        0.12,
        0.16,
        0.20,
    )

    muted_text = (
        0.40,
        0.44,
        0.48,
    )

    border_color = (
        0.82,
        0.85,
        0.87,
    )

    light_accent = (
        0.94,
        0.985,
        0.975,
    )

    output_pdf_path = Path(
        output_pdf_path
    )

    output_pdf_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_pdf_path = (
        output_pdf_path.parent
        / f".{output_pdf_path.name}.tmp"
    )

    annotations: list[FieldAnnotation] = []
    field_font_audit: list[dict[str, Any]] = []

    document = pymupdf.open()

    def add_field(
        page: pymupdf.Page,
        rectangle: pymupdf.Rect,
        text: str,
        field_name: str,
        font_name: str = "helv",
        font_size: float = 8.0,
        minimum_font_size: float = 6.5,
        font_color: tuple[
            float,
            float,
            float,
        ] = dark_text,
        alignment: str = "left",
    ) -> None:
        """Menambahkan field sekaligus mencatat audit font."""

        annotation, used_font_size = (
            insert_annotated_text(
                page=page,
                rectangle=rectangle,
                text=str(text),
                field_name=field_name,
                font_name=font_name,
                font_size=font_size,
                minimum_font_size=minimum_font_size,
                font_color=font_color,
                alignment=alignment,
            )
        )

        annotations.append(
            annotation
        )

        field_font_audit.append(
            {
                "field_name": field_name,
                "requested_font_size": font_size,
                "used_font_size": used_font_size,
                "adjusted": (
                    used_font_size < font_size
                ),
            }
        )

    try:
        page = document.new_page(
            width=page_width,
            height=page_height,
        )

        # --------------------------------------------------------
        # Accent sidebar
        # --------------------------------------------------------

        page.draw_rect(
            pymupdf.Rect(
                0,
                0,
                16,
                page_height,
            ),
            color=accent_color,
            fill=accent_color,
            width=0,
            overlay=True,
        )

        # --------------------------------------------------------
        # Header
        # --------------------------------------------------------

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                45,
                40,
                315,
                94,
            ),
            text=render_payload["title"],
            font_name="hebo",
            font_size=25.0,
            minimum_font_size=18.0,
            font_color=primary_color,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                47,
                99,
                315,
                119,
            ),
            text=(
                render_payload["footer"]
                ["document_reference"]
            ),
            font_name="helv",
            font_size=7.5,
            minimum_font_size=6.5,
            font_color=muted_text,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                47,
                126,
                315,
                143,
            ),
            text="MODERN MINIMAL · SYNTHETIC",
            font_name="hebo",
            font_size=7.0,
            minimum_font_size=6.0,
            font_color=accent_color,
        )

        # --------------------------------------------------------
        # Metadata invoice
        # --------------------------------------------------------

        metadata_rows = [
            (
                "invoice_number",
                42,
                64,
            ),
            (
                "invoice_date",
                70,
                92,
            ),
            (
                "due_date",
                98,
                120,
            ),
        ]

        for (
            field_name,
            row_y0,
            row_y1,
        ) in metadata_rows:
            field_payload = (
                render_payload["metadata"]
                [field_name]
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    345,
                    row_y0,
                    428,
                    row_y1,
                ),
                text=field_payload["label"],
                font_name="helv",
                font_size=7.1,
                minimum_font_size=6.0,
                font_color=muted_text,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    430,
                    row_y0,
                    page_width - 45,
                    row_y1,
                ),
                text=field_payload["display_value"],
                field_name=field_name,
                font_name="hebo",
                font_size=8.2,
                minimum_font_size=6.8,
                font_color=dark_text,
                alignment="right",
            )

        currency_label = (
            "Mata Uang"
            if language == "id"
            else "Currency"
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                345,
                126,
                428,
                148,
            ),
            text=currency_label,
            font_name="helv",
            font_size=7.1,
            minimum_font_size=6.0,
            font_color=muted_text,
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                430,
                126,
                page_width - 45,
                148,
            ),
            text=render_payload["currency"],
            field_name="currency",
            font_name="hebo",
            font_size=8.5,
            minimum_font_size=7.0,
            font_color=accent_color,
            alignment="right",
        )

        page.draw_line(
            pymupdf.Point(
                45,
                157,
            ),
            pymupdf.Point(
                page_width - 45,
                157,
            ),
            color=border_color,
            width=0.8,
            overlay=True,
        )

        # --------------------------------------------------------
        # Vendor dan buyer
        # --------------------------------------------------------

        party_sections = [
            (
                "vendor",
                render_payload["labels"]["vendor"],
                168,
                247,
            ),
            (
                "buyer",
                render_payload["labels"]["buyer"],
                258,
                337,
            ),
        ]

        for (
            party_type,
            party_label,
            section_y0,
            section_y1,
        ) in party_sections:
            party = render_payload[
                party_type
            ]

            page.draw_rect(
                pymupdf.Rect(
                    45,
                    section_y0,
                    page_width - 45,
                    section_y1,
                ),
                color=None,
                fill=light_accent,
                width=0,
                overlay=True,
            )

            page.draw_rect(
                pymupdf.Rect(
                    45,
                    section_y0,
                    49,
                    section_y1,
                ),
                color=accent_color,
                fill=accent_color,
                width=0,
                overlay=True,
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    57,
                    section_y0 + 7,
                    300,
                    section_y0 + 20,
                ),
                text=party_label.upper(),
                font_name="hebo",
                font_size=6.8,
                minimum_font_size=6.0,
                font_color=accent_color,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    57,
                    section_y0 + 23,
                    307,
                    section_y0 + 41,
                ),
                text=party["name"],
                field_name=f"{party_type}.name",
                font_name="hebo",
                font_size=8.8,
                minimum_font_size=7.0,
                font_color=dark_text,
            )

            address_text = "\n".join(
                party["address_lines"]
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    57,
                    section_y0 + 43,
                    307,
                    section_y1 - 6,
                ),
                text=address_text,
                field_name=(
                    f"{party_type}.address_lines"
                ),
                font_name="helv",
                font_size=6.9,
                minimum_font_size=6.5,
                font_color=dark_text,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    325,
                    section_y0 + 10,
                    page_width - 55,
                    section_y0 + 25,
                ),
                text=party["email"],
                field_name=f"{party_type}.email",
                font_name="helv",
                font_size=7.2,
                minimum_font_size=6.5,
                font_color=dark_text,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    325,
                    section_y0 + 31,
                    page_width - 55,
                    section_y0 + 46,
                ),
                text=party["phone"],
                field_name=f"{party_type}.phone",
                font_name="helv",
                font_size=7.2,
                minimum_font_size=6.5,
                font_color=dark_text,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    325,
                    section_y0 + 52,
                    page_width - 55,
                    section_y0 + 68,
                ),
                text=party[
                    "tax_identifier_display"
                ],
                field_name=(
                    f"{party_type}.tax_identifier"
                ),
                font_name="helv",
                font_size=7.2,
                minimum_font_size=6.5,
                font_color=dark_text,
            )

        # --------------------------------------------------------
        # Item table
        # --------------------------------------------------------

        table_x_positions = [
            45,
            70,
            315,
            365,
            460,
            page_width - 45,
        ]

        table_header_y0 = 354.0
        table_header_y1 = 384.0
        row_height = 26.0

        page.draw_rect(
            pymupdf.Rect(
                table_x_positions[0],
                table_header_y0,
                table_x_positions[-1],
                table_header_y1,
            ),
            color=None,
            fill=light_accent,
            width=0,
            overlay=True,
        )

        page.draw_line(
            pymupdf.Point(
                table_x_positions[0],
                table_header_y1,
            ),
            pymupdf.Point(
                table_x_positions[-1],
                table_header_y1,
            ),
            color=accent_color,
            width=1.2,
            overlay=True,
        )

        table_headers = [
            "#",
            render_payload["labels"]["description"],
            render_payload["labels"]["quantity"],
            render_payload["labels"]["unit_price"],
            render_payload["labels"]["line_total"],
        ]

        for (
            column_index,
            header_text,
        ) in enumerate(table_headers):
            header_alignment = (
                "left"
                if column_index == 1
                else "center"
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    (
                        table_x_positions[
                            column_index
                        ]
                        + 3
                    ),
                    table_header_y0 + 8,
                    (
                        table_x_positions[
                            column_index + 1
                        ]
                        - 3
                    ),
                    table_header_y1 - 3,
                ),
                text=header_text,
                font_name="hebo",
                font_size=7.0,
                minimum_font_size=6.0,
                font_color=primary_color,
                alignment=header_alignment,
            )

        for (
            item_index,
            item,
        ) in enumerate(
            render_payload["items"]
        ):
            row_y0 = (
                table_header_y1
                + item_index * row_height
            )

            row_y1 = (
                row_y0 + row_height
            )

            page.draw_line(
                pymupdf.Point(
                    table_x_positions[0],
                    row_y1,
                ),
                pymupdf.Point(
                    table_x_positions[-1],
                    row_y1,
                ),
                color=border_color,
                width=0.55,
                overlay=True,
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    table_x_positions[0] + 3,
                    row_y0 + 6,
                    table_x_positions[1] - 3,
                    row_y1 - 3,
                ),
                text=str(
                    item["item_number"]
                ),
                font_name="helv",
                font_size=7.4,
                minimum_font_size=6.5,
                font_color=muted_text,
                alignment="center",
            )

            item_fields = [
                (
                    "description",
                    1,
                    "left",
                ),
                (
                    "quantity",
                    2,
                    "center",
                ),
                (
                    "unit_price",
                    3,
                    "right",
                ),
                (
                    "line_total",
                    4,
                    "right",
                ),
            ]

            for (
                item_field,
                column_index,
                field_alignment,
            ) in item_fields:
                add_field(
                    page=page,
                    rectangle=pymupdf.Rect(
                        (
                            table_x_positions[
                                column_index
                            ]
                            + 4
                        ),
                        row_y0 + 6,
                        (
                            table_x_positions[
                                column_index + 1
                            ]
                            - 4
                        ),
                        row_y1 - 3,
                    ),
                    text=item[item_field],
                    field_name=(
                        f"items[{item_index}]"
                        f".{item_field}"
                    ),
                    font_name="helv",
                    font_size=7.4,
                    minimum_font_size=6.5,
                    font_color=dark_text,
                    alignment=field_alignment,
                )

        # --------------------------------------------------------
        # Financial summary — dynamic placement
        # --------------------------------------------------------

        table_end_y = (
            table_header_y1
            + item_count * row_height
        )

        totals_panel_gap = 24.0
        totals_panel_height = 107.0
        totals_minimum_y0 = 548.0
        totals_maximum_y0 = 614.0

        totals_x0 = 342.0
        totals_x1 = page_width - 45.0

        totals_y0 = min(
            max(
                table_end_y + totals_panel_gap,
                totals_minimum_y0,
            ),
            totals_maximum_y0,
        )

        totals_y1 = (
            totals_y0
            + totals_panel_height
        )

        actual_table_to_totals_gap = (
            totals_y0 - table_end_y
        )

        if actual_table_to_totals_gap < 20.0:
            raise RuntimeError(
                "Jarak tabel dan panel total terlalu sempit: "
                f"{actual_table_to_totals_gap:.2f} point."
            )

        footer_line_y = 760.0

        if totals_y1 >= footer_line_y:
            raise RuntimeError(
                "Panel total bertabrakan dengan footer. "
                f"Panel berakhir pada {totals_y1:.2f}; "
                f"footer dimulai pada {footer_line_y:.2f}."
            )

        page.draw_rect(
            pymupdf.Rect(
                totals_x0,
                totals_y0,
                totals_x1,
                totals_y1,
            ),
            color=None,
            fill=light_accent,
            width=0,
            overlay=True,
        )

        financial_order = [
            "subtotal",
            "tax",
            "discount",
            "total",
        ]

        for (
            row_index,
            field_name,
        ) in enumerate(financial_order):
            field_payload = (
                render_payload["financials"]
                [field_name]
            )

            row_y0 = (
                totals_y0
                + 9.0
                + row_index * 23.0
            )

            is_total = (
                field_name == "total"
            )

            if is_total:
                page.draw_line(
                    pymupdf.Point(
                        totals_x0 + 10,
                        row_y0 - 4,
                    ),
                    pymupdf.Point(
                        totals_x1 - 10,
                        row_y0 - 4,
                    ),
                    color=accent_color,
                    width=1.0,
                    overlay=True,
                )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    totals_x0 + 10,
                    row_y0,
                    totals_x0 + 90,
                    row_y0 + 18,
                ),
                text=field_payload["label"],
                font_name=(
                    "hebo"
                    if is_total
                    else "helv"
                ),
                font_size=(
                    8.3
                    if is_total
                    else 7.8
                ),
                minimum_font_size=6.5,
                font_color=(
                    primary_color
                    if is_total
                    else muted_text
                ),
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    totals_x0 + 92,
                    row_y0,
                    totals_x1 - 10,
                    row_y0 + 18,
                ),
                text=field_payload["display_value"],
                field_name=(
                    f"financials.{field_name}"
                ),
                font_name=(
                    "hebo"
                    if is_total
                    else "helv"
                ),
                font_size=(
                    8.7
                    if is_total
                    else 7.8
                ),
                minimum_font_size=6.5,
                font_color=(
                    accent_color
                    if is_total
                    else dark_text
                ),
                alignment="right",
            )

        # --------------------------------------------------------
        # Footer
        # --------------------------------------------------------

        page.draw_line(
            pymupdf.Point(
                45,
                footer_line_y,
            ),
            pymupdf.Point(
                page_width - 45,
                footer_line_y,
            ),
            color=border_color,
            width=0.7,
            overlay=True,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                45,
                769,
                page_width - 45,
                784,
            ),
            text=(
                render_payload["footer"]
                ["document_reference"]
            ),
            font_name="helv",
            font_size=6.8,
            minimum_font_size=6.0,
            font_color=muted_text,
            alignment="center",
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                45,
                792,
                page_width - 45,
                811,
            ),
            text=render_payload[
                "synthetic_notice"
            ],
            field_name=(
                "document.synthetic_notice"
            ),
            font_name="hebo",
            font_size=7.0,
            minimum_font_size=6.5,
            font_color=accent_color,
            alignment="center",
        )

        # --------------------------------------------------------
        # Pre-save controls
        # --------------------------------------------------------

        annotation_names = [
            annotation.field_name
            for annotation in annotations
        ]

        if (
            len(annotation_names)
            != len(set(annotation_names))
        ):
            raise RuntimeError(
                "Ditemukan nama anotasi duplikat."
            )

        expected_annotation_count = (
            19 + 4 * item_count
        )

        actual_annotation_count = len(
            annotations
        )

        if (
            actual_annotation_count
            != expected_annotation_count
        ):
            raise RuntimeError(
                "Jumlah anotasi tidak sesuai. "
                f"Expected: {expected_annotation_count}; "
                f"actual: {actual_annotation_count}."
            )

        if not field_font_audit:
            raise RuntimeError(
                "Audit font field tidak boleh kosong."
            )

        minimum_used_font_size = min(
            row["used_font_size"]
            for row in field_font_audit
        )

        if minimum_used_font_size < 6.5:
            raise RuntimeError(
                "Font field berada di bawah "
                "batas keterbacaan 6.5 point."
            )

        temporary_pdf_path.unlink(
            missing_ok=True
        )

        document.save(
            str(temporary_pdf_path),
            garbage=4,
            deflate=True,
        )

    except Exception:
        temporary_pdf_path.unlink(
            missing_ok=True
        )
        raise

    finally:
        document.close()

    if not temporary_pdf_path.is_file():
        raise RuntimeError(
            "Temporary PDF tidak berhasil dibuat."
        )

    if temporary_pdf_path.stat().st_size == 0:
        temporary_pdf_path.unlink(
            missing_ok=True
        )
        raise RuntimeError(
            "Temporary PDF memiliki ukuran nol byte."
        )

    os.replace(
        temporary_pdf_path,
        output_pdf_path,
    )

    rendering_metadata = {
        "renderer_version": (
            RENDER_ENGINE_VERSION
        ),
        "template_renderer_version": (
            TPL02_RENDERER_VERSION
        ),
        "template_id": TPL02_TEMPLATE_ID,
        "layout_family": (
            template_specification[
                "layout_family"
            ]
        ),
        "page_size": (
            template_specification[
                "page_size"
            ]
        ),
        "page_width_points": round(
            page_width,
            3,
        ),
        "page_height_points": round(
            page_height,
            3,
        ),
        "page_count": 1,
        "annotation_type": "field_region",
        "annotation_count": len(
            annotations
        ),
        "minimum_field_font_size": min(
            row["used_font_size"]
            for row in field_font_audit
        ),
        "font_adjustment_count": int(
            sum(
                bool(row["adjusted"])
                for row in field_font_audit
            )
        ),
        "layout_metrics": {
            "item_count": item_count,
            "table_end_y_points": round(
                table_end_y,
                3,
            ),
            "totals_panel_y0_points": round(
                totals_y0,
                3,
            ),
            "totals_panel_y1_points": round(
                totals_y1,
                3,
            ),
            "table_to_totals_gap_points": round(
                actual_table_to_totals_gap,
                3,
            ),
            "footer_line_y_points": (
                footer_line_y
            ),
        },
    }

    return (
        annotations,
        rendering_metadata,
    )


# ================================================================
# Memilih prototype pertama TPL-02
# ================================================================

tpl02_candidate_ids = sorted(
    canonical_id
    for canonical_id, payload
    in invoice_render_payloads.items()
    if payload["template_id"] == TPL02_TEMPLATE_ID
)

if len(tpl02_candidate_ids) != 20:
    raise RuntimeError(
        "TPL-02 harus memiliki tepat 20 payload. "
        f"Actual: {len(tpl02_candidate_ids)}."
    )

prototype_02_canonical_id = (
    tpl02_candidate_ids[0]
)

prototype_02_render_payload = (
    invoice_render_payloads[
        prototype_02_canonical_id
    ]
)

prototype_02_document_id = (
    prototype_02_render_payload[
        "document_id"
    ]
)

prototype_02_canonical_payload = (
    load_canonical_payload_by_id(
        prototype_02_canonical_id
    )
)


# ================================================================
# Lokasi artifact prototype
# ================================================================

prototype_02_pdf_path = (
    TPL02_PROTOTYPE_ROOT
    / (
        f"{prototype_02_document_id}"
        "_TPL-02_prototype.pdf"
    )
)

prototype_02_png_path = (
    TPL02_PROTOTYPE_ROOT
    / (
        f"{prototype_02_document_id}"
        "_TPL-02_preview.png"
    )
)

prototype_02_ground_truth_path = (
    TPL02_PROTOTYPE_ROOT
    / (
        f"{prototype_02_document_id}"
        "_TPL-02_ground_truth.json"
    )
)


# ================================================================
# Render prototype
# ================================================================

(
    prototype_02_annotations,
    prototype_02_rendering_metadata,
) = render_template_02(
    render_payload=(
        prototype_02_render_payload
    ),
    output_pdf_path=(
        prototype_02_pdf_path
    ),
)


# ================================================================
# Simpan ground truth prototype
# ================================================================

prototype_02_ground_truth = {
    "schema_version": "1.0.0",
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "document": {
        "document_id": (
            prototype_02_document_id
        ),
        "canonical_invoice_id": (
            prototype_02_canonical_id
        ),
        "split": (
            prototype_02_render_payload[
                "split"
            ]
        ),
        "template_id": (
            TPL02_TEMPLATE_ID
        ),
        "language": (
            prototype_02_render_payload[
                "language"
            ]
        ),
        "currency": (
            prototype_02_render_payload[
                "currency"
            ]
        ),
    },
    "canonical": (
        prototype_02_canonical_payload
    ),
    "rendering": (
        prototype_02_rendering_metadata
    ),
    "annotations": [
        annotation_to_dict(
            annotation
        )
        for annotation
        in prototype_02_annotations
    ],
}

write_json_atomically(
    prototype_02_ground_truth_path,
    prototype_02_ground_truth,
)


# ================================================================
# Buka ulang PDF dan buat preview
# ================================================================

temporary_png_path = (
    prototype_02_png_path.parent
    / (
        f".{prototype_02_png_path.name}"
        ".tmp.png"
    )
)

temporary_png_path.unlink(
    missing_ok=True
)

try:
    with pymupdf.open(
        str(prototype_02_pdf_path)
    ) as prototype_document:
        if prototype_document.page_count != 1:
            raise RuntimeError(
                "Prototype TPL-02 harus memiliki "
                "tepat satu halaman."
            )

        prototype_page = (
            prototype_document[0]
        )

        extracted_text_02 = (
            prototype_page.get_text("text")
        )

        prototype_pixmap_02 = (
            prototype_page.get_pixmap(
                matrix=pymupdf.Matrix(
                    150 / 72,
                    150 / 72,
                ),
                alpha=False,
            )
        )

        prototype_pixmap_02.save(
            str(temporary_png_path)
        )

    if not temporary_png_path.is_file():
        raise RuntimeError(
            "Preview sementara tidak berhasil dibuat."
        )

    if temporary_png_path.stat().st_size == 0:
        raise RuntimeError(
            "Preview sementara memiliki ukuran nol byte."
        )

    os.replace(
        temporary_png_path,
        prototype_02_png_path,
    )

except Exception:
    temporary_png_path.unlink(
        missing_ok=True
    )
    raise


# ================================================================
# Post-render controls
# ================================================================

required_text_fragments_02 = [
    (
        prototype_02_render_payload[
            "metadata"
        ]["invoice_number"]["display_value"]
    ),
    (
        prototype_02_render_payload[
            "vendor"
        ]["name"]
    ),
    (
        prototype_02_render_payload[
            "buyer"
        ]["name"]
    ),
    (
        prototype_02_render_payload[
            "labels"
        ]["quantity"]
    ),
    (
        prototype_02_render_payload[
            "financials"
        ]["total"]["display_value"]
    ),
]

missing_text_fragments_02 = [
    text_fragment
    for text_fragment
    in required_text_fragments_02
    if text_fragment not in extracted_text_02
]

if missing_text_fragments_02:
    raise RuntimeError(
        "Teks penting TPL-02 tidak ditemukan "
        "pada hasil ekstraksi PDF: "
        f"{missing_text_fragments_02}"
    )

annotation_names_02 = {
    annotation.field_name
    for annotation
    in prototype_02_annotations
}

required_annotation_names_02 = {
    "vendor.name",
    "buyer.name",
    "invoice_number",
    "invoice_date",
    "due_date",
    "currency",
    "financials.subtotal",
    "financials.tax",
    "financials.discount",
    "financials.total",
    "document.synthetic_notice",
}

missing_annotations_02 = sorted(
    required_annotation_names_02
    - annotation_names_02
)

if missing_annotations_02:
    raise RuntimeError(
        "Anotasi wajib TPL-02 tidak tersedia: "
        f"{missing_annotations_02}"
    )

layout_metrics_02 = (
    prototype_02_rendering_metadata[
        "layout_metrics"
    ]
)

prototype_02_summary = pd.DataFrame(
    [
        {
            "control": "pdf_created",
            "actual": (
                prototype_02_pdf_path.is_file()
            ),
            "status": "VALID",
        },
        {
            "control": "ground_truth_created",
            "actual": (
                prototype_02_ground_truth_path
                .is_file()
            ),
            "status": "VALID",
        },
        {
            "control": "annotation_count",
            "actual": len(
                prototype_02_annotations
            ),
            "status": "VALID",
        },
        {
            "control": "quantity_header",
            "actual": (
                prototype_02_render_payload[
                    "labels"
                ]["quantity"]
            ),
            "status": "VALID",
        },
        {
            "control": "minimum_field_font",
            "actual": (
                prototype_02_rendering_metadata[
                    "minimum_field_font_size"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "font_adjustments",
            "actual": (
                prototype_02_rendering_metadata[
                    "font_adjustment_count"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "table_to_totals_gap",
            "actual": (
                layout_metrics_02[
                    "table_to_totals_gap_points"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "missing_required_text",
            "actual": len(
                missing_text_fragments_02
            ),
            "status": "VALID",
        },
    ]
)

display(
    prototype_02_summary
)

with PILImage.open(
    prototype_02_png_path
) as prototype_image:
    display(
        prototype_image.copy()
    )


# ================================================================
# Ringkasan
# ================================================================

print(
    f"Canonical ID       : "
    f"{prototype_02_canonical_id}"
)

print(
    f"Document ID        : "
    f"{prototype_02_document_id}"
)

print(
    f"Prototype PDF      : "
    f"{prototype_02_pdf_path}"
)

print(
    f"Prototype preview  : "
    f"{prototype_02_png_path}"
)

print(
    f"Ground truth       : "
    f"{prototype_02_ground_truth_path}"
)

print(
    f"Annotations        : "
    f"{len(prototype_02_annotations)}"
)

print(
    f"Quantity header    : "
    f"{prototype_02_render_payload['labels']['quantity']}"
)

print(
    f"Minimum field font : "
    f"{prototype_02_rendering_metadata['minimum_field_font_size']}"
)

print(
    f"Adjusted fields    : "
    f"{prototype_02_rendering_metadata['font_adjustment_count']}"
)

print(
    f"Table-total gap    : "
    f"{layout_metrics_02['table_to_totals_gap_points']} pt"
)

print(
    f"Renderer version   : "
    f"{TPL02_RENDERER_VERSION}"
)

print()

print(
    "✅ Prototype TPL-02 berhasil dirender "
    "dengan header dan layout final, serta "
    "lolos pemeriksaan teknis awal."
)

**Cell 44 — Code: Audit teknis prototype TPL-02**

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os


def calculate_artifact_sha256(
    artifact_path: Path,
) -> str:
    """Menghitung SHA-256 artifact."""

    digest = hashlib.sha256()

    with Path(artifact_path).open(
        mode="rb"
    ) as artifact_file:
        for chunk in iter(
            lambda: artifact_file.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def audit_prototype_artifacts(
    pdf_path: Path,
    preview_path: Path,
    ground_truth_path: Path,
) -> tuple[dict, pd.DataFrame]:
    """Melakukan audit teknis prototype invoice."""

    required_files = {
        "pdf": Path(pdf_path),
        "preview": Path(preview_path),
        "ground_truth": Path(
            ground_truth_path
        ),
    }

    missing_files = [
        artifact_name
        for artifact_name, artifact_path
        in required_files.items()
        if (
            not artifact_path.is_file()
            or artifact_path.stat().st_size == 0
        )
    ]

    if missing_files:
        raise FileNotFoundError(
            "Artifact prototype tidak lengkap: "
            f"{missing_files}"
        )

    with Path(ground_truth_path).open(
        mode="r",
        encoding="utf-8",
    ) as ground_truth_file:
        ground_truth = json.load(
            ground_truth_file
        )

    annotations = ground_truth[
        "annotations"
    ]

    rendering = ground_truth[
        "rendering"
    ]

    canonical = ground_truth[
        "canonical"
    ]

    document_information = ground_truth[
        "document"
    ]

    annotation_lookup = {
        annotation["field_name"]: annotation
        for annotation in annotations
    }

    duplicate_annotation_count = (
        len(annotations)
        - len(annotation_lookup)
    )

    required_annotation_names = {
        "vendor.name",
        "buyer.name",
        "invoice_number",
        "invoice_date",
        "due_date",
        "currency",
        "financials.subtotal",
        "financials.tax",
        "financials.discount",
        "financials.total",
        "document.synthetic_notice",
    }

    missing_required_annotations = sorted(
        required_annotation_names
        - set(annotation_lookup)
    )

    expected_annotation_count = (
        19
        + 4 * len(
            canonical["items"]
        )
    )

    invalid_bbox_fields = []
    normalization_mismatch_fields = []
    annotations_without_words = []
    severe_overlap_pairs = []
    missing_extracted_values = []

    with pymupdf.open(
        str(pdf_path)
    ) as pdf_document:
        page_count = (
            pdf_document.page_count
        )

        if page_count != 1:
            raise RuntimeError(
                "Prototype harus memiliki "
                "tepat satu halaman."
            )

        page = pdf_document[0]
        page_rectangle = page.rect

        extracted_text = page.get_text(
            "text"
        )

        extracted_words = page.get_text(
            "words"
        )

        for annotation in annotations:
            field_name = annotation[
                "field_name"
            ]

            bbox_points = annotation[
                "bbox_points"
            ]

            bbox_normalized = annotation[
                "bbox_normalized"
            ]

            if (
                not isinstance(
                    bbox_points,
                    list,
                )
                or len(bbox_points) != 4
            ):
                invalid_bbox_fields.append(
                    field_name
                )
                continue

            annotation_rectangle = (
                pymupdf.Rect(
                    *bbox_points
                )
            )

            bbox_valid = (
                not annotation_rectangle.is_empty
                and not annotation_rectangle.is_infinite
                and annotation_rectangle.x0
                >= page_rectangle.x0
                and annotation_rectangle.y0
                >= page_rectangle.y0
                and annotation_rectangle.x1
                <= page_rectangle.x1
                and annotation_rectangle.y1
                <= page_rectangle.y1
            )

            if not bbox_valid:
                invalid_bbox_fields.append(
                    field_name
                )
                continue

            recalculated_normalized = (
                normalize_rectangle(
                    annotation_rectangle,
                    page_rectangle,
                )
            )

            maximum_difference = max(
                abs(
                    float(stored_value)
                    - float(calculated_value)
                )
                for (
                    stored_value,
                    calculated_value,
                ) in zip(
                    bbox_normalized,
                    recalculated_normalized,
                )
            )

            if maximum_difference > 1e-5:
                normalization_mismatch_fields.append(
                    field_name
                )

            words_inside_bbox = []

            for word in extracted_words:
                (
                    word_x0,
                    word_y0,
                    word_x1,
                    word_y1,
                    word_text,
                    *_,
                ) = word

                word_center = pymupdf.Point(
                    (
                        word_x0 + word_x1
                    ) / 2,
                    (
                        word_y0 + word_y1
                    ) / 2,
                )

                if annotation_rectangle.contains(
                    word_center
                ):
                    words_inside_bbox.append(
                        word_text
                    )

            if not words_inside_bbox:
                annotations_without_words.append(
                    field_name
                )

        for first_index in range(
            len(annotations)
        ):
            first_annotation = (
                annotations[first_index]
            )

            first_rectangle = pymupdf.Rect(
                *first_annotation[
                    "bbox_points"
                ]
            )

            for second_index in range(
                first_index + 1,
                len(annotations),
            ):
                second_annotation = (
                    annotations[second_index]
                )

                second_rectangle = (
                    pymupdf.Rect(
                        *second_annotation[
                            "bbox_points"
                        ]
                    )
                )

                intersection = (
                    first_rectangle
                    & second_rectangle
                )

                if intersection.is_empty:
                    continue

                smaller_area = min(
                    first_rectangle.get_area(),
                    second_rectangle.get_area(),
                )

                if smaller_area <= 0:
                    continue

                overlap_ratio = (
                    intersection.get_area()
                    / smaller_area
                )

                if overlap_ratio > 0.10:
                    severe_overlap_pairs.append(
                        {
                            "first_field": (
                                first_annotation[
                                    "field_name"
                                ]
                            ),
                            "second_field": (
                                second_annotation[
                                    "field_name"
                                ]
                            ),
                            "overlap_ratio": round(
                                overlap_ratio,
                                4,
                            ),
                        }
                    )

        expected_extracted_values = [
            annotation_lookup[
                field_name
            ]["text"]
            for field_name in (
                "vendor.name",
                "buyer.name",
                "invoice_number",
                "financials.total",
            )
            if field_name
            in annotation_lookup
        ]

        missing_extracted_values = [
            expected_value
            for expected_value
            in expected_extracted_values
            if expected_value
            not in extracted_text
        ]

    grayscale_image = cv2.imread(
        str(preview_path),
        cv2.IMREAD_GRAYSCALE,
    )

    if grayscale_image is None:
        raise RuntimeError(
            "Preview tidak dapat dibaca OpenCV."
        )

    image_height, image_width = (
        grayscale_image.shape
    )

    ink_pixel_ratio = float(
        np.mean(
            grayscale_image < 245
        )
    )

    white_pixel_ratio = float(
        np.mean(
            grayscale_image >= 250
        )
    )

    raster_contrast = float(
        np.std(grayscale_image)
    )

    minimum_field_font_size = float(
        rendering[
            "minimum_field_font_size"
        ]
    )

    font_adjustment_count = int(
        rendering[
            "font_adjustment_count"
        ]
    )

    font_adjustment_ratio = (
        font_adjustment_count
        / len(annotations)
    )

    controls = pd.DataFrame(
        [
            {
                "control": "pdf_page_count",
                "expected": 1,
                "actual": page_count,
                "status": (
                    "VALID"
                    if page_count == 1
                    else "INVALID"
                ),
            },
            {
                "control": "annotation_count",
                "expected": (
                    expected_annotation_count
                ),
                "actual": len(annotations),
                "status": (
                    "VALID"
                    if len(annotations)
                    == expected_annotation_count
                    else "INVALID"
                ),
            },
            {
                "control": "duplicate_annotations",
                "expected": 0,
                "actual": (
                    duplicate_annotation_count
                ),
                "status": (
                    "VALID"
                    if duplicate_annotation_count == 0
                    else "INVALID"
                ),
            },
            {
                "control": "missing_required_annotations",
                "expected": 0,
                "actual": len(
                    missing_required_annotations
                ),
                "status": (
                    "VALID"
                    if not missing_required_annotations
                    else "INVALID"
                ),
            },
            {
                "control": "invalid_bounding_boxes",
                "expected": 0,
                "actual": len(
                    invalid_bbox_fields
                ),
                "status": (
                    "VALID"
                    if not invalid_bbox_fields
                    else "INVALID"
                ),
            },
            {
                "control": "normalization_mismatches",
                "expected": 0,
                "actual": len(
                    normalization_mismatch_fields
                ),
                "status": (
                    "VALID"
                    if not normalization_mismatch_fields
                    else "INVALID"
                ),
            },
            {
                "control": "annotations_without_words",
                "expected": 0,
                "actual": len(
                    annotations_without_words
                ),
                "status": (
                    "VALID"
                    if not annotations_without_words
                    else "INVALID"
                ),
            },
            {
                "control": "severe_annotation_overlaps",
                "expected": 0,
                "actual": len(
                    severe_overlap_pairs
                ),
                "status": (
                    "VALID"
                    if not severe_overlap_pairs
                    else "INVALID"
                ),
            },
            {
                "control": "missing_extracted_values",
                "expected": 0,
                "actual": len(
                    missing_extracted_values
                ),
                "status": (
                    "VALID"
                    if not missing_extracted_values
                    else "INVALID"
                ),
            },
            {
                "control": "minimum_field_font_size",
                "expected": ">=6.5",
                "actual": (
                    minimum_field_font_size
                ),
                "status": (
                    "VALID"
                    if minimum_field_font_size
                    >= 6.5
                    else "INVALID"
                ),
            },
            {
                "control": "font_adjustment_ratio",
                "expected": "<=0.20",
                "actual": round(
                    font_adjustment_ratio,
                    4,
                ),
                "status": (
                    "VALID"
                    if font_adjustment_ratio
                    <= 0.20
                    else "INVALID"
                ),
            },
            {
                "control": "raster_content_ratio",
                "expected": "0.01–0.40",
                "actual": round(
                    ink_pixel_ratio,
                    4,
                ),
                "status": (
                    "VALID"
                    if 0.01
                    <= ink_pixel_ratio
                    <= 0.40
                    else "INVALID"
                ),
            },
            {
                "control": "raster_contrast",
                "expected": ">=10.0",
                "actual": round(
                    raster_contrast,
                    4,
                ),
                "status": (
                    "VALID"
                    if raster_contrast >= 10.0
                    else "INVALID"
                ),
            },
        ]
    )

    invalid_controls = controls.loc[
        controls["status"] != "VALID",
        "control",
    ].tolist()

    qa_report = {
        "schema_version": "1.0.0",
        "dataset_id": "SYNTHETIC-INVOICE-V1",
        "document_id": (
            document_information[
                "document_id"
            ]
        ),
        "canonical_invoice_id": (
            document_information[
                "canonical_invoice_id"
            ]
        ),
        "template_id": (
            document_information[
                "template_id"
            ]
        ),
        "status": (
            "PASSED"
            if not invalid_controls
            else "FAILED"
        ),
        "manual_visual_review": "PENDING",
        "metrics": {
            "page_count": page_count,
            "annotation_count": len(
                annotations
            ),
            "minimum_field_font_size": (
                minimum_field_font_size
            ),
            "font_adjustment_count": (
                font_adjustment_count
            ),
            "font_adjustment_ratio": round(
                font_adjustment_ratio,
                6,
            ),
            "invalid_bbox_count": len(
                invalid_bbox_fields
            ),
            "normalization_mismatch_count": len(
                normalization_mismatch_fields
            ),
            "annotation_without_words_count": len(
                annotations_without_words
            ),
            "severe_overlap_count": len(
                severe_overlap_pairs
            ),
            "ink_pixel_ratio": round(
                ink_pixel_ratio,
                6,
            ),
            "white_pixel_ratio": round(
                white_pixel_ratio,
                6,
            ),
            "raster_contrast": round(
                raster_contrast,
                6,
            ),
            "image_width_pixels": int(
                image_width
            ),
            "image_height_pixels": int(
                image_height
            ),
        },
        "checksums_sha256": {
            artifact_name: (
                calculate_artifact_sha256(
                    artifact_path
                )
            )
            for artifact_name, artifact_path
            in required_files.items()
        },
        "failures": {
            "invalid_controls": (
                invalid_controls
            ),
            "invalid_bbox_fields": (
                invalid_bbox_fields
            ),
            "normalization_mismatch_fields": (
                normalization_mismatch_fields
            ),
            "annotations_without_words": (
                annotations_without_words
            ),
            "severe_overlap_pairs": (
                severe_overlap_pairs
            ),
            "missing_extracted_values": (
                missing_extracted_values
            ),
            "missing_required_annotations": (
                missing_required_annotations
            ),
        },
    }

    return (
        qa_report,
        controls,
    )


# ================================================================
# Menjalankan audit TPL-02
# ================================================================

(
    prototype_02_qa_report,
    prototype_02_qa_controls,
) = audit_prototype_artifacts(
    pdf_path=prototype_02_pdf_path,
    preview_path=prototype_02_png_path,
    ground_truth_path=(
        prototype_02_ground_truth_path
    ),
)

prototype_02_qa_report_path = (
    TPL02_PROTOTYPE_ROOT
    / (
        f"{prototype_02_document_id}"
        "_TPL-02_qa_report.json"
    )
)

write_json_atomically(
    prototype_02_qa_report_path,
    prototype_02_qa_report,
)

display(
    prototype_02_qa_controls
)

if (
    prototype_02_qa_report["status"]
    != "PASSED"
):
    raise RuntimeError(
        "Prototype TPL-02 gagal audit teknis: "
        f"{prototype_02_qa_report['failures']}"
    )


# ================================================================
# Memperbarui build state
# ================================================================

with BUILD_STATE_PATH.open(
    mode="r",
    encoding="utf-8",
) as build_state_file:
    updated_build_state = json.load(
        build_state_file
    )

tpl02_artifacts = {
    "prototype_tpl_02_pdf": (
        prototype_02_pdf_path
    ),
    "prototype_tpl_02_preview": (
        prototype_02_png_path
    ),
    "prototype_tpl_02_ground_truth": (
        prototype_02_ground_truth_path
    ),
    "prototype_tpl_02_qa_report": (
        prototype_02_qa_report_path
    ),
}

for (
    artifact_name,
    artifact_path,
) in tpl02_artifacts.items():
    updated_build_state.setdefault(
        "artifacts",
        {},
    )[artifact_name] = {
        "relative_path": str(
            artifact_path.relative_to(
                PROJECT_DATA_ROOT
            )
        ),
        "size_bytes": int(
            artifact_path.stat().st_size
        ),
        "sha256": (
            calculate_artifact_sha256(
                artifact_path
            )
        ),
    }

updated_timestamp = datetime.now(
    timezone.utc
).isoformat()

updated_build_state[
    "build_status"
] = "PROTOTYPE_TPL_02_TECHNICAL_QA_PASSED"

updated_build_state[
    "last_completed_cell"
] = 44

updated_build_state[
    "last_completed_stage"
] = "prototype_tpl_02_technical_qa"

updated_build_state[
    "next_stage"
] = "prototype_tpl_02_manual_visual_review"

updated_build_state[
    "updated_utc"
] = updated_timestamp

updated_build_state.setdefault(
    "quality_gates",
    {},
)[
    "prototype_tpl_02_technical_qa"
] = "PASSED"

updated_build_state[
    "quality_gates"
][
    "prototype_tpl_02_manual_visual_review"
] = "PENDING"

write_json_atomically(
    BUILD_STATE_PATH,
    updated_build_state,
)

updated_build_state_checksum = (
    checkpoint_sha256(
        BUILD_STATE_PATH
    )
)

with ACTIVE_BUILD_POINTER_PATH.open(
    mode="r",
    encoding="utf-8",
) as pointer_file:
    updated_active_pointer = json.load(
        pointer_file
    )

updated_active_pointer[
    "build_state_sha256"
] = updated_build_state_checksum

updated_active_pointer[
    "build_status"
] = updated_build_state[
    "build_status"
]

updated_active_pointer[
    "updated_utc"
] = updated_timestamp

write_json_atomically(
    ACTIVE_BUILD_POINTER_PATH,
    updated_active_pointer,
)


# ================================================================
# Output
# ================================================================

print(
    f"QA report          : "
    f"{prototype_02_qa_report_path}"
)
print(
    f"QA status          : "
    f"{prototype_02_qa_report['status']}"
)
print(
    f"Minimum field font : "
    f"{prototype_02_qa_report['metrics']['minimum_field_font_size']}"
)
print(
    f"Font adjustments   : "
    f"{prototype_02_qa_report['metrics']['font_adjustment_count']}"
)
print(
    f"Ink pixel ratio    : "
    f"{prototype_02_qa_report['metrics']['ink_pixel_ratio']}"
)
print(
    f"Raster contrast    : "
    f"{prototype_02_qa_report['metrics']['raster_contrast']}"
)
print(
    "Manual review      : PENDING"
)
print()
print(
    "✅ Prototype TPL-02 lulus audit teknis. "
    "Review visual manual masih diperlukan."
)

In [ ]:
# CELL 44A — Finalisasi review visual TPL-02

import json


qa_report_path = prototype_02_qa_report_path

with qa_report_path.open("r", encoding="utf-8") as file:
    qa_report = json.load(file)

# Validasi identitas dan audit teknis
if qa_report.get("template_id") != "TPL-02":
    raise RuntimeError(
        "QA report bukan milik TPL-02."
    )

if qa_report.get("status") != "PASSED":
    raise RuntimeError(
        "Review visual tidak dapat disahkan karena "
        f"status teknis adalah {qa_report.get('status')!r}."
    )

current_review = qa_report.get(
    "manual_visual_review"
)

if current_review not in {"PENDING", "PASSED"}:
    raise RuntimeError(
        "Nilai manual_visual_review tidak dikenal: "
        f"{current_review!r}"
    )

# Review visual telah dilakukan dan dinyatakan lulus
qa_report["manual_visual_review"] = "PASSED"

write_json_atomically(
    qa_report_path,
    qa_report,
)

# Buka ulang untuk memastikan perubahan tersimpan
with qa_report_path.open("r", encoding="utf-8") as file:
    verified_report = json.load(file)

if (
    verified_report.get("manual_visual_review")
    != "PASSED"
):
    raise RuntimeError(
        "Status review visual gagal disimpan."
    )

print(f"QA report           : {qa_report_path}")
print(f"Technical status    : {verified_report['status']}")
print(
    "Manual visual review: "
    f"{verified_report['manual_visual_review']}"
)
print()
print(
    "✅ TPL-02 FINAL PASSED — audit teknis dan "
    "review visual telah selesai."
)

**CELL 46 — Prototype renderer TPL-03: Split Header**

In [ ]:
# ================================================================
# CELL 46 — Prototype renderer TPL-03: Split Header
# ================================================================

from pathlib import Path
from typing import Any
from PIL import Image as PILImage

import json
import os

import pandas as pd
import pymupdf


# ================================================================
# Validasi runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "PROTOTYPE_ROOT",
    "CANONICAL_RECORDS_JSONL_PATH",
    "FieldAnnotation",
    "template_registry",
    "invoice_render_payloads",
    "get_page_dimensions",
    "hex_to_rgb",
    "insert_annotated_text",
    "insert_static_text",
    "annotation_to_dict",
    "write_json_atomically",
    "RENDER_ENGINE_VERSION",
]

missing_runtime_objects = [
    name
    for name in REQUIRED_RUNTIME_OBJECTS
    if name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Runtime TPL-03 belum lengkap: "
        f"{missing_runtime_objects}"
    )


# ================================================================
# Konfigurasi
# ================================================================

TPL03_TEMPLATE_ID = "TPL-03"
TPL03_RENDERER_VERSION = "1.1.0"
TPL03_MAX_ITEMS_PER_PAGE = 7

TPL03_PROTOTYPE_ROOT = (
    Path(PROTOTYPE_ROOT)
    / TPL03_TEMPLATE_ID
)

TPL03_PROTOTYPE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ================================================================
# Helper canonical payload
# ================================================================

def load_tpl03_canonical_payload(
    canonical_id: str,
) -> dict[str, Any]:

    canonical_path = Path(
        CANONICAL_RECORDS_JSONL_PATH
    )

    if not canonical_path.is_file():
        raise FileNotFoundError(
            f"Canonical JSONL tidak ditemukan: {canonical_path}"
        )

    with canonical_path.open(
        "r",
        encoding="utf-8",
    ) as canonical_file:

        for line_number, line in enumerate(
            canonical_file,
            start=1,
        ):
            if not line.strip():
                continue

            try:
                payload = json.loads(line)
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    "Canonical JSONL rusak pada "
                    f"baris {line_number}."
                ) from error

            if (
                payload.get("canonical_invoice_id")
                == canonical_id
            ):
                return payload

    raise KeyError(
        f"Canonical payload tidak ditemukan: {canonical_id}"
    )


def normalize_pdf_notice(text: Any) -> str:
    """Mengganti separator yang tidak didukung font PDF bawaan."""

    return (
        str(text)
        .replace("—", "·")
        .replace("–", "-")
    )


# ================================================================
# Renderer
# ================================================================

def render_template_03(
    render_payload: dict[str, Any],
    output_pdf_path: Path,
) -> tuple[list[FieldAnnotation], dict[str, Any]]:

    if render_payload.get("template_id") != TPL03_TEMPLATE_ID:
        raise ValueError(
            "Renderer TPL-03 menerima payload TPL-03 saja."
        )

    required_keys = {
        "canonical_invoice_id",
        "document_id",
        "template_id",
        "language",
        "currency",
        "title",
        "labels",
        "metadata",
        "vendor",
        "buyer",
        "items",
        "financials",
        "footer",
        "synthetic_notice",
    }

    missing_keys = sorted(
        required_keys - set(render_payload)
    )

    if missing_keys:
        raise KeyError(
            f"Payload TPL-03 belum lengkap: {missing_keys}"
        )

    item_count = len(
        render_payload["items"]
    )

    if not 1 <= item_count <= TPL03_MAX_ITEMS_PER_PAGE:
        raise ValueError(
            "TPL-03 hanya mendukung 1–7 item per halaman."
        )

    template_specification = template_registry[
        TPL03_TEMPLATE_ID
    ]

    page_width, page_height = get_page_dimensions(
        template_specification["page_size"]
    )

    primary_color = hex_to_rgb(
        template_specification["primary_color"]
    )

    accent_color = hex_to_rgb(
        template_specification["accent_color"]
    )

    dark_text = (0.16, 0.17, 0.22)
    muted_text = (0.43, 0.45, 0.50)
    border_color = (0.84, 0.85, 0.88)
    light_primary = (0.965, 0.968, 0.978)
    light_accent = (0.992, 0.955, 0.940)
    white = (1.0, 1.0, 1.0)

    output_pdf_path = Path(output_pdf_path)

    output_pdf_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_pdf_path = output_pdf_path.with_name(
        f".{output_pdf_path.stem}.tmp.pdf"
    )

    temporary_pdf_path.unlink(
        missing_ok=True
    )

    annotations: list[FieldAnnotation] = []
    field_font_audit: list[dict[str, Any]] = []

    document = pymupdf.open()

    def add_field(
        page: pymupdf.Page,
        rectangle: pymupdf.Rect,
        text: str,
        field_name: str,
        font_name: str = "helv",
        font_size: float = 7.4,
        minimum_font_size: float = 6.5,
        font_color: tuple[float, float, float] = dark_text,
        alignment: str = "left",
    ) -> None:

        annotation, used_font_size = insert_annotated_text(
            page=page,
            rectangle=rectangle,
            text=str(text),
            field_name=field_name,
            font_name=font_name,
            font_size=font_size,
            minimum_font_size=minimum_font_size,
            font_color=font_color,
            alignment=alignment,
        )

        annotations.append(annotation)

        field_font_audit.append(
            {
                "field_name": field_name,
                "requested_font_size": font_size,
                "used_font_size": used_font_size,
                "adjusted": used_font_size < font_size,
            }
        )

    try:
        page = document.new_page(
            width=page_width,
            height=page_height,
        )

        # ============================================================
        # Header accent
        # ============================================================

        page.draw_rect(
            pymupdf.Rect(
                0,
                0,
                page_width,
                12,
            ),
            color=accent_color,
            fill=accent_color,
            width=0,
            overlay=True,
        )

        page.draw_rect(
            pymupdf.Rect(
                45,
                34,
                53,
                132,
            ),
            color=accent_color,
            fill=accent_color,
            width=0,
            overlay=True,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                68,
                39,
                310,
                78,
            ),
            text=render_payload["title"],
            font_name="hebo",
            font_size=23.0,
            minimum_font_size=18.0,
            font_color=primary_color,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                68,
                85,
                310,
                101,
            ),
            text=render_payload[
                "footer"
            ]["document_reference"],
            font_name="helv",
            font_size=7.2,
            minimum_font_size=6.0,
            font_color=muted_text,
        )

        currency_label = (
            "Mata Uang"
            if render_payload["language"] == "id"
            else "Currency"
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                68,
                108,
                125,
                124,
            ),
            text=currency_label,
            font_name="helv",
            font_size=6.8,
            minimum_font_size=6.0,
            font_color=muted_text,
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                127,
                106,
                188,
                126,
            ),
            text=render_payload["currency"],
            field_name="currency",
            font_name="hebo",
            font_size=8.2,
            minimum_font_size=6.5,
            font_color=accent_color,
        )

        # ============================================================
        # Metadata kanan
        # ============================================================

        metadata_rows = [
            ("invoice_number", 34, 56),
            ("invoice_date", 63, 85),
            ("due_date", 92, 114),
        ]

        for field_name, row_y0, row_y1 in metadata_rows:
            field_payload = render_payload[
                "metadata"
            ][field_name]

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    326,
                    row_y0,
                    417,
                    row_y1,
                ),
                text=field_payload["label"],
                font_name="helv",
                font_size=6.9,
                minimum_font_size=6.0,
                font_color=muted_text,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    419,
                    row_y0,
                    page_width - 45,
                    row_y1,
                ),
                text=field_payload["display_value"],
                field_name=field_name,
                font_name="hebo",
                font_size=7.8,
                minimum_font_size=6.5,
                font_color=dark_text,
                alignment="right",
            )

        page.draw_line(
            pymupdf.Point(45, 145),
            pymupdf.Point(page_width - 45, 145),
            color=border_color,
            width=0.8,
            overlay=True,
        )

        # ============================================================
        # Vendor dan buyer — dua kolom
        # ============================================================

        party_cards = [
            (
                "vendor",
                render_payload["labels"]["vendor"],
                45.0,
                299.0,
            ),
            (
                "buyer",
                render_payload["labels"]["buyer"],
                313.0,
                page_width - 45.0,
            ),
        ]

        party_y0 = 161.0
        party_y1 = 294.0

        for (
            party_type,
            party_label,
            card_x0,
            card_x1,
        ) in party_cards:

            party = render_payload[party_type]

            page.draw_rect(
                pymupdf.Rect(
                    card_x0,
                    party_y0,
                    card_x1,
                    party_y1,
                ),
                color=border_color,
                fill=white,
                width=0.7,
                overlay=True,
            )

            page.draw_rect(
                pymupdf.Rect(
                    card_x0,
                    party_y0,
                    card_x1,
                    party_y0 + 24,
                ),
                color=None,
                fill=light_accent,
                width=0,
                overlay=True,
            )

            page.draw_rect(
                pymupdf.Rect(
                    card_x0,
                    party_y0,
                    card_x0 + 5,
                    party_y0 + 24,
                ),
                color=accent_color,
                fill=accent_color,
                width=0,
                overlay=True,
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 14,
                    party_y0 + 6,
                    card_x1 - 10,
                    party_y0 + 20,
                ),
                text=party_label.upper(),
                font_name="hebo",
                font_size=6.8,
                minimum_font_size=6.0,
                font_color=accent_color,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 14,
                    party_y0 + 31,
                    card_x1 - 12,
                    party_y0 + 48,
                ),
                text=party["name"],
                field_name=f"{party_type}.name",
                font_name="hebo",
                font_size=8.0,
                minimum_font_size=6.5,
                font_color=primary_color,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 14,
                    party_y0 + 51,
                    card_x1 - 12,
                    party_y0 + 86,
                ),
                text="\n".join(
                    party["address_lines"]
                ),
                field_name=f"{party_type}.address_lines",
                font_name="helv",
                font_size=6.8,
                minimum_font_size=6.5,
                font_color=dark_text,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 14,
                    party_y0 + 88,
                    card_x1 - 12,
                    party_y0 + 101,
                ),
                text=party["email"],
                field_name=f"{party_type}.email",
                font_name="helv",
                font_size=6.7,
                minimum_font_size=6.5,
                font_color=dark_text,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 14,
                    party_y0 + 103,
                    card_x1 - 12,
                    party_y0 + 116,
                ),
                text=party["phone"],
                field_name=f"{party_type}.phone",
                font_name="helv",
                font_size=6.7,
                minimum_font_size=6.5,
                font_color=dark_text,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 14,
                    party_y0 + 118,
                    card_x1 - 12,
                    party_y0 + 131,
                ),
                text=party["tax_identifier_display"],
                field_name=f"{party_type}.tax_identifier",
                font_name="helv",
                font_size=6.7,
                minimum_font_size=6.5,
                font_color=dark_text,
            )

        # ============================================================
        # Tabel item — striped
        # ============================================================

        table_positions = [
            45.0,
            70.0,
            310.0,
            365.0,
            460.0,
            page_width - 45.0,
        ]

        table_header_y0 = 304.0
        table_header_y1 = 334.0
        row_height = 25.0

        page.draw_rect(
            pymupdf.Rect(
                table_positions[0],
                table_header_y0,
                table_positions[-1],
                table_header_y1,
            ),
            color=None,
            fill=primary_color,
            width=0,
            overlay=True,
        )

        table_headers = [
            "#",
            render_payload["labels"]["description"],
            render_payload["labels"]["quantity"],
            render_payload["labels"]["unit_price"],
            render_payload["labels"]["line_total"],
        ]

        for column_index, header_text in enumerate(
            table_headers
        ):
            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    table_positions[column_index] + 3,
                    table_header_y0 + 8,
                    table_positions[column_index + 1] - 3,
                    table_header_y1 - 3,
                ),
                text=header_text,
                font_name="hebo",
                font_size=6.8,
                minimum_font_size=6.0,
                font_color=white,
                alignment=(
                    "left"
                    if column_index == 1
                    else "center"
                ),
            )

        for item_index, item in enumerate(
            render_payload["items"]
        ):
            row_y0 = (
                table_header_y1
                + item_index * row_height
            )
            row_y1 = row_y0 + row_height

            if item_index % 2 == 1:
                page.draw_rect(
                    pymupdf.Rect(
                        table_positions[0],
                        row_y0,
                        table_positions[-1],
                        row_y1,
                    ),
                    color=None,
                    fill=light_primary,
                    width=0,
                    overlay=True,
                )

            page.draw_line(
                pymupdf.Point(
                    table_positions[0],
                    row_y1,
                ),
                pymupdf.Point(
                    table_positions[-1],
                    row_y1,
                ),
                color=border_color,
                width=0.45,
                overlay=True,
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    table_positions[0] + 3,
                    row_y0 + 6,
                    table_positions[1] - 3,
                    row_y1 - 3,
                ),
                text=str(item["item_number"]),
                font_name="helv",
                font_size=7.0,
                minimum_font_size=6.0,
                font_color=muted_text,
                alignment="center",
            )

            item_fields = [
                ("description", 1, "left"),
                ("quantity", 2, "center"),
                ("unit_price", 3, "right"),
                ("line_total", 4, "right"),
            ]

            for (
                field_name,
                column_index,
                alignment,
            ) in item_fields:

                add_field(
                    page=page,
                    rectangle=pymupdf.Rect(
                        table_positions[column_index] + 4,
                        row_y0 + 6,
                        table_positions[column_index + 1] - 4,
                        row_y1 - 3,
                    ),
                    text=item[field_name],
                    field_name=(
                        f"items[{item_index}].{field_name}"
                    ),
                    font_name="helv",
                    font_size=7.0,
                    minimum_font_size=6.5,
                    font_color=dark_text,
                    alignment=alignment,
                )

        table_end_y = (
            table_header_y1
            + item_count * row_height
        )

        # ============================================================
        # Bottom totals card
        # ============================================================

        totals_panel_height = 108.0

        totals_y0 = min(
            max(
                table_end_y + 38.0,
                500.0,
            ),
            535.0,
        )

        totals_y1 = (
            totals_y0
            + totals_panel_height
        )

        table_to_totals_gap = (
            totals_y0 - table_end_y
        )

        if table_to_totals_gap < 20.0:
            raise RuntimeError(
                "Tabel bertabrakan dengan kartu total. "
                f"Gap: {table_to_totals_gap:.2f} pt."
            )

        if totals_y1 >= 700.0:
            raise RuntimeError(
                "Kartu total terlalu dekat dengan footer. "
                f"Berakhir pada {totals_y1:.2f} pt."
            )

        page.draw_rect(
            pymupdf.Rect(
                45,
                totals_y0,
                page_width - 45,
                totals_y1,
            ),
            color=border_color,
            fill=light_accent,
            width=0.7,
            overlay=True,
        )

        page.draw_rect(
            pymupdf.Rect(
                45,
                totals_y0,
                page_width - 45,
                totals_y0 + 5,
            ),
            color=accent_color,
            fill=accent_color,
            width=0,
            overlay=True,
        )

        total_payload = render_payload[
            "financials"
        ]["total"]

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                62,
                totals_y0 + 25,
                285,
                totals_y0 + 44 + 44,
            ),
            text=total_payload["label"].upper(),
            font_name="hebo",
            font_size=7.2,
            minimum_font_size=6.0,
            font_color=muted_text,
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                62,
                totals_y0 + 48,
                300,
                totals_y0 + 79,
            ),
            text=total_payload["display_value"],
            field_name="financials.total",
            font_name="hebo",
            font_size=13.0,
            minimum_font_size=8.0,
            font_color=accent_color,
        )

        page.draw_line(
            pymupdf.Point(
                318,
                totals_y0 + 18,
            ),
            pymupdf.Point(
                318,
                totals_y1 - 18,
            ),
            color=border_color,
            width=0.7,
            overlay=True,
        )

        summary_fields = [
            "subtotal",
            "tax",
            "discount",
        ]

        for row_index, field_name in enumerate(
            summary_fields
        ):
            field_payload = render_payload[
                "financials"
            ][field_name]

            row_y0 = (
                totals_y0
                + 19
                + row_index * 26
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    337,
                    row_y0,
                    415,
                    row_y0 + 18,
                ),
                text=field_payload["label"],
                font_name="helv",
                font_size=7.3,
                minimum_font_size=6.0,
                font_color=muted_text,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    417,
                    row_y0,
                    page_width - 61,
                    row_y0 + 18,
                ),
                text=field_payload["display_value"],
                field_name=f"financials.{field_name}",
                font_name="helv",
                font_size=7.5,
                minimum_font_size=6.5,
                font_color=dark_text,
                alignment="right",
            )

        # ============================================================
        # Footer
        # ============================================================

        footer_line_y = 716.0

        page.draw_line(
            pymupdf.Point(45, footer_line_y),
            pymupdf.Point(
                page_width - 45,
                footer_line_y,
            ),
            color=border_color,
            width=0.7,
            overlay=True,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                45,
                724,
                page_width - 45,
                739,
            ),
            text=render_payload[
                "footer"
            ]["document_reference"],
            font_name="helv",
            font_size=6.6,
            minimum_font_size=6.0,
            font_color=muted_text,
            alignment="center",
        )

        notice_display_text = normalize_pdf_notice(
            render_payload[
                "synthetic_notice"
            ]
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                45,
                748,
                page_width - 45,
                766,
            ),
            text=notice_display_text,
            field_name="document.synthetic_notice",
            font_name="hebo",
            font_size=6.8,
            minimum_font_size=6.5,
            font_color=accent_color,
            alignment="center",
        )

        # ============================================================
        # Validasi sebelum simpan
        # ============================================================

        annotation_names = [
            annotation.field_name
            for annotation in annotations
        ]

        if len(annotation_names) != len(
            set(annotation_names)
        ):
            raise RuntimeError(
                "Ditemukan nama anotasi duplikat."
            )

        expected_annotation_count = (
            19 + 4 * item_count
        )

        if (
            len(annotations)
            != expected_annotation_count
        ):
            raise RuntimeError(
                "Jumlah anotasi tidak sesuai. "
                f"Expected: {expected_annotation_count}; "
                f"actual: {len(annotations)}."
            )

        minimum_used_font_size = min(
            row["used_font_size"]
            for row in field_font_audit
        )

        if minimum_used_font_size < 6.5:
            raise RuntimeError(
                "Ukuran font field berada di bawah 6.5 pt."
            )

        document.save(
            str(temporary_pdf_path),
            garbage=4,
            deflate=True,
        )

    except Exception:
        temporary_pdf_path.unlink(
            missing_ok=True
        )
        raise

    finally:
        document.close()

    if (
        not temporary_pdf_path.is_file()
        or temporary_pdf_path.stat().st_size == 0
    ):
        temporary_pdf_path.unlink(
            missing_ok=True
        )
        raise RuntimeError(
            "PDF sementara gagal dibuat."
        )

    os.replace(
        temporary_pdf_path,
        output_pdf_path,
    )

    rendering_metadata = {
        "renderer_version": RENDER_ENGINE_VERSION,
        "template_renderer_version": (
            TPL03_RENDERER_VERSION
        ),
        "template_id": TPL03_TEMPLATE_ID,
        "layout_family": template_specification[
            "layout_family"
        ],
        "page_size": template_specification[
            "page_size"
        ],
        "page_width_points": round(
            page_width,
            3,
        ),
        "page_height_points": round(
            page_height,
            3,
        ),
        "page_count": 1,
        "annotation_type": "field_region",
        "annotation_count": len(annotations),
        "minimum_field_font_size": min(
            row["used_font_size"]
            for row in field_font_audit
        ),
        "font_adjustment_count": sum(
            bool(row["adjusted"])
            for row in field_font_audit
        ),
        "layout_metrics": {
            "item_count": item_count,
            "table_end_y_points": round(
                table_end_y,
                3,
            ),
            "totals_panel_y0_points": totals_y0,
            "totals_panel_y1_points": totals_y1,
            "table_to_totals_gap_points": round(
                table_to_totals_gap,
                3,
            ),
            "footer_line_y_points": footer_line_y,
        },
    }

    return annotations, rendering_metadata


# ================================================================
# Pilih prototype TPL-03
# ================================================================

tpl03_candidate_ids = sorted(
    canonical_id
    for canonical_id, payload
    in invoice_render_payloads.items()
    if payload.get("template_id") == TPL03_TEMPLATE_ID
)

if len(tpl03_candidate_ids) != 20:
    raise RuntimeError(
        "TPL-03 harus memiliki tepat 20 payload. "
        f"Actual: {len(tpl03_candidate_ids)}."
    )

prototype_03_canonical_id = (
    tpl03_candidate_ids[0]
)

prototype_03_render_payload = (
    invoice_render_payloads[
        prototype_03_canonical_id
    ]
)

prototype_03_document_id = (
    prototype_03_render_payload[
        "document_id"
    ]
)

prototype_03_canonical_payload = (
    load_tpl03_canonical_payload(
        prototype_03_canonical_id
    )
)


# ================================================================
# Path artifact
# ================================================================

prototype_03_pdf_path = (
    TPL03_PROTOTYPE_ROOT
    / (
        f"{prototype_03_document_id}"
        "_TPL-03_prototype.pdf"
    )
)

prototype_03_png_path = (
    TPL03_PROTOTYPE_ROOT
    / (
        f"{prototype_03_document_id}"
        "_TPL-03_preview.png"
    )
)

prototype_03_ground_truth_path = (
    TPL03_PROTOTYPE_ROOT
    / (
        f"{prototype_03_document_id}"
        "_TPL-03_ground_truth.json"
    )
)


# ================================================================
# Render
# ================================================================

(
    prototype_03_annotations,
    prototype_03_rendering_metadata,
) = render_template_03(
    render_payload=prototype_03_render_payload,
    output_pdf_path=prototype_03_pdf_path,
)


# ================================================================
# Simpan ground truth
# ================================================================

prototype_03_ground_truth = {
    "schema_version": "1.0.0",
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "document": {
        "document_id": prototype_03_document_id,
        "canonical_invoice_id": (
            prototype_03_canonical_id
        ),
        "split": prototype_03_render_payload[
            "split"
        ],
        "template_id": TPL03_TEMPLATE_ID,
        "language": prototype_03_render_payload[
            "language"
        ],
        "currency": prototype_03_render_payload[
            "currency"
        ],
    },
    "canonical": prototype_03_canonical_payload,
    "rendering": (
        prototype_03_rendering_metadata
    ),
    "annotations": [
        annotation_to_dict(annotation)
        for annotation
        in prototype_03_annotations
    ],
}

write_json_atomically(
    prototype_03_ground_truth_path,
    prototype_03_ground_truth,
)


# ================================================================
# Buat preview PNG
# ================================================================

temporary_png_path = (
    prototype_03_png_path.with_name(
        f".{prototype_03_png_path.stem}.tmp.png"
    )
)

temporary_png_path.unlink(
    missing_ok=True
)

try:
    with pymupdf.open(
        str(prototype_03_pdf_path)
    ) as prototype_document:

        if prototype_document.page_count != 1:
            raise RuntimeError(
                "Prototype TPL-03 harus satu halaman."
            )

        prototype_page = prototype_document[0]

        extracted_text_03 = (
            prototype_page.get_text("text")
        )

        prototype_pixmap = (
            prototype_page.get_pixmap(
                matrix=pymupdf.Matrix(
                    150 / 72,
                    150 / 72,
                ),
                alpha=False,
            )
        )

        prototype_pixmap.save(
            str(temporary_png_path)
        )

    if (
        not temporary_png_path.is_file()
        or temporary_png_path.stat().st_size == 0
    ):
        raise RuntimeError(
            "Preview PNG gagal dibuat."
        )

    os.replace(
        temporary_png_path,
        prototype_03_png_path,
    )

except Exception:
    temporary_png_path.unlink(
        missing_ok=True
    )
    raise


# ================================================================
# Pemeriksaan awal
# ================================================================

required_text_fragments = [
    prototype_03_render_payload[
        "metadata"
    ]["invoice_number"]["display_value"],
    prototype_03_render_payload[
        "vendor"
    ]["name"],
    prototype_03_render_payload[
        "buyer"
    ]["name"],
    prototype_03_render_payload[
        "labels"
    ]["quantity"],
    prototype_03_render_payload[
        "financials"
    ]["total"]["display_value"],
    normalize_pdf_notice(
        prototype_03_render_payload[
            "synthetic_notice"
        ]
    ),
]

missing_text_fragments = [
    text
    for text in required_text_fragments
    if text not in extracted_text_03
]

if missing_text_fragments:
    raise RuntimeError(
        "Teks penting hilang dari PDF: "
        f"{missing_text_fragments}"
    )

layout_metrics = (
    prototype_03_rendering_metadata[
        "layout_metrics"
    ]
)

prototype_03_summary = pd.DataFrame(
    [
        {
            "control": "pdf_created",
            "actual": prototype_03_pdf_path.is_file(),
            "status": "VALID",
        },
        {
            "control": "ground_truth_created",
            "actual": (
                prototype_03_ground_truth_path.is_file()
            ),
            "status": "VALID",
        },
        {
            "control": "annotation_count",
            "actual": len(
                prototype_03_annotations
            ),
            "status": "VALID",
        },
        {
            "control": "item_count",
            "actual": layout_metrics[
                "item_count"
            ],
            "status": "VALID",
        },
        {
            "control": "minimum_field_font",
            "actual": (
                prototype_03_rendering_metadata[
                    "minimum_field_font_size"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "font_adjustments",
            "actual": (
                prototype_03_rendering_metadata[
                    "font_adjustment_count"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "table_to_totals_gap",
            "actual": layout_metrics[
                "table_to_totals_gap_points"
            ],
            "status": "VALID",
        },
        {
            "control": "missing_required_text",
            "actual": len(
                missing_text_fragments
            ),
            "status": "VALID",
        },
    ]
)

display(prototype_03_summary)

with PILImage.open(
    prototype_03_png_path
) as prototype_image:
    display(prototype_image.copy())


# ================================================================
# Ringkasan
# ================================================================

print(
    f"Canonical ID       : {prototype_03_canonical_id}"
)
print(
    f"Document ID        : {prototype_03_document_id}"
)
print(
    f"Prototype PDF      : {prototype_03_pdf_path}"
)
print(
    f"Prototype preview  : {prototype_03_png_path}"
)
print(
    f"Ground truth       : {prototype_03_ground_truth_path}"
)
print(
    f"Annotations        : {len(prototype_03_annotations)}"
)
print(
    "Minimum field font : "
    f"{prototype_03_rendering_metadata['minimum_field_font_size']}"
)
print(
    "Adjusted fields    : "
    f"{prototype_03_rendering_metadata['font_adjustment_count']}"
)
print(
    "Table-total gap    : "
    f"{layout_metrics['table_to_totals_gap_points']} pt"
)
print(
    f"Renderer version   : {TPL03_RENDERER_VERSION}"
)
print()
print(
    "✅ Prototype TPL-03 berhasil dibuat dan "
    "lolos pemeriksaan teknis awal."
)


**CELL 47 — FINAL Audit QA teknis prototype TPL-03**

In [ ]:
# ================================================================
# CELL 47 — FINAL
# Audit QA teknis prototype TPL-03
# ================================================================

from pathlib import Path
from typing import Any

import hashlib
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd
import pymupdf
from PIL import Image as PILImage


# ================================================================
# Validasi dependency runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "TPL03_PROTOTYPE_ROOT",
    "TPL03_TEMPLATE_ID",
    "prototype_03_canonical_id",
    "prototype_03_document_id",
    "prototype_03_pdf_path",
    "prototype_03_png_path",
    "prototype_03_ground_truth_path",
    "prototype_03_render_payload",
    "prototype_03_rendering_metadata",
    "prototype_03_annotations",
    "annotation_to_dict",
    "write_json_atomically",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 47 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali Cell 46 versi final."
    )


# ================================================================
# Konfigurasi audit
# ================================================================

QA_SCHEMA_VERSION = "1.0.0"
QA_AUDITOR_VERSION = "1.0.0"

MINIMUM_FIELD_FONT_SIZE = 6.5
MAXIMUM_FONT_ADJUSTMENT_RATIO = 0.20
MINIMUM_RASTER_CONTENT_RATIO = 0.01
MAXIMUM_RASTER_CONTENT_RATIO = 0.40
MINIMUM_RASTER_CONTRAST = 10.0
SEVERE_OVERLAP_THRESHOLD = 0.50

prototype_03_pdf_path = Path(
    prototype_03_pdf_path
)
prototype_03_png_path = Path(
    prototype_03_png_path
)
prototype_03_ground_truth_path = Path(
    prototype_03_ground_truth_path
)

prototype_03_qa_report_path = (
    Path(TPL03_PROTOTYPE_ROOT)
    / (
        f"{prototype_03_document_id}"
        "_TPL-03_qa_report.json"
    )
)

required_files = [
    prototype_03_pdf_path,
    prototype_03_png_path,
    prototype_03_ground_truth_path,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Artifact TPL-03 belum lengkap: "
        f"{missing_files}"
    )


# ================================================================
# Helper umum
# ================================================================

def sha256_file(file_path: Path) -> str:
    digest = hashlib.sha256()

    with Path(file_path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def normalize_text(value: Any) -> str:
    text = unicodedata.normalize(
        "NFKC",
        str(value or ""),
    )

    text = (
        text
        .replace("\u2014", "-")
        .replace("\u2013", "-")
        .replace("\u00b7", "-")
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip().casefold()


def normalize_notice_for_pdf(value: Any) -> str:
    return (
        str(value)
        .replace("\u2014", "\u00b7")
        .replace("\u2013", "-")
    )


def first_present(
    mapping: dict[str, Any],
    candidate_keys: tuple[str, ...],
) -> Any:
    for key in candidate_keys:
        if key in mapping:
            return mapping[key]

    return None


def annotation_field_name(
    annotation: dict[str, Any],
) -> str:
    value = first_present(
        annotation,
        (
            "field_name",
            "field_path",
            "field",
            "path",
            "name",
        ),
    )

    if value is None:
        raise KeyError(
            "Nama field tidak ditemukan pada anotasi. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return str(value)


def annotation_text(
    annotation: dict[str, Any],
) -> str:
    value = first_present(
        annotation,
        (
            "text",
            "value",
            "display_value",
            "field_value",
            "raw_value",
            "raw_text",
            "text_value",
            "content",
        ),
    )

    if isinstance(value, dict):
        value = first_present(
            value,
            (
                "text",
                "value",
                "display_value",
                "raw",
            ),
        )

    if value is None:
        raise KeyError(
            "Nilai teks tidak ditemukan pada anotasi. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return str(value)


def bbox_from_value(
    value: Any,
) -> tuple[float, float, float, float] | None:
    if isinstance(value, (list, tuple)):
        if len(value) != 4:
            return None

        try:
            return tuple(
                float(number)
                for number in value
            )
        except (TypeError, ValueError):
            return None

    if not isinstance(value, dict):
        return None

    direct_key_sets = [
        ("x0", "y0", "x1", "y1"),
        ("left", "top", "right", "bottom"),
    ]

    for key_set in direct_key_sets:
        if all(key in value for key in key_set):
            try:
                return tuple(
                    float(value[key])
                    for key in key_set
                )
            except (TypeError, ValueError):
                return None

    if all(
        key in value
        for key in ("x", "y", "width", "height")
    ):
        try:
            x0 = float(value["x"])
            y0 = float(value["y"])
            width = float(value["width"])
            height = float(value["height"])

            return (
                x0,
                y0,
                x0 + width,
                y0 + height,
            )
        except (TypeError, ValueError):
            return None

    if all(
        key in value
        for key in ("x", "y", "w", "h")
    ):
        try:
            x0 = float(value["x"])
            y0 = float(value["y"])
            width = float(value["w"])
            height = float(value["h"])

            return (
                x0,
                y0,
                x0 + width,
                y0 + height,
            )
        except (TypeError, ValueError):
            return None

    return None


def annotation_bbox(
    annotation: dict[str, Any],
) -> tuple[float, float, float, float]:
    candidate = first_present(
        annotation,
        (
            "bbox",
            "bbox_points",
            "bbox_pt",
            "bbox_pdf",
            "bounding_box",
            "rectangle",
            "rect",
            "coordinates",
            "coordinates_points",
        ),
    )

    if candidate is None:
        geometry = annotation.get("geometry")

        if isinstance(geometry, dict):
            candidate = first_present(
                geometry,
                (
                    "bbox",
                    "bounding_box",
                    "rectangle",
                ),
            )

    bbox = bbox_from_value(candidate)

    if bbox is None:
        raise KeyError(
            "Bounding box tidak ditemukan atau tidak valid. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return bbox


def rectangle_area(
    bbox: tuple[float, float, float, float],
) -> float:
    x0, y0, x1, y1 = bbox

    return max(0.0, x1 - x0) * max(
        0.0,
        y1 - y0,
    )


def intersection_area(
    first_bbox: tuple[float, float, float, float],
    second_bbox: tuple[float, float, float, float],
) -> float:
    first_x0, first_y0, first_x1, first_y1 = first_bbox
    second_x0, second_y0, second_x1, second_y1 = second_bbox

    width = max(
        0.0,
        min(first_x1, second_x1)
        - max(first_x0, second_x0),
    )

    height = max(
        0.0,
        min(first_y1, second_y1)
        - max(first_y0, second_y0),
    )

    return width * height


def bbox_intersects_word(
    annotation_box: tuple[float, float, float, float],
    word_box: tuple[float, float, float, float],
) -> bool:
    return intersection_area(
        annotation_box,
        word_box,
    ) > 0.0


# ================================================================
# Muat ground truth
# ================================================================

with prototype_03_ground_truth_path.open(
    "r",
    encoding="utf-8",
) as ground_truth_file:
    ground_truth = json.load(
        ground_truth_file
    )

ground_truth_document = ground_truth.get(
    "document",
    {},
)

if (
    ground_truth_document.get("document_id")
    != prototype_03_document_id
):
    raise RuntimeError(
        "Document ID ground truth tidak sesuai."
    )

if (
    ground_truth_document.get("template_id")
    != TPL03_TEMPLATE_ID
):
    raise RuntimeError(
        "Ground truth bukan milik TPL-03."
    )

annotation_records = ground_truth.get(
    "annotations",
    [],
)

if not isinstance(annotation_records, list):
    raise TypeError(
        "ground_truth.annotations wajib berupa list."
    )

if not annotation_records:
    raise RuntimeError(
        "Ground truth tidak memiliki anotasi."
    )


# ================================================================
# Expected field mapping
# ================================================================

render_payload = prototype_03_render_payload

expected_values: dict[str, str] = {
    "invoice_number": str(
        render_payload["metadata"]
        ["invoice_number"]
        ["display_value"]
    ),
    "invoice_date": str(
        render_payload["metadata"]
        ["invoice_date"]
        ["display_value"]
    ),
    "due_date": str(
        render_payload["metadata"]
        ["due_date"]
        ["display_value"]
    ),
    "currency": str(
        render_payload["currency"]
    ),
}

for party_type in ("vendor", "buyer"):
    party = render_payload[party_type]

    expected_values.update(
        {
            f"{party_type}.name": str(
                party["name"]
            ),
            f"{party_type}.address_lines": "\n".join(
                party["address_lines"]
            ),
            f"{party_type}.email": str(
                party["email"]
            ),
            f"{party_type}.phone": str(
                party["phone"]
            ),
            f"{party_type}.tax_identifier": str(
                party["tax_identifier_display"]
            ),
        }
    )

for item_index, item in enumerate(
    render_payload["items"]
):
    for field_name in (
        "description",
        "quantity",
        "unit_price",
        "line_total",
    ):
        expected_values[
            f"items[{item_index}].{field_name}"
        ] = str(item[field_name])

for field_name in (
    "subtotal",
    "tax",
    "discount",
    "total",
):
    expected_values[
        f"financials.{field_name}"
    ] = str(
        render_payload["financials"]
        [field_name]
        ["display_value"]
    )

expected_values[
    "document.synthetic_notice"
] = normalize_notice_for_pdf(
    render_payload["synthetic_notice"]
)

expected_annotation_count = len(
    expected_values
)


# ================================================================
# Parse annotations
# ================================================================

parsed_annotations: list[dict[str, Any]] = []
annotation_parse_failures: list[dict[str, Any]] = []

for annotation_index, annotation in enumerate(
    annotation_records
):
    try:
        parsed_annotations.append(
            {
                "index": annotation_index,
                "field_name": annotation_field_name(
                    annotation
                ),
                "text": annotation_text(
                    annotation
                ),
                "bbox": annotation_bbox(
                    annotation
                ),
            }
        )
    except (KeyError, TypeError, ValueError) as error:
        annotation_parse_failures.append(
            {
                "index": annotation_index,
                "error": str(error),
                "keys": sorted(
                    annotation.keys()
                ) if isinstance(
                    annotation,
                    dict,
                ) else [],
            }
        )

if annotation_parse_failures:
    raise RuntimeError(
        "Schema anotasi tidak dapat diproses: "
        f"{annotation_parse_failures[:3]}"
    )

annotation_names = [
    annotation["field_name"]
    for annotation in parsed_annotations
]

annotation_by_name = {
    annotation["field_name"]: annotation
    for annotation in parsed_annotations
}

duplicate_annotation_count = (
    len(annotation_names)
    - len(set(annotation_names))
)

missing_required_annotations = sorted(
    set(expected_values)
    - set(annotation_names)
)


# ================================================================
# Audit PDF dan anotasi
# ================================================================

with pymupdf.open(
    str(prototype_03_pdf_path)
) as pdf_document:
    pdf_page_count = pdf_document.page_count

    if pdf_page_count < 1:
        raise RuntimeError(
            "PDF TPL-03 tidak memiliki halaman."
        )

    first_page = pdf_document[0]
    page_rectangle = first_page.rect
    extracted_text = first_page.get_text("text")
    extracted_words = first_page.get_text("words")

page_width = float(page_rectangle.width)
page_height = float(page_rectangle.height)

word_boxes = [
    (
        float(word[0]),
        float(word[1]),
        float(word[2]),
        float(word[3]),
    )
    for word in extracted_words
    if len(word) >= 5
    and str(word[4]).strip()
]

invalid_bounding_boxes: list[str] = []
annotations_without_words: list[str] = []

for annotation in parsed_annotations:
    field_name = annotation["field_name"]
    x0, y0, x1, y1 = annotation["bbox"]

    coordinates_are_finite = all(
        math.isfinite(value)
        for value in (x0, y0, x1, y1)
    )

    bbox_is_valid = (
        coordinates_are_finite
        and x0 >= 0.0
        and y0 >= 0.0
        and x1 <= page_width
        and y1 <= page_height
        and x1 > x0
        and y1 > y0
    )

    if not bbox_is_valid:
        invalid_bounding_boxes.append(
            field_name
        )
        continue

    contains_word = any(
        bbox_intersects_word(
            annotation["bbox"],
            word_box,
        )
        for word_box in word_boxes
    )

    if not contains_word:
        annotations_without_words.append(
            field_name
        )


# ================================================================
# Audit nilai anotasi
# ================================================================

normalization_mismatches: list[dict[str, str]] = []

for field_name, expected_value in expected_values.items():
    annotation = annotation_by_name.get(
        field_name
    )

    if annotation is None:
        continue

    actual_value = annotation["text"]

    if normalize_text(actual_value) != normalize_text(
        expected_value
    ):
        normalization_mismatches.append(
            {
                "field_name": field_name,
                "expected": expected_value,
                "actual": actual_value,
            }
        )

normalized_extracted_text = normalize_text(
    extracted_text
)

missing_extracted_values: list[str] = []

for annotation in parsed_annotations:
    normalized_value = normalize_text(
        annotation["text"]
    )

    if (
        normalized_value
        and normalized_value
        not in normalized_extracted_text
    ):
        missing_extracted_values.append(
            annotation["field_name"]
        )


# ================================================================
# Audit overlap anotasi
# ================================================================

severe_annotation_overlaps: list[dict[str, Any]] = []

for first_index in range(
    len(parsed_annotations)
):
    first_annotation = parsed_annotations[
        first_index
    ]
    first_area = rectangle_area(
        first_annotation["bbox"]
    )

    if first_area <= 0.0:
        continue

    for second_index in range(
        first_index + 1,
        len(parsed_annotations),
    ):
        second_annotation = parsed_annotations[
            second_index
        ]
        second_area = rectangle_area(
            second_annotation["bbox"]
        )

        if second_area <= 0.0:
            continue

        overlap_area = intersection_area(
            first_annotation["bbox"],
            second_annotation["bbox"],
        )

        overlap_ratio = overlap_area / min(
            first_area,
            second_area,
        )

        if overlap_ratio >= SEVERE_OVERLAP_THRESHOLD:
            severe_annotation_overlaps.append(
                {
                    "first": first_annotation[
                        "field_name"
                    ],
                    "second": second_annotation[
                        "field_name"
                    ],
                    "overlap_ratio": round(
                        overlap_ratio,
                        6,
                    ),
                }
            )


# ================================================================
# Audit font dan raster
# ================================================================

minimum_field_font_size = float(
    prototype_03_rendering_metadata[
        "minimum_field_font_size"
    ]
)

font_adjustment_count = int(
    prototype_03_rendering_metadata[
        "font_adjustment_count"
    ]
)

font_adjustment_ratio = (
    font_adjustment_count
    / max(1, len(parsed_annotations))
)

with PILImage.open(
    prototype_03_png_path
) as preview_image:
    grayscale_array = np.asarray(
        preview_image.convert("L"),
        dtype=np.float32,
    ).copy()

if grayscale_array.size == 0:
    raise RuntimeError(
        "Preview PNG tidak memiliki piksel."
    )

raster_content_ratio = float(
    np.mean(grayscale_array < 245.0)
)

raster_contrast = float(
    np.std(grayscale_array)
)


# ================================================================
# Susun hasil pemeriksaan
# ================================================================

checks = [
    {
        "control": "pdf_page_count",
        "expected": 1,
        "actual": pdf_page_count,
        "valid": pdf_page_count == 1,
    },
    {
        "control": "annotation_count",
        "expected": expected_annotation_count,
        "actual": len(parsed_annotations),
        "valid": (
            len(parsed_annotations)
            == expected_annotation_count
        ),
    },
    {
        "control": "duplicate_annotations",
        "expected": 0,
        "actual": duplicate_annotation_count,
        "valid": duplicate_annotation_count == 0,
    },
    {
        "control": "missing_required_annotations",
        "expected": 0,
        "actual": len(
            missing_required_annotations
        ),
        "valid": not missing_required_annotations,
    },
    {
        "control": "invalid_bounding_boxes",
        "expected": 0,
        "actual": len(
            invalid_bounding_boxes
        ),
        "valid": not invalid_bounding_boxes,
    },
    {
        "control": "normalization_mismatches",
        "expected": 0,
        "actual": len(
            normalization_mismatches
        ),
        "valid": not normalization_mismatches,
    },
    {
        "control": "annotations_without_words",
        "expected": 0,
        "actual": len(
            annotations_without_words
        ),
        "valid": not annotations_without_words,
    },
    {
        "control": "severe_annotation_overlaps",
        "expected": 0,
        "actual": len(
            severe_annotation_overlaps
        ),
        "valid": not severe_annotation_overlaps,
    },
    {
        "control": "missing_extracted_values",
        "expected": 0,
        "actual": len(
            missing_extracted_values
        ),
        "valid": not missing_extracted_values,
    },
    {
        "control": "minimum_field_font_size",
        "expected": ">=6.5",
        "actual": round(
            minimum_field_font_size,
            4,
        ),
        "valid": (
            minimum_field_font_size
            >= MINIMUM_FIELD_FONT_SIZE
        ),
    },
    {
        "control": "font_adjustment_ratio",
        "expected": "<=0.20",
        "actual": round(
            font_adjustment_ratio,
            4,
        ),
        "valid": (
            font_adjustment_ratio
            <= MAXIMUM_FONT_ADJUSTMENT_RATIO
        ),
    },
    {
        "control": "raster_content_ratio",
        "expected": "0.01–0.40",
        "actual": round(
            raster_content_ratio,
            4,
        ),
        "valid": (
            MINIMUM_RASTER_CONTENT_RATIO
            <= raster_content_ratio
            <= MAXIMUM_RASTER_CONTENT_RATIO
        ),
    },
    {
        "control": "raster_contrast",
        "expected": ">=10.0",
        "actual": round(
            raster_contrast,
            4,
        ),
        "valid": (
            raster_contrast
            >= MINIMUM_RASTER_CONTRAST
        ),
    },
]

for check in checks:
    check["status"] = (
        "VALID"
        if check["valid"]
        else "INVALID"
    )

failed_checks = [
    check
    for check in checks
    if not check["valid"]
]

qa_status = (
    "PASSED"
    if not failed_checks
    else "FAILED"
)


# ================================================================
# Simpan QA report
# ================================================================

qa_report = {
    "schema_version": QA_SCHEMA_VERSION,
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "document_id": prototype_03_document_id,
    "canonical_invoice_id": (
        prototype_03_canonical_id
    ),
    "template_id": TPL03_TEMPLATE_ID,
    "status": qa_status,
    "manual_visual_review": "PENDING",
    "metrics": {
        "auditor_version": QA_AUDITOR_VERSION,
        "pdf_page_count": pdf_page_count,
        "annotation_count": len(
            parsed_annotations
        ),
        "expected_annotation_count": (
            expected_annotation_count
        ),
        "duplicate_annotation_count": (
            duplicate_annotation_count
        ),
        "missing_required_annotation_count": len(
            missing_required_annotations
        ),
        "invalid_bounding_box_count": len(
            invalid_bounding_boxes
        ),
        "normalization_mismatch_count": len(
            normalization_mismatches
        ),
        "annotations_without_words_count": len(
            annotations_without_words
        ),
        "severe_annotation_overlap_count": len(
            severe_annotation_overlaps
        ),
        "missing_extracted_value_count": len(
            missing_extracted_values
        ),
        "minimum_field_font_size": round(
            minimum_field_font_size,
            6,
        ),
        "font_adjustment_count": (
            font_adjustment_count
        ),
        "font_adjustment_ratio": round(
            font_adjustment_ratio,
            6,
        ),
        "raster_content_ratio": round(
            raster_content_ratio,
            6,
        ),
        "raster_contrast": round(
            raster_contrast,
            6,
        ),
    },
    "failures": {
        "failed_checks": [
            check["control"]
            for check in failed_checks
        ],
        "missing_required_annotations": (
            missing_required_annotations
        ),
        "invalid_bounding_boxes": (
            invalid_bounding_boxes
        ),
        "normalization_mismatches": (
            normalization_mismatches
        ),
        "annotations_without_words": (
            annotations_without_words
        ),
        "severe_annotation_overlaps": (
            severe_annotation_overlaps
        ),
        "missing_extracted_values": (
            missing_extracted_values
        ),
    },
    "checksums_sha256": {
        "prototype_pdf": sha256_file(
            prototype_03_pdf_path
        ),
        "preview": sha256_file(
            prototype_03_png_path
        ),
        "ground_truth": sha256_file(
            prototype_03_ground_truth_path
        ),
    },
}

write_json_atomically(
    prototype_03_qa_report_path,
    qa_report,
)


# ================================================================
# Verifikasi round-trip QA report
# ================================================================

with prototype_03_qa_report_path.open(
    "r",
    encoding="utf-8",
) as qa_report_file:
    verified_qa_report = json.load(
        qa_report_file
    )

if verified_qa_report != qa_report:
    raise RuntimeError(
        "QA report berubah setelah disimpan dan dibuka ulang."
    )


# ================================================================
# Output audit
# ================================================================

qa_summary = pd.DataFrame(
    [
        {
            "control": check["control"],
            "expected": check["expected"],
            "actual": check["actual"],
            "status": check["status"],
        }
        for check in checks
    ]
)

display(qa_summary)

print(
    f"QA report          : "
    f"{prototype_03_qa_report_path}"
)
print(
    f"QA status          : {qa_status}"
)
print(
    "Minimum field font : "
    f"{minimum_field_font_size:.2f}"
)
print(
    "Font adjustments   : "
    f"{font_adjustment_count}"
)
print(
    "Ink pixel ratio    : "
    f"{raster_content_ratio:.6f}"
)
print(
    "Raster contrast    : "
    f"{raster_contrast:.6f}"
)
print(
    "Manual review      : "
    f"{qa_report['manual_visual_review']}"
)
print()

if qa_status != "PASSED":
    print("Detail kegagalan:")
    print(
        json.dumps(
            qa_report["failures"],
            ensure_ascii=False,
            indent=2,
        )
    )

    raise RuntimeError(
        "Prototype TPL-03 gagal audit QA teknis."
    )

print(
    "✅ Prototype TPL-03 lulus audit teknis. "
    "Review visual manual masih perlu dicatat "
    "ke QA report."
)


In [ ]:
# CELL 47A — Finalisasi review visual TPL-03

import json


qa_report_path = prototype_03_qa_report_path

with qa_report_path.open(
    "r",
    encoding="utf-8",
) as file:
    qa_report = json.load(file)


# Validasi identitas dan status teknis
if qa_report.get("template_id") != "TPL-03":
    raise RuntimeError(
        "QA report bukan milik TPL-03."
    )

if qa_report.get("document_id") != (
    prototype_03_document_id
):
    raise RuntimeError(
        "Document ID QA report tidak sesuai."
    )

if qa_report.get("status") != "PASSED":
    raise RuntimeError(
        "Review visual tidak dapat disahkan karena "
        f"status teknis adalah "
        f"{qa_report.get('status')!r}."
    )


# Pastikan artifact tidak berubah setelah audit
artifact_checks = {
    "prototype_pdf": prototype_03_pdf_path,
    "preview": prototype_03_png_path,
    "ground_truth": (
        prototype_03_ground_truth_path
    ),
}

for artifact_name, artifact_path in (
    artifact_checks.items()
):
    expected_checksum = (
        qa_report[
            "checksums_sha256"
        ][artifact_name]
    )

    actual_checksum = sha256_file(
        artifact_path
    )

    if actual_checksum != expected_checksum:
        raise RuntimeError(
            f"Artifact {artifact_name!r} berubah "
            "setelah audit. Jalankan ulang Cell 47."
        )


# Catat hasil review visual
qa_report[
    "manual_visual_review"
] = "PASSED"

write_json_atomically(
    qa_report_path,
    qa_report,
)


# Buka ulang untuk verifikasi
with qa_report_path.open(
    "r",
    encoding="utf-8",
) as file:
    verified_report = json.load(file)

if (
    verified_report.get(
        "manual_visual_review"
    )
    != "PASSED"
):
    raise RuntimeError(
        "Status review visual gagal disimpan."
    )


print(f"QA report           : {qa_report_path}")
print(
    "Technical status    : "
    f"{verified_report['status']}"
)
print(
    "Manual visual review: "
    f"{verified_report['manual_visual_review']}"
)
print()
print(
    "✅ TPL-03 FINAL PASSED — audit teknis "
    "dan review visual telah selesai."
)

**CELL 48 — Inspeksi konfigurasi dan payload TPL-04**

In [ ]:
# ================================================================
# CELL 48 — Inspeksi konfigurasi dan payload TPL-04
# Tidak membuat atau mengubah artifact
# ================================================================

import json
import pandas as pd


# ================================================================
# Pastikan TPL-03 sudah final
# ================================================================

with prototype_03_qa_report_path.open(
    "r",
    encoding="utf-8",
) as file:
    tpl03_final_report = json.load(file)

if (
    tpl03_final_report.get("status")
    != "PASSED"
    or tpl03_final_report.get(
        "manual_visual_review"
    )
    != "PASSED"
):
    raise RuntimeError(
        "TPL-03 belum FINAL PASSED. "
        "Jalankan Cell 47A terlebih dahulu."
    )

print("TPL-03 status       : FINAL PASSED")


# ================================================================
# Konfigurasi TPL-04
# ================================================================

TPL04_TEMPLATE_ID = "TPL-04"

required_objects = [
    "template_registry",
    "invoice_render_payloads",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Runtime belum lengkap: "
        f"{missing_objects}"
    )

if TPL04_TEMPLATE_ID not in template_registry:
    raise KeyError(
        "TPL-04 tidak ditemukan dalam template_registry."
    )


# ================================================================
# Ambil seluruh payload TPL-04
# ================================================================

tpl04_payloads = {
    canonical_id: payload
    for canonical_id, payload
    in invoice_render_payloads.items()
    if payload.get("template_id")
    == TPL04_TEMPLATE_ID
}

tpl04_candidate_ids = sorted(
    tpl04_payloads
)

if len(tpl04_candidate_ids) != 20:
    raise RuntimeError(
        "TPL-04 harus memiliki tepat 20 payload. "
        f"Actual: {len(tpl04_candidate_ids)}."
    )

sample_canonical_id = (
    tpl04_candidate_ids[0]
)

sample_payload = tpl04_payloads[
    sample_canonical_id
]


# ================================================================
# Helper statistik teks
# ================================================================

def text_length(value):
    return len(str(value or ""))


def address_length(party):
    return text_length(
        " ".join(
            party.get(
                "address_lines",
                [],
            )
        )
    )


# ================================================================
# Statistik seluruh payload
# ================================================================

payload_statistics = []

for canonical_id in tpl04_candidate_ids:
    payload = tpl04_payloads[
        canonical_id
    ]

    vendor = payload.get("vendor", {})
    buyer = payload.get("buyer", {})
    items = payload.get("items", [])
    financials = payload.get(
        "financials",
        {},
    )

    payload_statistics.append(
        {
            "canonical_id": canonical_id,
            "document_id": payload.get(
                "document_id"
            ),
            "language": payload.get(
                "language"
            ),
            "currency": payload.get(
                "currency"
            ),
            "item_count": len(items),
            "vendor_name_length": text_length(
                vendor.get("name")
            ),
            "vendor_address_length": (
                address_length(vendor)
            ),
            "vendor_email_length": text_length(
                vendor.get("email")
            ),
            "vendor_phone_length": text_length(
                vendor.get("phone")
            ),
            "vendor_tax_length": text_length(
                vendor.get(
                    "tax_identifier_display"
                )
            ),
            "buyer_name_length": text_length(
                buyer.get("name")
            ),
            "buyer_address_length": (
                address_length(buyer)
            ),
            "buyer_email_length": text_length(
                buyer.get("email")
            ),
            "buyer_phone_length": text_length(
                buyer.get("phone")
            ),
            "buyer_tax_length": text_length(
                buyer.get(
                    "tax_identifier_display"
                )
            ),
            "maximum_description_length": max(
                (
                    text_length(
                        item.get("description")
                    )
                    for item in items
                ),
                default=0,
            ),
            "maximum_money_length": max(
                (
                    text_length(
                        field.get(
                            "display_value"
                        )
                    )
                    for field
                    in financials.values()
                    if isinstance(field, dict)
                ),
                default=0,
            ),
        }
    )

statistics_frame = pd.DataFrame(
    payload_statistics
)


# ================================================================
# Struktur payload sampel
# ================================================================

sample_structure = {
    "canonical_invoice_id": (
        sample_payload.get(
            "canonical_invoice_id"
        )
    ),
    "document_id": sample_payload.get(
        "document_id"
    ),
    "template_id": sample_payload.get(
        "template_id"
    ),
    "language": sample_payload.get(
        "language"
    ),
    "currency": sample_payload.get(
        "currency"
    ),
    "title": sample_payload.get(
        "title"
    ),
    "top_level_keys": sorted(
        sample_payload.keys()
    ),
    "label_keys": sorted(
        sample_payload.get(
            "labels",
            {},
        ).keys()
    ),
    "metadata_keys": sorted(
        sample_payload.get(
            "metadata",
            {},
        ).keys()
    ),
    "vendor_keys": sorted(
        sample_payload.get(
            "vendor",
            {},
        ).keys()
    ),
    "buyer_keys": sorted(
        sample_payload.get(
            "buyer",
            {},
        ).keys()
    ),
    "item_keys": sorted(
        (
            sample_payload.get(
                "items",
                [{}],
            )[0]
        ).keys()
        if sample_payload.get("items")
        else []
    ),
    "financial_keys": sorted(
        sample_payload.get(
            "financials",
            {},
        ).keys()
    ),
    "footer_keys": sorted(
        sample_payload.get(
            "footer",
            {},
        ).keys()
    ),
}


# ================================================================
# Output
# ================================================================

print("\nTEMPLATE SPECIFICATION")
print(
    json.dumps(
        template_registry[
            TPL04_TEMPLATE_ID
        ],
        ensure_ascii=False,
        indent=2,
        default=str,
    )
)

print("\nPAYLOAD COUNT")
print(len(tpl04_candidate_ids))

print("\nSAMPLE STRUCTURE")
print(
    json.dumps(
        sample_structure,
        ensure_ascii=False,
        indent=2,
        default=str,
    )
)

print("\nLABELS")
print(
    json.dumps(
        sample_payload.get(
            "labels",
            {},
        ),
        ensure_ascii=False,
        indent=2,
        default=str,
    )
)

print("\nPAYLOAD STATISTICS")
display(
    statistics_frame.describe(
        include="all"
    ).transpose()
)

print("\nITEM-COUNT DISTRIBUTION")
display(
    statistics_frame[
        "item_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("item_count")
    .reset_index(
        name="document_count"
    )
)

print()
print(
    "✅ Inspeksi TPL-04 selesai. "
    "Belum ada artifact yang dibuat atau diubah."
)

**CELL 49 — FINAL Prototype renderer TPL-04: Compact Left Sidebar**

In [ ]:
# ================================================================
# CELL 49 — FINAL
# Prototype renderer TPL-04: Compact Left Sidebar
# ================================================================

from pathlib import Path
from typing import Any
from PIL import Image as PILImage

import json
import os

import pandas as pd
import pymupdf


# ================================================================
# Validasi runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "PROTOTYPE_ROOT",
    "CANONICAL_RECORDS_JSONL_PATH",
    "FieldAnnotation",
    "template_registry",
    "invoice_render_payloads",
    "get_page_dimensions",
    "hex_to_rgb",
    "insert_annotated_text",
    "insert_static_text",
    "annotation_to_dict",
    "write_json_atomically",
    "RENDER_ENGINE_VERSION",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 49 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali cell pemulihan runtime."
    )


# ================================================================
# Konfigurasi TPL-04
# ================================================================

TPL04_TEMPLATE_ID = "TPL-04"
TPL04_RENDERER_VERSION = "1.0.0"
TPL04_MIN_ITEMS_PER_PAGE = 2
TPL04_MAX_ITEMS_PER_PAGE = 8

TPL04_PROTOTYPE_ROOT = (
    Path(PROTOTYPE_ROOT)
    / TPL04_TEMPLATE_ID
)

TPL04_PROTOTYPE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ================================================================
# Helper canonical payload dan teks PDF
# ================================================================

def load_tpl04_canonical_payload(
    canonical_id: str,
) -> dict[str, Any]:
    canonical_path = Path(
        CANONICAL_RECORDS_JSONL_PATH
    )

    if not canonical_path.is_file():
        raise FileNotFoundError(
            "Canonical JSONL tidak ditemukan: "
            f"{canonical_path}"
        )

    with canonical_path.open(
        "r",
        encoding="utf-8",
    ) as canonical_file:
        for line_number, line in enumerate(
            canonical_file,
            start=1,
        ):
            clean_line = line.strip()

            if not clean_line:
                continue

            try:
                payload = json.loads(clean_line)
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    "Canonical JSONL tidak valid pada "
                    f"baris {line_number}."
                ) from error

            if (
                payload.get("canonical_invoice_id")
                == canonical_id
            ):
                return payload

    raise KeyError(
        "Canonical payload tidak ditemukan: "
        f"{canonical_id}"
    )


def normalize_pdf_notice(text: Any) -> str:
    """Mengganti separator yang tidak didukung font PDF bawaan."""

    return (
        str(text)
        .replace("—", "·")
        .replace("–", "-")
    )


# ================================================================
# Renderer TPL-04
# ================================================================

def render_template_04(
    render_payload: dict[str, Any],
    output_pdf_path: Path,
) -> tuple[list[FieldAnnotation], dict[str, Any]]:
    if not isinstance(render_payload, dict):
        raise TypeError(
            "render_payload wajib berupa dictionary."
        )

    if render_payload.get("template_id") != TPL04_TEMPLATE_ID:
        raise ValueError(
            "Renderer TPL-04 hanya menerima payload TPL-04."
        )

    required_payload_keys = {
        "canonical_invoice_id",
        "document_id",
        "template_id",
        "language",
        "currency",
        "title",
        "labels",
        "metadata",
        "vendor",
        "buyer",
        "items",
        "financials",
        "footer",
        "synthetic_notice",
    }

    missing_payload_keys = sorted(
        required_payload_keys
        - set(render_payload)
    )

    if missing_payload_keys:
        raise KeyError(
            "Payload TPL-04 belum lengkap: "
            f"{missing_payload_keys}"
        )

    item_count = len(render_payload["items"])

    if not (
        TPL04_MIN_ITEMS_PER_PAGE
        <= item_count
        <= TPL04_MAX_ITEMS_PER_PAGE
    ):
        raise ValueError(
            "TPL-04 hanya mendukung 2–8 item per halaman."
        )

    template_specification = template_registry[
        TPL04_TEMPLATE_ID
    ]

    page_width, page_height = get_page_dimensions(
        template_specification["page_size"]
    )

    primary_color = hex_to_rgb(
        template_specification["primary_color"]
    )

    accent_color = hex_to_rgb(
        template_specification["accent_color"]
    )

    dark_text = (0.13, 0.20, 0.23)
    muted_text = (0.40, 0.46, 0.48)
    border_color = (0.82, 0.86, 0.86)
    sidebar_fill = (0.935, 0.972, 0.966)
    light_accent = (0.955, 0.982, 0.978)
    white = (1.0, 1.0, 1.0)

    output_pdf_path = Path(output_pdf_path)

    output_pdf_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_pdf_path = output_pdf_path.with_name(
        f".{output_pdf_path.stem}.tmp.pdf"
    )

    temporary_pdf_path.unlink(
        missing_ok=True
    )

    annotations: list[FieldAnnotation] = []
    field_font_audit: list[dict[str, Any]] = []

    document = pymupdf.open()

    def add_field(
        page: pymupdf.Page,
        rectangle: pymupdf.Rect,
        text: str,
        field_name: str,
        font_name: str = "helv",
        font_size: float = 7.0,
        minimum_font_size: float = 6.5,
        font_color: tuple[
            float,
            float,
            float,
        ] = dark_text,
        alignment: str = "left",
    ) -> None:
        annotation, used_font_size = insert_annotated_text(
            page=page,
            rectangle=rectangle,
            text=str(text),
            field_name=field_name,
            font_name=font_name,
            font_size=font_size,
            minimum_font_size=minimum_font_size,
            font_color=font_color,
            alignment=alignment,
        )

        annotations.append(annotation)

        field_font_audit.append(
            {
                "field_name": field_name,
                "requested_font_size": font_size,
                "used_font_size": used_font_size,
                "adjusted": used_font_size < font_size,
            }
        )

    try:
        page = document.new_page(
            width=page_width,
            height=page_height,
        )

        # --------------------------------------------------------
        # Sidebar kiri
        # --------------------------------------------------------

        sidebar_x1 = 172.0

        page.draw_rect(
            pymupdf.Rect(
                0,
                0,
                sidebar_x1,
                page_height,
            ),
            color=None,
            fill=sidebar_fill,
            width=0,
            overlay=True,
        )

        page.draw_rect(
            pymupdf.Rect(
                0,
                0,
                10,
                page_height,
            ),
            color=accent_color,
            fill=accent_color,
            width=0,
            overlay=True,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                28,
                44,
                154,
                61,
            ),
            text=render_payload["labels"]["vendor"].upper(),
            font_name="hebo",
            font_size=7.3,
            minimum_font_size=6.0,
            font_color=accent_color,
        )

        vendor = render_payload["vendor"]

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                28,
                72,
                155,
                108,
            ),
            text=vendor["name"],
            field_name="vendor.name",
            font_name="hebo",
            font_size=8.2,
            minimum_font_size=6.5,
            font_color=primary_color,
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                28,
                118,
                155,
                158,
            ),
            text="\n".join(
                vendor["address_lines"]
            ),
            field_name="vendor.address_lines",
            font_name="helv",
            font_size=6.9,
            minimum_font_size=6.5,
            font_color=dark_text,
        )

        page.draw_line(
            pymupdf.Point(28, 174),
            pymupdf.Point(155, 174),
            color=border_color,
            width=0.7,
            overlay=True,
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                28,
                185,
                155,
                202,
            ),
            text=vendor["email"],
            field_name="vendor.email",
            font_name="helv",
            font_size=6.7,
            minimum_font_size=6.5,
            font_color=dark_text,
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                28,
                211,
                155,
                228,
            ),
            text=vendor["phone"],
            field_name="vendor.phone",
            font_name="helv",
            font_size=6.7,
            minimum_font_size=6.5,
            font_color=dark_text,
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                28,
                237,
                155,
                263,
            ),
            text=vendor["tax_identifier_display"],
            field_name="vendor.tax_identifier",
            font_name="helv",
            font_size=6.7,
            minimum_font_size=6.5,
            font_color=dark_text,
        )

        page.draw_line(
            pymupdf.Point(28, 292),
            pymupdf.Point(155, 292),
            color=border_color,
            width=0.7,
            overlay=True,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                28,
                310,
                155,
                326,
            ),
            text="DOCUMENT",
            font_name="hebo",
            font_size=6.5,
            minimum_font_size=6.0,
            font_color=muted_text,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                28,
                332,
                155,
                351,
            ),
            text=render_payload["footer"][
                "document_reference"
            ],
            font_name="helv",
            font_size=6.5,
            minimum_font_size=6.0,
            font_color=primary_color,
        )

        currency_label = (
            "Mata Uang"
            if render_payload["language"] == "id"
            else "Currency"
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                28,
                377,
                92,
                393,
            ),
            text=currency_label,
            font_name="helv",
            font_size=6.5,
            minimum_font_size=6.0,
            font_color=muted_text,
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                28,
                398,
                155,
                418,
            ),
            text=render_payload["currency"],
            field_name="currency",
            font_name="hebo",
            font_size=8.5,
            minimum_font_size=6.5,
            font_color=accent_color,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                28,
                450,
                155,
                468,
            ),
            text="COMPACT SIDEBAR",
            font_name="hebo",
            font_size=6.2,
            minimum_font_size=5.8,
            font_color=accent_color,
        )

        # --------------------------------------------------------
        # Header utama dan metadata
        # --------------------------------------------------------

        main_x0 = 198.0
        main_x1 = page_width - 45.0

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                main_x0,
                42,
                350,
                82,
            ),
            text=render_payload["title"],
            font_name="hebo",
            font_size=25.0,
            minimum_font_size=18.0,
            font_color=primary_color,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                main_x0 + 2,
                91,
                350,
                108,
            ),
            text="LEFT SIDEBAR · SYNTHETIC",
            font_name="hebo",
            font_size=6.5,
            minimum_font_size=6.0,
            font_color=accent_color,
        )

        page.draw_rect(
            pymupdf.Rect(
                360,
                36,
                main_x1,
                132,
            ),
            color=None,
            fill=light_accent,
            width=0,
            overlay=True,
        )

        metadata_rows = [
            ("invoice_number", 45, 65),
            ("invoice_date", 75, 95),
            ("due_date", 105, 125),
        ]

        for field_name, row_y0, row_y1 in metadata_rows:
            field_payload = render_payload[
                "metadata"
            ][field_name]

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    370,
                    row_y0,
                    433,
                    row_y1,
                ),
                text=field_payload["label"],
                font_name="helv",
                font_size=6.3,
                minimum_font_size=5.8,
                font_color=muted_text,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    435,
                    row_y0,
                    main_x1 - 10,
                    row_y1,
                ),
                text=field_payload["display_value"],
                field_name=field_name,
                font_name="hebo",
                font_size=7.0,
                minimum_font_size=6.5,
                font_color=primary_color,
                alignment="right",
            )

        page.draw_line(
            pymupdf.Point(main_x0, 154),
            pymupdf.Point(main_x1, 154),
            color=border_color,
            width=0.8,
            overlay=True,
        )

        # --------------------------------------------------------
        # Buyer di area utama
        # --------------------------------------------------------

        buyer_card_y0 = 170.0
        buyer_card_y1 = 284.0

        page.draw_rect(
            pymupdf.Rect(
                main_x0,
                buyer_card_y0,
                main_x1,
                buyer_card_y1,
            ),
            color=border_color,
            fill=white,
            width=0.7,
            overlay=True,
        )

        page.draw_rect(
            pymupdf.Rect(
                main_x0,
                buyer_card_y0,
                main_x0 + 5,
                buyer_card_y1,
            ),
            color=accent_color,
            fill=accent_color,
            width=0,
            overlay=True,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                main_x0 + 14,
                buyer_card_y0 + 8,
                main_x1 - 12,
                buyer_card_y0 + 23,
            ),
            text=render_payload["labels"]["buyer"].upper(),
            font_name="hebo",
            font_size=6.7,
            minimum_font_size=6.0,
            font_color=accent_color,
        )

        buyer = render_payload["buyer"]

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                main_x0 + 14,
                buyer_card_y0 + 31,
                350,
                buyer_card_y0 + 51,
            ),
            text=buyer["name"],
            field_name="buyer.name",
            font_name="hebo",
            font_size=7.6,
            minimum_font_size=6.5,
            font_color=primary_color,
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                main_x0 + 14,
                buyer_card_y0 + 55,
                350,
                buyer_card_y0 + 94,
            ),
            text="\n".join(
                buyer["address_lines"]
            ),
            field_name="buyer.address_lines",
            font_name="helv",
            font_size=6.8,
            minimum_font_size=6.5,
            font_color=dark_text,
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                365,
                buyer_card_y0 + 32,
                main_x1 - 12,
                buyer_card_y0 + 48,
            ),
            text=buyer["email"],
            field_name="buyer.email",
            font_name="helv",
            font_size=6.7,
            minimum_font_size=6.5,
            font_color=dark_text,
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                365,
                buyer_card_y0 + 56,
                main_x1 - 12,
                buyer_card_y0 + 72,
            ),
            text=buyer["phone"],
            field_name="buyer.phone",
            font_name="helv",
            font_size=6.7,
            minimum_font_size=6.5,
            font_color=dark_text,
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                365,
                buyer_card_y0 + 80,
                main_x1 - 12,
                buyer_card_y0 + 100,
            ),
            text=buyer["tax_identifier_display"],
            field_name="buyer.tax_identifier",
            font_name="helv",
            font_size=6.7,
            minimum_font_size=6.5,
            font_color=dark_text,
        )

        # --------------------------------------------------------
        # Tabel item minimal
        # --------------------------------------------------------

        table_positions = [
            main_x0,
            218.0,
            342.0,
            382.0,
            465.0,
            main_x1,
        ]

        table_header_y0 = 307.0
        table_header_y1 = 331.0
        row_height = 24.0

        page.draw_line(
            pymupdf.Point(
                table_positions[0],
                table_header_y0,
            ),
            pymupdf.Point(
                table_positions[-1],
                table_header_y0,
            ),
            color=accent_color,
            width=1.1,
            overlay=True,
        )

        page.draw_line(
            pymupdf.Point(
                table_positions[0],
                table_header_y1,
            ),
            pymupdf.Point(
                table_positions[-1],
                table_header_y1,
            ),
            color=primary_color,
            width=0.8,
            overlay=True,
        )

        table_headers = [
            "#",
            render_payload["labels"]["description"],
            render_payload["labels"]["quantity"],
            render_payload["labels"]["unit_price"],
            render_payload["labels"]["line_total"],
        ]

        for column_index, header_text in enumerate(
            table_headers
        ):
            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    table_positions[column_index] + 2,
                    table_header_y0 + 6,
                    table_positions[column_index + 1] - 2,
                    table_header_y1 - 2,
                ),
                text=header_text,
                font_name="hebo",
                font_size=6.0,
                minimum_font_size=5.5,
                font_color=primary_color,
                alignment=(
                    "left"
                    if column_index == 1
                    else "center"
                ),
            )

        for item_index, item in enumerate(
            render_payload["items"]
        ):
            row_y0 = (
                table_header_y1
                + item_index * row_height
            )
            row_y1 = row_y0 + row_height

            page.draw_line(
                pymupdf.Point(
                    table_positions[0],
                    row_y1,
                ),
                pymupdf.Point(
                    table_positions[-1],
                    row_y1,
                ),
                color=border_color,
                width=0.45,
                overlay=True,
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    table_positions[0] + 2,
                    row_y0 + 6,
                    table_positions[1] - 2,
                    row_y1 - 2,
                ),
                text=str(item["item_number"]),
                font_name="helv",
                font_size=6.6,
                minimum_font_size=5.8,
                font_color=muted_text,
                alignment="center",
            )

            item_fields = [
                ("description", 1, "left"),
                ("quantity", 2, "center"),
                ("unit_price", 3, "right"),
                ("line_total", 4, "right"),
            ]

            for (
                field_name,
                column_index,
                alignment,
            ) in item_fields:
                add_field(
                    page=page,
                    rectangle=pymupdf.Rect(
                        table_positions[column_index] + 3,
                        row_y0 + 6,
                        table_positions[column_index + 1] - 3,
                        row_y1 - 2,
                    ),
                    text=item[field_name],
                    field_name=(
                        f"items[{item_index}].{field_name}"
                    ),
                    font_name="helv",
                    font_size=6.7,
                    minimum_font_size=6.5,
                    font_color=dark_text,
                    alignment=alignment,
                )

        table_end_y = (
            table_header_y1
            + item_count * row_height
        )

        # --------------------------------------------------------
        # Ringkasan finansial kanan bawah
        # --------------------------------------------------------

        totals_panel_height = 112.0

        totals_y0 = min(
            max(
                table_end_y + 30.0,
                540.0,
            ),
            570.0,
        )

        totals_y1 = (
            totals_y0
            + totals_panel_height
        )

        table_to_totals_gap = (
            totals_y0 - table_end_y
        )

        if table_to_totals_gap < 24.0:
            raise RuntimeError(
                "Jarak tabel dan panel total terlalu sempit: "
                f"{table_to_totals_gap:.2f} pt."
            )

        footer_line_y = 762.0

        if totals_y1 >= footer_line_y - 30.0:
            raise RuntimeError(
                "Panel total terlalu dekat dengan footer."
            )

        totals_x0 = 350.0
        totals_x1 = main_x1

        summary_title = (
            "RINGKASAN"
            if render_payload["language"] == "id"
            else "SUMMARY"
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                totals_x0,
                totals_y0 - 20,
                totals_x1,
                totals_y0 - 5,
            ),
            text=summary_title,
            font_name="hebo",
            font_size=6.4,
            minimum_font_size=5.8,
            font_color=accent_color,
            alignment="right",
        )

        page.draw_rect(
            pymupdf.Rect(
                totals_x0,
                totals_y0,
                totals_x1,
                totals_y1,
            ),
            color=None,
            fill=light_accent,
            width=0,
            overlay=True,
        )

        page.draw_rect(
            pymupdf.Rect(
                totals_x0,
                totals_y0,
                totals_x0 + 4,
                totals_y1,
            ),
            color=accent_color,
            fill=accent_color,
            width=0,
            overlay=True,
        )

        financial_order = [
            "subtotal",
            "tax",
            "discount",
            "total",
        ]

        for row_index, field_name in enumerate(
            financial_order
        ):
            field_payload = render_payload[
                "financials"
            ][field_name]

            row_y0 = (
                totals_y0
                + 10.0
                + row_index * 24.0
            )

            is_total = field_name == "total"

            if is_total:
                page.draw_line(
                    pymupdf.Point(
                        totals_x0 + 12,
                        row_y0 - 4,
                    ),
                    pymupdf.Point(
                        totals_x1 - 10,
                        row_y0 - 4,
                    ),
                    color=accent_color,
                    width=0.9,
                    overlay=True,
                )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    totals_x0 + 13,
                    row_y0,
                    totals_x0 + 80,
                    row_y0 + 17,
                ),
                text=field_payload["label"],
                font_name=(
                    "hebo"
                    if is_total
                    else "helv"
                ),
                font_size=(
                    7.4
                    if is_total
                    else 6.9
                ),
                minimum_font_size=6.0,
                font_color=(
                    primary_color
                    if is_total
                    else muted_text
                ),
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    totals_x0 + 82,
                    row_y0,
                    totals_x1 - 10,
                    row_y0 + 17,
                ),
                text=field_payload["display_value"],
                field_name=f"financials.{field_name}",
                font_name=(
                    "hebo"
                    if is_total
                    else "helv"
                ),
                font_size=(
                    8.0
                    if is_total
                    else 7.0
                ),
                minimum_font_size=6.5,
                font_color=(
                    accent_color
                    if is_total
                    else dark_text
                ),
                alignment="right",
            )

        # --------------------------------------------------------
        # Footer
        # --------------------------------------------------------

        page.draw_line(
            pymupdf.Point(main_x0, footer_line_y),
            pymupdf.Point(main_x1, footer_line_y),
            color=border_color,
            width=0.7,
            overlay=True,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                main_x0,
                772,
                main_x1,
                787,
            ),
            text=render_payload["footer"][
                "document_reference"
            ],
            font_name="helv",
            font_size=6.4,
            minimum_font_size=5.8,
            font_color=muted_text,
            alignment="center",
        )

        notice_display_text = normalize_pdf_notice(
            render_payload["synthetic_notice"]
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                main_x0,
                802,
                main_x1,
                821,
            ),
            text=notice_display_text,
            field_name="document.synthetic_notice",
            font_name="hebo",
            font_size=6.5,
            minimum_font_size=6.5,
            font_color=accent_color,
            alignment="center",
        )

        # --------------------------------------------------------
        # Validasi sebelum simpan
        # --------------------------------------------------------

        annotation_names = [
            annotation.field_name
            for annotation in annotations
        ]

        if len(annotation_names) != len(
            set(annotation_names)
        ):
            raise RuntimeError(
                "Ditemukan nama anotasi duplikat."
            )

        expected_annotation_count = (
            19 + 4 * item_count
        )

        if len(annotations) != expected_annotation_count:
            raise RuntimeError(
                "Jumlah anotasi tidak sesuai. "
                f"Expected: {expected_annotation_count}; "
                f"actual: {len(annotations)}."
            )

        if not field_font_audit:
            raise RuntimeError(
                "Audit font field tidak boleh kosong."
            )

        minimum_used_font_size = min(
            row["used_font_size"]
            for row in field_font_audit
        )

        if minimum_used_font_size < 6.5:
            raise RuntimeError(
                "Font field berada di bawah batas 6.5 pt."
            )

        document.save(
            str(temporary_pdf_path),
            garbage=4,
            deflate=True,
        )

    except Exception:
        temporary_pdf_path.unlink(
            missing_ok=True
        )
        raise

    finally:
        document.close()

    if (
        not temporary_pdf_path.is_file()
        or temporary_pdf_path.stat().st_size == 0
    ):
        temporary_pdf_path.unlink(
            missing_ok=True
        )
        raise RuntimeError(
            "PDF sementara TPL-04 gagal dibuat."
        )

    os.replace(
        temporary_pdf_path,
        output_pdf_path,
    )

    rendering_metadata = {
        "renderer_version": RENDER_ENGINE_VERSION,
        "template_renderer_version": (
            TPL04_RENDERER_VERSION
        ),
        "template_id": TPL04_TEMPLATE_ID,
        "layout_family": template_specification[
            "layout_family"
        ],
        "page_size": template_specification[
            "page_size"
        ],
        "page_width_points": round(
            page_width,
            3,
        ),
        "page_height_points": round(
            page_height,
            3,
        ),
        "page_count": 1,
        "annotation_type": "field_region",
        "annotation_count": len(annotations),
        "minimum_field_font_size": min(
            row["used_font_size"]
            for row in field_font_audit
        ),
        "font_adjustment_count": int(
            sum(
                bool(row["adjusted"])
                for row in field_font_audit
            )
        ),
        "layout_metrics": {
            "item_count": item_count,
            "table_end_y_points": round(
                table_end_y,
                3,
            ),
            "totals_panel_y0_points": round(
                totals_y0,
                3,
            ),
            "totals_panel_y1_points": round(
                totals_y1,
                3,
            ),
            "table_to_totals_gap_points": round(
                table_to_totals_gap,
                3,
            ),
            "footer_line_y_points": footer_line_y,
            "sidebar_width_points": sidebar_x1,
        },
    }

    return annotations, rendering_metadata


# ================================================================
# Pilih prototype pertama TPL-04
# ================================================================

tpl04_candidate_ids = sorted(
    canonical_id
    for canonical_id, payload
    in invoice_render_payloads.items()
    if payload.get("template_id")
    == TPL04_TEMPLATE_ID
)

if len(tpl04_candidate_ids) != 20:
    raise RuntimeError(
        "TPL-04 harus memiliki tepat 20 payload. "
        f"Actual: {len(tpl04_candidate_ids)}."
    )

prototype_04_canonical_id = (
    tpl04_candidate_ids[0]
)

prototype_04_render_payload = (
    invoice_render_payloads[
        prototype_04_canonical_id
    ]
)

prototype_04_document_id = (
    prototype_04_render_payload[
        "document_id"
    ]
)

prototype_04_canonical_payload = (
    load_tpl04_canonical_payload(
        prototype_04_canonical_id
    )
)


# ================================================================
# Lokasi artifact
# ================================================================

prototype_04_pdf_path = (
    TPL04_PROTOTYPE_ROOT
    / (
        f"{prototype_04_document_id}"
        "_TPL-04_prototype.pdf"
    )
)

prototype_04_png_path = (
    TPL04_PROTOTYPE_ROOT
    / (
        f"{prototype_04_document_id}"
        "_TPL-04_preview.png"
    )
)

prototype_04_ground_truth_path = (
    TPL04_PROTOTYPE_ROOT
    / (
        f"{prototype_04_document_id}"
        "_TPL-04_ground_truth.json"
    )
)


# ================================================================
# Render prototype
# ================================================================

(
    prototype_04_annotations,
    prototype_04_rendering_metadata,
) = render_template_04(
    render_payload=prototype_04_render_payload,
    output_pdf_path=prototype_04_pdf_path,
)


# ================================================================
# Simpan ground truth
# ================================================================

prototype_04_ground_truth = {
    "schema_version": "1.0.0",
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "document": {
        "document_id": prototype_04_document_id,
        "canonical_invoice_id": (
            prototype_04_canonical_id
        ),
        "split": prototype_04_render_payload[
            "split"
        ],
        "template_id": TPL04_TEMPLATE_ID,
        "language": prototype_04_render_payload[
            "language"
        ],
        "currency": prototype_04_render_payload[
            "currency"
        ],
    },
    "canonical": prototype_04_canonical_payload,
    "rendering": (
        prototype_04_rendering_metadata
    ),
    "annotations": [
        annotation_to_dict(annotation)
        for annotation in prototype_04_annotations
    ],
}

write_json_atomically(
    prototype_04_ground_truth_path,
    prototype_04_ground_truth,
)


# ================================================================
# Buka ulang PDF dan buat preview
# ================================================================

temporary_png_path = (
    prototype_04_png_path.with_name(
        f".{prototype_04_png_path.stem}.tmp.png"
    )
)

temporary_png_path.unlink(
    missing_ok=True
)

try:
    with pymupdf.open(
        str(prototype_04_pdf_path)
    ) as prototype_document:
        if prototype_document.page_count != 1:
            raise RuntimeError(
                "Prototype TPL-04 harus satu halaman."
            )

        prototype_page = prototype_document[0]
        extracted_text_04 = prototype_page.get_text(
            "text"
        )

        prototype_pixmap = prototype_page.get_pixmap(
            matrix=pymupdf.Matrix(
                150 / 72,
                150 / 72,
            ),
            alpha=False,
        )

        prototype_pixmap.save(
            str(temporary_png_path)
        )

    if (
        not temporary_png_path.is_file()
        or temporary_png_path.stat().st_size == 0
    ):
        raise RuntimeError(
            "Preview PNG TPL-04 gagal dibuat."
        )

    os.replace(
        temporary_png_path,
        prototype_04_png_path,
    )

except Exception:
    temporary_png_path.unlink(
        missing_ok=True
    )
    raise


# ================================================================
# Pemeriksaan teknis awal
# ================================================================

required_text_fragments = [
    prototype_04_render_payload[
        "metadata"
    ]["invoice_number"]["display_value"],
    prototype_04_render_payload[
        "vendor"
    ]["name"],
    prototype_04_render_payload[
        "buyer"
    ]["name"],
    prototype_04_render_payload[
        "labels"
    ]["quantity"],
    prototype_04_render_payload[
        "financials"
    ]["total"]["display_value"],
    normalize_pdf_notice(
        prototype_04_render_payload[
            "synthetic_notice"
        ]
    ),
]

missing_text_fragments = [
    text
    for text in required_text_fragments
    if text not in extracted_text_04
]

if missing_text_fragments:
    raise RuntimeError(
        "Teks penting TPL-04 hilang dari PDF: "
        f"{missing_text_fragments}"
    )

annotation_names = {
    annotation.field_name
    for annotation in prototype_04_annotations
}

required_annotation_names = {
    "vendor.name",
    "vendor.address_lines",
    "vendor.email",
    "vendor.phone",
    "vendor.tax_identifier",
    "buyer.name",
    "buyer.address_lines",
    "buyer.email",
    "buyer.phone",
    "buyer.tax_identifier",
    "invoice_number",
    "invoice_date",
    "due_date",
    "currency",
    "financials.subtotal",
    "financials.tax",
    "financials.discount",
    "financials.total",
    "document.synthetic_notice",
}

missing_annotations = sorted(
    required_annotation_names
    - annotation_names
)

if missing_annotations:
    raise RuntimeError(
        "Anotasi wajib TPL-04 tidak tersedia: "
        f"{missing_annotations}"
    )

layout_metrics = (
    prototype_04_rendering_metadata[
        "layout_metrics"
    ]
)

prototype_04_summary = pd.DataFrame(
    [
        {
            "control": "pdf_created",
            "actual": prototype_04_pdf_path.is_file(),
            "status": "VALID",
        },
        {
            "control": "ground_truth_created",
            "actual": (
                prototype_04_ground_truth_path.is_file()
            ),
            "status": "VALID",
        },
        {
            "control": "annotation_count",
            "actual": len(
                prototype_04_annotations
            ),
            "status": "VALID",
        },
        {
            "control": "item_count",
            "actual": layout_metrics[
                "item_count"
            ],
            "status": "VALID",
        },
        {
            "control": "minimum_field_font",
            "actual": (
                prototype_04_rendering_metadata[
                    "minimum_field_font_size"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "font_adjustments",
            "actual": (
                prototype_04_rendering_metadata[
                    "font_adjustment_count"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "table_to_totals_gap",
            "actual": layout_metrics[
                "table_to_totals_gap_points"
            ],
            "status": "VALID",
        },
        {
            "control": "missing_required_text",
            "actual": len(
                missing_text_fragments
            ),
            "status": "VALID",
        },
    ]
)

display(prototype_04_summary)

with PILImage.open(
    prototype_04_png_path
) as prototype_image:
    display(prototype_image.copy())


# ================================================================
# Ringkasan
# ================================================================

print(
    f"Canonical ID       : {prototype_04_canonical_id}"
)
print(
    f"Document ID        : {prototype_04_document_id}"
)
print(
    f"Prototype PDF      : {prototype_04_pdf_path}"
)
print(
    f"Prototype preview  : {prototype_04_png_path}"
)
print(
    f"Ground truth       : {prototype_04_ground_truth_path}"
)
print(
    f"Annotations        : {len(prototype_04_annotations)}"
)
print(
    "Minimum field font : "
    f"{prototype_04_rendering_metadata['minimum_field_font_size']}"
)
print(
    "Adjusted fields    : "
    f"{prototype_04_rendering_metadata['font_adjustment_count']}"
)
print(
    "Table-total gap    : "
    f"{layout_metrics['table_to_totals_gap_points']} pt"
)
print(
    f"Renderer version   : {TPL04_RENDERER_VERSION}"
)
print()
print(
    "✅ Prototype TPL-04 berhasil dibuat dan "
    "lolos pemeriksaan teknis awal."
)


**CELL 50 — FINAL Audit QA teknis prototype TPL-04**

In [ ]:
# ================================================================
# CELL 50 — FINAL
# Audit QA teknis prototype TPL-04
# ================================================================

from pathlib import Path
from typing import Any

import hashlib
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd
import pymupdf
from PIL import Image as PILImage


# ================================================================
# Validasi dependency runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "TPL04_PROTOTYPE_ROOT",
    "TPL04_TEMPLATE_ID",
    "prototype_04_canonical_id",
    "prototype_04_document_id",
    "prototype_04_pdf_path",
    "prototype_04_png_path",
    "prototype_04_ground_truth_path",
    "prototype_04_render_payload",
    "prototype_04_rendering_metadata",
    "prototype_04_annotations",
    "annotation_to_dict",
    "write_json_atomically",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 50 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali Cell 49 versi final."
    )


# ================================================================
# Konfigurasi audit
# ================================================================

QA_SCHEMA_VERSION = "1.0.0"
QA_AUDITOR_VERSION = "1.0.0"

MINIMUM_FIELD_FONT_SIZE = 6.5
MAXIMUM_FONT_ADJUSTMENT_RATIO = 0.20
MINIMUM_RASTER_CONTENT_RATIO = 0.01
MAXIMUM_RASTER_CONTENT_RATIO = 0.40
MINIMUM_RASTER_CONTRAST = 10.0
SEVERE_OVERLAP_THRESHOLD = 0.50

prototype_04_pdf_path = Path(
    prototype_04_pdf_path
)
prototype_04_png_path = Path(
    prototype_04_png_path
)
prototype_04_ground_truth_path = Path(
    prototype_04_ground_truth_path
)

prototype_04_qa_report_path = (
    Path(TPL04_PROTOTYPE_ROOT)
    / (
        f"{prototype_04_document_id}"
        "_TPL-04_qa_report.json"
    )
)

required_files = [
    prototype_04_pdf_path,
    prototype_04_png_path,
    prototype_04_ground_truth_path,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Artifact TPL-04 belum lengkap: "
        f"{missing_files}"
    )


# ================================================================
# Helper umum
# ================================================================

def sha256_file(file_path: Path) -> str:
    digest = hashlib.sha256()

    with Path(file_path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def normalize_text(value: Any) -> str:
    text = unicodedata.normalize(
        "NFKC",
        str(value or ""),
    )

    text = (
        text
        .replace("\u2014", "-")
        .replace("\u2013", "-")
        .replace("\u00b7", "-")
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip().casefold()


def normalize_notice_for_pdf(value: Any) -> str:
    return (
        str(value)
        .replace("\u2014", "\u00b7")
        .replace("\u2013", "-")
    )


def first_present(
    mapping: dict[str, Any],
    candidate_keys: tuple[str, ...],
) -> Any:
    for key in candidate_keys:
        if key in mapping:
            return mapping[key]

    return None


def annotation_field_name(
    annotation: dict[str, Any],
) -> str:
    value = first_present(
        annotation,
        (
            "field_name",
            "field_path",
            "field",
            "path",
            "name",
        ),
    )

    if value is None:
        raise KeyError(
            "Nama field tidak ditemukan pada anotasi. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return str(value)


def annotation_text(
    annotation: dict[str, Any],
) -> str:
    value = first_present(
        annotation,
        (
            "text",
            "value",
            "display_value",
            "field_value",
            "raw_value",
            "raw_text",
            "text_value",
            "content",
        ),
    )

    if isinstance(value, dict):
        value = first_present(
            value,
            (
                "text",
                "value",
                "display_value",
                "raw",
            ),
        )

    if value is None:
        raise KeyError(
            "Nilai teks tidak ditemukan pada anotasi. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return str(value)


def bbox_from_value(
    value: Any,
) -> tuple[float, float, float, float] | None:
    if isinstance(value, (list, tuple)):
        if len(value) != 4:
            return None

        try:
            return tuple(
                float(number)
                for number in value
            )
        except (TypeError, ValueError):
            return None

    if not isinstance(value, dict):
        return None

    direct_key_sets = [
        ("x0", "y0", "x1", "y1"),
        ("left", "top", "right", "bottom"),
    ]

    for key_set in direct_key_sets:
        if all(key in value for key in key_set):
            try:
                return tuple(
                    float(value[key])
                    for key in key_set
                )
            except (TypeError, ValueError):
                return None

    if all(
        key in value
        for key in ("x", "y", "width", "height")
    ):
        try:
            x0 = float(value["x"])
            y0 = float(value["y"])
            width = float(value["width"])
            height = float(value["height"])

            return (
                x0,
                y0,
                x0 + width,
                y0 + height,
            )
        except (TypeError, ValueError):
            return None

    if all(
        key in value
        for key in ("x", "y", "w", "h")
    ):
        try:
            x0 = float(value["x"])
            y0 = float(value["y"])
            width = float(value["w"])
            height = float(value["h"])

            return (
                x0,
                y0,
                x0 + width,
                y0 + height,
            )
        except (TypeError, ValueError):
            return None

    return None


def annotation_bbox(
    annotation: dict[str, Any],
) -> tuple[float, float, float, float]:
    candidate = first_present(
        annotation,
        (
            "bbox",
            "bbox_points",
            "bbox_pt",
            "bbox_pdf",
            "bounding_box",
            "rectangle",
            "rect",
            "coordinates",
            "coordinates_points",
        ),
    )

    if candidate is None:
        geometry = annotation.get("geometry")

        if isinstance(geometry, dict):
            candidate = first_present(
                geometry,
                (
                    "bbox",
                    "bounding_box",
                    "rectangle",
                ),
            )

    bbox = bbox_from_value(candidate)

    if bbox is None:
        raise KeyError(
            "Bounding box tidak ditemukan atau tidak valid. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return bbox


def rectangle_area(
    bbox: tuple[float, float, float, float],
) -> float:
    x0, y0, x1, y1 = bbox

    return max(0.0, x1 - x0) * max(
        0.0,
        y1 - y0,
    )


def intersection_area(
    first_bbox: tuple[float, float, float, float],
    second_bbox: tuple[float, float, float, float],
) -> float:
    first_x0, first_y0, first_x1, first_y1 = first_bbox
    second_x0, second_y0, second_x1, second_y1 = second_bbox

    width = max(
        0.0,
        min(first_x1, second_x1)
        - max(first_x0, second_x0),
    )

    height = max(
        0.0,
        min(first_y1, second_y1)
        - max(first_y0, second_y0),
    )

    return width * height


def bbox_intersects_word(
    annotation_box: tuple[float, float, float, float],
    word_box: tuple[float, float, float, float],
) -> bool:
    return intersection_area(
        annotation_box,
        word_box,
    ) > 0.0


# ================================================================
# Muat ground truth
# ================================================================

with prototype_04_ground_truth_path.open(
    "r",
    encoding="utf-8",
) as ground_truth_file:
    ground_truth = json.load(
        ground_truth_file
    )

ground_truth_document = ground_truth.get(
    "document",
    {},
)

if (
    ground_truth_document.get("document_id")
    != prototype_04_document_id
):
    raise RuntimeError(
        "Document ID ground truth tidak sesuai."
    )

if (
    ground_truth_document.get("template_id")
    != TPL04_TEMPLATE_ID
):
    raise RuntimeError(
        "Ground truth bukan milik TPL-04."
    )

annotation_records = ground_truth.get(
    "annotations",
    [],
)

if not isinstance(annotation_records, list):
    raise TypeError(
        "ground_truth.annotations wajib berupa list."
    )

if not annotation_records:
    raise RuntimeError(
        "Ground truth tidak memiliki anotasi."
    )


# ================================================================
# Expected field mapping
# ================================================================

render_payload = prototype_04_render_payload

expected_values: dict[str, str] = {
    "invoice_number": str(
        render_payload["metadata"]
        ["invoice_number"]
        ["display_value"]
    ),
    "invoice_date": str(
        render_payload["metadata"]
        ["invoice_date"]
        ["display_value"]
    ),
    "due_date": str(
        render_payload["metadata"]
        ["due_date"]
        ["display_value"]
    ),
    "currency": str(
        render_payload["currency"]
    ),
}

for party_type in ("vendor", "buyer"):
    party = render_payload[party_type]

    expected_values.update(
        {
            f"{party_type}.name": str(
                party["name"]
            ),
            f"{party_type}.address_lines": "\n".join(
                party["address_lines"]
            ),
            f"{party_type}.email": str(
                party["email"]
            ),
            f"{party_type}.phone": str(
                party["phone"]
            ),
            f"{party_type}.tax_identifier": str(
                party["tax_identifier_display"]
            ),
        }
    )

for item_index, item in enumerate(
    render_payload["items"]
):
    for field_name in (
        "description",
        "quantity",
        "unit_price",
        "line_total",
    ):
        expected_values[
            f"items[{item_index}].{field_name}"
        ] = str(item[field_name])

for field_name in (
    "subtotal",
    "tax",
    "discount",
    "total",
):
    expected_values[
        f"financials.{field_name}"
    ] = str(
        render_payload["financials"]
        [field_name]
        ["display_value"]
    )

expected_values[
    "document.synthetic_notice"
] = normalize_notice_for_pdf(
    render_payload["synthetic_notice"]
)

expected_annotation_count = len(
    expected_values
)


# ================================================================
# Parse annotations
# ================================================================

parsed_annotations: list[dict[str, Any]] = []
annotation_parse_failures: list[dict[str, Any]] = []

for annotation_index, annotation in enumerate(
    annotation_records
):
    try:
        parsed_annotations.append(
            {
                "index": annotation_index,
                "field_name": annotation_field_name(
                    annotation
                ),
                "text": annotation_text(
                    annotation
                ),
                "bbox": annotation_bbox(
                    annotation
                ),
            }
        )
    except (KeyError, TypeError, ValueError) as error:
        annotation_parse_failures.append(
            {
                "index": annotation_index,
                "error": str(error),
                "keys": sorted(
                    annotation.keys()
                ) if isinstance(
                    annotation,
                    dict,
                ) else [],
            }
        )

if annotation_parse_failures:
    raise RuntimeError(
        "Schema anotasi tidak dapat diproses: "
        f"{annotation_parse_failures[:3]}"
    )

annotation_names = [
    annotation["field_name"]
    for annotation in parsed_annotations
]

annotation_by_name = {
    annotation["field_name"]: annotation
    for annotation in parsed_annotations
}

duplicate_annotation_count = (
    len(annotation_names)
    - len(set(annotation_names))
)

missing_required_annotations = sorted(
    set(expected_values)
    - set(annotation_names)
)


# ================================================================
# Audit PDF dan anotasi
# ================================================================

with pymupdf.open(
    str(prototype_04_pdf_path)
) as pdf_document:
    pdf_page_count = pdf_document.page_count

    if pdf_page_count < 1:
        raise RuntimeError(
            "PDF TPL-04 tidak memiliki halaman."
        )

    first_page = pdf_document[0]
    page_rectangle = first_page.rect
    extracted_text = first_page.get_text("text")
    extracted_words = first_page.get_text("words")

page_width = float(page_rectangle.width)
page_height = float(page_rectangle.height)

word_boxes = [
    (
        float(word[0]),
        float(word[1]),
        float(word[2]),
        float(word[3]),
    )
    for word in extracted_words
    if len(word) >= 5
    and str(word[4]).strip()
]

invalid_bounding_boxes: list[str] = []
annotations_without_words: list[str] = []

for annotation in parsed_annotations:
    field_name = annotation["field_name"]
    x0, y0, x1, y1 = annotation["bbox"]

    coordinates_are_finite = all(
        math.isfinite(value)
        for value in (x0, y0, x1, y1)
    )

    bbox_is_valid = (
        coordinates_are_finite
        and x0 >= 0.0
        and y0 >= 0.0
        and x1 <= page_width
        and y1 <= page_height
        and x1 > x0
        and y1 > y0
    )

    if not bbox_is_valid:
        invalid_bounding_boxes.append(
            field_name
        )
        continue

    contains_word = any(
        bbox_intersects_word(
            annotation["bbox"],
            word_box,
        )
        for word_box in word_boxes
    )

    if not contains_word:
        annotations_without_words.append(
            field_name
        )


# ================================================================
# Audit nilai anotasi
# ================================================================

normalization_mismatches: list[dict[str, str]] = []

for field_name, expected_value in expected_values.items():
    annotation = annotation_by_name.get(
        field_name
    )

    if annotation is None:
        continue

    actual_value = annotation["text"]

    if normalize_text(actual_value) != normalize_text(
        expected_value
    ):
        normalization_mismatches.append(
            {
                "field_name": field_name,
                "expected": expected_value,
                "actual": actual_value,
            }
        )

normalized_extracted_text = normalize_text(
    extracted_text
)

missing_extracted_values: list[str] = []

for annotation in parsed_annotations:
    normalized_value = normalize_text(
        annotation["text"]
    )

    if (
        normalized_value
        and normalized_value
        not in normalized_extracted_text
    ):
        missing_extracted_values.append(
            annotation["field_name"]
        )


# ================================================================
# Audit overlap anotasi
# ================================================================

severe_annotation_overlaps: list[dict[str, Any]] = []

for first_index in range(
    len(parsed_annotations)
):
    first_annotation = parsed_annotations[
        first_index
    ]
    first_area = rectangle_area(
        first_annotation["bbox"]
    )

    if first_area <= 0.0:
        continue

    for second_index in range(
        first_index + 1,
        len(parsed_annotations),
    ):
        second_annotation = parsed_annotations[
            second_index
        ]
        second_area = rectangle_area(
            second_annotation["bbox"]
        )

        if second_area <= 0.0:
            continue

        overlap_area = intersection_area(
            first_annotation["bbox"],
            second_annotation["bbox"],
        )

        overlap_ratio = overlap_area / min(
            first_area,
            second_area,
        )

        if overlap_ratio >= SEVERE_OVERLAP_THRESHOLD:
            severe_annotation_overlaps.append(
                {
                    "first": first_annotation[
                        "field_name"
                    ],
                    "second": second_annotation[
                        "field_name"
                    ],
                    "overlap_ratio": round(
                        overlap_ratio,
                        6,
                    ),
                }
            )


# ================================================================
# Audit font dan raster
# ================================================================

minimum_field_font_size = float(
    prototype_04_rendering_metadata[
        "minimum_field_font_size"
    ]
)

font_adjustment_count = int(
    prototype_04_rendering_metadata[
        "font_adjustment_count"
    ]
)

font_adjustment_ratio = (
    font_adjustment_count
    / max(1, len(parsed_annotations))
)

with PILImage.open(
    prototype_04_png_path
) as preview_image:
    grayscale_array = np.asarray(
        preview_image.convert("L"),
        dtype=np.float32,
    ).copy()

if grayscale_array.size == 0:
    raise RuntimeError(
        "Preview PNG tidak memiliki piksel."
    )

raster_content_ratio = float(
    np.mean(grayscale_array < 245.0)
)

raster_contrast = float(
    np.std(grayscale_array)
)


# ================================================================
# Susun hasil pemeriksaan
# ================================================================

checks = [
    {
        "control": "pdf_page_count",
        "expected": 1,
        "actual": pdf_page_count,
        "valid": pdf_page_count == 1,
    },
    {
        "control": "annotation_count",
        "expected": expected_annotation_count,
        "actual": len(parsed_annotations),
        "valid": (
            len(parsed_annotations)
            == expected_annotation_count
        ),
    },
    {
        "control": "duplicate_annotations",
        "expected": 0,
        "actual": duplicate_annotation_count,
        "valid": duplicate_annotation_count == 0,
    },
    {
        "control": "missing_required_annotations",
        "expected": 0,
        "actual": len(
            missing_required_annotations
        ),
        "valid": not missing_required_annotations,
    },
    {
        "control": "invalid_bounding_boxes",
        "expected": 0,
        "actual": len(
            invalid_bounding_boxes
        ),
        "valid": not invalid_bounding_boxes,
    },
    {
        "control": "normalization_mismatches",
        "expected": 0,
        "actual": len(
            normalization_mismatches
        ),
        "valid": not normalization_mismatches,
    },
    {
        "control": "annotations_without_words",
        "expected": 0,
        "actual": len(
            annotations_without_words
        ),
        "valid": not annotations_without_words,
    },
    {
        "control": "severe_annotation_overlaps",
        "expected": 0,
        "actual": len(
            severe_annotation_overlaps
        ),
        "valid": not severe_annotation_overlaps,
    },
    {
        "control": "missing_extracted_values",
        "expected": 0,
        "actual": len(
            missing_extracted_values
        ),
        "valid": not missing_extracted_values,
    },
    {
        "control": "minimum_field_font_size",
        "expected": ">=6.5",
        "actual": round(
            minimum_field_font_size,
            4,
        ),
        "valid": (
            minimum_field_font_size
            >= MINIMUM_FIELD_FONT_SIZE
        ),
    },
    {
        "control": "font_adjustment_ratio",
        "expected": "<=0.20",
        "actual": round(
            font_adjustment_ratio,
            4,
        ),
        "valid": (
            font_adjustment_ratio
            <= MAXIMUM_FONT_ADJUSTMENT_RATIO
        ),
    },
    {
        "control": "raster_content_ratio",
        "expected": "0.01–0.40",
        "actual": round(
            raster_content_ratio,
            4,
        ),
        "valid": (
            MINIMUM_RASTER_CONTENT_RATIO
            <= raster_content_ratio
            <= MAXIMUM_RASTER_CONTENT_RATIO
        ),
    },
    {
        "control": "raster_contrast",
        "expected": ">=10.0",
        "actual": round(
            raster_contrast,
            4,
        ),
        "valid": (
            raster_contrast
            >= MINIMUM_RASTER_CONTRAST
        ),
    },
]

for check in checks:
    check["status"] = (
        "VALID"
        if check["valid"]
        else "INVALID"
    )

failed_checks = [
    check
    for check in checks
    if not check["valid"]
]

qa_status = (
    "PASSED"
    if not failed_checks
    else "FAILED"
)


# ================================================================
# Simpan QA report
# ================================================================

qa_report = {
    "schema_version": QA_SCHEMA_VERSION,
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "document_id": prototype_04_document_id,
    "canonical_invoice_id": (
        prototype_04_canonical_id
    ),
    "template_id": TPL04_TEMPLATE_ID,
    "status": qa_status,
    "manual_visual_review": "PENDING",
    "metrics": {
        "auditor_version": QA_AUDITOR_VERSION,
        "pdf_page_count": pdf_page_count,
        "annotation_count": len(
            parsed_annotations
        ),
        "expected_annotation_count": (
            expected_annotation_count
        ),
        "duplicate_annotation_count": (
            duplicate_annotation_count
        ),
        "missing_required_annotation_count": len(
            missing_required_annotations
        ),
        "invalid_bounding_box_count": len(
            invalid_bounding_boxes
        ),
        "normalization_mismatch_count": len(
            normalization_mismatches
        ),
        "annotations_without_words_count": len(
            annotations_without_words
        ),
        "severe_annotation_overlap_count": len(
            severe_annotation_overlaps
        ),
        "missing_extracted_value_count": len(
            missing_extracted_values
        ),
        "minimum_field_font_size": round(
            minimum_field_font_size,
            6,
        ),
        "font_adjustment_count": (
            font_adjustment_count
        ),
        "font_adjustment_ratio": round(
            font_adjustment_ratio,
            6,
        ),
        "raster_content_ratio": round(
            raster_content_ratio,
            6,
        ),
        "raster_contrast": round(
            raster_contrast,
            6,
        ),
    },
    "failures": {
        "failed_checks": [
            check["control"]
            for check in failed_checks
        ],
        "missing_required_annotations": (
            missing_required_annotations
        ),
        "invalid_bounding_boxes": (
            invalid_bounding_boxes
        ),
        "normalization_mismatches": (
            normalization_mismatches
        ),
        "annotations_without_words": (
            annotations_without_words
        ),
        "severe_annotation_overlaps": (
            severe_annotation_overlaps
        ),
        "missing_extracted_values": (
            missing_extracted_values
        ),
    },
    "checksums_sha256": {
        "prototype_pdf": sha256_file(
            prototype_04_pdf_path
        ),
        "preview": sha256_file(
            prototype_04_png_path
        ),
        "ground_truth": sha256_file(
            prototype_04_ground_truth_path
        ),
    },
}

write_json_atomically(
    prototype_04_qa_report_path,
    qa_report,
)


# ================================================================
# Verifikasi round-trip QA report
# ================================================================

with prototype_04_qa_report_path.open(
    "r",
    encoding="utf-8",
) as qa_report_file:
    verified_qa_report = json.load(
        qa_report_file
    )

if verified_qa_report != qa_report:
    raise RuntimeError(
        "QA report berubah setelah disimpan dan dibuka ulang."
    )


# ================================================================
# Output audit
# ================================================================

qa_summary = pd.DataFrame(
    [
        {
            "control": check["control"],
            "expected": check["expected"],
            "actual": check["actual"],
            "status": check["status"],
        }
        for check in checks
    ]
)

display(qa_summary)

print(
    f"QA report          : "
    f"{prototype_04_qa_report_path}"
)
print(
    f"QA status          : {qa_status}"
)
print(
    "Minimum field font : "
    f"{minimum_field_font_size:.2f}"
)
print(
    "Font adjustments   : "
    f"{font_adjustment_count}"
)
print(
    "Ink pixel ratio    : "
    f"{raster_content_ratio:.6f}"
)
print(
    "Raster contrast    : "
    f"{raster_contrast:.6f}"
)
print(
    "Manual review      : "
    f"{qa_report['manual_visual_review']}"
)
print()

if qa_status != "PASSED":
    print("Detail kegagalan:")
    print(
        json.dumps(
            qa_report["failures"],
            ensure_ascii=False,
            indent=2,
        )
    )

    raise RuntimeError(
        "Prototype TPL-04 gagal audit QA teknis."
    )

print(
    "✅ Prototype TPL-04 lulus audit teknis. "
    "Review visual manual masih perlu dicatat "
    "ke QA report."
)


In [ ]:
# CELL 50A — Finalisasi review visual TPL-04

import json


qa_report_path = prototype_04_qa_report_path

with qa_report_path.open(
    "r",
    encoding="utf-8",
) as file:
    qa_report = json.load(file)


# Validasi identitas dan status
if qa_report.get("template_id") != "TPL-04":
    raise RuntimeError(
        "QA report bukan milik TPL-04."
    )

if qa_report.get("document_id") != (
    prototype_04_document_id
):
    raise RuntimeError(
        "Document ID QA report tidak sesuai."
    )

if qa_report.get("status") != "PASSED":
    raise RuntimeError(
        "Review visual tidak dapat disahkan karena "
        f"status teknis adalah "
        f"{qa_report.get('status')!r}."
    )


# Pastikan artifact tidak berubah setelah audit
artifact_checks = {
    "prototype_pdf": prototype_04_pdf_path,
    "preview": prototype_04_png_path,
    "ground_truth": (
        prototype_04_ground_truth_path
    ),
}

for artifact_name, artifact_path in (
    artifact_checks.items()
):
    expected_checksum = (
        qa_report[
            "checksums_sha256"
        ][artifact_name]
    )

    actual_checksum = sha256_file(
        artifact_path
    )

    if actual_checksum != expected_checksum:
        raise RuntimeError(
            f"Artifact {artifact_name!r} berubah "
            "setelah audit. Jalankan ulang Cell 50."
        )


# Catat review visual
qa_report[
    "manual_visual_review"
] = "PASSED"

write_json_atomically(
    qa_report_path,
    qa_report,
)


# Verifikasi hasil penyimpanan
with qa_report_path.open(
    "r",
    encoding="utf-8",
) as file:
    verified_report = json.load(file)

if (
    verified_report.get(
        "manual_visual_review"
    )
    != "PASSED"
):
    raise RuntimeError(
        "Status review visual gagal disimpan."
    )


print(f"QA report           : {qa_report_path}")
print(
    "Technical status    : "
    f"{verified_report['status']}"
)
print(
    "Manual visual review: "
    f"{verified_report['manual_visual_review']}"
)
print()
print(
    "✅ TPL-04 FINAL PASSED — audit teknis "
    "dan review visual telah selesai."
)

**CELL 51 — Inspeksi konfigurasi dan payload TPL-05**

In [ ]:
# ================================================================
# CELL 51 — Inspeksi konfigurasi dan payload TPL-05
# ================================================================

import json
import pandas as pd


# Pastikan TPL-04 sudah final
with prototype_04_qa_report_path.open(
    "r",
    encoding="utf-8",
) as file:
    tpl04_final_report = json.load(file)

if (
    tpl04_final_report.get("status") != "PASSED"
    or tpl04_final_report.get(
        "manual_visual_review"
    ) != "PASSED"
):
    raise RuntimeError(
        "TPL-04 belum FINAL PASSED."
    )

print("TPL-04 status       : FINAL PASSED")


# Ambil payload TPL-05
TPL05_TEMPLATE_ID = "TPL-05"

if TPL05_TEMPLATE_ID not in template_registry:
    raise KeyError(
        "TPL-05 tidak ditemukan dalam template_registry."
    )

tpl05_payloads = {
    canonical_id: payload
    for canonical_id, payload
    in invoice_render_payloads.items()
    if payload.get("template_id")
    == TPL05_TEMPLATE_ID
}

tpl05_candidate_ids = sorted(
    tpl05_payloads
)

if len(tpl05_candidate_ids) != 20:
    raise RuntimeError(
        "TPL-05 harus memiliki tepat 20 payload. "
        f"Actual: {len(tpl05_candidate_ids)}."
    )

sample_payload = tpl05_payloads[
    tpl05_candidate_ids[0]
]


def text_length(value):
    return len(str(value or ""))


def address_length(party):
    return text_length(
        " ".join(
            party.get("address_lines", [])
        )
    )


statistics = []

for canonical_id in tpl05_candidate_ids:
    payload = tpl05_payloads[canonical_id]
    vendor = payload["vendor"]
    buyer = payload["buyer"]
    items = payload["items"]
    financials = payload["financials"]

    statistics.append(
        {
            "canonical_id": canonical_id,
            "document_id": payload["document_id"],
            "language": payload["language"],
            "currency": payload["currency"],
            "item_count": len(items),
            "vendor_name_length": text_length(
                vendor["name"]
            ),
            "vendor_address_length": (
                address_length(vendor)
            ),
            "vendor_email_length": text_length(
                vendor["email"]
            ),
            "vendor_phone_length": text_length(
                vendor["phone"]
            ),
            "vendor_tax_length": text_length(
                vendor[
                    "tax_identifier_display"
                ]
            ),
            "buyer_name_length": text_length(
                buyer["name"]
            ),
            "buyer_address_length": (
                address_length(buyer)
            ),
            "buyer_email_length": text_length(
                buyer["email"]
            ),
            "buyer_phone_length": text_length(
                buyer["phone"]
            ),
            "buyer_tax_length": text_length(
                buyer[
                    "tax_identifier_display"
                ]
            ),
            "maximum_description_length": max(
                text_length(
                    item["description"]
                )
                for item in items
            ),
            "maximum_money_length": max(
                text_length(
                    field["display_value"]
                )
                for field
                in financials.values()
            ),
        }
    )

statistics_frame = pd.DataFrame(
    statistics
)

sample_structure = {
    "canonical_invoice_id": (
        sample_payload["canonical_invoice_id"]
    ),
    "document_id": sample_payload[
        "document_id"
    ],
    "template_id": sample_payload[
        "template_id"
    ],
    "language": sample_payload["language"],
    "currency": sample_payload["currency"],
    "title": sample_payload["title"],
    "top_level_keys": sorted(
        sample_payload.keys()
    ),
    "label_keys": sorted(
        sample_payload["labels"].keys()
    ),
    "metadata_keys": sorted(
        sample_payload["metadata"].keys()
    ),
    "vendor_keys": sorted(
        sample_payload["vendor"].keys()
    ),
    "buyer_keys": sorted(
        sample_payload["buyer"].keys()
    ),
    "item_keys": sorted(
        sample_payload["items"][0].keys()
    ),
    "financial_keys": sorted(
        sample_payload["financials"].keys()
    ),
    "footer_keys": sorted(
        sample_payload["footer"].keys()
    ),
}


print("\nTEMPLATE SPECIFICATION")
print(
    json.dumps(
        template_registry[
            TPL05_TEMPLATE_ID
        ],
        ensure_ascii=False,
        indent=2,
        default=str,
    )
)

print("\nPAYLOAD COUNT")
print(len(tpl05_candidate_ids))

print("\nSAMPLE STRUCTURE")
print(
    json.dumps(
        sample_structure,
        ensure_ascii=False,
        indent=2,
    )
)

print("\nLABELS")
print(
    json.dumps(
        sample_payload["labels"],
        ensure_ascii=False,
        indent=2,
    )
)

print("\nPAYLOAD STATISTICS")
display(
    statistics_frame.describe(
        include="all"
    ).transpose()
)

print("\nITEM-COUNT DISTRIBUTION")
display(
    statistics_frame[
        "item_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("item_count")
    .reset_index(name="document_count")
)

print()
print(
    "✅ Inspeksi TPL-05 selesai. "
    "Belum ada artifact yang dibuat atau diubah."
)

CELL 52 — FINAL
 Prototype renderer TPL-05: Centered Editorial

In [ ]:
# ================================================================
# CELL 52 — FINAL
# Prototype renderer TPL-05: Centered Editorial
# ================================================================

from pathlib import Path
from typing import Any
from PIL import Image as PILImage

import json
import os

import pandas as pd
import pymupdf


# ================================================================
# Validasi runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "PROTOTYPE_ROOT",
    "CANONICAL_RECORDS_JSONL_PATH",
    "FieldAnnotation",
    "template_registry",
    "invoice_render_payloads",
    "get_page_dimensions",
    "hex_to_rgb",
    "insert_annotated_text",
    "insert_static_text",
    "annotation_to_dict",
    "write_json_atomically",
    "RENDER_ENGINE_VERSION",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 52 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali cell pemulihan runtime."
    )


# ================================================================
# Konfigurasi TPL-05
# ================================================================

TPL05_TEMPLATE_ID = "TPL-05"
TPL05_RENDERER_VERSION = "1.0.0"
TPL05_MIN_ITEMS_PER_PAGE = 2
TPL05_MAX_ITEMS_PER_PAGE = 8

TPL05_PROTOTYPE_ROOT = (
    Path(PROTOTYPE_ROOT)
    / TPL05_TEMPLATE_ID
)

TPL05_PROTOTYPE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ================================================================
# Helper payload dan teks PDF
# ================================================================

def load_tpl05_canonical_payload(
    canonical_id: str,
) -> dict[str, Any]:
    canonical_path = Path(
        CANONICAL_RECORDS_JSONL_PATH
    )

    if not canonical_path.is_file():
        raise FileNotFoundError(
            "Canonical JSONL tidak ditemukan: "
            f"{canonical_path}"
        )

    with canonical_path.open(
        "r",
        encoding="utf-8",
    ) as canonical_file:
        for line_number, line in enumerate(
            canonical_file,
            start=1,
        ):
            clean_line = line.strip()

            if not clean_line:
                continue

            try:
                payload = json.loads(clean_line)
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    "Canonical JSONL tidak valid pada "
                    f"baris {line_number}."
                ) from error

            if (
                payload.get("canonical_invoice_id")
                == canonical_id
            ):
                return payload

    raise KeyError(
        "Canonical payload tidak ditemukan: "
        f"{canonical_id}"
    )


def normalize_pdf_notice(text: Any) -> str:
    return (
        str(text)
        .replace("—", "·")
        .replace("–", "-")
    )


# ================================================================
# Renderer TPL-05
# ================================================================

def render_template_05(
    render_payload: dict[str, Any],
    output_pdf_path: Path,
) -> tuple[list[FieldAnnotation], dict[str, Any]]:
    if not isinstance(render_payload, dict):
        raise TypeError(
            "render_payload wajib berupa dictionary."
        )

    if render_payload.get("template_id") != TPL05_TEMPLATE_ID:
        raise ValueError(
            "Renderer TPL-05 hanya menerima payload TPL-05."
        )

    required_payload_keys = {
        "canonical_invoice_id",
        "document_id",
        "template_id",
        "language",
        "currency",
        "title",
        "labels",
        "metadata",
        "vendor",
        "buyer",
        "items",
        "financials",
        "footer",
        "synthetic_notice",
    }

    missing_payload_keys = sorted(
        required_payload_keys
        - set(render_payload)
    )

    if missing_payload_keys:
        raise KeyError(
            "Payload TPL-05 belum lengkap: "
            f"{missing_payload_keys}"
        )

    item_count = len(render_payload["items"])

    if not (
        TPL05_MIN_ITEMS_PER_PAGE
        <= item_count
        <= TPL05_MAX_ITEMS_PER_PAGE
    ):
        raise ValueError(
            "TPL-05 hanya mendukung 2–8 item per halaman."
        )

    template_specification = template_registry[
        TPL05_TEMPLATE_ID
    ]

    page_width, page_height = get_page_dimensions(
        template_specification["page_size"]
    )

    primary_color = hex_to_rgb(
        template_specification["primary_color"]
    )

    accent_color = hex_to_rgb(
        template_specification["accent_color"]
    )

    dark_text = (0.17, 0.18, 0.24)
    muted_text = (0.43, 0.43, 0.49)
    border_color = (0.84, 0.83, 0.86)
    light_fill = (0.972, 0.968, 0.974)
    total_fill = (0.935, 0.925, 0.940)
    white = (1.0, 1.0, 1.0)

    output_pdf_path = Path(output_pdf_path)

    output_pdf_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_pdf_path = output_pdf_path.with_name(
        f".{output_pdf_path.stem}.tmp.pdf"
    )

    temporary_pdf_path.unlink(
        missing_ok=True
    )

    annotations: list[FieldAnnotation] = []
    field_font_audit: list[dict[str, Any]] = []

    document = pymupdf.open()

    def add_field(
        page: pymupdf.Page,
        rectangle: pymupdf.Rect,
        text: str,
        field_name: str,
        font_name: str = "helv",
        font_size: float = 7.2,
        minimum_font_size: float = 6.5,
        font_color: tuple[
            float,
            float,
            float,
        ] = dark_text,
        alignment: str = "left",
    ) -> None:
        annotation, used_font_size = insert_annotated_text(
            page=page,
            rectangle=rectangle,
            text=str(text),
            field_name=field_name,
            font_name=font_name,
            font_size=font_size,
            minimum_font_size=minimum_font_size,
            font_color=font_color,
            alignment=alignment,
        )

        annotations.append(annotation)

        field_font_audit.append(
            {
                "field_name": field_name,
                "requested_font_size": font_size,
                "used_font_size": used_font_size,
                "adjusted": used_font_size < font_size,
            }
        )

    try:
        page = document.new_page(
            width=page_width,
            height=page_height,
        )

        margin_x0 = 50.0
        margin_x1 = page_width - 50.0

        # --------------------------------------------------------
        # Header terpusat
        # --------------------------------------------------------

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                38,
                margin_x1,
                77,
            ),
            text=render_payload["title"],
            font_name="hebo",
            font_size=26.0,
            minimum_font_size=19.0,
            font_color=primary_color,
            alignment="center",
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                82,
                margin_x1,
                98,
            ),
            text="EDITORIAL · SYNTHETIC",
            font_name="hebo",
            font_size=6.5,
            minimum_font_size=6.0,
            font_color=accent_color,
            alignment="center",
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                104,
                margin_x1,
                119,
            ),
            text=render_payload["footer"][
                "document_reference"
            ],
            font_name="helv",
            font_size=6.7,
            minimum_font_size=6.0,
            font_color=muted_text,
            alignment="center",
        )

        # --------------------------------------------------------
        # Metadata empat kolom
        # --------------------------------------------------------

        metadata_y0 = 134.0
        metadata_y1 = 187.0

        page.draw_rect(
            pymupdf.Rect(
                margin_x0,
                metadata_y0,
                margin_x1,
                metadata_y1,
            ),
            color=None,
            fill=light_fill,
            width=0,
            overlay=True,
        )

        metadata_edges = [
            margin_x0,
            174.0,
            298.0,
            422.0,
            margin_x1,
        ]

        currency_label = (
            "Mata Uang"
            if render_payload["language"] == "id"
            else "Currency"
        )

        metadata_entries = [
            (
                "invoice_number",
                render_payload["metadata"]
                ["invoice_number"]["label"],
                render_payload["metadata"]
                ["invoice_number"]["display_value"],
            ),
            (
                "invoice_date",
                render_payload["metadata"]
                ["invoice_date"]["label"],
                render_payload["metadata"]
                ["invoice_date"]["display_value"],
            ),
            (
                "due_date",
                render_payload["metadata"]
                ["due_date"]["label"],
                render_payload["metadata"]
                ["due_date"]["display_value"],
            ),
            (
                "currency",
                currency_label,
                render_payload["currency"],
            ),
        ]

        for column_index, (
            field_name,
            label,
            value,
        ) in enumerate(metadata_entries):
            column_x0 = metadata_edges[column_index]
            column_x1 = metadata_edges[column_index + 1]

            if column_index > 0:
                page.draw_line(
                    pymupdf.Point(
                        column_x0,
                        metadata_y0 + 10,
                    ),
                    pymupdf.Point(
                        column_x0,
                        metadata_y1 - 10,
                    ),
                    color=border_color,
                    width=0.6,
                    overlay=True,
                )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    column_x0 + 8,
                    metadata_y0 + 8,
                    column_x1 - 8,
                    metadata_y0 + 23,
                ),
                text=label,
                font_name="helv",
                font_size=6.2,
                minimum_font_size=5.7,
                font_color=muted_text,
                alignment="center",
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    column_x0 + 7,
                    metadata_y0 + 29,
                    column_x1 - 7,
                    metadata_y0 + 47,
                ),
                text=value,
                field_name=field_name,
                font_name="hebo",
                font_size=7.3,
                minimum_font_size=6.5,
                font_color=primary_color,
                alignment="center",
            )

        # --------------------------------------------------------
        # Vendor dan buyer horizontal
        # --------------------------------------------------------

        party_y0 = 205.0
        party_y1 = 353.0

        party_cards = [
            (
                "vendor",
                render_payload["labels"]["vendor"],
                margin_x0,
                285.0,
            ),
            (
                "buyer",
                render_payload["labels"]["buyer"],
                310.0,
                margin_x1,
            ),
        ]

        for (
            party_type,
            party_label,
            card_x0,
            card_x1,
        ) in party_cards:
            party = render_payload[party_type]

            page.draw_rect(
                pymupdf.Rect(
                    card_x0,
                    party_y0,
                    card_x1,
                    party_y1,
                ),
                color=border_color,
                fill=white,
                width=0.7,
                overlay=True,
            )

            page.draw_line(
                pymupdf.Point(
                    card_x0,
                    party_y0,
                ),
                pymupdf.Point(
                    card_x1,
                    party_y0,
                ),
                color=accent_color,
                width=2.0,
                overlay=True,
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 14,
                    party_y0 + 10,
                    card_x1 - 14,
                    party_y0 + 25,
                ),
                text=party_label.upper(),
                font_name="hebo",
                font_size=6.6,
                minimum_font_size=6.0,
                font_color=accent_color,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 14,
                    party_y0 + 32,
                    card_x1 - 14,
                    party_y0 + 51,
                ),
                text=party["name"],
                field_name=f"{party_type}.name",
                font_name="hebo",
                font_size=8.0,
                minimum_font_size=6.5,
                font_color=primary_color,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 14,
                    party_y0 + 55,
                    card_x1 - 14,
                    party_y0 + 90,
                ),
                text="\n".join(
                    party["address_lines"]
                ),
                field_name=f"{party_type}.address_lines",
                font_name="helv",
                font_size=6.8,
                minimum_font_size=6.5,
                font_color=dark_text,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 14,
                    party_y0 + 94,
                    card_x1 - 14,
                    party_y0 + 108,
                ),
                text=party["email"],
                field_name=f"{party_type}.email",
                font_name="helv",
                font_size=6.7,
                minimum_font_size=6.5,
                font_color=dark_text,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 14,
                    party_y0 + 111,
                    card_x1 - 14,
                    party_y0 + 125,
                ),
                text=party["phone"],
                field_name=f"{party_type}.phone",
                font_name="helv",
                font_size=6.7,
                minimum_font_size=6.5,
                font_color=dark_text,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 14,
                    party_y0 + 128,
                    card_x1 - 14,
                    party_y0 + 143,
                ),
                text=party["tax_identifier_display"],
                field_name=f"{party_type}.tax_identifier",
                font_name="helv",
                font_size=6.7,
                minimum_font_size=6.5,
                font_color=dark_text,
            )

        # --------------------------------------------------------
        # Tabel open rows
        # --------------------------------------------------------

        table_positions = [
            margin_x0,
            75.0,
            320.0,
            375.0,
            460.0,
            margin_x1,
        ]

        table_header_y0 = 379.0
        table_header_y1 = 405.0
        row_height = 26.0

        page.draw_line(
            pymupdf.Point(
                table_positions[0],
                table_header_y0,
            ),
            pymupdf.Point(
                table_positions[-1],
                table_header_y0,
            ),
            color=primary_color,
            width=0.9,
            overlay=True,
        )

        page.draw_line(
            pymupdf.Point(
                table_positions[0],
                table_header_y1,
            ),
            pymupdf.Point(
                table_positions[-1],
                table_header_y1,
            ),
            color=accent_color,
            width=1.1,
            overlay=True,
        )

        table_headers = [
            "#",
            render_payload["labels"]["description"],
            render_payload["labels"]["quantity"],
            render_payload["labels"]["unit_price"],
            render_payload["labels"]["line_total"],
        ]

        for column_index, header_text in enumerate(
            table_headers
        ):
            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    table_positions[column_index] + 3,
                    table_header_y0 + 7,
                    table_positions[column_index + 1] - 3,
                    table_header_y1 - 2,
                ),
                text=header_text,
                font_name="hebo",
                font_size=6.5,
                minimum_font_size=5.8,
                font_color=primary_color,
                alignment=(
                    "left"
                    if column_index == 1
                    else "center"
                ),
            )

        for item_index, item in enumerate(
            render_payload["items"]
        ):
            row_y0 = (
                table_header_y1
                + item_index * row_height
            )
            row_y1 = row_y0 + row_height

            page.draw_line(
                pymupdf.Point(
                    table_positions[0],
                    row_y1,
                ),
                pymupdf.Point(
                    table_positions[-1],
                    row_y1,
                ),
                color=border_color,
                width=0.45,
                overlay=True,
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    table_positions[0] + 3,
                    row_y0 + 7,
                    table_positions[1] - 3,
                    row_y1 - 2,
                ),
                text=str(item["item_number"]),
                font_name="helv",
                font_size=6.8,
                minimum_font_size=6.0,
                font_color=muted_text,
                alignment="center",
            )

            item_fields = [
                ("description", 1, "left"),
                ("quantity", 2, "center"),
                ("unit_price", 3, "right"),
                ("line_total", 4, "right"),
            ]

            for (
                field_name,
                column_index,
                alignment,
            ) in item_fields:
                add_field(
                    page=page,
                    rectangle=pymupdf.Rect(
                        table_positions[column_index] + 4,
                        row_y0 + 7,
                        table_positions[column_index + 1] - 4,
                        row_y1 - 2,
                    ),
                    text=item[field_name],
                    field_name=(
                        f"items[{item_index}].{field_name}"
                    ),
                    font_name="helv",
                    font_size=7.0,
                    minimum_font_size=6.5,
                    font_color=dark_text,
                    alignment=alignment,
                )

        table_end_y = (
            table_header_y1
            + item_count * row_height
        )

        # --------------------------------------------------------
        # Full-width financial summary
        # --------------------------------------------------------

        totals_panel_height = 84.0

        totals_y0 = min(
            max(
                table_end_y + 24.0,
                634.0,
            ),
            650.0,
        )

        totals_y1 = (
            totals_y0
            + totals_panel_height
        )

        table_to_totals_gap = (
            totals_y0 - table_end_y
        )

        if table_to_totals_gap < 22.0:
            raise RuntimeError(
                "Jarak tabel dan ringkasan total terlalu sempit: "
                f"{table_to_totals_gap:.2f} pt."
            )

        footer_line_y = 768.0

        if totals_y1 >= footer_line_y - 24.0:
            raise RuntimeError(
                "Ringkasan total terlalu dekat dengan footer."
            )

        page.draw_rect(
            pymupdf.Rect(
                margin_x0,
                totals_y0,
                margin_x1,
                totals_y1,
            ),
            color=None,
            fill=light_fill,
            width=0,
            overlay=True,
        )

        page.draw_line(
            pymupdf.Point(
                margin_x0,
                totals_y1,
            ),
            pymupdf.Point(
                margin_x1,
                totals_y1,
            ),
            color=accent_color,
            width=2.0,
            overlay=True,
        )

        totals_edges = [
            margin_x0,
            174.0,
            298.0,
            422.0,
            margin_x1,
        ]

        financial_order = [
            "subtotal",
            "tax",
            "discount",
            "total",
        ]

        for column_index, field_name in enumerate(
            financial_order
        ):
            field_payload = render_payload[
                "financials"
            ][field_name]

            column_x0 = totals_edges[column_index]
            column_x1 = totals_edges[column_index + 1]
            is_total = field_name == "total"

            if is_total:
                page.draw_rect(
                    pymupdf.Rect(
                        column_x0,
                        totals_y0,
                        column_x1,
                        totals_y1,
                    ),
                    color=None,
                    fill=total_fill,
                    width=0,
                    overlay=True,
                )

            elif column_index > 0:
                page.draw_line(
                    pymupdf.Point(
                        column_x0,
                        totals_y0 + 13,
                    ),
                    pymupdf.Point(
                        column_x0,
                        totals_y1 - 13,
                    ),
                    color=border_color,
                    width=0.6,
                    overlay=True,
                )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    column_x0 + 9,
                    totals_y0 + 14,
                    column_x1 - 9,
                    totals_y0 + 30,
                ),
                text=field_payload["label"],
                font_name=(
                    "hebo"
                    if is_total
                    else "helv"
                ),
                font_size=6.8,
                minimum_font_size=6.0,
                font_color=(
                    primary_color
                    if is_total
                    else muted_text
                ),
                alignment="center",
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    column_x0 + 8,
                    totals_y0 + 41,
                    column_x1 - 8,
                    totals_y0 + 63,
                ),
                text=field_payload["display_value"],
                field_name=f"financials.{field_name}",
                font_name=(
                    "hebo"
                    if is_total
                    else "helv"
                ),
                font_size=(
                    8.0
                    if is_total
                    else 7.2
                ),
                minimum_font_size=6.5,
                font_color=(
                    accent_color
                    if is_total
                    else dark_text
                ),
                alignment="center",
            )

        # --------------------------------------------------------
        # Footer dan aksen bawah
        # --------------------------------------------------------

        page.draw_line(
            pymupdf.Point(margin_x0, footer_line_y),
            pymupdf.Point(margin_x1, footer_line_y),
            color=border_color,
            width=0.7,
            overlay=True,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                777,
                margin_x1,
                792,
            ),
            text=render_payload["footer"][
                "document_reference"
            ],
            font_name="helv",
            font_size=6.5,
            minimum_font_size=6.0,
            font_color=muted_text,
            alignment="center",
        )

        notice_display_text = normalize_pdf_notice(
            render_payload["synthetic_notice"]
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                805,
                margin_x1,
                823,
            ),
            text=notice_display_text,
            field_name="document.synthetic_notice",
            font_name="hebo",
            font_size=6.6,
            minimum_font_size=6.5,
            font_color=accent_color,
            alignment="center",
        )

        page.draw_rect(
            pymupdf.Rect(
                0,
                page_height - 8,
                page_width,
                page_height,
            ),
            color=accent_color,
            fill=accent_color,
            width=0,
            overlay=True,
        )

        # --------------------------------------------------------
        # Validasi sebelum simpan
        # --------------------------------------------------------

        annotation_names = [
            annotation.field_name
            for annotation in annotations
        ]

        if len(annotation_names) != len(
            set(annotation_names)
        ):
            raise RuntimeError(
                "Ditemukan nama anotasi duplikat."
            )

        expected_annotation_count = (
            19 + 4 * item_count
        )

        if len(annotations) != expected_annotation_count:
            raise RuntimeError(
                "Jumlah anotasi tidak sesuai. "
                f"Expected: {expected_annotation_count}; "
                f"actual: {len(annotations)}."
            )

        if not field_font_audit:
            raise RuntimeError(
                "Audit font field tidak boleh kosong."
            )

        minimum_used_font_size = min(
            row["used_font_size"]
            for row in field_font_audit
        )

        if minimum_used_font_size < 6.5:
            raise RuntimeError(
                "Font field berada di bawah batas 6.5 pt."
            )

        document.save(
            str(temporary_pdf_path),
            garbage=4,
            deflate=True,
        )

    except Exception:
        temporary_pdf_path.unlink(
            missing_ok=True
        )
        raise

    finally:
        document.close()

    if (
        not temporary_pdf_path.is_file()
        or temporary_pdf_path.stat().st_size == 0
    ):
        temporary_pdf_path.unlink(
            missing_ok=True
        )
        raise RuntimeError(
            "PDF sementara TPL-05 gagal dibuat."
        )

    os.replace(
        temporary_pdf_path,
        output_pdf_path,
    )

    rendering_metadata = {
        "renderer_version": RENDER_ENGINE_VERSION,
        "template_renderer_version": (
            TPL05_RENDERER_VERSION
        ),
        "template_id": TPL05_TEMPLATE_ID,
        "layout_family": template_specification[
            "layout_family"
        ],
        "page_size": template_specification[
            "page_size"
        ],
        "page_width_points": round(
            page_width,
            3,
        ),
        "page_height_points": round(
            page_height,
            3,
        ),
        "page_count": 1,
        "annotation_type": "field_region",
        "annotation_count": len(annotations),
        "minimum_field_font_size": min(
            row["used_font_size"]
            for row in field_font_audit
        ),
        "font_adjustment_count": int(
            sum(
                bool(row["adjusted"])
                for row in field_font_audit
            )
        ),
        "layout_metrics": {
            "item_count": item_count,
            "table_end_y_points": round(
                table_end_y,
                3,
            ),
            "totals_panel_y0_points": round(
                totals_y0,
                3,
            ),
            "totals_panel_y1_points": round(
                totals_y1,
                3,
            ),
            "table_to_totals_gap_points": round(
                table_to_totals_gap,
                3,
            ),
            "footer_line_y_points": footer_line_y,
        },
    }

    return annotations, rendering_metadata


# ================================================================
# Pilih prototype pertama TPL-05
# ================================================================

tpl05_candidate_ids = sorted(
    canonical_id
    for canonical_id, payload
    in invoice_render_payloads.items()
    if payload.get("template_id")
    == TPL05_TEMPLATE_ID
)

if len(tpl05_candidate_ids) != 20:
    raise RuntimeError(
        "TPL-05 harus memiliki tepat 20 payload. "
        f"Actual: {len(tpl05_candidate_ids)}."
    )

prototype_05_canonical_id = (
    tpl05_candidate_ids[0]
)

prototype_05_render_payload = (
    invoice_render_payloads[
        prototype_05_canonical_id
    ]
)

prototype_05_document_id = (
    prototype_05_render_payload[
        "document_id"
    ]
)

prototype_05_canonical_payload = (
    load_tpl05_canonical_payload(
        prototype_05_canonical_id
    )
)


# ================================================================
# Lokasi artifact
# ================================================================

prototype_05_pdf_path = (
    TPL05_PROTOTYPE_ROOT
    / (
        f"{prototype_05_document_id}"
        "_TPL-05_prototype.pdf"
    )
)

prototype_05_png_path = (
    TPL05_PROTOTYPE_ROOT
    / (
        f"{prototype_05_document_id}"
        "_TPL-05_preview.png"
    )
)

prototype_05_ground_truth_path = (
    TPL05_PROTOTYPE_ROOT
    / (
        f"{prototype_05_document_id}"
        "_TPL-05_ground_truth.json"
    )
)


# ================================================================
# Render prototype
# ================================================================

(
    prototype_05_annotations,
    prototype_05_rendering_metadata,
) = render_template_05(
    render_payload=prototype_05_render_payload,
    output_pdf_path=prototype_05_pdf_path,
)


# ================================================================
# Simpan ground truth
# ================================================================

prototype_05_ground_truth = {
    "schema_version": "1.0.0",
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "document": {
        "document_id": prototype_05_document_id,
        "canonical_invoice_id": (
            prototype_05_canonical_id
        ),
        "split": prototype_05_render_payload[
            "split"
        ],
        "template_id": TPL05_TEMPLATE_ID,
        "language": prototype_05_render_payload[
            "language"
        ],
        "currency": prototype_05_render_payload[
            "currency"
        ],
    },
    "canonical": prototype_05_canonical_payload,
    "rendering": (
        prototype_05_rendering_metadata
    ),
    "annotations": [
        annotation_to_dict(annotation)
        for annotation in prototype_05_annotations
    ],
}

write_json_atomically(
    prototype_05_ground_truth_path,
    prototype_05_ground_truth,
)


# ================================================================
# Buat preview PNG
# ================================================================

temporary_png_path = (
    prototype_05_png_path.with_name(
        f".{prototype_05_png_path.stem}.tmp.png"
    )
)

temporary_png_path.unlink(
    missing_ok=True
)

try:
    with pymupdf.open(
        str(prototype_05_pdf_path)
    ) as prototype_document:
        if prototype_document.page_count != 1:
            raise RuntimeError(
                "Prototype TPL-05 harus satu halaman."
            )

        prototype_page = prototype_document[0]
        extracted_text_05 = prototype_page.get_text(
            "text"
        )

        prototype_pixmap = prototype_page.get_pixmap(
            matrix=pymupdf.Matrix(
                150 / 72,
                150 / 72,
            ),
            alpha=False,
        )

        prototype_pixmap.save(
            str(temporary_png_path)
        )

    if (
        not temporary_png_path.is_file()
        or temporary_png_path.stat().st_size == 0
    ):
        raise RuntimeError(
            "Preview PNG TPL-05 gagal dibuat."
        )

    os.replace(
        temporary_png_path,
        prototype_05_png_path,
    )

except Exception:
    temporary_png_path.unlink(
        missing_ok=True
    )
    raise


# ================================================================
# Pemeriksaan teknis awal
# ================================================================

required_text_fragments = [
    prototype_05_render_payload[
        "metadata"
    ]["invoice_number"]["display_value"],
    prototype_05_render_payload[
        "vendor"
    ]["name"],
    prototype_05_render_payload[
        "buyer"
    ]["name"],
    prototype_05_render_payload[
        "labels"
    ]["quantity"],
    prototype_05_render_payload[
        "financials"
    ]["total"]["display_value"],
    normalize_pdf_notice(
        prototype_05_render_payload[
            "synthetic_notice"
        ]
    ),
]

missing_text_fragments = [
    text
    for text in required_text_fragments
    if text not in extracted_text_05
]

if missing_text_fragments:
    raise RuntimeError(
        "Teks penting TPL-05 hilang dari PDF: "
        f"{missing_text_fragments}"
    )

annotation_names = {
    annotation.field_name
    for annotation in prototype_05_annotations
}

required_annotation_names = {
    "vendor.name",
    "vendor.address_lines",
    "vendor.email",
    "vendor.phone",
    "vendor.tax_identifier",
    "buyer.name",
    "buyer.address_lines",
    "buyer.email",
    "buyer.phone",
    "buyer.tax_identifier",
    "invoice_number",
    "invoice_date",
    "due_date",
    "currency",
    "financials.subtotal",
    "financials.tax",
    "financials.discount",
    "financials.total",
    "document.synthetic_notice",
}

missing_annotations = sorted(
    required_annotation_names
    - annotation_names
)

if missing_annotations:
    raise RuntimeError(
        "Anotasi wajib TPL-05 tidak tersedia: "
        f"{missing_annotations}"
    )

layout_metrics = (
    prototype_05_rendering_metadata[
        "layout_metrics"
    ]
)

prototype_05_summary = pd.DataFrame(
    [
        {
            "control": "pdf_created",
            "actual": prototype_05_pdf_path.is_file(),
            "status": "VALID",
        },
        {
            "control": "ground_truth_created",
            "actual": (
                prototype_05_ground_truth_path.is_file()
            ),
            "status": "VALID",
        },
        {
            "control": "annotation_count",
            "actual": len(
                prototype_05_annotations
            ),
            "status": "VALID",
        },
        {
            "control": "item_count",
            "actual": layout_metrics[
                "item_count"
            ],
            "status": "VALID",
        },
        {
            "control": "minimum_field_font",
            "actual": (
                prototype_05_rendering_metadata[
                    "minimum_field_font_size"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "font_adjustments",
            "actual": (
                prototype_05_rendering_metadata[
                    "font_adjustment_count"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "table_to_totals_gap",
            "actual": layout_metrics[
                "table_to_totals_gap_points"
            ],
            "status": "VALID",
        },
        {
            "control": "missing_required_text",
            "actual": len(
                missing_text_fragments
            ),
            "status": "VALID",
        },
    ]
)

display(prototype_05_summary)

with PILImage.open(
    prototype_05_png_path
) as prototype_image:
    display(prototype_image.copy())


# ================================================================
# Ringkasan
# ================================================================

print(
    f"Canonical ID       : {prototype_05_canonical_id}"
)
print(
    f"Document ID        : {prototype_05_document_id}"
)
print(
    f"Prototype PDF      : {prototype_05_pdf_path}"
)
print(
    f"Prototype preview  : {prototype_05_png_path}"
)
print(
    f"Ground truth       : {prototype_05_ground_truth_path}"
)
print(
    f"Annotations        : {len(prototype_05_annotations)}"
)
print(
    "Minimum field font : "
    f"{prototype_05_rendering_metadata['minimum_field_font_size']}"
)
print(
    "Adjusted fields    : "
    f"{prototype_05_rendering_metadata['font_adjustment_count']}"
)
print(
    "Table-total gap    : "
    f"{layout_metrics['table_to_totals_gap_points']} pt"
)
print(
    f"Renderer version   : {TPL05_RENDERER_VERSION}"
)
print()
print(
    "✅ Prototype TPL-05 berhasil dibuat dan "
    "lolos pemeriksaan teknis awal."
)


**CELL 53 — FINAL Audit QA teknis prototype TPL-05**

In [ ]:
# ================================================================
# CELL 53 — FINAL
# Audit QA teknis prototype TPL-05
# ================================================================

from pathlib import Path
from typing import Any

import hashlib
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd
import pymupdf
from PIL import Image as PILImage


# ================================================================
# Validasi dependency runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "TPL05_PROTOTYPE_ROOT",
    "TPL05_TEMPLATE_ID",
    "prototype_05_canonical_id",
    "prototype_05_document_id",
    "prototype_05_pdf_path",
    "prototype_05_png_path",
    "prototype_05_ground_truth_path",
    "prototype_05_render_payload",
    "prototype_05_rendering_metadata",
    "prototype_05_annotations",
    "annotation_to_dict",
    "write_json_atomically",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 53 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali Cell 52 versi final."
    )


# ================================================================
# Konfigurasi audit
# ================================================================

QA_SCHEMA_VERSION = "1.0.0"
QA_AUDITOR_VERSION = "1.0.0"

MINIMUM_FIELD_FONT_SIZE = 6.5
MAXIMUM_FONT_ADJUSTMENT_RATIO = 0.20
MINIMUM_RASTER_CONTENT_RATIO = 0.01
MAXIMUM_RASTER_CONTENT_RATIO = 0.40
MINIMUM_RASTER_CONTRAST = 10.0
SEVERE_OVERLAP_THRESHOLD = 0.50

prototype_05_pdf_path = Path(
    prototype_05_pdf_path
)
prototype_05_png_path = Path(
    prototype_05_png_path
)
prototype_05_ground_truth_path = Path(
    prototype_05_ground_truth_path
)

prototype_05_qa_report_path = (
    Path(TPL05_PROTOTYPE_ROOT)
    / (
        f"{prototype_05_document_id}"
        "_TPL-05_qa_report.json"
    )
)

required_files = [
    prototype_05_pdf_path,
    prototype_05_png_path,
    prototype_05_ground_truth_path,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Artifact TPL-05 belum lengkap: "
        f"{missing_files}"
    )


# ================================================================
# Helper umum
# ================================================================

def sha256_file(file_path: Path) -> str:
    digest = hashlib.sha256()

    with Path(file_path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def normalize_text(value: Any) -> str:
    text = unicodedata.normalize(
        "NFKC",
        str(value or ""),
    )

    text = (
        text
        .replace("\u2014", "-")
        .replace("\u2013", "-")
        .replace("\u00b7", "-")
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip().casefold()


def normalize_notice_for_pdf(value: Any) -> str:
    return (
        str(value)
        .replace("\u2014", "\u00b7")
        .replace("\u2013", "-")
    )


def first_present(
    mapping: dict[str, Any],
    candidate_keys: tuple[str, ...],
) -> Any:
    for key in candidate_keys:
        if key in mapping:
            return mapping[key]

    return None


def annotation_field_name(
    annotation: dict[str, Any],
) -> str:
    value = first_present(
        annotation,
        (
            "field_name",
            "field_path",
            "field",
            "path",
            "name",
        ),
    )

    if value is None:
        raise KeyError(
            "Nama field tidak ditemukan pada anotasi. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return str(value)


def annotation_text(
    annotation: dict[str, Any],
) -> str:
    value = first_present(
        annotation,
        (
            "text",
            "value",
            "display_value",
            "field_value",
            "raw_value",
            "raw_text",
            "text_value",
            "content",
        ),
    )

    if isinstance(value, dict):
        value = first_present(
            value,
            (
                "text",
                "value",
                "display_value",
                "raw",
            ),
        )

    if value is None:
        raise KeyError(
            "Nilai teks tidak ditemukan pada anotasi. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return str(value)


def bbox_from_value(
    value: Any,
) -> tuple[float, float, float, float] | None:
    if isinstance(value, (list, tuple)):
        if len(value) != 4:
            return None

        try:
            return tuple(
                float(number)
                for number in value
            )
        except (TypeError, ValueError):
            return None

    if not isinstance(value, dict):
        return None

    direct_key_sets = [
        ("x0", "y0", "x1", "y1"),
        ("left", "top", "right", "bottom"),
    ]

    for key_set in direct_key_sets:
        if all(key in value for key in key_set):
            try:
                return tuple(
                    float(value[key])
                    for key in key_set
                )
            except (TypeError, ValueError):
                return None

    if all(
        key in value
        for key in ("x", "y", "width", "height")
    ):
        try:
            x0 = float(value["x"])
            y0 = float(value["y"])
            width = float(value["width"])
            height = float(value["height"])

            return (
                x0,
                y0,
                x0 + width,
                y0 + height,
            )
        except (TypeError, ValueError):
            return None

    if all(
        key in value
        for key in ("x", "y", "w", "h")
    ):
        try:
            x0 = float(value["x"])
            y0 = float(value["y"])
            width = float(value["w"])
            height = float(value["h"])

            return (
                x0,
                y0,
                x0 + width,
                y0 + height,
            )
        except (TypeError, ValueError):
            return None

    return None


def annotation_bbox(
    annotation: dict[str, Any],
) -> tuple[float, float, float, float]:
    candidate = first_present(
        annotation,
        (
            "bbox",
            "bbox_points",
            "bbox_pt",
            "bbox_pdf",
            "bounding_box",
            "rectangle",
            "rect",
            "coordinates",
            "coordinates_points",
        ),
    )

    if candidate is None:
        geometry = annotation.get("geometry")

        if isinstance(geometry, dict):
            candidate = first_present(
                geometry,
                (
                    "bbox",
                    "bounding_box",
                    "rectangle",
                ),
            )

    bbox = bbox_from_value(candidate)

    if bbox is None:
        raise KeyError(
            "Bounding box tidak ditemukan atau tidak valid. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return bbox


def rectangle_area(
    bbox: tuple[float, float, float, float],
) -> float:
    x0, y0, x1, y1 = bbox

    return max(0.0, x1 - x0) * max(
        0.0,
        y1 - y0,
    )


def intersection_area(
    first_bbox: tuple[float, float, float, float],
    second_bbox: tuple[float, float, float, float],
) -> float:
    first_x0, first_y0, first_x1, first_y1 = first_bbox
    second_x0, second_y0, second_x1, second_y1 = second_bbox

    width = max(
        0.0,
        min(first_x1, second_x1)
        - max(first_x0, second_x0),
    )

    height = max(
        0.0,
        min(first_y1, second_y1)
        - max(first_y0, second_y0),
    )

    return width * height


def bbox_intersects_word(
    annotation_box: tuple[float, float, float, float],
    word_box: tuple[float, float, float, float],
) -> bool:
    return intersection_area(
        annotation_box,
        word_box,
    ) > 0.0


# ================================================================
# Muat ground truth
# ================================================================

with prototype_05_ground_truth_path.open(
    "r",
    encoding="utf-8",
) as ground_truth_file:
    ground_truth = json.load(
        ground_truth_file
    )

ground_truth_document = ground_truth.get(
    "document",
    {},
)

if (
    ground_truth_document.get("document_id")
    != prototype_05_document_id
):
    raise RuntimeError(
        "Document ID ground truth tidak sesuai."
    )

if (
    ground_truth_document.get("template_id")
    != TPL05_TEMPLATE_ID
):
    raise RuntimeError(
        "Ground truth bukan milik TPL-05."
    )

annotation_records = ground_truth.get(
    "annotations",
    [],
)

if not isinstance(annotation_records, list):
    raise TypeError(
        "ground_truth.annotations wajib berupa list."
    )

if not annotation_records:
    raise RuntimeError(
        "Ground truth tidak memiliki anotasi."
    )


# ================================================================
# Expected field mapping
# ================================================================

render_payload = prototype_05_render_payload

expected_values: dict[str, str] = {
    "invoice_number": str(
        render_payload["metadata"]
        ["invoice_number"]
        ["display_value"]
    ),
    "invoice_date": str(
        render_payload["metadata"]
        ["invoice_date"]
        ["display_value"]
    ),
    "due_date": str(
        render_payload["metadata"]
        ["due_date"]
        ["display_value"]
    ),
    "currency": str(
        render_payload["currency"]
    ),
}

for party_type in ("vendor", "buyer"):
    party = render_payload[party_type]

    expected_values.update(
        {
            f"{party_type}.name": str(
                party["name"]
            ),
            f"{party_type}.address_lines": "\n".join(
                party["address_lines"]
            ),
            f"{party_type}.email": str(
                party["email"]
            ),
            f"{party_type}.phone": str(
                party["phone"]
            ),
            f"{party_type}.tax_identifier": str(
                party["tax_identifier_display"]
            ),
        }
    )

for item_index, item in enumerate(
    render_payload["items"]
):
    for field_name in (
        "description",
        "quantity",
        "unit_price",
        "line_total",
    ):
        expected_values[
            f"items[{item_index}].{field_name}"
        ] = str(item[field_name])

for field_name in (
    "subtotal",
    "tax",
    "discount",
    "total",
):
    expected_values[
        f"financials.{field_name}"
    ] = str(
        render_payload["financials"]
        [field_name]
        ["display_value"]
    )

expected_values[
    "document.synthetic_notice"
] = normalize_notice_for_pdf(
    render_payload["synthetic_notice"]
)

expected_annotation_count = len(
    expected_values
)


# ================================================================
# Parse annotations
# ================================================================

parsed_annotations: list[dict[str, Any]] = []
annotation_parse_failures: list[dict[str, Any]] = []

for annotation_index, annotation in enumerate(
    annotation_records
):
    try:
        parsed_annotations.append(
            {
                "index": annotation_index,
                "field_name": annotation_field_name(
                    annotation
                ),
                "text": annotation_text(
                    annotation
                ),
                "bbox": annotation_bbox(
                    annotation
                ),
            }
        )
    except (KeyError, TypeError, ValueError) as error:
        annotation_parse_failures.append(
            {
                "index": annotation_index,
                "error": str(error),
                "keys": sorted(
                    annotation.keys()
                ) if isinstance(
                    annotation,
                    dict,
                ) else [],
            }
        )

if annotation_parse_failures:
    raise RuntimeError(
        "Schema anotasi tidak dapat diproses: "
        f"{annotation_parse_failures[:3]}"
    )

annotation_names = [
    annotation["field_name"]
    for annotation in parsed_annotations
]

annotation_by_name = {
    annotation["field_name"]: annotation
    for annotation in parsed_annotations
}

duplicate_annotation_count = (
    len(annotation_names)
    - len(set(annotation_names))
)

missing_required_annotations = sorted(
    set(expected_values)
    - set(annotation_names)
)


# ================================================================
# Audit PDF dan anotasi
# ================================================================

with pymupdf.open(
    str(prototype_05_pdf_path)
) as pdf_document:
    pdf_page_count = pdf_document.page_count

    if pdf_page_count < 1:
        raise RuntimeError(
            "PDF TPL-05 tidak memiliki halaman."
        )

    first_page = pdf_document[0]
    page_rectangle = first_page.rect
    extracted_text = first_page.get_text("text")
    extracted_words = first_page.get_text("words")

page_width = float(page_rectangle.width)
page_height = float(page_rectangle.height)

word_boxes = [
    (
        float(word[0]),
        float(word[1]),
        float(word[2]),
        float(word[3]),
    )
    for word in extracted_words
    if len(word) >= 5
    and str(word[4]).strip()
]

invalid_bounding_boxes: list[str] = []
annotations_without_words: list[str] = []

for annotation in parsed_annotations:
    field_name = annotation["field_name"]
    x0, y0, x1, y1 = annotation["bbox"]

    coordinates_are_finite = all(
        math.isfinite(value)
        for value in (x0, y0, x1, y1)
    )

    bbox_is_valid = (
        coordinates_are_finite
        and x0 >= 0.0
        and y0 >= 0.0
        and x1 <= page_width
        and y1 <= page_height
        and x1 > x0
        and y1 > y0
    )

    if not bbox_is_valid:
        invalid_bounding_boxes.append(
            field_name
        )
        continue

    contains_word = any(
        bbox_intersects_word(
            annotation["bbox"],
            word_box,
        )
        for word_box in word_boxes
    )

    if not contains_word:
        annotations_without_words.append(
            field_name
        )


# ================================================================
# Audit nilai anotasi
# ================================================================

normalization_mismatches: list[dict[str, str]] = []

for field_name, expected_value in expected_values.items():
    annotation = annotation_by_name.get(
        field_name
    )

    if annotation is None:
        continue

    actual_value = annotation["text"]

    if normalize_text(actual_value) != normalize_text(
        expected_value
    ):
        normalization_mismatches.append(
            {
                "field_name": field_name,
                "expected": expected_value,
                "actual": actual_value,
            }
        )

normalized_extracted_text = normalize_text(
    extracted_text
)

missing_extracted_values: list[str] = []

for annotation in parsed_annotations:
    normalized_value = normalize_text(
        annotation["text"]
    )

    if (
        normalized_value
        and normalized_value
        not in normalized_extracted_text
    ):
        missing_extracted_values.append(
            annotation["field_name"]
        )


# ================================================================
# Audit overlap anotasi
# ================================================================

severe_annotation_overlaps: list[dict[str, Any]] = []

for first_index in range(
    len(parsed_annotations)
):
    first_annotation = parsed_annotations[
        first_index
    ]
    first_area = rectangle_area(
        first_annotation["bbox"]
    )

    if first_area <= 0.0:
        continue

    for second_index in range(
        first_index + 1,
        len(parsed_annotations),
    ):
        second_annotation = parsed_annotations[
            second_index
        ]
        second_area = rectangle_area(
            second_annotation["bbox"]
        )

        if second_area <= 0.0:
            continue

        overlap_area = intersection_area(
            first_annotation["bbox"],
            second_annotation["bbox"],
        )

        overlap_ratio = overlap_area / min(
            first_area,
            second_area,
        )

        if overlap_ratio >= SEVERE_OVERLAP_THRESHOLD:
            severe_annotation_overlaps.append(
                {
                    "first": first_annotation[
                        "field_name"
                    ],
                    "second": second_annotation[
                        "field_name"
                    ],
                    "overlap_ratio": round(
                        overlap_ratio,
                        6,
                    ),
                }
            )


# ================================================================
# Audit font dan raster
# ================================================================

minimum_field_font_size = float(
    prototype_05_rendering_metadata[
        "minimum_field_font_size"
    ]
)

font_adjustment_count = int(
    prototype_05_rendering_metadata[
        "font_adjustment_count"
    ]
)

font_adjustment_ratio = (
    font_adjustment_count
    / max(1, len(parsed_annotations))
)

with PILImage.open(
    prototype_05_png_path
) as preview_image:
    grayscale_array = np.asarray(
        preview_image.convert("L"),
        dtype=np.float32,
    ).copy()

if grayscale_array.size == 0:
    raise RuntimeError(
        "Preview PNG tidak memiliki piksel."
    )

raster_content_ratio = float(
    np.mean(grayscale_array < 245.0)
)

raster_contrast = float(
    np.std(grayscale_array)
)


# ================================================================
# Susun hasil pemeriksaan
# ================================================================

checks = [
    {
        "control": "pdf_page_count",
        "expected": 1,
        "actual": pdf_page_count,
        "valid": pdf_page_count == 1,
    },
    {
        "control": "annotation_count",
        "expected": expected_annotation_count,
        "actual": len(parsed_annotations),
        "valid": (
            len(parsed_annotations)
            == expected_annotation_count
        ),
    },
    {
        "control": "duplicate_annotations",
        "expected": 0,
        "actual": duplicate_annotation_count,
        "valid": duplicate_annotation_count == 0,
    },
    {
        "control": "missing_required_annotations",
        "expected": 0,
        "actual": len(
            missing_required_annotations
        ),
        "valid": not missing_required_annotations,
    },
    {
        "control": "invalid_bounding_boxes",
        "expected": 0,
        "actual": len(
            invalid_bounding_boxes
        ),
        "valid": not invalid_bounding_boxes,
    },
    {
        "control": "normalization_mismatches",
        "expected": 0,
        "actual": len(
            normalization_mismatches
        ),
        "valid": not normalization_mismatches,
    },
    {
        "control": "annotations_without_words",
        "expected": 0,
        "actual": len(
            annotations_without_words
        ),
        "valid": not annotations_without_words,
    },
    {
        "control": "severe_annotation_overlaps",
        "expected": 0,
        "actual": len(
            severe_annotation_overlaps
        ),
        "valid": not severe_annotation_overlaps,
    },
    {
        "control": "missing_extracted_values",
        "expected": 0,
        "actual": len(
            missing_extracted_values
        ),
        "valid": not missing_extracted_values,
    },
    {
        "control": "minimum_field_font_size",
        "expected": ">=6.5",
        "actual": round(
            minimum_field_font_size,
            4,
        ),
        "valid": (
            minimum_field_font_size
            >= MINIMUM_FIELD_FONT_SIZE
        ),
    },
    {
        "control": "font_adjustment_ratio",
        "expected": "<=0.20",
        "actual": round(
            font_adjustment_ratio,
            4,
        ),
        "valid": (
            font_adjustment_ratio
            <= MAXIMUM_FONT_ADJUSTMENT_RATIO
        ),
    },
    {
        "control": "raster_content_ratio",
        "expected": "0.01–0.40",
        "actual": round(
            raster_content_ratio,
            4,
        ),
        "valid": (
            MINIMUM_RASTER_CONTENT_RATIO
            <= raster_content_ratio
            <= MAXIMUM_RASTER_CONTENT_RATIO
        ),
    },
    {
        "control": "raster_contrast",
        "expected": ">=10.0",
        "actual": round(
            raster_contrast,
            4,
        ),
        "valid": (
            raster_contrast
            >= MINIMUM_RASTER_CONTRAST
        ),
    },
]

for check in checks:
    check["status"] = (
        "VALID"
        if check["valid"]
        else "INVALID"
    )

failed_checks = [
    check
    for check in checks
    if not check["valid"]
]

qa_status = (
    "PASSED"
    if not failed_checks
    else "FAILED"
)


# ================================================================
# Simpan QA report
# ================================================================

qa_report = {
    "schema_version": QA_SCHEMA_VERSION,
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "document_id": prototype_05_document_id,
    "canonical_invoice_id": (
        prototype_05_canonical_id
    ),
    "template_id": TPL05_TEMPLATE_ID,
    "status": qa_status,
    "manual_visual_review": "PENDING",
    "metrics": {
        "auditor_version": QA_AUDITOR_VERSION,
        "pdf_page_count": pdf_page_count,
        "annotation_count": len(
            parsed_annotations
        ),
        "expected_annotation_count": (
            expected_annotation_count
        ),
        "duplicate_annotation_count": (
            duplicate_annotation_count
        ),
        "missing_required_annotation_count": len(
            missing_required_annotations
        ),
        "invalid_bounding_box_count": len(
            invalid_bounding_boxes
        ),
        "normalization_mismatch_count": len(
            normalization_mismatches
        ),
        "annotations_without_words_count": len(
            annotations_without_words
        ),
        "severe_annotation_overlap_count": len(
            severe_annotation_overlaps
        ),
        "missing_extracted_value_count": len(
            missing_extracted_values
        ),
        "minimum_field_font_size": round(
            minimum_field_font_size,
            6,
        ),
        "font_adjustment_count": (
            font_adjustment_count
        ),
        "font_adjustment_ratio": round(
            font_adjustment_ratio,
            6,
        ),
        "raster_content_ratio": round(
            raster_content_ratio,
            6,
        ),
        "raster_contrast": round(
            raster_contrast,
            6,
        ),
    },
    "failures": {
        "failed_checks": [
            check["control"]
            for check in failed_checks
        ],
        "missing_required_annotations": (
            missing_required_annotations
        ),
        "invalid_bounding_boxes": (
            invalid_bounding_boxes
        ),
        "normalization_mismatches": (
            normalization_mismatches
        ),
        "annotations_without_words": (
            annotations_without_words
        ),
        "severe_annotation_overlaps": (
            severe_annotation_overlaps
        ),
        "missing_extracted_values": (
            missing_extracted_values
        ),
    },
    "checksums_sha256": {
        "prototype_pdf": sha256_file(
            prototype_05_pdf_path
        ),
        "preview": sha256_file(
            prototype_05_png_path
        ),
        "ground_truth": sha256_file(
            prototype_05_ground_truth_path
        ),
    },
}

write_json_atomically(
    prototype_05_qa_report_path,
    qa_report,
)


# ================================================================
# Verifikasi round-trip QA report
# ================================================================

with prototype_05_qa_report_path.open(
    "r",
    encoding="utf-8",
) as qa_report_file:
    verified_qa_report = json.load(
        qa_report_file
    )

if verified_qa_report != qa_report:
    raise RuntimeError(
        "QA report berubah setelah disimpan dan dibuka ulang."
    )


# ================================================================
# Output audit
# ================================================================

qa_summary = pd.DataFrame(
    [
        {
            "control": check["control"],
            "expected": check["expected"],
            "actual": check["actual"],
            "status": check["status"],
        }
        for check in checks
    ]
)

display(qa_summary)

print(
    f"QA report          : "
    f"{prototype_05_qa_report_path}"
)
print(
    f"QA status          : {qa_status}"
)
print(
    "Minimum field font : "
    f"{minimum_field_font_size:.2f}"
)
print(
    "Font adjustments   : "
    f"{font_adjustment_count}"
)
print(
    "Ink pixel ratio    : "
    f"{raster_content_ratio:.6f}"
)
print(
    "Raster contrast    : "
    f"{raster_contrast:.6f}"
)
print(
    "Manual review      : "
    f"{qa_report['manual_visual_review']}"
)
print()

if qa_status != "PASSED":
    print("Detail kegagalan:")
    print(
        json.dumps(
            qa_report["failures"],
            ensure_ascii=False,
            indent=2,
        )
    )

    raise RuntimeError(
        "Prototype TPL-05 gagal audit QA teknis."
    )

print(
    "✅ Prototype TPL-05 lulus audit teknis. "
    "Review visual manual masih perlu dicatat "
    "ke QA report."
)


In [ ]:
# ================================================================
# CELL 53A — FINAL
# Finalisasi review visual manual prototype TPL-05
# ================================================================

from pathlib import Path

import hashlib
import json


# ================================================================
# Validasi dependency runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "prototype_05_qa_report_path",
    "prototype_05_png_path",
    "prototype_05_document_id",
    "write_json_atomically",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 53A belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali Cell 52 dan Cell 53."
    )


# ================================================================
# Helper checksum
# ================================================================

def calculate_sha256(file_path: Path) -> str:
    """Menghitung SHA-256 file tanpa memuat seluruh file ke memori."""

    digest = hashlib.sha256()

    with file_path.open("rb") as input_file:
        for chunk in iter(
            lambda: input_file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ================================================================
# Muat artifact yang akan difinalisasi
# ================================================================

qa_report_path = Path(
    prototype_05_qa_report_path
)

preview_path = Path(
    prototype_05_png_path
)

if not qa_report_path.is_file():
    raise FileNotFoundError(
        "QA report TPL-05 tidak ditemukan: "
        f"{qa_report_path}"
    )

if not preview_path.is_file():
    raise FileNotFoundError(
        "Preview TPL-05 tidak ditemukan: "
        f"{preview_path}"
    )

with qa_report_path.open(
    mode="r",
    encoding="utf-8",
) as qa_report_file:
    qa_report = json.load(qa_report_file)


# ================================================================
# Validasi identitas dan audit teknis
# ================================================================

if qa_report.get("template_id") != "TPL-05":
    raise RuntimeError(
        "QA report bukan milik TPL-05."
    )

if (
    qa_report.get("document_id")
    != prototype_05_document_id
):
    raise RuntimeError(
        "Document ID QA report tidak sesuai. "
        f"Expected: {prototype_05_document_id!r}; "
        f"actual: {qa_report.get('document_id')!r}."
    )

if qa_report.get("status") != "PASSED":
    raise RuntimeError(
        "Review visual tidak boleh difinalisasi karena "
        "audit teknis belum PASSED. "
        f"Status saat ini: {qa_report.get('status')!r}."
    )

failure_details = qa_report.get(
    "failures",
    {},
)

if not isinstance(failure_details, dict):
    raise TypeError(
        "Field 'failures' pada QA report wajib "
        "berupa dictionary."
    )

nonempty_failures = {
    failure_name: failure_value
    for failure_name, failure_value
    in failure_details.items()
    if bool(failure_value)
}

if nonempty_failures:
    raise RuntimeError(
        "QA report masih memiliki kegagalan teknis: "
        f"{nonempty_failures}"
    )

if "manual_visual_review" not in qa_report:
    raise KeyError(
        "Field 'manual_visual_review' tidak ditemukan "
        "dalam QA report."
    )


# ================================================================
# Pastikan preview yang direview sama dengan preview yang diaudit
# ================================================================

recorded_preview_checksum = (
    qa_report
    .get("checksums_sha256", {})
    .get("preview")
)

if not recorded_preview_checksum:
    raise KeyError(
        "Checksum preview tidak ditemukan pada "
        "qa_report['checksums_sha256']['preview']."
    )

actual_preview_checksum = calculate_sha256(
    preview_path
)

if actual_preview_checksum != recorded_preview_checksum:
    raise RuntimeError(
        "Preview berubah setelah audit teknis. "
        "Jalankan ulang Cell 53 dan lakukan review visual kembali. "
        f"Expected SHA-256: {recorded_preview_checksum}; "
        f"actual: {actual_preview_checksum}."
    )


# ================================================================
# Catat hasil review visual secara idempoten
# ================================================================

qa_report["manual_visual_review"] = "PASSED"

write_json_atomically(
    qa_report_path,
    qa_report,
)


# ================================================================
# Buka ulang dan verifikasi hasil persistensi
# ================================================================

with qa_report_path.open(
    mode="r",
    encoding="utf-8",
) as qa_report_file:
    persisted_qa_report = json.load(
        qa_report_file
    )

if persisted_qa_report.get("status") != "PASSED":
    raise RuntimeError(
        "Status audit teknis berubah setelah finalisasi."
    )

if (
    persisted_qa_report.get("manual_visual_review")
    != "PASSED"
):
    raise RuntimeError(
        "Status review visual gagal disimpan."
    )

persisted_preview_checksum = (
    persisted_qa_report
    .get("checksums_sha256", {})
    .get("preview")
)

if persisted_preview_checksum != actual_preview_checksum:
    raise RuntimeError(
        "Checksum preview berubah pada QA report."
    )


# ================================================================
# Ringkasan final
# ================================================================

print(
    f"QA report           : {qa_report_path}"
)

print(
    "Technical status    : "
    f"{persisted_qa_report['status']}"
)

print(
    "Manual visual review: "
    f"{persisted_qa_report['manual_visual_review']}"
)

print(
    "Preview SHA-256     : "
    f"{persisted_preview_checksum}"
)

print()

print(
    "✅ TPL-05 FINAL PASSED — audit teknis dan "
    "review visual telah selesai."
)


**CELL 54 — FINAL Inspeksi spesifikasi dan presentation payload TPL-06**

In [ ]:
# ================================================================
# CELL 54 — FINAL
# Inspeksi spesifikasi dan presentation payload TPL-06
# Read-only: tidak membuat atau mengubah artifact build
# ================================================================

from collections import Counter
from typing import Any

import json

import pandas as pd
from IPython.display import display


# ================================================================
# Validasi dependency runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "template_registry",
    "invoice_render_payloads",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 54 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali cell pembentukan template registry "
        "dan presentation payload."
    )

if not isinstance(template_registry, dict):
    raise TypeError(
        "template_registry wajib berupa dictionary."
    )

if not isinstance(invoice_render_payloads, dict):
    raise TypeError(
        "invoice_render_payloads wajib berupa dictionary."
    )


# ================================================================
# Konfigurasi inspeksi
# ================================================================

TPL06_TEMPLATE_ID = "TPL-06"
TPL06_EXPECTED_PAYLOAD_COUNT = 20

if TPL06_TEMPLATE_ID not in template_registry:
    raise KeyError(
        "Spesifikasi TPL-06 tidak ditemukan dalam "
        "template_registry."
    )

tpl06_template_specification = template_registry[
    TPL06_TEMPLATE_ID
]

if not isinstance(tpl06_template_specification, dict):
    raise TypeError(
        "Spesifikasi TPL-06 wajib berupa dictionary."
    )

tpl06_payloads = {
    canonical_id: payload
    for canonical_id, payload
    in invoice_render_payloads.items()
    if isinstance(payload, dict)
    and payload.get("template_id") == TPL06_TEMPLATE_ID
}

if len(tpl06_payloads) != TPL06_EXPECTED_PAYLOAD_COUNT:
    raise RuntimeError(
        "TPL-06 harus memiliki tepat "
        f"{TPL06_EXPECTED_PAYLOAD_COUNT} payload. "
        f"Actual: {len(tpl06_payloads)}."
    )


# ================================================================
# Kontrak struktur presentation payload
# ================================================================

REQUIRED_TOP_LEVEL_KEYS = {
    "buyer",
    "canonical_invoice_id",
    "currency",
    "document_id",
    "financials",
    "footer",
    "items",
    "labels",
    "language",
    "metadata",
    "split",
    "synthetic_notice",
    "template_id",
    "title",
    "vendor",
}

REQUIRED_LABEL_KEYS = {
    "buyer",
    "description",
    "discount",
    "document_title",
    "due_date",
    "invoice_date",
    "invoice_number",
    "line_total",
    "page",
    "quantity",
    "subtotal",
    "synthetic_notice",
    "tax",
    "tax_identifier",
    "total",
    "unit_price",
    "vendor",
}

REQUIRED_METADATA_KEYS = {
    "due_date",
    "invoice_date",
    "invoice_number",
}

REQUIRED_PARTY_KEYS = {
    "address_lines",
    "email",
    "name",
    "party_id",
    "phone",
    "tax_identifier",
    "tax_identifier_display",
}

REQUIRED_ITEM_KEYS = {
    "description",
    "item_number",
    "line_total",
    "quantity",
    "raw",
    "unit_price",
}

REQUIRED_FINANCIAL_KEYS = {
    "discount",
    "subtotal",
    "tax",
    "total",
}

REQUIRED_FOOTER_KEYS = {
    "document_reference",
    "synthetic_notice",
}


def require_dictionary(
    value: Any,
    field_path: str,
) -> dict[str, Any]:
    """Memastikan sebuah field merupakan dictionary."""

    if not isinstance(value, dict):
        raise TypeError(
            f"{field_path} wajib berupa dictionary."
        )

    return value


def require_key_set(
    value: dict[str, Any],
    required_keys: set[str],
    field_path: str,
) -> None:
    """Memastikan seluruh key wajib tersedia."""

    missing_keys = sorted(
        required_keys - set(value)
    )

    if missing_keys:
        raise KeyError(
            f"{field_path} kehilangan key wajib: "
            f"{missing_keys}"
        )


expected_split = tpl06_template_specification.get(
    "split"
)

document_ids: list[str] = []

for canonical_id in sorted(tpl06_payloads):
    payload = tpl06_payloads[canonical_id]

    require_key_set(
        payload,
        REQUIRED_TOP_LEVEL_KEYS,
        f"payload[{canonical_id!r}]",
    )

    if payload["canonical_invoice_id"] != canonical_id:
        raise RuntimeError(
            "Canonical ID pada dictionary dan payload "
            f"tidak sama: {canonical_id!r}."
        )

    if payload["template_id"] != TPL06_TEMPLATE_ID:
        raise RuntimeError(
            f"Template ID tidak sesuai pada {canonical_id}."
        )

    if expected_split is not None and (
        payload["split"] != expected_split
    ):
        raise RuntimeError(
            f"Split payload {canonical_id} tidak sesuai. "
            f"Expected: {expected_split!r}; "
            f"actual: {payload['split']!r}."
        )

    if payload["language"] not in {"id", "en"}:
        raise RuntimeError(
            f"Language tidak didukung pada {canonical_id}: "
            f"{payload['language']!r}."
        )

    labels = require_dictionary(
        payload["labels"],
        f"payload[{canonical_id!r}].labels",
    )

    metadata = require_dictionary(
        payload["metadata"],
        f"payload[{canonical_id!r}].metadata",
    )

    vendor = require_dictionary(
        payload["vendor"],
        f"payload[{canonical_id!r}].vendor",
    )

    buyer = require_dictionary(
        payload["buyer"],
        f"payload[{canonical_id!r}].buyer",
    )

    financials = require_dictionary(
        payload["financials"],
        f"payload[{canonical_id!r}].financials",
    )

    footer = require_dictionary(
        payload["footer"],
        f"payload[{canonical_id!r}].footer",
    )

    require_key_set(
        labels,
        REQUIRED_LABEL_KEYS,
        f"payload[{canonical_id!r}].labels",
    )

    require_key_set(
        metadata,
        REQUIRED_METADATA_KEYS,
        f"payload[{canonical_id!r}].metadata",
    )

    require_key_set(
        vendor,
        REQUIRED_PARTY_KEYS,
        f"payload[{canonical_id!r}].vendor",
    )

    require_key_set(
        buyer,
        REQUIRED_PARTY_KEYS,
        f"payload[{canonical_id!r}].buyer",
    )

    require_key_set(
        financials,
        REQUIRED_FINANCIAL_KEYS,
        f"payload[{canonical_id!r}].financials",
    )

    require_key_set(
        footer,
        REQUIRED_FOOTER_KEYS,
        f"payload[{canonical_id!r}].footer",
    )

    items = payload["items"]

    if not isinstance(items, list) or not items:
        raise RuntimeError(
            f"payload[{canonical_id!r}].items wajib "
            "berupa list yang tidak kosong."
        )

    for item_index, item in enumerate(items):
        item = require_dictionary(
            item,
            (
                f"payload[{canonical_id!r}]"
                f".items[{item_index}]"
            ),
        )

        require_key_set(
            item,
            REQUIRED_ITEM_KEYS,
            (
                f"payload[{canonical_id!r}]"
                f".items[{item_index}]"
            ),
        )

    document_ids.append(
        str(payload["document_id"])
    )

if len(document_ids) != len(set(document_ids)):
    raise RuntimeError(
        "Ditemukan document_id duplikat pada payload TPL-06."
    )


# ================================================================
# Helper statistik panjang teks
# ================================================================

def joined_text_length(values: Any) -> int:
    """Mengukur panjang gabungan list teks secara konsisten."""

    if not isinstance(values, list):
        return len(str(values))

    return len(
        " ".join(str(value) for value in values)
    )


def maximum_money_length(
    payload: dict[str, Any],
) -> int:
    """Mengukur nilai uang terpanjang yang perlu dirender."""

    money_values: list[str] = []

    for item in payload["items"]:
        money_values.extend(
            [
                str(item["unit_price"]),
                str(item["line_total"]),
            ]
        )

    for field_name in (
        "subtotal",
        "tax",
        "discount",
        "total",
    ):
        financial_value = payload[
            "financials"
        ][field_name]

        if isinstance(financial_value, dict):
            financial_value = financial_value.get(
                "display_value",
                financial_value.get("value", ""),
            )

        money_values.append(
            str(financial_value)
        )

    return max(
        map(len, money_values),
        default=0,
    )


# ================================================================
# Bangun ringkasan inspeksi
# ================================================================

tpl06_candidate_ids = sorted(
    tpl06_payloads
)

tpl06_sample_id = tpl06_candidate_ids[0]
tpl06_sample_payload = tpl06_payloads[
    tpl06_sample_id
]

tpl06_sample_structure = {
    "canonical_invoice_id": (
        tpl06_sample_payload["canonical_invoice_id"]
    ),
    "document_id": tpl06_sample_payload[
        "document_id"
    ],
    "template_id": tpl06_sample_payload[
        "template_id"
    ],
    "language": tpl06_sample_payload["language"],
    "currency": tpl06_sample_payload["currency"],
    "title": tpl06_sample_payload["title"],
    "top_level_keys": sorted(
        tpl06_sample_payload
    ),
    "label_keys": sorted(
        tpl06_sample_payload["labels"]
    ),
    "metadata_keys": sorted(
        tpl06_sample_payload["metadata"]
    ),
    "vendor_keys": sorted(
        tpl06_sample_payload["vendor"]
    ),
    "buyer_keys": sorted(
        tpl06_sample_payload["buyer"]
    ),
    "item_keys": sorted(
        tpl06_sample_payload["items"][0]
    ),
    "financial_keys": sorted(
        tpl06_sample_payload["financials"]
    ),
    "footer_keys": sorted(
        tpl06_sample_payload["footer"]
    ),
}

statistics_rows: list[dict[str, Any]] = []

for canonical_id in tpl06_candidate_ids:
    payload = tpl06_payloads[canonical_id]
    descriptions = [
        str(item["description"])
        for item in payload["items"]
    ]

    statistics_rows.append(
        {
            "canonical_id": canonical_id,
            "document_id": payload["document_id"],
            "language": payload["language"],
            "currency": payload["currency"],
            "item_count": len(payload["items"]),
            "vendor_name_length": len(
                str(payload["vendor"]["name"])
            ),
            "vendor_address_length": joined_text_length(
                payload["vendor"]["address_lines"]
            ),
            "vendor_email_length": len(
                str(payload["vendor"]["email"])
            ),
            "buyer_name_length": len(
                str(payload["buyer"]["name"])
            ),
            "buyer_address_length": joined_text_length(
                payload["buyer"]["address_lines"]
            ),
            "buyer_email_length": len(
                str(payload["buyer"]["email"])
            ),
            "maximum_description_length": max(
                map(len, descriptions),
                default=0,
            ),
            "maximum_money_length": maximum_money_length(
                payload
            ),
        }
    )

tpl06_statistics = pd.DataFrame(
    statistics_rows
)

tpl06_payload_statistics = (
    tpl06_statistics
    .describe(include="all")
    .transpose()
)

tpl06_item_count_distribution = (
    tpl06_statistics["item_count"]
    .value_counts()
    .sort_index()
    .rename_axis("item_count")
    .reset_index(name="document_count")
)

tpl06_language_distribution = (
    tpl06_statistics["language"]
    .value_counts()
    .sort_index()
    .rename_axis("language")
    .reset_index(name="document_count")
)

tpl06_currency_distribution = (
    tpl06_statistics["currency"]
    .value_counts()
    .sort_index()
    .rename_axis("currency")
    .reset_index(name="document_count")
)


# ================================================================
# Output inspeksi
# ================================================================

print("TEMPLATE SPECIFICATION")
print(
    json.dumps(
        tpl06_template_specification,
        ensure_ascii=False,
        indent=2,
    )
)

print()
print("PAYLOAD COUNT")
print(len(tpl06_payloads))

print()
print("SAMPLE STRUCTURE")
print(
    json.dumps(
        tpl06_sample_structure,
        ensure_ascii=False,
        indent=2,
    )
)

print()
print("LABELS")
print(
    json.dumps(
        tpl06_sample_payload["labels"],
        ensure_ascii=False,
        indent=2,
    )
)

print()
print("PAYLOAD STATISTICS")
display(tpl06_payload_statistics)

print()
print("ITEM-COUNT DISTRIBUTION")
display(tpl06_item_count_distribution)

print()
print("LANGUAGE DISTRIBUTION")
display(tpl06_language_distribution)

print()
print("CURRENCY DISTRIBUTION")
display(tpl06_currency_distribution)

print(
    "✅ Inspeksi TPL-06 selesai. "
    "Belum ada artifact yang dibuat atau diubah."
)


**CELL 55 — Prototype renderer TPL-06: Boxed Enterprise**

In [ ]:
# ================================================================
# CELL 55 — Prototype renderer TPL-06: Boxed Enterprise
# ================================================================

from pathlib import Path
from typing import Any
from PIL import Image as PILImage

import json
import os

import pandas as pd
import pymupdf


# ================================================================
# Validasi runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "PROTOTYPE_ROOT",
    "CANONICAL_RECORDS_JSONL_PATH",
    "FieldAnnotation",
    "template_registry",
    "invoice_render_payloads",
    "get_page_dimensions",
    "hex_to_rgb",
    "insert_annotated_text",
    "insert_static_text",
    "annotation_to_dict",
    "write_json_atomically",
    "RENDER_ENGINE_VERSION",
]

missing_runtime_objects = [
    name
    for name in REQUIRED_RUNTIME_OBJECTS
    if name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Runtime TPL-06 belum lengkap: "
        f"{missing_runtime_objects}"
    )


# ================================================================
# Konfigurasi
# ================================================================

TPL06_TEMPLATE_ID = "TPL-06"
TPL06_RENDERER_VERSION = "1.0.0"
TPL06_MIN_ITEMS_PER_PAGE = 2
TPL06_MAX_ITEMS_PER_PAGE = 8

TPL06_PROTOTYPE_ROOT = (
    Path(PROTOTYPE_ROOT)
    / TPL06_TEMPLATE_ID
)

TPL06_PROTOTYPE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ================================================================
# Helper canonical payload
# ================================================================

def load_tpl06_canonical_payload(
    canonical_id: str,
) -> dict[str, Any]:

    canonical_path = Path(
        CANONICAL_RECORDS_JSONL_PATH
    )

    if not canonical_path.is_file():
        raise FileNotFoundError(
            f"Canonical JSONL tidak ditemukan: {canonical_path}"
        )

    with canonical_path.open(
        "r",
        encoding="utf-8",
    ) as canonical_file:

        for line_number, line in enumerate(
            canonical_file,
            start=1,
        ):
            if not line.strip():
                continue

            try:
                payload = json.loads(line)
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    "Canonical JSONL rusak pada "
                    f"baris {line_number}."
                ) from error

            if (
                payload.get("canonical_invoice_id")
                == canonical_id
            ):
                return payload

    raise KeyError(
        f"Canonical payload tidak ditemukan: {canonical_id}"
    )


def normalize_pdf_notice(text: Any) -> str:
    """Mengganti separator yang tidak didukung font PDF bawaan."""

    return (
        str(text)
        .replace("—", "·")
        .replace("–", "-")
    )


# ================================================================
# Renderer final TPL-06 — Boxed Enterprise
# ================================================================

def render_template_06(
    render_payload: dict[str, Any],
    output_pdf_path: Path,
) -> tuple[list[FieldAnnotation], dict[str, Any]]:
    """Merender satu halaman TPL-06 dengan layout enterprise compact."""

    if not isinstance(render_payload, dict):
        raise TypeError("render_payload wajib berupa dictionary.")

    if render_payload.get("template_id") != TPL06_TEMPLATE_ID:
        raise ValueError(
            "Renderer TPL-06 hanya menerima payload TPL-06."
        )

    required_keys = {
        "canonical_invoice_id", "document_id", "template_id",
        "language", "currency", "title", "labels", "metadata",
        "vendor", "buyer", "items", "financials", "footer",
        "synthetic_notice",
    }
    missing_keys = sorted(required_keys - set(render_payload))

    if missing_keys:
        raise KeyError(
            f"Payload TPL-06 belum lengkap: {missing_keys}"
        )

    item_count = len(render_payload["items"])

    if not (
        TPL06_MIN_ITEMS_PER_PAGE
        <= item_count
        <= TPL06_MAX_ITEMS_PER_PAGE
    ):
        raise ValueError(
            "TPL-06 hanya mendukung 2–8 item per halaman."
        )

    if (
        render_payload["language"] == "id"
        and render_payload["labels"]["quantity"] != "Kuantitas"
    ):
        raise RuntimeError(
            "Label quantity bahasa Indonesia wajib 'Kuantitas'."
        )

    template_specification = template_registry[TPL06_TEMPLATE_ID]
    expected_template_contract = {
        "layout_family": "boxed_enterprise",
        "page_size": "LETTER",
        "header_layout": "boxed_right",
        "party_layout": "two_columns_boxed",
        "table_style": "full_grid",
        "totals_position": "side_panel",
        "accent_position": "right",
        "density": "compact",
    }
    contract_mismatches = {
        key: {
            "expected": expected_value,
            "actual": template_specification.get(key),
        }
        for key, expected_value in expected_template_contract.items()
        if template_specification.get(key) != expected_value
    }

    if contract_mismatches:
        raise RuntimeError(
            "Spesifikasi layout TPL-06 tidak sesuai: "
            f"{contract_mismatches}"
        )

    page_width, page_height = get_page_dimensions(
        template_specification["page_size"]
    )
    primary_color = hex_to_rgb(
        template_specification["primary_color"]
    )
    accent_color = hex_to_rgb(
        template_specification["accent_color"]
    )
    dark_text = (0.12, 0.16, 0.21)
    muted_text = (0.40, 0.45, 0.49)
    border_color = (0.74, 0.79, 0.83)
    light_primary = (0.945, 0.965, 0.975)
    light_accent = (0.925, 0.955, 0.970)
    white = (1.0, 1.0, 1.0)

    output_pdf_path = Path(output_pdf_path)
    output_pdf_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_pdf_path = output_pdf_path.with_name(
        f".{output_pdf_path.stem}.tmp.pdf"
    )
    temporary_pdf_path.unlink(missing_ok=True)

    annotations: list[FieldAnnotation] = []
    field_font_audit: list[dict[str, Any]] = []
    document = pymupdf.open()

    def add_field(
        page: pymupdf.Page,
        rectangle: pymupdf.Rect,
        text: str,
        field_name: str,
        font_name: str = "helv",
        font_size: float = 7.0,
        minimum_font_size: float = 6.5,
        font_color: tuple[float, float, float] = dark_text,
        alignment: str = "left",
    ) -> None:
        annotation, used_font_size = insert_annotated_text(
            page=page,
            rectangle=rectangle,
            text=str(text),
            field_name=field_name,
            font_name=font_name,
            font_size=font_size,
            minimum_font_size=minimum_font_size,
            font_color=font_color,
            alignment=alignment,
        )
        annotations.append(annotation)
        field_font_audit.append(
            {
                "field_name": field_name,
                "requested_font_size": font_size,
                "used_font_size": used_font_size,
                "adjusted": used_font_size < font_size,
            }
        )

    try:
        page = document.new_page(
            width=page_width,
            height=page_height,
        )
        margin_x0 = 40.0
        margin_x1 = page_width - 40.0

        # Aksen vertikal kanan.
        page.draw_rect(
            pymupdf.Rect(page_width - 10, 0, page_width, page_height),
            color=accent_color,
            fill=accent_color,
            width=0,
            overlay=True,
        )

        # Header kiri.
        page.draw_rect(
            pymupdf.Rect(40, 34, 46, 126),
            color=primary_color,
            fill=primary_color,
            width=0,
            overlay=True,
        )
        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(58, 38, 305, 75),
            text=render_payload["title"],
            font_name="hebo",
            font_size=22.0,
            minimum_font_size=17.0,
            font_color=primary_color,
        )
        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(59, 78, 305, 93),
            text="BOXED ENTERPRISE · SYNTHETIC",
            font_name="hebo",
            font_size=6.7,
            minimum_font_size=6.0,
            font_color=accent_color,
        )
        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(59, 98, 305, 113),
            text=render_payload["footer"]["document_reference"],
            font_name="helv",
            font_size=6.7,
            minimum_font_size=6.0,
            font_color=muted_text,
        )
        currency_label = (
            "Mata Uang"
            if render_payload["language"] == "id"
            else "Currency"
        )
        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(59, 116, 115, 132),
            text=currency_label,
            font_name="helv",
            font_size=6.4,
            minimum_font_size=5.8,
            font_color=muted_text,
        )
        add_field(
            page=page,
            rectangle=pymupdf.Rect(117, 114, 190, 132),
            text=render_payload["currency"],
            field_name="currency",
            font_name="hebo",
            font_size=7.8,
            minimum_font_size=6.5,
            font_color=accent_color,
        )

        # Kotak metadata kanan.
        metadata_x0 = 326.0
        metadata_x1 = margin_x1
        metadata_y0 = 32.0
        metadata_y1 = 130.0
        page.draw_rect(
            pymupdf.Rect(
                metadata_x0, metadata_y0, metadata_x1, metadata_y1
            ),
            color=border_color,
            fill=light_primary,
            width=0.75,
            overlay=True,
        )
        page.draw_rect(
            pymupdf.Rect(
                metadata_x1 - 5, metadata_y0, metadata_x1, metadata_y1
            ),
            color=accent_color,
            fill=accent_color,
            width=0,
            overlay=True,
        )
        metadata_rows = [
            ("invoice_number", 43.0, 61.0),
            ("invoice_date", 70.0, 88.0),
            ("due_date", 97.0, 115.0),
        ]

        for field_name, row_y0, row_y1 in metadata_rows:
            field_payload = render_payload["metadata"][field_name]
            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    metadata_x0 + 13,
                    row_y0,
                    metadata_x0 + 101,
                    row_y1,
                ),
                text=field_payload["label"],
                font_name="helv",
                font_size=6.5,
                minimum_font_size=5.8,
                font_color=muted_text,
            )
            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    metadata_x0 + 103,
                    row_y0,
                    metadata_x1 - 14,
                    row_y1,
                ),
                text=field_payload["display_value"],
                field_name=field_name,
                font_name="hebo",
                font_size=7.5,
                minimum_font_size=6.5,
                font_color=primary_color,
                alignment="right",
            )

        page.draw_line(
            pymupdf.Point(margin_x0, 145),
            pymupdf.Point(margin_x1, 145),
            color=primary_color,
            width=0.9,
            overlay=True,
        )

        # Vendor dan buyer — dua kotak enterprise.
        party_cards = [
            ("vendor", render_payload["labels"]["vendor"], margin_x0, 296.0),
            ("buyer", render_payload["labels"]["buyer"], 310.0, margin_x1),
        ]
        party_y0 = 160.0
        party_y1 = 292.0

        for party_type, party_label, card_x0, card_x1 in party_cards:
            party = render_payload[party_type]
            page.draw_rect(
                pymupdf.Rect(card_x0, party_y0, card_x1, party_y1),
                color=border_color,
                fill=white,
                width=0.75,
                overlay=True,
            )
            page.draw_rect(
                pymupdf.Rect(card_x0, party_y0, card_x1, party_y0 + 23),
                color=None,
                fill=light_accent,
                width=0,
                overlay=True,
            )
            page.draw_rect(
                pymupdf.Rect(card_x1 - 5, party_y0, card_x1, party_y0 + 23),
                color=accent_color,
                fill=accent_color,
                width=0,
                overlay=True,
            )
            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 13,
                    party_y0 + 6,
                    card_x1 - 12,
                    party_y0 + 19,
                ),
                text=party_label.upper(),
                font_name="hebo",
                font_size=6.6,
                minimum_font_size=5.8,
                font_color=accent_color,
            )
            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 13,
                    party_y0 + 30,
                    card_x1 - 13,
                    party_y0 + 47,
                ),
                text=party["name"],
                field_name=f"{party_type}.name",
                font_name="hebo",
                font_size=7.8,
                minimum_font_size=6.5,
                font_color=primary_color,
            )
            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 13,
                    party_y0 + 50,
                    card_x1 - 13,
                    party_y0 + 84,
                ),
                text="\n".join(party["address_lines"]),
                field_name=f"{party_type}.address_lines",
                font_name="helv",
                font_size=6.7,
                minimum_font_size=6.5,
                font_color=dark_text,
            )
            party_contact_fields = [
                ("email", 86.0, 99.0),
                ("phone", 101.0, 114.0),
                ("tax_identifier", 116.0, 129.0),
            ]

            for field_name, field_y0, field_y1 in party_contact_fields:
                party_value = (
                    party["tax_identifier_display"]
                    if field_name == "tax_identifier"
                    else party[field_name]
                )
                add_field(
                    page=page,
                    rectangle=pymupdf.Rect(
                        card_x0 + 13,
                        party_y0 + field_y0,
                        card_x1 - 13,
                        party_y0 + field_y1,
                    ),
                    text=party_value,
                    field_name=f"{party_type}.{field_name}",
                    font_name="helv",
                    font_size=6.6,
                    minimum_font_size=6.5,
                    font_color=dark_text,
                )

        # Tabel item — full grid.
        table_positions = [
            margin_x0, 69.0, 302.0, 362.0, 455.0, margin_x1
        ]
        table_header_y0 = 307.0
        table_header_y1 = 335.0
        row_height = 24.0
        page.draw_rect(
            pymupdf.Rect(
                table_positions[0],
                table_header_y0,
                table_positions[-1],
                table_header_y1,
            ),
            color=primary_color,
            fill=primary_color,
            width=0.7,
            overlay=True,
        )
        table_headers = [
            "#",
            render_payload["labels"]["description"],
            render_payload["labels"]["quantity"],
            render_payload["labels"]["unit_price"],
            render_payload["labels"]["line_total"],
        ]

        for column_index, header_text in enumerate(table_headers):
            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    table_positions[column_index] + 3,
                    table_header_y0 + 7,
                    table_positions[column_index + 1] - 3,
                    table_header_y1 - 2,
                ),
                text=header_text,
                font_name="hebo",
                font_size=6.5,
                minimum_font_size=5.8,
                font_color=white,
                alignment=("left" if column_index == 1 else "center"),
            )

        for item_index, item in enumerate(render_payload["items"]):
            row_y0 = table_header_y1 + item_index * row_height
            row_y1 = row_y0 + row_height

            if item_index % 2 == 1:
                page.draw_rect(
                    pymupdf.Rect(
                        table_positions[0], row_y0, table_positions[-1], row_y1
                    ),
                    color=None,
                    fill=light_primary,
                    width=0,
                    overlay=True,
                )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    table_positions[0] + 3,
                    row_y0 + 6,
                    table_positions[1] - 3,
                    row_y1 - 2,
                ),
                text=str(item["item_number"]),
                font_name="helv",
                font_size=6.7,
                minimum_font_size=6.0,
                font_color=muted_text,
                alignment="center",
            )
            item_fields = [
                ("description", 1, "left"),
                ("quantity", 2, "center"),
                ("unit_price", 3, "right"),
                ("line_total", 4, "right"),
            ]

            for field_name, column_index, alignment in item_fields:
                add_field(
                    page=page,
                    rectangle=pymupdf.Rect(
                        table_positions[column_index] + 4,
                        row_y0 + 6,
                        table_positions[column_index + 1] - 4,
                        row_y1 - 2,
                    ),
                    text=item[field_name],
                    field_name=f"items[{item_index}].{field_name}",
                    font_name="helv",
                    font_size=6.8,
                    minimum_font_size=6.5,
                    font_color=dark_text,
                    alignment=alignment,
                )

        table_end_y = table_header_y1 + item_count * row_height

        # Garis grid digambar setelah teks agar seluruh batas sel konsisten.
        for column_x in table_positions:
            page.draw_line(
                pymupdf.Point(column_x, table_header_y0),
                pymupdf.Point(column_x, table_end_y),
                color=border_color,
                width=0.55,
                overlay=True,
            )

        for row_index in range(item_count + 1):
            row_y = table_header_y1 + row_index * row_height
            page.draw_line(
                pymupdf.Point(table_positions[0], row_y),
                pymupdf.Point(table_positions[-1], row_y),
                color=border_color,
                width=0.55,
                overlay=True,
            )

        page.draw_rect(
            pymupdf.Rect(
                table_positions[0],
                table_header_y0,
                table_positions[-1],
                table_end_y,
            ),
            color=primary_color,
            fill=None,
            width=0.8,
            overlay=True,
        )

        # Financial summary — side panel dinamis.
        totals_panel_height = 112.0
        totals_y0 = min(max(table_end_y + 24.0, 520.0), 552.0)
        totals_y1 = totals_y0 + totals_panel_height
        table_to_totals_gap = totals_y0 - table_end_y

        if table_to_totals_gap < 24.0:
            raise RuntimeError(
                "Tabel bertabrakan dengan side panel total. "
                f"Gap: {table_to_totals_gap:.2f} pt."
            )

        footer_line_y = 716.0

        if totals_y1 >= footer_line_y - 28.0:
            raise RuntimeError(
                "Side panel total terlalu dekat dengan footer. "
                f"Panel berakhir pada {totals_y1:.2f} pt."
            )

        totals_x0 = 360.0
        totals_x1 = margin_x1
        summary_heading = (
            "RINGKASAN"
            if render_payload["language"] == "id"
            else "SUMMARY"
        )
        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                totals_x0, totals_y0 - 19, totals_x1, totals_y0 - 4
            ),
            text=summary_heading,
            font_name="hebo",
            font_size=6.7,
            minimum_font_size=6.0,
            font_color=accent_color,
            alignment="right",
        )
        page.draw_rect(
            pymupdf.Rect(totals_x0, totals_y0, totals_x1, totals_y1),
            color=border_color,
            fill=light_primary,
            width=0.75,
            overlay=True,
        )
        page.draw_rect(
            pymupdf.Rect(totals_x1 - 5, totals_y0, totals_x1, totals_y1),
            color=accent_color,
            fill=accent_color,
            width=0,
            overlay=True,
        )
        financial_rows = [
            ("subtotal", 12.0),
            ("tax", 34.0),
            ("discount", 56.0),
            ("total", 86.0),
        ]

        for field_name, row_offset in financial_rows:
            field_payload = render_payload["financials"][field_name]
            is_total = field_name == "total"
            row_y0 = totals_y0 + row_offset

            if is_total:
                page.draw_line(
                    pymupdf.Point(totals_x0 + 12, totals_y0 + 79),
                    pymupdf.Point(totals_x1 - 13, totals_y0 + 79),
                    color=accent_color,
                    width=0.9,
                    overlay=True,
                )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    totals_x0 + 13,
                    row_y0,
                    totals_x0 + 82,
                    row_y0 + 16,
                ),
                text=field_payload["label"],
                font_name=("hebo" if is_total else "helv"),
                font_size=(7.2 if is_total else 6.8),
                minimum_font_size=6.0,
                font_color=(primary_color if is_total else muted_text),
            )
            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    totals_x0 + 84,
                    row_y0,
                    totals_x1 - 13,
                    row_y0 + 17,
                ),
                text=field_payload["display_value"],
                field_name=f"financials.{field_name}",
                font_name=("hebo" if is_total else "helv"),
                font_size=(8.0 if is_total else 7.0),
                minimum_font_size=6.5,
                font_color=(accent_color if is_total else dark_text),
                alignment="right",
            )

        page.draw_line(
            pymupdf.Point(margin_x0, totals_y0 + 12),
            pymupdf.Point(totals_x0 - 24, totals_y0 + 12),
            color=border_color,
            width=0.7,
            overlay=True,
        )
        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                totals_y0 + 20,
                totals_x0 - 24,
                totals_y0 + 37,
            ),
            text="BOXED ENTERPRISE · TPL-06",
            font_name="hebo",
            font_size=6.4,
            minimum_font_size=5.8,
            font_color=muted_text,
        )

        # Footer.
        page.draw_line(
            pymupdf.Point(margin_x0, footer_line_y),
            pymupdf.Point(margin_x1, footer_line_y),
            color=border_color,
            width=0.7,
            overlay=True,
        )
        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(margin_x0, 724, margin_x1, 739),
            text=render_payload["footer"]["document_reference"],
            font_name="helv",
            font_size=6.5,
            minimum_font_size=6.0,
            font_color=muted_text,
            alignment="center",
        )
        notice_display_text = normalize_pdf_notice(
            render_payload["synthetic_notice"]
        )
        add_field(
            page=page,
            rectangle=pymupdf.Rect(margin_x0, 748, margin_x1, 766),
            text=notice_display_text,
            field_name="document.synthetic_notice",
            font_name="hebo",
            font_size=6.7,
            minimum_font_size=6.5,
            font_color=accent_color,
            alignment="center",
        )

        annotation_names = [
            annotation.field_name for annotation in annotations
        ]

        if len(annotation_names) != len(set(annotation_names)):
            raise RuntimeError("Ditemukan nama anotasi duplikat.")

        expected_annotation_count = 19 + 4 * item_count

        if len(annotations) != expected_annotation_count:
            raise RuntimeError(
                "Jumlah anotasi tidak sesuai. "
                f"Expected: {expected_annotation_count}; "
                f"actual: {len(annotations)}."
            )

        if not field_font_audit:
            raise RuntimeError("Audit font field tidak boleh kosong.")

        minimum_used_font_size = min(
            row["used_font_size"] for row in field_font_audit
        )

        if minimum_used_font_size < 6.5:
            raise RuntimeError(
                "Ukuran font field berada di bawah 6.5 pt."
            )

        document.save(
            str(temporary_pdf_path),
            garbage=4,
            deflate=True,
        )

    except Exception:
        temporary_pdf_path.unlink(missing_ok=True)
        raise

    finally:
        document.close()

    if (
        not temporary_pdf_path.is_file()
        or temporary_pdf_path.stat().st_size == 0
    ):
        temporary_pdf_path.unlink(missing_ok=True)
        raise RuntimeError("PDF sementara TPL-06 gagal dibuat.")

    os.replace(temporary_pdf_path, output_pdf_path)

    rendering_metadata = {
        "renderer_version": RENDER_ENGINE_VERSION,
        "template_renderer_version": TPL06_RENDERER_VERSION,
        "template_id": TPL06_TEMPLATE_ID,
        "layout_family": template_specification["layout_family"],
        "page_size": template_specification["page_size"],
        "page_width_points": round(page_width, 3),
        "page_height_points": round(page_height, 3),
        "page_count": 1,
        "annotation_type": "field_region",
        "annotation_count": len(annotations),
        "minimum_field_font_size": min(
            row["used_font_size"] for row in field_font_audit
        ),
        "font_adjustment_count": int(
            sum(bool(row["adjusted"]) for row in field_font_audit)
        ),
        "layout_metrics": {
            "item_count": item_count,
            "table_end_y_points": round(table_end_y, 3),
            "totals_panel_y0_points": round(totals_y0, 3),
            "totals_panel_y1_points": round(totals_y1, 3),
            "table_to_totals_gap_points": round(
                table_to_totals_gap,
                3,
            ),
            "footer_line_y_points": footer_line_y,
        },
    }

    return annotations, rendering_metadata


# ================================================================
# Pilih prototype TPL-06
# ================================================================

tpl06_candidate_ids = sorted(
    canonical_id
    for canonical_id, payload
    in invoice_render_payloads.items()
    if payload.get("template_id") == TPL06_TEMPLATE_ID
)

if len(tpl06_candidate_ids) != 20:
    raise RuntimeError(
        "TPL-06 harus memiliki tepat 20 payload. "
        f"Actual: {len(tpl06_candidate_ids)}."
    )

prototype_06_canonical_id = (
    tpl06_candidate_ids[0]
)

prototype_06_render_payload = (
    invoice_render_payloads[
        prototype_06_canonical_id
    ]
)

prototype_06_document_id = (
    prototype_06_render_payload[
        "document_id"
    ]
)

prototype_06_canonical_payload = (
    load_tpl06_canonical_payload(
        prototype_06_canonical_id
    )
)


# ================================================================
# Path artifact
# ================================================================

prototype_06_pdf_path = (
    TPL06_PROTOTYPE_ROOT
    / (
        f"{prototype_06_document_id}"
        "_TPL-06_prototype.pdf"
    )
)

prototype_06_png_path = (
    TPL06_PROTOTYPE_ROOT
    / (
        f"{prototype_06_document_id}"
        "_TPL-06_preview.png"
    )
)

prototype_06_ground_truth_path = (
    TPL06_PROTOTYPE_ROOT
    / (
        f"{prototype_06_document_id}"
        "_TPL-06_ground_truth.json"
    )
)


# ================================================================
# Render
# ================================================================

(
    prototype_06_annotations,
    prototype_06_rendering_metadata,
) = render_template_06(
    render_payload=prototype_06_render_payload,
    output_pdf_path=prototype_06_pdf_path,
)


# ================================================================
# Simpan ground truth
# ================================================================

prototype_06_ground_truth = {
    "schema_version": "1.0.0",
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "document": {
        "document_id": prototype_06_document_id,
        "canonical_invoice_id": (
            prototype_06_canonical_id
        ),
        "split": prototype_06_render_payload[
            "split"
        ],
        "template_id": TPL06_TEMPLATE_ID,
        "language": prototype_06_render_payload[
            "language"
        ],
        "currency": prototype_06_render_payload[
            "currency"
        ],
    },
    "canonical": prototype_06_canonical_payload,
    "rendering": (
        prototype_06_rendering_metadata
    ),
    "annotations": [
        annotation_to_dict(annotation)
        for annotation
        in prototype_06_annotations
    ],
}

write_json_atomically(
    prototype_06_ground_truth_path,
    prototype_06_ground_truth,
)


# ================================================================
# Buat preview PNG
# ================================================================

temporary_png_path = (
    prototype_06_png_path.with_name(
        f".{prototype_06_png_path.stem}.tmp.png"
    )
)

temporary_png_path.unlink(
    missing_ok=True
)

try:
    with pymupdf.open(
        str(prototype_06_pdf_path)
    ) as prototype_document:

        if prototype_document.page_count != 1:
            raise RuntimeError(
                "Prototype TPL-06 harus satu halaman."
            )

        prototype_page = prototype_document[0]

        extracted_text_06 = (
            prototype_page.get_text("text")
        )

        prototype_pixmap = (
            prototype_page.get_pixmap(
                matrix=pymupdf.Matrix(
                    150 / 72,
                    150 / 72,
                ),
                alpha=False,
            )
        )

        prototype_pixmap.save(
            str(temporary_png_path)
        )

    if (
        not temporary_png_path.is_file()
        or temporary_png_path.stat().st_size == 0
    ):
        raise RuntimeError(
            "Preview PNG gagal dibuat."
        )

    os.replace(
        temporary_png_path,
        prototype_06_png_path,
    )

except Exception:
    temporary_png_path.unlink(
        missing_ok=True
    )
    raise


# ================================================================
# Pemeriksaan awal
# ================================================================

required_text_fragments = [
    prototype_06_render_payload[
        "metadata"
    ]["invoice_number"]["display_value"],
    prototype_06_render_payload[
        "vendor"
    ]["name"],
    prototype_06_render_payload[
        "buyer"
    ]["name"],
    prototype_06_render_payload[
        "labels"
    ]["quantity"],
    prototype_06_render_payload[
        "financials"
    ]["total"]["display_value"],
    normalize_pdf_notice(
        prototype_06_render_payload[
            "synthetic_notice"
        ]
    ),
]

missing_text_fragments = [
    text
    for text in required_text_fragments
    if text not in extracted_text_06
]

if missing_text_fragments:
    raise RuntimeError(
        "Teks penting hilang dari PDF: "
        f"{missing_text_fragments}"
    )

layout_metrics = (
    prototype_06_rendering_metadata[
        "layout_metrics"
    ]
)

prototype_06_summary = pd.DataFrame(
    [
        {
            "control": "pdf_created",
            "actual": prototype_06_pdf_path.is_file(),
            "status": "VALID",
        },
        {
            "control": "ground_truth_created",
            "actual": (
                prototype_06_ground_truth_path.is_file()
            ),
            "status": "VALID",
        },
        {
            "control": "annotation_count",
            "actual": len(
                prototype_06_annotations
            ),
            "status": "VALID",
        },
        {
            "control": "item_count",
            "actual": layout_metrics[
                "item_count"
            ],
            "status": "VALID",
        },
        {
            "control": "minimum_field_font",
            "actual": (
                prototype_06_rendering_metadata[
                    "minimum_field_font_size"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "font_adjustments",
            "actual": (
                prototype_06_rendering_metadata[
                    "font_adjustment_count"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "table_to_totals_gap",
            "actual": layout_metrics[
                "table_to_totals_gap_points"
            ],
            "status": "VALID",
        },
        {
            "control": "missing_required_text",
            "actual": len(
                missing_text_fragments
            ),
            "status": "VALID",
        },
    ]
)

display(prototype_06_summary)

with PILImage.open(
    prototype_06_png_path
) as prototype_image:
    display(prototype_image.copy())


# ================================================================
# Ringkasan
# ================================================================

print(
    f"Canonical ID       : {prototype_06_canonical_id}"
)
print(
    f"Document ID        : {prototype_06_document_id}"
)
print(
    f"Prototype PDF      : {prototype_06_pdf_path}"
)
print(
    f"Prototype preview  : {prototype_06_png_path}"
)
print(
    f"Ground truth       : {prototype_06_ground_truth_path}"
)
print(
    f"Annotations        : {len(prototype_06_annotations)}"
)
print(
    "Minimum field font : "
    f"{prototype_06_rendering_metadata['minimum_field_font_size']}"
)
print(
    "Adjusted fields    : "
    f"{prototype_06_rendering_metadata['font_adjustment_count']}"
)
print(
    "Table-total gap    : "
    f"{layout_metrics['table_to_totals_gap_points']} pt"
)
print(
    f"Renderer version   : {TPL06_RENDERER_VERSION}"
)
print()
print(
    "✅ Prototype TPL-06 berhasil dibuat dan "
    "lolos pemeriksaan teknis awal."
)


**CELL 56 — FINAL Audit QA teknis prototype TPL-06**

In [ ]:
# ================================================================
# CELL 56 — FINAL
# Audit QA teknis prototype TPL-06
# ================================================================

from pathlib import Path
from typing import Any

import hashlib
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd
import pymupdf
from PIL import Image as PILImage


# ================================================================
# Validasi dependency runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "TPL06_PROTOTYPE_ROOT",
    "TPL06_TEMPLATE_ID",
    "prototype_06_canonical_id",
    "prototype_06_document_id",
    "prototype_06_pdf_path",
    "prototype_06_png_path",
    "prototype_06_ground_truth_path",
    "prototype_06_render_payload",
    "prototype_06_rendering_metadata",
    "prototype_06_annotations",
    "annotation_to_dict",
    "write_json_atomically",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 56 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali Cell 55 versi final."
    )


# ================================================================
# Konfigurasi audit
# ================================================================

QA_SCHEMA_VERSION = "1.0.0"
QA_AUDITOR_VERSION = "1.0.0"

MINIMUM_FIELD_FONT_SIZE = 6.5
MAXIMUM_FONT_ADJUSTMENT_RATIO = 0.20
MINIMUM_RASTER_CONTENT_RATIO = 0.01
MAXIMUM_RASTER_CONTENT_RATIO = 0.40
MINIMUM_RASTER_CONTRAST = 10.0
SEVERE_OVERLAP_THRESHOLD = 0.50

prototype_06_pdf_path = Path(
    prototype_06_pdf_path
)
prototype_06_png_path = Path(
    prototype_06_png_path
)
prototype_06_ground_truth_path = Path(
    prototype_06_ground_truth_path
)

prototype_06_qa_report_path = (
    Path(TPL06_PROTOTYPE_ROOT)
    / (
        f"{prototype_06_document_id}"
        "_TPL-06_qa_report.json"
    )
)

required_files = [
    prototype_06_pdf_path,
    prototype_06_png_path,
    prototype_06_ground_truth_path,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Artifact TPL-06 belum lengkap: "
        f"{missing_files}"
    )


# ================================================================
# Helper umum
# ================================================================

def sha256_file(file_path: Path) -> str:
    digest = hashlib.sha256()

    with Path(file_path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def normalize_text(value: Any) -> str:
    text = unicodedata.normalize(
        "NFKC",
        str(value or ""),
    )

    text = (
        text
        .replace("\u2014", "-")
        .replace("\u2013", "-")
        .replace("\u00b7", "-")
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip().casefold()


def normalize_notice_for_pdf(value: Any) -> str:
    return (
        str(value)
        .replace("\u2014", "\u00b7")
        .replace("\u2013", "-")
    )


def first_present(
    mapping: dict[str, Any],
    candidate_keys: tuple[str, ...],
) -> Any:
    for key in candidate_keys:
        if key in mapping:
            return mapping[key]

    return None


def annotation_field_name(
    annotation: dict[str, Any],
) -> str:
    value = first_present(
        annotation,
        (
            "field_name",
            "field_path",
            "field",
            "path",
            "name",
        ),
    )

    if value is None:
        raise KeyError(
            "Nama field tidak ditemukan pada anotasi. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return str(value)


def annotation_text(
    annotation: dict[str, Any],
) -> str:
    value = first_present(
        annotation,
        (
            "text",
            "value",
            "display_value",
            "field_value",
            "raw_value",
            "raw_text",
            "text_value",
            "content",
        ),
    )

    if isinstance(value, dict):
        value = first_present(
            value,
            (
                "text",
                "value",
                "display_value",
                "raw",
            ),
        )

    if value is None:
        raise KeyError(
            "Nilai teks tidak ditemukan pada anotasi. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return str(value)


def bbox_from_value(
    value: Any,
) -> tuple[float, float, float, float] | None:
    if isinstance(value, (list, tuple)):
        if len(value) != 4:
            return None

        try:
            return tuple(
                float(number)
                for number in value
            )
        except (TypeError, ValueError):
            return None

    if not isinstance(value, dict):
        return None

    direct_key_sets = [
        ("x0", "y0", "x1", "y1"),
        ("left", "top", "right", "bottom"),
    ]

    for key_set in direct_key_sets:
        if all(key in value for key in key_set):
            try:
                return tuple(
                    float(value[key])
                    for key in key_set
                )
            except (TypeError, ValueError):
                return None

    if all(
        key in value
        for key in ("x", "y", "width", "height")
    ):
        try:
            x0 = float(value["x"])
            y0 = float(value["y"])
            width = float(value["width"])
            height = float(value["height"])

            return (
                x0,
                y0,
                x0 + width,
                y0 + height,
            )
        except (TypeError, ValueError):
            return None

    if all(
        key in value
        for key in ("x", "y", "w", "h")
    ):
        try:
            x0 = float(value["x"])
            y0 = float(value["y"])
            width = float(value["w"])
            height = float(value["h"])

            return (
                x0,
                y0,
                x0 + width,
                y0 + height,
            )
        except (TypeError, ValueError):
            return None

    return None


def annotation_bbox(
    annotation: dict[str, Any],
) -> tuple[float, float, float, float]:
    candidate = first_present(
        annotation,
        (
            "bbox",
            "bbox_points",
            "bbox_pt",
            "bbox_pdf",
            "bounding_box",
            "rectangle",
            "rect",
            "coordinates",
            "coordinates_points",
        ),
    )

    if candidate is None:
        geometry = annotation.get("geometry")

        if isinstance(geometry, dict):
            candidate = first_present(
                geometry,
                (
                    "bbox",
                    "bounding_box",
                    "rectangle",
                ),
            )

    bbox = bbox_from_value(candidate)

    if bbox is None:
        raise KeyError(
            "Bounding box tidak ditemukan atau tidak valid. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return bbox


def rectangle_area(
    bbox: tuple[float, float, float, float],
) -> float:
    x0, y0, x1, y1 = bbox

    return max(0.0, x1 - x0) * max(
        0.0,
        y1 - y0,
    )


def intersection_area(
    first_bbox: tuple[float, float, float, float],
    second_bbox: tuple[float, float, float, float],
) -> float:
    first_x0, first_y0, first_x1, first_y1 = first_bbox
    second_x0, second_y0, second_x1, second_y1 = second_bbox

    width = max(
        0.0,
        min(first_x1, second_x1)
        - max(first_x0, second_x0),
    )

    height = max(
        0.0,
        min(first_y1, second_y1)
        - max(first_y0, second_y0),
    )

    return width * height


def bbox_intersects_word(
    annotation_box: tuple[float, float, float, float],
    word_box: tuple[float, float, float, float],
) -> bool:
    return intersection_area(
        annotation_box,
        word_box,
    ) > 0.0


# ================================================================
# Muat ground truth
# ================================================================

with prototype_06_ground_truth_path.open(
    "r",
    encoding="utf-8",
) as ground_truth_file:
    ground_truth = json.load(
        ground_truth_file
    )

ground_truth_document = ground_truth.get(
    "document",
    {},
)

if (
    ground_truth_document.get("document_id")
    != prototype_06_document_id
):
    raise RuntimeError(
        "Document ID ground truth tidak sesuai."
    )

if (
    ground_truth_document.get("template_id")
    != TPL06_TEMPLATE_ID
):
    raise RuntimeError(
        "Ground truth bukan milik TPL-06."
    )

annotation_records = ground_truth.get(
    "annotations",
    [],
)

if not isinstance(annotation_records, list):
    raise TypeError(
        "ground_truth.annotations wajib berupa list."
    )

if not annotation_records:
    raise RuntimeError(
        "Ground truth tidak memiliki anotasi."
    )


# ================================================================
# Expected field mapping
# ================================================================

render_payload = prototype_06_render_payload

expected_values: dict[str, str] = {
    "invoice_number": str(
        render_payload["metadata"]
        ["invoice_number"]
        ["display_value"]
    ),
    "invoice_date": str(
        render_payload["metadata"]
        ["invoice_date"]
        ["display_value"]
    ),
    "due_date": str(
        render_payload["metadata"]
        ["due_date"]
        ["display_value"]
    ),
    "currency": str(
        render_payload["currency"]
    ),
}

for party_type in ("vendor", "buyer"):
    party = render_payload[party_type]

    expected_values.update(
        {
            f"{party_type}.name": str(
                party["name"]
            ),
            f"{party_type}.address_lines": "\n".join(
                party["address_lines"]
            ),
            f"{party_type}.email": str(
                party["email"]
            ),
            f"{party_type}.phone": str(
                party["phone"]
            ),
            f"{party_type}.tax_identifier": str(
                party["tax_identifier_display"]
            ),
        }
    )

for item_index, item in enumerate(
    render_payload["items"]
):
    for field_name in (
        "description",
        "quantity",
        "unit_price",
        "line_total",
    ):
        expected_values[
            f"items[{item_index}].{field_name}"
        ] = str(item[field_name])

for field_name in (
    "subtotal",
    "tax",
    "discount",
    "total",
):
    expected_values[
        f"financials.{field_name}"
    ] = str(
        render_payload["financials"]
        [field_name]
        ["display_value"]
    )

expected_values[
    "document.synthetic_notice"
] = normalize_notice_for_pdf(
    render_payload["synthetic_notice"]
)

expected_annotation_count = len(
    expected_values
)


# ================================================================
# Parse annotations
# ================================================================

parsed_annotations: list[dict[str, Any]] = []
annotation_parse_failures: list[dict[str, Any]] = []

for annotation_index, annotation in enumerate(
    annotation_records
):
    try:
        parsed_annotations.append(
            {
                "index": annotation_index,
                "field_name": annotation_field_name(
                    annotation
                ),
                "text": annotation_text(
                    annotation
                ),
                "bbox": annotation_bbox(
                    annotation
                ),
            }
        )
    except (KeyError, TypeError, ValueError) as error:
        annotation_parse_failures.append(
            {
                "index": annotation_index,
                "error": str(error),
                "keys": sorted(
                    annotation.keys()
                ) if isinstance(
                    annotation,
                    dict,
                ) else [],
            }
        )

if annotation_parse_failures:
    raise RuntimeError(
        "Schema anotasi tidak dapat diproses: "
        f"{annotation_parse_failures[:3]}"
    )

annotation_names = [
    annotation["field_name"]
    for annotation in parsed_annotations
]

annotation_by_name = {
    annotation["field_name"]: annotation
    for annotation in parsed_annotations
}

duplicate_annotation_count = (
    len(annotation_names)
    - len(set(annotation_names))
)

missing_required_annotations = sorted(
    set(expected_values)
    - set(annotation_names)
)


# ================================================================
# Audit PDF dan anotasi
# ================================================================

with pymupdf.open(
    str(prototype_06_pdf_path)
) as pdf_document:
    pdf_page_count = pdf_document.page_count

    if pdf_page_count < 1:
        raise RuntimeError(
            "PDF TPL-06 tidak memiliki halaman."
        )

    first_page = pdf_document[0]
    page_rectangle = first_page.rect
    extracted_text = first_page.get_text("text")
    extracted_words = first_page.get_text("words")

page_width = float(page_rectangle.width)
page_height = float(page_rectangle.height)

word_boxes = [
    (
        float(word[0]),
        float(word[1]),
        float(word[2]),
        float(word[3]),
    )
    for word in extracted_words
    if len(word) >= 5
    and str(word[4]).strip()
]

invalid_bounding_boxes: list[str] = []
annotations_without_words: list[str] = []

for annotation in parsed_annotations:
    field_name = annotation["field_name"]
    x0, y0, x1, y1 = annotation["bbox"]

    coordinates_are_finite = all(
        math.isfinite(value)
        for value in (x0, y0, x1, y1)
    )

    bbox_is_valid = (
        coordinates_are_finite
        and x0 >= 0.0
        and y0 >= 0.0
        and x1 <= page_width
        and y1 <= page_height
        and x1 > x0
        and y1 > y0
    )

    if not bbox_is_valid:
        invalid_bounding_boxes.append(
            field_name
        )
        continue

    contains_word = any(
        bbox_intersects_word(
            annotation["bbox"],
            word_box,
        )
        for word_box in word_boxes
    )

    if not contains_word:
        annotations_without_words.append(
            field_name
        )


# ================================================================
# Audit nilai anotasi
# ================================================================

normalization_mismatches: list[dict[str, str]] = []

for field_name, expected_value in expected_values.items():
    annotation = annotation_by_name.get(
        field_name
    )

    if annotation is None:
        continue

    actual_value = annotation["text"]

    if normalize_text(actual_value) != normalize_text(
        expected_value
    ):
        normalization_mismatches.append(
            {
                "field_name": field_name,
                "expected": expected_value,
                "actual": actual_value,
            }
        )

normalized_extracted_text = normalize_text(
    extracted_text
)

missing_extracted_values: list[str] = []

for annotation in parsed_annotations:
    normalized_value = normalize_text(
        annotation["text"]
    )

    if (
        normalized_value
        and normalized_value
        not in normalized_extracted_text
    ):
        missing_extracted_values.append(
            annotation["field_name"]
        )


# ================================================================
# Audit overlap anotasi
# ================================================================

severe_annotation_overlaps: list[dict[str, Any]] = []

for first_index in range(
    len(parsed_annotations)
):
    first_annotation = parsed_annotations[
        first_index
    ]
    first_area = rectangle_area(
        first_annotation["bbox"]
    )

    if first_area <= 0.0:
        continue

    for second_index in range(
        first_index + 1,
        len(parsed_annotations),
    ):
        second_annotation = parsed_annotations[
            second_index
        ]
        second_area = rectangle_area(
            second_annotation["bbox"]
        )

        if second_area <= 0.0:
            continue

        overlap_area = intersection_area(
            first_annotation["bbox"],
            second_annotation["bbox"],
        )

        overlap_ratio = overlap_area / min(
            first_area,
            second_area,
        )

        if overlap_ratio >= SEVERE_OVERLAP_THRESHOLD:
            severe_annotation_overlaps.append(
                {
                    "first": first_annotation[
                        "field_name"
                    ],
                    "second": second_annotation[
                        "field_name"
                    ],
                    "overlap_ratio": round(
                        overlap_ratio,
                        6,
                    ),
                }
            )


# ================================================================
# Audit font dan raster
# ================================================================

minimum_field_font_size = float(
    prototype_06_rendering_metadata[
        "minimum_field_font_size"
    ]
)

font_adjustment_count = int(
    prototype_06_rendering_metadata[
        "font_adjustment_count"
    ]
)

font_adjustment_ratio = (
    font_adjustment_count
    / max(1, len(parsed_annotations))
)

with PILImage.open(
    prototype_06_png_path
) as preview_image:
    grayscale_array = np.asarray(
        preview_image.convert("L"),
        dtype=np.float32,
    ).copy()

if grayscale_array.size == 0:
    raise RuntimeError(
        "Preview PNG tidak memiliki piksel."
    )

raster_content_ratio = float(
    np.mean(grayscale_array < 245.0)
)

raster_contrast = float(
    np.std(grayscale_array)
)


# ================================================================
# Susun hasil pemeriksaan
# ================================================================

checks = [
    {
        "control": "pdf_page_count",
        "expected": 1,
        "actual": pdf_page_count,
        "valid": pdf_page_count == 1,
    },
    {
        "control": "annotation_count",
        "expected": expected_annotation_count,
        "actual": len(parsed_annotations),
        "valid": (
            len(parsed_annotations)
            == expected_annotation_count
        ),
    },
    {
        "control": "duplicate_annotations",
        "expected": 0,
        "actual": duplicate_annotation_count,
        "valid": duplicate_annotation_count == 0,
    },
    {
        "control": "missing_required_annotations",
        "expected": 0,
        "actual": len(
            missing_required_annotations
        ),
        "valid": not missing_required_annotations,
    },
    {
        "control": "invalid_bounding_boxes",
        "expected": 0,
        "actual": len(
            invalid_bounding_boxes
        ),
        "valid": not invalid_bounding_boxes,
    },
    {
        "control": "normalization_mismatches",
        "expected": 0,
        "actual": len(
            normalization_mismatches
        ),
        "valid": not normalization_mismatches,
    },
    {
        "control": "annotations_without_words",
        "expected": 0,
        "actual": len(
            annotations_without_words
        ),
        "valid": not annotations_without_words,
    },
    {
        "control": "severe_annotation_overlaps",
        "expected": 0,
        "actual": len(
            severe_annotation_overlaps
        ),
        "valid": not severe_annotation_overlaps,
    },
    {
        "control": "missing_extracted_values",
        "expected": 0,
        "actual": len(
            missing_extracted_values
        ),
        "valid": not missing_extracted_values,
    },
    {
        "control": "minimum_field_font_size",
        "expected": ">=6.5",
        "actual": round(
            minimum_field_font_size,
            4,
        ),
        "valid": (
            minimum_field_font_size
            >= MINIMUM_FIELD_FONT_SIZE
        ),
    },
    {
        "control": "font_adjustment_ratio",
        "expected": "<=0.20",
        "actual": round(
            font_adjustment_ratio,
            4,
        ),
        "valid": (
            font_adjustment_ratio
            <= MAXIMUM_FONT_ADJUSTMENT_RATIO
        ),
    },
    {
        "control": "raster_content_ratio",
        "expected": "0.01–0.40",
        "actual": round(
            raster_content_ratio,
            4,
        ),
        "valid": (
            MINIMUM_RASTER_CONTENT_RATIO
            <= raster_content_ratio
            <= MAXIMUM_RASTER_CONTENT_RATIO
        ),
    },
    {
        "control": "raster_contrast",
        "expected": ">=10.0",
        "actual": round(
            raster_contrast,
            4,
        ),
        "valid": (
            raster_contrast
            >= MINIMUM_RASTER_CONTRAST
        ),
    },
]

for check in checks:
    check["status"] = (
        "VALID"
        if check["valid"]
        else "INVALID"
    )

failed_checks = [
    check
    for check in checks
    if not check["valid"]
]

qa_status = (
    "PASSED"
    if not failed_checks
    else "FAILED"
)


# ================================================================
# Simpan QA report
# ================================================================

qa_report = {
    "schema_version": QA_SCHEMA_VERSION,
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "document_id": prototype_06_document_id,
    "canonical_invoice_id": (
        prototype_06_canonical_id
    ),
    "template_id": TPL06_TEMPLATE_ID,
    "status": qa_status,
    "manual_visual_review": "PENDING",
    "metrics": {
        "auditor_version": QA_AUDITOR_VERSION,
        "pdf_page_count": pdf_page_count,
        "annotation_count": len(
            parsed_annotations
        ),
        "expected_annotation_count": (
            expected_annotation_count
        ),
        "duplicate_annotation_count": (
            duplicate_annotation_count
        ),
        "missing_required_annotation_count": len(
            missing_required_annotations
        ),
        "invalid_bounding_box_count": len(
            invalid_bounding_boxes
        ),
        "normalization_mismatch_count": len(
            normalization_mismatches
        ),
        "annotations_without_words_count": len(
            annotations_without_words
        ),
        "severe_annotation_overlap_count": len(
            severe_annotation_overlaps
        ),
        "missing_extracted_value_count": len(
            missing_extracted_values
        ),
        "minimum_field_font_size": round(
            minimum_field_font_size,
            6,
        ),
        "font_adjustment_count": (
            font_adjustment_count
        ),
        "font_adjustment_ratio": round(
            font_adjustment_ratio,
            6,
        ),
        "raster_content_ratio": round(
            raster_content_ratio,
            6,
        ),
        "raster_contrast": round(
            raster_contrast,
            6,
        ),
    },
    "failures": {
        "failed_checks": [
            check["control"]
            for check in failed_checks
        ],
        "missing_required_annotations": (
            missing_required_annotations
        ),
        "invalid_bounding_boxes": (
            invalid_bounding_boxes
        ),
        "normalization_mismatches": (
            normalization_mismatches
        ),
        "annotations_without_words": (
            annotations_without_words
        ),
        "severe_annotation_overlaps": (
            severe_annotation_overlaps
        ),
        "missing_extracted_values": (
            missing_extracted_values
        ),
    },
    "checksums_sha256": {
        "prototype_pdf": sha256_file(
            prototype_06_pdf_path
        ),
        "preview": sha256_file(
            prototype_06_png_path
        ),
        "ground_truth": sha256_file(
            prototype_06_ground_truth_path
        ),
    },
}

write_json_atomically(
    prototype_06_qa_report_path,
    qa_report,
)


# ================================================================
# Verifikasi round-trip QA report
# ================================================================

with prototype_06_qa_report_path.open(
    "r",
    encoding="utf-8",
) as qa_report_file:
    verified_qa_report = json.load(
        qa_report_file
    )

if verified_qa_report != qa_report:
    raise RuntimeError(
        "QA report berubah setelah disimpan dan dibuka ulang."
    )


# ================================================================
# Output audit
# ================================================================

qa_summary = pd.DataFrame(
    [
        {
            "control": check["control"],
            "expected": check["expected"],
            "actual": check["actual"],
            "status": check["status"],
        }
        for check in checks
    ]
)

display(qa_summary)

print(
    f"QA report          : "
    f"{prototype_06_qa_report_path}"
)
print(
    f"QA status          : {qa_status}"
)
print(
    "Minimum field font : "
    f"{minimum_field_font_size:.2f}"
)
print(
    "Font adjustments   : "
    f"{font_adjustment_count}"
)
print(
    "Ink pixel ratio    : "
    f"{raster_content_ratio:.6f}"
)
print(
    "Raster contrast    : "
    f"{raster_contrast:.6f}"
)
print(
    "Manual review      : "
    f"{qa_report['manual_visual_review']}"
)
print()

if qa_status != "PASSED":
    print("Detail kegagalan:")
    print(
        json.dumps(
            qa_report["failures"],
            ensure_ascii=False,
            indent=2,
        )
    )

    raise RuntimeError(
        "Prototype TPL-06 gagal audit QA teknis."
    )

print(
    "✅ Prototype TPL-06 lulus audit teknis. "
    "Review visual manual masih perlu dicatat "
    "ke QA report."
)


In [ ]:
# ================================================================
# CELL 56A — FINAL
# Finalisasi review visual manual prototype TPL-06
# ================================================================

from pathlib import Path

import hashlib
import json


# ================================================================
# Validasi dependency runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "prototype_06_qa_report_path",
    "prototype_06_png_path",
    "prototype_06_document_id",
    "write_json_atomically",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 56A belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali Cell 55 dan Cell 56."
    )


# ================================================================
# Helper checksum
# ================================================================

def calculate_sha256(file_path: Path) -> str:
    """Menghitung SHA-256 file tanpa memuat seluruh file ke memori."""

    digest = hashlib.sha256()

    with file_path.open("rb") as input_file:
        for chunk in iter(
            lambda: input_file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ================================================================
# Muat artifact yang akan difinalisasi
# ================================================================

qa_report_path = Path(
    prototype_06_qa_report_path
)

preview_path = Path(
    prototype_06_png_path
)

if not qa_report_path.is_file():
    raise FileNotFoundError(
        "QA report TPL-06 tidak ditemukan: "
        f"{qa_report_path}"
    )

if not preview_path.is_file():
    raise FileNotFoundError(
        "Preview TPL-06 tidak ditemukan: "
        f"{preview_path}"
    )

with qa_report_path.open(
    mode="r",
    encoding="utf-8",
) as qa_report_file:
    qa_report = json.load(qa_report_file)


# ================================================================
# Validasi identitas dan audit teknis
# ================================================================

if qa_report.get("template_id") != "TPL-06":
    raise RuntimeError(
        "QA report bukan milik TPL-06."
    )

if (
    qa_report.get("document_id")
    != prototype_06_document_id
):
    raise RuntimeError(
        "Document ID QA report tidak sesuai. "
        f"Expected: {prototype_06_document_id!r}; "
        f"actual: {qa_report.get('document_id')!r}."
    )

if qa_report.get("status") != "PASSED":
    raise RuntimeError(
        "Review visual tidak boleh difinalisasi karena "
        "audit teknis belum PASSED. "
        f"Status saat ini: {qa_report.get('status')!r}."
    )

failure_details = qa_report.get(
    "failures",
    {},
)

if not isinstance(failure_details, dict):
    raise TypeError(
        "Field 'failures' pada QA report wajib "
        "berupa dictionary."
    )

nonempty_failures = {
    failure_name: failure_value
    for failure_name, failure_value
    in failure_details.items()
    if bool(failure_value)
}

if nonempty_failures:
    raise RuntimeError(
        "QA report masih memiliki kegagalan teknis: "
        f"{nonempty_failures}"
    )

if "manual_visual_review" not in qa_report:
    raise KeyError(
        "Field 'manual_visual_review' tidak ditemukan "
        "dalam QA report."
    )


# ================================================================
# Pastikan preview yang direview sama dengan preview yang diaudit
# ================================================================

recorded_preview_checksum = (
    qa_report
    .get("checksums_sha256", {})
    .get("preview")
)

if not recorded_preview_checksum:
    raise KeyError(
        "Checksum preview tidak ditemukan pada "
        "qa_report['checksums_sha256']['preview']."
    )

actual_preview_checksum = calculate_sha256(
    preview_path
)

if actual_preview_checksum != recorded_preview_checksum:
    raise RuntimeError(
        "Preview berubah setelah audit teknis. "
        "Jalankan ulang Cell 56 dan lakukan review visual kembali. "
        f"Expected SHA-256: {recorded_preview_checksum}; "
        f"actual: {actual_preview_checksum}."
    )


# ================================================================
# Catat hasil review visual secara idempoten
# ================================================================

qa_report["manual_visual_review"] = "PASSED"

write_json_atomically(
    qa_report_path,
    qa_report,
)


# ================================================================
# Buka ulang dan verifikasi hasil persistensi
# ================================================================

with qa_report_path.open(
    mode="r",
    encoding="utf-8",
) as qa_report_file:
    persisted_qa_report = json.load(
        qa_report_file
    )

if persisted_qa_report.get("status") != "PASSED":
    raise RuntimeError(
        "Status audit teknis berubah setelah finalisasi."
    )

if (
    persisted_qa_report.get("manual_visual_review")
    != "PASSED"
):
    raise RuntimeError(
        "Status review visual gagal disimpan."
    )

persisted_preview_checksum = (
    persisted_qa_report
    .get("checksums_sha256", {})
    .get("preview")
)

if persisted_preview_checksum != actual_preview_checksum:
    raise RuntimeError(
        "Checksum preview berubah pada QA report."
    )


# ================================================================
# Ringkasan final
# ================================================================

print(
    f"QA report           : {qa_report_path}"
)

print(
    "Technical status    : "
    f"{persisted_qa_report['status']}"
)

print(
    "Manual visual review: "
    f"{persisted_qa_report['manual_visual_review']}"
)

print(
    "Preview SHA-256     : "
    f"{persisted_preview_checksum}"
)

print()

print(
    "✅ TPL-06 FINAL PASSED — audit teknis dan "
    "review visual telah selesai."
)


In [ ]:
# ================================================================
# CELL 57 — FINAL
# Inspeksi spesifikasi dan presentation payload TPL-07
# Read-only: tidak membuat atau mengubah artifact build
# ================================================================

from collections import Counter
from typing import Any

import json

import pandas as pd
from IPython.display import display


# ================================================================
# Validasi dependency runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "template_registry",
    "invoice_render_payloads",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 57 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali cell pembentukan template registry "
        "dan presentation payload."
    )

if not isinstance(template_registry, dict):
    raise TypeError(
        "template_registry wajib berupa dictionary."
    )

if not isinstance(invoice_render_payloads, dict):
    raise TypeError(
        "invoice_render_payloads wajib berupa dictionary."
    )


# ================================================================
# Konfigurasi inspeksi
# ================================================================

TPL07_TEMPLATE_ID = "TPL-07"
TPL07_EXPECTED_PAYLOAD_COUNT = 20

if TPL07_TEMPLATE_ID not in template_registry:
    raise KeyError(
        "Spesifikasi TPL-07 tidak ditemukan dalam "
        "template_registry."
    )

tpl07_template_specification = template_registry[
    TPL07_TEMPLATE_ID
]

if not isinstance(tpl07_template_specification, dict):
    raise TypeError(
        "Spesifikasi TPL-07 wajib berupa dictionary."
    )

tpl07_payloads = {
    canonical_id: payload
    for canonical_id, payload
    in invoice_render_payloads.items()
    if isinstance(payload, dict)
    and payload.get("template_id") == TPL07_TEMPLATE_ID
}

if len(tpl07_payloads) != TPL07_EXPECTED_PAYLOAD_COUNT:
    raise RuntimeError(
        "TPL-07 harus memiliki tepat "
        f"{TPL07_EXPECTED_PAYLOAD_COUNT} payload. "
        f"Actual: {len(tpl07_payloads)}."
    )


# ================================================================
# Kontrak struktur presentation payload
# ================================================================

REQUIRED_TOP_LEVEL_KEYS = {
    "buyer",
    "canonical_invoice_id",
    "currency",
    "document_id",
    "financials",
    "footer",
    "items",
    "labels",
    "language",
    "metadata",
    "split",
    "synthetic_notice",
    "template_id",
    "title",
    "vendor",
}

REQUIRED_LABEL_KEYS = {
    "buyer",
    "description",
    "discount",
    "document_title",
    "due_date",
    "invoice_date",
    "invoice_number",
    "line_total",
    "page",
    "quantity",
    "subtotal",
    "synthetic_notice",
    "tax",
    "tax_identifier",
    "total",
    "unit_price",
    "vendor",
}

REQUIRED_METADATA_KEYS = {
    "due_date",
    "invoice_date",
    "invoice_number",
}

REQUIRED_PARTY_KEYS = {
    "address_lines",
    "email",
    "name",
    "party_id",
    "phone",
    "tax_identifier",
    "tax_identifier_display",
}

REQUIRED_ITEM_KEYS = {
    "description",
    "item_number",
    "line_total",
    "quantity",
    "raw",
    "unit_price",
}

REQUIRED_FINANCIAL_KEYS = {
    "discount",
    "subtotal",
    "tax",
    "total",
}

REQUIRED_FOOTER_KEYS = {
    "document_reference",
    "synthetic_notice",
}


def require_dictionary(
    value: Any,
    field_path: str,
) -> dict[str, Any]:
    """Memastikan sebuah field merupakan dictionary."""

    if not isinstance(value, dict):
        raise TypeError(
            f"{field_path} wajib berupa dictionary."
        )

    return value


def require_key_set(
    value: dict[str, Any],
    required_keys: set[str],
    field_path: str,
) -> None:
    """Memastikan seluruh key wajib tersedia."""

    missing_keys = sorted(
        required_keys - set(value)
    )

    if missing_keys:
        raise KeyError(
            f"{field_path} kehilangan key wajib: "
            f"{missing_keys}"
        )


expected_split = tpl07_template_specification.get(
    "split"
)

document_ids: list[str] = []

for canonical_id in sorted(tpl07_payloads):
    payload = tpl07_payloads[canonical_id]

    require_key_set(
        payload,
        REQUIRED_TOP_LEVEL_KEYS,
        f"payload[{canonical_id!r}]",
    )

    if payload["canonical_invoice_id"] != canonical_id:
        raise RuntimeError(
            "Canonical ID pada dictionary dan payload "
            f"tidak sama: {canonical_id!r}."
        )

    if payload["template_id"] != TPL07_TEMPLATE_ID:
        raise RuntimeError(
            f"Template ID tidak sesuai pada {canonical_id}."
        )

    if expected_split is not None and (
        payload["split"] != expected_split
    ):
        raise RuntimeError(
            f"Split payload {canonical_id} tidak sesuai. "
            f"Expected: {expected_split!r}; "
            f"actual: {payload['split']!r}."
        )

    if payload["language"] not in {"id", "en"}:
        raise RuntimeError(
            f"Language tidak didukung pada {canonical_id}: "
            f"{payload['language']!r}."
        )

    labels = require_dictionary(
        payload["labels"],
        f"payload[{canonical_id!r}].labels",
    )

    metadata = require_dictionary(
        payload["metadata"],
        f"payload[{canonical_id!r}].metadata",
    )

    vendor = require_dictionary(
        payload["vendor"],
        f"payload[{canonical_id!r}].vendor",
    )

    buyer = require_dictionary(
        payload["buyer"],
        f"payload[{canonical_id!r}].buyer",
    )

    financials = require_dictionary(
        payload["financials"],
        f"payload[{canonical_id!r}].financials",
    )

    footer = require_dictionary(
        payload["footer"],
        f"payload[{canonical_id!r}].footer",
    )

    require_key_set(
        labels,
        REQUIRED_LABEL_KEYS,
        f"payload[{canonical_id!r}].labels",
    )

    require_key_set(
        metadata,
        REQUIRED_METADATA_KEYS,
        f"payload[{canonical_id!r}].metadata",
    )

    require_key_set(
        vendor,
        REQUIRED_PARTY_KEYS,
        f"payload[{canonical_id!r}].vendor",
    )

    require_key_set(
        buyer,
        REQUIRED_PARTY_KEYS,
        f"payload[{canonical_id!r}].buyer",
    )

    require_key_set(
        financials,
        REQUIRED_FINANCIAL_KEYS,
        f"payload[{canonical_id!r}].financials",
    )

    require_key_set(
        footer,
        REQUIRED_FOOTER_KEYS,
        f"payload[{canonical_id!r}].footer",
    )

    items = payload["items"]

    if not isinstance(items, list) or not items:
        raise RuntimeError(
            f"payload[{canonical_id!r}].items wajib "
            "berupa list yang tidak kosong."
        )

    for item_index, item in enumerate(items):
        item = require_dictionary(
            item,
            (
                f"payload[{canonical_id!r}]"
                f".items[{item_index}]"
            ),
        )

        require_key_set(
            item,
            REQUIRED_ITEM_KEYS,
            (
                f"payload[{canonical_id!r}]"
                f".items[{item_index}]"
            ),
        )

    document_ids.append(
        str(payload["document_id"])
    )

if len(document_ids) != len(set(document_ids)):
    raise RuntimeError(
        "Ditemukan document_id duplikat pada payload TPL-07."
    )


# ================================================================
# Helper statistik panjang teks
# ================================================================

def joined_text_length(values: Any) -> int:
    """Mengukur panjang gabungan list teks secara konsisten."""

    if not isinstance(values, list):
        return len(str(values))

    return len(
        " ".join(str(value) for value in values)
    )


def maximum_money_length(
    payload: dict[str, Any],
) -> int:
    """Mengukur nilai uang terpanjang yang perlu dirender."""

    money_values: list[str] = []

    for item in payload["items"]:
        money_values.extend(
            [
                str(item["unit_price"]),
                str(item["line_total"]),
            ]
        )

    for field_name in (
        "subtotal",
        "tax",
        "discount",
        "total",
    ):
        financial_value = payload[
            "financials"
        ][field_name]

        if isinstance(financial_value, dict):
            financial_value = financial_value.get(
                "display_value",
                financial_value.get("value", ""),
            )

        money_values.append(
            str(financial_value)
        )

    return max(
        map(len, money_values),
        default=0,
    )


# ================================================================
# Bangun ringkasan inspeksi
# ================================================================

tpl07_candidate_ids = sorted(
    tpl07_payloads
)

tpl07_sample_id = tpl07_candidate_ids[0]
tpl07_sample_payload = tpl07_payloads[
    tpl07_sample_id
]

tpl07_sample_structure = {
    "canonical_invoice_id": (
        tpl07_sample_payload["canonical_invoice_id"]
    ),
    "document_id": tpl07_sample_payload[
        "document_id"
    ],
    "template_id": tpl07_sample_payload[
        "template_id"
    ],
    "language": tpl07_sample_payload["language"],
    "currency": tpl07_sample_payload["currency"],
    "title": tpl07_sample_payload["title"],
    "top_level_keys": sorted(
        tpl07_sample_payload
    ),
    "label_keys": sorted(
        tpl07_sample_payload["labels"]
    ),
    "metadata_keys": sorted(
        tpl07_sample_payload["metadata"]
    ),
    "vendor_keys": sorted(
        tpl07_sample_payload["vendor"]
    ),
    "buyer_keys": sorted(
        tpl07_sample_payload["buyer"]
    ),
    "item_keys": sorted(
        tpl07_sample_payload["items"][0]
    ),
    "financial_keys": sorted(
        tpl07_sample_payload["financials"]
    ),
    "footer_keys": sorted(
        tpl07_sample_payload["footer"]
    ),
}

statistics_rows: list[dict[str, Any]] = []

for canonical_id in tpl07_candidate_ids:
    payload = tpl07_payloads[canonical_id]
    descriptions = [
        str(item["description"])
        for item in payload["items"]
    ]

    statistics_rows.append(
        {
            "canonical_id": canonical_id,
            "document_id": payload["document_id"],
            "language": payload["language"],
            "currency": payload["currency"],
            "item_count": len(payload["items"]),
            "vendor_name_length": len(
                str(payload["vendor"]["name"])
            ),
            "vendor_address_length": joined_text_length(
                payload["vendor"]["address_lines"]
            ),
            "vendor_email_length": len(
                str(payload["vendor"]["email"])
            ),
            "buyer_name_length": len(
                str(payload["buyer"]["name"])
            ),
            "buyer_address_length": joined_text_length(
                payload["buyer"]["address_lines"]
            ),
            "buyer_email_length": len(
                str(payload["buyer"]["email"])
            ),
            "maximum_description_length": max(
                map(len, descriptions),
                default=0,
            ),
            "maximum_money_length": maximum_money_length(
                payload
            ),
        }
    )

tpl07_statistics = pd.DataFrame(
    statistics_rows
)

tpl07_payload_statistics = (
    tpl07_statistics
    .describe(include="all")
    .transpose()
)

tpl07_item_count_distribution = (
    tpl07_statistics["item_count"]
    .value_counts()
    .sort_index()
    .rename_axis("item_count")
    .reset_index(name="document_count")
)

tpl07_language_distribution = (
    tpl07_statistics["language"]
    .value_counts()
    .sort_index()
    .rename_axis("language")
    .reset_index(name="document_count")
)

tpl07_currency_distribution = (
    tpl07_statistics["currency"]
    .value_counts()
    .sort_index()
    .rename_axis("currency")
    .reset_index(name="document_count")
)


# ================================================================
# Output inspeksi
# ================================================================

print("TEMPLATE SPECIFICATION")
print(
    json.dumps(
        tpl07_template_specification,
        ensure_ascii=False,
        indent=2,
    )
)

print()
print("PAYLOAD COUNT")
print(len(tpl07_payloads))

print()
print("SAMPLE STRUCTURE")
print(
    json.dumps(
        tpl07_sample_structure,
        ensure_ascii=False,
        indent=2,
    )
)

print()
print("LABELS")
print(
    json.dumps(
        tpl07_sample_payload["labels"],
        ensure_ascii=False,
        indent=2,
    )
)

print()
print("PAYLOAD STATISTICS")
display(tpl07_payload_statistics)

print()
print("ITEM-COUNT DISTRIBUTION")
display(tpl07_item_count_distribution)

print()
print("LANGUAGE DISTRIBUTION")
display(tpl07_language_distribution)

print()
print("CURRENCY DISTRIBUTION")
display(tpl07_currency_distribution)

print(
    "✅ Inspeksi TPL-07 selesai. "
    "Belum ada artifact yang dibuat atau diubah."
)


In [ ]:
# ================================================================
# CELL 58 — FINAL
# Prototype renderer TPL-07: Top Stripe
# ================================================================

from pathlib import Path
from typing import Any
from PIL import Image as PILImage

import json
import os

import pandas as pd
import pymupdf


# ================================================================
# Validasi runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "PROTOTYPE_ROOT",
    "CANONICAL_RECORDS_JSONL_PATH",
    "FieldAnnotation",
    "template_registry",
    "invoice_render_payloads",
    "get_page_dimensions",
    "hex_to_rgb",
    "insert_annotated_text",
    "insert_static_text",
    "annotation_to_dict",
    "write_json_atomically",
    "RENDER_ENGINE_VERSION",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 58 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali cell pemulihan runtime."
    )


# ================================================================
# Konfigurasi TPL-07
# ================================================================

TPL07_TEMPLATE_ID = "TPL-07"
TPL07_RENDERER_VERSION = "1.0.0"
TPL07_MIN_ITEMS_PER_PAGE = 2
TPL07_MAX_ITEMS_PER_PAGE = 8

TPL07_PROTOTYPE_ROOT = (
    Path(PROTOTYPE_ROOT)
    / TPL07_TEMPLATE_ID
)

TPL07_PROTOTYPE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ================================================================
# Helper payload dan teks PDF
# ================================================================

def load_tpl07_canonical_payload(
    canonical_id: str,
) -> dict[str, Any]:
    canonical_path = Path(
        CANONICAL_RECORDS_JSONL_PATH
    )

    if not canonical_path.is_file():
        raise FileNotFoundError(
            "Canonical JSONL tidak ditemukan: "
            f"{canonical_path}"
        )

    with canonical_path.open(
        "r",
        encoding="utf-8",
    ) as canonical_file:
        for line_number, line in enumerate(
            canonical_file,
            start=1,
        ):
            clean_line = line.strip()

            if not clean_line:
                continue

            try:
                payload = json.loads(clean_line)
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    "Canonical JSONL tidak valid pada "
                    f"baris {line_number}."
                ) from error

            if (
                payload.get("canonical_invoice_id")
                == canonical_id
            ):
                return payload

    raise KeyError(
        "Canonical payload tidak ditemukan: "
        f"{canonical_id}"
    )


def normalize_pdf_notice(text: Any) -> str:
    return (
        str(text)
        .replace("—", "·")
        .replace("–", "-")
    )


# ================================================================
# Renderer TPL-07
# ================================================================

def render_template_07(
    render_payload: dict[str, Any],
    output_pdf_path: Path,
) -> tuple[list[FieldAnnotation], dict[str, Any]]:
    """Merender invoice TPL-07 dengan top stripe dan stacked split."""

    if not isinstance(render_payload, dict):
        raise TypeError("render_payload wajib berupa dictionary.")

    if render_payload.get("template_id") != TPL07_TEMPLATE_ID:
        raise ValueError(
            "Renderer TPL-07 hanya menerima payload TPL-07."
        )

    required_keys = {
        "canonical_invoice_id", "document_id", "template_id",
        "language", "currency", "title", "labels", "metadata",
        "vendor", "buyer", "items", "financials", "footer",
        "synthetic_notice",
    }
    missing_keys = sorted(required_keys - set(render_payload))

    if missing_keys:
        raise KeyError(
            f"Payload TPL-07 belum lengkap: {missing_keys}"
        )

    item_count = len(render_payload["items"])

    if not (
        TPL07_MIN_ITEMS_PER_PAGE
        <= item_count
        <= TPL07_MAX_ITEMS_PER_PAGE
    ):
        raise ValueError(
            "TPL-07 hanya mendukung 2–8 item per halaman."
        )

    if (
        render_payload["language"] == "id"
        and render_payload["labels"]["quantity"] != "Kuantitas"
    ):
        raise RuntimeError(
            "Label quantity bahasa Indonesia wajib 'Kuantitas'."
        )

    template_specification = template_registry[TPL07_TEMPLATE_ID]
    expected_template_contract = {
        "layout_family": "top_stripe",
        "page_size": "A4",
        "header_layout": "title_with_top_stripe",
        "party_layout": "stacked_split",
        "table_style": "striped",
        "totals_position": "bottom_card",
        "accent_position": "top",
        "density": "regular",
    }
    contract_mismatches = {
        key: {
            "expected": expected_value,
            "actual": template_specification.get(key),
        }
        for key, expected_value in expected_template_contract.items()
        if template_specification.get(key) != expected_value
    }

    if contract_mismatches:
        raise RuntimeError(
            "Spesifikasi layout TPL-07 tidak sesuai: "
            f"{contract_mismatches}"
        )

    page_width, page_height = get_page_dimensions(
        template_specification["page_size"]
    )
    primary_color = hex_to_rgb(
        template_specification["primary_color"]
    )
    accent_color = hex_to_rgb(
        template_specification["accent_color"]
    )
    dark_text = (0.14, 0.17, 0.12)
    muted_text = (0.40, 0.44, 0.37)
    border_color = (0.79, 0.82, 0.76)
    light_primary = (0.958, 0.970, 0.945)
    light_accent = (0.990, 0.965, 0.925)
    white = (1.0, 1.0, 1.0)

    output_pdf_path = Path(output_pdf_path)
    output_pdf_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_pdf_path = output_pdf_path.with_name(
        f".{output_pdf_path.stem}.tmp.pdf"
    )
    temporary_pdf_path.unlink(missing_ok=True)

    annotations: list[FieldAnnotation] = []
    field_font_audit: list[dict[str, Any]] = []
    document = pymupdf.open()

    def add_field(
        page: pymupdf.Page,
        rectangle: pymupdf.Rect,
        text: str,
        field_name: str,
        font_name: str = "helv",
        font_size: float = 7.0,
        minimum_font_size: float = 6.5,
        font_color: tuple[float, float, float] = dark_text,
        alignment: str = "left",
    ) -> None:
        annotation, used_font_size = insert_annotated_text(
            page=page,
            rectangle=rectangle,
            text=str(text),
            field_name=field_name,
            font_name=font_name,
            font_size=font_size,
            minimum_font_size=minimum_font_size,
            font_color=font_color,
            alignment=alignment,
        )
        annotations.append(annotation)
        field_font_audit.append(
            {
                "field_name": field_name,
                "requested_font_size": font_size,
                "used_font_size": used_font_size,
                "adjusted": used_font_size < font_size,
            }
        )

    try:
        page = document.new_page(width=page_width, height=page_height)
        margin_x0 = 45.0
        margin_x1 = page_width - 45.0

        # Aksen utama di bagian atas.
        page.draw_rect(
            pymupdf.Rect(0, 0, page_width, 14),
            color=accent_color,
            fill=accent_color,
            width=0,
            overlay=True,
        )
        page.draw_rect(
            pymupdf.Rect(0, 14, page_width, 18),
            color=primary_color,
            fill=primary_color,
            width=0,
            overlay=True,
        )

        # Header judul.
        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(45, 42, 294, 82),
            text=render_payload["title"],
            font_name="hebo",
            font_size=23.0,
            minimum_font_size=18.0,
            font_color=primary_color,
        )
        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(47, 86, 294, 101),
            text="TOP STRIPE · SYNTHETIC",
            font_name="hebo",
            font_size=6.8,
            minimum_font_size=6.0,
            font_color=accent_color,
        )
        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(47, 106, 294, 121),
            text=render_payload["footer"]["document_reference"],
            font_name="helv",
            font_size=6.7,
            minimum_font_size=6.0,
            font_color=muted_text,
        )
        currency_label = (
            "Mata Uang"
            if render_payload["language"] == "id"
            else "Currency"
        )
        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(47, 128, 107, 144),
            text=currency_label,
            font_name="helv",
            font_size=6.5,
            minimum_font_size=5.8,
            font_color=muted_text,
        )
        add_field(
            page=page,
            rectangle=pymupdf.Rect(109, 126, 180, 145),
            text=render_payload["currency"],
            field_name="currency",
            font_name="hebo",
            font_size=8.0,
            minimum_font_size=6.5,
            font_color=accent_color,
        )

        # Metadata kanan.
        metadata_x0 = 310.0
        metadata_x1 = margin_x1
        metadata_y0 = 39.0
        metadata_y1 = 145.0
        page.draw_rect(
            pymupdf.Rect(
                metadata_x0, metadata_y0, metadata_x1, metadata_y1
            ),
            color=None,
            fill=light_primary,
            width=0,
            overlay=True,
        )
        page.draw_rect(
            pymupdf.Rect(
                metadata_x0, metadata_y0, metadata_x0 + 5, metadata_y1
            ),
            color=accent_color,
            fill=accent_color,
            width=0,
            overlay=True,
        )
        metadata_rows = [
            ("invoice_number", 50.0, 69.0),
            ("invoice_date", 80.0, 99.0),
            ("due_date", 110.0, 129.0),
        ]

        for field_name, row_y0, row_y1 in metadata_rows:
            field_payload = render_payload["metadata"][field_name]
            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    metadata_x0 + 16,
                    row_y0,
                    metadata_x0 + 96,
                    row_y1,
                ),
                text=field_payload["label"],
                font_name="helv",
                font_size=6.5,
                minimum_font_size=5.8,
                font_color=muted_text,
            )
            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    metadata_x0 + 98,
                    row_y0,
                    metadata_x1 - 12,
                    row_y1,
                ),
                text=field_payload["display_value"],
                field_name=field_name,
                font_name="hebo",
                font_size=7.4,
                minimum_font_size=6.5,
                font_color=primary_color,
                alignment="right",
            )

        page.draw_line(
            pymupdf.Point(margin_x0, 157),
            pymupdf.Point(margin_x1, 157),
            color=primary_color,
            width=0.8,
            overlay=True,
        )

        # Vendor dan buyer — stacked split.
        party_sections = [
            ("vendor", render_payload["labels"]["vendor"], 169.0, 246.0),
            ("buyer", render_payload["labels"]["buyer"], 258.0, 335.0),
        ]

        for party_type, party_label, section_y0, section_y1 in party_sections:
            party = render_payload[party_type]
            page.draw_rect(
                pymupdf.Rect(
                    margin_x0, section_y0, margin_x1, section_y1
                ),
                color=border_color,
                fill=white,
                width=0.7,
                overlay=True,
            )
            page.draw_rect(
                pymupdf.Rect(
                    margin_x0, section_y0, margin_x0 + 5, section_y1
                ),
                color=accent_color,
                fill=accent_color,
                width=0,
                overlay=True,
            )
            page.draw_line(
                pymupdf.Point(320, section_y0 + 11),
                pymupdf.Point(320, section_y1 - 11),
                color=border_color,
                width=0.6,
                overlay=True,
            )
            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    margin_x0 + 14,
                    section_y0 + 7,
                    306,
                    section_y0 + 20,
                ),
                text=party_label.upper(),
                font_name="hebo",
                font_size=6.5,
                minimum_font_size=5.8,
                font_color=accent_color,
            )
            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    margin_x0 + 14,
                    section_y0 + 22,
                    306,
                    section_y0 + 39,
                ),
                text=party["name"],
                field_name=f"{party_type}.name",
                font_name="hebo",
                font_size=7.8,
                minimum_font_size=6.5,
                font_color=primary_color,
            )
            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    margin_x0 + 14,
                    section_y0 + 41,
                    306,
                    section_y1 - 5,
                ),
                text="\n".join(party["address_lines"]),
                field_name=f"{party_type}.address_lines",
                font_name="helv",
                font_size=6.6,
                minimum_font_size=6.5,
                font_color=dark_text,
            )
            contact_fields = [
                ("email", 10.0, 25.0),
                ("phone", 31.0, 46.0),
                ("tax_identifier", 52.0, 69.0),
            ]

            for field_name, field_y0, field_y1 in contact_fields:
                value = (
                    party["tax_identifier_display"]
                    if field_name == "tax_identifier"
                    else party[field_name]
                )
                add_field(
                    page=page,
                    rectangle=pymupdf.Rect(
                        335,
                        section_y0 + field_y0,
                        margin_x1 - 12,
                        section_y0 + field_y1,
                    ),
                    text=value,
                    field_name=f"{party_type}.{field_name}",
                    font_name="helv",
                    font_size=6.7,
                    minimum_font_size=6.5,
                    font_color=dark_text,
                )

        # Tabel striped.
        table_positions = [
            margin_x0, 70.0, 315.0, 370.0, 458.0, margin_x1
        ]
        table_header_y0 = 350.0
        table_header_y1 = 378.0
        row_height = 25.0
        page.draw_rect(
            pymupdf.Rect(
                table_positions[0],
                table_header_y0,
                table_positions[-1],
                table_header_y1,
            ),
            color=primary_color,
            fill=primary_color,
            width=0,
            overlay=True,
        )
        table_headers = [
            "#",
            render_payload["labels"]["description"],
            render_payload["labels"]["quantity"],
            render_payload["labels"]["unit_price"],
            render_payload["labels"]["line_total"],
        ]

        for column_index, header_text in enumerate(table_headers):
            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    table_positions[column_index] + 3,
                    table_header_y0 + 7,
                    table_positions[column_index + 1] - 3,
                    table_header_y1 - 2,
                ),
                text=header_text,
                font_name="hebo",
                font_size=6.5,
                minimum_font_size=5.8,
                font_color=white,
                alignment=("left" if column_index == 1 else "center"),
            )

        for item_index, item in enumerate(render_payload["items"]):
            row_y0 = table_header_y1 + item_index * row_height
            row_y1 = row_y0 + row_height

            if item_index % 2 == 1:
                page.draw_rect(
                    pymupdf.Rect(
                        table_positions[0], row_y0, table_positions[-1], row_y1
                    ),
                    color=None,
                    fill=light_primary,
                    width=0,
                    overlay=True,
                )

            page.draw_line(
                pymupdf.Point(table_positions[0], row_y1),
                pymupdf.Point(table_positions[-1], row_y1),
                color=border_color,
                width=0.5,
                overlay=True,
            )
            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    table_positions[0] + 3,
                    row_y0 + 6,
                    table_positions[1] - 3,
                    row_y1 - 2,
                ),
                text=str(item["item_number"]),
                font_name="helv",
                font_size=6.8,
                minimum_font_size=6.0,
                font_color=muted_text,
                alignment="center",
            )
            item_fields = [
                ("description", 1, "left"),
                ("quantity", 2, "center"),
                ("unit_price", 3, "right"),
                ("line_total", 4, "right"),
            ]

            for field_name, column_index, alignment in item_fields:
                add_field(
                    page=page,
                    rectangle=pymupdf.Rect(
                        table_positions[column_index] + 4,
                        row_y0 + 6,
                        table_positions[column_index + 1] - 4,
                        row_y1 - 2,
                    ),
                    text=item[field_name],
                    field_name=f"items[{item_index}].{field_name}",
                    font_name="helv",
                    font_size=6.9,
                    minimum_font_size=6.5,
                    font_color=dark_text,
                    alignment=alignment,
                )

        table_end_y = table_header_y1 + item_count * row_height

        # Bottom totals card.
        totals_panel_height = 106.0
        totals_y0 = min(max(table_end_y + 26.0, 604.0), 640.0)
        totals_y1 = totals_y0 + totals_panel_height
        table_to_totals_gap = totals_y0 - table_end_y

        if table_to_totals_gap < 26.0:
            raise RuntimeError(
                "Tabel bertabrakan dengan kartu total. "
                f"Gap: {table_to_totals_gap:.2f} pt."
            )

        footer_line_y = 770.0

        if totals_y1 >= footer_line_y - 24.0:
            raise RuntimeError(
                "Kartu total terlalu dekat dengan footer. "
                f"Panel berakhir pada {totals_y1:.2f} pt."
            )

        totals_x0 = 300.0
        totals_x1 = margin_x1
        page.draw_rect(
            pymupdf.Rect(totals_x0, totals_y0, totals_x1, totals_y1),
            color=border_color,
            fill=light_accent,
            width=0.7,
            overlay=True,
        )
        page.draw_rect(
            pymupdf.Rect(totals_x0, totals_y0, totals_x1, totals_y0 + 5),
            color=accent_color,
            fill=accent_color,
            width=0,
            overlay=True,
        )
        summary_heading = (
            "RINGKASAN"
            if render_payload["language"] == "id"
            else "SUMMARY"
        )
        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                totals_y0 + 18,
                totals_x0 - 22,
                totals_y0 + 36,
            ),
            text=summary_heading,
            font_name="hebo",
            font_size=7.0,
            minimum_font_size=6.0,
            font_color=accent_color,
        )
        page.draw_line(
            pymupdf.Point(margin_x0, totals_y0 + 42),
            pymupdf.Point(totals_x0 - 22, totals_y0 + 42),
            color=border_color,
            width=0.7,
            overlay=True,
        )
        financial_rows = [
            ("subtotal", 14.0),
            ("tax", 36.0),
            ("discount", 58.0),
            ("total", 84.0),
        ]

        for field_name, row_offset in financial_rows:
            field_payload = render_payload["financials"][field_name]
            is_total = field_name == "total"
            row_y0 = totals_y0 + row_offset

            if is_total:
                page.draw_line(
                    pymupdf.Point(totals_x0 + 12, totals_y0 + 78),
                    pymupdf.Point(totals_x1 - 12, totals_y0 + 78),
                    color=accent_color,
                    width=0.9,
                    overlay=True,
                )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    totals_x0 + 13,
                    row_y0,
                    totals_x0 + 88,
                    row_y0 + 16,
                ),
                text=field_payload["label"],
                font_name=("hebo" if is_total else "helv"),
                font_size=(7.2 if is_total else 6.8),
                minimum_font_size=6.0,
                font_color=(primary_color if is_total else muted_text),
            )
            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    totals_x0 + 90,
                    row_y0,
                    totals_x1 - 12,
                    row_y0 + 17,
                ),
                text=field_payload["display_value"],
                field_name=f"financials.{field_name}",
                font_name=("hebo" if is_total else "helv"),
                font_size=(8.0 if is_total else 7.0),
                minimum_font_size=6.5,
                font_color=(accent_color if is_total else dark_text),
                alignment="right",
            )

        # Footer.
        page.draw_line(
            pymupdf.Point(margin_x0, footer_line_y),
            pymupdf.Point(margin_x1, footer_line_y),
            color=border_color,
            width=0.7,
            overlay=True,
        )
        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(margin_x0, 779, margin_x1, 794),
            text=render_payload["footer"]["document_reference"],
            font_name="helv",
            font_size=6.5,
            minimum_font_size=6.0,
            font_color=muted_text,
            alignment="center",
        )
        notice_display_text = normalize_pdf_notice(
            render_payload["synthetic_notice"]
        )
        add_field(
            page=page,
            rectangle=pymupdf.Rect(margin_x0, 806, margin_x1, 824),
            text=notice_display_text,
            field_name="document.synthetic_notice",
            font_name="hebo",
            font_size=6.7,
            minimum_font_size=6.5,
            font_color=accent_color,
            alignment="center",
        )

        annotation_names = [
            annotation.field_name for annotation in annotations
        ]

        if len(annotation_names) != len(set(annotation_names)):
            raise RuntimeError("Ditemukan nama anotasi duplikat.")

        expected_annotation_count = 19 + 4 * item_count

        if len(annotations) != expected_annotation_count:
            raise RuntimeError(
                "Jumlah anotasi tidak sesuai. "
                f"Expected: {expected_annotation_count}; "
                f"actual: {len(annotations)}."
            )

        if not field_font_audit:
            raise RuntimeError("Audit font field tidak boleh kosong.")

        minimum_used_font_size = min(
            row["used_font_size"] for row in field_font_audit
        )

        if minimum_used_font_size < 6.5:
            raise RuntimeError(
                "Ukuran font field berada di bawah 6.5 pt."
            )

        document.save(
            str(temporary_pdf_path),
            garbage=4,
            deflate=True,
        )

    except Exception:
        temporary_pdf_path.unlink(missing_ok=True)
        raise

    finally:
        document.close()

    if (
        not temporary_pdf_path.is_file()
        or temporary_pdf_path.stat().st_size == 0
    ):
        temporary_pdf_path.unlink(missing_ok=True)
        raise RuntimeError("PDF sementara TPL-07 gagal dibuat.")

    os.replace(temporary_pdf_path, output_pdf_path)

    rendering_metadata = {
        "renderer_version": RENDER_ENGINE_VERSION,
        "template_renderer_version": TPL07_RENDERER_VERSION,
        "template_id": TPL07_TEMPLATE_ID,
        "layout_family": template_specification["layout_family"],
        "page_size": template_specification["page_size"],
        "page_width_points": round(page_width, 3),
        "page_height_points": round(page_height, 3),
        "page_count": 1,
        "annotation_type": "field_region",
        "annotation_count": len(annotations),
        "minimum_field_font_size": min(
            row["used_font_size"] for row in field_font_audit
        ),
        "font_adjustment_count": int(
            sum(bool(row["adjusted"]) for row in field_font_audit)
        ),
        "layout_metrics": {
            "item_count": item_count,
            "table_end_y_points": round(table_end_y, 3),
            "totals_panel_y0_points": round(totals_y0, 3),
            "totals_panel_y1_points": round(totals_y1, 3),
            "table_to_totals_gap_points": round(
                table_to_totals_gap,
                3,
            ),
            "footer_line_y_points": footer_line_y,
        },
    }

    return annotations, rendering_metadata


# ================================================================
# Pilih prototype pertama TPL-07
# ================================================================

tpl07_candidate_ids = sorted(
    canonical_id
    for canonical_id, payload
    in invoice_render_payloads.items()
    if payload.get("template_id")
    == TPL07_TEMPLATE_ID
)

if len(tpl07_candidate_ids) != 20:
    raise RuntimeError(
        "TPL-07 harus memiliki tepat 20 payload. "
        f"Actual: {len(tpl07_candidate_ids)}."
    )

prototype_07_canonical_id = (
    tpl07_candidate_ids[0]
)

prototype_07_render_payload = (
    invoice_render_payloads[
        prototype_07_canonical_id
    ]
)

prototype_07_document_id = (
    prototype_07_render_payload[
        "document_id"
    ]
)

prototype_07_canonical_payload = (
    load_tpl07_canonical_payload(
        prototype_07_canonical_id
    )
)


# ================================================================
# Lokasi artifact
# ================================================================

prototype_07_pdf_path = (
    TPL07_PROTOTYPE_ROOT
    / (
        f"{prototype_07_document_id}"
        "_TPL-07_prototype.pdf"
    )
)

prototype_07_png_path = (
    TPL07_PROTOTYPE_ROOT
    / (
        f"{prototype_07_document_id}"
        "_TPL-07_preview.png"
    )
)

prototype_07_ground_truth_path = (
    TPL07_PROTOTYPE_ROOT
    / (
        f"{prototype_07_document_id}"
        "_TPL-07_ground_truth.json"
    )
)


# ================================================================
# Render prototype
# ================================================================

(
    prototype_07_annotations,
    prototype_07_rendering_metadata,
) = render_template_07(
    render_payload=prototype_07_render_payload,
    output_pdf_path=prototype_07_pdf_path,
)


# ================================================================
# Simpan ground truth
# ================================================================

prototype_07_ground_truth = {
    "schema_version": "1.0.0",
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "document": {
        "document_id": prototype_07_document_id,
        "canonical_invoice_id": (
            prototype_07_canonical_id
        ),
        "split": prototype_07_render_payload[
            "split"
        ],
        "template_id": TPL07_TEMPLATE_ID,
        "language": prototype_07_render_payload[
            "language"
        ],
        "currency": prototype_07_render_payload[
            "currency"
        ],
    },
    "canonical": prototype_07_canonical_payload,
    "rendering": (
        prototype_07_rendering_metadata
    ),
    "annotations": [
        annotation_to_dict(annotation)
        for annotation in prototype_07_annotations
    ],
}

write_json_atomically(
    prototype_07_ground_truth_path,
    prototype_07_ground_truth,
)


# ================================================================
# Buat preview PNG
# ================================================================

temporary_png_path = (
    prototype_07_png_path.with_name(
        f".{prototype_07_png_path.stem}.tmp.png"
    )
)

temporary_png_path.unlink(
    missing_ok=True
)

try:
    with pymupdf.open(
        str(prototype_07_pdf_path)
    ) as prototype_document:
        if prototype_document.page_count != 1:
            raise RuntimeError(
                "Prototype TPL-07 harus satu halaman."
            )

        prototype_page = prototype_document[0]
        extracted_text_07 = prototype_page.get_text(
            "text"
        )

        prototype_pixmap = prototype_page.get_pixmap(
            matrix=pymupdf.Matrix(
                150 / 72,
                150 / 72,
            ),
            alpha=False,
        )

        prototype_pixmap.save(
            str(temporary_png_path)
        )

    if (
        not temporary_png_path.is_file()
        or temporary_png_path.stat().st_size == 0
    ):
        raise RuntimeError(
            "Preview PNG TPL-07 gagal dibuat."
        )

    os.replace(
        temporary_png_path,
        prototype_07_png_path,
    )

except Exception:
    temporary_png_path.unlink(
        missing_ok=True
    )
    raise


# ================================================================
# Pemeriksaan teknis awal
# ================================================================

required_text_fragments = [
    prototype_07_render_payload[
        "metadata"
    ]["invoice_number"]["display_value"],
    prototype_07_render_payload[
        "vendor"
    ]["name"],
    prototype_07_render_payload[
        "buyer"
    ]["name"],
    prototype_07_render_payload[
        "labels"
    ]["quantity"],
    prototype_07_render_payload[
        "financials"
    ]["total"]["display_value"],
    normalize_pdf_notice(
        prototype_07_render_payload[
            "synthetic_notice"
        ]
    ),
]

missing_text_fragments = [
    text
    for text in required_text_fragments
    if text not in extracted_text_07
]

if missing_text_fragments:
    raise RuntimeError(
        "Teks penting TPL-07 hilang dari PDF: "
        f"{missing_text_fragments}"
    )

annotation_names = {
    annotation.field_name
    for annotation in prototype_07_annotations
}

required_annotation_names = {
    "vendor.name",
    "vendor.address_lines",
    "vendor.email",
    "vendor.phone",
    "vendor.tax_identifier",
    "buyer.name",
    "buyer.address_lines",
    "buyer.email",
    "buyer.phone",
    "buyer.tax_identifier",
    "invoice_number",
    "invoice_date",
    "due_date",
    "currency",
    "financials.subtotal",
    "financials.tax",
    "financials.discount",
    "financials.total",
    "document.synthetic_notice",
}

missing_annotations = sorted(
    required_annotation_names
    - annotation_names
)

if missing_annotations:
    raise RuntimeError(
        "Anotasi wajib TPL-07 tidak tersedia: "
        f"{missing_annotations}"
    )

layout_metrics = (
    prototype_07_rendering_metadata[
        "layout_metrics"
    ]
)

prototype_07_summary = pd.DataFrame(
    [
        {
            "control": "pdf_created",
            "actual": prototype_07_pdf_path.is_file(),
            "status": "VALID",
        },
        {
            "control": "ground_truth_created",
            "actual": (
                prototype_07_ground_truth_path.is_file()
            ),
            "status": "VALID",
        },
        {
            "control": "annotation_count",
            "actual": len(
                prototype_07_annotations
            ),
            "status": "VALID",
        },
        {
            "control": "item_count",
            "actual": layout_metrics[
                "item_count"
            ],
            "status": "VALID",
        },
        {
            "control": "minimum_field_font",
            "actual": (
                prototype_07_rendering_metadata[
                    "minimum_field_font_size"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "font_adjustments",
            "actual": (
                prototype_07_rendering_metadata[
                    "font_adjustment_count"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "table_to_totals_gap",
            "actual": layout_metrics[
                "table_to_totals_gap_points"
            ],
            "status": "VALID",
        },
        {
            "control": "missing_required_text",
            "actual": len(
                missing_text_fragments
            ),
            "status": "VALID",
        },
    ]
)

display(prototype_07_summary)

with PILImage.open(
    prototype_07_png_path
) as prototype_image:
    display(prototype_image.copy())


# ================================================================
# Ringkasan
# ================================================================

print(
    f"Canonical ID       : {prototype_07_canonical_id}"
)
print(
    f"Document ID        : {prototype_07_document_id}"
)
print(
    f"Prototype PDF      : {prototype_07_pdf_path}"
)
print(
    f"Prototype preview  : {prototype_07_png_path}"
)
print(
    f"Ground truth       : {prototype_07_ground_truth_path}"
)
print(
    f"Annotations        : {len(prototype_07_annotations)}"
)
print(
    "Minimum field font : "
    f"{prototype_07_rendering_metadata['minimum_field_font_size']}"
)
print(
    "Adjusted fields    : "
    f"{prototype_07_rendering_metadata['font_adjustment_count']}"
)
print(
    "Table-total gap    : "
    f"{layout_metrics['table_to_totals_gap_points']} pt"
)
print(
    f"Renderer version   : {TPL07_RENDERER_VERSION}"
)
print()
print(
    "✅ Prototype TPL-07 berhasil dibuat dan "
    "lolos pemeriksaan teknis awal."
)


In [ ]:
# ================================================================
# CELL 59 — FINAL
# Audit QA teknis prototype TPL-07
# ================================================================

from pathlib import Path
from typing import Any

import hashlib
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd
import pymupdf
from PIL import Image as PILImage


# ================================================================
# Validasi dependency runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "TPL07_PROTOTYPE_ROOT",
    "TPL07_TEMPLATE_ID",
    "prototype_07_canonical_id",
    "prototype_07_document_id",
    "prototype_07_pdf_path",
    "prototype_07_png_path",
    "prototype_07_ground_truth_path",
    "prototype_07_render_payload",
    "prototype_07_rendering_metadata",
    "prototype_07_annotations",
    "annotation_to_dict",
    "write_json_atomically",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 59 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali Cell 58 versi final."
    )


# ================================================================
# Konfigurasi audit
# ================================================================

QA_SCHEMA_VERSION = "1.0.0"
QA_AUDITOR_VERSION = "1.0.0"

MINIMUM_FIELD_FONT_SIZE = 6.5
MAXIMUM_FONT_ADJUSTMENT_RATIO = 0.20
MINIMUM_RASTER_CONTENT_RATIO = 0.01
MAXIMUM_RASTER_CONTENT_RATIO = 0.40
MINIMUM_RASTER_CONTRAST = 10.0
SEVERE_OVERLAP_THRESHOLD = 0.50

prototype_07_pdf_path = Path(
    prototype_07_pdf_path
)
prototype_07_png_path = Path(
    prototype_07_png_path
)
prototype_07_ground_truth_path = Path(
    prototype_07_ground_truth_path
)

prototype_07_qa_report_path = (
    Path(TPL07_PROTOTYPE_ROOT)
    / (
        f"{prototype_07_document_id}"
        "_TPL-07_qa_report.json"
    )
)

required_files = [
    prototype_07_pdf_path,
    prototype_07_png_path,
    prototype_07_ground_truth_path,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Artifact TPL-07 belum lengkap: "
        f"{missing_files}"
    )


# ================================================================
# Helper umum
# ================================================================

def sha256_file(file_path: Path) -> str:
    digest = hashlib.sha256()

    with Path(file_path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def normalize_text(value: Any) -> str:
    text = unicodedata.normalize(
        "NFKC",
        str(value or ""),
    )

    text = (
        text
        .replace("\u2014", "-")
        .replace("\u2013", "-")
        .replace("\u00b7", "-")
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip().casefold()


def normalize_notice_for_pdf(value: Any) -> str:
    return (
        str(value)
        .replace("\u2014", "\u00b7")
        .replace("\u2013", "-")
    )


def first_present(
    mapping: dict[str, Any],
    candidate_keys: tuple[str, ...],
) -> Any:
    for key in candidate_keys:
        if key in mapping:
            return mapping[key]

    return None


def annotation_field_name(
    annotation: dict[str, Any],
) -> str:
    value = first_present(
        annotation,
        (
            "field_name",
            "field_path",
            "field",
            "path",
            "name",
        ),
    )

    if value is None:
        raise KeyError(
            "Nama field tidak ditemukan pada anotasi. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return str(value)


def annotation_text(
    annotation: dict[str, Any],
) -> str:
    value = first_present(
        annotation,
        (
            "text",
            "value",
            "display_value",
            "field_value",
            "raw_value",
            "raw_text",
            "text_value",
            "content",
        ),
    )

    if isinstance(value, dict):
        value = first_present(
            value,
            (
                "text",
                "value",
                "display_value",
                "raw",
            ),
        )

    if value is None:
        raise KeyError(
            "Nilai teks tidak ditemukan pada anotasi. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return str(value)


def bbox_from_value(
    value: Any,
) -> tuple[float, float, float, float] | None:
    if isinstance(value, (list, tuple)):
        if len(value) != 4:
            return None

        try:
            return tuple(
                float(number)
                for number in value
            )
        except (TypeError, ValueError):
            return None

    if not isinstance(value, dict):
        return None

    direct_key_sets = [
        ("x0", "y0", "x1", "y1"),
        ("left", "top", "right", "bottom"),
    ]

    for key_set in direct_key_sets:
        if all(key in value for key in key_set):
            try:
                return tuple(
                    float(value[key])
                    for key in key_set
                )
            except (TypeError, ValueError):
                return None

    if all(
        key in value
        for key in ("x", "y", "width", "height")
    ):
        try:
            x0 = float(value["x"])
            y0 = float(value["y"])
            width = float(value["width"])
            height = float(value["height"])

            return (
                x0,
                y0,
                x0 + width,
                y0 + height,
            )
        except (TypeError, ValueError):
            return None

    if all(
        key in value
        for key in ("x", "y", "w", "h")
    ):
        try:
            x0 = float(value["x"])
            y0 = float(value["y"])
            width = float(value["w"])
            height = float(value["h"])

            return (
                x0,
                y0,
                x0 + width,
                y0 + height,
            )
        except (TypeError, ValueError):
            return None

    return None


def annotation_bbox(
    annotation: dict[str, Any],
) -> tuple[float, float, float, float]:
    candidate = first_present(
        annotation,
        (
            "bbox",
            "bbox_points",
            "bbox_pt",
            "bbox_pdf",
            "bounding_box",
            "rectangle",
            "rect",
            "coordinates",
            "coordinates_points",
        ),
    )

    if candidate is None:
        geometry = annotation.get("geometry")

        if isinstance(geometry, dict):
            candidate = first_present(
                geometry,
                (
                    "bbox",
                    "bounding_box",
                    "rectangle",
                ),
            )

    bbox = bbox_from_value(candidate)

    if bbox is None:
        raise KeyError(
            "Bounding box tidak ditemukan atau tidak valid. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return bbox


def rectangle_area(
    bbox: tuple[float, float, float, float],
) -> float:
    x0, y0, x1, y1 = bbox

    return max(0.0, x1 - x0) * max(
        0.0,
        y1 - y0,
    )


def intersection_area(
    first_bbox: tuple[float, float, float, float],
    second_bbox: tuple[float, float, float, float],
) -> float:
    first_x0, first_y0, first_x1, first_y1 = first_bbox
    second_x0, second_y0, second_x1, second_y1 = second_bbox

    width = max(
        0.0,
        min(first_x1, second_x1)
        - max(first_x0, second_x0),
    )

    height = max(
        0.0,
        min(first_y1, second_y1)
        - max(first_y0, second_y0),
    )

    return width * height


def bbox_intersects_word(
    annotation_box: tuple[float, float, float, float],
    word_box: tuple[float, float, float, float],
) -> bool:
    return intersection_area(
        annotation_box,
        word_box,
    ) > 0.0


# ================================================================
# Muat ground truth
# ================================================================

with prototype_07_ground_truth_path.open(
    "r",
    encoding="utf-8",
) as ground_truth_file:
    ground_truth = json.load(
        ground_truth_file
    )

ground_truth_document = ground_truth.get(
    "document",
    {},
)

if (
    ground_truth_document.get("document_id")
    != prototype_07_document_id
):
    raise RuntimeError(
        "Document ID ground truth tidak sesuai."
    )

if (
    ground_truth_document.get("template_id")
    != TPL07_TEMPLATE_ID
):
    raise RuntimeError(
        "Ground truth bukan milik TPL-07."
    )

annotation_records = ground_truth.get(
    "annotations",
    [],
)

if not isinstance(annotation_records, list):
    raise TypeError(
        "ground_truth.annotations wajib berupa list."
    )

if not annotation_records:
    raise RuntimeError(
        "Ground truth tidak memiliki anotasi."
    )


# ================================================================
# Expected field mapping
# ================================================================

render_payload = prototype_07_render_payload

expected_values: dict[str, str] = {
    "invoice_number": str(
        render_payload["metadata"]
        ["invoice_number"]
        ["display_value"]
    ),
    "invoice_date": str(
        render_payload["metadata"]
        ["invoice_date"]
        ["display_value"]
    ),
    "due_date": str(
        render_payload["metadata"]
        ["due_date"]
        ["display_value"]
    ),
    "currency": str(
        render_payload["currency"]
    ),
}

for party_type in ("vendor", "buyer"):
    party = render_payload[party_type]

    expected_values.update(
        {
            f"{party_type}.name": str(
                party["name"]
            ),
            f"{party_type}.address_lines": "\n".join(
                party["address_lines"]
            ),
            f"{party_type}.email": str(
                party["email"]
            ),
            f"{party_type}.phone": str(
                party["phone"]
            ),
            f"{party_type}.tax_identifier": str(
                party["tax_identifier_display"]
            ),
        }
    )

for item_index, item in enumerate(
    render_payload["items"]
):
    for field_name in (
        "description",
        "quantity",
        "unit_price",
        "line_total",
    ):
        expected_values[
            f"items[{item_index}].{field_name}"
        ] = str(item[field_name])

for field_name in (
    "subtotal",
    "tax",
    "discount",
    "total",
):
    expected_values[
        f"financials.{field_name}"
    ] = str(
        render_payload["financials"]
        [field_name]
        ["display_value"]
    )

expected_values[
    "document.synthetic_notice"
] = normalize_notice_for_pdf(
    render_payload["synthetic_notice"]
)

expected_annotation_count = len(
    expected_values
)


# ================================================================
# Parse annotations
# ================================================================

parsed_annotations: list[dict[str, Any]] = []
annotation_parse_failures: list[dict[str, Any]] = []

for annotation_index, annotation in enumerate(
    annotation_records
):
    try:
        parsed_annotations.append(
            {
                "index": annotation_index,
                "field_name": annotation_field_name(
                    annotation
                ),
                "text": annotation_text(
                    annotation
                ),
                "bbox": annotation_bbox(
                    annotation
                ),
            }
        )
    except (KeyError, TypeError, ValueError) as error:
        annotation_parse_failures.append(
            {
                "index": annotation_index,
                "error": str(error),
                "keys": sorted(
                    annotation.keys()
                ) if isinstance(
                    annotation,
                    dict,
                ) else [],
            }
        )

if annotation_parse_failures:
    raise RuntimeError(
        "Schema anotasi tidak dapat diproses: "
        f"{annotation_parse_failures[:3]}"
    )

annotation_names = [
    annotation["field_name"]
    for annotation in parsed_annotations
]

annotation_by_name = {
    annotation["field_name"]: annotation
    for annotation in parsed_annotations
}

duplicate_annotation_count = (
    len(annotation_names)
    - len(set(annotation_names))
)

missing_required_annotations = sorted(
    set(expected_values)
    - set(annotation_names)
)


# ================================================================
# Audit PDF dan anotasi
# ================================================================

with pymupdf.open(
    str(prototype_07_pdf_path)
) as pdf_document:
    pdf_page_count = pdf_document.page_count

    if pdf_page_count < 1:
        raise RuntimeError(
            "PDF TPL-07 tidak memiliki halaman."
        )

    first_page = pdf_document[0]
    page_rectangle = first_page.rect
    extracted_text = first_page.get_text("text")
    extracted_words = first_page.get_text("words")

page_width = float(page_rectangle.width)
page_height = float(page_rectangle.height)

word_boxes = [
    (
        float(word[0]),
        float(word[1]),
        float(word[2]),
        float(word[3]),
    )
    for word in extracted_words
    if len(word) >= 5
    and str(word[4]).strip()
]

invalid_bounding_boxes: list[str] = []
annotations_without_words: list[str] = []

for annotation in parsed_annotations:
    field_name = annotation["field_name"]
    x0, y0, x1, y1 = annotation["bbox"]

    coordinates_are_finite = all(
        math.isfinite(value)
        for value in (x0, y0, x1, y1)
    )

    bbox_is_valid = (
        coordinates_are_finite
        and x0 >= 0.0
        and y0 >= 0.0
        and x1 <= page_width
        and y1 <= page_height
        and x1 > x0
        and y1 > y0
    )

    if not bbox_is_valid:
        invalid_bounding_boxes.append(
            field_name
        )
        continue

    contains_word = any(
        bbox_intersects_word(
            annotation["bbox"],
            word_box,
        )
        for word_box in word_boxes
    )

    if not contains_word:
        annotations_without_words.append(
            field_name
        )


# ================================================================
# Audit nilai anotasi
# ================================================================

normalization_mismatches: list[dict[str, str]] = []

for field_name, expected_value in expected_values.items():
    annotation = annotation_by_name.get(
        field_name
    )

    if annotation is None:
        continue

    actual_value = annotation["text"]

    if normalize_text(actual_value) != normalize_text(
        expected_value
    ):
        normalization_mismatches.append(
            {
                "field_name": field_name,
                "expected": expected_value,
                "actual": actual_value,
            }
        )

normalized_extracted_text = normalize_text(
    extracted_text
)

missing_extracted_values: list[str] = []

for annotation in parsed_annotations:
    normalized_value = normalize_text(
        annotation["text"]
    )

    if (
        normalized_value
        and normalized_value
        not in normalized_extracted_text
    ):
        missing_extracted_values.append(
            annotation["field_name"]
        )


# ================================================================
# Audit overlap anotasi
# ================================================================

severe_annotation_overlaps: list[dict[str, Any]] = []

for first_index in range(
    len(parsed_annotations)
):
    first_annotation = parsed_annotations[
        first_index
    ]
    first_area = rectangle_area(
        first_annotation["bbox"]
    )

    if first_area <= 0.0:
        continue

    for second_index in range(
        first_index + 1,
        len(parsed_annotations),
    ):
        second_annotation = parsed_annotations[
            second_index
        ]
        second_area = rectangle_area(
            second_annotation["bbox"]
        )

        if second_area <= 0.0:
            continue

        overlap_area = intersection_area(
            first_annotation["bbox"],
            second_annotation["bbox"],
        )

        overlap_ratio = overlap_area / min(
            first_area,
            second_area,
        )

        if overlap_ratio >= SEVERE_OVERLAP_THRESHOLD:
            severe_annotation_overlaps.append(
                {
                    "first": first_annotation[
                        "field_name"
                    ],
                    "second": second_annotation[
                        "field_name"
                    ],
                    "overlap_ratio": round(
                        overlap_ratio,
                        6,
                    ),
                }
            )


# ================================================================
# Audit font dan raster
# ================================================================

minimum_field_font_size = float(
    prototype_07_rendering_metadata[
        "minimum_field_font_size"
    ]
)

font_adjustment_count = int(
    prototype_07_rendering_metadata[
        "font_adjustment_count"
    ]
)

font_adjustment_ratio = (
    font_adjustment_count
    / max(1, len(parsed_annotations))
)

with PILImage.open(
    prototype_07_png_path
) as preview_image:
    grayscale_array = np.asarray(
        preview_image.convert("L"),
        dtype=np.float32,
    ).copy()

if grayscale_array.size == 0:
    raise RuntimeError(
        "Preview PNG tidak memiliki piksel."
    )

raster_content_ratio = float(
    np.mean(grayscale_array < 245.0)
)

raster_contrast = float(
    np.std(grayscale_array)
)


# ================================================================
# Susun hasil pemeriksaan
# ================================================================

checks = [
    {
        "control": "pdf_page_count",
        "expected": 1,
        "actual": pdf_page_count,
        "valid": pdf_page_count == 1,
    },
    {
        "control": "annotation_count",
        "expected": expected_annotation_count,
        "actual": len(parsed_annotations),
        "valid": (
            len(parsed_annotations)
            == expected_annotation_count
        ),
    },
    {
        "control": "duplicate_annotations",
        "expected": 0,
        "actual": duplicate_annotation_count,
        "valid": duplicate_annotation_count == 0,
    },
    {
        "control": "missing_required_annotations",
        "expected": 0,
        "actual": len(
            missing_required_annotations
        ),
        "valid": not missing_required_annotations,
    },
    {
        "control": "invalid_bounding_boxes",
        "expected": 0,
        "actual": len(
            invalid_bounding_boxes
        ),
        "valid": not invalid_bounding_boxes,
    },
    {
        "control": "normalization_mismatches",
        "expected": 0,
        "actual": len(
            normalization_mismatches
        ),
        "valid": not normalization_mismatches,
    },
    {
        "control": "annotations_without_words",
        "expected": 0,
        "actual": len(
            annotations_without_words
        ),
        "valid": not annotations_without_words,
    },
    {
        "control": "severe_annotation_overlaps",
        "expected": 0,
        "actual": len(
            severe_annotation_overlaps
        ),
        "valid": not severe_annotation_overlaps,
    },
    {
        "control": "missing_extracted_values",
        "expected": 0,
        "actual": len(
            missing_extracted_values
        ),
        "valid": not missing_extracted_values,
    },
    {
        "control": "minimum_field_font_size",
        "expected": ">=6.5",
        "actual": round(
            minimum_field_font_size,
            4,
        ),
        "valid": (
            minimum_field_font_size
            >= MINIMUM_FIELD_FONT_SIZE
        ),
    },
    {
        "control": "font_adjustment_ratio",
        "expected": "<=0.20",
        "actual": round(
            font_adjustment_ratio,
            4,
        ),
        "valid": (
            font_adjustment_ratio
            <= MAXIMUM_FONT_ADJUSTMENT_RATIO
        ),
    },
    {
        "control": "raster_content_ratio",
        "expected": "0.01–0.40",
        "actual": round(
            raster_content_ratio,
            4,
        ),
        "valid": (
            MINIMUM_RASTER_CONTENT_RATIO
            <= raster_content_ratio
            <= MAXIMUM_RASTER_CONTENT_RATIO
        ),
    },
    {
        "control": "raster_contrast",
        "expected": ">=10.0",
        "actual": round(
            raster_contrast,
            4,
        ),
        "valid": (
            raster_contrast
            >= MINIMUM_RASTER_CONTRAST
        ),
    },
]

for check in checks:
    check["status"] = (
        "VALID"
        if check["valid"]
        else "INVALID"
    )

failed_checks = [
    check
    for check in checks
    if not check["valid"]
]

qa_status = (
    "PASSED"
    if not failed_checks
    else "FAILED"
)


# ================================================================
# Simpan QA report
# ================================================================

qa_report = {
    "schema_version": QA_SCHEMA_VERSION,
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "document_id": prototype_07_document_id,
    "canonical_invoice_id": (
        prototype_07_canonical_id
    ),
    "template_id": TPL07_TEMPLATE_ID,
    "status": qa_status,
    "manual_visual_review": "PENDING",
    "metrics": {
        "auditor_version": QA_AUDITOR_VERSION,
        "pdf_page_count": pdf_page_count,
        "annotation_count": len(
            parsed_annotations
        ),
        "expected_annotation_count": (
            expected_annotation_count
        ),
        "duplicate_annotation_count": (
            duplicate_annotation_count
        ),
        "missing_required_annotation_count": len(
            missing_required_annotations
        ),
        "invalid_bounding_box_count": len(
            invalid_bounding_boxes
        ),
        "normalization_mismatch_count": len(
            normalization_mismatches
        ),
        "annotations_without_words_count": len(
            annotations_without_words
        ),
        "severe_annotation_overlap_count": len(
            severe_annotation_overlaps
        ),
        "missing_extracted_value_count": len(
            missing_extracted_values
        ),
        "minimum_field_font_size": round(
            minimum_field_font_size,
            6,
        ),
        "font_adjustment_count": (
            font_adjustment_count
        ),
        "font_adjustment_ratio": round(
            font_adjustment_ratio,
            6,
        ),
        "raster_content_ratio": round(
            raster_content_ratio,
            6,
        ),
        "raster_contrast": round(
            raster_contrast,
            6,
        ),
    },
    "failures": {
        "failed_checks": [
            check["control"]
            for check in failed_checks
        ],
        "missing_required_annotations": (
            missing_required_annotations
        ),
        "invalid_bounding_boxes": (
            invalid_bounding_boxes
        ),
        "normalization_mismatches": (
            normalization_mismatches
        ),
        "annotations_without_words": (
            annotations_without_words
        ),
        "severe_annotation_overlaps": (
            severe_annotation_overlaps
        ),
        "missing_extracted_values": (
            missing_extracted_values
        ),
    },
    "checksums_sha256": {
        "prototype_pdf": sha256_file(
            prototype_07_pdf_path
        ),
        "preview": sha256_file(
            prototype_07_png_path
        ),
        "ground_truth": sha256_file(
            prototype_07_ground_truth_path
        ),
    },
}

write_json_atomically(
    prototype_07_qa_report_path,
    qa_report,
)


# ================================================================
# Verifikasi round-trip QA report
# ================================================================

with prototype_07_qa_report_path.open(
    "r",
    encoding="utf-8",
) as qa_report_file:
    verified_qa_report = json.load(
        qa_report_file
    )

if verified_qa_report != qa_report:
    raise RuntimeError(
        "QA report berubah setelah disimpan dan dibuka ulang."
    )


# ================================================================
# Output audit
# ================================================================

qa_summary = pd.DataFrame(
    [
        {
            "control": check["control"],
            "expected": check["expected"],
            "actual": check["actual"],
            "status": check["status"],
        }
        for check in checks
    ]
)

display(qa_summary)

print(
    f"QA report          : "
    f"{prototype_07_qa_report_path}"
)
print(
    f"QA status          : {qa_status}"
)
print(
    "Minimum field font : "
    f"{minimum_field_font_size:.2f}"
)
print(
    "Font adjustments   : "
    f"{font_adjustment_count}"
)
print(
    "Ink pixel ratio    : "
    f"{raster_content_ratio:.6f}"
)
print(
    "Raster contrast    : "
    f"{raster_contrast:.6f}"
)
print(
    "Manual review      : "
    f"{qa_report['manual_visual_review']}"
)
print()

if qa_status != "PASSED":
    print("Detail kegagalan:")
    print(
        json.dumps(
            qa_report["failures"],
            ensure_ascii=False,
            indent=2,
        )
    )

    raise RuntimeError(
        "Prototype TPL-07 gagal audit QA teknis."
    )

print(
    "✅ Prototype TPL-07 lulus audit teknis. "
    "Review visual manual masih perlu dicatat "
    "ke QA report."
)


In [ ]:
# ================================================================
# CELL 59A — FINAL
# Finalisasi review visual manual prototype TPL-07
# ================================================================

from pathlib import Path

import hashlib
import json


# ================================================================
# Validasi dependency runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "prototype_07_qa_report_path",
    "prototype_07_png_path",
    "prototype_07_document_id",
    "write_json_atomically",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 59A belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali Cell 58 dan Cell 59."
    )


# ================================================================
# Helper checksum
# ================================================================

def calculate_sha256(file_path: Path) -> str:
    """Menghitung SHA-256 file tanpa memuat seluruh file ke memori."""

    digest = hashlib.sha256()

    with file_path.open("rb") as input_file:
        for chunk in iter(
            lambda: input_file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ================================================================
# Muat artifact yang akan difinalisasi
# ================================================================

qa_report_path = Path(
    prototype_07_qa_report_path
)

preview_path = Path(
    prototype_07_png_path
)

if not qa_report_path.is_file():
    raise FileNotFoundError(
        "QA report TPL-07 tidak ditemukan: "
        f"{qa_report_path}"
    )

if not preview_path.is_file():
    raise FileNotFoundError(
        "Preview TPL-07 tidak ditemukan: "
        f"{preview_path}"
    )

with qa_report_path.open(
    mode="r",
    encoding="utf-8",
) as qa_report_file:
    qa_report = json.load(qa_report_file)


# ================================================================
# Validasi identitas dan audit teknis
# ================================================================

if qa_report.get("template_id") != "TPL-07":
    raise RuntimeError(
        "QA report bukan milik TPL-07."
    )

if (
    qa_report.get("document_id")
    != prototype_07_document_id
):
    raise RuntimeError(
        "Document ID QA report tidak sesuai. "
        f"Expected: {prototype_07_document_id!r}; "
        f"actual: {qa_report.get('document_id')!r}."
    )

if qa_report.get("status") != "PASSED":
    raise RuntimeError(
        "Review visual tidak boleh difinalisasi karena "
        "audit teknis belum PASSED. "
        f"Status saat ini: {qa_report.get('status')!r}."
    )

failure_details = qa_report.get(
    "failures",
    {},
)

if not isinstance(failure_details, dict):
    raise TypeError(
        "Field 'failures' pada QA report wajib "
        "berupa dictionary."
    )

nonempty_failures = {
    failure_name: failure_value
    for failure_name, failure_value
    in failure_details.items()
    if bool(failure_value)
}

if nonempty_failures:
    raise RuntimeError(
        "QA report masih memiliki kegagalan teknis: "
        f"{nonempty_failures}"
    )

if "manual_visual_review" not in qa_report:
    raise KeyError(
        "Field 'manual_visual_review' tidak ditemukan "
        "dalam QA report."
    )


# ================================================================
# Pastikan preview yang direview sama dengan preview yang diaudit
# ================================================================

recorded_preview_checksum = (
    qa_report
    .get("checksums_sha256", {})
    .get("preview")
)

if not recorded_preview_checksum:
    raise KeyError(
        "Checksum preview tidak ditemukan pada "
        "qa_report['checksums_sha256']['preview']."
    )

actual_preview_checksum = calculate_sha256(
    preview_path
)

if actual_preview_checksum != recorded_preview_checksum:
    raise RuntimeError(
        "Preview berubah setelah audit teknis. "
        "Jalankan ulang Cell 59 dan lakukan review visual kembali. "
        f"Expected SHA-256: {recorded_preview_checksum}; "
        f"actual: {actual_preview_checksum}."
    )


# ================================================================
# Catat hasil review visual secara idempoten
# ================================================================

qa_report["manual_visual_review"] = "PASSED"

write_json_atomically(
    qa_report_path,
    qa_report,
)


# ================================================================
# Buka ulang dan verifikasi hasil persistensi
# ================================================================

with qa_report_path.open(
    mode="r",
    encoding="utf-8",
) as qa_report_file:
    persisted_qa_report = json.load(
        qa_report_file
    )

if persisted_qa_report.get("status") != "PASSED":
    raise RuntimeError(
        "Status audit teknis berubah setelah finalisasi."
    )

if (
    persisted_qa_report.get("manual_visual_review")
    != "PASSED"
):
    raise RuntimeError(
        "Status review visual gagal disimpan."
    )

persisted_preview_checksum = (
    persisted_qa_report
    .get("checksums_sha256", {})
    .get("preview")
)

if persisted_preview_checksum != actual_preview_checksum:
    raise RuntimeError(
        "Checksum preview berubah pada QA report."
    )


# ================================================================
# Ringkasan final
# ================================================================

print(
    f"QA report           : {qa_report_path}"
)

print(
    "Technical status    : "
    f"{persisted_qa_report['status']}"
)

print(
    "Manual visual review: "
    f"{persisted_qa_report['manual_visual_review']}"
)

print(
    "Preview SHA-256     : "
    f"{persisted_preview_checksum}"
)

print()

print(
    "✅ TPL-07 FINAL PASSED — audit teknis dan "
    "review visual telah selesai."
)


In [ ]:
# ================================================================
# CELL 60 — FINAL
# Inspeksi spesifikasi dan presentation payload TPL-08
# Read-only: tidak membuat atau mengubah artifact build
# ================================================================

from collections import Counter
from typing import Any

import json

import pandas as pd
from IPython.display import display


# ================================================================
# Validasi dependency runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "template_registry",
    "invoice_render_payloads",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 60 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali cell pembentukan template registry "
        "dan presentation payload."
    )

if not isinstance(template_registry, dict):
    raise TypeError(
        "template_registry wajib berupa dictionary."
    )

if not isinstance(invoice_render_payloads, dict):
    raise TypeError(
        "invoice_render_payloads wajib berupa dictionary."
    )


# ================================================================
# Konfigurasi inspeksi
# ================================================================

TPL08_TEMPLATE_ID = "TPL-08"
TPL08_EXPECTED_PAYLOAD_COUNT = 20

if TPL08_TEMPLATE_ID not in template_registry:
    raise KeyError(
        "Spesifikasi TPL-08 tidak ditemukan dalam "
        "template_registry."
    )

tpl08_template_specification = template_registry[
    TPL08_TEMPLATE_ID
]

if not isinstance(tpl08_template_specification, dict):
    raise TypeError(
        "Spesifikasi TPL-08 wajib berupa dictionary."
    )

tpl08_payloads = {
    canonical_id: payload
    for canonical_id, payload
    in invoice_render_payloads.items()
    if isinstance(payload, dict)
    and payload.get("template_id") == TPL08_TEMPLATE_ID
}

if len(tpl08_payloads) != TPL08_EXPECTED_PAYLOAD_COUNT:
    raise RuntimeError(
        "TPL-08 harus memiliki tepat "
        f"{TPL08_EXPECTED_PAYLOAD_COUNT} payload. "
        f"Actual: {len(tpl08_payloads)}."
    )


# ================================================================
# Kontrak struktur presentation payload
# ================================================================

REQUIRED_TOP_LEVEL_KEYS = {
    "buyer",
    "canonical_invoice_id",
    "currency",
    "document_id",
    "financials",
    "footer",
    "items",
    "labels",
    "language",
    "metadata",
    "split",
    "synthetic_notice",
    "template_id",
    "title",
    "vendor",
}

REQUIRED_LABEL_KEYS = {
    "buyer",
    "description",
    "discount",
    "document_title",
    "due_date",
    "invoice_date",
    "invoice_number",
    "line_total",
    "page",
    "quantity",
    "subtotal",
    "synthetic_notice",
    "tax",
    "tax_identifier",
    "total",
    "unit_price",
    "vendor",
}

REQUIRED_METADATA_KEYS = {
    "due_date",
    "invoice_date",
    "invoice_number",
}

REQUIRED_PARTY_KEYS = {
    "address_lines",
    "email",
    "name",
    "party_id",
    "phone",
    "tax_identifier",
    "tax_identifier_display",
}

REQUIRED_ITEM_KEYS = {
    "description",
    "item_number",
    "line_total",
    "quantity",
    "raw",
    "unit_price",
}

REQUIRED_FINANCIAL_KEYS = {
    "discount",
    "subtotal",
    "tax",
    "total",
}

REQUIRED_FOOTER_KEYS = {
    "document_reference",
    "synthetic_notice",
}


def require_dictionary(
    value: Any,
    field_path: str,
) -> dict[str, Any]:
    """Memastikan sebuah field merupakan dictionary."""

    if not isinstance(value, dict):
        raise TypeError(
            f"{field_path} wajib berupa dictionary."
        )

    return value


def require_key_set(
    value: dict[str, Any],
    required_keys: set[str],
    field_path: str,
) -> None:
    """Memastikan seluruh key wajib tersedia."""

    missing_keys = sorted(
        required_keys - set(value)
    )

    if missing_keys:
        raise KeyError(
            f"{field_path} kehilangan key wajib: "
            f"{missing_keys}"
        )


expected_split = tpl08_template_specification.get(
    "split"
)

document_ids: list[str] = []

for canonical_id in sorted(tpl08_payloads):
    payload = tpl08_payloads[canonical_id]

    require_key_set(
        payload,
        REQUIRED_TOP_LEVEL_KEYS,
        f"payload[{canonical_id!r}]",
    )

    if payload["canonical_invoice_id"] != canonical_id:
        raise RuntimeError(
            "Canonical ID pada dictionary dan payload "
            f"tidak sama: {canonical_id!r}."
        )

    if payload["template_id"] != TPL08_TEMPLATE_ID:
        raise RuntimeError(
            f"Template ID tidak sesuai pada {canonical_id}."
        )

    if expected_split is not None and (
        payload["split"] != expected_split
    ):
        raise RuntimeError(
            f"Split payload {canonical_id} tidak sesuai. "
            f"Expected: {expected_split!r}; "
            f"actual: {payload['split']!r}."
        )

    if payload["language"] not in {"id", "en"}:
        raise RuntimeError(
            f"Language tidak didukung pada {canonical_id}: "
            f"{payload['language']!r}."
        )

    labels = require_dictionary(
        payload["labels"],
        f"payload[{canonical_id!r}].labels",
    )

    metadata = require_dictionary(
        payload["metadata"],
        f"payload[{canonical_id!r}].metadata",
    )

    vendor = require_dictionary(
        payload["vendor"],
        f"payload[{canonical_id!r}].vendor",
    )

    buyer = require_dictionary(
        payload["buyer"],
        f"payload[{canonical_id!r}].buyer",
    )

    financials = require_dictionary(
        payload["financials"],
        f"payload[{canonical_id!r}].financials",
    )

    footer = require_dictionary(
        payload["footer"],
        f"payload[{canonical_id!r}].footer",
    )

    require_key_set(
        labels,
        REQUIRED_LABEL_KEYS,
        f"payload[{canonical_id!r}].labels",
    )

    require_key_set(
        metadata,
        REQUIRED_METADATA_KEYS,
        f"payload[{canonical_id!r}].metadata",
    )

    require_key_set(
        vendor,
        REQUIRED_PARTY_KEYS,
        f"payload[{canonical_id!r}].vendor",
    )

    require_key_set(
        buyer,
        REQUIRED_PARTY_KEYS,
        f"payload[{canonical_id!r}].buyer",
    )

    require_key_set(
        financials,
        REQUIRED_FINANCIAL_KEYS,
        f"payload[{canonical_id!r}].financials",
    )

    require_key_set(
        footer,
        REQUIRED_FOOTER_KEYS,
        f"payload[{canonical_id!r}].footer",
    )

    items = payload["items"]

    if not isinstance(items, list) or not items:
        raise RuntimeError(
            f"payload[{canonical_id!r}].items wajib "
            "berupa list yang tidak kosong."
        )

    for item_index, item in enumerate(items):
        item = require_dictionary(
            item,
            (
                f"payload[{canonical_id!r}]"
                f".items[{item_index}]"
            ),
        )

        require_key_set(
            item,
            REQUIRED_ITEM_KEYS,
            (
                f"payload[{canonical_id!r}]"
                f".items[{item_index}]"
            ),
        )

    document_ids.append(
        str(payload["document_id"])
    )

if len(document_ids) != len(set(document_ids)):
    raise RuntimeError(
        "Ditemukan document_id duplikat pada payload TPL-08."
    )


# ================================================================
# Helper statistik panjang teks
# ================================================================

def joined_text_length(values: Any) -> int:
    """Mengukur panjang gabungan list teks secara konsisten."""

    if not isinstance(values, list):
        return len(str(values))

    return len(
        " ".join(str(value) for value in values)
    )


def maximum_money_length(
    payload: dict[str, Any],
) -> int:
    """Mengukur nilai uang terpanjang yang perlu dirender."""

    money_values: list[str] = []

    for item in payload["items"]:
        money_values.extend(
            [
                str(item["unit_price"]),
                str(item["line_total"]),
            ]
        )

    for field_name in (
        "subtotal",
        "tax",
        "discount",
        "total",
    ):
        financial_value = payload[
            "financials"
        ][field_name]

        if isinstance(financial_value, dict):
            financial_value = financial_value.get(
                "display_value",
                financial_value.get("value", ""),
            )

        money_values.append(
            str(financial_value)
        )

    return max(
        map(len, money_values),
        default=0,
    )


# ================================================================
# Bangun ringkasan inspeksi
# ================================================================

tpl08_candidate_ids = sorted(
    tpl08_payloads
)

tpl08_sample_id = tpl08_candidate_ids[0]
tpl08_sample_payload = tpl08_payloads[
    tpl08_sample_id
]

tpl08_sample_structure = {
    "canonical_invoice_id": (
        tpl08_sample_payload["canonical_invoice_id"]
    ),
    "document_id": tpl08_sample_payload[
        "document_id"
    ],
    "template_id": tpl08_sample_payload[
        "template_id"
    ],
    "language": tpl08_sample_payload["language"],
    "currency": tpl08_sample_payload["currency"],
    "title": tpl08_sample_payload["title"],
    "top_level_keys": sorted(
        tpl08_sample_payload
    ),
    "label_keys": sorted(
        tpl08_sample_payload["labels"]
    ),
    "metadata_keys": sorted(
        tpl08_sample_payload["metadata"]
    ),
    "vendor_keys": sorted(
        tpl08_sample_payload["vendor"]
    ),
    "buyer_keys": sorted(
        tpl08_sample_payload["buyer"]
    ),
    "item_keys": sorted(
        tpl08_sample_payload["items"][0]
    ),
    "financial_keys": sorted(
        tpl08_sample_payload["financials"]
    ),
    "footer_keys": sorted(
        tpl08_sample_payload["footer"]
    ),
}

statistics_rows: list[dict[str, Any]] = []

for canonical_id in tpl08_candidate_ids:
    payload = tpl08_payloads[canonical_id]
    descriptions = [
        str(item["description"])
        for item in payload["items"]
    ]

    statistics_rows.append(
        {
            "canonical_id": canonical_id,
            "document_id": payload["document_id"],
            "language": payload["language"],
            "currency": payload["currency"],
            "item_count": len(payload["items"]),
            "vendor_name_length": len(
                str(payload["vendor"]["name"])
            ),
            "vendor_address_length": joined_text_length(
                payload["vendor"]["address_lines"]
            ),
            "vendor_email_length": len(
                str(payload["vendor"]["email"])
            ),
            "buyer_name_length": len(
                str(payload["buyer"]["name"])
            ),
            "buyer_address_length": joined_text_length(
                payload["buyer"]["address_lines"]
            ),
            "buyer_email_length": len(
                str(payload["buyer"]["email"])
            ),
            "maximum_description_length": max(
                map(len, descriptions),
                default=0,
            ),
            "maximum_money_length": maximum_money_length(
                payload
            ),
        }
    )

tpl08_statistics = pd.DataFrame(
    statistics_rows
)

tpl08_payload_statistics = (
    tpl08_statistics
    .describe(include="all")
    .transpose()
)

tpl08_item_count_distribution = (
    tpl08_statistics["item_count"]
    .value_counts()
    .sort_index()
    .rename_axis("item_count")
    .reset_index(name="document_count")
)

tpl08_language_distribution = (
    tpl08_statistics["language"]
    .value_counts()
    .sort_index()
    .rename_axis("language")
    .reset_index(name="document_count")
)

tpl08_currency_distribution = (
    tpl08_statistics["currency"]
    .value_counts()
    .sort_index()
    .rename_axis("currency")
    .reset_index(name="document_count")
)


# ================================================================
# Output inspeksi
# ================================================================

print("TEMPLATE SPECIFICATION")
print(
    json.dumps(
        tpl08_template_specification,
        ensure_ascii=False,
        indent=2,
    )
)

print()
print("PAYLOAD COUNT")
print(len(tpl08_payloads))

print()
print("SAMPLE STRUCTURE")
print(
    json.dumps(
        tpl08_sample_structure,
        ensure_ascii=False,
        indent=2,
    )
)

print()
print("LABELS")
print(
    json.dumps(
        tpl08_sample_payload["labels"],
        ensure_ascii=False,
        indent=2,
    )
)

print()
print("PAYLOAD STATISTICS")
display(tpl08_payload_statistics)

print()
print("ITEM-COUNT DISTRIBUTION")
display(tpl08_item_count_distribution)

print()
print("LANGUAGE DISTRIBUTION")
display(tpl08_language_distribution)

print()
print("CURRENCY DISTRIBUTION")
display(tpl08_currency_distribution)

print(
    "✅ Inspeksi TPL-08 selesai. "
    "Belum ada artifact yang dibuat atau diubah."
)


In [ ]:
# ================================================================
# CELL 61 — Prototype renderer TPL-08: Modular Cards
# ================================================================

from pathlib import Path
from typing import Any
from PIL import Image as PILImage

import json
import os

import pandas as pd
import pymupdf


# ================================================================
# Validasi runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "PROTOTYPE_ROOT",
    "CANONICAL_RECORDS_JSONL_PATH",
    "FieldAnnotation",
    "template_registry",
    "invoice_render_payloads",
    "get_page_dimensions",
    "hex_to_rgb",
    "insert_annotated_text",
    "insert_static_text",
    "annotation_to_dict",
    "write_json_atomically",
    "RENDER_ENGINE_VERSION",
]

missing_runtime_objects = [
    name
    for name in REQUIRED_RUNTIME_OBJECTS
    if name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Runtime TPL-08 belum lengkap: "
        f"{missing_runtime_objects}"
    )


# ================================================================
# Konfigurasi
# ================================================================

TPL08_TEMPLATE_ID = "TPL-08"
TPL08_RENDERER_VERSION = "1.0.0"
TPL08_MIN_ITEMS_PER_PAGE = 2
TPL08_MAX_ITEMS_PER_PAGE = 8

TPL08_PROTOTYPE_ROOT = (
    Path(PROTOTYPE_ROOT)
    / TPL08_TEMPLATE_ID
)

TPL08_PROTOTYPE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ================================================================
# Helper canonical payload
# ================================================================

def load_tpl08_canonical_payload(
    canonical_id: str,
) -> dict[str, Any]:

    canonical_path = Path(
        CANONICAL_RECORDS_JSONL_PATH
    )

    if not canonical_path.is_file():
        raise FileNotFoundError(
            f"Canonical JSONL tidak ditemukan: {canonical_path}"
        )

    with canonical_path.open(
        "r",
        encoding="utf-8",
    ) as canonical_file:

        for line_number, line in enumerate(
            canonical_file,
            start=1,
        ):
            if not line.strip():
                continue

            try:
                payload = json.loads(line)
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    "Canonical JSONL rusak pada "
                    f"baris {line_number}."
                ) from error

            if (
                payload.get("canonical_invoice_id")
                == canonical_id
            ):
                return payload

    raise KeyError(
        f"Canonical payload tidak ditemukan: {canonical_id}"
    )


def normalize_pdf_notice(text: Any) -> str:
    """Mengganti separator yang tidak didukung font PDF bawaan."""

    return (
        str(text)
        .replace("—", "·")
        .replace("–", "-")
    )


# ================================================================
# Renderer final TPL-08 — Modular Cards
# ================================================================

def render_template_08(
    render_payload: dict[str, Any],
    output_pdf_path: Path,
) -> tuple[list[FieldAnnotation], dict[str, Any]]:
    """Merender satu halaman TPL-08 dengan layout modular cards."""

    if not isinstance(render_payload, dict):
        raise TypeError("render_payload wajib berupa dictionary.")

    if render_payload.get("template_id") != TPL08_TEMPLATE_ID:
        raise ValueError(
            "Renderer TPL-08 hanya menerima payload TPL-08."
        )

    required_keys = {
        "canonical_invoice_id", "document_id", "template_id",
        "language", "currency", "title", "labels", "metadata",
        "vendor", "buyer", "items", "financials", "footer",
        "synthetic_notice",
    }
    missing_keys = sorted(required_keys - set(render_payload))

    if missing_keys:
        raise KeyError(
            f"Payload TPL-08 belum lengkap: {missing_keys}"
        )

    item_count = len(render_payload["items"])

    if not (
        TPL08_MIN_ITEMS_PER_PAGE
        <= item_count
        <= TPL08_MAX_ITEMS_PER_PAGE
    ):
        raise ValueError(
            "TPL-08 hanya mendukung 2–8 item per halaman."
        )

    if (
        render_payload["language"] == "id"
        and render_payload["labels"]["quantity"] != "Kuantitas"
    ):
        raise RuntimeError(
            "Label quantity bahasa Indonesia wajib 'Kuantitas'."
        )

    template_specification = template_registry[TPL08_TEMPLATE_ID]
    expected_template_contract = {
        "layout_family": "modular_cards",
        "page_size": "LETTER",
        "header_layout": "metadata_cards",
        "party_layout": "horizontal_cards",
        "table_style": "boxed_rows",
        "totals_position": "side_panel",
        "accent_position": "header",
        "density": "spacious",
    }
    contract_mismatches = {
        key: {
            "expected": expected_value,
            "actual": template_specification.get(key),
        }
        for key, expected_value in expected_template_contract.items()
        if template_specification.get(key) != expected_value
    }

    if contract_mismatches:
        raise RuntimeError(
            "Spesifikasi layout TPL-08 tidak sesuai: "
            f"{contract_mismatches}"
        )

    page_width, page_height = get_page_dimensions(
        template_specification["page_size"]
    )
    primary_color = hex_to_rgb(
        template_specification["primary_color"]
    )
    accent_color = hex_to_rgb(
        template_specification["accent_color"]
    )
    dark_text = (0.12, 0.16, 0.21)
    muted_text = (0.40, 0.45, 0.49)
    border_color = (0.78, 0.83, 0.86)
    light_primary = (0.945, 0.970, 0.977)
    light_accent = (0.995, 0.955, 0.905)
    white = (1.0, 1.0, 1.0)

    output_pdf_path = Path(output_pdf_path)
    output_pdf_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_pdf_path = output_pdf_path.with_name(
        f".{output_pdf_path.stem}.tmp.pdf"
    )
    temporary_pdf_path.unlink(missing_ok=True)

    annotations: list[FieldAnnotation] = []
    field_font_audit: list[dict[str, Any]] = []
    document = pymupdf.open()

    def add_field(
        page: pymupdf.Page,
        rectangle: pymupdf.Rect,
        text: str,
        field_name: str,
        font_name: str = "helv",
        font_size: float = 7.0,
        minimum_font_size: float = 6.5,
        font_color: tuple[float, float, float] = dark_text,
        alignment: str = "left",
    ) -> None:
        annotation, used_font_size = insert_annotated_text(
            page=page,
            rectangle=rectangle,
            text=str(text),
            field_name=field_name,
            font_name=font_name,
            font_size=font_size,
            minimum_font_size=minimum_font_size,
            font_color=font_color,
            alignment=alignment,
        )
        annotations.append(annotation)
        field_font_audit.append(
            {
                "field_name": field_name,
                "requested_font_size": font_size,
                "used_font_size": used_font_size,
                "adjusted": used_font_size < font_size,
            }
        )

    try:
        page = document.new_page(
            width=page_width,
            height=page_height,
        )
        margin_x0 = 45.0
        margin_x1 = page_width - 45.0

        # Aksen header.
        page.draw_rect(
            pymupdf.Rect(0, 0, page_width, 11),
            color=accent_color,
            fill=accent_color,
            width=0,
            overlay=True,
        )
        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(margin_x0, 31, 330, 69),
            text=render_payload["title"],
            font_name="hebo",
            font_size=23.0,
            minimum_font_size=17.0,
            font_color=primary_color,
        )
        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(margin_x0, 73, 330, 88),
            text="MODULAR CARDS · SYNTHETIC",
            font_name="hebo",
            font_size=6.8,
            minimum_font_size=6.0,
            font_color=accent_color,
        )
        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(margin_x0, 94, 330, 109),
            text=render_payload["footer"]["document_reference"],
            font_name="helv",
            font_size=6.7,
            minimum_font_size=6.0,
            font_color=muted_text,
        )

        # Empat kartu metadata sejajar.
        currency_label = (
            "Mata Uang" if render_payload["language"] == "id" else "Currency"
        )
        metadata_cards = [
            (
                "invoice_number",
                render_payload["metadata"]["invoice_number"]["label"],
                render_payload["metadata"]["invoice_number"]["display_value"],
                margin_x0,
                201.0,
            ),
            (
                "invoice_date",
                render_payload["metadata"]["invoice_date"]["label"],
                render_payload["metadata"]["invoice_date"]["display_value"],
                207.0,
                345.0,
            ),
            (
                "due_date",
                render_payload["metadata"]["due_date"]["label"],
                render_payload["metadata"]["due_date"]["display_value"],
                351.0,
                475.0,
            ),
            (
                "currency",
                currency_label,
                render_payload["currency"],
                481.0,
                margin_x1,
            ),
        ]
        metadata_y0 = 121.0
        metadata_y1 = 177.0

        for field_name, label_text, display_value, card_x0, card_x1 in metadata_cards:
            page.draw_rect(
                pymupdf.Rect(card_x0, metadata_y0, card_x1, metadata_y1),
                color=border_color,
                fill=light_primary,
                width=0.7,
                overlay=True,
            )
            page.draw_rect(
                pymupdf.Rect(card_x0, metadata_y0, card_x1, metadata_y0 + 4),
                color=accent_color,
                fill=accent_color,
                width=0,
                overlay=True,
            )
            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 10,
                    metadata_y0 + 12,
                    card_x1 - 10,
                    metadata_y0 + 27,
                ),
                text=label_text,
                font_name="helv",
                font_size=6.3,
                minimum_font_size=5.8,
                font_color=muted_text,
            )
            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 10,
                    metadata_y0 + 32,
                    card_x1 - 10,
                    metadata_y1 - 7,
                ),
                text=display_value,
                field_name=field_name,
                font_name="hebo",
                font_size=7.2,
                minimum_font_size=6.5,
                font_color=(accent_color if field_name == "currency" else primary_color),
            )

        page.draw_line(
            pymupdf.Point(margin_x0, 190),
            pymupdf.Point(margin_x1, 190),
            color=primary_color,
            width=0.9,
            overlay=True,
        )

        # Vendor dan buyer — dua kartu horizontal.
        party_cards = [
            ("vendor", render_payload["labels"]["vendor"], margin_x0, 299.0),
            ("buyer", render_payload["labels"]["buyer"], 313.0, margin_x1),
        ]
        party_y0 = 204.0
        party_y1 = 339.0

        for party_type, party_label, card_x0, card_x1 in party_cards:
            party = render_payload[party_type]
            page.draw_rect(
                pymupdf.Rect(card_x0, party_y0, card_x1, party_y1),
                color=border_color,
                fill=white,
                width=0.75,
                overlay=True,
            )
            page.draw_rect(
                pymupdf.Rect(card_x0, party_y0, card_x1, party_y0 + 24),
                color=None,
                fill=light_accent,
                width=0,
                overlay=True,
            )
            page.draw_rect(
                pymupdf.Rect(card_x0, party_y0, card_x0 + 5, party_y0 + 24),
                color=accent_color,
                fill=accent_color,
                width=0,
                overlay=True,
            )
            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 14,
                    party_y0 + 6,
                    card_x1 - 12,
                    party_y0 + 19,
                ),
                text=party_label.upper(),
                font_name="hebo",
                font_size=6.6,
                minimum_font_size=5.8,
                font_color=accent_color,
            )
            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 14,
                    party_y0 + 30,
                    card_x1 - 14,
                    party_y0 + 47,
                ),
                text=party["name"],
                field_name=f"{party_type}.name",
                font_name="hebo",
                font_size=7.8,
                minimum_font_size=6.5,
                font_color=primary_color,
            )
            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 14,
                    party_y0 + 50,
                    card_x1 - 14,
                    party_y0 + 84,
                ),
                text="\n".join(party["address_lines"]),
                field_name=f"{party_type}.address_lines",
                font_name="helv",
                font_size=6.7,
                minimum_font_size=6.5,
                font_color=dark_text,
            )
            party_contact_fields = [
                ("email", 87.0, 101.0),
                ("phone", 103.0, 117.0),
                ("tax_identifier", 119.0, 133.0),
            ]

            for field_name, field_y0, field_y1 in party_contact_fields:
                party_value = (
                    party["tax_identifier_display"]
                    if field_name == "tax_identifier"
                    else party[field_name]
                )
                add_field(
                    page=page,
                    rectangle=pymupdf.Rect(
                        card_x0 + 14,
                        party_y0 + field_y0,
                        card_x1 - 14,
                        party_y0 + field_y1,
                    ),
                    text=party_value,
                    field_name=f"{party_type}.{field_name}",
                    font_name="helv",
                    font_size=6.6,
                    minimum_font_size=6.5,
                    font_color=dark_text,
                )

        # Tabel item — boxed rows dengan ruang antarkartu.
        table_positions = [
            margin_x0, 70.0, 306.0, 363.0, 457.0, margin_x1
        ]
        table_header_y0 = 354.0
        table_header_y1 = 380.0
        row_height = 22.0
        row_gap = 4.0
        row_step = row_height + row_gap
        page.draw_rect(
            pymupdf.Rect(
                table_positions[0],
                table_header_y0,
                table_positions[-1],
                table_header_y1,
            ),
            color=primary_color,
            fill=primary_color,
            width=0.7,
            overlay=True,
        )
        table_headers = [
            "#",
            render_payload["labels"]["description"],
            render_payload["labels"]["quantity"],
            render_payload["labels"]["unit_price"],
            render_payload["labels"]["line_total"],
        ]

        for column_index, header_text in enumerate(table_headers):
            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    table_positions[column_index] + 3,
                    table_header_y0 + 7,
                    table_positions[column_index + 1] - 3,
                    table_header_y1 - 2,
                ),
                text=header_text,
                font_name="hebo",
                font_size=6.5,
                minimum_font_size=5.8,
                font_color=white,
                alignment=("left" if column_index == 1 else "center"),
            )

        for item_index, item in enumerate(render_payload["items"]):
            row_y0 = table_header_y1 + item_index * row_step
            row_y1 = row_y0 + row_height

            page.draw_rect(
                pymupdf.Rect(
                    table_positions[0], row_y0, table_positions[-1], row_y1
                ),
                color=border_color,
                fill=(light_primary if item_index % 2 else white),
                width=0.55,
                overlay=True,
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    table_positions[0] + 3,
                    row_y0 + 6,
                    table_positions[1] - 3,
                    row_y1 - 2,
                ),
                text=str(item["item_number"]),
                font_name="helv",
                font_size=6.7,
                minimum_font_size=6.0,
                font_color=muted_text,
                alignment="center",
            )
            item_fields = [
                ("description", 1, "left"),
                ("quantity", 2, "center"),
                ("unit_price", 3, "right"),
                ("line_total", 4, "right"),
            ]

            for field_name, column_index, alignment in item_fields:
                add_field(
                    page=page,
                    rectangle=pymupdf.Rect(
                        table_positions[column_index] + 4,
                        row_y0 + 6,
                        table_positions[column_index + 1] - 4,
                        row_y1 - 2,
                    ),
                    text=item[field_name],
                    field_name=f"items[{item_index}].{field_name}",
                    font_name="helv",
                    font_size=6.8,
                    minimum_font_size=6.5,
                    font_color=dark_text,
                    alignment=alignment,
                )

        table_end_y = (
            table_header_y1
            + item_count * row_step
            - row_gap
        )

        # Financial summary — side panel dinamis.
        totals_panel_height = 95.0
        totals_y0 = min(max(table_end_y + 22.0, 560.0), 609.0)
        totals_y1 = totals_y0 + totals_panel_height
        table_to_totals_gap = totals_y0 - table_end_y

        if table_to_totals_gap < 22.0:
            raise RuntimeError(
                "Tabel bertabrakan dengan side panel total. "
                f"Gap: {table_to_totals_gap:.2f} pt."
            )

        footer_line_y = 735.0

        if totals_y1 >= footer_line_y - 24.0:
            raise RuntimeError(
                "Side panel total terlalu dekat dengan footer. "
                f"Panel berakhir pada {totals_y1:.2f} pt."
            )

        totals_x0 = 350.0
        totals_x1 = margin_x1
        summary_heading = (
            "RINGKASAN"
            if render_payload["language"] == "id"
            else "SUMMARY"
        )
        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                totals_x0, totals_y0 - 19, totals_x1, totals_y0 - 4
            ),
            text=summary_heading,
            font_name="hebo",
            font_size=6.7,
            minimum_font_size=6.0,
            font_color=accent_color,
            alignment="right",
        )
        page.draw_rect(
            pymupdf.Rect(totals_x0, totals_y0, totals_x1, totals_y1),
            color=border_color,
            fill=light_primary,
            width=0.75,
            overlay=True,
        )
        page.draw_rect(
            pymupdf.Rect(totals_x1 - 5, totals_y0, totals_x1, totals_y1),
            color=accent_color,
            fill=accent_color,
            width=0,
            overlay=True,
        )
        financial_rows = [
            ("subtotal", 9.0),
            ("tax", 27.0),
            ("discount", 45.0),
            ("total", 69.0),
        ]

        for field_name, row_offset in financial_rows:
            field_payload = render_payload["financials"][field_name]
            is_total = field_name == "total"
            row_y0 = totals_y0 + row_offset

            if is_total:
                page.draw_line(
                    pymupdf.Point(totals_x0 + 12, totals_y0 + 64),
                    pymupdf.Point(totals_x1 - 13, totals_y0 + 64),
                    color=accent_color,
                    width=0.9,
                    overlay=True,
                )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    totals_x0 + 13,
                    row_y0,
                    totals_x0 + 82,
                    row_y0 + 16,
                ),
                text=field_payload["label"],
                font_name=("hebo" if is_total else "helv"),
                font_size=(7.2 if is_total else 6.8),
                minimum_font_size=6.0,
                font_color=(primary_color if is_total else muted_text),
            )
            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    totals_x0 + 84,
                    row_y0,
                    totals_x1 - 13,
                    row_y0 + 17,
                ),
                text=field_payload["display_value"],
                field_name=f"financials.{field_name}",
                font_name=("hebo" if is_total else "helv"),
                font_size=(8.0 if is_total else 7.0),
                minimum_font_size=6.5,
                font_color=(accent_color if is_total else dark_text),
                alignment="right",
            )

        page.draw_line(
            pymupdf.Point(margin_x0, totals_y0 + 12),
            pymupdf.Point(totals_x0 - 24, totals_y0 + 12),
            color=border_color,
            width=0.7,
            overlay=True,
        )
        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                totals_y0 + 20,
                totals_x0 - 24,
                totals_y0 + 37,
            ),
            text="MODULAR CARDS · TPL-08",
            font_name="hebo",
            font_size=6.4,
            minimum_font_size=5.8,
            font_color=muted_text,
        )

        # Footer.
        page.draw_line(
            pymupdf.Point(margin_x0, footer_line_y),
            pymupdf.Point(margin_x1, footer_line_y),
            color=border_color,
            width=0.7,
            overlay=True,
        )
        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(margin_x0, 743, margin_x1, 758),
            text=render_payload["footer"]["document_reference"],
            font_name="helv",
            font_size=6.5,
            minimum_font_size=6.0,
            font_color=muted_text,
            alignment="center",
        )
        notice_display_text = normalize_pdf_notice(
            render_payload["synthetic_notice"]
        )
        add_field(
            page=page,
            rectangle=pymupdf.Rect(margin_x0, 768, margin_x1, 786),
            text=notice_display_text,
            field_name="document.synthetic_notice",
            font_name="hebo",
            font_size=6.7,
            minimum_font_size=6.5,
            font_color=accent_color,
            alignment="center",
        )

        annotation_names = [
            annotation.field_name for annotation in annotations
        ]

        if len(annotation_names) != len(set(annotation_names)):
            raise RuntimeError("Ditemukan nama anotasi duplikat.")

        expected_annotation_count = 19 + 4 * item_count

        if len(annotations) != expected_annotation_count:
            raise RuntimeError(
                "Jumlah anotasi tidak sesuai. "
                f"Expected: {expected_annotation_count}; "
                f"actual: {len(annotations)}."
            )

        if not field_font_audit:
            raise RuntimeError("Audit font field tidak boleh kosong.")

        minimum_used_font_size = min(
            row["used_font_size"] for row in field_font_audit
        )

        if minimum_used_font_size < 6.5:
            raise RuntimeError(
                "Ukuran font field berada di bawah 6.5 pt."
            )

        document.save(
            str(temporary_pdf_path),
            garbage=4,
            deflate=True,
        )

    except Exception:
        temporary_pdf_path.unlink(missing_ok=True)
        raise

    finally:
        document.close()

    if (
        not temporary_pdf_path.is_file()
        or temporary_pdf_path.stat().st_size == 0
    ):
        temporary_pdf_path.unlink(missing_ok=True)
        raise RuntimeError("PDF sementara TPL-08 gagal dibuat.")

    os.replace(temporary_pdf_path, output_pdf_path)

    rendering_metadata = {
        "renderer_version": RENDER_ENGINE_VERSION,
        "template_renderer_version": TPL08_RENDERER_VERSION,
        "template_id": TPL08_TEMPLATE_ID,
        "layout_family": template_specification["layout_family"],
        "page_size": template_specification["page_size"],
        "page_width_points": round(page_width, 3),
        "page_height_points": round(page_height, 3),
        "page_count": 1,
        "annotation_type": "field_region",
        "annotation_count": len(annotations),
        "minimum_field_font_size": min(
            row["used_font_size"] for row in field_font_audit
        ),
        "font_adjustment_count": int(
            sum(bool(row["adjusted"]) for row in field_font_audit)
        ),
        "layout_metrics": {
            "item_count": item_count,
            "table_end_y_points": round(table_end_y, 3),
            "totals_panel_y0_points": round(totals_y0, 3),
            "totals_panel_y1_points": round(totals_y1, 3),
            "table_to_totals_gap_points": round(
                table_to_totals_gap,
                3,
            ),
            "footer_line_y_points": footer_line_y,
        },
    }

    return annotations, rendering_metadata


# ================================================================
# Pilih prototype TPL-08
# ================================================================

tpl08_candidate_ids = sorted(
    canonical_id
    for canonical_id, payload
    in invoice_render_payloads.items()
    if payload.get("template_id") == TPL08_TEMPLATE_ID
)

if len(tpl08_candidate_ids) != 20:
    raise RuntimeError(
        "TPL-08 harus memiliki tepat 20 payload. "
        f"Actual: {len(tpl08_candidate_ids)}."
    )

prototype_08_canonical_id = (
    tpl08_candidate_ids[0]
)

prototype_08_render_payload = (
    invoice_render_payloads[
        prototype_08_canonical_id
    ]
)

prototype_08_document_id = (
    prototype_08_render_payload[
        "document_id"
    ]
)

prototype_08_canonical_payload = (
    load_tpl08_canonical_payload(
        prototype_08_canonical_id
    )
)


# ================================================================
# Path artifact
# ================================================================

prototype_08_pdf_path = (
    TPL08_PROTOTYPE_ROOT
    / (
        f"{prototype_08_document_id}"
        "_TPL-08_prototype.pdf"
    )
)

prototype_08_png_path = (
    TPL08_PROTOTYPE_ROOT
    / (
        f"{prototype_08_document_id}"
        "_TPL-08_preview.png"
    )
)

prototype_08_ground_truth_path = (
    TPL08_PROTOTYPE_ROOT
    / (
        f"{prototype_08_document_id}"
        "_TPL-08_ground_truth.json"
    )
)


# ================================================================
# Render
# ================================================================

(
    prototype_08_annotations,
    prototype_08_rendering_metadata,
) = render_template_08(
    render_payload=prototype_08_render_payload,
    output_pdf_path=prototype_08_pdf_path,
)


# ================================================================
# Simpan ground truth
# ================================================================

prototype_08_ground_truth = {
    "schema_version": "1.0.0",
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "document": {
        "document_id": prototype_08_document_id,
        "canonical_invoice_id": (
            prototype_08_canonical_id
        ),
        "split": prototype_08_render_payload[
            "split"
        ],
        "template_id": TPL08_TEMPLATE_ID,
        "language": prototype_08_render_payload[
            "language"
        ],
        "currency": prototype_08_render_payload[
            "currency"
        ],
    },
    "canonical": prototype_08_canonical_payload,
    "rendering": (
        prototype_08_rendering_metadata
    ),
    "annotations": [
        annotation_to_dict(annotation)
        for annotation
        in prototype_08_annotations
    ],
}

write_json_atomically(
    prototype_08_ground_truth_path,
    prototype_08_ground_truth,
)


# ================================================================
# Buat preview PNG
# ================================================================

temporary_png_path = (
    prototype_08_png_path.with_name(
        f".{prototype_08_png_path.stem}.tmp.png"
    )
)

temporary_png_path.unlink(
    missing_ok=True
)

try:
    with pymupdf.open(
        str(prototype_08_pdf_path)
    ) as prototype_document:

        if prototype_document.page_count != 1:
            raise RuntimeError(
                "Prototype TPL-08 harus satu halaman."
            )

        prototype_page = prototype_document[0]

        extracted_text_08 = (
            prototype_page.get_text("text")
        )

        prototype_pixmap = (
            prototype_page.get_pixmap(
                matrix=pymupdf.Matrix(
                    150 / 72,
                    150 / 72,
                ),
                alpha=False,
            )
        )

        prototype_pixmap.save(
            str(temporary_png_path)
        )

    if (
        not temporary_png_path.is_file()
        or temporary_png_path.stat().st_size == 0
    ):
        raise RuntimeError(
            "Preview PNG gagal dibuat."
        )

    os.replace(
        temporary_png_path,
        prototype_08_png_path,
    )

except Exception:
    temporary_png_path.unlink(
        missing_ok=True
    )
    raise


# ================================================================
# Pemeriksaan awal
# ================================================================

required_text_fragments = [
    prototype_08_render_payload[
        "metadata"
    ]["invoice_number"]["display_value"],
    prototype_08_render_payload[
        "vendor"
    ]["name"],
    prototype_08_render_payload[
        "buyer"
    ]["name"],
    prototype_08_render_payload[
        "labels"
    ]["quantity"],
    prototype_08_render_payload[
        "financials"
    ]["total"]["display_value"],
    normalize_pdf_notice(
        prototype_08_render_payload[
            "synthetic_notice"
        ]
    ),
]

missing_text_fragments = [
    text
    for text in required_text_fragments
    if text not in extracted_text_08
]

if missing_text_fragments:
    raise RuntimeError(
        "Teks penting hilang dari PDF: "
        f"{missing_text_fragments}"
    )

layout_metrics = (
    prototype_08_rendering_metadata[
        "layout_metrics"
    ]
)

prototype_08_summary = pd.DataFrame(
    [
        {
            "control": "pdf_created",
            "actual": prototype_08_pdf_path.is_file(),
            "status": "VALID",
        },
        {
            "control": "ground_truth_created",
            "actual": (
                prototype_08_ground_truth_path.is_file()
            ),
            "status": "VALID",
        },
        {
            "control": "annotation_count",
            "actual": len(
                prototype_08_annotations
            ),
            "status": "VALID",
        },
        {
            "control": "item_count",
            "actual": layout_metrics[
                "item_count"
            ],
            "status": "VALID",
        },
        {
            "control": "minimum_field_font",
            "actual": (
                prototype_08_rendering_metadata[
                    "minimum_field_font_size"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "font_adjustments",
            "actual": (
                prototype_08_rendering_metadata[
                    "font_adjustment_count"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "table_to_totals_gap",
            "actual": layout_metrics[
                "table_to_totals_gap_points"
            ],
            "status": "VALID",
        },
        {
            "control": "missing_required_text",
            "actual": len(
                missing_text_fragments
            ),
            "status": "VALID",
        },
    ]
)

display(prototype_08_summary)

with PILImage.open(
    prototype_08_png_path
) as prototype_image:
    display(prototype_image.copy())


# ================================================================
# Ringkasan
# ================================================================

print(
    f"Canonical ID       : {prototype_08_canonical_id}"
)
print(
    f"Document ID        : {prototype_08_document_id}"
)
print(
    f"Prototype PDF      : {prototype_08_pdf_path}"
)
print(
    f"Prototype preview  : {prototype_08_png_path}"
)
print(
    f"Ground truth       : {prototype_08_ground_truth_path}"
)
print(
    f"Annotations        : {len(prototype_08_annotations)}"
)
print(
    "Minimum field font : "
    f"{prototype_08_rendering_metadata['minimum_field_font_size']}"
)
print(
    "Adjusted fields    : "
    f"{prototype_08_rendering_metadata['font_adjustment_count']}"
)
print(
    "Table-total gap    : "
    f"{layout_metrics['table_to_totals_gap_points']} pt"
)
print(
    f"Renderer version   : {TPL08_RENDERER_VERSION}"
)
print()
print(
    "✅ Prototype TPL-08 berhasil dibuat dan "
    "lolos pemeriksaan teknis awal."
)


In [ ]:
# ================================================================
# CELL 62 — FINAL
# Audit QA teknis prototype TPL-08
# ================================================================

from pathlib import Path
from typing import Any

import hashlib
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd
import pymupdf
from PIL import Image as PILImage


# ================================================================
# Validasi dependency runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "TPL08_PROTOTYPE_ROOT",
    "TPL08_TEMPLATE_ID",
    "prototype_08_canonical_id",
    "prototype_08_document_id",
    "prototype_08_pdf_path",
    "prototype_08_png_path",
    "prototype_08_ground_truth_path",
    "prototype_08_render_payload",
    "prototype_08_rendering_metadata",
    "prototype_08_annotations",
    "annotation_to_dict",
    "write_json_atomically",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 62 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali Cell 61 versi final."
    )


# ================================================================
# Konfigurasi audit
# ================================================================

QA_SCHEMA_VERSION = "1.0.0"
QA_AUDITOR_VERSION = "1.0.0"

MINIMUM_FIELD_FONT_SIZE = 6.5
MAXIMUM_FONT_ADJUSTMENT_RATIO = 0.20
MINIMUM_RASTER_CONTENT_RATIO = 0.01
MAXIMUM_RASTER_CONTENT_RATIO = 0.40
MINIMUM_RASTER_CONTRAST = 10.0
SEVERE_OVERLAP_THRESHOLD = 0.50

prototype_08_pdf_path = Path(
    prototype_08_pdf_path
)
prototype_08_png_path = Path(
    prototype_08_png_path
)
prototype_08_ground_truth_path = Path(
    prototype_08_ground_truth_path
)

prototype_08_qa_report_path = (
    Path(TPL08_PROTOTYPE_ROOT)
    / (
        f"{prototype_08_document_id}"
        "_TPL-08_qa_report.json"
    )
)

required_files = [
    prototype_08_pdf_path,
    prototype_08_png_path,
    prototype_08_ground_truth_path,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Artifact TPL-08 belum lengkap: "
        f"{missing_files}"
    )


# ================================================================
# Helper umum
# ================================================================

def sha256_file(file_path: Path) -> str:
    digest = hashlib.sha256()

    with Path(file_path).open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def normalize_text(value: Any) -> str:
    text = unicodedata.normalize(
        "NFKC",
        str(value or ""),
    )

    text = (
        text
        .replace("\u2014", "-")
        .replace("\u2013", "-")
        .replace("\u00b7", "-")
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip().casefold()


def normalize_notice_for_pdf(value: Any) -> str:
    return (
        str(value)
        .replace("\u2014", "\u00b7")
        .replace("\u2013", "-")
    )


def first_present(
    mapping: dict[str, Any],
    candidate_keys: tuple[str, ...],
) -> Any:
    for key in candidate_keys:
        if key in mapping:
            return mapping[key]

    return None


def annotation_field_name(
    annotation: dict[str, Any],
) -> str:
    value = first_present(
        annotation,
        (
            "field_name",
            "field_path",
            "field",
            "path",
            "name",
        ),
    )

    if value is None:
        raise KeyError(
            "Nama field tidak ditemukan pada anotasi. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return str(value)


def annotation_text(
    annotation: dict[str, Any],
) -> str:
    value = first_present(
        annotation,
        (
            "text",
            "value",
            "display_value",
            "field_value",
            "raw_value",
            "raw_text",
            "text_value",
            "content",
        ),
    )

    if isinstance(value, dict):
        value = first_present(
            value,
            (
                "text",
                "value",
                "display_value",
                "raw",
            ),
        )

    if value is None:
        raise KeyError(
            "Nilai teks tidak ditemukan pada anotasi. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return str(value)


def bbox_from_value(
    value: Any,
) -> tuple[float, float, float, float] | None:
    if isinstance(value, (list, tuple)):
        if len(value) != 4:
            return None

        try:
            return tuple(
                float(number)
                for number in value
            )
        except (TypeError, ValueError):
            return None

    if not isinstance(value, dict):
        return None

    direct_key_sets = [
        ("x0", "y0", "x1", "y1"),
        ("left", "top", "right", "bottom"),
    ]

    for key_set in direct_key_sets:
        if all(key in value for key in key_set):
            try:
                return tuple(
                    float(value[key])
                    for key in key_set
                )
            except (TypeError, ValueError):
                return None

    if all(
        key in value
        for key in ("x", "y", "width", "height")
    ):
        try:
            x0 = float(value["x"])
            y0 = float(value["y"])
            width = float(value["width"])
            height = float(value["height"])

            return (
                x0,
                y0,
                x0 + width,
                y0 + height,
            )
        except (TypeError, ValueError):
            return None

    if all(
        key in value
        for key in ("x", "y", "w", "h")
    ):
        try:
            x0 = float(value["x"])
            y0 = float(value["y"])
            width = float(value["w"])
            height = float(value["h"])

            return (
                x0,
                y0,
                x0 + width,
                y0 + height,
            )
        except (TypeError, ValueError):
            return None

    return None


def annotation_bbox(
    annotation: dict[str, Any],
) -> tuple[float, float, float, float]:
    candidate = first_present(
        annotation,
        (
            "bbox",
            "bbox_points",
            "bbox_pt",
            "bbox_pdf",
            "bounding_box",
            "rectangle",
            "rect",
            "coordinates",
            "coordinates_points",
        ),
    )

    if candidate is None:
        geometry = annotation.get("geometry")

        if isinstance(geometry, dict):
            candidate = first_present(
                geometry,
                (
                    "bbox",
                    "bounding_box",
                    "rectangle",
                ),
            )

    bbox = bbox_from_value(candidate)

    if bbox is None:
        raise KeyError(
            "Bounding box tidak ditemukan atau tidak valid. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return bbox


def rectangle_area(
    bbox: tuple[float, float, float, float],
) -> float:
    x0, y0, x1, y1 = bbox

    return max(0.0, x1 - x0) * max(
        0.0,
        y1 - y0,
    )


def intersection_area(
    first_bbox: tuple[float, float, float, float],
    second_bbox: tuple[float, float, float, float],
) -> float:
    first_x0, first_y0, first_x1, first_y1 = first_bbox
    second_x0, second_y0, second_x1, second_y1 = second_bbox

    width = max(
        0.0,
        min(first_x1, second_x1)
        - max(first_x0, second_x0),
    )

    height = max(
        0.0,
        min(first_y1, second_y1)
        - max(first_y0, second_y0),
    )

    return width * height


def bbox_intersects_word(
    annotation_box: tuple[float, float, float, float],
    word_box: tuple[float, float, float, float],
) -> bool:
    return intersection_area(
        annotation_box,
        word_box,
    ) > 0.0


# ================================================================
# Muat ground truth
# ================================================================

with prototype_08_ground_truth_path.open(
    "r",
    encoding="utf-8",
) as ground_truth_file:
    ground_truth = json.load(
        ground_truth_file
    )

ground_truth_document = ground_truth.get(
    "document",
    {},
)

if (
    ground_truth_document.get("document_id")
    != prototype_08_document_id
):
    raise RuntimeError(
        "Document ID ground truth tidak sesuai."
    )

if (
    ground_truth_document.get("template_id")
    != TPL08_TEMPLATE_ID
):
    raise RuntimeError(
        "Ground truth bukan milik TPL-08."
    )

annotation_records = ground_truth.get(
    "annotations",
    [],
)

if not isinstance(annotation_records, list):
    raise TypeError(
        "ground_truth.annotations wajib berupa list."
    )

if not annotation_records:
    raise RuntimeError(
        "Ground truth tidak memiliki anotasi."
    )


# ================================================================
# Expected field mapping
# ================================================================

render_payload = prototype_08_render_payload

expected_values: dict[str, str] = {
    "invoice_number": str(
        render_payload["metadata"]
        ["invoice_number"]
        ["display_value"]
    ),
    "invoice_date": str(
        render_payload["metadata"]
        ["invoice_date"]
        ["display_value"]
    ),
    "due_date": str(
        render_payload["metadata"]
        ["due_date"]
        ["display_value"]
    ),
    "currency": str(
        render_payload["currency"]
    ),
}

for party_type in ("vendor", "buyer"):
    party = render_payload[party_type]

    expected_values.update(
        {
            f"{party_type}.name": str(
                party["name"]
            ),
            f"{party_type}.address_lines": "\n".join(
                party["address_lines"]
            ),
            f"{party_type}.email": str(
                party["email"]
            ),
            f"{party_type}.phone": str(
                party["phone"]
            ),
            f"{party_type}.tax_identifier": str(
                party["tax_identifier_display"]
            ),
        }
    )

for item_index, item in enumerate(
    render_payload["items"]
):
    for field_name in (
        "description",
        "quantity",
        "unit_price",
        "line_total",
    ):
        expected_values[
            f"items[{item_index}].{field_name}"
        ] = str(item[field_name])

for field_name in (
    "subtotal",
    "tax",
    "discount",
    "total",
):
    expected_values[
        f"financials.{field_name}"
    ] = str(
        render_payload["financials"]
        [field_name]
        ["display_value"]
    )

expected_values[
    "document.synthetic_notice"
] = normalize_notice_for_pdf(
    render_payload["synthetic_notice"]
)

expected_annotation_count = len(
    expected_values
)


# ================================================================
# Parse annotations
# ================================================================

parsed_annotations: list[dict[str, Any]] = []
annotation_parse_failures: list[dict[str, Any]] = []

for annotation_index, annotation in enumerate(
    annotation_records
):
    try:
        parsed_annotations.append(
            {
                "index": annotation_index,
                "field_name": annotation_field_name(
                    annotation
                ),
                "text": annotation_text(
                    annotation
                ),
                "bbox": annotation_bbox(
                    annotation
                ),
            }
        )
    except (KeyError, TypeError, ValueError) as error:
        annotation_parse_failures.append(
            {
                "index": annotation_index,
                "error": str(error),
                "keys": sorted(
                    annotation.keys()
                ) if isinstance(
                    annotation,
                    dict,
                ) else [],
            }
        )

if annotation_parse_failures:
    raise RuntimeError(
        "Schema anotasi tidak dapat diproses: "
        f"{annotation_parse_failures[:3]}"
    )

annotation_names = [
    annotation["field_name"]
    for annotation in parsed_annotations
]

annotation_by_name = {
    annotation["field_name"]: annotation
    for annotation in parsed_annotations
}

duplicate_annotation_count = (
    len(annotation_names)
    - len(set(annotation_names))
)

missing_required_annotations = sorted(
    set(expected_values)
    - set(annotation_names)
)


# ================================================================
# Audit PDF dan anotasi
# ================================================================

with pymupdf.open(
    str(prototype_08_pdf_path)
) as pdf_document:
    pdf_page_count = pdf_document.page_count

    if pdf_page_count < 1:
        raise RuntimeError(
            "PDF TPL-08 tidak memiliki halaman."
        )

    first_page = pdf_document[0]
    page_rectangle = first_page.rect
    extracted_text = first_page.get_text("text")
    extracted_words = first_page.get_text("words")

page_width = float(page_rectangle.width)
page_height = float(page_rectangle.height)

word_boxes = [
    (
        float(word[0]),
        float(word[1]),
        float(word[2]),
        float(word[3]),
    )
    for word in extracted_words
    if len(word) >= 5
    and str(word[4]).strip()
]

invalid_bounding_boxes: list[str] = []
annotations_without_words: list[str] = []

for annotation in parsed_annotations:
    field_name = annotation["field_name"]
    x0, y0, x1, y1 = annotation["bbox"]

    coordinates_are_finite = all(
        math.isfinite(value)
        for value in (x0, y0, x1, y1)
    )

    bbox_is_valid = (
        coordinates_are_finite
        and x0 >= 0.0
        and y0 >= 0.0
        and x1 <= page_width
        and y1 <= page_height
        and x1 > x0
        and y1 > y0
    )

    if not bbox_is_valid:
        invalid_bounding_boxes.append(
            field_name
        )
        continue

    contains_word = any(
        bbox_intersects_word(
            annotation["bbox"],
            word_box,
        )
        for word_box in word_boxes
    )

    if not contains_word:
        annotations_without_words.append(
            field_name
        )


# ================================================================
# Audit nilai anotasi
# ================================================================

normalization_mismatches: list[dict[str, str]] = []

for field_name, expected_value in expected_values.items():
    annotation = annotation_by_name.get(
        field_name
    )

    if annotation is None:
        continue

    actual_value = annotation["text"]

    if normalize_text(actual_value) != normalize_text(
        expected_value
    ):
        normalization_mismatches.append(
            {
                "field_name": field_name,
                "expected": expected_value,
                "actual": actual_value,
            }
        )

normalized_extracted_text = normalize_text(
    extracted_text
)

missing_extracted_values: list[str] = []

for annotation in parsed_annotations:
    normalized_value = normalize_text(
        annotation["text"]
    )

    if (
        normalized_value
        and normalized_value
        not in normalized_extracted_text
    ):
        missing_extracted_values.append(
            annotation["field_name"]
        )


# ================================================================
# Audit overlap anotasi
# ================================================================

severe_annotation_overlaps: list[dict[str, Any]] = []

for first_index in range(
    len(parsed_annotations)
):
    first_annotation = parsed_annotations[
        first_index
    ]
    first_area = rectangle_area(
        first_annotation["bbox"]
    )

    if first_area <= 0.0:
        continue

    for second_index in range(
        first_index + 1,
        len(parsed_annotations),
    ):
        second_annotation = parsed_annotations[
            second_index
        ]
        second_area = rectangle_area(
            second_annotation["bbox"]
        )

        if second_area <= 0.0:
            continue

        overlap_area = intersection_area(
            first_annotation["bbox"],
            second_annotation["bbox"],
        )

        overlap_ratio = overlap_area / min(
            first_area,
            second_area,
        )

        if overlap_ratio >= SEVERE_OVERLAP_THRESHOLD:
            severe_annotation_overlaps.append(
                {
                    "first": first_annotation[
                        "field_name"
                    ],
                    "second": second_annotation[
                        "field_name"
                    ],
                    "overlap_ratio": round(
                        overlap_ratio,
                        6,
                    ),
                }
            )


# ================================================================
# Audit font dan raster
# ================================================================

minimum_field_font_size = float(
    prototype_08_rendering_metadata[
        "minimum_field_font_size"
    ]
)

font_adjustment_count = int(
    prototype_08_rendering_metadata[
        "font_adjustment_count"
    ]
)

font_adjustment_ratio = (
    font_adjustment_count
    / max(1, len(parsed_annotations))
)

with PILImage.open(
    prototype_08_png_path
) as preview_image:
    grayscale_array = np.asarray(
        preview_image.convert("L"),
        dtype=np.float32,
    ).copy()

if grayscale_array.size == 0:
    raise RuntimeError(
        "Preview PNG tidak memiliki piksel."
    )

raster_content_ratio = float(
    np.mean(grayscale_array < 245.0)
)

raster_contrast = float(
    np.std(grayscale_array)
)


# ================================================================
# Susun hasil pemeriksaan
# ================================================================

checks = [
    {
        "control": "pdf_page_count",
        "expected": 1,
        "actual": pdf_page_count,
        "valid": pdf_page_count == 1,
    },
    {
        "control": "annotation_count",
        "expected": expected_annotation_count,
        "actual": len(parsed_annotations),
        "valid": (
            len(parsed_annotations)
            == expected_annotation_count
        ),
    },
    {
        "control": "duplicate_annotations",
        "expected": 0,
        "actual": duplicate_annotation_count,
        "valid": duplicate_annotation_count == 0,
    },
    {
        "control": "missing_required_annotations",
        "expected": 0,
        "actual": len(
            missing_required_annotations
        ),
        "valid": not missing_required_annotations,
    },
    {
        "control": "invalid_bounding_boxes",
        "expected": 0,
        "actual": len(
            invalid_bounding_boxes
        ),
        "valid": not invalid_bounding_boxes,
    },
    {
        "control": "normalization_mismatches",
        "expected": 0,
        "actual": len(
            normalization_mismatches
        ),
        "valid": not normalization_mismatches,
    },
    {
        "control": "annotations_without_words",
        "expected": 0,
        "actual": len(
            annotations_without_words
        ),
        "valid": not annotations_without_words,
    },
    {
        "control": "severe_annotation_overlaps",
        "expected": 0,
        "actual": len(
            severe_annotation_overlaps
        ),
        "valid": not severe_annotation_overlaps,
    },
    {
        "control": "missing_extracted_values",
        "expected": 0,
        "actual": len(
            missing_extracted_values
        ),
        "valid": not missing_extracted_values,
    },
    {
        "control": "minimum_field_font_size",
        "expected": ">=6.5",
        "actual": round(
            minimum_field_font_size,
            4,
        ),
        "valid": (
            minimum_field_font_size
            >= MINIMUM_FIELD_FONT_SIZE
        ),
    },
    {
        "control": "font_adjustment_ratio",
        "expected": "<=0.20",
        "actual": round(
            font_adjustment_ratio,
            4,
        ),
        "valid": (
            font_adjustment_ratio
            <= MAXIMUM_FONT_ADJUSTMENT_RATIO
        ),
    },
    {
        "control": "raster_content_ratio",
        "expected": "0.01–0.40",
        "actual": round(
            raster_content_ratio,
            4,
        ),
        "valid": (
            MINIMUM_RASTER_CONTENT_RATIO
            <= raster_content_ratio
            <= MAXIMUM_RASTER_CONTENT_RATIO
        ),
    },
    {
        "control": "raster_contrast",
        "expected": ">=10.0",
        "actual": round(
            raster_contrast,
            4,
        ),
        "valid": (
            raster_contrast
            >= MINIMUM_RASTER_CONTRAST
        ),
    },
]

for check in checks:
    check["status"] = (
        "VALID"
        if check["valid"]
        else "INVALID"
    )

failed_checks = [
    check
    for check in checks
    if not check["valid"]
]

qa_status = (
    "PASSED"
    if not failed_checks
    else "FAILED"
)


# ================================================================
# Simpan QA report
# ================================================================

qa_report = {
    "schema_version": QA_SCHEMA_VERSION,
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "document_id": prototype_08_document_id,
    "canonical_invoice_id": (
        prototype_08_canonical_id
    ),
    "template_id": TPL08_TEMPLATE_ID,
    "status": qa_status,
    "manual_visual_review": "PENDING",
    "metrics": {
        "auditor_version": QA_AUDITOR_VERSION,
        "pdf_page_count": pdf_page_count,
        "annotation_count": len(
            parsed_annotations
        ),
        "expected_annotation_count": (
            expected_annotation_count
        ),
        "duplicate_annotation_count": (
            duplicate_annotation_count
        ),
        "missing_required_annotation_count": len(
            missing_required_annotations
        ),
        "invalid_bounding_box_count": len(
            invalid_bounding_boxes
        ),
        "normalization_mismatch_count": len(
            normalization_mismatches
        ),
        "annotations_without_words_count": len(
            annotations_without_words
        ),
        "severe_annotation_overlap_count": len(
            severe_annotation_overlaps
        ),
        "missing_extracted_value_count": len(
            missing_extracted_values
        ),
        "minimum_field_font_size": round(
            minimum_field_font_size,
            6,
        ),
        "font_adjustment_count": (
            font_adjustment_count
        ),
        "font_adjustment_ratio": round(
            font_adjustment_ratio,
            6,
        ),
        "raster_content_ratio": round(
            raster_content_ratio,
            6,
        ),
        "raster_contrast": round(
            raster_contrast,
            6,
        ),
    },
    "failures": {
        "failed_checks": [
            check["control"]
            for check in failed_checks
        ],
        "missing_required_annotations": (
            missing_required_annotations
        ),
        "invalid_bounding_boxes": (
            invalid_bounding_boxes
        ),
        "normalization_mismatches": (
            normalization_mismatches
        ),
        "annotations_without_words": (
            annotations_without_words
        ),
        "severe_annotation_overlaps": (
            severe_annotation_overlaps
        ),
        "missing_extracted_values": (
            missing_extracted_values
        ),
    },
    "checksums_sha256": {
        "prototype_pdf": sha256_file(
            prototype_08_pdf_path
        ),
        "preview": sha256_file(
            prototype_08_png_path
        ),
        "ground_truth": sha256_file(
            prototype_08_ground_truth_path
        ),
    },
}

write_json_atomically(
    prototype_08_qa_report_path,
    qa_report,
)


# ================================================================
# Verifikasi round-trip QA report
# ================================================================

with prototype_08_qa_report_path.open(
    "r",
    encoding="utf-8",
) as qa_report_file:
    verified_qa_report = json.load(
        qa_report_file
    )

if verified_qa_report != qa_report:
    raise RuntimeError(
        "QA report berubah setelah disimpan dan dibuka ulang."
    )


# ================================================================
# Output audit
# ================================================================

qa_summary = pd.DataFrame(
    [
        {
            "control": check["control"],
            "expected": check["expected"],
            "actual": check["actual"],
            "status": check["status"],
        }
        for check in checks
    ]
)

display(qa_summary)

print(
    f"QA report          : "
    f"{prototype_08_qa_report_path}"
)
print(
    f"QA status          : {qa_status}"
)
print(
    "Minimum field font : "
    f"{minimum_field_font_size:.2f}"
)
print(
    "Font adjustments   : "
    f"{font_adjustment_count}"
)
print(
    "Ink pixel ratio    : "
    f"{raster_content_ratio:.6f}"
)
print(
    "Raster contrast    : "
    f"{raster_contrast:.6f}"
)
print(
    "Manual review      : "
    f"{qa_report['manual_visual_review']}"
)
print()

if qa_status != "PASSED":
    print("Detail kegagalan:")
    print(
        json.dumps(
            qa_report["failures"],
            ensure_ascii=False,
            indent=2,
        )
    )

    raise RuntimeError(
        "Prototype TPL-08 gagal audit QA teknis."
    )

print(
    "✅ Prototype TPL-08 lulus audit teknis. "
    "Review visual manual masih perlu dicatat "
    "ke QA report."
)


In [ ]:
# ================================================================
# CELL 62A — FINAL
# Finalisasi review visual manual prototype TPL-08
# ================================================================

from pathlib import Path

import hashlib
import json


# ================================================================
# Validasi dependency runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "prototype_08_qa_report_path",
    "prototype_08_png_path",
    "prototype_08_document_id",
    "write_json_atomically",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 62A belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali Cell 61 dan Cell 62."
    )


# ================================================================
# Helper checksum
# ================================================================

def calculate_sha256(
    file_path: Path,
) -> str:
    """Menghitung SHA-256 tanpa memuat seluruh file ke memori."""

    digest = hashlib.sha256()

    with file_path.open(
        mode="rb",
    ) as input_file:
        for chunk in iter(
            lambda: input_file.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ================================================================
# Muat artifact yang akan difinalisasi
# ================================================================

qa_report_path = Path(
    prototype_08_qa_report_path
)

preview_path = Path(
    prototype_08_png_path
)

if not qa_report_path.is_file():
    raise FileNotFoundError(
        "QA report TPL-08 tidak ditemukan: "
        f"{qa_report_path}"
    )

if not preview_path.is_file():
    raise FileNotFoundError(
        "Preview TPL-08 tidak ditemukan: "
        f"{preview_path}"
    )

with qa_report_path.open(
    mode="r",
    encoding="utf-8",
) as qa_report_file:
    qa_report = json.load(
        qa_report_file
    )


# ================================================================
# Validasi identitas dan status audit teknis
# ================================================================

if qa_report.get("template_id") != "TPL-08":
    raise RuntimeError(
        "QA report bukan milik TPL-08."
    )

if (
    qa_report.get("document_id")
    != prototype_08_document_id
):
    raise RuntimeError(
        "Document ID QA report tidak sesuai. "
        f"Expected: {prototype_08_document_id!r}; "
        f"actual: {qa_report.get('document_id')!r}."
    )

if qa_report.get("status") != "PASSED":
    raise RuntimeError(
        "Review visual tidak boleh difinalisasi "
        "karena audit teknis belum PASSED. "
        f"Status saat ini: "
        f"{qa_report.get('status')!r}."
    )


# ================================================================
# Pastikan tidak ada kegagalan teknis
# ================================================================

failure_details = qa_report.get(
    "failures",
    {},
)

if not isinstance(
    failure_details,
    dict,
):
    raise TypeError(
        "Field 'failures' pada QA report "
        "wajib berupa dictionary."
    )

# Dictionary failures sendiri selalu bernilai True jika memiliki key.
# Karena itu, periksa isi setiap daftar kegagalannya.
nonempty_failures = {
    failure_name: failure_value
    for failure_name, failure_value
    in failure_details.items()
    if bool(failure_value)
}

if nonempty_failures:
    raise RuntimeError(
        "QA report masih memiliki kegagalan teknis: "
        f"{nonempty_failures}"
    )

if "manual_visual_review" not in qa_report:
    raise KeyError(
        "Field 'manual_visual_review' tidak ditemukan "
        "dalam QA report."
    )

current_manual_status = qa_report[
    "manual_visual_review"
]

if current_manual_status not in {
    "PENDING",
    "PASSED",
}:
    raise RuntimeError(
        "Status manual visual review tidak dikenali: "
        f"{current_manual_status!r}."
    )


# ================================================================
# Pastikan preview yang direview sama dengan yang diaudit
# ================================================================

recorded_preview_checksum = (
    qa_report
    .get("checksums_sha256", {})
    .get("preview")
)

if not recorded_preview_checksum:
    raise KeyError(
        "Checksum preview tidak ditemukan pada "
        "qa_report['checksums_sha256']['preview']."
    )

actual_preview_checksum = (
    calculate_sha256(
        preview_path
    )
)

if (
    actual_preview_checksum
    != recorded_preview_checksum
):
    raise RuntimeError(
        "Preview berubah setelah audit teknis. "
        "Jalankan ulang Cell 62 dan lakukan "
        "review visual kembali. "
        f"Expected SHA-256: "
        f"{recorded_preview_checksum}; "
        f"actual: {actual_preview_checksum}."
    )


# ================================================================
# Catat hasil review visual secara idempoten
# ================================================================

qa_report[
    "manual_visual_review"
] = "PASSED"

write_json_atomically(
    qa_report_path,
    qa_report,
)


# ================================================================
# Buka ulang dan verifikasi persistensi
# ================================================================

with qa_report_path.open(
    mode="r",
    encoding="utf-8",
) as qa_report_file:
    persisted_qa_report = json.load(
        qa_report_file
    )

if (
    persisted_qa_report.get("template_id")
    != "TPL-08"
):
    raise RuntimeError(
        "Template ID berubah setelah finalisasi."
    )

if (
    persisted_qa_report.get("document_id")
    != prototype_08_document_id
):
    raise RuntimeError(
        "Document ID berubah setelah finalisasi."
    )

if (
    persisted_qa_report.get("status")
    != "PASSED"
):
    raise RuntimeError(
        "Status audit teknis berubah "
        "setelah finalisasi."
    )

if (
    persisted_qa_report.get(
        "manual_visual_review"
    )
    != "PASSED"
):
    raise RuntimeError(
        "Status review visual gagal disimpan."
    )

persisted_preview_checksum = (
    persisted_qa_report
    .get("checksums_sha256", {})
    .get("preview")
)

if (
    persisted_preview_checksum
    != actual_preview_checksum
):
    raise RuntimeError(
        "Checksum preview berubah "
        "pada QA report."
    )


# ================================================================
# Ringkasan final
# ================================================================

print(
    f"QA report           : "
    f"{qa_report_path}"
)

print(
    "Technical status    : "
    f"{persisted_qa_report['status']}"
)

print(
    "Manual visual review: "
    f"{persisted_qa_report['manual_visual_review']}"
)

print(
    "Preview SHA-256     : "
    f"{persisted_preview_checksum}"
)

print()

print(
    "✅ TPL-08 FINAL PASSED — audit teknis dan "
    "review visual telah selesai."
)

In [ ]:
# ================================================================
# CELL 63 — FINAL
# Inspeksi spesifikasi dan presentation payload TPL-09
# Read-only: tidak membuat atau mengubah artifact build
# ================================================================

from typing import Any

import json

import pandas as pd
from IPython.display import display


# ================================================================
# Validasi dependency runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "template_registry",
    "invoice_render_payloads",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 63 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali cell pembentukan template registry "
        "dan presentation payload."
    )

if not isinstance(
    template_registry,
    dict,
):
    raise TypeError(
        "template_registry wajib berupa dictionary."
    )

if not isinstance(
    invoice_render_payloads,
    dict,
):
    raise TypeError(
        "invoice_render_payloads wajib berupa dictionary."
    )


# ================================================================
# Konfigurasi inspeksi
# ================================================================

TPL09_TEMPLATE_ID = "TPL-09"
TPL09_EXPECTED_PAYLOAD_COUNT = 20

if TPL09_TEMPLATE_ID not in template_registry:
    raise KeyError(
        "Spesifikasi TPL-09 tidak ditemukan dalam "
        "template_registry."
    )

tpl09_template_specification = (
    template_registry[
        TPL09_TEMPLATE_ID
    ]
)

if not isinstance(
    tpl09_template_specification,
    dict,
):
    raise TypeError(
        "Spesifikasi TPL-09 wajib berupa dictionary."
    )

tpl09_payloads = {
    canonical_id: payload
    for canonical_id, payload
    in invoice_render_payloads.items()
    if isinstance(payload, dict)
    and payload.get("template_id")
    == TPL09_TEMPLATE_ID
}

if (
    len(tpl09_payloads)
    != TPL09_EXPECTED_PAYLOAD_COUNT
):
    raise RuntimeError(
        "TPL-09 harus memiliki tepat "
        f"{TPL09_EXPECTED_PAYLOAD_COUNT} payload. "
        f"Actual: {len(tpl09_payloads)}."
    )


# ================================================================
# Kontrak struktur presentation payload
# ================================================================

REQUIRED_TOP_LEVEL_KEYS = {
    "buyer",
    "canonical_invoice_id",
    "currency",
    "document_id",
    "financials",
    "footer",
    "items",
    "labels",
    "language",
    "metadata",
    "split",
    "synthetic_notice",
    "template_id",
    "title",
    "vendor",
}

REQUIRED_LABEL_KEYS = {
    "buyer",
    "description",
    "discount",
    "document_title",
    "due_date",
    "invoice_date",
    "invoice_number",
    "line_total",
    "page",
    "quantity",
    "subtotal",
    "synthetic_notice",
    "tax",
    "tax_identifier",
    "total",
    "unit_price",
    "vendor",
}

REQUIRED_METADATA_KEYS = {
    "due_date",
    "invoice_date",
    "invoice_number",
}

REQUIRED_PARTY_KEYS = {
    "address_lines",
    "email",
    "name",
    "party_id",
    "phone",
    "tax_identifier",
    "tax_identifier_display",
}

REQUIRED_ITEM_KEYS = {
    "description",
    "item_number",
    "line_total",
    "quantity",
    "raw",
    "unit_price",
}

REQUIRED_FINANCIAL_KEYS = {
    "discount",
    "subtotal",
    "tax",
    "total",
}

REQUIRED_FOOTER_KEYS = {
    "document_reference",
    "synthetic_notice",
}


# ================================================================
# Helper validasi
# ================================================================

def require_dictionary(
    value: Any,
    field_path: str,
) -> dict[str, Any]:
    """Memastikan field merupakan dictionary."""

    if not isinstance(value, dict):
        raise TypeError(
            f"{field_path} wajib berupa dictionary."
        )

    return value


def require_key_set(
    value: dict[str, Any],
    required_keys: set[str],
    field_path: str,
) -> None:
    """Memastikan seluruh key wajib tersedia."""

    missing_keys = sorted(
        required_keys - set(value)
    )

    if missing_keys:
        raise KeyError(
            f"{field_path} kehilangan key wajib: "
            f"{missing_keys}"
        )


def joined_text_length(
    values: Any,
) -> int:
    """Mengukur panjang gabungan list teks."""

    if not isinstance(values, list):
        return len(str(values))

    return len(
        " ".join(
            str(value)
            for value in values
        )
    )


def get_display_value(
    value: Any,
) -> str:
    """Mengambil display value dari field terstruktur."""

    if isinstance(value, dict):
        return str(
            value.get(
                "display_value",
                value.get(
                    "value",
                    "",
                ),
            )
        )

    return str(value)


def maximum_money_length(
    payload: dict[str, Any],
) -> int:
    """Mengukur nilai uang terpanjang yang perlu dirender."""

    money_values: list[str] = []

    for item in payload["items"]:
        money_values.extend(
            [
                str(item["unit_price"]),
                str(item["line_total"]),
            ]
        )

    for field_name in (
        "subtotal",
        "tax",
        "discount",
        "total",
    ):
        money_values.append(
            get_display_value(
                payload[
                    "financials"
                ][field_name]
            )
        )

    return max(
        map(len, money_values),
        default=0,
    )


# ================================================================
# Validasi seluruh payload TPL-09
# ================================================================

expected_split = (
    tpl09_template_specification.get(
        "split"
    )
)

document_ids: list[str] = []

for canonical_id in sorted(
    tpl09_payloads
):
    payload = tpl09_payloads[
        canonical_id
    ]

    require_key_set(
        payload,
        REQUIRED_TOP_LEVEL_KEYS,
        f"payload[{canonical_id!r}]",
    )

    if (
        payload["canonical_invoice_id"]
        != canonical_id
    ):
        raise RuntimeError(
            "Canonical ID pada dictionary dan payload "
            f"tidak sama: {canonical_id!r}."
        )

    if (
        payload["template_id"]
        != TPL09_TEMPLATE_ID
    ):
        raise RuntimeError(
            "Template ID tidak sesuai pada "
            f"{canonical_id}."
        )

    if (
        expected_split is not None
        and payload["split"] != expected_split
    ):
        raise RuntimeError(
            f"Split payload {canonical_id} tidak sesuai. "
            f"Expected: {expected_split!r}; "
            f"actual: {payload['split']!r}."
        )

    if payload["language"] not in {
        "id",
        "en",
    }:
        raise RuntimeError(
            "Language tidak didukung pada "
            f"{canonical_id}: "
            f"{payload['language']!r}."
        )

    if not str(
        payload["document_id"]
    ).strip():
        raise RuntimeError(
            f"document_id kosong pada {canonical_id}."
        )

    labels = require_dictionary(
        payload["labels"],
        f"payload[{canonical_id!r}].labels",
    )

    metadata = require_dictionary(
        payload["metadata"],
        f"payload[{canonical_id!r}].metadata",
    )

    vendor = require_dictionary(
        payload["vendor"],
        f"payload[{canonical_id!r}].vendor",
    )

    buyer = require_dictionary(
        payload["buyer"],
        f"payload[{canonical_id!r}].buyer",
    )

    financials = require_dictionary(
        payload["financials"],
        f"payload[{canonical_id!r}].financials",
    )

    footer = require_dictionary(
        payload["footer"],
        f"payload[{canonical_id!r}].footer",
    )

    require_key_set(
        labels,
        REQUIRED_LABEL_KEYS,
        f"payload[{canonical_id!r}].labels",
    )

    require_key_set(
        metadata,
        REQUIRED_METADATA_KEYS,
        f"payload[{canonical_id!r}].metadata",
    )

    require_key_set(
        vendor,
        REQUIRED_PARTY_KEYS,
        f"payload[{canonical_id!r}].vendor",
    )

    require_key_set(
        buyer,
        REQUIRED_PARTY_KEYS,
        f"payload[{canonical_id!r}].buyer",
    )

    require_key_set(
        financials,
        REQUIRED_FINANCIAL_KEYS,
        f"payload[{canonical_id!r}].financials",
    )

    require_key_set(
        footer,
        REQUIRED_FOOTER_KEYS,
        f"payload[{canonical_id!r}].footer",
    )

    for metadata_name in REQUIRED_METADATA_KEYS:
        metadata_field = require_dictionary(
            metadata[metadata_name],
            (
                f"payload[{canonical_id!r}]"
                f".metadata[{metadata_name!r}]"
            ),
        )

        require_key_set(
            metadata_field,
            {
                "label",
                "display_value",
            },
            (
                f"payload[{canonical_id!r}]"
                f".metadata[{metadata_name!r}]"
            ),
        )

    for party_name, party in (
        ("vendor", vendor),
        ("buyer", buyer),
    ):
        address_lines = party[
            "address_lines"
        ]

        if (
            not isinstance(address_lines, list)
            or not address_lines
        ):
            raise RuntimeError(
                f"payload[{canonical_id!r}]"
                f".{party_name}.address_lines "
                "wajib berupa list yang tidak kosong."
            )

    items = payload["items"]

    if (
        not isinstance(items, list)
        or not items
    ):
        raise RuntimeError(
            f"payload[{canonical_id!r}].items "
            "wajib berupa list yang tidak kosong."
        )

    for item_index, item in enumerate(
        items
    ):
        item = require_dictionary(
            item,
            (
                f"payload[{canonical_id!r}]"
                f".items[{item_index}]"
            ),
        )

        require_key_set(
            item,
            REQUIRED_ITEM_KEYS,
            (
                f"payload[{canonical_id!r}]"
                f".items[{item_index}]"
            ),
        )

    document_ids.append(
        str(payload["document_id"])
    )

if (
    len(document_ids)
    != len(set(document_ids))
):
    raise RuntimeError(
        "Ditemukan document_id duplikat "
        "pada payload TPL-09."
    )


# ================================================================
# Bangun struktur sampel
# ================================================================

tpl09_candidate_ids = sorted(
    tpl09_payloads
)

tpl09_sample_id = (
    tpl09_candidate_ids[0]
)

tpl09_sample_payload = (
    tpl09_payloads[
        tpl09_sample_id
    ]
)

tpl09_sample_structure = {
    "canonical_invoice_id": (
        tpl09_sample_payload[
            "canonical_invoice_id"
        ]
    ),
    "document_id": (
        tpl09_sample_payload[
            "document_id"
        ]
    ),
    "template_id": (
        tpl09_sample_payload[
            "template_id"
        ]
    ),
    "language": (
        tpl09_sample_payload[
            "language"
        ]
    ),
    "currency": (
        tpl09_sample_payload[
            "currency"
        ]
    ),
    "title": (
        tpl09_sample_payload[
            "title"
        ]
    ),
    "top_level_keys": sorted(
        tpl09_sample_payload
    ),
    "label_keys": sorted(
        tpl09_sample_payload[
            "labels"
        ]
    ),
    "metadata_keys": sorted(
        tpl09_sample_payload[
            "metadata"
        ]
    ),
    "vendor_keys": sorted(
        tpl09_sample_payload[
            "vendor"
        ]
    ),
    "buyer_keys": sorted(
        tpl09_sample_payload[
            "buyer"
        ]
    ),
    "item_keys": sorted(
        tpl09_sample_payload[
            "items"
        ][0]
    ),
    "financial_keys": sorted(
        tpl09_sample_payload[
            "financials"
        ]
    ),
    "footer_keys": sorted(
        tpl09_sample_payload[
            "footer"
        ]
    ),
}


# ================================================================
# Bangun statistik payload
# ================================================================

statistics_rows: list[
    dict[str, Any]
] = []

for canonical_id in tpl09_candidate_ids:
    payload = tpl09_payloads[
        canonical_id
    ]

    descriptions = [
        str(item["description"])
        for item in payload["items"]
    ]

    statistics_rows.append(
        {
            "canonical_id": canonical_id,
            "document_id": (
                payload["document_id"]
            ),
            "language": (
                payload["language"]
            ),
            "currency": (
                payload["currency"]
            ),
            "item_count": len(
                payload["items"]
            ),
            "vendor_name_length": len(
                str(
                    payload[
                        "vendor"
                    ]["name"]
                )
            ),
            "vendor_address_length": (
                joined_text_length(
                    payload[
                        "vendor"
                    ]["address_lines"]
                )
            ),
            "vendor_email_length": len(
                str(
                    payload[
                        "vendor"
                    ]["email"]
                )
            ),
            "buyer_name_length": len(
                str(
                    payload[
                        "buyer"
                    ]["name"]
                )
            ),
            "buyer_address_length": (
                joined_text_length(
                    payload[
                        "buyer"
                    ]["address_lines"]
                )
            ),
            "buyer_email_length": len(
                str(
                    payload[
                        "buyer"
                    ]["email"]
                )
            ),
            "maximum_description_length": max(
                map(
                    len,
                    descriptions,
                ),
                default=0,
            ),
            "maximum_money_length": (
                maximum_money_length(
                    payload
                )
            ),
        }
    )

tpl09_statistics = pd.DataFrame(
    statistics_rows
)

tpl09_payload_statistics = (
    tpl09_statistics
    .describe(
        include="all"
    )
    .transpose()
)

tpl09_item_count_distribution = (
    tpl09_statistics[
        "item_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "item_count"
    )
    .reset_index(
        name="document_count"
    )
)

tpl09_language_distribution = (
    tpl09_statistics[
        "language"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "language"
    )
    .reset_index(
        name="document_count"
    )
)

tpl09_currency_distribution = (
    tpl09_statistics[
        "currency"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "currency"
    )
    .reset_index(
        name="document_count"
    )
)


# ================================================================
# Output inspeksi
# ================================================================

print(
    "TEMPLATE SPECIFICATION"
)

print(
    json.dumps(
        tpl09_template_specification,
        ensure_ascii=False,
        indent=2,
    )
)

print()
print("PAYLOAD COUNT")
print(
    len(tpl09_payloads)
)

print()
print("SAMPLE STRUCTURE")

print(
    json.dumps(
        tpl09_sample_structure,
        ensure_ascii=False,
        indent=2,
    )
)

print()
print("LABELS")

print(
    json.dumps(
        tpl09_sample_payload[
            "labels"
        ],
        ensure_ascii=False,
        indent=2,
    )
)

print()
print("PAYLOAD STATISTICS")

display(
    tpl09_payload_statistics
)

print()
print("ITEM-COUNT DISTRIBUTION")

display(
    tpl09_item_count_distribution
)

print()
print("LANGUAGE DISTRIBUTION")

display(
    tpl09_language_distribution
)

print()
print("CURRENCY DISTRIBUTION")

display(
    tpl09_currency_distribution
)

print()

print(
    "✅ Inspeksi TPL-09 selesai. "
    "Belum ada artifact yang dibuat atau diubah."
)

In [ ]:
# ================================================================
# CELL 64 — FINAL
# Prototype renderer TPL-09: Compact Ledger
# ================================================================

from pathlib import Path
from typing import Any
from PIL import Image as PILImage

import json
import os

import pandas as pd
import pymupdf


# ================================================================
# Validasi dependency runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "PROTOTYPE_ROOT",
    "CANONICAL_RECORDS_JSONL_PATH",
    "FieldAnnotation",
    "template_registry",
    "invoice_render_payloads",
    "get_page_dimensions",
    "hex_to_rgb",
    "insert_annotated_text",
    "insert_static_text",
    "annotation_to_dict",
    "write_json_atomically",
    "RENDER_ENGINE_VERSION",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 64 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali cell pemulihan runtime, "
        "template registry, dan presentation payload."
    )


# ================================================================
# Konfigurasi TPL-09
# ================================================================

TPL09_TEMPLATE_ID = "TPL-09"
TPL09_RENDERER_VERSION = "1.0.0"

TPL09_MIN_ITEMS_PER_PAGE = 2
TPL09_MAX_ITEMS_PER_PAGE = 8

TPL09_PROTOTYPE_ROOT = (
    Path(PROTOTYPE_ROOT)
    / TPL09_TEMPLATE_ID
)

TPL09_PROTOTYPE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ================================================================
# Helper canonical payload
# ================================================================

def load_tpl09_canonical_payload(
    canonical_id: str,
) -> dict[str, Any]:
    """Memuat satu canonical invoice dari JSONL."""

    if (
        not isinstance(canonical_id, str)
        or not canonical_id.strip()
    ):
        raise ValueError(
            "canonical_id wajib berupa string "
            "yang tidak kosong."
        )

    canonical_path = Path(
        CANONICAL_RECORDS_JSONL_PATH
    )

    if not canonical_path.is_file():
        raise FileNotFoundError(
            "Canonical JSONL tidak ditemukan: "
            f"{canonical_path}"
        )

    with canonical_path.open(
        mode="r",
        encoding="utf-8",
    ) as canonical_file:
        for line_number, line in enumerate(
            canonical_file,
            start=1,
        ):
            clean_line = line.strip()

            if not clean_line:
                continue

            try:
                payload = json.loads(
                    clean_line
                )
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    "Canonical JSONL tidak valid "
                    f"pada baris {line_number}."
                ) from error

            if (
                payload.get(
                    "canonical_invoice_id"
                )
                == canonical_id
            ):
                return payload

    raise KeyError(
        "Canonical payload tidak ditemukan: "
        f"{canonical_id}"
    )


def normalize_pdf_notice(
    value: Any,
) -> str:
    """Mengganti separator yang tidak didukung font PDF bawaan."""

    return (
        str(value)
        .replace("—", "·")
        .replace("–", "-")
    )


# ================================================================
# Renderer final TPL-09
# ================================================================

def render_template_09(
    render_payload: dict[str, Any],
    output_pdf_path: Path,
) -> tuple[
    list[FieldAnnotation],
    dict[str, Any],
]:
    """Merender satu halaman A4 dengan layout compact ledger."""

    if not isinstance(
        render_payload,
        dict,
    ):
        raise TypeError(
            "render_payload wajib berupa dictionary."
        )

    if (
        render_payload.get("template_id")
        != TPL09_TEMPLATE_ID
    ):
        raise ValueError(
            "Renderer TPL-09 hanya menerima "
            "payload TPL-09."
        )

    required_payload_keys = {
        "canonical_invoice_id",
        "document_id",
        "template_id",
        "language",
        "currency",
        "title",
        "labels",
        "metadata",
        "vendor",
        "buyer",
        "items",
        "financials",
        "footer",
        "synthetic_notice",
    }

    missing_payload_keys = sorted(
        required_payload_keys
        - set(render_payload)
    )

    if missing_payload_keys:
        raise KeyError(
            "Render payload TPL-09 belum lengkap: "
            f"{missing_payload_keys}"
        )

    item_count = len(
        render_payload["items"]
    )

    if not (
        TPL09_MIN_ITEMS_PER_PAGE
        <= item_count
        <= TPL09_MAX_ITEMS_PER_PAGE
    ):
        raise ValueError(
            "TPL-09 hanya mendukung 2–8 item "
            "dalam satu halaman."
        )

    language = render_payload[
        "language"
    ]

    if (
        language == "id"
        and render_payload[
            "labels"
        ]["quantity"] != "Kuantitas"
    ):
        raise RuntimeError(
            "Label quantity bahasa Indonesia "
            "wajib menggunakan 'Kuantitas'."
        )

    template_specification = (
        template_registry[
            TPL09_TEMPLATE_ID
        ]
    )

    expected_template_contract = {
        "layout_family": "compact_ledger",
        "page_size": "A4",
        "header_layout": "compact_metadata",
        "party_layout": "two_columns",
        "table_style": "ledger",
        "totals_position": "inline",
        "accent_position": "bottom",
        "density": "compact",
    }

    contract_mismatches = {
        key: {
            "expected": expected_value,
            "actual": (
                template_specification.get(
                    key
                )
            ),
        }
        for key, expected_value
        in expected_template_contract.items()
        if (
            template_specification.get(key)
            != expected_value
        )
    }

    if contract_mismatches:
        raise RuntimeError(
            "Spesifikasi layout TPL-09 "
            "tidak sesuai: "
            f"{contract_mismatches}"
        )

    page_width, page_height = (
        get_page_dimensions(
            template_specification[
                "page_size"
            ]
        )
    )

    primary_color = hex_to_rgb(
        template_specification[
            "primary_color"
        ]
    )

    accent_color = hex_to_rgb(
        template_specification[
            "accent_color"
        ]
    )

    dark_text = (
        0.12,
        0.14,
        0.19,
    )

    muted_text = (
        0.38,
        0.42,
        0.48,
    )

    border_color = (
        0.77,
        0.80,
        0.85,
    )

    light_primary = (
        0.955,
        0.960,
        0.972,
    )

    light_accent = (
        0.945,
        0.950,
        0.960,
    )

    white = (
        1.0,
        1.0,
        1.0,
    )

    output_pdf_path = Path(
        output_pdf_path
    )

    output_pdf_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_pdf_path = (
        output_pdf_path.with_name(
            f".{output_pdf_path.stem}"
            ".tmp.pdf"
        )
    )

    temporary_pdf_path.unlink(
        missing_ok=True
    )

    annotations: list[
        FieldAnnotation
    ] = []

    field_font_audit: list[
        dict[str, Any]
    ] = []

    document = pymupdf.open()

    def add_field(
        page: pymupdf.Page,
        rectangle: pymupdf.Rect,
        text: str,
        field_name: str,
        font_name: str = "helv",
        font_size: float = 7.0,
        minimum_font_size: float = 6.5,
        font_color: tuple[
            float,
            float,
            float,
        ] = dark_text,
        alignment: str = "left",
    ) -> None:
        """Menambahkan field dan mencatat audit ukuran font."""

        annotation, used_font_size = (
            insert_annotated_text(
                page=page,
                rectangle=rectangle,
                text=str(text),
                field_name=field_name,
                font_name=font_name,
                font_size=font_size,
                minimum_font_size=(
                    minimum_font_size
                ),
                font_color=font_color,
                alignment=alignment,
            )
        )

        annotations.append(
            annotation
        )

        field_font_audit.append(
            {
                "field_name": field_name,
                "requested_font_size": (
                    font_size
                ),
                "used_font_size": (
                    used_font_size
                ),
                "adjusted": (
                    used_font_size
                    < font_size
                ),
            }
        )

    try:
        page = document.new_page(
            width=page_width,
            height=page_height,
        )

        margin_x0 = 42.0
        margin_x1 = (
            page_width - 42.0
        )

        # --------------------------------------------------------
        # Header ringkas
        # --------------------------------------------------------

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                31,
                290,
                67,
            ),
            text=render_payload[
                "title"
            ],
            font_name="hebo",
            font_size=22.0,
            minimum_font_size=17.0,
            font_color=primary_color,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                71,
                290,
                86,
            ),
            text=(
                "COMPACT LEDGER · SYNTHETIC"
            ),
            font_name="hebo",
            font_size=6.7,
            minimum_font_size=6.0,
            font_color=accent_color,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                91,
                290,
                106,
            ),
            text=(
                render_payload[
                    "footer"
                ]["document_reference"]
            ),
            font_name="helv",
            font_size=6.6,
            minimum_font_size=6.0,
            font_color=muted_text,
        )

        currency_label = (
            "Mata Uang"
            if language == "id"
            else "Currency"
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                110,
                103,
                125,
            ),
            text=currency_label,
            font_name="helv",
            font_size=6.3,
            minimum_font_size=5.8,
            font_color=muted_text,
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                105,
                108,
                180,
                126,
            ),
            text=render_payload[
                "currency"
            ],
            field_name="currency",
            font_name="hebo",
            font_size=7.6,
            minimum_font_size=6.5,
            font_color=primary_color,
        )

        # --------------------------------------------------------
        # Metadata invoice kanan
        # --------------------------------------------------------

        metadata_x0 = 306.0
        metadata_x1 = margin_x1

        metadata_rows = [
            (
                "invoice_number",
                36.0,
                54.0,
            ),
            (
                "invoice_date",
                61.0,
                79.0,
            ),
            (
                "due_date",
                86.0,
                104.0,
            ),
        ]

        for (
            field_name,
            row_y0,
            row_y1,
        ) in metadata_rows:
            field_payload = (
                render_payload[
                    "metadata"
                ][field_name]
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    metadata_x0,
                    row_y0,
                    402,
                    row_y1,
                ),
                text=field_payload[
                    "label"
                ],
                font_name="helv",
                font_size=6.4,
                minimum_font_size=5.8,
                font_color=muted_text,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    404,
                    row_y0,
                    metadata_x1,
                    row_y1,
                ),
                text=field_payload[
                    "display_value"
                ],
                field_name=field_name,
                font_name="hebo",
                font_size=7.2,
                minimum_font_size=6.5,
                font_color=primary_color,
                alignment="right",
            )

        page.draw_line(
            pymupdf.Point(
                margin_x0,
                127,
            ),
            pymupdf.Point(
                margin_x1,
                127,
            ),
            color=primary_color,
            width=0.9,
            overlay=True,
        )

        # --------------------------------------------------------
        # Vendor dan buyer
        # --------------------------------------------------------

        party_cards = [
            (
                "vendor",
                render_payload[
                    "labels"
                ]["vendor"],
                margin_x0,
                291.0,
            ),
            (
                "buyer",
                render_payload[
                    "labels"
                ]["buyer"],
                304.0,
                margin_x1,
            ),
        ]

        party_y0 = 140.0
        party_y1 = 274.0

        for (
            party_type,
            party_label,
            card_x0,
            card_x1,
        ) in party_cards:
            party = render_payload[
                party_type
            ]

            page.draw_rect(
                pymupdf.Rect(
                    card_x0,
                    party_y0,
                    card_x1,
                    party_y1,
                ),
                color=border_color,
                fill=white,
                width=0.65,
                overlay=True,
            )

            page.draw_rect(
                pymupdf.Rect(
                    card_x0,
                    party_y0,
                    card_x1,
                    party_y0 + 22,
                ),
                color=None,
                fill=light_accent,
                width=0,
                overlay=True,
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 12,
                    party_y0 + 6,
                    card_x1 - 12,
                    party_y0 + 19,
                ),
                text=party_label.upper(),
                font_name="hebo",
                font_size=6.5,
                minimum_font_size=5.8,
                font_color=primary_color,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 12,
                    party_y0 + 28,
                    card_x1 - 12,
                    party_y0 + 46,
                ),
                text=party["name"],
                field_name=(
                    f"{party_type}.name"
                ),
                font_name="hebo",
                font_size=7.6,
                minimum_font_size=6.5,
                font_color=primary_color,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 12,
                    party_y0 + 49,
                    card_x1 - 12,
                    party_y0 + 84,
                ),
                text="\n".join(
                    party[
                        "address_lines"
                    ]
                ),
                field_name=(
                    f"{party_type}"
                    ".address_lines"
                ),
                font_name="helv",
                font_size=6.7,
                minimum_font_size=6.5,
                font_color=dark_text,
            )

            contact_fields = [
                (
                    "email",
                    party["email"],
                    87.0,
                    100.0,
                ),
                (
                    "phone",
                    party["phone"],
                    102.0,
                    115.0,
                ),
                (
                    "tax_identifier",
                    party[
                        "tax_identifier_display"
                    ],
                    117.0,
                    131.0,
                ),
            ]

            for (
                field_name,
                field_value,
                field_y0,
                field_y1,
            ) in contact_fields:
                add_field(
                    page=page,
                    rectangle=pymupdf.Rect(
                        card_x0 + 12,
                        party_y0 + field_y0,
                        card_x1 - 12,
                        party_y0 + field_y1,
                    ),
                    text=field_value,
                    field_name=(
                        f"{party_type}."
                        f"{field_name}"
                    ),
                    font_name="helv",
                    font_size=6.6,
                    minimum_font_size=6.5,
                    font_color=dark_text,
                )

        # --------------------------------------------------------
        # Tabel item bergaya ledger
        # --------------------------------------------------------

        table_positions = [
            margin_x0,
            67.0,
            305.0,
            360.0,
            455.0,
            margin_x1,
        ]

        table_header_y0 = 291.0
        table_header_y1 = 316.0
        row_height = 23.0

        page.draw_rect(
            pymupdf.Rect(
                table_positions[0],
                table_header_y0,
                table_positions[-1],
                table_header_y1,
            ),
            color=primary_color,
            fill=primary_color,
            width=0,
            overlay=True,
        )

        table_headers = [
            "#",
            render_payload[
                "labels"
            ]["description"],
            render_payload[
                "labels"
            ]["quantity"],
            render_payload[
                "labels"
            ]["unit_price"],
            render_payload[
                "labels"
            ]["line_total"],
        ]

        for (
            column_index,
            header_text,
        ) in enumerate(
            table_headers
        ):
            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    (
                        table_positions[
                            column_index
                        ]
                        + 3
                    ),
                    table_header_y0 + 7,
                    (
                        table_positions[
                            column_index + 1
                        ]
                        - 3
                    ),
                    table_header_y1 - 2,
                ),
                text=header_text,
                font_name="hebo",
                font_size=6.4,
                minimum_font_size=5.8,
                font_color=white,
                alignment=(
                    "left"
                    if column_index == 1
                    else "center"
                ),
            )

        for (
            item_index,
            item,
        ) in enumerate(
            render_payload["items"]
        ):
            row_y0 = (
                table_header_y1
                + item_index * row_height
            )

            row_y1 = (
                row_y0
                + row_height
            )

            if item_index % 2 == 1:
                page.draw_rect(
                    pymupdf.Rect(
                        table_positions[0],
                        row_y0,
                        table_positions[-1],
                        row_y1,
                    ),
                    color=None,
                    fill=light_primary,
                    width=0,
                    overlay=True,
                )

            page.draw_line(
                pymupdf.Point(
                    table_positions[0],
                    row_y1,
                ),
                pymupdf.Point(
                    table_positions[-1],
                    row_y1,
                ),
                color=border_color,
                width=0.55,
                overlay=True,
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    table_positions[0] + 3,
                    row_y0 + 6,
                    table_positions[1] - 3,
                    row_y1 - 2,
                ),
                text=str(
                    item[
                        "item_number"
                    ]
                ),
                font_name="helv",
                font_size=6.7,
                minimum_font_size=6.0,
                font_color=muted_text,
                alignment="center",
            )

            item_fields = [
                (
                    "description",
                    1,
                    "left",
                ),
                (
                    "quantity",
                    2,
                    "center",
                ),
                (
                    "unit_price",
                    3,
                    "right",
                ),
                (
                    "line_total",
                    4,
                    "right",
                ),
            ]

            for (
                field_name,
                column_index,
                field_alignment,
            ) in item_fields:
                add_field(
                    page=page,
                    rectangle=pymupdf.Rect(
                        (
                            table_positions[
                                column_index
                            ]
                            + 4
                        ),
                        row_y0 + 6,
                        (
                            table_positions[
                                column_index + 1
                            ]
                            - 4
                        ),
                        row_y1 - 2,
                    ),
                    text=item[
                        field_name
                    ],
                    field_name=(
                        f"items[{item_index}]"
                        f".{field_name}"
                    ),
                    font_name="helv",
                    font_size=6.8,
                    minimum_font_size=6.5,
                    font_color=dark_text,
                    alignment=(
                        field_alignment
                    ),
                )

        table_end_y = (
            table_header_y1
            + item_count * row_height
        )

        # --------------------------------------------------------
        # Ringkasan keuangan inline
        # --------------------------------------------------------

        totals_gap = 10.0
        totals_row_height = 21.0

        totals_y0 = (
            table_end_y
            + totals_gap
        )

        totals_y1 = (
            totals_y0
            + 4 * totals_row_height
        )

        table_to_totals_gap = (
            totals_y0
            - table_end_y
        )

        footer_line_y = 792.0

        if table_to_totals_gap < 10.0:
            raise RuntimeError(
                "Ringkasan keuangan bertabrakan "
                "dengan tabel."
            )

        if (
            totals_y1
            >= footer_line_y - 40.0
        ):
            raise RuntimeError(
                "Ringkasan keuangan terlalu dekat "
                "dengan footer. "
                f"Panel berakhir pada "
                f"{totals_y1:.2f} pt."
            )

        totals_x0 = 318.0
        totals_x1 = margin_x1

        summary_heading = (
            "RINGKASAN"
            if language == "id"
            else "SUMMARY"
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                totals_y0 + 4,
                totals_x0 - 20,
                totals_y0 + 20,
            ),
            text=summary_heading,
            font_name="hebo",
            font_size=6.6,
            minimum_font_size=6.0,
            font_color=accent_color,
        )

        page.draw_line(
            pymupdf.Point(
                totals_x0,
                totals_y0,
            ),
            pymupdf.Point(
                totals_x1,
                totals_y0,
            ),
            color=primary_color,
            width=0.9,
            overlay=True,
        )

        financial_order = [
            "subtotal",
            "tax",
            "discount",
            "total",
        ]

        for (
            row_index,
            field_name,
        ) in enumerate(
            financial_order
        ):
            field_payload = (
                render_payload[
                    "financials"
                ][field_name]
            )

            row_y0 = (
                totals_y0
                + row_index
                * totals_row_height
            )

            row_y1 = (
                row_y0
                + totals_row_height
            )

            is_total = (
                field_name == "total"
            )

            if is_total:
                page.draw_rect(
                    pymupdf.Rect(
                        totals_x0,
                        row_y0,
                        totals_x1,
                        row_y1,
                    ),
                    color=primary_color,
                    fill=primary_color,
                    width=0,
                    overlay=True,
                )
            else:
                if row_index % 2 == 1:
                    page.draw_rect(
                        pymupdf.Rect(
                            totals_x0,
                            row_y0,
                            totals_x1,
                            row_y1,
                        ),
                        color=None,
                        fill=light_primary,
                        width=0,
                        overlay=True,
                    )

                page.draw_line(
                    pymupdf.Point(
                        totals_x0,
                        row_y1,
                    ),
                    pymupdf.Point(
                        totals_x1,
                        row_y1,
                    ),
                    color=border_color,
                    width=0.5,
                    overlay=True,
                )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    totals_x0 + 9,
                    row_y0 + 5,
                    totals_x0 + 91,
                    row_y1 - 2,
                ),
                text=field_payload[
                    "label"
                ],
                font_name=(
                    "hebo"
                    if is_total
                    else "helv"
                ),
                font_size=(
                    7.0
                    if is_total
                    else 6.7
                ),
                minimum_font_size=6.0,
                font_color=(
                    white
                    if is_total
                    else muted_text
                ),
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    totals_x0 + 93,
                    row_y0 + 5,
                    totals_x1 - 9,
                    row_y1 - 2,
                ),
                text=field_payload[
                    "display_value"
                ],
                field_name=(
                    f"financials."
                    f"{field_name}"
                ),
                font_name=(
                    "hebo"
                    if is_total
                    else "helv"
                ),
                font_size=(
                    7.6
                    if is_total
                    else 6.9
                ),
                minimum_font_size=6.5,
                font_color=(
                    white
                    if is_total
                    else dark_text
                ),
                alignment="right",
            )

        # --------------------------------------------------------
        # Footer dan aksen bawah
        # --------------------------------------------------------

        page.draw_line(
            pymupdf.Point(
                margin_x0,
                footer_line_y,
            ),
            pymupdf.Point(
                margin_x1,
                footer_line_y,
            ),
            color=border_color,
            width=0.7,
            overlay=True,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                799,
                margin_x1,
                813,
            ),
            text=(
                render_payload[
                    "footer"
                ]["document_reference"]
            ),
            font_name="helv",
            font_size=6.4,
            minimum_font_size=6.0,
            font_color=muted_text,
            alignment="center",
        )

        notice_display_text = (
            normalize_pdf_notice(
                render_payload[
                    "synthetic_notice"
                ]
            )
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                818,
                margin_x1,
                835,
            ),
            text=notice_display_text,
            field_name=(
                "document.synthetic_notice"
            ),
            font_name="hebo",
            font_size=6.6,
            minimum_font_size=6.5,
            font_color=primary_color,
            alignment="center",
        )

        page.draw_rect(
            pymupdf.Rect(
                0,
                page_height - 7,
                page_width,
                page_height,
            ),
            color=accent_color,
            fill=accent_color,
            width=0,
            overlay=True,
        )

        # --------------------------------------------------------
        # Pre-save controls
        # --------------------------------------------------------

        annotation_names = [
            annotation.field_name
            for annotation in annotations
        ]

        if (
            len(annotation_names)
            != len(set(annotation_names))
        ):
            raise RuntimeError(
                "Ditemukan nama anotasi duplikat."
            )

        expected_annotation_count = (
            19
            + 4 * item_count
        )

        actual_annotation_count = len(
            annotations
        )

        if (
            actual_annotation_count
            != expected_annotation_count
        ):
            raise RuntimeError(
                "Jumlah anotasi tidak sesuai. "
                f"Expected: "
                f"{expected_annotation_count}; "
                f"actual: "
                f"{actual_annotation_count}."
            )

        if not field_font_audit:
            raise RuntimeError(
                "Audit font field tidak boleh kosong."
            )

        minimum_used_font_size = min(
            row["used_font_size"]
            for row in field_font_audit
        )

        if (
            minimum_used_font_size
            < 6.5
        ):
            raise RuntimeError(
                "Ukuran font field berada "
                "di bawah 6.5 pt."
            )

        document.save(
            str(temporary_pdf_path),
            garbage=4,
            deflate=True,
        )

    except Exception:
        temporary_pdf_path.unlink(
            missing_ok=True
        )
        raise

    finally:
        document.close()

    if (
        not temporary_pdf_path.is_file()
        or temporary_pdf_path.stat().st_size
        == 0
    ):
        temporary_pdf_path.unlink(
            missing_ok=True
        )

        raise RuntimeError(
            "PDF sementara TPL-09 "
            "gagal dibuat."
        )

    os.replace(
        temporary_pdf_path,
        output_pdf_path,
    )

    rendering_metadata = {
        "renderer_version": (
            RENDER_ENGINE_VERSION
        ),
        "template_renderer_version": (
            TPL09_RENDERER_VERSION
        ),
        "template_id": (
            TPL09_TEMPLATE_ID
        ),
        "layout_family": (
            template_specification[
                "layout_family"
            ]
        ),
        "page_size": (
            template_specification[
                "page_size"
            ]
        ),
        "page_width_points": round(
            page_width,
            3,
        ),
        "page_height_points": round(
            page_height,
            3,
        ),
        "page_count": 1,
        "annotation_type": (
            "field_region"
        ),
        "annotation_count": len(
            annotations
        ),
        "minimum_field_font_size": min(
            row["used_font_size"]
            for row in field_font_audit
        ),
        "font_adjustment_count": int(
            sum(
                bool(row["adjusted"])
                for row in field_font_audit
            )
        ),
        "layout_metrics": {
            "item_count": item_count,
            "table_end_y_points": round(
                table_end_y,
                3,
            ),
            "totals_panel_y0_points": round(
                totals_y0,
                3,
            ),
            "totals_panel_y1_points": round(
                totals_y1,
                3,
            ),
            "table_to_totals_gap_points": round(
                table_to_totals_gap,
                3,
            ),
            "footer_line_y_points": (
                footer_line_y
            ),
        },
    }

    return (
        annotations,
        rendering_metadata,
    )


# ================================================================
# Pilih prototype pertama TPL-09
# ================================================================

tpl09_candidate_ids = sorted(
    canonical_id
    for canonical_id, payload
    in invoice_render_payloads.items()
    if (
        isinstance(payload, dict)
        and payload.get("template_id")
        == TPL09_TEMPLATE_ID
    )
)

if len(tpl09_candidate_ids) != 20:
    raise RuntimeError(
        "TPL-09 harus memiliki tepat "
        "20 presentation payload. "
        f"Actual: {len(tpl09_candidate_ids)}."
    )

prototype_09_canonical_id = (
    tpl09_candidate_ids[0]
)

prototype_09_render_payload = (
    invoice_render_payloads[
        prototype_09_canonical_id
    ]
)

prototype_09_document_id = (
    prototype_09_render_payload[
        "document_id"
    ]
)

prototype_09_canonical_payload = (
    load_tpl09_canonical_payload(
        prototype_09_canonical_id
    )
)


# ================================================================
# Path artifact prototype
# ================================================================

prototype_09_pdf_path = (
    TPL09_PROTOTYPE_ROOT
    / (
        f"{prototype_09_document_id}"
        "_TPL-09_prototype.pdf"
    )
)

prototype_09_png_path = (
    TPL09_PROTOTYPE_ROOT
    / (
        f"{prototype_09_document_id}"
        "_TPL-09_preview.png"
    )
)

prototype_09_ground_truth_path = (
    TPL09_PROTOTYPE_ROOT
    / (
        f"{prototype_09_document_id}"
        "_TPL-09_ground_truth.json"
    )
)


# ================================================================
# Render prototype
# ================================================================

(
    prototype_09_annotations,
    prototype_09_rendering_metadata,
) = render_template_09(
    render_payload=(
        prototype_09_render_payload
    ),
    output_pdf_path=(
        prototype_09_pdf_path
    ),
)


# ================================================================
# Simpan ground truth
# ================================================================

prototype_09_ground_truth = {
    "schema_version": "1.0.0",
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "document": {
        "document_id": (
            prototype_09_document_id
        ),
        "canonical_invoice_id": (
            prototype_09_canonical_id
        ),
        "split": (
            prototype_09_render_payload[
                "split"
            ]
        ),
        "template_id": (
            TPL09_TEMPLATE_ID
        ),
        "language": (
            prototype_09_render_payload[
                "language"
            ]
        ),
        "currency": (
            prototype_09_render_payload[
                "currency"
            ]
        ),
    },
    "canonical": (
        prototype_09_canonical_payload
    ),
    "rendering": (
        prototype_09_rendering_metadata
    ),
    "annotations": [
        annotation_to_dict(
            annotation
        )
        for annotation
        in prototype_09_annotations
    ],
}

write_json_atomically(
    prototype_09_ground_truth_path,
    prototype_09_ground_truth,
)


# ================================================================
# Buka ulang PDF dan buat preview
# ================================================================

temporary_png_path = (
    prototype_09_png_path.with_name(
        f".{prototype_09_png_path.stem}"
        ".tmp.png"
    )
)

temporary_png_path.unlink(
    missing_ok=True
)

try:
    with pymupdf.open(
        str(prototype_09_pdf_path)
    ) as prototype_document:
        if (
            prototype_document.page_count
            != 1
        ):
            raise RuntimeError(
                "Prototype TPL-09 harus "
                "memiliki tepat satu halaman."
            )

        prototype_page = (
            prototype_document[0]
        )

        extracted_text_09 = (
            prototype_page.get_text(
                "text"
            )
        )

        prototype_pixmap_09 = (
            prototype_page.get_pixmap(
                matrix=pymupdf.Matrix(
                    150 / 72,
                    150 / 72,
                ),
                alpha=False,
            )
        )

        prototype_pixmap_09.save(
            str(temporary_png_path)
        )

    if (
        not temporary_png_path.is_file()
        or temporary_png_path.stat().st_size
        == 0
    ):
        raise RuntimeError(
            "Preview sementara TPL-09 "
            "gagal dibuat."
        )

    os.replace(
        temporary_png_path,
        prototype_09_png_path,
    )

except Exception:
    temporary_png_path.unlink(
        missing_ok=True
    )
    raise


# ================================================================
# Post-render controls
# ================================================================

required_text_fragments_09 = [
    (
        prototype_09_render_payload[
            "metadata"
        ]["invoice_number"][
            "display_value"
        ]
    ),
    (
        prototype_09_render_payload[
            "vendor"
        ]["name"]
    ),
    (
        prototype_09_render_payload[
            "buyer"
        ]["name"]
    ),
    (
        prototype_09_render_payload[
            "labels"
        ]["quantity"]
    ),
    (
        prototype_09_render_payload[
            "financials"
        ]["total"]["display_value"]
    ),
    normalize_pdf_notice(
        prototype_09_render_payload[
            "synthetic_notice"
        ]
    ),
]

missing_text_fragments_09 = [
    text_fragment
    for text_fragment
    in required_text_fragments_09
    if (
        text_fragment
        not in extracted_text_09
    )
]

if missing_text_fragments_09:
    raise RuntimeError(
        "Teks penting TPL-09 tidak ditemukan "
        "pada hasil ekstraksi PDF: "
        f"{missing_text_fragments_09}"
    )

annotation_names_09 = {
    annotation.field_name
    for annotation
    in prototype_09_annotations
}

required_annotation_names_09 = {
    "vendor.name",
    "vendor.address_lines",
    "vendor.email",
    "vendor.phone",
    "vendor.tax_identifier",
    "buyer.name",
    "buyer.address_lines",
    "buyer.email",
    "buyer.phone",
    "buyer.tax_identifier",
    "invoice_number",
    "invoice_date",
    "due_date",
    "currency",
    "financials.subtotal",
    "financials.tax",
    "financials.discount",
    "financials.total",
    "document.synthetic_notice",
}

missing_annotations_09 = sorted(
    required_annotation_names_09
    - annotation_names_09
)

if missing_annotations_09:
    raise RuntimeError(
        "Anotasi wajib TPL-09 "
        "tidak tersedia: "
        f"{missing_annotations_09}"
    )

layout_metrics_09 = (
    prototype_09_rendering_metadata[
        "layout_metrics"
    ]
)

prototype_09_summary = pd.DataFrame(
    [
        {
            "control": "pdf_created",
            "actual": (
                prototype_09_pdf_path
                .is_file()
            ),
            "status": "VALID",
        },
        {
            "control": (
                "ground_truth_created"
            ),
            "actual": (
                prototype_09_ground_truth_path
                .is_file()
            ),
            "status": "VALID",
        },
        {
            "control": "annotation_count",
            "actual": len(
                prototype_09_annotations
            ),
            "status": "VALID",
        },
        {
            "control": "item_count",
            "actual": (
                layout_metrics_09[
                    "item_count"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": (
                "minimum_field_font"
            ),
            "actual": (
                prototype_09_rendering_metadata[
                    "minimum_field_font_size"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "font_adjustments",
            "actual": (
                prototype_09_rendering_metadata[
                    "font_adjustment_count"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": (
                "table_to_totals_gap"
            ),
            "actual": (
                layout_metrics_09[
                    "table_to_totals_gap_points"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": (
                "missing_required_text"
            ),
            "actual": len(
                missing_text_fragments_09
            ),
            "status": "VALID",
        },
    ]
)

display(
    prototype_09_summary
)

with PILImage.open(
    prototype_09_png_path
) as prototype_image:
    display(
        prototype_image.copy()
    )


# ================================================================
# Ringkasan
# ================================================================

print(
    f"Canonical ID       : "
    f"{prototype_09_canonical_id}"
)

print(
    f"Document ID        : "
    f"{prototype_09_document_id}"
)

print(
    f"Prototype PDF      : "
    f"{prototype_09_pdf_path}"
)

print(
    f"Prototype preview  : "
    f"{prototype_09_png_path}"
)

print(
    f"Ground truth       : "
    f"{prototype_09_ground_truth_path}"
)

print(
    f"Annotations        : "
    f"{len(prototype_09_annotations)}"
)

print(
    "Minimum field font : "
    f"{prototype_09_rendering_metadata['minimum_field_font_size']}"
)

print(
    "Adjusted fields    : "
    f"{prototype_09_rendering_metadata['font_adjustment_count']}"
)

print(
    "Table-total gap    : "
    f"{layout_metrics_09['table_to_totals_gap_points']} pt"
)

print(
    f"Renderer version   : "
    f"{TPL09_RENDERER_VERSION}"
)

print()

print(
    "✅ Prototype TPL-09 berhasil dibuat dan "
    "lolos pemeriksaan teknis awal."
)

In [ ]:
# ================================================================
# CELL 65 — FINAL
# Audit QA teknis prototype TPL-09
# ================================================================

from pathlib import Path
from typing import Any

import hashlib
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd
import pymupdf
from PIL import Image as PILImage


# ================================================================
# Validasi dependency runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "TPL09_PROTOTYPE_ROOT",
    "TPL09_TEMPLATE_ID",
    "prototype_09_canonical_id",
    "prototype_09_document_id",
    "prototype_09_pdf_path",
    "prototype_09_png_path",
    "prototype_09_ground_truth_path",
    "prototype_09_render_payload",
    "prototype_09_rendering_metadata",
    "prototype_09_annotations",
    "write_json_atomically",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 65 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali Cell 64 versi final."
    )


# ================================================================
# Konfigurasi audit
# ================================================================

QA_SCHEMA_VERSION = "1.0.0"
QA_AUDITOR_VERSION = "1.0.0"

MINIMUM_FIELD_FONT_SIZE = 6.5
MAXIMUM_FONT_ADJUSTMENT_RATIO = 0.20

MINIMUM_RASTER_CONTENT_RATIO = 0.01
MAXIMUM_RASTER_CONTENT_RATIO = 0.40
MINIMUM_RASTER_CONTRAST = 10.0

SEVERE_OVERLAP_THRESHOLD = 0.50


# ================================================================
# Normalisasi path
# ================================================================

prototype_09_pdf_path = Path(
    prototype_09_pdf_path
)

prototype_09_png_path = Path(
    prototype_09_png_path
)

prototype_09_ground_truth_path = Path(
    prototype_09_ground_truth_path
)

prototype_09_qa_report_path = (
    Path(TPL09_PROTOTYPE_ROOT)
    / (
        f"{prototype_09_document_id}"
        "_TPL-09_qa_report.json"
    )
)

required_files = [
    prototype_09_pdf_path,
    prototype_09_png_path,
    prototype_09_ground_truth_path,
]

missing_files = [
    str(file_path)
    for file_path in required_files
    if not file_path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Artifact TPL-09 belum lengkap: "
        f"{missing_files}"
    )


# ================================================================
# Helper umum
# ================================================================

def sha256_file(
    file_path: Path,
) -> str:
    """Menghitung SHA-256 file."""

    digest = hashlib.sha256()

    with Path(file_path).open(
        mode="rb",
    ) as input_file:
        for chunk in iter(
            lambda: input_file.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def normalize_text(
    value: Any,
) -> str:
    """Menormalisasi teks untuk perbandingan audit."""

    normalized_value = unicodedata.normalize(
        "NFKC",
        str(value or ""),
    )

    normalized_value = (
        normalized_value
        .replace("\u2014", "-")
        .replace("\u2013", "-")
        .replace("\u00b7", "-")
    )

    normalized_value = re.sub(
        r"\s+",
        " ",
        normalized_value,
    )

    return (
        normalized_value
        .strip()
        .casefold()
    )


def normalize_notice_for_pdf(
    value: Any,
) -> str:
    """Menyesuaikan notice dengan teks yang ditulis ke PDF."""

    return (
        str(value)
        .replace("\u2014", "\u00b7")
        .replace("\u2013", "-")
    )


def first_present(
    mapping: dict[str, Any],
    candidate_keys: tuple[str, ...],
) -> Any:
    """Mengambil key pertama yang tersedia."""

    for key in candidate_keys:
        if key in mapping:
            return mapping[key]

    return None


def annotation_field_name(
    annotation: dict[str, Any],
) -> str:
    """Mengambil nama field dari variasi schema anotasi."""

    value = first_present(
        annotation,
        (
            "field_name",
            "field_path",
            "field",
            "path",
            "name",
        ),
    )

    if value is None:
        raise KeyError(
            "Nama field tidak ditemukan pada anotasi. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return str(value)


def annotation_text(
    annotation: dict[str, Any],
) -> str:
    """Mengambil nilai teks dari variasi schema anotasi."""

    value = first_present(
        annotation,
        (
            "text",
            "value",
            "display_value",
            "field_value",
            "raw_value",
            "raw_text",
            "text_value",
            "content",
        ),
    )

    if isinstance(value, dict):
        value = first_present(
            value,
            (
                "text",
                "value",
                "display_value",
                "raw",
            ),
        )

    if value is None:
        raise KeyError(
            "Nilai teks tidak ditemukan pada anotasi. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return str(value)


def bbox_from_value(
    value: Any,
) -> tuple[
    float,
    float,
    float,
    float,
] | None:
    """Mengubah beberapa variasi bounding box ke x0,y0,x1,y1."""

    if isinstance(
        value,
        (list, tuple),
    ):
        if len(value) != 4:
            return None

        try:
            return tuple(
                float(number)
                for number in value
            )
        except (
            TypeError,
            ValueError,
        ):
            return None

    if not isinstance(
        value,
        dict,
    ):
        return None

    direct_key_sets = [
        (
            "x0",
            "y0",
            "x1",
            "y1",
        ),
        (
            "left",
            "top",
            "right",
            "bottom",
        ),
    ]

    for key_set in direct_key_sets:
        if all(
            key in value
            for key in key_set
        ):
            try:
                return tuple(
                    float(value[key])
                    for key in key_set
                )
            except (
                TypeError,
                ValueError,
            ):
                return None

    for size_keys in (
        (
            "x",
            "y",
            "width",
            "height",
        ),
        (
            "x",
            "y",
            "w",
            "h",
        ),
    ):
        if all(
            key in value
            for key in size_keys
        ):
            try:
                x0 = float(value["x"])
                y0 = float(value["y"])
                width = float(
                    value[size_keys[2]]
                )
                height = float(
                    value[size_keys[3]]
                )

                return (
                    x0,
                    y0,
                    x0 + width,
                    y0 + height,
                )
            except (
                TypeError,
                ValueError,
            ):
                return None

    return None


def annotation_bbox(
    annotation: dict[str, Any],
) -> tuple[
    float,
    float,
    float,
    float,
]:
    """Mengambil bounding box anotasi."""

    candidate = first_present(
        annotation,
        (
            "bbox",
            "bbox_points",
            "bbox_pt",
            "bbox_pdf",
            "bounding_box",
            "rectangle",
            "rect",
            "coordinates",
            "coordinates_points",
        ),
    )

    if candidate is None:
        geometry = annotation.get(
            "geometry"
        )

        if isinstance(
            geometry,
            dict,
        ):
            candidate = first_present(
                geometry,
                (
                    "bbox",
                    "bounding_box",
                    "rectangle",
                ),
            )

    bbox = bbox_from_value(
        candidate
    )

    if bbox is None:
        raise KeyError(
            "Bounding box tidak ditemukan "
            "atau tidak valid. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return bbox


def rectangle_area(
    bbox: tuple[
        float,
        float,
        float,
        float,
    ],
) -> float:
    """Menghitung luas bounding box."""

    x0, y0, x1, y1 = bbox

    return (
        max(
            0.0,
            x1 - x0,
        )
        * max(
            0.0,
            y1 - y0,
        )
    )


def intersection_area(
    first_bbox: tuple[
        float,
        float,
        float,
        float,
    ],
    second_bbox: tuple[
        float,
        float,
        float,
        float,
    ],
) -> float:
    """Menghitung luas perpotongan dua bounding box."""

    (
        first_x0,
        first_y0,
        first_x1,
        first_y1,
    ) = first_bbox

    (
        second_x0,
        second_y0,
        second_x1,
        second_y1,
    ) = second_bbox

    intersection_width = max(
        0.0,
        min(
            first_x1,
            second_x1,
        )
        - max(
            first_x0,
            second_x0,
        ),
    )

    intersection_height = max(
        0.0,
        min(
            first_y1,
            second_y1,
        )
        - max(
            first_y0,
            second_y0,
        ),
    )

    return (
        intersection_width
        * intersection_height
    )


def bbox_intersects_word(
    annotation_box: tuple[
        float,
        float,
        float,
        float,
    ],
    word_box: tuple[
        float,
        float,
        float,
        float,
    ],
) -> bool:
    """Memeriksa apakah anotasi berpotongan dengan kata PDF."""

    return (
        intersection_area(
            annotation_box,
            word_box,
        )
        > 0.0
    )


# ================================================================
# Muat ground truth
# ================================================================

with prototype_09_ground_truth_path.open(
    mode="r",
    encoding="utf-8",
) as ground_truth_file:
    ground_truth = json.load(
        ground_truth_file
    )

ground_truth_document = (
    ground_truth.get(
        "document",
        {},
    )
)

if (
    ground_truth_document.get(
        "document_id"
    )
    != prototype_09_document_id
):
    raise RuntimeError(
        "Document ID ground truth "
        "tidak sesuai."
    )

if (
    ground_truth_document.get(
        "template_id"
    )
    != TPL09_TEMPLATE_ID
):
    raise RuntimeError(
        "Ground truth bukan milik TPL-09."
    )

annotation_records = (
    ground_truth.get(
        "annotations",
        [],
    )
)

if not isinstance(
    annotation_records,
    list,
):
    raise TypeError(
        "ground_truth.annotations "
        "wajib berupa list."
    )

if not annotation_records:
    raise RuntimeError(
        "Ground truth tidak memiliki anotasi."
    )


# ================================================================
# Expected field mapping
# ================================================================

render_payload = (
    prototype_09_render_payload
)

expected_values: dict[
    str,
    str,
] = {
    "invoice_number": str(
        render_payload[
            "metadata"
        ]["invoice_number"][
            "display_value"
        ]
    ),
    "invoice_date": str(
        render_payload[
            "metadata"
        ]["invoice_date"][
            "display_value"
        ]
    ),
    "due_date": str(
        render_payload[
            "metadata"
        ]["due_date"][
            "display_value"
        ]
    ),
    "currency": str(
        render_payload[
            "currency"
        ]
    ),
}

for party_type in (
    "vendor",
    "buyer",
):
    party = render_payload[
        party_type
    ]

    expected_values.update(
        {
            (
                f"{party_type}.name"
            ): str(
                party["name"]
            ),
            (
                f"{party_type}"
                ".address_lines"
            ): "\n".join(
                party[
                    "address_lines"
                ]
            ),
            (
                f"{party_type}.email"
            ): str(
                party["email"]
            ),
            (
                f"{party_type}.phone"
            ): str(
                party["phone"]
            ),
            (
                f"{party_type}"
                ".tax_identifier"
            ): str(
                party[
                    "tax_identifier_display"
                ]
            ),
        }
    )

for (
    item_index,
    item,
) in enumerate(
    render_payload["items"]
):
    for field_name in (
        "description",
        "quantity",
        "unit_price",
        "line_total",
    ):
        expected_values[
            (
                f"items[{item_index}]"
                f".{field_name}"
            )
        ] = str(
            item[field_name]
        )

for field_name in (
    "subtotal",
    "tax",
    "discount",
    "total",
):
    expected_values[
        f"financials.{field_name}"
    ] = str(
        render_payload[
            "financials"
        ][field_name][
            "display_value"
        ]
    )

expected_values[
    "document.synthetic_notice"
] = normalize_notice_for_pdf(
    render_payload[
        "synthetic_notice"
    ]
)

expected_annotation_count = len(
    expected_values
)


# ================================================================
# Parse annotations
# ================================================================

parsed_annotations: list[
    dict[str, Any]
] = []

annotation_parse_failures: list[
    dict[str, Any]
] = []

for (
    annotation_index,
    annotation,
) in enumerate(
    annotation_records
):
    try:
        if not isinstance(
            annotation,
            dict,
        ):
            raise TypeError(
                "Anotasi wajib berupa dictionary."
            )

        parsed_annotations.append(
            {
                "index": annotation_index,
                "field_name": (
                    annotation_field_name(
                        annotation
                    )
                ),
                "text": (
                    annotation_text(
                        annotation
                    )
                ),
                "bbox": (
                    annotation_bbox(
                        annotation
                    )
                ),
            }
        )

    except (
        KeyError,
        TypeError,
        ValueError,
    ) as error:
        annotation_parse_failures.append(
            {
                "index": annotation_index,
                "error": str(error),
                "keys": (
                    sorted(
                        annotation.keys()
                    )
                    if isinstance(
                        annotation,
                        dict,
                    )
                    else []
                ),
            }
        )

if annotation_parse_failures:
    raise RuntimeError(
        "Schema anotasi tidak dapat diproses: "
        f"{annotation_parse_failures[:3]}"
    )

annotation_names = [
    annotation["field_name"]
    for annotation
    in parsed_annotations
]

annotation_by_name = {
    annotation["field_name"]: annotation
    for annotation
    in parsed_annotations
}

duplicate_annotation_count = (
    len(annotation_names)
    - len(set(annotation_names))
)

missing_required_annotations = sorted(
    set(expected_values)
    - set(annotation_names)
)


# ================================================================
# Audit PDF
# ================================================================

with pymupdf.open(
    str(prototype_09_pdf_path)
) as pdf_document:
    pdf_page_count = (
        pdf_document.page_count
    )

    if pdf_page_count < 1:
        raise RuntimeError(
            "PDF TPL-09 tidak memiliki halaman."
        )

    first_page = pdf_document[0]
    page_rectangle = first_page.rect

    extracted_text = (
        first_page.get_text(
            "text"
        )
    )

    extracted_words = (
        first_page.get_text(
            "words"
        )
    )

page_width = float(
    page_rectangle.width
)

page_height = float(
    page_rectangle.height
)

word_boxes = [
    (
        float(word[0]),
        float(word[1]),
        float(word[2]),
        float(word[3]),
    )
    for word in extracted_words
    if (
        len(word) >= 5
        and str(word[4]).strip()
    )
]


# ================================================================
# Audit bounding box dan word coverage
# ================================================================

invalid_bounding_boxes: list[
    str
] = []

annotations_without_words: list[
    str
] = []

for annotation in parsed_annotations:
    field_name = annotation[
        "field_name"
    ]

    x0, y0, x1, y1 = (
        annotation["bbox"]
    )

    coordinates_are_finite = all(
        math.isfinite(value)
        for value in (
            x0,
            y0,
            x1,
            y1,
        )
    )

    bbox_is_valid = (
        coordinates_are_finite
        and x0 >= 0.0
        and y0 >= 0.0
        and x1 <= page_width
        and y1 <= page_height
        and x1 > x0
        and y1 > y0
    )

    if not bbox_is_valid:
        invalid_bounding_boxes.append(
            field_name
        )
        continue

    contains_word = any(
        bbox_intersects_word(
            annotation["bbox"],
            word_box,
        )
        for word_box in word_boxes
    )

    if not contains_word:
        annotations_without_words.append(
            field_name
        )


# ================================================================
# Audit nilai anotasi
# ================================================================

normalization_mismatches: list[
    dict[str, str]
] = []

for (
    field_name,
    expected_value,
) in expected_values.items():
    annotation = (
        annotation_by_name.get(
            field_name
        )
    )

    if annotation is None:
        continue

    actual_value = annotation[
        "text"
    ]

    if (
        normalize_text(actual_value)
        != normalize_text(
            expected_value
        )
    ):
        normalization_mismatches.append(
            {
                "field_name": field_name,
                "expected": expected_value,
                "actual": actual_value,
            }
        )

normalized_extracted_text = (
    normalize_text(
        extracted_text
    )
)

missing_extracted_values: list[
    str
] = []

for annotation in parsed_annotations:
    normalized_value = (
        normalize_text(
            annotation["text"]
        )
    )

    if (
        normalized_value
        and normalized_value
        not in normalized_extracted_text
    ):
        missing_extracted_values.append(
            annotation["field_name"]
        )


# ================================================================
# Audit overlap anotasi
# ================================================================

severe_annotation_overlaps: list[
    dict[str, Any]
] = []

for first_index in range(
    len(parsed_annotations)
):
    first_annotation = (
        parsed_annotations[
            first_index
        ]
    )

    first_area = rectangle_area(
        first_annotation["bbox"]
    )

    if first_area <= 0.0:
        continue

    for second_index in range(
        first_index + 1,
        len(parsed_annotations),
    ):
        second_annotation = (
            parsed_annotations[
                second_index
            ]
        )

        second_area = rectangle_area(
            second_annotation["bbox"]
        )

        if second_area <= 0.0:
            continue

        overlap_area = intersection_area(
            first_annotation["bbox"],
            second_annotation["bbox"],
        )

        overlap_ratio = (
            overlap_area
            / min(
                first_area,
                second_area,
            )
        )

        if (
            overlap_ratio
            >= SEVERE_OVERLAP_THRESHOLD
        ):
            severe_annotation_overlaps.append(
                {
                    "first": (
                        first_annotation[
                            "field_name"
                        ]
                    ),
                    "second": (
                        second_annotation[
                            "field_name"
                        ]
                    ),
                    "overlap_ratio": round(
                        overlap_ratio,
                        6,
                    ),
                }
            )


# ================================================================
# Audit font
# ================================================================

minimum_field_font_size = float(
    prototype_09_rendering_metadata[
        "minimum_field_font_size"
    ]
)

font_adjustment_count = int(
    prototype_09_rendering_metadata[
        "font_adjustment_count"
    ]
)

font_adjustment_ratio = (
    font_adjustment_count
    / max(
        1,
        len(parsed_annotations),
    )
)


# ================================================================
# Audit raster preview
# ================================================================

with PILImage.open(
    prototype_09_png_path
) as preview_image:
    grayscale_array = np.asarray(
        preview_image.convert("L"),
        dtype=np.float32,
    ).copy()

if grayscale_array.size == 0:
    raise RuntimeError(
        "Preview PNG tidak memiliki piksel."
    )

raster_content_ratio = float(
    np.mean(
        grayscale_array < 245.0
    )
)

raster_contrast = float(
    np.std(
        grayscale_array
    )
)


# ================================================================
# Susun hasil pemeriksaan
# ================================================================

checks = [
    {
        "control": "pdf_page_count",
        "expected": 1,
        "actual": pdf_page_count,
        "valid": (
            pdf_page_count == 1
        ),
    },
    {
        "control": "annotation_count",
        "expected": (
            expected_annotation_count
        ),
        "actual": len(
            parsed_annotations
        ),
        "valid": (
            len(parsed_annotations)
            == expected_annotation_count
        ),
    },
    {
        "control": (
            "duplicate_annotations"
        ),
        "expected": 0,
        "actual": (
            duplicate_annotation_count
        ),
        "valid": (
            duplicate_annotation_count
            == 0
        ),
    },
    {
        "control": (
            "missing_required_annotations"
        ),
        "expected": 0,
        "actual": len(
            missing_required_annotations
        ),
        "valid": (
            not missing_required_annotations
        ),
    },
    {
        "control": (
            "invalid_bounding_boxes"
        ),
        "expected": 0,
        "actual": len(
            invalid_bounding_boxes
        ),
        "valid": (
            not invalid_bounding_boxes
        ),
    },
    {
        "control": (
            "normalization_mismatches"
        ),
        "expected": 0,
        "actual": len(
            normalization_mismatches
        ),
        "valid": (
            not normalization_mismatches
        ),
    },
    {
        "control": (
            "annotations_without_words"
        ),
        "expected": 0,
        "actual": len(
            annotations_without_words
        ),
        "valid": (
            not annotations_without_words
        ),
    },
    {
        "control": (
            "severe_annotation_overlaps"
        ),
        "expected": 0,
        "actual": len(
            severe_annotation_overlaps
        ),
        "valid": (
            not severe_annotation_overlaps
        ),
    },
    {
        "control": (
            "missing_extracted_values"
        ),
        "expected": 0,
        "actual": len(
            missing_extracted_values
        ),
        "valid": (
            not missing_extracted_values
        ),
    },
    {
        "control": (
            "minimum_field_font_size"
        ),
        "expected": ">=6.5",
        "actual": round(
            minimum_field_font_size,
            4,
        ),
        "valid": (
            minimum_field_font_size
            >= MINIMUM_FIELD_FONT_SIZE
        ),
    },
    {
        "control": (
            "font_adjustment_ratio"
        ),
        "expected": "<=0.20",
        "actual": round(
            font_adjustment_ratio,
            4,
        ),
        "valid": (
            font_adjustment_ratio
            <= MAXIMUM_FONT_ADJUSTMENT_RATIO
        ),
    },
    {
        "control": (
            "raster_content_ratio"
        ),
        "expected": "0.01–0.40",
        "actual": round(
            raster_content_ratio,
            4,
        ),
        "valid": (
            MINIMUM_RASTER_CONTENT_RATIO
            <= raster_content_ratio
            <= MAXIMUM_RASTER_CONTENT_RATIO
        ),
    },
    {
        "control": "raster_contrast",
        "expected": ">=10.0",
        "actual": round(
            raster_contrast,
            4,
        ),
        "valid": (
            raster_contrast
            >= MINIMUM_RASTER_CONTRAST
        ),
    },
]

for check in checks:
    check["status"] = (
        "VALID"
        if check["valid"]
        else "INVALID"
    )

failed_checks = [
    check
    for check in checks
    if not check["valid"]
]

qa_status = (
    "PASSED"
    if not failed_checks
    else "FAILED"
)


# ================================================================
# Susun dan simpan QA report
# ================================================================

qa_report = {
    "schema_version": (
        QA_SCHEMA_VERSION
    ),
    "dataset_id": (
        "SYNTHETIC-INVOICE-V1"
    ),
    "document_id": (
        prototype_09_document_id
    ),
    "canonical_invoice_id": (
        prototype_09_canonical_id
    ),
    "template_id": (
        TPL09_TEMPLATE_ID
    ),
    "status": qa_status,
    "manual_visual_review": (
        "PENDING"
    ),
    "metrics": {
        "auditor_version": (
            QA_AUDITOR_VERSION
        ),
        "pdf_page_count": (
            pdf_page_count
        ),
        "annotation_count": len(
            parsed_annotations
        ),
        "expected_annotation_count": (
            expected_annotation_count
        ),
        "duplicate_annotation_count": (
            duplicate_annotation_count
        ),
        "missing_required_annotation_count": len(
            missing_required_annotations
        ),
        "invalid_bounding_box_count": len(
            invalid_bounding_boxes
        ),
        "normalization_mismatch_count": len(
            normalization_mismatches
        ),
        "annotations_without_words_count": len(
            annotations_without_words
        ),
        "severe_annotation_overlap_count": len(
            severe_annotation_overlaps
        ),
        "missing_extracted_value_count": len(
            missing_extracted_values
        ),
        "minimum_field_font_size": round(
            minimum_field_font_size,
            6,
        ),
        "font_adjustment_count": (
            font_adjustment_count
        ),
        "font_adjustment_ratio": round(
            font_adjustment_ratio,
            6,
        ),
        "raster_content_ratio": round(
            raster_content_ratio,
            6,
        ),
        "raster_contrast": round(
            raster_contrast,
            6,
        ),
    },
    "failures": {
        "failed_checks": [
            check["control"]
            for check in failed_checks
        ],
        "missing_required_annotations": (
            missing_required_annotations
        ),
        "invalid_bounding_boxes": (
            invalid_bounding_boxes
        ),
        "normalization_mismatches": (
            normalization_mismatches
        ),
        "annotations_without_words": (
            annotations_without_words
        ),
        "severe_annotation_overlaps": (
            severe_annotation_overlaps
        ),
        "missing_extracted_values": (
            missing_extracted_values
        ),
    },
    "checksums_sha256": {
        "prototype_pdf": sha256_file(
            prototype_09_pdf_path
        ),
        "preview": sha256_file(
            prototype_09_png_path
        ),
        "ground_truth": sha256_file(
            prototype_09_ground_truth_path
        ),
    },
}

write_json_atomically(
    prototype_09_qa_report_path,
    qa_report,
)


# ================================================================
# Verifikasi round-trip QA report
# ================================================================

with prototype_09_qa_report_path.open(
    mode="r",
    encoding="utf-8",
) as qa_report_file:
    verified_qa_report = json.load(
        qa_report_file
    )

if verified_qa_report != qa_report:
    raise RuntimeError(
        "QA report berubah setelah "
        "disimpan dan dibuka ulang."
    )


# ================================================================
# Output audit
# ================================================================

qa_summary = pd.DataFrame(
    [
        {
            "control": check[
                "control"
            ],
            "expected": check[
                "expected"
            ],
            "actual": check[
                "actual"
            ],
            "status": check[
                "status"
            ],
        }
        for check in checks
    ]
)

display(
    qa_summary
)

print(
    f"QA report          : "
    f"{prototype_09_qa_report_path}"
)

print(
    f"QA status          : "
    f"{qa_status}"
)

print(
    "Minimum field font : "
    f"{minimum_field_font_size:.2f}"
)

print(
    "Font adjustments   : "
    f"{font_adjustment_count}"
)

print(
    "Ink pixel ratio    : "
    f"{raster_content_ratio:.6f}"
)

print(
    "Raster contrast    : "
    f"{raster_contrast:.6f}"
)

print(
    "Manual review      : "
    f"{qa_report['manual_visual_review']}"
)

print()

if qa_status != "PASSED":
    print(
        "Detail kegagalan:"
    )

    print(
        json.dumps(
            qa_report["failures"],
            ensure_ascii=False,
            indent=2,
        )
    )

    raise RuntimeError(
        "Prototype TPL-09 gagal "
        "audit QA teknis."
    )

print(
    "✅ Prototype TPL-09 lulus audit teknis. "
    "Review visual manual masih perlu dicatat "
    "ke QA report."
)

In [ ]:
# ================================================================
# CELL 65A — FINAL
# Finalisasi review visual manual prototype TPL-09
# ================================================================

from pathlib import Path

import hashlib
import json


# ================================================================
# Validasi dependency runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "prototype_09_qa_report_path",
    "prototype_09_png_path",
    "prototype_09_document_id",
    "write_json_atomically",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 65A belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali Cell 64 dan Cell 65."
    )


# ================================================================
# Helper checksum
# ================================================================

def calculate_sha256(
    file_path: Path,
) -> str:
    """Menghitung SHA-256 tanpa memuat seluruh file ke memori."""

    digest = hashlib.sha256()

    with file_path.open(
        mode="rb",
    ) as input_file:
        for chunk in iter(
            lambda: input_file.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ================================================================
# Muat artifact
# ================================================================

qa_report_path = Path(
    prototype_09_qa_report_path
)

preview_path = Path(
    prototype_09_png_path
)

if not qa_report_path.is_file():
    raise FileNotFoundError(
        "QA report TPL-09 tidak ditemukan: "
        f"{qa_report_path}"
    )

if not preview_path.is_file():
    raise FileNotFoundError(
        "Preview TPL-09 tidak ditemukan: "
        f"{preview_path}"
    )

with qa_report_path.open(
    mode="r",
    encoding="utf-8",
) as qa_report_file:
    qa_report = json.load(
        qa_report_file
    )


# ================================================================
# Validasi identitas dan status teknis
# ================================================================

if qa_report.get(
    "template_id"
) != "TPL-09":
    raise RuntimeError(
        "QA report bukan milik TPL-09."
    )

if (
    qa_report.get("document_id")
    != prototype_09_document_id
):
    raise RuntimeError(
        "Document ID QA report tidak sesuai. "
        f"Expected: {prototype_09_document_id!r}; "
        f"actual: {qa_report.get('document_id')!r}."
    )

if qa_report.get(
    "status"
) != "PASSED":
    raise RuntimeError(
        "Review visual tidak boleh difinalisasi "
        "karena audit teknis belum PASSED. "
        f"Status saat ini: "
        f"{qa_report.get('status')!r}."
    )


# ================================================================
# Pastikan tidak ada kegagalan teknis
# ================================================================

failure_details = qa_report.get(
    "failures",
    {},
)

if not isinstance(
    failure_details,
    dict,
):
    raise TypeError(
        "Field 'failures' wajib berupa dictionary."
    )

nonempty_failures = {
    failure_name: failure_value
    for failure_name, failure_value
    in failure_details.items()
    if bool(failure_value)
}

if nonempty_failures:
    raise RuntimeError(
        "QA report masih memiliki kegagalan teknis: "
        f"{nonempty_failures}"
    )

if "manual_visual_review" not in qa_report:
    raise KeyError(
        "Field 'manual_visual_review' tidak ditemukan "
        "dalam QA report."
    )

current_manual_status = qa_report[
    "manual_visual_review"
]

if current_manual_status not in {
    "PENDING",
    "PASSED",
}:
    raise RuntimeError(
        "Status manual visual review tidak dikenali: "
        f"{current_manual_status!r}."
    )


# ================================================================
# Verifikasi checksum preview
# ================================================================

recorded_preview_checksum = (
    qa_report
    .get(
        "checksums_sha256",
        {},
    )
    .get("preview")
)

if not recorded_preview_checksum:
    raise KeyError(
        "Checksum preview tidak ditemukan pada "
        "qa_report['checksums_sha256']['preview']."
    )

actual_preview_checksum = (
    calculate_sha256(
        preview_path
    )
)

if (
    actual_preview_checksum
    != recorded_preview_checksum
):
    raise RuntimeError(
        "Preview berubah setelah audit teknis. "
        "Jalankan ulang Cell 65 dan lakukan "
        "review visual kembali. "
        f"Expected SHA-256: "
        f"{recorded_preview_checksum}; "
        f"actual: {actual_preview_checksum}."
    )


# ================================================================
# Catat review visual secara idempoten
# ================================================================

qa_report[
    "manual_visual_review"
] = "PASSED"

write_json_atomically(
    qa_report_path,
    qa_report,
)


# ================================================================
# Buka ulang dan verifikasi persistensi
# ================================================================

with qa_report_path.open(
    mode="r",
    encoding="utf-8",
) as qa_report_file:
    persisted_qa_report = json.load(
        qa_report_file
    )

if (
    persisted_qa_report.get("template_id")
    != "TPL-09"
):
    raise RuntimeError(
        "Template ID berubah setelah finalisasi."
    )

if (
    persisted_qa_report.get("document_id")
    != prototype_09_document_id
):
    raise RuntimeError(
        "Document ID berubah setelah finalisasi."
    )

if (
    persisted_qa_report.get("status")
    != "PASSED"
):
    raise RuntimeError(
        "Status audit teknis berubah "
        "setelah finalisasi."
    )

if (
    persisted_qa_report.get(
        "manual_visual_review"
    )
    != "PASSED"
):
    raise RuntimeError(
        "Status review visual gagal disimpan."
    )

persisted_preview_checksum = (
    persisted_qa_report
    .get(
        "checksums_sha256",
        {},
    )
    .get("preview")
)

if (
    persisted_preview_checksum
    != actual_preview_checksum
):
    raise RuntimeError(
        "Checksum preview berubah "
        "pada QA report."
    )


# ================================================================
# Ringkasan final
# ================================================================

print(
    f"QA report           : "
    f"{qa_report_path}"
)

print(
    "Technical status    : "
    f"{persisted_qa_report['status']}"
)

print(
    "Manual visual review: "
    f"{persisted_qa_report['manual_visual_review']}"
)

print(
    "Preview SHA-256     : "
    f"{persisted_preview_checksum}"
)

print()

print(
    "✅ TPL-09 FINAL PASSED — audit teknis dan "
    "review visual telah selesai."
)

In [ ]:
# ================================================================
# CELL 66 — FINAL
# Inspeksi spesifikasi dan presentation payload TPL-10
# Read-only: tidak membuat atau mengubah artifact
# ================================================================

from typing import Any

import json

import pandas as pd
from IPython.display import display


# ================================================================
# Validasi runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "template_registry",
    "invoice_render_payloads",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 66 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali cell template registry "
        "dan presentation payload."
    )

if not isinstance(
    template_registry,
    dict,
):
    raise TypeError(
        "template_registry wajib berupa dictionary."
    )

if not isinstance(
    invoice_render_payloads,
    dict,
):
    raise TypeError(
        "invoice_render_payloads wajib berupa dictionary."
    )


# ================================================================
# Konfigurasi
# ================================================================

TPL10_TEMPLATE_ID = "TPL-10"
TPL10_EXPECTED_PAYLOAD_COUNT = 20

if TPL10_TEMPLATE_ID not in template_registry:
    raise KeyError(
        "Spesifikasi TPL-10 tidak ditemukan "
        "dalam template_registry."
    )

tpl10_template_specification = (
    template_registry[
        TPL10_TEMPLATE_ID
    ]
)

if not isinstance(
    tpl10_template_specification,
    dict,
):
    raise TypeError(
        "Spesifikasi TPL-10 wajib berupa dictionary."
    )

tpl10_payloads = {
    canonical_id: payload
    for canonical_id, payload
    in invoice_render_payloads.items()
    if (
        isinstance(payload, dict)
        and payload.get("template_id")
        == TPL10_TEMPLATE_ID
    )
}

if (
    len(tpl10_payloads)
    != TPL10_EXPECTED_PAYLOAD_COUNT
):
    raise RuntimeError(
        "TPL-10 harus memiliki tepat "
        f"{TPL10_EXPECTED_PAYLOAD_COUNT} payload. "
        f"Actual: {len(tpl10_payloads)}."
    )


# ================================================================
# Kontrak payload
# ================================================================

REQUIRED_KEYS = {
    "top_level": {
        "buyer",
        "canonical_invoice_id",
        "currency",
        "document_id",
        "financials",
        "footer",
        "items",
        "labels",
        "language",
        "metadata",
        "split",
        "synthetic_notice",
        "template_id",
        "title",
        "vendor",
    },
    "labels": {
        "buyer",
        "description",
        "discount",
        "document_title",
        "due_date",
        "invoice_date",
        "invoice_number",
        "line_total",
        "page",
        "quantity",
        "subtotal",
        "synthetic_notice",
        "tax",
        "tax_identifier",
        "total",
        "unit_price",
        "vendor",
    },
    "metadata": {
        "due_date",
        "invoice_date",
        "invoice_number",
    },
    "party": {
        "address_lines",
        "email",
        "name",
        "party_id",
        "phone",
        "tax_identifier",
        "tax_identifier_display",
    },
    "item": {
        "description",
        "item_number",
        "line_total",
        "quantity",
        "raw",
        "unit_price",
    },
    "financials": {
        "discount",
        "subtotal",
        "tax",
        "total",
    },
    "footer": {
        "document_reference",
        "synthetic_notice",
    },
}


# ================================================================
# Helper
# ================================================================

def require_dictionary(
    value: Any,
    field_path: str,
) -> dict[str, Any]:
    if not isinstance(value, dict):
        raise TypeError(
            f"{field_path} wajib berupa dictionary."
        )

    return value


def require_keys(
    mapping: dict[str, Any],
    required_keys: set[str],
    field_path: str,
) -> None:
    missing_keys = sorted(
        required_keys - set(mapping)
    )

    if missing_keys:
        raise KeyError(
            f"{field_path} kehilangan key wajib: "
            f"{missing_keys}"
        )


def joined_text_length(
    values: Any,
) -> int:
    if isinstance(values, list):
        return len(
            " ".join(
                str(value)
                for value in values
            )
        )

    return len(str(values))


def display_value(
    value: Any,
) -> str:
    if isinstance(value, dict):
        return str(
            value.get(
                "display_value",
                value.get("value", ""),
            )
        )

    return str(value)


def maximum_money_length(
    payload: dict[str, Any],
) -> int:
    money_values = []

    for item in payload["items"]:
        money_values.extend(
            [
                str(item["unit_price"]),
                str(item["line_total"]),
            ]
        )

    for field_name in (
        "subtotal",
        "tax",
        "discount",
        "total",
    ):
        money_values.append(
            display_value(
                payload[
                    "financials"
                ][field_name]
            )
        )

    return max(
        map(len, money_values),
        default=0,
    )


# ================================================================
# Validasi seluruh payload
# ================================================================

expected_split = (
    tpl10_template_specification.get(
        "split"
    )
)

document_ids = []

for canonical_id in sorted(
    tpl10_payloads
):
    payload = tpl10_payloads[
        canonical_id
    ]

    require_keys(
        payload,
        REQUIRED_KEYS["top_level"],
        f"payload[{canonical_id!r}]",
    )

    if (
        payload["canonical_invoice_id"]
        != canonical_id
    ):
        raise RuntimeError(
            "Canonical ID dictionary dan payload "
            f"tidak sama: {canonical_id!r}."
        )

    if (
        payload["template_id"]
        != TPL10_TEMPLATE_ID
    ):
        raise RuntimeError(
            f"Template ID tidak sesuai pada {canonical_id}."
        )

    if (
        expected_split is not None
        and payload["split"] != expected_split
    ):
        raise RuntimeError(
            f"Split {canonical_id} tidak sesuai. "
            f"Expected: {expected_split!r}; "
            f"actual: {payload['split']!r}."
        )

    if payload["language"] not in {
        "id",
        "en",
    }:
        raise RuntimeError(
            f"Language tidak didukung pada {canonical_id}: "
            f"{payload['language']!r}."
        )

    labels = require_dictionary(
        payload["labels"],
        f"payload[{canonical_id!r}].labels",
    )

    metadata = require_dictionary(
        payload["metadata"],
        f"payload[{canonical_id!r}].metadata",
    )

    vendor = require_dictionary(
        payload["vendor"],
        f"payload[{canonical_id!r}].vendor",
    )

    buyer = require_dictionary(
        payload["buyer"],
        f"payload[{canonical_id!r}].buyer",
    )

    financials = require_dictionary(
        payload["financials"],
        f"payload[{canonical_id!r}].financials",
    )

    footer = require_dictionary(
        payload["footer"],
        f"payload[{canonical_id!r}].footer",
    )

    require_keys(
        labels,
        REQUIRED_KEYS["labels"],
        f"payload[{canonical_id!r}].labels",
    )

    require_keys(
        metadata,
        REQUIRED_KEYS["metadata"],
        f"payload[{canonical_id!r}].metadata",
    )

    require_keys(
        vendor,
        REQUIRED_KEYS["party"],
        f"payload[{canonical_id!r}].vendor",
    )

    require_keys(
        buyer,
        REQUIRED_KEYS["party"],
        f"payload[{canonical_id!r}].buyer",
    )

    require_keys(
        financials,
        REQUIRED_KEYS["financials"],
        f"payload[{canonical_id!r}].financials",
    )

    require_keys(
        footer,
        REQUIRED_KEYS["footer"],
        f"payload[{canonical_id!r}].footer",
    )

    for metadata_name in REQUIRED_KEYS[
        "metadata"
    ]:
        metadata_field = require_dictionary(
            metadata[metadata_name],
            (
                f"payload[{canonical_id!r}]"
                f".metadata[{metadata_name!r}]"
            ),
        )

        require_keys(
            metadata_field,
            {
                "label",
                "display_value",
            },
            (
                f"payload[{canonical_id!r}]"
                f".metadata[{metadata_name!r}]"
            ),
        )

    for party_name, party in (
        ("vendor", vendor),
        ("buyer", buyer),
    ):
        if (
            not isinstance(
                party["address_lines"],
                list,
            )
            or not party["address_lines"]
        ):
            raise RuntimeError(
                f"payload[{canonical_id!r}]"
                f".{party_name}.address_lines "
                "wajib berupa list yang tidak kosong."
            )

    items = payload["items"]

    if (
        not isinstance(items, list)
        or not items
    ):
        raise RuntimeError(
            f"payload[{canonical_id!r}].items "
            "wajib berupa list yang tidak kosong."
        )

    for item_index, item in enumerate(
        items
    ):
        item = require_dictionary(
            item,
            (
                f"payload[{canonical_id!r}]"
                f".items[{item_index}]"
            ),
        )

        require_keys(
            item,
            REQUIRED_KEYS["item"],
            (
                f"payload[{canonical_id!r}]"
                f".items[{item_index}]"
            ),
        )

    document_ids.append(
        str(payload["document_id"])
    )

if (
    len(document_ids)
    != len(set(document_ids))
):
    raise RuntimeError(
        "Ditemukan document_id duplikat "
        "pada payload TPL-10."
    )


# ================================================================
# Struktur sampel
# ================================================================

tpl10_candidate_ids = sorted(
    tpl10_payloads
)

tpl10_sample_id = (
    tpl10_candidate_ids[0]
)

tpl10_sample_payload = (
    tpl10_payloads[
        tpl10_sample_id
    ]
)

tpl10_sample_structure = {
    "canonical_invoice_id": (
        tpl10_sample_payload[
            "canonical_invoice_id"
        ]
    ),
    "document_id": (
        tpl10_sample_payload[
            "document_id"
        ]
    ),
    "template_id": (
        tpl10_sample_payload[
            "template_id"
        ]
    ),
    "language": (
        tpl10_sample_payload[
            "language"
        ]
    ),
    "currency": (
        tpl10_sample_payload[
            "currency"
        ]
    ),
    "title": (
        tpl10_sample_payload[
            "title"
        ]
    ),
    "top_level_keys": sorted(
        tpl10_sample_payload
    ),
    "label_keys": sorted(
        tpl10_sample_payload["labels"]
    ),
    "metadata_keys": sorted(
        tpl10_sample_payload["metadata"]
    ),
    "vendor_keys": sorted(
        tpl10_sample_payload["vendor"]
    ),
    "buyer_keys": sorted(
        tpl10_sample_payload["buyer"]
    ),
    "item_keys": sorted(
        tpl10_sample_payload["items"][0]
    ),
    "financial_keys": sorted(
        tpl10_sample_payload["financials"]
    ),
    "footer_keys": sorted(
        tpl10_sample_payload["footer"]
    ),
}


# ================================================================
# Statistik payload
# ================================================================

statistics_rows = []

for canonical_id in tpl10_candidate_ids:
    payload = tpl10_payloads[
        canonical_id
    ]

    descriptions = [
        str(item["description"])
        for item in payload["items"]
    ]

    statistics_rows.append(
        {
            "canonical_id": canonical_id,
            "document_id": payload[
                "document_id"
            ],
            "language": payload[
                "language"
            ],
            "currency": payload[
                "currency"
            ],
            "item_count": len(
                payload["items"]
            ),
            "vendor_name_length": len(
                str(
                    payload[
                        "vendor"
                    ]["name"]
                )
            ),
            "vendor_address_length": (
                joined_text_length(
                    payload[
                        "vendor"
                    ]["address_lines"]
                )
            ),
            "vendor_email_length": len(
                str(
                    payload[
                        "vendor"
                    ]["email"]
                )
            ),
            "buyer_name_length": len(
                str(
                    payload[
                        "buyer"
                    ]["name"]
                )
            ),
            "buyer_address_length": (
                joined_text_length(
                    payload[
                        "buyer"
                    ]["address_lines"]
                )
            ),
            "buyer_email_length": len(
                str(
                    payload[
                        "buyer"
                    ]["email"]
                )
            ),
            "maximum_description_length": max(
                map(
                    len,
                    descriptions,
                ),
                default=0,
            ),
            "maximum_money_length": (
                maximum_money_length(
                    payload
                )
            ),
        }
    )

tpl10_statistics = pd.DataFrame(
    statistics_rows
)

tpl10_payload_statistics = (
    tpl10_statistics
    .describe(include="all")
    .transpose()
)

tpl10_item_count_distribution = (
    tpl10_statistics["item_count"]
    .value_counts()
    .sort_index()
    .rename_axis("item_count")
    .reset_index(
        name="document_count"
    )
)

tpl10_language_distribution = (
    tpl10_statistics["language"]
    .value_counts()
    .sort_index()
    .rename_axis("language")
    .reset_index(
        name="document_count"
    )
)

tpl10_currency_distribution = (
    tpl10_statistics["currency"]
    .value_counts()
    .sort_index()
    .rename_axis("currency")
    .reset_index(
        name="document_count"
    )
)


# ================================================================
# Output inspeksi
# ================================================================

print("TEMPLATE SPECIFICATION")

print(
    json.dumps(
        tpl10_template_specification,
        ensure_ascii=False,
        indent=2,
    )
)

print()
print("PAYLOAD COUNT")
print(len(tpl10_payloads))

print()
print("SAMPLE STRUCTURE")

print(
    json.dumps(
        tpl10_sample_structure,
        ensure_ascii=False,
        indent=2,
    )
)

print()
print("LABELS")

print(
    json.dumps(
        tpl10_sample_payload["labels"],
        ensure_ascii=False,
        indent=2,
    )
)

print()
print("PAYLOAD STATISTICS")
display(tpl10_payload_statistics)

print()
print("ITEM-COUNT DISTRIBUTION")
display(tpl10_item_count_distribution)

print()
print("LANGUAGE DISTRIBUTION")
display(tpl10_language_distribution)

print()
print("CURRENCY DISTRIBUTION")
display(tpl10_currency_distribution)

print()

print(
    "✅ Inspeksi TPL-10 selesai. "
    "Belum ada artifact yang dibuat atau diubah."
)

In [ ]:
# ================================================================
# CELL 67 — FINAL
# Prototype renderer TPL-10: International Clean
# ================================================================

from pathlib import Path
from typing import Any
from PIL import Image as PILImage

import json
import os

import pandas as pd
import pymupdf


# ================================================================
# Validasi dependency runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "PROTOTYPE_ROOT",
    "CANONICAL_RECORDS_JSONL_PATH",
    "FieldAnnotation",
    "template_registry",
    "invoice_render_payloads",
    "get_page_dimensions",
    "hex_to_rgb",
    "insert_annotated_text",
    "insert_static_text",
    "annotation_to_dict",
    "write_json_atomically",
    "RENDER_ENGINE_VERSION",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 67 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali cell pemulihan runtime, "
        "template registry, dan presentation payload."
    )


# ================================================================
# Konfigurasi TPL-10
# ================================================================

TPL10_TEMPLATE_ID = "TPL-10"
TPL10_RENDERER_VERSION = "1.0.0"

TPL10_MIN_ITEMS_PER_PAGE = 2
TPL10_MAX_ITEMS_PER_PAGE = 8

TPL10_PROTOTYPE_ROOT = (
    Path(PROTOTYPE_ROOT)
    / TPL10_TEMPLATE_ID
)

TPL10_PROTOTYPE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ================================================================
# Helper canonical payload
# ================================================================

def load_tpl10_canonical_payload(
    canonical_id: str,
) -> dict[str, Any]:
    """Memuat satu canonical invoice dari JSONL."""

    if (
        not isinstance(canonical_id, str)
        or not canonical_id.strip()
    ):
        raise ValueError(
            "canonical_id wajib berupa string "
            "yang tidak kosong."
        )

    canonical_path = Path(
        CANONICAL_RECORDS_JSONL_PATH
    )

    if not canonical_path.is_file():
        raise FileNotFoundError(
            "Canonical JSONL tidak ditemukan: "
            f"{canonical_path}"
        )

    with canonical_path.open(
        mode="r",
        encoding="utf-8",
    ) as canonical_file:
        for line_number, line in enumerate(
            canonical_file,
            start=1,
        ):
            clean_line = line.strip()

            if not clean_line:
                continue

            try:
                payload = json.loads(
                    clean_line
                )
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    "Canonical JSONL tidak valid "
                    f"pada baris {line_number}."
                ) from error

            if (
                payload.get(
                    "canonical_invoice_id"
                )
                == canonical_id
            ):
                return payload

    raise KeyError(
        "Canonical payload tidak ditemukan: "
        f"{canonical_id}"
    )


def normalize_pdf_notice(
    value: Any,
) -> str:
    """Mengganti separator yang tidak didukung font PDF bawaan."""

    return (
        str(value)
        .replace("—", "·")
        .replace("–", "-")
    )


# ================================================================
# Renderer final TPL-10
# ================================================================

def render_template_10(
    render_payload: dict[str, Any],
    output_pdf_path: Path,
) -> tuple[
    list[FieldAnnotation],
    dict[str, Any],
]:
    """Merender invoice satu halaman dengan layout international clean."""

    if not isinstance(
        render_payload,
        dict,
    ):
        raise TypeError(
            "render_payload wajib berupa dictionary."
        )

    if (
        render_payload.get("template_id")
        != TPL10_TEMPLATE_ID
    ):
        raise ValueError(
            "Renderer TPL-10 hanya menerima "
            "payload TPL-10."
        )

    required_payload_keys = {
        "canonical_invoice_id",
        "document_id",
        "template_id",
        "language",
        "currency",
        "title",
        "labels",
        "metadata",
        "vendor",
        "buyer",
        "items",
        "financials",
        "footer",
        "synthetic_notice",
    }

    missing_payload_keys = sorted(
        required_payload_keys
        - set(render_payload)
    )

    if missing_payload_keys:
        raise KeyError(
            "Render payload TPL-10 belum lengkap: "
            f"{missing_payload_keys}"
        )

    item_count = len(
        render_payload["items"]
    )

    if not (
        TPL10_MIN_ITEMS_PER_PAGE
        <= item_count
        <= TPL10_MAX_ITEMS_PER_PAGE
    ):
        raise ValueError(
            "TPL-10 hanya mendukung 2–8 item "
            "dalam satu halaman."
        )

    language = render_payload[
        "language"
    ]

    if (
        language == "id"
        and render_payload[
            "labels"
        ]["quantity"] != "Kuantitas"
    ):
        raise RuntimeError(
            "Label quantity bahasa Indonesia "
            "wajib menggunakan 'Kuantitas'."
        )

    template_specification = (
        template_registry[
            TPL10_TEMPLATE_ID
        ]
    )

    expected_template_contract = {
        "layout_family": "international_clean",
        "page_size": "LETTER",
        "header_layout": "open_right",
        "party_layout": "stacked",
        "table_style": "horizontal_rules",
        "totals_position": "full_width",
        "accent_position": "right",
        "density": "regular",
    }

    contract_mismatches = {
        key: {
            "expected": expected_value,
            "actual": (
                template_specification.get(key)
            ),
        }
        for key, expected_value
        in expected_template_contract.items()
        if (
            template_specification.get(key)
            != expected_value
        )
    }

    if contract_mismatches:
        raise RuntimeError(
            "Spesifikasi layout TPL-10 "
            "tidak sesuai: "
            f"{contract_mismatches}"
        )

    page_width, page_height = (
        get_page_dimensions(
            template_specification[
                "page_size"
            ]
        )
    )

    primary_color = hex_to_rgb(
        template_specification[
            "primary_color"
        ]
    )

    accent_color = hex_to_rgb(
        template_specification[
            "accent_color"
        ]
    )

    dark_text = (
        0.12,
        0.15,
        0.20,
    )

    muted_text = (
        0.39,
        0.43,
        0.48,
    )

    border_color = (
        0.80,
        0.83,
        0.86,
    )

    light_primary = (
        0.955,
        0.965,
        0.975,
    )

    light_accent = (
        0.995,
        0.970,
        0.920,
    )

    white = (
        1.0,
        1.0,
        1.0,
    )

    output_pdf_path = Path(
        output_pdf_path
    )

    output_pdf_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_pdf_path = (
        output_pdf_path.with_name(
            f".{output_pdf_path.stem}"
            ".tmp.pdf"
        )
    )

    temporary_pdf_path.unlink(
        missing_ok=True
    )

    annotations: list[
        FieldAnnotation
    ] = []

    field_font_audit: list[
        dict[str, Any]
    ] = []

    document = pymupdf.open()

    def add_field(
        page: pymupdf.Page,
        rectangle: pymupdf.Rect,
        text: str,
        field_name: str,
        font_name: str = "helv",
        font_size: float = 7.0,
        minimum_font_size: float = 6.5,
        font_color: tuple[
            float,
            float,
            float,
        ] = dark_text,
        alignment: str = "left",
    ) -> None:
        """Menambahkan field dan mencatat ukuran font."""

        annotation, used_font_size = (
            insert_annotated_text(
                page=page,
                rectangle=rectangle,
                text=str(text),
                field_name=field_name,
                font_name=font_name,
                font_size=font_size,
                minimum_font_size=(
                    minimum_font_size
                ),
                font_color=font_color,
                alignment=alignment,
            )
        )

        annotations.append(
            annotation
        )

        field_font_audit.append(
            {
                "field_name": field_name,
                "requested_font_size": (
                    font_size
                ),
                "used_font_size": (
                    used_font_size
                ),
                "adjusted": (
                    used_font_size
                    < font_size
                ),
            }
        )

    try:
        page = document.new_page(
            width=page_width,
            height=page_height,
        )

        margin_x0 = 44.0
        margin_x1 = (
            page_width - 44.0
        )

        # --------------------------------------------------------
        # Aksen vertikal kanan
        # --------------------------------------------------------

        page.draw_rect(
            pymupdf.Rect(
                page_width - 8,
                0,
                page_width,
                page_height,
            ),
            color=accent_color,
            fill=accent_color,
            width=0,
            overlay=True,
        )

        # --------------------------------------------------------
        # Header open-right
        # --------------------------------------------------------

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                330,
                32,
                margin_x1,
                70,
            ),
            text=render_payload[
                "title"
            ],
            font_name="hebo",
            font_size=23.0,
            minimum_font_size=17.0,
            font_color=primary_color,
            alignment="right",
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                37,
                295,
                53,
            ),
            text=(
                "INTERNATIONAL CLEAN · SYNTHETIC"
            ),
            font_name="hebo",
            font_size=6.7,
            minimum_font_size=6.0,
            font_color=accent_color,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                59,
                295,
                74,
            ),
            text=(
                render_payload[
                    "footer"
                ]["document_reference"]
            ),
            font_name="helv",
            font_size=6.6,
            minimum_font_size=6.0,
            font_color=muted_text,
        )

        currency_label = (
            "Mata Uang"
            if language == "id"
            else "Currency"
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                83,
                106,
                98,
            ),
            text=currency_label,
            font_name="helv",
            font_size=6.3,
            minimum_font_size=5.8,
            font_color=muted_text,
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                108,
                81,
                182,
                99,
            ),
            text=render_payload[
                "currency"
            ],
            field_name="currency",
            font_name="hebo",
            font_size=7.6,
            minimum_font_size=6.5,
            font_color=primary_color,
        )

        # --------------------------------------------------------
        # Metadata tiga kolom
        # --------------------------------------------------------

        metadata_cards = [
            (
                "invoice_number",
                250.0,
                356.0,
            ),
            (
                "invoice_date",
                362.0,
                462.0,
            ),
            (
                "due_date",
                468.0,
                margin_x1,
            ),
        ]

        for (
            field_name,
            card_x0,
            card_x1,
        ) in metadata_cards:
            field_payload = (
                render_payload[
                    "metadata"
                ][field_name]
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0,
                    82,
                    card_x1,
                    96,
                ),
                text=field_payload[
                    "label"
                ],
                font_name="helv",
                font_size=6.0,
                minimum_font_size=5.6,
                font_color=muted_text,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0,
                    100,
                    card_x1,
                    118,
                ),
                text=field_payload[
                    "display_value"
                ],
                field_name=field_name,
                font_name="hebo",
                font_size=6.8,
                minimum_font_size=6.5,
                font_color=primary_color,
            )

        page.draw_line(
            pymupdf.Point(
                margin_x0,
                130,
            ),
            pymupdf.Point(
                margin_x1,
                130,
            ),
            color=primary_color,
            width=0.9,
            overlay=True,
        )

        # --------------------------------------------------------
        # Vendor dan buyer bertumpuk
        # --------------------------------------------------------

        party_sections = [
            (
                "vendor",
                render_payload[
                    "labels"
                ]["vendor"],
                144.0,
                215.0,
            ),
            (
                "buyer",
                render_payload[
                    "labels"
                ]["buyer"],
                224.0,
                295.0,
            ),
        ]

        for (
            party_type,
            party_label,
            section_y0,
            section_y1,
        ) in party_sections:
            party = render_payload[
                party_type
            ]

            page.draw_rect(
                pymupdf.Rect(
                    margin_x0,
                    section_y0,
                    margin_x1,
                    section_y1,
                ),
                color=None,
                fill=(
                    light_primary
                    if party_type == "vendor"
                    else white
                ),
                width=0,
                overlay=True,
            )

            page.draw_line(
                pymupdf.Point(
                    margin_x0,
                    section_y1,
                ),
                pymupdf.Point(
                    margin_x1,
                    section_y1,
                ),
                color=border_color,
                width=0.65,
                overlay=True,
            )

            page.draw_line(
                pymupdf.Point(
                    320,
                    section_y0 + 10,
                ),
                pymupdf.Point(
                    320,
                    section_y1 - 10,
                ),
                color=border_color,
                width=0.6,
                overlay=True,
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    margin_x0,
                    section_y0 + 7,
                    120,
                    section_y0 + 20,
                ),
                text=party_label.upper(),
                font_name="hebo",
                font_size=6.4,
                minimum_font_size=5.8,
                font_color=accent_color,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    122,
                    section_y0 + 6,
                    310,
                    section_y0 + 23,
                ),
                text=party["name"],
                field_name=(
                    f"{party_type}.name"
                ),
                font_name="hebo",
                font_size=7.4,
                minimum_font_size=6.5,
                font_color=primary_color,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    margin_x0,
                    section_y0 + 29,
                    310,
                    section_y1 - 7,
                ),
                text="\n".join(
                    party[
                        "address_lines"
                    ]
                ),
                field_name=(
                    f"{party_type}"
                    ".address_lines"
                ),
                font_name="helv",
                font_size=6.7,
                minimum_font_size=6.5,
                font_color=dark_text,
            )

            contact_fields = [
                (
                    "email",
                    party["email"],
                    10.0,
                    24.0,
                ),
                (
                    "phone",
                    party["phone"],
                    29.0,
                    43.0,
                ),
                (
                    "tax_identifier",
                    party[
                        "tax_identifier_display"
                    ],
                    48.0,
                    63.0,
                ),
            ]

            for (
                field_name,
                field_value,
                offset_y0,
                offset_y1,
            ) in contact_fields:
                add_field(
                    page=page,
                    rectangle=pymupdf.Rect(
                        334,
                        section_y0 + offset_y0,
                        margin_x1,
                        section_y0 + offset_y1,
                    ),
                    text=field_value,
                    field_name=(
                        f"{party_type}."
                        f"{field_name}"
                    ),
                    font_name="helv",
                    font_size=6.6,
                    minimum_font_size=6.5,
                    font_color=dark_text,
                )

        # --------------------------------------------------------
        # Tabel horizontal rules
        # --------------------------------------------------------

        table_positions = [
            margin_x0,
            69.0,
            305.0,
            361.0,
            458.0,
            margin_x1,
        ]

        table_header_y0 = 313.0
        table_header_y1 = 338.0
        row_height = 23.0

        page.draw_line(
            pymupdf.Point(
                table_positions[0],
                table_header_y0,
            ),
            pymupdf.Point(
                table_positions[-1],
                table_header_y0,
            ),
            color=primary_color,
            width=1.1,
            overlay=True,
        )

        page.draw_line(
            pymupdf.Point(
                table_positions[0],
                table_header_y1,
            ),
            pymupdf.Point(
                table_positions[-1],
                table_header_y1,
            ),
            color=accent_color,
            width=0.9,
            overlay=True,
        )

        table_headers = [
            "#",
            render_payload[
                "labels"
            ]["description"],
            render_payload[
                "labels"
            ]["quantity"],
            render_payload[
                "labels"
            ]["unit_price"],
            render_payload[
                "labels"
            ]["line_total"],
        ]

        for (
            column_index,
            header_text,
        ) in enumerate(
            table_headers
        ):
            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    (
                        table_positions[
                            column_index
                        ]
                        + 3
                    ),
                    table_header_y0 + 7,
                    (
                        table_positions[
                            column_index + 1
                        ]
                        - 3
                    ),
                    table_header_y1 - 2,
                ),
                text=header_text,
                font_name="hebo",
                font_size=6.4,
                minimum_font_size=5.8,
                font_color=primary_color,
                alignment=(
                    "left"
                    if column_index == 1
                    else "center"
                ),
            )

        for (
            item_index,
            item,
        ) in enumerate(
            render_payload["items"]
        ):
            row_y0 = (
                table_header_y1
                + item_index
                * row_height
            )

            row_y1 = (
                row_y0
                + row_height
            )

            page.draw_line(
                pymupdf.Point(
                    table_positions[0],
                    row_y1,
                ),
                pymupdf.Point(
                    table_positions[-1],
                    row_y1,
                ),
                color=border_color,
                width=0.55,
                overlay=True,
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    table_positions[0] + 3,
                    row_y0 + 6,
                    table_positions[1] - 3,
                    row_y1 - 2,
                ),
                text=str(
                    item[
                        "item_number"
                    ]
                ),
                font_name="helv",
                font_size=6.7,
                minimum_font_size=6.0,
                font_color=muted_text,
                alignment="center",
            )

            item_fields = [
                (
                    "description",
                    1,
                    "left",
                ),
                (
                    "quantity",
                    2,
                    "center",
                ),
                (
                    "unit_price",
                    3,
                    "right",
                ),
                (
                    "line_total",
                    4,
                    "right",
                ),
            ]

            for (
                field_name,
                column_index,
                field_alignment,
            ) in item_fields:
                add_field(
                    page=page,
                    rectangle=pymupdf.Rect(
                        (
                            table_positions[
                                column_index
                            ]
                            + 4
                        ),
                        row_y0 + 6,
                        (
                            table_positions[
                                column_index + 1
                            ]
                            - 4
                        ),
                        row_y1 - 2,
                    ),
                    text=item[
                        field_name
                    ],
                    field_name=(
                        f"items[{item_index}]"
                        f".{field_name}"
                    ),
                    font_name="helv",
                    font_size=6.8,
                    minimum_font_size=6.5,
                    font_color=dark_text,
                    alignment=(
                        field_alignment
                    ),
                )

        table_end_y = (
            table_header_y1
            + item_count
            * row_height
        )

        # --------------------------------------------------------
        # Ringkasan finansial full-width
        # --------------------------------------------------------

        totals_panel_height = 82.0

        totals_y0 = min(
            max(
                table_end_y + 18.0,
                500.0,
            ),
            545.0,
        )

        totals_y1 = (
            totals_y0
            + totals_panel_height
        )

        table_to_totals_gap = (
            totals_y0
            - table_end_y
        )

        footer_line_y = 727.0

        if table_to_totals_gap < 18.0:
            raise RuntimeError(
                "Tabel bertabrakan dengan "
                "ringkasan finansial. "
                f"Gap: "
                f"{table_to_totals_gap:.2f} pt."
            )

        if (
            totals_y1
            >= footer_line_y - 28.0
        ):
            raise RuntimeError(
                "Ringkasan finansial terlalu dekat "
                "dengan footer. "
                f"Panel berakhir pada "
                f"{totals_y1:.2f} pt."
            )

        summary_heading = (
            "RINGKASAN"
            if language == "id"
            else "SUMMARY"
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                totals_y0 - 18,
                margin_x1,
                totals_y0 - 4,
            ),
            text=summary_heading,
            font_name="hebo",
            font_size=6.6,
            minimum_font_size=6.0,
            font_color=accent_color,
        )

        page.draw_rect(
            pymupdf.Rect(
                margin_x0,
                totals_y0,
                margin_x1,
                totals_y1,
            ),
            color=border_color,
            fill=light_primary,
            width=0.65,
            overlay=True,
        )

        total_column_positions = [
            margin_x0,
            175.0,
            306.0,
            437.0,
            margin_x1,
        ]

        financial_order = [
            "subtotal",
            "tax",
            "discount",
            "total",
        ]

        for (
            row_index,
            field_name,
        ) in enumerate(
            financial_order
        ):
            column_x0 = (
                total_column_positions[
                    row_index
                ]
            )

            column_x1 = (
                total_column_positions[
                    row_index + 1
                ]
            )

            field_payload = (
                render_payload[
                    "financials"
                ][field_name]
            )

            is_total = (
                field_name == "total"
            )

            if row_index > 0:
                page.draw_line(
                    pymupdf.Point(
                        column_x0,
                        totals_y0 + 10,
                    ),
                    pymupdf.Point(
                        column_x0,
                        totals_y1 - 10,
                    ),
                    color=border_color,
                    width=0.6,
                    overlay=True,
                )

            if is_total:
                page.draw_rect(
                    pymupdf.Rect(
                        column_x0,
                        totals_y0,
                        column_x1,
                        totals_y1,
                    ),
                    color=primary_color,
                    fill=primary_color,
                    width=0,
                    overlay=True,
                )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    column_x0 + 8,
                    totals_y0 + 15,
                    column_x1 - 8,
                    totals_y0 + 31,
                ),
                text=field_payload[
                    "label"
                ],
                font_name=(
                    "hebo"
                    if is_total
                    else "helv"
                ),
                font_size=6.7,
                minimum_font_size=6.0,
                font_color=(
                    white
                    if is_total
                    else muted_text
                ),
                alignment="center",
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    column_x0 + 8,
                    totals_y0 + 43,
                    column_x1 - 8,
                    totals_y0 + 63,
                ),
                text=field_payload[
                    "display_value"
                ],
                field_name=(
                    f"financials."
                    f"{field_name}"
                ),
                font_name=(
                    "hebo"
                    if is_total
                    else "helv"
                ),
                font_size=(
                    7.5
                    if is_total
                    else 6.9
                ),
                minimum_font_size=6.5,
                font_color=(
                    white
                    if is_total
                    else dark_text
                ),
                alignment="center",
            )

        # --------------------------------------------------------
        # Footer
        # --------------------------------------------------------

        page.draw_line(
            pymupdf.Point(
                margin_x0,
                footer_line_y,
            ),
            pymupdf.Point(
                margin_x1,
                footer_line_y,
            ),
            color=border_color,
            width=0.7,
            overlay=True,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                735,
                margin_x1,
                749,
            ),
            text=(
                render_payload[
                    "footer"
                ]["document_reference"]
            ),
            font_name="helv",
            font_size=6.4,
            minimum_font_size=6.0,
            font_color=muted_text,
            alignment="center",
        )

        notice_display_text = (
            normalize_pdf_notice(
                render_payload[
                    "synthetic_notice"
                ]
            )
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                758,
                margin_x1,
                777,
            ),
            text=notice_display_text,
            field_name=(
                "document.synthetic_notice"
            ),
            font_name="hebo",
            font_size=6.6,
            minimum_font_size=6.5,
            font_color=accent_color,
            alignment="center",
        )

        # --------------------------------------------------------
        # Pre-save controls
        # --------------------------------------------------------

        annotation_names = [
            annotation.field_name
            for annotation in annotations
        ]

        if (
            len(annotation_names)
            != len(set(annotation_names))
        ):
            raise RuntimeError(
                "Ditemukan nama anotasi duplikat."
            )

        expected_annotation_count = (
            19
            + 4 * item_count
        )

        actual_annotation_count = len(
            annotations
        )

        if (
            actual_annotation_count
            != expected_annotation_count
        ):
            raise RuntimeError(
                "Jumlah anotasi tidak sesuai. "
                f"Expected: "
                f"{expected_annotation_count}; "
                f"actual: "
                f"{actual_annotation_count}."
            )

        if not field_font_audit:
            raise RuntimeError(
                "Audit font field tidak boleh kosong."
            )

        minimum_used_font_size = min(
            row["used_font_size"]
            for row in field_font_audit
        )

        if (
            minimum_used_font_size
            < 6.5
        ):
            raise RuntimeError(
                "Ukuran font field berada "
                "di bawah 6.5 pt."
            )

        document.save(
            str(temporary_pdf_path),
            garbage=4,
            deflate=True,
        )

    except Exception:
        temporary_pdf_path.unlink(
            missing_ok=True
        )
        raise

    finally:
        document.close()

    if (
        not temporary_pdf_path.is_file()
        or temporary_pdf_path.stat().st_size
        == 0
    ):
        temporary_pdf_path.unlink(
            missing_ok=True
        )

        raise RuntimeError(
            "PDF sementara TPL-10 "
            "gagal dibuat."
        )

    os.replace(
        temporary_pdf_path,
        output_pdf_path,
    )

    rendering_metadata = {
        "renderer_version": (
            RENDER_ENGINE_VERSION
        ),
        "template_renderer_version": (
            TPL10_RENDERER_VERSION
        ),
        "template_id": (
            TPL10_TEMPLATE_ID
        ),
        "layout_family": (
            template_specification[
                "layout_family"
            ]
        ),
        "page_size": (
            template_specification[
                "page_size"
            ]
        ),
        "page_width_points": round(
            page_width,
            3,
        ),
        "page_height_points": round(
            page_height,
            3,
        ),
        "page_count": 1,
        "annotation_type": (
            "field_region"
        ),
        "annotation_count": len(
            annotations
        ),
        "minimum_field_font_size": min(
            row["used_font_size"]
            for row in field_font_audit
        ),
        "font_adjustment_count": int(
            sum(
                bool(row["adjusted"])
                for row in field_font_audit
            )
        ),
        "layout_metrics": {
            "item_count": item_count,
            "table_end_y_points": round(
                table_end_y,
                3,
            ),
            "totals_panel_y0_points": round(
                totals_y0,
                3,
            ),
            "totals_panel_y1_points": round(
                totals_y1,
                3,
            ),
            "table_to_totals_gap_points": round(
                table_to_totals_gap,
                3,
            ),
            "footer_line_y_points": (
                footer_line_y
            ),
        },
    }

    return (
        annotations,
        rendering_metadata,
    )


# ================================================================
# Pilih prototype pertama TPL-10
# ================================================================

tpl10_candidate_ids = sorted(
    canonical_id
    for canonical_id, payload
    in invoice_render_payloads.items()
    if (
        isinstance(payload, dict)
        and payload.get("template_id")
        == TPL10_TEMPLATE_ID
    )
)

if len(tpl10_candidate_ids) != 20:
    raise RuntimeError(
        "TPL-10 harus memiliki tepat "
        "20 presentation payload. "
        f"Actual: {len(tpl10_candidate_ids)}."
    )

prototype_10_canonical_id = (
    tpl10_candidate_ids[0]
)

prototype_10_render_payload = (
    invoice_render_payloads[
        prototype_10_canonical_id
    ]
)

prototype_10_document_id = (
    prototype_10_render_payload[
        "document_id"
    ]
)

prototype_10_canonical_payload = (
    load_tpl10_canonical_payload(
        prototype_10_canonical_id
    )
)


# ================================================================
# Path artifact
# ================================================================

prototype_10_pdf_path = (
    TPL10_PROTOTYPE_ROOT
    / (
        f"{prototype_10_document_id}"
        "_TPL-10_prototype.pdf"
    )
)

prototype_10_png_path = (
    TPL10_PROTOTYPE_ROOT
    / (
        f"{prototype_10_document_id}"
        "_TPL-10_preview.png"
    )
)

prototype_10_ground_truth_path = (
    TPL10_PROTOTYPE_ROOT
    / (
        f"{prototype_10_document_id}"
        "_TPL-10_ground_truth.json"
    )
)


# ================================================================
# Render prototype
# ================================================================

(
    prototype_10_annotations,
    prototype_10_rendering_metadata,
) = render_template_10(
    render_payload=(
        prototype_10_render_payload
    ),
    output_pdf_path=(
        prototype_10_pdf_path
    ),
)


# ================================================================
# Simpan ground truth
# ================================================================

prototype_10_ground_truth = {
    "schema_version": "1.0.0",
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "document": {
        "document_id": (
            prototype_10_document_id
        ),
        "canonical_invoice_id": (
            prototype_10_canonical_id
        ),
        "split": (
            prototype_10_render_payload[
                "split"
            ]
        ),
        "template_id": (
            TPL10_TEMPLATE_ID
        ),
        "language": (
            prototype_10_render_payload[
                "language"
            ]
        ),
        "currency": (
            prototype_10_render_payload[
                "currency"
            ]
        ),
    },
    "canonical": (
        prototype_10_canonical_payload
    ),
    "rendering": (
        prototype_10_rendering_metadata
    ),
    "annotations": [
        annotation_to_dict(
            annotation
        )
        for annotation
        in prototype_10_annotations
    ],
}

write_json_atomically(
    prototype_10_ground_truth_path,
    prototype_10_ground_truth,
)


# ================================================================
# Buka ulang PDF dan buat preview
# ================================================================

temporary_png_path = (
    prototype_10_png_path.with_name(
        f".{prototype_10_png_path.stem}"
        ".tmp.png"
    )
)

temporary_png_path.unlink(
    missing_ok=True
)

try:
    with pymupdf.open(
        str(prototype_10_pdf_path)
    ) as prototype_document:
        if (
            prototype_document.page_count
            != 1
        ):
            raise RuntimeError(
                "Prototype TPL-10 harus "
                "memiliki tepat satu halaman."
            )

        prototype_page = (
            prototype_document[0]
        )

        extracted_text_10 = (
            prototype_page.get_text(
                "text"
            )
        )

        prototype_pixmap_10 = (
            prototype_page.get_pixmap(
                matrix=pymupdf.Matrix(
                    150 / 72,
                    150 / 72,
                ),
                alpha=False,
            )
        )

        prototype_pixmap_10.save(
            str(temporary_png_path)
        )

    if (
        not temporary_png_path.is_file()
        or temporary_png_path.stat().st_size
        == 0
    ):
        raise RuntimeError(
            "Preview sementara TPL-10 "
            "gagal dibuat."
        )

    os.replace(
        temporary_png_path,
        prototype_10_png_path,
    )

except Exception:
    temporary_png_path.unlink(
        missing_ok=True
    )
    raise


# ================================================================
# Post-render controls
# ================================================================

required_text_fragments_10 = [
    (
        prototype_10_render_payload[
            "metadata"
        ]["invoice_number"][
            "display_value"
        ]
    ),
    (
        prototype_10_render_payload[
            "vendor"
        ]["name"]
    ),
    (
        prototype_10_render_payload[
            "buyer"
        ]["name"]
    ),
    (
        prototype_10_render_payload[
            "labels"
        ]["quantity"]
    ),
    (
        prototype_10_render_payload[
            "financials"
        ]["total"]["display_value"]
    ),
    normalize_pdf_notice(
        prototype_10_render_payload[
            "synthetic_notice"
        ]
    ),
]

missing_text_fragments_10 = [
    text_fragment
    for text_fragment
    in required_text_fragments_10
    if (
        text_fragment
        not in extracted_text_10
    )
]

if missing_text_fragments_10:
    raise RuntimeError(
        "Teks penting TPL-10 tidak ditemukan "
        "pada hasil ekstraksi PDF: "
        f"{missing_text_fragments_10}"
    )

annotation_names_10 = {
    annotation.field_name
    for annotation
    in prototype_10_annotations
}

required_annotation_names_10 = {
    "vendor.name",
    "vendor.address_lines",
    "vendor.email",
    "vendor.phone",
    "vendor.tax_identifier",
    "buyer.name",
    "buyer.address_lines",
    "buyer.email",
    "buyer.phone",
    "buyer.tax_identifier",
    "invoice_number",
    "invoice_date",
    "due_date",
    "currency",
    "financials.subtotal",
    "financials.tax",
    "financials.discount",
    "financials.total",
    "document.synthetic_notice",
}

missing_annotations_10 = sorted(
    required_annotation_names_10
    - annotation_names_10
)

if missing_annotations_10:
    raise RuntimeError(
        "Anotasi wajib TPL-10 "
        "tidak tersedia: "
        f"{missing_annotations_10}"
    )

layout_metrics_10 = (
    prototype_10_rendering_metadata[
        "layout_metrics"
    ]
)

prototype_10_summary = pd.DataFrame(
    [        {
            "control": "pdf_created",
            "actual": prototype_10_pdf_path.is_file(),
            "status": "VALID",
        },
        {
            "control": "ground_truth_created",
            "actual": (
                prototype_10_ground_truth_path.is_file()
            ),
            "status": "VALID",
        },
        {
            "control": "annotation_count",
            "actual": len(
                prototype_10_annotations
            ),
            "status": "VALID",
        },
        {
            "control": "item_count",
            "actual": layout_metrics_10[
                "item_count"
            ],
            "status": "VALID",
        },
        {
            "control": "minimum_field_font",
            "actual": (
                prototype_10_rendering_metadata[
                    "minimum_field_font_size"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "font_adjustments",
            "actual": (
                prototype_10_rendering_metadata[
                    "font_adjustment_count"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "table_to_totals_gap",
            "actual": layout_metrics_10[
                "table_to_totals_gap_points"
            ],
            "status": "VALID",
        },
        {
            "control": "missing_required_text",
            "actual": len(
                missing_text_fragments_10
            ),
            "status": "VALID",
        },
    ]
)

display(
    prototype_10_summary
)

with PILImage.open(
    prototype_10_png_path
) as prototype_image:
    display(
        prototype_image.copy()
    )


# ================================================================
# Ringkasan
# ================================================================

print(
    f"Canonical ID       : "
    f"{prototype_10_canonical_id}"
)

print(
    f"Document ID        : "
    f"{prototype_10_document_id}"
)

print(
    f"Prototype PDF      : "
    f"{prototype_10_pdf_path}"
)

print(
    f"Prototype preview  : "
    f"{prototype_10_png_path}"
)

print(
    f"Ground truth       : "
    f"{prototype_10_ground_truth_path}"
)

print(
    f"Annotations        : "
    f"{len(prototype_10_annotations)}"
)

print(
    "Minimum field font : "
    f"{prototype_10_rendering_metadata['minimum_field_font_size']}"
)

print(
    "Adjusted fields    : "
    f"{prototype_10_rendering_metadata['font_adjustment_count']}"
)

print(
    "Table-total gap    : "
    f"{layout_metrics_10['table_to_totals_gap_points']} pt"
)

print(
    f"Renderer version   : "
    f"{TPL10_RENDERER_VERSION}"
)

print()

print(
    "✅ Prototype TPL-10 berhasil dibuat dan "
    "lolos pemeriksaan teknis awal."
)

In [ ]:
# ================================================================
# CELL 68 — FINAL
# Audit QA teknis prototype TPL-10
# ================================================================

from pathlib import Path
from typing import Any

import hashlib
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd
import pymupdf
from PIL import Image as PILImage


# ================================================================
# Validasi runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "TPL10_PROTOTYPE_ROOT",
    "TPL10_TEMPLATE_ID",
    "prototype_10_canonical_id",
    "prototype_10_document_id",
    "prototype_10_pdf_path",
    "prototype_10_png_path",
    "prototype_10_ground_truth_path",
    "prototype_10_render_payload",
    "prototype_10_rendering_metadata",
    "prototype_10_annotations",
    "write_json_atomically",
]

missing_runtime_objects = [
    name
    for name in REQUIRED_RUNTIME_OBJECTS
    if name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 68 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali Cell 67."
    )


# ================================================================
# Konfigurasi audit
# ================================================================

QA_SCHEMA_VERSION = "1.0.0"
QA_AUDITOR_VERSION = "1.0.0"

MINIMUM_FIELD_FONT_SIZE = 6.5
MAXIMUM_FONT_ADJUSTMENT_RATIO = 0.20
MINIMUM_RASTER_CONTENT_RATIO = 0.01
MAXIMUM_RASTER_CONTENT_RATIO = 0.40
MINIMUM_RASTER_CONTRAST = 10.0
SEVERE_OVERLAP_THRESHOLD = 0.50


# ================================================================
# Path artifact
# ================================================================

prototype_10_pdf_path = Path(
    prototype_10_pdf_path
)

prototype_10_png_path = Path(
    prototype_10_png_path
)

prototype_10_ground_truth_path = Path(
    prototype_10_ground_truth_path
)

prototype_10_qa_report_path = (
    Path(TPL10_PROTOTYPE_ROOT)
    / (
        f"{prototype_10_document_id}"
        "_TPL-10_qa_report.json"
    )
)

required_files = [
    prototype_10_pdf_path,
    prototype_10_png_path,
    prototype_10_ground_truth_path,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Artifact TPL-10 belum lengkap: "
        f"{missing_files}"
    )


# ================================================================
# Helper
# ================================================================

def qa_sha256(
    file_path: Path,
) -> str:
    digest = hashlib.sha256()

    with Path(file_path).open("rb") as input_file:
        for chunk in iter(
            lambda: input_file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def qa_normalize_text(
    value: Any,
) -> str:
    text = unicodedata.normalize(
        "NFKC",
        str(value or ""),
    )

    text = (
        text
        .replace("\u2014", "-")
        .replace("\u2013", "-")
        .replace("\u00b7", "-")
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip().casefold()


def qa_normalize_notice(
    value: Any,
) -> str:
    return (
        str(value)
        .replace("\u2014", "\u00b7")
        .replace("\u2013", "-")
    )


def qa_first_present(
    mapping: dict[str, Any],
    keys: tuple[str, ...],
) -> Any:
    for key in keys:
        if key in mapping:
            return mapping[key]

    return None


def qa_annotation_name(
    annotation: dict[str, Any],
) -> str:
    value = qa_first_present(
        annotation,
        (
            "field_name",
            "field_path",
            "field",
            "path",
            "name",
        ),
    )

    if value is None:
        raise KeyError(
            "Nama field anotasi tidak ditemukan."
        )

    return str(value)


def qa_annotation_text(
    annotation: dict[str, Any],
) -> str:
    value = qa_first_present(
        annotation,
        (
            "text",
            "value",
            "display_value",
            "field_value",
            "raw_value",
            "raw_text",
            "text_value",
            "content",
        ),
    )

    if isinstance(value, dict):
        value = qa_first_present(
            value,
            (
                "text",
                "value",
                "display_value",
                "raw",
            ),
        )

    if value is None:
        raise KeyError(
            "Teks anotasi tidak ditemukan."
        )

    return str(value)


def qa_bbox_from_value(
    value: Any,
) -> tuple[float, float, float, float] | None:
    if isinstance(value, (list, tuple)):
        if len(value) != 4:
            return None

        try:
            return tuple(float(number) for number in value)
        except (TypeError, ValueError):
            return None

    if not isinstance(value, dict):
        return None

    for key_set in (
        ("x0", "y0", "x1", "y1"),
        ("left", "top", "right", "bottom"),
    ):
        if all(key in value for key in key_set):
            try:
                return tuple(
                    float(value[key])
                    for key in key_set
                )
            except (TypeError, ValueError):
                return None

    for key_set in (
        ("x", "y", "width", "height"),
        ("x", "y", "w", "h"),
    ):
        if all(key in value for key in key_set):
            try:
                x0 = float(value["x"])
                y0 = float(value["y"])
                width = float(value[key_set[2]])
                height = float(value[key_set[3]])

                return (
                    x0,
                    y0,
                    x0 + width,
                    y0 + height,
                )
            except (TypeError, ValueError):
                return None

    return None


def qa_annotation_bbox(
    annotation: dict[str, Any],
) -> tuple[float, float, float, float]:
    value = qa_first_present(
        annotation,
        (
            "bbox",
            "bbox_points",
            "bbox_pt",
            "bbox_pdf",
            "bounding_box",
            "rectangle",
            "rect",
            "coordinates",
            "coordinates_points",
        ),
    )

    if value is None:
        geometry = annotation.get("geometry")

        if isinstance(geometry, dict):
            value = qa_first_present(
                geometry,
                (
                    "bbox",
                    "bounding_box",
                    "rectangle",
                ),
            )

    bbox = qa_bbox_from_value(value)

    if bbox is None:
        raise KeyError(
            "Bounding box anotasi tidak ditemukan."
        )

    return bbox


def qa_rectangle_area(
    bbox: tuple[float, float, float, float],
) -> float:
    x0, y0, x1, y1 = bbox

    return (
        max(0.0, x1 - x0)
        * max(0.0, y1 - y0)
    )


def qa_intersection_area(
    first_bbox: tuple[float, float, float, float],
    second_bbox: tuple[float, float, float, float],
) -> float:
    first_x0, first_y0, first_x1, first_y1 = first_bbox
    second_x0, second_y0, second_x1, second_y1 = second_bbox

    width = max(
        0.0,
        min(first_x1, second_x1)
        - max(first_x0, second_x0),
    )

    height = max(
        0.0,
        min(first_y1, second_y1)
        - max(first_y0, second_y0),
    )

    return width * height


# ================================================================
# Muat ground truth
# ================================================================

with prototype_10_ground_truth_path.open(
    "r",
    encoding="utf-8",
) as ground_truth_file:
    ground_truth = json.load(
        ground_truth_file
    )

ground_truth_document = ground_truth.get(
    "document",
    {},
)

if (
    ground_truth_document.get("document_id")
    != prototype_10_document_id
):
    raise RuntimeError(
        "Document ID ground truth tidak sesuai."
    )

if (
    ground_truth_document.get("template_id")
    != TPL10_TEMPLATE_ID
):
    raise RuntimeError(
        "Ground truth bukan milik TPL-10."
    )

annotation_records = ground_truth.get(
    "annotations",
    [],
)

if (
    not isinstance(annotation_records, list)
    or not annotation_records
):
    raise RuntimeError(
        "Ground truth tidak memiliki daftar anotasi valid."
    )


# ================================================================
# Expected values
# ================================================================

render_payload = prototype_10_render_payload

expected_values = {
    "invoice_number": str(
        render_payload["metadata"]
        ["invoice_number"]["display_value"]
    ),
    "invoice_date": str(
        render_payload["metadata"]
        ["invoice_date"]["display_value"]
    ),
    "due_date": str(
        render_payload["metadata"]
        ["due_date"]["display_value"]
    ),
    "currency": str(
        render_payload["currency"]
    ),
}

for party_type in ("vendor", "buyer"):
    party = render_payload[party_type]

    expected_values.update(
        {
            f"{party_type}.name": str(
                party["name"]
            ),
            f"{party_type}.address_lines": "\n".join(
                party["address_lines"]
            ),
            f"{party_type}.email": str(
                party["email"]
            ),
            f"{party_type}.phone": str(
                party["phone"]
            ),
            f"{party_type}.tax_identifier": str(
                party["tax_identifier_display"]
            ),
        }
    )

for item_index, item in enumerate(
    render_payload["items"]
):
    for field_name in (
        "description",
        "quantity",
        "unit_price",
        "line_total",
    ):
        expected_values[
            f"items[{item_index}].{field_name}"
        ] = str(item[field_name])

for field_name in (
    "subtotal",
    "tax",
    "discount",
    "total",
):
    expected_values[
        f"financials.{field_name}"
    ] = str(
        render_payload["financials"]
        [field_name]["display_value"]
    )

expected_values[
    "document.synthetic_notice"
] = qa_normalize_notice(
    render_payload["synthetic_notice"]
)

expected_annotation_count = len(
    expected_values
)


# ================================================================
# Parse annotations
# ================================================================

parsed_annotations = []
annotation_parse_failures = []

for annotation_index, annotation in enumerate(
    annotation_records
):
    try:
        if not isinstance(annotation, dict):
            raise TypeError(
                "Anotasi wajib berupa dictionary."
            )

        parsed_annotations.append(
            {
                "index": annotation_index,
                "field_name": qa_annotation_name(
                    annotation
                ),
                "text": qa_annotation_text(
                    annotation
                ),
                "bbox": qa_annotation_bbox(
                    annotation
                ),
            }
        )

    except (KeyError, TypeError, ValueError) as error:
        annotation_parse_failures.append(
            {
                "index": annotation_index,
                "error": str(error),
            }
        )

if annotation_parse_failures:
    raise RuntimeError(
        "Schema anotasi tidak dapat diproses: "
        f"{annotation_parse_failures[:3]}"
    )

annotation_names = [
    annotation["field_name"]
    for annotation in parsed_annotations
]

annotation_by_name = {
    annotation["field_name"]: annotation
    for annotation in parsed_annotations
}

duplicate_annotation_count = (
    len(annotation_names)
    - len(set(annotation_names))
)

missing_required_annotations = sorted(
    set(expected_values)
    - set(annotation_names)
)


# ================================================================
# Audit PDF dan word coverage
# ================================================================

with pymupdf.open(
    str(prototype_10_pdf_path)
) as pdf_document:
    pdf_page_count = pdf_document.page_count

    if pdf_page_count < 1:
        raise RuntimeError(
            "PDF TPL-10 tidak memiliki halaman."
        )

    first_page = pdf_document[0]
    page_rectangle = first_page.rect
    extracted_text = first_page.get_text("text")
    extracted_words = first_page.get_text("words")

page_width = float(page_rectangle.width)
page_height = float(page_rectangle.height)

word_boxes = [
    (
        float(word[0]),
        float(word[1]),
        float(word[2]),
        float(word[3]),
    )
    for word in extracted_words
    if (
        len(word) >= 5
        and str(word[4]).strip()
    )
]

invalid_bounding_boxes = []
annotations_without_words = []

for annotation in parsed_annotations:
    field_name = annotation["field_name"]
    x0, y0, x1, y1 = annotation["bbox"]

    bbox_is_valid = (
        all(
            math.isfinite(value)
            for value in (x0, y0, x1, y1)
        )
        and x0 >= 0.0
        and y0 >= 0.0
        and x1 <= page_width
        and y1 <= page_height
        and x1 > x0
        and y1 > y0
    )

    if not bbox_is_valid:
        invalid_bounding_boxes.append(
            field_name
        )
        continue

    contains_word = any(
        qa_intersection_area(
            annotation["bbox"],
            word_box,
        ) > 0.0
        for word_box in word_boxes
    )

    if not contains_word:
        annotations_without_words.append(
            field_name
        )


# ================================================================
# Audit nilai teks
# ================================================================

normalization_mismatches = []

for field_name, expected_value in expected_values.items():
    annotation = annotation_by_name.get(
        field_name
    )

    if annotation is None:
        continue

    actual_value = annotation["text"]

    if (
        qa_normalize_text(actual_value)
        != qa_normalize_text(expected_value)
    ):
        normalization_mismatches.append(
            {
                "field_name": field_name,
                "expected": expected_value,
                "actual": actual_value,
            }
        )

normalized_extracted_text = qa_normalize_text(
    extracted_text
)

missing_extracted_values = []

for annotation in parsed_annotations:
    normalized_value = qa_normalize_text(
        annotation["text"]
    )

    if (
        normalized_value
        and normalized_value
        not in normalized_extracted_text
    ):
        missing_extracted_values.append(
            annotation["field_name"]
        )


# ================================================================
# Audit overlap
# ================================================================

severe_annotation_overlaps = []

for first_index in range(
    len(parsed_annotations)
):
    first_annotation = parsed_annotations[
        first_index
    ]

    first_area = qa_rectangle_area(
        first_annotation["bbox"]
    )

    if first_area <= 0.0:
        continue

    for second_index in range(
        first_index + 1,
        len(parsed_annotations),
    ):
        second_annotation = parsed_annotations[
            second_index
        ]

        second_area = qa_rectangle_area(
            second_annotation["bbox"]
        )

        if second_area <= 0.0:
            continue

        overlap_area = qa_intersection_area(
            first_annotation["bbox"],
            second_annotation["bbox"],
        )

        overlap_ratio = (
            overlap_area
            / min(first_area, second_area)
        )

        if (
            overlap_ratio
            >= SEVERE_OVERLAP_THRESHOLD
        ):
            severe_annotation_overlaps.append(
                {
                    "first": first_annotation[
                        "field_name"
                    ],
                    "second": second_annotation[
                        "field_name"
                    ],
                    "overlap_ratio": round(
                        overlap_ratio,
                        6,
                    ),
                }
            )


# ================================================================
# Audit font dan raster
# ================================================================

minimum_field_font_size = float(
    prototype_10_rendering_metadata[
        "minimum_field_font_size"
    ]
)

font_adjustment_count = int(
    prototype_10_rendering_metadata[
        "font_adjustment_count"
    ]
)

font_adjustment_ratio = (
    font_adjustment_count
    / max(1, len(parsed_annotations))
)

with PILImage.open(
    prototype_10_png_path
) as preview_image:
    grayscale_array = np.asarray(
        preview_image.convert("L"),
        dtype=np.float32,
    ).copy()

if grayscale_array.size == 0:
    raise RuntimeError(
        "Preview PNG tidak memiliki piksel."
    )

raster_content_ratio = float(
    np.mean(grayscale_array < 245.0)
)

raster_contrast = float(
    np.std(grayscale_array)
)


# ================================================================
# Susun checks
# ================================================================

checks = [
    {
        "control": "pdf_page_count",
        "expected": 1,
        "actual": pdf_page_count,
        "valid": pdf_page_count == 1,
    },
    {
        "control": "annotation_count",
        "expected": expected_annotation_count,
        "actual": len(parsed_annotations),
        "valid": (
            len(parsed_annotations)
            == expected_annotation_count
        ),
    },
    {
        "control": "duplicate_annotations",
        "expected": 0,
        "actual": duplicate_annotation_count,
        "valid": duplicate_annotation_count == 0,
    },
    {
        "control": "missing_required_annotations",
        "expected": 0,
        "actual": len(
            missing_required_annotations
        ),
        "valid": not missing_required_annotations,
    },
    {
        "control": "invalid_bounding_boxes",
        "expected": 0,
        "actual": len(
            invalid_bounding_boxes
        ),
        "valid": not invalid_bounding_boxes,
    },
    {
        "control": "normalization_mismatches",
        "expected": 0,
        "actual": len(
            normalization_mismatches
        ),
        "valid": not normalization_mismatches,
    },
    {
        "control": "annotations_without_words",
        "expected": 0,
        "actual": len(
            annotations_without_words
        ),
        "valid": not annotations_without_words,
    },
    {
        "control": "severe_annotation_overlaps",
        "expected": 0,
        "actual": len(
            severe_annotation_overlaps
        ),
        "valid": not severe_annotation_overlaps,
    },
    {
        "control": "missing_extracted_values",
        "expected": 0,
        "actual": len(
            missing_extracted_values
        ),
        "valid": not missing_extracted_values,
    },
    {
        "control": "minimum_field_font_size",
        "expected": ">=6.5",
        "actual": round(
            minimum_field_font_size,
            4,
        ),
        "valid": (
            minimum_field_font_size
            >= MINIMUM_FIELD_FONT_SIZE
        ),
    },
    {
        "control": "font_adjustment_ratio",
        "expected": "<=0.20",
        "actual": round(
            font_adjustment_ratio,
            4,
        ),
        "valid": (
            font_adjustment_ratio
            <= MAXIMUM_FONT_ADJUSTMENT_RATIO
        ),
    },
    {
        "control": "raster_content_ratio",
        "expected": "0.01–0.40",
        "actual": round(
            raster_content_ratio,
            4,
        ),
        "valid": (
            MINIMUM_RASTER_CONTENT_RATIO
            <= raster_content_ratio
            <= MAXIMUM_RASTER_CONTENT_RATIO
        ),
    },
    {
        "control": "raster_contrast",
        "expected": ">=10.0",
        "actual": round(
            raster_contrast,
            4,
        ),
        "valid": (
            raster_contrast
            >= MINIMUM_RASTER_CONTRAST
        ),
    },
]

for check in checks:
    check["status"] = (
        "VALID"
        if check["valid"]
        else "INVALID"
    )

failed_checks = [
    check
    for check in checks
    if not check["valid"]
]

qa_status = (
    "PASSED"
    if not failed_checks
    else "FAILED"
)


# ================================================================
# Simpan QA report
# ================================================================

qa_report = {
    "schema_version": QA_SCHEMA_VERSION,
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "document_id": prototype_10_document_id,
    "canonical_invoice_id": (
        prototype_10_canonical_id
    ),
    "template_id": TPL10_TEMPLATE_ID,
    "status": qa_status,
    "manual_visual_review": "PENDING",
    "metrics": {
        "auditor_version": QA_AUDITOR_VERSION,
        "pdf_page_count": pdf_page_count,
        "annotation_count": len(
            parsed_annotations
        ),
        "expected_annotation_count": (
            expected_annotation_count
        ),
        "duplicate_annotation_count": (
            duplicate_annotation_count
        ),
        "missing_required_annotation_count": len(
            missing_required_annotations
        ),
        "invalid_bounding_box_count": len(
            invalid_bounding_boxes
        ),
        "normalization_mismatch_count": len(
            normalization_mismatches
        ),
        "annotations_without_words_count": len(
            annotations_without_words
        ),
        "severe_annotation_overlap_count": len(
            severe_annotation_overlaps
        ),
        "missing_extracted_value_count": len(
            missing_extracted_values
        ),
        "minimum_field_font_size": round(
            minimum_field_font_size,
            6,
        ),
        "font_adjustment_count": (
            font_adjustment_count
        ),
        "font_adjustment_ratio": round(
            font_adjustment_ratio,
            6,
        ),
        "raster_content_ratio": round(
            raster_content_ratio,
            6,
        ),
        "raster_contrast": round(
            raster_contrast,
            6,
        ),
    },
    "failures": {
        "failed_checks": [
            check["control"]
            for check in failed_checks
        ],
        "missing_required_annotations": (
            missing_required_annotations
        ),
        "invalid_bounding_boxes": (
            invalid_bounding_boxes
        ),
        "normalization_mismatches": (
            normalization_mismatches
        ),
        "annotations_without_words": (
            annotations_without_words
        ),
        "severe_annotation_overlaps": (
            severe_annotation_overlaps
        ),
        "missing_extracted_values": (
            missing_extracted_values
        ),
    },
    "checksums_sha256": {
        "prototype_pdf": qa_sha256(
            prototype_10_pdf_path
        ),
        "preview": qa_sha256(
            prototype_10_png_path
        ),
        "ground_truth": qa_sha256(
            prototype_10_ground_truth_path
        ),
    },
}

write_json_atomically(
    prototype_10_qa_report_path,
    qa_report,
)


# ================================================================
# Verifikasi dan output
# ================================================================

with prototype_10_qa_report_path.open(
    "r",
    encoding="utf-8",
) as qa_report_file:
    verified_qa_report = json.load(
        qa_report_file
    )

if verified_qa_report != qa_report:
    raise RuntimeError(
        "QA report berubah setelah round-trip."
    )

qa_summary = pd.DataFrame(
    [
        {
            "control": check["control"],
            "expected": check["expected"],
            "actual": check["actual"],
            "status": check["status"],
        }
        for check in checks
    ]
)

display(qa_summary)

print(
    f"QA report          : "
    f"{prototype_10_qa_report_path}"
)
print(
    f"QA status          : {qa_status}"
)
print(
    "Minimum field font : "
    f"{minimum_field_font_size:.2f}"
)
print(
    "Font adjustments   : "
    f"{font_adjustment_count}"
)
print(
    "Ink pixel ratio    : "
    f"{raster_content_ratio:.6f}"
)
print(
    "Raster contrast    : "
    f"{raster_contrast:.6f}"
)
print(
    "Manual review      : "
    f"{qa_report['manual_visual_review']}"
)
print()

if qa_status != "PASSED":
    print("Detail kegagalan:")
    print(
        json.dumps(
            qa_report["failures"],
            ensure_ascii=False,
            indent=2,
        )
    )

    raise RuntimeError(
        "Prototype TPL-10 gagal audit QA teknis."
    )

print(
    "✅ Prototype TPL-10 lulus audit teknis. "
    "Review visual manual masih perlu dicatat "
    "ke QA report."
)

In [ ]:
# ================================================================
# CELL 68A — FINAL
# Finalisasi review visual manual prototype TPL-10
# ================================================================

from pathlib import Path

import hashlib
import json


# ================================================================
# Validasi runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "prototype_10_qa_report_path",
    "prototype_10_png_path",
    "prototype_10_document_id",
    "write_json_atomically",
]

missing_runtime_objects = [
    name
    for name in REQUIRED_RUNTIME_OBJECTS
    if name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 68A belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali Cell 67 dan Cell 68."
    )


# ================================================================
# Helper checksum
# ================================================================

def calculate_tpl10_sha256(
    file_path: Path,
) -> str:
    digest = hashlib.sha256()

    with file_path.open("rb") as input_file:
        for chunk in iter(
            lambda: input_file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ================================================================
# Muat artifact
# ================================================================

qa_report_path = Path(
    prototype_10_qa_report_path
)

preview_path = Path(
    prototype_10_png_path
)

if not qa_report_path.is_file():
    raise FileNotFoundError(
        "QA report TPL-10 tidak ditemukan: "
        f"{qa_report_path}"
    )

if not preview_path.is_file():
    raise FileNotFoundError(
        "Preview TPL-10 tidak ditemukan: "
        f"{preview_path}"
    )

with qa_report_path.open(
    "r",
    encoding="utf-8",
) as qa_report_file:
    qa_report = json.load(
        qa_report_file
    )


# ================================================================
# Validasi identitas dan audit teknis
# ================================================================

if qa_report.get("template_id") != "TPL-10":
    raise RuntimeError(
        "QA report bukan milik TPL-10."
    )

if (
    qa_report.get("document_id")
    != prototype_10_document_id
):
    raise RuntimeError(
        "Document ID QA report tidak sesuai. "
        f"Expected: {prototype_10_document_id!r}; "
        f"actual: {qa_report.get('document_id')!r}."
    )

if qa_report.get("status") != "PASSED":
    raise RuntimeError(
        "Review visual tidak boleh difinalisasi "
        "karena audit teknis belum PASSED. "
        f"Status: {qa_report.get('status')!r}."
    )

failure_details = qa_report.get(
    "failures",
    {},
)

if not isinstance(failure_details, dict):
    raise TypeError(
        "Field 'failures' wajib berupa dictionary."
    )

nonempty_failures = {
    failure_name: failure_value
    for failure_name, failure_value
    in failure_details.items()
    if bool(failure_value)
}

if nonempty_failures:
    raise RuntimeError(
        "QA report masih memiliki kegagalan teknis: "
        f"{nonempty_failures}"
    )

if "manual_visual_review" not in qa_report:
    raise KeyError(
        "Field 'manual_visual_review' tidak ditemukan."
    )

if qa_report["manual_visual_review"] not in {
    "PENDING",
    "PASSED",
}:
    raise RuntimeError(
        "Status manual visual review tidak dikenali: "
        f"{qa_report['manual_visual_review']!r}."
    )


# ================================================================
# Verifikasi checksum preview
# ================================================================

recorded_preview_checksum = (
    qa_report
    .get("checksums_sha256", {})
    .get("preview")
)

if not recorded_preview_checksum:
    raise KeyError(
        "Checksum preview tidak ditemukan pada "
        "qa_report['checksums_sha256']['preview']."
    )

actual_preview_checksum = (
    calculate_tpl10_sha256(
        preview_path
    )
)

if (
    actual_preview_checksum
    != recorded_preview_checksum
):
    raise RuntimeError(
        "Preview berubah setelah audit teknis. "
        "Jalankan ulang Cell 68 dan lakukan "
        "review visual kembali. "
        f"Expected SHA-256: "
        f"{recorded_preview_checksum}; "
        f"actual: {actual_preview_checksum}."
    )


# ================================================================
# Simpan status manual secara idempoten
# ================================================================

qa_report[
    "manual_visual_review"
] = "PASSED"

write_json_atomically(
    qa_report_path,
    qa_report,
)


# ================================================================
# Verifikasi persistensi
# ================================================================

with qa_report_path.open(
    "r",
    encoding="utf-8",
) as qa_report_file:
    persisted_qa_report = json.load(
        qa_report_file
    )

if (
    persisted_qa_report.get("template_id")
    != "TPL-10"
):
    raise RuntimeError(
        "Template ID berubah setelah finalisasi."
    )

if (
    persisted_qa_report.get("document_id")
    != prototype_10_document_id
):
    raise RuntimeError(
        "Document ID berubah setelah finalisasi."
    )

if (
    persisted_qa_report.get("status")
    != "PASSED"
):
    raise RuntimeError(
        "Status teknis berubah setelah finalisasi."
    )

if (
    persisted_qa_report.get(
        "manual_visual_review"
    )
    != "PASSED"
):
    raise RuntimeError(
        "Status review visual gagal disimpan."
    )

persisted_preview_checksum = (
    persisted_qa_report
    .get("checksums_sha256", {})
    .get("preview")
)

if (
    persisted_preview_checksum
    != actual_preview_checksum
):
    raise RuntimeError(
        "Checksum preview berubah "
        "pada QA report."
    )


# ================================================================
# Ringkasan
# ================================================================

print(
    f"QA report           : "
    f"{qa_report_path}"
)

print(
    "Technical status    : "
    f"{persisted_qa_report['status']}"
)

print(
    "Manual visual review: "
    f"{persisted_qa_report['manual_visual_review']}"
)

print(
    "Preview SHA-256     : "
    f"{persisted_preview_checksum}"
)

print()

print(
    "✅ TPL-10 FINAL PASSED — audit teknis dan "
    "review visual telah selesai."
)

In [ ]:
# ================================================================
# CELL 69 — FINAL
# Pemeriksaan ulang artifact prototype TPL-01
# Read-only: tidak membuat atau mengubah artifact
# ================================================================

from pathlib import Path

import hashlib
import json

import pandas as pd
import pymupdf
from PIL import Image as PILImage
from IPython.display import display


# ================================================================
# Validasi runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "PROTOTYPE_ROOT",
    "invoice_render_payloads",
]

missing_runtime_objects = [
    name
    for name in REQUIRED_RUNTIME_OBJECTS
    if name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 69 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali cell pembentukan "
        "presentation payload."
    )


# ================================================================
# Helper
# ================================================================

def tpl01_sha256(
    file_path: Path,
) -> str | None:
    if not file_path.is_file():
        return None

    digest = hashlib.sha256()

    with file_path.open("rb") as input_file:
        for chunk in iter(
            lambda: input_file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def tpl01_pdf_notice(
    value: str,
) -> str:
    return (
        str(value)
        .replace("—", "·")
        .replace("–", "-")
    )


# ================================================================
# Temukan payload prototype TPL-01
# ================================================================

TPL01_TEMPLATE_ID = "TPL-01"

tpl01_candidate_ids = sorted(
    canonical_id
    for canonical_id, payload
    in invoice_render_payloads.items()
    if (
        isinstance(payload, dict)
        and payload.get("template_id")
        == TPL01_TEMPLATE_ID
    )
)

if len(tpl01_candidate_ids) != 20:
    raise RuntimeError(
        "TPL-01 harus memiliki tepat 20 payload. "
        f"Actual: {len(tpl01_candidate_ids)}."
    )

prototype_01_canonical_id = (
    tpl01_candidate_ids[0]
)

prototype_01_render_payload = (
    invoice_render_payloads[
        prototype_01_canonical_id
    ]
)

prototype_01_document_id = (
    prototype_01_render_payload[
        "document_id"
    ]
)

TPL01_PROTOTYPE_ROOT = (
    Path(PROTOTYPE_ROOT)
    / TPL01_TEMPLATE_ID
)

prototype_01_pdf_path = (
    TPL01_PROTOTYPE_ROOT
    / (
        f"{prototype_01_document_id}"
        "_TPL-01_prototype.pdf"
    )
)

prototype_01_png_path = (
    TPL01_PROTOTYPE_ROOT
    / (
        f"{prototype_01_document_id}"
        "_TPL-01_preview.png"
    )
)

prototype_01_ground_truth_path = (
    TPL01_PROTOTYPE_ROOT
    / (
        f"{prototype_01_document_id}"
        "_TPL-01_ground_truth.json"
    )
)

prototype_01_qa_report_path = (
    TPL01_PROTOTYPE_ROOT
    / (
        f"{prototype_01_document_id}"
        "_TPL-01_qa_report.json"
    )
)


# ================================================================
# Periksa label presentation payload terbaru
# ================================================================

current_quantity_label = (
    prototype_01_render_payload[
        "labels"
    ]["quantity"]
)

current_line_total_label = (
    prototype_01_render_payload[
        "labels"
    ]["line_total"]
)

expected_notice_text = (
    tpl01_pdf_notice(
        prototype_01_render_payload[
            "synthetic_notice"
        ]
    )
)

if (
    prototype_01_render_payload["language"]
    == "id"
    and current_quantity_label
    != "Kuantitas"
):
    raise RuntimeError(
        "Presentation payload TPL-01 masih memakai "
        "label kuantitas yang salah. "
        f"Actual: {current_quantity_label!r}."
    )


# ================================================================
# Periksa PDF lama
# ================================================================

pdf_page_count = None
pdf_extracted_text = ""
pdf_contains_quantity_label = False
pdf_contains_notice = False

if prototype_01_pdf_path.is_file():
    with pymupdf.open(
        str(prototype_01_pdf_path)
    ) as pdf_document:
        pdf_page_count = (
            pdf_document.page_count
        )

        if pdf_page_count >= 1:
            pdf_extracted_text = (
                pdf_document[0].get_text(
                    "text"
                )
            )

    pdf_contains_quantity_label = (
        current_quantity_label
        in pdf_extracted_text
    )

    pdf_contains_notice = (
        expected_notice_text
        in pdf_extracted_text
    )


# ================================================================
# Periksa QA report lama
# ================================================================

qa_status = "MISSING"
manual_visual_review = "MISSING"
recorded_preview_checksum = None

if prototype_01_qa_report_path.is_file():
    with prototype_01_qa_report_path.open(
        "r",
        encoding="utf-8",
    ) as qa_report_file:
        tpl01_qa_report = json.load(
            qa_report_file
        )

    qa_status = tpl01_qa_report.get(
        "status",
        "MISSING",
    )

    manual_visual_review = (
        tpl01_qa_report.get(
            "manual_visual_review",
            tpl01_qa_report.get(
                "manual_review",
                "MISSING",
            ),
        )
    )

    recorded_preview_checksum = (
        tpl01_qa_report
        .get("checksums_sha256", {})
        .get("preview")
    )


# ================================================================
# Periksa checksum preview
# ================================================================

actual_preview_checksum = (
    tpl01_sha256(
        prototype_01_png_path
    )
)

preview_checksum_matches = (
    recorded_preview_checksum is not None
    and actual_preview_checksum is not None
    and recorded_preview_checksum
    == actual_preview_checksum
)


# ================================================================
# Tentukan apakah perlu render ulang
# ================================================================

required_artifacts_exist = all(
    path.is_file()
    for path in (
        prototype_01_pdf_path,
        prototype_01_png_path,
        prototype_01_ground_truth_path,
    )
)

requires_rerender = (
    not required_artifacts_exist
    or pdf_page_count != 1
    or not pdf_contains_quantity_label
    or not pdf_contains_notice
)

requires_qa_rerun = (
    requires_rerender
    or qa_status != "PASSED"
    or manual_visual_review != "PASSED"
    or not preview_checksum_matches
)


# ================================================================
# Ringkasan
# ================================================================

tpl01_recheck_summary = pd.DataFrame(
    [
        {
            "control": "payload_quantity_label",
            "expected": "Kuantitas",
            "actual": current_quantity_label,
            "status": (
                "VALID"
                if current_quantity_label
                == "Kuantitas"
                else "INVALID"
            ),
        },
        {
            "control": "payload_line_total_label",
            "expected": "Jumlah",
            "actual": current_line_total_label,
            "status": (
                "VALID"
                if current_line_total_label
                == "Jumlah"
                else "INVALID"
            ),
        },
        {
            "control": "pdf_created",
            "expected": True,
            "actual": (
                prototype_01_pdf_path.is_file()
            ),
            "status": (
                "VALID"
                if prototype_01_pdf_path.is_file()
                else "INVALID"
            ),
        },
        {
            "control": "preview_created",
            "expected": True,
            "actual": (
                prototype_01_png_path.is_file()
            ),
            "status": (
                "VALID"
                if prototype_01_png_path.is_file()
                else "INVALID"
            ),
        },
        {
            "control": "ground_truth_created",
            "expected": True,
            "actual": (
                prototype_01_ground_truth_path
                .is_file()
            ),
            "status": (
                "VALID"
                if prototype_01_ground_truth_path
                .is_file()
                else "INVALID"
            ),
        },
        {
            "control": "pdf_page_count",
            "expected": 1,
            "actual": pdf_page_count,
            "status": (
                "VALID"
                if pdf_page_count == 1
                else "INVALID"
            ),
        },
        {
            "control": "pdf_quantity_label",
            "expected": current_quantity_label,
            "actual": (
                pdf_contains_quantity_label
            ),
            "status": (
                "VALID"
                if pdf_contains_quantity_label
                else "INVALID"
            ),
        },
        {
            "control": "pdf_synthetic_notice",
            "expected": expected_notice_text,
            "actual": pdf_contains_notice,
            "status": (
                "VALID"
                if pdf_contains_notice
                else "INVALID"
            ),
        },
        {
            "control": "qa_status",
            "expected": "PASSED",
            "actual": qa_status,
            "status": (
                "VALID"
                if qa_status == "PASSED"
                else "INVALID"
            ),
        },
        {
            "control": "manual_visual_review",
            "expected": "PASSED",
            "actual": manual_visual_review,
            "status": (
                "VALID"
                if manual_visual_review
                == "PASSED"
                else "INVALID"
            ),
        },
        {
            "control": "preview_checksum",
            "expected": "MATCH",
            "actual": (
                "MATCH"
                if preview_checksum_matches
                else "MISMATCH_OR_MISSING"
            ),
            "status": (
                "VALID"
                if preview_checksum_matches
                else "INVALID"
            ),
        },
    ]
)

display(
    tpl01_recheck_summary
)


# ================================================================
# Tampilkan preview lama
# ================================================================

if prototype_01_png_path.is_file():
    with PILImage.open(
        prototype_01_png_path
    ) as preview_image:
        display(
            preview_image.copy()
        )


# ================================================================
# Output
# ================================================================

print(
    f"Canonical ID       : "
    f"{prototype_01_canonical_id}"
)

print(
    f"Document ID        : "
    f"{prototype_01_document_id}"
)

print(
    f"Prototype PDF      : "
    f"{prototype_01_pdf_path}"
)

print(
    f"Prototype preview  : "
    f"{prototype_01_png_path}"
)

print(
    f"Ground truth       : "
    f"{prototype_01_ground_truth_path}"
)

print(
    f"QA report          : "
    f"{prototype_01_qa_report_path}"
)

print(
    f"Current quantity   : "
    f"{current_quantity_label}"
)

print(
    f"PDF has quantity   : "
    f"{pdf_contains_quantity_label}"
)

print(
    f"PDF has notice     : "
    f"{pdf_contains_notice}"
)

print(
    f"Technical QA       : "
    f"{qa_status}"
)

print(
    f"Manual review      : "
    f"{manual_visual_review}"
)

print(
    f"Requires rerender  : "
    f"{requires_rerender}"
)

print(
    f"Requires QA rerun  : "
    f"{requires_qa_rerun}"
)

print()

print(
    "✅ Pemeriksaan ulang TPL-01 selesai. "
    "Tidak ada artifact yang dibuat atau diubah."
)

In [ ]:
# ================================================================
# CELL 69A — Read-only
# Menampilkan kontrak layout TPL-01
# ================================================================

import json


if "template_registry" not in globals():
    raise RuntimeError(
        "template_registry belum tersedia."
    )

if "TPL-01" not in template_registry:
    raise KeyError(
        "TPL-01 tidak ditemukan dalam template_registry."
    )

tpl01_template_specification = (
    template_registry["TPL-01"]
)

if not isinstance(
    tpl01_template_specification,
    dict,
):
    raise TypeError(
        "Spesifikasi TPL-01 wajib berupa dictionary."
    )

print("TEMPLATE SPECIFICATION TPL-01")

print(
    json.dumps(
        tpl01_template_specification,
        ensure_ascii=False,
        indent=2,
    )
)

print()
print(
    "✅ Spesifikasi TPL-01 berhasil dibaca. "
    "Tidak ada artifact yang dibuat atau diubah."
)

In [ ]:
# ================================================================
# CELL 70A — FINAL
# Definisi renderer TPL-01: Classic Corporate
# Belum membuat PDF, PNG, atau ground truth
# ================================================================

from pathlib import Path
from typing import Any

import json
import os

import pymupdf


# ================================================================
# Validasi runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "PROTOTYPE_ROOT",
    "CANONICAL_RECORDS_JSONL_PATH",
    "FieldAnnotation",
    "template_registry",
    "invoice_render_payloads",
    "get_page_dimensions",
    "hex_to_rgb",
    "insert_annotated_text",
    "insert_static_text",
    "annotation_to_dict",
    "write_json_atomically",
    "RENDER_ENGINE_VERSION",
]

missing_runtime_objects = [
    name
    for name in REQUIRED_RUNTIME_OBJECTS
    if name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 70A belum tersedia: "
        f"{missing_runtime_objects}."
    )


# ================================================================
# Konfigurasi
# ================================================================

TPL01_TEMPLATE_ID = "TPL-01"
TPL01_RENDERER_VERSION = "1.0.0"

TPL01_MIN_ITEMS_PER_PAGE = 2
TPL01_MAX_ITEMS_PER_PAGE = 8

TPL01_PROTOTYPE_ROOT = (
    Path(PROTOTYPE_ROOT)
    / TPL01_TEMPLATE_ID
)

TPL01_PROTOTYPE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ================================================================
# Helper
# ================================================================

def load_tpl01_canonical_payload(
    canonical_id: str,
) -> dict[str, Any]:
    canonical_path = Path(
        CANONICAL_RECORDS_JSONL_PATH
    )

    if not canonical_path.is_file():
        raise FileNotFoundError(
            "Canonical JSONL tidak ditemukan: "
            f"{canonical_path}"
        )

    with canonical_path.open(
        "r",
        encoding="utf-8",
    ) as canonical_file:
        for line_number, line in enumerate(
            canonical_file,
            start=1,
        ):
            clean_line = line.strip()

            if not clean_line:
                continue

            try:
                payload = json.loads(
                    clean_line
                )
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    "Canonical JSONL rusak pada "
                    f"baris {line_number}."
                ) from error

            if (
                payload.get(
                    "canonical_invoice_id"
                )
                == canonical_id
            ):
                return payload

    raise KeyError(
        "Canonical payload tidak ditemukan: "
        f"{canonical_id}"
    )


def normalize_tpl01_notice(
    value: Any,
) -> str:
    return (
        str(value)
        .replace("—", "·")
        .replace("–", "-")
    )


# ================================================================
# Renderer
# ================================================================

def render_template_01(
    render_payload: dict[str, Any],
    output_pdf_path: Path,
) -> tuple[
    list[FieldAnnotation],
    dict[str, Any],
]:
    """Merender satu halaman TPL-01 Classic Corporate."""

    if not isinstance(render_payload, dict):
        raise TypeError(
            "render_payload wajib berupa dictionary."
        )

    if (
        render_payload.get("template_id")
        != TPL01_TEMPLATE_ID
    ):
        raise ValueError(
            "Renderer ini hanya menerima TPL-01."
        )

    required_keys = {
        "canonical_invoice_id",
        "document_id",
        "template_id",
        "language",
        "currency",
        "title",
        "labels",
        "metadata",
        "vendor",
        "buyer",
        "items",
        "financials",
        "footer",
        "synthetic_notice",
    }

    missing_keys = sorted(
        required_keys - set(render_payload)
    )

    if missing_keys:
        raise KeyError(
            "Payload TPL-01 belum lengkap: "
            f"{missing_keys}"
        )

    item_count = len(
        render_payload["items"]
    )

    if not (
        TPL01_MIN_ITEMS_PER_PAGE
        <= item_count
        <= TPL01_MAX_ITEMS_PER_PAGE
    ):
        raise ValueError(
            "TPL-01 hanya mendukung 2–8 item."
        )

    language = render_payload["language"]

    if (
        language == "id"
        and render_payload[
            "labels"
        ]["quantity"] != "Kuantitas"
    ):
        raise RuntimeError(
            "Label quantity bahasa Indonesia "
            "wajib 'Kuantitas'."
        )

    template_specification = (
        template_registry[
            TPL01_TEMPLATE_ID
        ]
    )

    expected_contract = {
        "layout_family": "classic_corporate",
        "page_size": "A4",
        "header_layout": "full_width_band",
        "party_layout": "two_columns",
        "table_style": "full_grid",
        "totals_position": "bottom_right",
        "accent_position": "top",
        "density": "regular",
    }

    contract_mismatches = {
        key: {
            "expected": expected_value,
            "actual": (
                template_specification.get(key)
            ),
        }
        for key, expected_value
        in expected_contract.items()
        if (
            template_specification.get(key)
            != expected_value
        )
    }

    if contract_mismatches:
        raise RuntimeError(
            "Kontrak TPL-01 tidak sesuai: "
            f"{contract_mismatches}"
        )

    page_width, page_height = (
        get_page_dimensions(
            template_specification[
                "page_size"
            ]
        )
    )

    primary_color = hex_to_rgb(
        template_specification[
            "primary_color"
        ]
    )

    accent_color = hex_to_rgb(
        template_specification[
            "accent_color"
        ]
    )

    dark_text = (0.12, 0.15, 0.19)
    muted_text = (0.38, 0.43, 0.48)
    border_color = (0.78, 0.82, 0.86)
    light_fill = (0.955, 0.970, 0.985)
    white = (1.0, 1.0, 1.0)

    output_pdf_path = Path(
        output_pdf_path
    )

    output_pdf_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_pdf_path = (
        output_pdf_path.with_name(
            f".{output_pdf_path.stem}"
            ".tmp.pdf"
        )
    )

    temporary_pdf_path.unlink(
        missing_ok=True
    )

    annotations: list[
        FieldAnnotation
    ] = []

    field_font_audit: list[
        dict[str, Any]
    ] = []

    document = pymupdf.open()

    def add_field(
        page: pymupdf.Page,
        rectangle: pymupdf.Rect,
        text: str,
        field_name: str,
        font_name: str = "helv",
        font_size: float = 7.0,
        minimum_font_size: float = 6.5,
        font_color: tuple[
            float,
            float,
            float,
        ] = dark_text,
        alignment: str = "left",
    ) -> None:
        annotation, used_font_size = (
            insert_annotated_text(
                page=page,
                rectangle=rectangle,
                text=str(text),
                field_name=field_name,
                font_name=font_name,
                font_size=font_size,
                minimum_font_size=(
                    minimum_font_size
                ),
                font_color=font_color,
                alignment=alignment,
            )
        )

        annotations.append(annotation)

        field_font_audit.append(
            {
                "field_name": field_name,
                "requested_font_size": font_size,
                "used_font_size": used_font_size,
                "adjusted": (
                    used_font_size < font_size
                ),
            }
        )

    try:
        page = document.new_page(
            width=page_width,
            height=page_height,
        )

        margin_x0 = 42.0
        margin_x1 = page_width - 42.0

        # --------------------------------------------------------
        # Full-width top band
        # --------------------------------------------------------

        page.draw_rect(
            pymupdf.Rect(
                0,
                0,
                page_width,
                12,
            ),
            color=accent_color,
            fill=accent_color,
            width=0,
            overlay=True,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                34,
                330,
                52,
            ),
            text=(
                "SYNTHETIC COMMERCE DOCUMENT"
            ),
            font_name="hebo",
            font_size=8.0,
            minimum_font_size=6.5,
            font_color=primary_color,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                58,
                330,
                74,
            ),
            text=(
                render_payload[
                    "footer"
                ]["document_reference"]
            ),
            font_name="helv",
            font_size=6.8,
            minimum_font_size=6.0,
            font_color=muted_text,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                350,
                29,
                margin_x1,
                70,
            ),
            text=render_payload["title"],
            font_name="hebo",
            font_size=23.0,
            minimum_font_size=17.0,
            font_color=primary_color,
            alignment="right",
        )

        page.draw_line(
            pymupdf.Point(
                margin_x0,
                92,
            ),
            pymupdf.Point(
                margin_x1,
                92,
            ),
            color=primary_color,
            width=1.0,
            overlay=True,
        )

        # --------------------------------------------------------
        # Vendor dan buyer
        # --------------------------------------------------------

        party_cards = [
            (
                "vendor",
                render_payload[
                    "labels"
                ]["vendor"],
                margin_x0,
                292.0,
            ),
            (
                "buyer",
                render_payload[
                    "labels"
                ]["buyer"],
                310.0,
                margin_x1,
            ),
        ]

        party_y0 = 112.0
        party_y1 = 250.0

        for (
            party_type,
            party_label,
            card_x0,
            card_x1,
        ) in party_cards:
            party = render_payload[
                party_type
            ]

            page.draw_rect(
                pymupdf.Rect(
                    card_x0,
                    party_y0,
                    card_x1,
                    party_y1,
                ),
                color=border_color,
                fill=light_fill,
                width=0.7,
                overlay=True,
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 10,
                    party_y0 + 10,
                    card_x1 - 10,
                    party_y0 + 24,
                ),
                text=party_label.upper(),
                font_name="hebo",
                font_size=6.7,
                minimum_font_size=6.0,
                font_color=accent_color,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 10,
                    party_y0 + 31,
                    card_x1 - 10,
                    party_y0 + 50,
                ),
                text=party["name"],
                field_name=(
                    f"{party_type}.name"
                ),
                font_name="hebo",
                font_size=8.0,
                minimum_font_size=6.5,
                font_color=dark_text,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 10,
                    party_y0 + 54,
                    card_x1 - 10,
                    party_y0 + 89,
                ),
                text="\n".join(
                    party[
                        "address_lines"
                    ]
                ),
                field_name=(
                    f"{party_type}"
                    ".address_lines"
                ),
                font_name="helv",
                font_size=6.8,
                minimum_font_size=6.5,
                font_color=dark_text,
            )

            contact_rows = [
                (
                    "email",
                    party["email"],
                    92.0,
                    105.0,
                ),
                (
                    "phone",
                    party["phone"],
                    107.0,
                    120.0,
                ),
                (
                    "tax_identifier",
                    party[
                        "tax_identifier_display"
                    ],
                    122.0,
                    136.0,
                ),
            ]

            for (
                field_name,
                field_value,
                field_y0,
                field_y1,
            ) in contact_rows:
                add_field(
                    page=page,
                    rectangle=pymupdf.Rect(
                        card_x0 + 10,
                        party_y0 + field_y0,
                        card_x1 - 10,
                        party_y0 + field_y1,
                    ),
                    text=field_value,
                    field_name=(
                        f"{party_type}."
                        f"{field_name}"
                    ),
                    font_name="helv",
                    font_size=6.6,
                    minimum_font_size=6.5,
                    font_color=dark_text,
                )

        # --------------------------------------------------------
        # Empat kartu metadata
        # --------------------------------------------------------

        currency_label = (
            "Mata Uang"
            if language == "id"
            else "Currency"
        )

        metadata_cards = [
            (
                "invoice_number",
                render_payload[
                    "metadata"
                ]["invoice_number"]["label"],
                render_payload[
                    "metadata"
                ]["invoice_number"][
                    "display_value"
                ],
                margin_x0,
                198.0,
            ),
            (
                "invoice_date",
                render_payload[
                    "metadata"
                ]["invoice_date"]["label"],
                render_payload[
                    "metadata"
                ]["invoice_date"][
                    "display_value"
                ],
                207.0,
                351.0,
            ),
            (
                "due_date",
                render_payload[
                    "metadata"
                ]["due_date"]["label"],
                render_payload[
                    "metadata"
                ]["due_date"][
                    "display_value"
                ],
                360.0,
                480.0,
            ),
            (
                "currency",
                currency_label,
                render_payload["currency"],
                489.0,
                margin_x1,
            ),
        ]

        metadata_y0 = 266.0
        metadata_y1 = 326.0

        for (
            field_name,
            label_text,
            display_value,
            card_x0,
            card_x1,
        ) in metadata_cards:
            page.draw_rect(
                pymupdf.Rect(
                    card_x0,
                    metadata_y0,
                    card_x1,
                    metadata_y1,
                ),
                color=border_color,
                fill=white,
                width=0.7,
                overlay=True,
            )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 8,
                    metadata_y0 + 10,
                    card_x1 - 8,
                    metadata_y0 + 25,
                ),
                text=label_text,
                font_name="hebo",
                font_size=6.3,
                minimum_font_size=5.8,
                font_color=muted_text,
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    card_x0 + 8,
                    metadata_y0 + 33,
                    card_x1 - 8,
                    metadata_y1 - 7,
                ),
                text=display_value,
                field_name=field_name,
                font_name="hebo",
                font_size=7.4,
                minimum_font_size=6.5,
                font_color=dark_text,
            )

        # --------------------------------------------------------
        # Tabel full grid
        # --------------------------------------------------------

        table_positions = [
            margin_x0,
            68.0,
            308.0,
            363.0,
            458.0,
            margin_x1,
        ]

        table_header_y0 = 342.0
        table_header_y1 = 371.0
        row_height = 25.0

        page.draw_rect(
            pymupdf.Rect(
                table_positions[0],
                table_header_y0,
                table_positions[-1],
                table_header_y1,
            ),
            color=primary_color,
            fill=primary_color,
            width=0,
            overlay=True,
        )

        table_headers = [
            "#",
            render_payload[
                "labels"
            ]["description"],
            render_payload[
                "labels"
            ]["quantity"],
            render_payload[
                "labels"
            ]["unit_price"],
            render_payload[
                "labels"
            ]["line_total"],
        ]

        for column_index, header_text in enumerate(
            table_headers
        ):
            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    (
                        table_positions[
                            column_index
                        ]
                        + 3
                    ),
                    table_header_y0 + 8,
                    (
                        table_positions[
                            column_index + 1
                        ]
                        - 3
                    ),
                    table_header_y1 - 2,
                ),
                text=header_text,
                font_name="hebo",
                font_size=6.5,
                minimum_font_size=5.8,
                font_color=white,
                alignment=(
                    "left"
                    if column_index == 1
                    else "center"
                ),
            )

        for item_index, item in enumerate(
            render_payload["items"]
        ):
            row_y0 = (
                table_header_y1
                + item_index * row_height
            )

            row_y1 = row_y0 + row_height

            if item_index % 2 == 0:
                page.draw_rect(
                    pymupdf.Rect(
                        table_positions[0],
                        row_y0,
                        table_positions[-1],
                        row_y1,
                    ),
                    color=None,
                    fill=light_fill,
                    width=0,
                    overlay=True,
                )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    table_positions[0] + 3,
                    row_y0 + 6,
                    table_positions[1] - 3,
                    row_y1 - 2,
                ),
                text=str(
                    item["item_number"]
                ),
                font_name="helv",
                font_size=6.7,
                minimum_font_size=6.0,
                font_color=muted_text,
                alignment="center",
            )

            item_fields = [
                ("description", 1, "left"),
                ("quantity", 2, "center"),
                ("unit_price", 3, "right"),
                ("line_total", 4, "right"),
            ]

            for (
                field_name,
                column_index,
                alignment,
            ) in item_fields:
                add_field(
                    page=page,
                    rectangle=pymupdf.Rect(
                        (
                            table_positions[
                                column_index
                            ]
                            + 4
                        ),
                        row_y0 + 6,
                        (
                            table_positions[
                                column_index + 1
                            ]
                            - 4
                        ),
                        row_y1 - 2,
                    ),
                    text=item[field_name],
                    field_name=(
                        f"items[{item_index}]"
                        f".{field_name}"
                    ),
                    font_name="helv",
                    font_size=6.8,
                    minimum_font_size=6.5,
                    font_color=dark_text,
                    alignment=alignment,
                )

        table_end_y = (
            table_header_y1
            + item_count * row_height
        )

        for column_x in table_positions:
            page.draw_line(
                pymupdf.Point(
                    column_x,
                    table_header_y0,
                ),
                pymupdf.Point(
                    column_x,
                    table_end_y,
                ),
                color=border_color,
                width=0.55,
                overlay=True,
            )

        for row_index in range(
            item_count + 1
        ):
            row_y = (
                table_header_y1
                + row_index * row_height
            )

            page.draw_line(
                pymupdf.Point(
                    table_positions[0],
                    row_y,
                ),
                pymupdf.Point(
                    table_positions[-1],
                    row_y,
                ),
                color=border_color,
                width=0.55,
                overlay=True,
            )

        page.draw_rect(
            pymupdf.Rect(
                table_positions[0],
                table_header_y0,
                table_positions[-1],
                table_end_y,
            ),
            color=primary_color,
            fill=None,
            width=0.8,
            overlay=True,
        )

        # --------------------------------------------------------
        # Panel total kanan bawah
        # --------------------------------------------------------

        totals_panel_height = 108.0

        totals_y0 = max(
            590.0,
            table_end_y + 24.0,
        )

        totals_y1 = (
            totals_y0
            + totals_panel_height
        )

        table_to_totals_gap = (
            totals_y0
            - table_end_y
        )

        footer_line_y = 770.0

        if table_to_totals_gap < 24.0:
            raise RuntimeError(
                "Tabel bertabrakan dengan panel total."
            )

        if (
            totals_y1
            >= footer_line_y - 24.0
        ):
            raise RuntimeError(
                "Panel total terlalu dekat dengan footer."
            )

        totals_x0 = 330.0
        totals_x1 = margin_x1

        page.draw_rect(
            pymupdf.Rect(
                totals_x0,
                totals_y0,
                totals_x1,
                totals_y1,
            ),
            color=border_color,
            fill=light_fill,
            width=0.7,
            overlay=True,
        )

        financial_rows = [
            ("subtotal", 10.0),
            ("tax", 32.0),
            ("discount", 54.0),
            ("total", 82.0),
        ]

        for field_name, row_offset in financial_rows:
            field_payload = (
                render_payload[
                    "financials"
                ][field_name]
            )

            is_total = (
                field_name == "total"
            )

            row_y0 = (
                totals_y0 + row_offset
            )

            if is_total:
                page.draw_line(
                    pymupdf.Point(
                        totals_x0 + 12,
                        totals_y0 + 76,
                    ),
                    pymupdf.Point(
                        totals_x1 - 12,
                        totals_y0 + 76,
                    ),
                    color=accent_color,
                    width=0.9,
                    overlay=True,
                )

            insert_static_text(
                page=page,
                rectangle=pymupdf.Rect(
                    totals_x0 + 12,
                    row_y0,
                    totals_x0 + 86,
                    row_y0 + 17,
                ),
                text=field_payload["label"],
                font_name=(
                    "hebo"
                    if is_total
                    else "helv"
                ),
                font_size=(
                    7.3
                    if is_total
                    else 6.8
                ),
                minimum_font_size=6.0,
                font_color=(
                    primary_color
                    if is_total
                    else muted_text
                ),
            )

            add_field(
                page=page,
                rectangle=pymupdf.Rect(
                    totals_x0 + 88,
                    row_y0,
                    totals_x1 - 12,
                    row_y0 + 17,
                ),
                text=field_payload[
                    "display_value"
                ],
                field_name=(
                    f"financials.{field_name}"
                ),
                font_name=(
                    "hebo"
                    if is_total
                    else "helv"
                ),
                font_size=(
                    8.0
                    if is_total
                    else 7.0
                ),
                minimum_font_size=6.5,
                font_color=(
                    accent_color
                    if is_total
                    else dark_text
                ),
                alignment="right",
            )

        # --------------------------------------------------------
        # Footer
        # --------------------------------------------------------

        page.draw_line(
            pymupdf.Point(
                margin_x0,
                footer_line_y,
            ),
            pymupdf.Point(
                margin_x1,
                footer_line_y,
            ),
            color=border_color,
            width=0.7,
            overlay=True,
        )

        insert_static_text(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                778,
                margin_x1,
                793,
            ),
            text=(
                render_payload[
                    "footer"
                ]["document_reference"]
            ),
            font_name="helv",
            font_size=6.5,
            minimum_font_size=6.0,
            font_color=muted_text,
            alignment="center",
        )

        notice_text = normalize_tpl01_notice(
            render_payload[
                "synthetic_notice"
            ]
        )

        add_field(
            page=page,
            rectangle=pymupdf.Rect(
                margin_x0,
                807,
                margin_x1,
                826,
            ),
            text=notice_text,
            field_name=(
                "document.synthetic_notice"
            ),
            font_name="hebo",
            font_size=6.7,
            minimum_font_size=6.5,
            font_color=accent_color,
            alignment="center",
        )

        # --------------------------------------------------------
        # Validasi sebelum penyimpanan
        # --------------------------------------------------------

        annotation_names = [
            annotation.field_name
            for annotation in annotations
        ]

        if (
            len(annotation_names)
            != len(set(annotation_names))
        ):
            raise RuntimeError(
                "Ditemukan anotasi duplikat."
            )

        expected_annotation_count = (
            19 + 4 * item_count
        )

        if (
            len(annotations)
            != expected_annotation_count
        ):
            raise RuntimeError(
                "Jumlah anotasi tidak sesuai. "
                f"Expected: "
                f"{expected_annotation_count}; "
                f"actual: {len(annotations)}."
            )

        minimum_used_font_size = min(
            row["used_font_size"]
            for row in field_font_audit
        )

        if minimum_used_font_size < 6.5:
            raise RuntimeError(
                "Ukuran font field di bawah 6.5 pt."
            )

        document.save(
            str(temporary_pdf_path),
            garbage=4,
            deflate=True,
        )

    except Exception:
        temporary_pdf_path.unlink(
            missing_ok=True
        )
        raise

    finally:
        document.close()

    if (
        not temporary_pdf_path.is_file()
        or temporary_pdf_path.stat().st_size
        == 0
    ):
        temporary_pdf_path.unlink(
            missing_ok=True
        )

        raise RuntimeError(
            "PDF sementara TPL-01 gagal dibuat."
        )

    os.replace(
        temporary_pdf_path,
        output_pdf_path,
    )

    rendering_metadata = {
        "renderer_version": (
            RENDER_ENGINE_VERSION
        ),
        "template_renderer_version": (
            TPL01_RENDERER_VERSION
        ),
        "template_id": TPL01_TEMPLATE_ID,
        "layout_family": (
            template_specification[
                "layout_family"
            ]
        ),
        "page_size": (
            template_specification[
                "page_size"
            ]
        ),
        "page_width_points": round(
            page_width,
            3,
        ),
        "page_height_points": round(
            page_height,
            3,
        ),
        "page_count": 1,
        "annotation_type": "field_region",
        "annotation_count": len(
            annotations
        ),
        "minimum_field_font_size": min(
            row["used_font_size"]
            for row in field_font_audit
        ),
        "font_adjustment_count": int(
            sum(
                bool(row["adjusted"])
                for row in field_font_audit
            )
        ),
        "layout_metrics": {
            "item_count": item_count,
            "table_end_y_points": round(
                table_end_y,
                3,
            ),
            "totals_panel_y0_points": round(
                totals_y0,
                3,
            ),
            "totals_panel_y1_points": round(
                totals_y1,
                3,
            ),
            "table_to_totals_gap_points": round(
                table_to_totals_gap,
                3,
            ),
            "footer_line_y_points": footer_line_y,
        },
    }

    return (
        annotations,
        rendering_metadata,
    )


print(
    "✅ Cell 70A siap — renderer TPL-01 berhasil "
    "didefinisikan. Belum ada artifact yang dibuat."
)

In [ ]:
# ================================================================
# CELL 70B — FINAL
# Render prototype, preview, dan ground truth TPL-01
# ================================================================

from PIL import Image as PILImage

import pandas as pd


# ================================================================
# Validasi dependency Cell 70A
# ================================================================

REQUIRED_CELL_70A_OBJECTS = [
    "TPL01_TEMPLATE_ID",
    "TPL01_RENDERER_VERSION",
    "TPL01_PROTOTYPE_ROOT",
    "render_template_01",
    "load_tpl01_canonical_payload",
    "normalize_tpl01_notice",
]

missing_cell_70a_objects = [
    name
    for name in REQUIRED_CELL_70A_OBJECTS
    if name not in globals()
]

if missing_cell_70a_objects:
    raise RuntimeError(
        "Dependency Cell 70A belum tersedia: "
        f"{missing_cell_70a_objects}. "
        "Jalankan Cell 70A terlebih dahulu."
    )


# ================================================================
# Pilih prototype pertama TPL-01
# ================================================================

tpl01_candidate_ids = sorted(
    canonical_id
    for canonical_id, payload
    in invoice_render_payloads.items()
    if (
        isinstance(payload, dict)
        and payload.get("template_id")
        == TPL01_TEMPLATE_ID
    )
)

if len(tpl01_candidate_ids) != 20:
    raise RuntimeError(
        "TPL-01 harus memiliki tepat 20 payload. "
        f"Actual: {len(tpl01_candidate_ids)}."
    )

prototype_01_canonical_id = (
    tpl01_candidate_ids[0]
)

prototype_01_render_payload = (
    invoice_render_payloads[
        prototype_01_canonical_id
    ]
)

prototype_01_document_id = (
    prototype_01_render_payload[
        "document_id"
    ]
)

prototype_01_canonical_payload = (
    load_tpl01_canonical_payload(
        prototype_01_canonical_id
    )
)


# ================================================================
# Path artifact
# ================================================================

prototype_01_pdf_path = (
    TPL01_PROTOTYPE_ROOT
    / (
        f"{prototype_01_document_id}"
        "_TPL-01_prototype.pdf"
    )
)

prototype_01_png_path = (
    TPL01_PROTOTYPE_ROOT
    / (
        f"{prototype_01_document_id}"
        "_TPL-01_preview.png"
    )
)

prototype_01_ground_truth_path = (
    TPL01_PROTOTYPE_ROOT
    / (
        f"{prototype_01_document_id}"
        "_TPL-01_ground_truth.json"
    )
)


# ================================================================
# Render prototype
# ================================================================

(
    prototype_01_annotations,
    prototype_01_rendering_metadata,
) = render_template_01(
    render_payload=(
        prototype_01_render_payload
    ),
    output_pdf_path=(
        prototype_01_pdf_path
    ),
)


# ================================================================
# Simpan ground truth
# ================================================================

prototype_01_ground_truth = {
    "schema_version": "1.0.0",
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "document": {
        "document_id": (
            prototype_01_document_id
        ),
        "canonical_invoice_id": (
            prototype_01_canonical_id
        ),
        "split": (
            prototype_01_render_payload[
                "split"
            ]
        ),
        "template_id": (
            TPL01_TEMPLATE_ID
        ),
        "language": (
            prototype_01_render_payload[
                "language"
            ]
        ),
        "currency": (
            prototype_01_render_payload[
                "currency"
            ]
        ),
    },
    "canonical": (
        prototype_01_canonical_payload
    ),
    "rendering": (
        prototype_01_rendering_metadata
    ),
    "annotations": [
        annotation_to_dict(
            annotation
        )
        for annotation
        in prototype_01_annotations
    ],
}

write_json_atomically(
    prototype_01_ground_truth_path,
    prototype_01_ground_truth,
)


# ================================================================
# Buat preview secara atomik
# ================================================================

temporary_png_path = (
    prototype_01_png_path.with_name(
        f".{prototype_01_png_path.stem}"
        ".tmp.png"
    )
)

temporary_png_path.unlink(
    missing_ok=True
)

try:
    with pymupdf.open(
        str(prototype_01_pdf_path)
    ) as prototype_document:
        if (
            prototype_document.page_count
            != 1
        ):
            raise RuntimeError(
                "Prototype TPL-01 harus "
                "memiliki tepat satu halaman."
            )

        prototype_page = (
            prototype_document[0]
        )

        extracted_text_01 = (
            prototype_page.get_text(
                "text"
            )
        )

        prototype_pixmap_01 = (
            prototype_page.get_pixmap(
                matrix=pymupdf.Matrix(
                    150 / 72,
                    150 / 72,
                ),
                alpha=False,
            )
        )

        prototype_pixmap_01.save(
            str(temporary_png_path)
        )

    if (
        not temporary_png_path.is_file()
        or temporary_png_path.stat().st_size
        == 0
    ):
        raise RuntimeError(
            "Preview sementara TPL-01 "
            "gagal dibuat."
        )

    os.replace(
        temporary_png_path,
        prototype_01_png_path,
    )

except Exception:
    temporary_png_path.unlink(
        missing_ok=True
    )
    raise


# ================================================================
# Pemeriksaan teks penting
# ================================================================

required_text_fragments_01 = [
    (
        prototype_01_render_payload[
            "metadata"
        ]["invoice_number"][
            "display_value"
        ]
    ),
    (
        prototype_01_render_payload[
            "vendor"
        ]["name"]
    ),
    (
        prototype_01_render_payload[
            "buyer"
        ]["name"]
    ),
    (
        prototype_01_render_payload[
            "labels"
        ]["quantity"]
    ),
    (
        prototype_01_render_payload[
            "financials"
        ]["total"][
            "display_value"
        ]
    ),
    normalize_tpl01_notice(
        prototype_01_render_payload[
            "synthetic_notice"
        ]
    ),
]

missing_text_fragments_01 = [
    text_fragment
    for text_fragment
    in required_text_fragments_01
    if (
        text_fragment
        not in extracted_text_01
    )
]

if missing_text_fragments_01:
    raise RuntimeError(
        "Teks penting TPL-01 hilang dari PDF: "
        f"{missing_text_fragments_01}"
    )


# ================================================================
# Pemeriksaan anotasi wajib
# ================================================================

annotation_names_01 = {
    annotation.field_name
    for annotation
    in prototype_01_annotations
}

required_annotation_names_01 = {
    "invoice_number",
    "invoice_date",
    "due_date",
    "currency",
    "vendor.name",
    "vendor.address_lines",
    "vendor.email",
    "vendor.phone",
    "vendor.tax_identifier",
    "buyer.name",
    "buyer.address_lines",
    "buyer.email",
    "buyer.phone",
    "buyer.tax_identifier",
    "financials.subtotal",
    "financials.tax",
    "financials.discount",
    "financials.total",
    "document.synthetic_notice",
}

missing_annotations_01 = sorted(
    required_annotation_names_01
    - annotation_names_01
)

if missing_annotations_01:
    raise RuntimeError(
        "Anotasi wajib TPL-01 tidak tersedia: "
        f"{missing_annotations_01}"
    )


# ================================================================
# Ringkasan teknis awal
# ================================================================

layout_metrics_01 = (
    prototype_01_rendering_metadata[
        "layout_metrics"
    ]
)

prototype_01_summary = pd.DataFrame(
    [
        {
            "control": "pdf_created",
            "actual": (
                prototype_01_pdf_path.is_file()
            ),
            "status": "VALID",
        },
        {
            "control": "ground_truth_created",
            "actual": (
                prototype_01_ground_truth_path
                .is_file()
            ),
            "status": "VALID",
        },
        {
            "control": "annotation_count",
            "actual": len(
                prototype_01_annotations
            ),
            "status": "VALID",
        },
        {
            "control": "item_count",
            "actual": (
                layout_metrics_01[
                    "item_count"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "quantity_header",
            "actual": (
                prototype_01_render_payload[
                    "labels"
                ]["quantity"]
            ),
            "status": "VALID",
        },
        {
            "control": "minimum_field_font",
            "actual": (
                prototype_01_rendering_metadata[
                    "minimum_field_font_size"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "font_adjustments",
            "actual": (
                prototype_01_rendering_metadata[
                    "font_adjustment_count"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "table_to_totals_gap",
            "actual": (
                layout_metrics_01[
                    "table_to_totals_gap_points"
                ]
            ),
            "status": "VALID",
        },
        {
            "control": "missing_required_text",
            "actual": len(
                missing_text_fragments_01
            ),
            "status": "VALID",
        },
    ]
)

display(
    prototype_01_summary
)

with PILImage.open(
    prototype_01_png_path
) as prototype_image:
    display(
        prototype_image.copy()
    )


# ================================================================
# Ringkasan output
# ================================================================

print(
    f"Canonical ID       : "
    f"{prototype_01_canonical_id}"
)

print(
    f"Document ID        : "
    f"{prototype_01_document_id}"
)

print(
    f"Prototype PDF      : "
    f"{prototype_01_pdf_path}"
)

print(
    f"Prototype preview  : "
    f"{prototype_01_png_path}"
)

print(
    f"Ground truth       : "
    f"{prototype_01_ground_truth_path}"
)

print(
    f"Annotations        : "
    f"{len(prototype_01_annotations)}"
)

print(
    f"Quantity header    : "
    f"{prototype_01_render_payload['labels']['quantity']}"
)

print(
    "Minimum field font : "
    f"{prototype_01_rendering_metadata['minimum_field_font_size']}"
)

print(
    "Adjusted fields    : "
    f"{prototype_01_rendering_metadata['font_adjustment_count']}"
)

print(
    "Table-total gap    : "
    f"{layout_metrics_01['table_to_totals_gap_points']} pt"
)

print(
    f"Renderer version   : "
    f"{TPL01_RENDERER_VERSION}"
)

print()

print(
    "✅ Prototype TPL-01 berhasil dibuat ulang "
    "dan lolos pemeriksaan teknis awal."
)

In [ ]:
# ================================================================
# CELL 71 — FINAL
# Audit QA teknis prototype TPL-01
# Menggunakan helper QA generik dari Cell 68
# ================================================================

from pathlib import Path

import json
import math

import numpy as np
import pandas as pd
import pymupdf
from PIL import Image as PILImage


# ================================================================
# Validasi runtime dan helper
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "TPL01_PROTOTYPE_ROOT",
    "TPL01_TEMPLATE_ID",
    "prototype_01_canonical_id",
    "prototype_01_document_id",
    "prototype_01_pdf_path",
    "prototype_01_png_path",
    "prototype_01_ground_truth_path",
    "prototype_01_render_payload",
    "prototype_01_rendering_metadata",
    "prototype_01_annotations",
    "write_json_atomically",
    "qa_sha256",
    "qa_normalize_text",
    "qa_normalize_notice",
    "qa_annotation_name",
    "qa_annotation_text",
    "qa_annotation_bbox",
    "qa_rectangle_area",
    "qa_intersection_area",
]

missing_runtime_objects = [
    name
    for name in REQUIRED_RUNTIME_OBJECTS
    if name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 71 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali Cell 68, Cell 70A, "
        "dan Cell 70B."
    )


# ================================================================
# Konfigurasi audit
# ================================================================

QA_SCHEMA_VERSION = "1.0.0"
QA_AUDITOR_VERSION = "1.0.0"

MINIMUM_FIELD_FONT_SIZE = 6.5
MAXIMUM_FONT_ADJUSTMENT_RATIO = 0.20
MINIMUM_RASTER_CONTENT_RATIO = 0.01
MAXIMUM_RASTER_CONTENT_RATIO = 0.40
MINIMUM_RASTER_CONTRAST = 10.0
SEVERE_OVERLAP_THRESHOLD = 0.50


# ================================================================
# Path artifact
# ================================================================

prototype_01_pdf_path = Path(
    prototype_01_pdf_path
)

prototype_01_png_path = Path(
    prototype_01_png_path
)

prototype_01_ground_truth_path = Path(
    prototype_01_ground_truth_path
)

prototype_01_qa_report_path = (
    Path(TPL01_PROTOTYPE_ROOT)
    / (
        f"{prototype_01_document_id}"
        "_TPL-01_qa_report.json"
    )
)

required_files = [
    prototype_01_pdf_path,
    prototype_01_png_path,
    prototype_01_ground_truth_path,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Artifact TPL-01 belum lengkap: "
        f"{missing_files}"
    )


# ================================================================
# Muat ground truth
# ================================================================

with prototype_01_ground_truth_path.open(
    "r",
    encoding="utf-8",
) as ground_truth_file:
    ground_truth = json.load(
        ground_truth_file
    )

ground_truth_document = ground_truth.get(
    "document",
    {},
)

if (
    ground_truth_document.get("document_id")
    != prototype_01_document_id
):
    raise RuntimeError(
        "Document ID ground truth tidak sesuai."
    )

if (
    ground_truth_document.get("template_id")
    != TPL01_TEMPLATE_ID
):
    raise RuntimeError(
        "Ground truth bukan milik TPL-01."
    )

annotation_records = ground_truth.get(
    "annotations",
    [],
)

if (
    not isinstance(annotation_records, list)
    or not annotation_records
):
    raise RuntimeError(
        "Ground truth tidak memiliki anotasi valid."
    )


# ================================================================
# Expected values
# ================================================================

render_payload = prototype_01_render_payload

expected_values = {
    "invoice_number": str(
        render_payload["metadata"]
        ["invoice_number"]["display_value"]
    ),
    "invoice_date": str(
        render_payload["metadata"]
        ["invoice_date"]["display_value"]
    ),
    "due_date": str(
        render_payload["metadata"]
        ["due_date"]["display_value"]
    ),
    "currency": str(
        render_payload["currency"]
    ),
}

for party_type in ("vendor", "buyer"):
    party = render_payload[party_type]

    expected_values.update(
        {
            f"{party_type}.name": str(
                party["name"]
            ),
            f"{party_type}.address_lines": "\n".join(
                party["address_lines"]
            ),
            f"{party_type}.email": str(
                party["email"]
            ),
            f"{party_type}.phone": str(
                party["phone"]
            ),
            f"{party_type}.tax_identifier": str(
                party["tax_identifier_display"]
            ),
        }
    )

for item_index, item in enumerate(
    render_payload["items"]
):
    for field_name in (
        "description",
        "quantity",
        "unit_price",
        "line_total",
    ):
        expected_values[
            f"items[{item_index}].{field_name}"
        ] = str(item[field_name])

for field_name in (
    "subtotal",
    "tax",
    "discount",
    "total",
):
    expected_values[
        f"financials.{field_name}"
    ] = str(
        render_payload["financials"]
        [field_name]["display_value"]
    )

expected_values[
    "document.synthetic_notice"
] = qa_normalize_notice(
    render_payload["synthetic_notice"]
)

expected_annotation_count = len(
    expected_values
)


# ================================================================
# Parse annotations
# ================================================================

parsed_annotations = []
annotation_parse_failures = []

for annotation_index, annotation in enumerate(
    annotation_records
):
    try:
        if not isinstance(annotation, dict):
            raise TypeError(
                "Anotasi wajib berupa dictionary."
            )

        parsed_annotations.append(
            {
                "index": annotation_index,
                "field_name": qa_annotation_name(
                    annotation
                ),
                "text": qa_annotation_text(
                    annotation
                ),
                "bbox": qa_annotation_bbox(
                    annotation
                ),
            }
        )

    except (KeyError, TypeError, ValueError) as error:
        annotation_parse_failures.append(
            {
                "index": annotation_index,
                "error": str(error),
            }
        )

if annotation_parse_failures:
    raise RuntimeError(
        "Schema anotasi tidak dapat diproses: "
        f"{annotation_parse_failures[:3]}"
    )

annotation_names = [
    annotation["field_name"]
    for annotation in parsed_annotations
]

annotation_by_name = {
    annotation["field_name"]: annotation
    for annotation in parsed_annotations
}

duplicate_annotation_count = (
    len(annotation_names)
    - len(set(annotation_names))
)

missing_required_annotations = sorted(
    set(expected_values)
    - set(annotation_names)
)


# ================================================================
# Audit PDF
# ================================================================

with pymupdf.open(
    str(prototype_01_pdf_path)
) as pdf_document:
    pdf_page_count = pdf_document.page_count

    if pdf_page_count < 1:
        raise RuntimeError(
            "PDF TPL-01 tidak memiliki halaman."
        )

    first_page = pdf_document[0]
    page_rectangle = first_page.rect
    extracted_text = first_page.get_text("text")
    extracted_words = first_page.get_text("words")

page_width = float(page_rectangle.width)
page_height = float(page_rectangle.height)

word_boxes = [
    (
        float(word[0]),
        float(word[1]),
        float(word[2]),
        float(word[3]),
    )
    for word in extracted_words
    if (
        len(word) >= 5
        and str(word[4]).strip()
    )
]


# ================================================================
# Audit bounding box
# ================================================================

invalid_bounding_boxes = []
annotations_without_words = []

for annotation in parsed_annotations:
    field_name = annotation["field_name"]
    x0, y0, x1, y1 = annotation["bbox"]

    bbox_is_valid = (
        all(
            math.isfinite(value)
            for value in (x0, y0, x1, y1)
        )
        and x0 >= 0.0
        and y0 >= 0.0
        and x1 <= page_width
        and y1 <= page_height
        and x1 > x0
        and y1 > y0
    )

    if not bbox_is_valid:
        invalid_bounding_boxes.append(
            field_name
        )
        continue

    contains_word = any(
        qa_intersection_area(
            annotation["bbox"],
            word_box,
        ) > 0.0
        for word_box in word_boxes
    )

    if not contains_word:
        annotations_without_words.append(
            field_name
        )


# ================================================================
# Audit nilai teks
# ================================================================

normalization_mismatches = []

for field_name, expected_value in expected_values.items():
    annotation = annotation_by_name.get(
        field_name
    )

    if annotation is None:
        continue

    actual_value = annotation["text"]

    if (
        qa_normalize_text(actual_value)
        != qa_normalize_text(expected_value)
    ):
        normalization_mismatches.append(
            {
                "field_name": field_name,
                "expected": expected_value,
                "actual": actual_value,
            }
        )

normalized_extracted_text = qa_normalize_text(
    extracted_text
)

missing_extracted_values = []

for annotation in parsed_annotations:
    normalized_value = qa_normalize_text(
        annotation["text"]
    )

    if (
        normalized_value
        and normalized_value
        not in normalized_extracted_text
    ):
        missing_extracted_values.append(
            annotation["field_name"]
        )


# ================================================================
# Audit overlap
# ================================================================

severe_annotation_overlaps = []

for first_index in range(
    len(parsed_annotations)
):
    first_annotation = parsed_annotations[
        first_index
    ]

    first_area = qa_rectangle_area(
        first_annotation["bbox"]
    )

    if first_area <= 0.0:
        continue

    for second_index in range(
        first_index + 1,
        len(parsed_annotations),
    ):
        second_annotation = parsed_annotations[
            second_index
        ]

        second_area = qa_rectangle_area(
            second_annotation["bbox"]
        )

        if second_area <= 0.0:
            continue

        overlap_area = qa_intersection_area(
            first_annotation["bbox"],
            second_annotation["bbox"],
        )

        overlap_ratio = (
            overlap_area
            / min(first_area, second_area)
        )

        if (
            overlap_ratio
            >= SEVERE_OVERLAP_THRESHOLD
        ):
            severe_annotation_overlaps.append(
                {
                    "first": first_annotation[
                        "field_name"
                    ],
                    "second": second_annotation[
                        "field_name"
                    ],
                    "overlap_ratio": round(
                        overlap_ratio,
                        6,
                    ),
                }
            )


# ================================================================
# Audit font dan raster
# ================================================================

minimum_field_font_size = float(
    prototype_01_rendering_metadata[
        "minimum_field_font_size"
    ]
)

font_adjustment_count = int(
    prototype_01_rendering_metadata[
        "font_adjustment_count"
    ]
)

font_adjustment_ratio = (
    font_adjustment_count
    / max(1, len(parsed_annotations))
)

with PILImage.open(
    prototype_01_png_path
) as preview_image:
    grayscale_array = np.asarray(
        preview_image.convert("L"),
        dtype=np.float32,
    ).copy()

if grayscale_array.size == 0:
    raise RuntimeError(
        "Preview PNG tidak memiliki piksel."
    )

raster_content_ratio = float(
    np.mean(grayscale_array < 245.0)
)

raster_contrast = float(
    np.std(grayscale_array)
)


# ================================================================
# Susun checks
# ================================================================

checks = [
    {
        "control": "pdf_page_count",
        "expected": 1,
        "actual": pdf_page_count,
        "valid": pdf_page_count == 1,
    },
    {
        "control": "annotation_count",
        "expected": expected_annotation_count,
        "actual": len(parsed_annotations),
        "valid": (
            len(parsed_annotations)
            == expected_annotation_count
        ),
    },
    {
        "control": "duplicate_annotations",
        "expected": 0,
        "actual": duplicate_annotation_count,
        "valid": duplicate_annotation_count == 0,
    },
    {
        "control": "missing_required_annotations",
        "expected": 0,
        "actual": len(
            missing_required_annotations
        ),
        "valid": not missing_required_annotations,
    },
    {
        "control": "invalid_bounding_boxes",
        "expected": 0,
        "actual": len(
            invalid_bounding_boxes
        ),
        "valid": not invalid_bounding_boxes,
    },
    {
        "control": "normalization_mismatches",
        "expected": 0,
        "actual": len(
            normalization_mismatches
        ),
        "valid": not normalization_mismatches,
    },
    {
        "control": "annotations_without_words",
        "expected": 0,
        "actual": len(
            annotations_without_words
        ),
        "valid": not annotations_without_words,
    },
    {
        "control": "severe_annotation_overlaps",
        "expected": 0,
        "actual": len(
            severe_annotation_overlaps
        ),
        "valid": not severe_annotation_overlaps,
    },
    {
        "control": "missing_extracted_values",
        "expected": 0,
        "actual": len(
            missing_extracted_values
        ),
        "valid": not missing_extracted_values,
    },
    {
        "control": "minimum_field_font_size",
        "expected": ">=6.5",
        "actual": round(
            minimum_field_font_size,
            4,
        ),
        "valid": (
            minimum_field_font_size
            >= MINIMUM_FIELD_FONT_SIZE
        ),
    },
    {
        "control": "font_adjustment_ratio",
        "expected": "<=0.20",
        "actual": round(
            font_adjustment_ratio,
            4,
        ),
        "valid": (
            font_adjustment_ratio
            <= MAXIMUM_FONT_ADJUSTMENT_RATIO
        ),
    },
    {
        "control": "raster_content_ratio",
        "expected": "0.01–0.40",
        "actual": round(
            raster_content_ratio,
            4,
        ),
        "valid": (
            MINIMUM_RASTER_CONTENT_RATIO
            <= raster_content_ratio
            <= MAXIMUM_RASTER_CONTENT_RATIO
        ),
    },
    {
        "control": "raster_contrast",
        "expected": ">=10.0",
        "actual": round(
            raster_contrast,
            4,
        ),
        "valid": (
            raster_contrast
            >= MINIMUM_RASTER_CONTRAST
        ),
    },
]

for check in checks:
    check["status"] = (
        "VALID"
        if check["valid"]
        else "INVALID"
    )

failed_checks = [
    check
    for check in checks
    if not check["valid"]
]

qa_status = (
    "PASSED"
    if not failed_checks
    else "FAILED"
)


# ================================================================
# Simpan QA report
# ================================================================

qa_report = {
    "schema_version": QA_SCHEMA_VERSION,
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "document_id": prototype_01_document_id,
    "canonical_invoice_id": (
        prototype_01_canonical_id
    ),
    "template_id": TPL01_TEMPLATE_ID,
    "status": qa_status,
    "manual_visual_review": "PENDING",
    "metrics": {
        "auditor_version": QA_AUDITOR_VERSION,
        "pdf_page_count": pdf_page_count,
        "annotation_count": len(
            parsed_annotations
        ),
        "expected_annotation_count": (
            expected_annotation_count
        ),
        "duplicate_annotation_count": (
            duplicate_annotation_count
        ),
        "missing_required_annotation_count": len(
            missing_required_annotations
        ),
        "invalid_bounding_box_count": len(
            invalid_bounding_boxes
        ),
        "normalization_mismatch_count": len(
            normalization_mismatches
        ),
        "annotations_without_words_count": len(
            annotations_without_words
        ),
        "severe_annotation_overlap_count": len(
            severe_annotation_overlaps
        ),
        "missing_extracted_value_count": len(
            missing_extracted_values
        ),
        "minimum_field_font_size": round(
            minimum_field_font_size,
            6,
        ),
        "font_adjustment_count": (
            font_adjustment_count
        ),
        "font_adjustment_ratio": round(
            font_adjustment_ratio,
            6,
        ),
        "raster_content_ratio": round(
            raster_content_ratio,
            6,
        ),
        "raster_contrast": round(
            raster_contrast,
            6,
        ),
    },
    "failures": {
        "failed_checks": [
            check["control"]
            for check in failed_checks
        ],
        "missing_required_annotations": (
            missing_required_annotations
        ),
        "invalid_bounding_boxes": (
            invalid_bounding_boxes
        ),
        "normalization_mismatches": (
            normalization_mismatches
        ),
        "annotations_without_words": (
            annotations_without_words
        ),
        "severe_annotation_overlaps": (
            severe_annotation_overlaps
        ),
        "missing_extracted_values": (
            missing_extracted_values
        ),
    },
    "checksums_sha256": {
        "prototype_pdf": qa_sha256(
            prototype_01_pdf_path
        ),
        "preview": qa_sha256(
            prototype_01_png_path
        ),
        "ground_truth": qa_sha256(
            prototype_01_ground_truth_path
        ),
    },
}

write_json_atomically(
    prototype_01_qa_report_path,
    qa_report,
)


# ================================================================
# Verifikasi dan output
# ================================================================

with prototype_01_qa_report_path.open(
    "r",
    encoding="utf-8",
) as qa_report_file:
    verified_qa_report = json.load(
        qa_report_file
    )

if verified_qa_report != qa_report:
    raise RuntimeError(
        "QA report berubah setelah round-trip."
    )

qa_summary = pd.DataFrame(
    [
        {
            "control": check["control"],
            "expected": check["expected"],
            "actual": check["actual"],
            "status": check["status"],
        }
        for check in checks
    ]
)

display(qa_summary)

print(
    f"QA report          : "
    f"{prototype_01_qa_report_path}"
)
print(
    f"QA status          : {qa_status}"
)
print(
    "Minimum field font : "
    f"{minimum_field_font_size:.2f}"
)
print(
    "Font adjustments   : "
    f"{font_adjustment_count}"
)
print(
    "Ink pixel ratio    : "
    f"{raster_content_ratio:.6f}"
)
print(
    "Raster contrast    : "
    f"{raster_contrast:.6f}"
)
print(
    "Manual review      : "
    f"{qa_report['manual_visual_review']}"
)
print()

if qa_status != "PASSED":
    print("Detail kegagalan:")
    print(
        json.dumps(
            qa_report["failures"],
            ensure_ascii=False,
            indent=2,
        )
    )

    raise RuntimeError(
        "Prototype TPL-01 gagal audit QA teknis."
    )

print(
    "✅ Prototype TPL-01 lulus audit teknis. "
    "Review visual manual masih perlu dicatat "
    "ke QA report."
)

In [ ]:
# ================================================================
# CELL 71A — FINAL
# Finalisasi review visual manual prototype TPL-01
# ================================================================

from pathlib import Path

import json


# ================================================================
# Validasi runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "prototype_01_qa_report_path",
    "prototype_01_png_path",
    "prototype_01_document_id",
    "write_json_atomically",
    "qa_sha256",
]

missing_runtime_objects = [
    name
    for name in REQUIRED_RUNTIME_OBJECTS
    if name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 71A belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali Cell 68 dan Cell 71."
    )


# ================================================================
# Muat artifact
# ================================================================

qa_report_path = Path(
    prototype_01_qa_report_path
)

preview_path = Path(
    prototype_01_png_path
)

if not qa_report_path.is_file():
    raise FileNotFoundError(
        "QA report TPL-01 tidak ditemukan: "
        f"{qa_report_path}"
    )

if not preview_path.is_file():
    raise FileNotFoundError(
        "Preview TPL-01 tidak ditemukan: "
        f"{preview_path}"
    )

with qa_report_path.open(
    "r",
    encoding="utf-8",
) as qa_report_file:
    qa_report = json.load(
        qa_report_file
    )


# ================================================================
# Validasi identitas dan audit teknis
# ================================================================

if qa_report.get("template_id") != "TPL-01":
    raise RuntimeError(
        "QA report bukan milik TPL-01."
    )

if (
    qa_report.get("document_id")
    != prototype_01_document_id
):
    raise RuntimeError(
        "Document ID QA report tidak sesuai. "
        f"Expected: {prototype_01_document_id!r}; "
        f"actual: {qa_report.get('document_id')!r}."
    )

if qa_report.get("status") != "PASSED":
    raise RuntimeError(
        "Review visual tidak boleh difinalisasi "
        "karena audit teknis belum PASSED. "
        f"Status: {qa_report.get('status')!r}."
    )

failure_details = qa_report.get(
    "failures",
    {},
)

if not isinstance(failure_details, dict):
    raise TypeError(
        "Field 'failures' wajib berupa dictionary."
    )

nonempty_failures = {
    failure_name: failure_value
    for failure_name, failure_value
    in failure_details.items()
    if bool(failure_value)
}

if nonempty_failures:
    raise RuntimeError(
        "QA report masih memiliki kegagalan teknis: "
        f"{nonempty_failures}"
    )

if "manual_visual_review" not in qa_report:
    raise KeyError(
        "Field 'manual_visual_review' tidak ditemukan."
    )

if qa_report["manual_visual_review"] not in {
    "PENDING",
    "PASSED",
}:
    raise RuntimeError(
        "Status manual visual review tidak dikenali: "
        f"{qa_report['manual_visual_review']!r}."
    )


# ================================================================
# Verifikasi checksum preview
# ================================================================

recorded_preview_checksum = (
    qa_report
    .get("checksums_sha256", {})
    .get("preview")
)

if not recorded_preview_checksum:
    raise KeyError(
        "Checksum preview tidak ditemukan pada "
        "qa_report['checksums_sha256']['preview']."
    )

actual_preview_checksum = qa_sha256(
    preview_path
)

if (
    actual_preview_checksum
    != recorded_preview_checksum
):
    raise RuntimeError(
        "Preview berubah setelah audit teknis. "
        "Jalankan ulang Cell 71 dan lakukan "
        "review visual kembali. "
        f"Expected SHA-256: "
        f"{recorded_preview_checksum}; "
        f"actual: {actual_preview_checksum}."
    )


# ================================================================
# Simpan status manual secara idempoten
# ================================================================

qa_report[
    "manual_visual_review"
] = "PASSED"

write_json_atomically(
    qa_report_path,
    qa_report,
)


# ================================================================
# Verifikasi persistensi
# ================================================================

with qa_report_path.open(
    "r",
    encoding="utf-8",
) as qa_report_file:
    persisted_qa_report = json.load(
        qa_report_file
    )

if (
    persisted_qa_report.get("template_id")
    != "TPL-01"
):
    raise RuntimeError(
        "Template ID berubah setelah finalisasi."
    )

if (
    persisted_qa_report.get("document_id")
    != prototype_01_document_id
):
    raise RuntimeError(
        "Document ID berubah setelah finalisasi."
    )

if (
    persisted_qa_report.get("status")
    != "PASSED"
):
    raise RuntimeError(
        "Status teknis berubah setelah finalisasi."
    )

if (
    persisted_qa_report.get(
        "manual_visual_review"
    )
    != "PASSED"
):
    raise RuntimeError(
        "Status review visual gagal disimpan."
    )

persisted_preview_checksum = (
    persisted_qa_report
    .get("checksums_sha256", {})
    .get("preview")
)

if (
    persisted_preview_checksum
    != actual_preview_checksum
):
    raise RuntimeError(
        "Checksum preview berubah "
        "pada QA report."
    )


# ================================================================
# Ringkasan
# ================================================================

print(
    f"QA report           : "
    f"{qa_report_path}"
)

print(
    "Technical status    : "
    f"{persisted_qa_report['status']}"
)

print(
    "Manual visual review: "
    f"{persisted_qa_report['manual_visual_review']}"
)

print(
    "Preview SHA-256     : "
    f"{persisted_preview_checksum}"
)

print()

print(
    "✅ TPL-01 FINAL PASSED — audit teknis dan "
    "review visual telah selesai."
)

In [ ]:
# ================================================================
# CELL 72 — FINAL
# Audit konsolidasi prototype TPL-01 sampai TPL-10
# Read-only: tidak membuat atau mengubah artifact
# ================================================================

from pathlib import Path
from typing import Any

import hashlib
import json

import pandas as pd
import pymupdf
from PIL import Image as PILImage
from IPython.display import display


# ================================================================
# Validasi runtime
# ================================================================

if "PROTOTYPE_ROOT" not in globals():
    raise RuntimeError(
        "PROTOTYPE_ROOT belum tersedia."
    )

prototype_root = Path(
    PROTOTYPE_ROOT
)

if not prototype_root.is_dir():
    raise FileNotFoundError(
        "Folder prototype tidak ditemukan: "
        f"{prototype_root}"
    )


# ================================================================
# Konfigurasi
# ================================================================

EXPECTED_TEMPLATE_IDS = [
    f"TPL-{template_number:02d}"
    for template_number in range(1, 11)
]

EXPECTED_TEMPLATE_COUNT = 10


# ================================================================
# Helper
# ================================================================

def consolidation_sha256(
    file_path: Path,
) -> str:
    digest = hashlib.sha256()

    with file_path.open("rb") as input_file:
        for chunk in iter(
            lambda: input_file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def nonempty_failure_details(
    failures: Any,
) -> dict[str, Any]:
    if not isinstance(failures, dict):
        return {
            "invalid_failures_schema": failures
        }

    return {
        key: value
        for key, value in failures.items()
        if bool(value)
    }


def resolve_single_artifact(
    template_directory: Path,
    pattern: str,
) -> tuple[Path | None, list[str]]:
    matches = sorted(
        template_directory.glob(pattern)
    )

    return (
        matches[0] if len(matches) == 1 else None,
        [
            str(path)
            for path in matches
        ],
    )


# ================================================================
# Audit setiap template
# ================================================================

summary_rows = []
failure_records = []

for template_id in EXPECTED_TEMPLATE_IDS:
    template_directory = (
        prototype_root
        / template_id
    )

    template_failures = []

    if not template_directory.is_dir():
        template_failures.append(
            "template_directory_missing"
        )

        summary_rows.append(
            {
                "template_id": template_id,
                "document_id": None,
                "pdf_pages": None,
                "annotations": None,
                "minimum_font": None,
                "technical_status": "MISSING",
                "manual_review": "MISSING",
                "checksums": "MISSING",
                "artifact_counts": "INVALID",
                "status": "INVALID",
            }
        )

        failure_records.append(
            {
                "template_id": template_id,
                "failures": template_failures,
            }
        )

        continue

    qa_report_path, qa_matches = (
        resolve_single_artifact(
            template_directory,
            "*_qa_report.json",
        )
    )

    pdf_path, pdf_matches = (
        resolve_single_artifact(
            template_directory,
            "*_prototype.pdf",
        )
    )

    preview_path, preview_matches = (
        resolve_single_artifact(
            template_directory,
            "*_preview.png",
        )
    )

    ground_truth_path, ground_truth_matches = (
        resolve_single_artifact(
            template_directory,
            "*_ground_truth.json",
        )
    )

    artifact_counts_valid = all(
        len(matches) == 1
        for matches in (
            qa_matches,
            pdf_matches,
            preview_matches,
            ground_truth_matches,
        )
    )

    if len(qa_matches) != 1:
        template_failures.append(
            {
                "qa_report_count": len(
                    qa_matches
                ),
                "matches": qa_matches,
            }
        )

    if len(pdf_matches) != 1:
        template_failures.append(
            {
                "prototype_pdf_count": len(
                    pdf_matches
                ),
                "matches": pdf_matches,
            }
        )

    if len(preview_matches) != 1:
        template_failures.append(
            {
                "preview_png_count": len(
                    preview_matches
                ),
                "matches": preview_matches,
            }
        )

    if len(ground_truth_matches) != 1:
        template_failures.append(
            {
                "ground_truth_count": len(
                    ground_truth_matches
                ),
                "matches": ground_truth_matches,
            }
        )

    if not artifact_counts_valid:
        summary_rows.append(
            {
                "template_id": template_id,
                "document_id": None,
                "pdf_pages": None,
                "annotations": None,
                "minimum_font": None,
                "technical_status": "UNKNOWN",
                "manual_review": "UNKNOWN",
                "checksums": "UNKNOWN",
                "artifact_counts": "INVALID",
                "status": "INVALID",
            }
        )

        failure_records.append(
            {
                "template_id": template_id,
                "failures": template_failures,
            }
        )

        continue

    required_paths = [
        qa_report_path,
        pdf_path,
        preview_path,
        ground_truth_path,
    ]

    for artifact_path in required_paths:
        if (
            artifact_path is None
            or not artifact_path.is_file()
            or artifact_path.stat().st_size == 0
        ):
            template_failures.append(
                {
                    "missing_or_empty_artifact": (
                        str(artifact_path)
                    )
                }
            )

    if template_failures:
        summary_rows.append(
            {
                "template_id": template_id,
                "document_id": None,
                "pdf_pages": None,
                "annotations": None,
                "minimum_font": None,
                "technical_status": "UNKNOWN",
                "manual_review": "UNKNOWN",
                "checksums": "UNKNOWN",
                "artifact_counts": "VALID",
                "status": "INVALID",
            }
        )

        failure_records.append(
            {
                "template_id": template_id,
                "failures": template_failures,
            }
        )

        continue

    with qa_report_path.open(
        "r",
        encoding="utf-8",
    ) as qa_file:
        qa_report = json.load(
            qa_file
        )

    with ground_truth_path.open(
        "r",
        encoding="utf-8",
    ) as ground_truth_file:
        ground_truth = json.load(
            ground_truth_file
        )

    document_id = qa_report.get(
        "document_id"
    )

    technical_status = qa_report.get(
        "status"
    )

    manual_review = qa_report.get(
        "manual_visual_review",
        qa_report.get(
            "manual_review"
        ),
    )

    if qa_report.get(
        "template_id"
    ) != template_id:
        template_failures.append(
            "qa_template_id_mismatch"
        )

    if technical_status != "PASSED":
        template_failures.append(
            {
                "technical_status": (
                    technical_status
                )
            }
        )

    if manual_review != "PASSED":
        template_failures.append(
            {
                "manual_visual_review": (
                    manual_review
                )
            }
        )

    remaining_failures = (
        nonempty_failure_details(
            qa_report.get(
                "failures",
                {},
            )
        )
    )

    if remaining_failures:
        template_failures.append(
            {
                "qa_failures": (
                    remaining_failures
                )
            }
        )

    ground_truth_document = (
        ground_truth.get(
            "document",
            {},
        )
    )

    if (
        ground_truth_document.get(
            "template_id"
        )
        != template_id
    ):
        template_failures.append(
            "ground_truth_template_id_mismatch"
        )

    if (
        ground_truth_document.get(
            "document_id"
        )
        != document_id
    ):
        template_failures.append(
            "ground_truth_document_id_mismatch"
        )

    annotations = ground_truth.get(
        "annotations",
        [],
    )

    if not isinstance(annotations, list):
        annotations = []

        template_failures.append(
            "invalid_annotation_schema"
        )

    annotation_count = len(
        annotations
    )

    metrics = qa_report.get(
        "metrics",
        {},
    )

    reported_annotation_count = (
        metrics.get(
            "annotation_count"
        )
    )

    expected_annotation_count = (
        metrics.get(
            "expected_annotation_count"
        )
    )

    if (
        reported_annotation_count
        != annotation_count
    ):
        template_failures.append(
            {
                "annotation_count_mismatch": {
                    "ground_truth": annotation_count,
                    "qa_report": (
                        reported_annotation_count
                    ),
                }
            }
        )

    if (
        expected_annotation_count
        != annotation_count
    ):
        template_failures.append(
            {
                "expected_annotation_count_mismatch": {
                    "ground_truth": annotation_count,
                    "expected": (
                        expected_annotation_count
                    ),
                }
            }
        )

    annotation_names = []

    for annotation in annotations:
        if not isinstance(annotation, dict):
            continue

        annotation_name = (
            annotation.get(
                "field_name",
                annotation.get(
                    "field_path",
                    annotation.get(
                        "name"
                    ),
                ),
            )
        )

        if annotation_name is not None:
            annotation_names.append(
                str(annotation_name)
            )

    if (
        len(annotation_names)
        != len(set(annotation_names))
    ):
        template_failures.append(
            "duplicate_annotation_names"
        )

    with pymupdf.open(
        str(pdf_path)
    ) as pdf_document:
        pdf_page_count = (
            pdf_document.page_count
        )

    if pdf_page_count != 1:
        template_failures.append(
            {
                "pdf_page_count": (
                    pdf_page_count
                )
            }
        )

    try:
        with PILImage.open(
            preview_path
        ) as preview_image:
            preview_image.verify()

    except Exception as error:
        template_failures.append(
            {
                "invalid_preview_image": (
                    str(error)
                )
            }
        )

    recorded_checksums = (
        qa_report.get(
            "checksums_sha256",
            {},
        )
    )

    actual_checksums = {
        "prototype_pdf": (
            consolidation_sha256(
                pdf_path
            )
        ),
        "preview": (
            consolidation_sha256(
                preview_path
            )
        ),
        "ground_truth": (
            consolidation_sha256(
                ground_truth_path
            )
        ),
    }

    checksum_mismatches = {
        checksum_name: {
            "expected": (
                recorded_checksums.get(
                    checksum_name
                )
            ),
            "actual": actual_checksum,
        }
        for checksum_name, actual_checksum
        in actual_checksums.items()
        if (
            recorded_checksums.get(
                checksum_name
            )
            != actual_checksum
        )
    }

    if checksum_mismatches:
        template_failures.append(
            {
                "checksum_mismatches": (
                    checksum_mismatches
                )
            }
        )

    minimum_font = metrics.get(
        "minimum_field_font_size"
    )

    if (
        minimum_font is None
        or float(minimum_font) < 6.5
    ):
        template_failures.append(
            {
                "minimum_field_font_size": (
                    minimum_font
                )
            }
        )

    template_status = (
        "VALID"
        if not template_failures
        else "INVALID"
    )

    summary_rows.append(
        {
            "template_id": template_id,
            "document_id": document_id,
            "pdf_pages": pdf_page_count,
            "annotations": annotation_count,
            "minimum_font": minimum_font,
            "technical_status": technical_status,
            "manual_review": manual_review,
            "checksums": (
                "MATCH"
                if not checksum_mismatches
                else "MISMATCH"
            ),
            "artifact_counts": (
                "VALID"
                if artifact_counts_valid
                else "INVALID"
            ),
            "status": template_status,
        }
    )

    if template_failures:
        failure_records.append(
            {
                "template_id": template_id,
                "failures": template_failures,
            }
        )


# ================================================================
# Hasil konsolidasi
# ================================================================

prototype_consolidation_summary = (
    pd.DataFrame(
        summary_rows
    )
)

display(
    prototype_consolidation_summary
)

valid_template_count = int(
    (
        prototype_consolidation_summary[
            "status"
        ]
        == "VALID"
    ).sum()
)

overall_status = (
    "PASSED"
    if (
        valid_template_count
        == EXPECTED_TEMPLATE_COUNT
        and not failure_records
    )
    else "FAILED"
)

print(
    f"Prototype root      : "
    f"{prototype_root}"
)

print(
    f"Expected templates  : "
    f"{EXPECTED_TEMPLATE_COUNT}"
)

print(
    f"Valid templates     : "
    f"{valid_template_count}"
)

print(
    f"Invalid templates   : "
    f"{EXPECTED_TEMPLATE_COUNT - valid_template_count}"
)

print(
    f"Consolidated status : "
    f"{overall_status}"
)

print()

if failure_records:
    print("DETAIL KEGAGALAN")

    print(
        json.dumps(
            failure_records,
            ensure_ascii=False,
            indent=2,
        )
    )

    raise RuntimeError(
        "Audit konsolidasi prototype gagal. "
        "Periksa detail kegagalan di atas."
    )

print(
    "✅ SELURUH PROTOTYPE TPL-01 SAMPAI TPL-10 "
    "VALID — audit teknis, review visual, artifact, "
    "anotasi, dan checksum konsisten."
)

In [ ]:
# ================================================================
# CELL 73 — READ-ONLY
# Batch readiness audit:
# renderer mapping, payload count, item capacity, dan prototype gate
# ================================================================

from collections import Counter
from pathlib import Path

import inspect
import json

import pandas as pd


# ================================================================
# Konfigurasi audit
# ================================================================

EXPECTED_TEMPLATE_IDS = [
    f"TPL-{template_number:02d}"
    for template_number in range(1, 11)
]

EXPECTED_TEMPLATE_COUNT = 10
EXPECTED_PAYLOAD_COUNT = 200
EXPECTED_PAYLOADS_PER_TEMPLATE = 20


# Kapasitas sesuai batas masing-masing renderer final.
EXPECTED_RENDER_CAPACITIES = {
    "TPL-01": (2, 8),
    "TPL-02": (1, 8),
    "TPL-03": (1, 7),
    "TPL-04": (2, 8),
    "TPL-05": (2, 8),
    "TPL-06": (2, 8),
    "TPL-07": (2, 8),
    "TPL-08": (2, 8),
    "TPL-09": (2, 8),
    "TPL-10": (2, 8),
}


EXPECTED_RENDERER_NAMES = {
    template_id: (
        f"render_template_"
        f"{int(template_id.split('-')[1]):02d}"
    )
    for template_id in EXPECTED_TEMPLATE_IDS
}


# ================================================================
# Validasi dependency runtime
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "PROTOTYPE_ROOT",
    "template_registry",
    "invoice_render_payloads",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency runtime untuk Cell 73 belum tersedia: "
        f"{missing_runtime_objects}"
    )


if not isinstance(template_registry, dict):
    raise TypeError(
        "template_registry wajib berupa dictionary."
    )

if not isinstance(invoice_render_payloads, dict):
    raise TypeError(
        "invoice_render_payloads wajib berupa dictionary."
    )


prototype_root = Path(PROTOTYPE_ROOT)

if not prototype_root.is_dir():
    raise FileNotFoundError(
        "Prototype root tidak ditemukan: "
        f"{prototype_root}"
    )


# ================================================================
# Helper audit
# ================================================================

def contains_nonempty_value(value):
    """Mendeteksi isi kegagalan secara rekursif."""

    if value is None:
        return False

    if isinstance(value, dict):
        return any(
            contains_nonempty_value(child_value)
            for child_value in value.values()
        )

    if isinstance(value, (list, tuple, set)):
        return any(
            contains_nonempty_value(child_value)
            for child_value in value
        )

    if isinstance(value, str):
        return bool(value.strip())

    return bool(value)


def get_runtime_capacity(
    template_id,
    expected_minimum,
    expected_maximum,
):
    """Membaca konstanta kapasitas renderer jika tersedia."""

    template_number = int(
        template_id.split("-")[1]
    )

    prefix = f"TPL{template_number:02d}"

    minimum_name = (
        f"{prefix}_MIN_ITEMS_PER_PAGE"
    )

    maximum_name = (
        f"{prefix}_MAX_ITEMS_PER_PAGE"
    )

    runtime_minimum = globals().get(
        minimum_name,
        expected_minimum,
    )

    runtime_maximum = globals().get(
        maximum_name,
        expected_maximum,
    )

    return {
        "minimum_name": minimum_name,
        "maximum_name": maximum_name,
        "minimum": runtime_minimum,
        "maximum": runtime_maximum,
    }


def load_prototype_gate(template_id):
    """Membaca status prototype tanpa mengubah QA report."""

    template_root = (
        prototype_root
        / template_id
    )

    qa_report_paths = sorted(
        template_root.glob(
            "*_qa_report.json"
        )
    )

    if len(qa_report_paths) != 1:
        return {
            "passed": False,
            "qa_report_count": len(
                qa_report_paths
            ),
            "technical_status": "MISSING",
            "manual_review": "MISSING",
            "failure_data_present": None,
            "qa_report_path": None,
        }

    qa_report_path = qa_report_paths[0]

    try:
        with qa_report_path.open(
            mode="r",
            encoding="utf-8",
        ) as qa_file:
            qa_report = json.load(
                qa_file
            )
    except Exception as error:
        return {
            "passed": False,
            "qa_report_count": 1,
            "technical_status": "UNREADABLE",
            "manual_review": "UNREADABLE",
            "failure_data_present": True,
            "qa_report_path": str(
                qa_report_path
            ),
            "read_error": repr(error),
        }

    technical_status = qa_report.get(
        "status",
        "MISSING",
    )

    manual_review = qa_report.get(
        "manual_visual_review",
        "MISSING",
    )

    failure_data_present = (
        contains_nonempty_value(
            qa_report.get(
                "failures",
                {},
            )
        )
    )

    passed = all(
        [
            technical_status == "PASSED",
            manual_review == "PASSED",
            not failure_data_present,
        ]
    )

    return {
        "passed": passed,
        "qa_report_count": 1,
        "technical_status": technical_status,
        "manual_review": manual_review,
        "failure_data_present": (
            failure_data_present
        ),
        "qa_report_path": str(
            qa_report_path
        ),
    }


# ================================================================
# Audit renderer mapping
# ================================================================

renderer_mapping = {}
runtime_capacities = {}

renderer_rows = []
renderer_failures = []

for template_id in EXPECTED_TEMPLATE_IDS:
    renderer_name = (
        EXPECTED_RENDERER_NAMES[
            template_id
        ]
    )

    renderer_function = globals().get(
        renderer_name
    )

    renderer_is_callable = callable(
        renderer_function
    )

    signature_text = None
    signature_valid = False

    if renderer_is_callable:
        try:
            renderer_signature = (
                inspect.signature(
                    renderer_function
                )
            )

            signature_text = str(
                renderer_signature
            )

            parameter_names = set(
                renderer_signature
                .parameters
            )

            signature_valid = {
                "render_payload",
                "output_pdf_path",
            }.issubset(parameter_names)

        except Exception:
            signature_valid = False

    expected_minimum, expected_maximum = (
        EXPECTED_RENDER_CAPACITIES[
            template_id
        ]
    )

    runtime_capacity = (
        get_runtime_capacity(
            template_id=template_id,
            expected_minimum=expected_minimum,
            expected_maximum=expected_maximum,
        )
    )

    capacity_matches = (
        runtime_capacity["minimum"]
        == expected_minimum
        and runtime_capacity["maximum"]
        == expected_maximum
    )

    registry_present = (
        template_id
        in template_registry
    )

    prototype_gate = (
        load_prototype_gate(
            template_id
        )
    )

    row_valid = all(
        [
            renderer_is_callable,
            signature_valid,
            capacity_matches,
            registry_present,
            prototype_gate["passed"],
        ]
    )

    if renderer_is_callable:
        renderer_mapping[
            template_id
        ] = renderer_function

    runtime_capacities[
        template_id
    ] = (
        runtime_capacity["minimum"],
        runtime_capacity["maximum"],
    )

    renderer_rows.append(
        {
            "template_id": template_id,
            "renderer": renderer_name,
            "callable": renderer_is_callable,
            "signature_valid": signature_valid,
            "capacity": (
                f"{expected_minimum}–"
                f"{expected_maximum}"
            ),
            "runtime_capacity": (
                f"{runtime_capacity['minimum']}–"
                f"{runtime_capacity['maximum']}"
            ),
            "registry_present": registry_present,
            "prototype_gate": (
                "PASSED"
                if prototype_gate["passed"]
                else "FAILED"
            ),
            "status": (
                "VALID"
                if row_valid
                else "INVALID"
            ),
        }
    )

    if not row_valid:
        renderer_failures.append(
            {
                "template_id": template_id,
                "renderer_name": renderer_name,
                "callable": renderer_is_callable,
                "signature": signature_text,
                "signature_valid": (
                    signature_valid
                ),
                "expected_capacity": [
                    expected_minimum,
                    expected_maximum,
                ],
                "runtime_capacity": [
                    runtime_capacity["minimum"],
                    runtime_capacity["maximum"],
                ],
                "registry_present": (
                    registry_present
                ),
                "prototype_gate": (
                    prototype_gate
                ),
            }
        )


renderer_summary = pd.DataFrame(
    renderer_rows
)

display(
    renderer_summary
)


# ================================================================
# Audit seluruh payload
# ================================================================

payload_failures = []
capacity_violations = []
split_mismatches = []
invalid_quantity_labels = []
invalid_payload_structures = []
canonical_key_mismatches = []

canonical_ids = []
document_ids = []
payload_template_ids = []

payloads_by_template = Counter()


for dictionary_key, payload in (
    invoice_render_payloads.items()
):
    if not isinstance(payload, dict):
        invalid_payload_structures.append(
            {
                "dictionary_key": str(
                    dictionary_key
                ),
                "reason": (
                    "payload bukan dictionary"
                ),
            }
        )
        continue

    canonical_id = payload.get(
        "canonical_invoice_id"
    )

    document_id = payload.get(
        "document_id"
    )

    template_id = payload.get(
        "template_id"
    )

    canonical_ids.append(
        canonical_id
    )

    document_ids.append(
        document_id
    )

    payload_template_ids.append(
        template_id
    )

    payloads_by_template[
        template_id
    ] += 1

    if dictionary_key != canonical_id:
        canonical_key_mismatches.append(
            {
                "dictionary_key": (
                    dictionary_key
                ),
                "canonical_invoice_id": (
                    canonical_id
                ),
            }
        )

    required_values = {
        "canonical_invoice_id": canonical_id,
        "document_id": document_id,
        "template_id": template_id,
    }

    missing_identity_fields = [
        field_name
        for field_name, field_value
        in required_values.items()
        if not isinstance(field_value, str)
        or not field_value.strip()
    ]

    if missing_identity_fields:
        invalid_payload_structures.append(
            {
                "dictionary_key": str(
                    dictionary_key
                ),
                "reason": (
                    "identity field tidak valid"
                ),
                "fields": (
                    missing_identity_fields
                ),
            }
        )

    if template_id not in (
        EXPECTED_RENDER_CAPACITIES
    ):
        invalid_payload_structures.append(
            {
                "canonical_invoice_id": (
                    canonical_id
                ),
                "reason": (
                    "template_id tidak dikenal"
                ),
                "template_id": template_id,
            }
        )
        continue

    items = payload.get(
        "items"
    )

    if not isinstance(items, list):
        invalid_payload_structures.append(
            {
                "canonical_invoice_id": (
                    canonical_id
                ),
                "reason": (
                    "items bukan list"
                ),
            }
        )
        continue

    if not all(
        isinstance(item, dict)
        for item in items
    ):
        invalid_payload_structures.append(
            {
                "canonical_invoice_id": (
                    canonical_id
                ),
                "reason": (
                    "salah satu item bukan dictionary"
                ),
            }
        )

    item_count = len(items)

    minimum_items, maximum_items = (
        EXPECTED_RENDER_CAPACITIES[
            template_id
        ]
    )

    if not (
        minimum_items
        <= item_count
        <= maximum_items
    ):
        capacity_violations.append(
            {
                "canonical_invoice_id": (
                    canonical_id
                ),
                "document_id": document_id,
                "template_id": template_id,
                "item_count": item_count,
                "minimum_items": (
                    minimum_items
                ),
                "maximum_items": (
                    maximum_items
                ),
            }
        )

    registry_specification = (
        template_registry.get(
            template_id,
            {},
        )
    )

    expected_split = (
        registry_specification.get(
            "split"
        )
    )

    actual_split = payload.get(
        "split"
    )

    if actual_split != expected_split:
        split_mismatches.append(
            {
                "canonical_invoice_id": (
                    canonical_id
                ),
                "template_id": template_id,
                "expected_split": (
                    expected_split
                ),
                "actual_split": (
                    actual_split
                ),
            }
        )

    if payload.get("language") == "id":
        quantity_label = (
            payload.get(
                "labels",
                {},
            ).get(
                "quantity"
            )
        )

        if quantity_label != "Kuantitas":
            invalid_quantity_labels.append(
                {
                    "canonical_invoice_id": (
                        canonical_id
                    ),
                    "template_id": (
                        template_id
                    ),
                    "expected": "Kuantitas",
                    "actual": quantity_label,
                }
            )


# ================================================================
# Audit keunikan ID dan nama output logis
# ================================================================

valid_canonical_ids = [
    canonical_id
    for canonical_id in canonical_ids
    if isinstance(canonical_id, str)
    and canonical_id.strip()
]

valid_document_ids = [
    document_id
    for document_id in document_ids
    if isinstance(document_id, str)
    and document_id.strip()
]


duplicate_canonical_ids = sorted(
    canonical_id
    for canonical_id, count
    in Counter(
        valid_canonical_ids
    ).items()
    if count > 1
)

duplicate_document_ids = sorted(
    document_id
    for document_id, count
    in Counter(
        valid_document_ids
    ).items()
    if count > 1
)


# Hanya memeriksa nama yang akan digunakan.
# Tidak ada direktori atau file yang dibuat.
planned_output_names = []

for document_id in valid_document_ids:
    planned_output_names.extend(
        [
            f"{document_id}.pdf",
            f"{document_id}.png",
            (
                f"{document_id}"
                "_ground_truth.json"
            ),
        ]
    )

planned_output_name_conflicts = sorted(
    output_name
    for output_name, count
    in Counter(
        planned_output_names
    ).items()
    if count > 1
)


# ================================================================
# Ringkasan per template
# ================================================================

payload_summary_rows = []

for template_id in EXPECTED_TEMPLATE_IDS:
    minimum_items, maximum_items = (
        EXPECTED_RENDER_CAPACITIES[
            template_id
        ]
    )

    template_payloads = [
        payload
        for payload
        in invoice_render_payloads.values()
        if isinstance(payload, dict)
        and payload.get("template_id")
        == template_id
    ]

    item_counts = [
        len(payload.get("items", []))
        for payload in template_payloads
        if isinstance(
            payload.get("items"),
            list,
        )
    ]

    template_capacity_violations = [
        violation
        for violation in capacity_violations
        if violation["template_id"]
        == template_id
    ]

    template_split_mismatches = [
        mismatch
        for mismatch in split_mismatches
        if mismatch["template_id"]
        == template_id
    ]

    template_label_failures = [
        failure
        for failure
        in invalid_quantity_labels
        if failure["template_id"]
        == template_id
    ]

    actual_payload_count = len(
        template_payloads
    )

    row_valid = all(
        [
            actual_payload_count
            == EXPECTED_PAYLOADS_PER_TEMPLATE,
            len(
                template_capacity_violations
            ) == 0,
            len(
                template_split_mismatches
            ) == 0,
            len(
                template_label_failures
            ) == 0,
        ]
    )

    payload_summary_rows.append(
        {
            "template_id": template_id,
            "expected_payloads": (
                EXPECTED_PAYLOADS_PER_TEMPLATE
            ),
            "actual_payloads": (
                actual_payload_count
            ),
            "allowed_items": (
                f"{minimum_items}–"
                f"{maximum_items}"
            ),
            "actual_min_items": (
                min(item_counts)
                if item_counts
                else None
            ),
            "actual_max_items": (
                max(item_counts)
                if item_counts
                else None
            ),
            "capacity_violations": len(
                template_capacity_violations
            ),
            "split_mismatches": len(
                template_split_mismatches
            ),
            "invalid_id_labels": len(
                template_label_failures
            ),
            "status": (
                "VALID"
                if row_valid
                else "INVALID"
            ),
        }
    )


payload_summary = pd.DataFrame(
    payload_summary_rows
)

display(
    payload_summary
)


# ================================================================
# Kontrol keseluruhan
# ================================================================

unexpected_template_ids = sorted(
    set(payload_template_ids)
    - set(EXPECTED_TEMPLATE_IDS),
    key=lambda value: str(value),
)

missing_template_ids = sorted(
    set(EXPECTED_TEMPLATE_IDS)
    - set(payload_template_ids)
)


global_controls = [
    {
        "control": "template_count",
        "expected": EXPECTED_TEMPLATE_COUNT,
        "actual": len(
            EXPECTED_TEMPLATE_IDS
        ),
    },
    {
        "control": "mapped_renderers",
        "expected": EXPECTED_TEMPLATE_COUNT,
        "actual": len(
            renderer_mapping
        ),
    },
    {
        "control": "valid_renderer_rows",
        "expected": EXPECTED_TEMPLATE_COUNT,
        "actual": int(
            (
                renderer_summary["status"]
                == "VALID"
            ).sum()
        ),
    },
    {
        "control": "payload_count",
        "expected": EXPECTED_PAYLOAD_COUNT,
        "actual": len(
            invoice_render_payloads
        ),
    },
    {
        "control": "templates_with_20_payloads",
        "expected": EXPECTED_TEMPLATE_COUNT,
        "actual": sum(
            payloads_by_template[
                template_id
            ]
            == EXPECTED_PAYLOADS_PER_TEMPLATE
            for template_id
            in EXPECTED_TEMPLATE_IDS
        ),
    },
    {
        "control": "unique_canonical_ids",
        "expected": EXPECTED_PAYLOAD_COUNT,
        "actual": len(
            set(valid_canonical_ids)
        ),
    },
    {
        "control": "unique_document_ids",
        "expected": EXPECTED_PAYLOAD_COUNT,
        "actual": len(
            set(valid_document_ids)
        ),
    },
    {
        "control": "canonical_key_mismatches",
        "expected": 0,
        "actual": len(
            canonical_key_mismatches
        ),
    },
    {
        "control": "invalid_payload_structures",
        "expected": 0,
        "actual": len(
            invalid_payload_structures
        ),
    },
    {
        "control": "capacity_violations",
        "expected": 0,
        "actual": len(
            capacity_violations
        ),
    },
    {
        "control": "split_mismatches",
        "expected": 0,
        "actual": len(
            split_mismatches
        ),
    },
    {
        "control": "invalid_quantity_labels",
        "expected": 0,
        "actual": len(
            invalid_quantity_labels
        ),
    },
    {
        "control": "unexpected_template_ids",
        "expected": 0,
        "actual": len(
            unexpected_template_ids
        ),
    },
    {
        "control": "missing_template_ids",
        "expected": 0,
        "actual": len(
            missing_template_ids
        ),
    },
    {
        "control": "duplicate_canonical_ids",
        "expected": 0,
        "actual": len(
            duplicate_canonical_ids
        ),
    },
    {
        "control": "duplicate_document_ids",
        "expected": 0,
        "actual": len(
            duplicate_document_ids
        ),
    },
    {
        "control": "output_name_conflicts",
        "expected": 0,
        "actual": len(
            planned_output_name_conflicts
        ),
    },
]


for control in global_controls:
    control["status"] = (
        "VALID"
        if control["actual"]
        == control["expected"]
        else "INVALID"
    )


global_summary = pd.DataFrame(
    global_controls
)

display(
    global_summary
)


# ================================================================
# Kumpulkan detail kegagalan
# ================================================================

audit_failures = {
    "renderer_failures": (
        renderer_failures
    ),
    "canonical_key_mismatches": (
        canonical_key_mismatches
    ),
    "invalid_payload_structures": (
        invalid_payload_structures
    ),
    "capacity_violations": (
        capacity_violations
    ),
    "split_mismatches": (
        split_mismatches
    ),
    "invalid_quantity_labels": (
        invalid_quantity_labels
    ),
    "unexpected_template_ids": (
        unexpected_template_ids
    ),
    "missing_template_ids": (
        missing_template_ids
    ),
    "duplicate_canonical_ids": (
        duplicate_canonical_ids
    ),
    "duplicate_document_ids": (
        duplicate_document_ids
    ),
    "output_name_conflicts": (
        planned_output_name_conflicts
    ),
}


batch_readiness_passed = all(
    global_summary["status"]
    == "VALID"
)


# Mapping ini hanya disimpan di memori runtime.
# Cell batch-render berikutnya dapat menggunakannya.
BATCH_RENDERER_MAPPING = dict(
    renderer_mapping
)

BATCH_RENDER_CAPACITIES = dict(
    runtime_capacities
)


# ================================================================
# Ringkasan akhir
# ================================================================

print(
    f"Expected templates    : "
    f"{EXPECTED_TEMPLATE_COUNT}"
)

print(
    f"Mapped renderers      : "
    f"{len(BATCH_RENDERER_MAPPING)}"
)

print(
    f"Expected payloads     : "
    f"{EXPECTED_PAYLOAD_COUNT}"
)

print(
    f"Validated payloads    : "
    f"{len(invoice_render_payloads)}"
)

print(
    f"Capacity violations   : "
    f"{len(capacity_violations)}"
)

print(
    f"Split mismatches      : "
    f"{len(split_mismatches)}"
)

print(
    f"Output-name conflicts : "
    f"{len(planned_output_name_conflicts)}"
)

print(
    "Artifact writes      : 0"
)

print(
    "Batch readiness      : "
    + (
        "PASSED"
        if batch_readiness_passed
        else "FAILED"
    )
)


if not batch_readiness_passed:
    print()
    print(
        "DETAIL KEGAGALAN"
    )

    print(
        json.dumps(
            audit_failures,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
    )

    raise RuntimeError(
        "Batch readiness audit gagal. "
        "Periksa detail kegagalan di atas."
    )


print()
print(
    "✅ BATCH READINESS PASSED — "
    "10 renderer terpetakan, 200 payload tervalidasi, "
    "kapasitas item sesuai, seluruh prototype telah PASSED, "
    "dan tidak ada benturan nama output."
)

print(
    "Belum ada PDF, preview, ground truth, "
    "atau artifact batch yang dibuat."
)

In [ ]:
# ================================================================
# CELL 72A — Read-only
# Inspeksi schema QA report TPL-02
# ================================================================

from pathlib import Path

import hashlib
import json


def inspect_sha256(
    file_path: Path,
) -> str:
    digest = hashlib.sha256()

    with file_path.open("rb") as input_file:
        for chunk in iter(
            lambda: input_file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


tpl02_directory = (
    Path(PROTOTYPE_ROOT)
    / "TPL-02"
)

qa_matches = sorted(
    tpl02_directory.glob(
        "*_qa_report.json"
    )
)

pdf_matches = sorted(
    tpl02_directory.glob(
        "*_prototype.pdf"
    )
)

preview_matches = sorted(
    tpl02_directory.glob(
        "*_preview.png"
    )
)

ground_truth_matches = sorted(
    tpl02_directory.glob(
        "*_ground_truth.json"
    )
)

artifact_counts = {
    "qa_report": len(qa_matches),
    "prototype_pdf": len(pdf_matches),
    "preview": len(preview_matches),
    "ground_truth": len(
        ground_truth_matches
    ),
}

if any(
    count != 1
    for count in artifact_counts.values()
):
    raise RuntimeError(
        "Jumlah artifact TPL-02 tidak sesuai: "
        f"{artifact_counts}"
    )

qa_report_path = qa_matches[0]
pdf_path = pdf_matches[0]
preview_path = preview_matches[0]
ground_truth_path = (
    ground_truth_matches[0]
)

with qa_report_path.open(
    "r",
    encoding="utf-8",
) as input_file:
    qa_report = json.load(
        input_file
    )

with ground_truth_path.open(
    "r",
    encoding="utf-8",
) as input_file:
    ground_truth = json.load(
        input_file
    )

metrics = qa_report.get(
    "metrics",
    {},
)

checksums = qa_report.get(
    "checksums_sha256",
    {},
)

annotations = ground_truth.get(
    "annotations",
    [],
)

print(
    "QA REPORT PATH"
)

print(
    qa_report_path
)

print()
print(
    "TOP-LEVEL KEYS"
)

print(
    sorted(
        qa_report.keys()
    )
)

print()
print(
    "METRICS KEYS"
)

print(
    sorted(
        metrics.keys()
    )
    if isinstance(metrics, dict)
    else type(metrics).__name__
)

print()
print(
    "CHECKSUM KEYS"
)

print(
    sorted(
        checksums.keys()
    )
    if isinstance(checksums, dict)
    else type(checksums).__name__
)

print()
print(
    "RECORDED CHECKSUMS"
)

print(
    json.dumps(
        checksums,
        ensure_ascii=False,
        indent=2,
    )
)

print()
print(
    "CURRENT ARTIFACT CHECKSUMS"
)

print(
    json.dumps(
        {
            "prototype_pdf": (
                inspect_sha256(
                    pdf_path
                )
            ),
            "preview": (
                inspect_sha256(
                    preview_path
                )
            ),
            "ground_truth": (
                inspect_sha256(
                    ground_truth_path
                )
            ),
        },
        ensure_ascii=False,
        indent=2,
    )
)

print()
print(
    "ANNOTATION COUNTS"
)

print(
    json.dumps(
        {
            "ground_truth": len(
                annotations
            ),
            "metrics.annotation_count": (
                metrics.get(
                    "annotation_count"
                )
                if isinstance(
                    metrics,
                    dict,
                )
                else None
            ),
            (
                "metrics."
                "expected_annotation_count"
            ): (
                metrics.get(
                    "expected_annotation_count"
                )
                if isinstance(
                    metrics,
                    dict,
                )
                else None
            ),
        },
        ensure_ascii=False,
        indent=2,
    )
)

print()
print(
    "STATUSES"
)

print(
    json.dumps(
        {
            "technical": (
                qa_report.get("status")
            ),
            "manual_visual_review": (
                qa_report.get(
                    "manual_visual_review"
                )
            ),
            "nonempty_failures": {
                key: value
                for key, value
                in qa_report.get(
                    "failures",
                    {},
                ).items()
                if bool(value)
            },
        },
        ensure_ascii=False,
        indent=2,
    )
)

print()
print(
    "✅ Inspeksi schema QA TPL-02 selesai. "
    "Tidak ada artifact yang diubah."
)

In [ ]:
# ================================================================
# CELL 72B — FINAL
# Migrasi kompatibilitas schema QA TPL-02
# Tidak merender ulang PDF, PNG, atau ground truth
# ================================================================

from pathlib import Path

import hashlib
import json


# ================================================================
# Validasi runtime
# ================================================================

if "PROTOTYPE_ROOT" not in globals():
    raise RuntimeError(
        "PROTOTYPE_ROOT belum tersedia."
    )

if "write_json_atomically" not in globals():
    raise RuntimeError(
        "write_json_atomically belum tersedia."
    )


# ================================================================
# Helper checksum
# ================================================================

def migration_sha256(
    file_path: Path,
) -> str:
    digest = hashlib.sha256()

    with file_path.open("rb") as input_file:
        for chunk in iter(
            lambda: input_file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ================================================================
# Resolve artifact TPL-02
# ================================================================

tpl02_directory = (
    Path(PROTOTYPE_ROOT)
    / "TPL-02"
)

artifact_patterns = {
    "qa_report": "*_qa_report.json",
    "prototype_pdf": "*_prototype.pdf",
    "preview": "*_preview.png",
    "ground_truth": "*_ground_truth.json",
}

resolved_artifacts = {}

for artifact_name, pattern in artifact_patterns.items():
    matches = sorted(
        tpl02_directory.glob(pattern)
    )

    if len(matches) != 1:
        raise RuntimeError(
            f"Artifact {artifact_name!r} harus "
            "berjumlah tepat satu. "
            f"Actual: {len(matches)}; "
            f"matches: {[str(path) for path in matches]}."
        )

    resolved_artifacts[
        artifact_name
    ] = matches[0]

qa_report_path = (
    resolved_artifacts[
        "qa_report"
    ]
)

pdf_path = (
    resolved_artifacts[
        "prototype_pdf"
    ]
)

preview_path = (
    resolved_artifacts[
        "preview"
    ]
)

ground_truth_path = (
    resolved_artifacts[
        "ground_truth"
    ]
)


# ================================================================
# Muat QA report dan ground truth
# ================================================================

with qa_report_path.open(
    "r",
    encoding="utf-8",
) as input_file:
    qa_report = json.load(
        input_file
    )

with ground_truth_path.open(
    "r",
    encoding="utf-8",
) as input_file:
    ground_truth = json.load(
        input_file
    )


# ================================================================
# Validasi status dan identitas
# ================================================================

if qa_report.get("template_id") != "TPL-02":
    raise RuntimeError(
        "QA report bukan milik TPL-02."
    )

if qa_report.get("status") != "PASSED":
    raise RuntimeError(
        "QA teknis TPL-02 belum PASSED."
    )

if (
    qa_report.get("manual_visual_review")
    != "PASSED"
):
    raise RuntimeError(
        "Review visual TPL-02 belum PASSED."
    )

nonempty_failures = {
    failure_name: failure_value
    for failure_name, failure_value
    in qa_report.get(
        "failures",
        {},
    ).items()
    if bool(failure_value)
}

if nonempty_failures:
    raise RuntimeError(
        "QA report masih memiliki kegagalan: "
        f"{nonempty_failures}"
    )

ground_truth_document = (
    ground_truth.get(
        "document",
        {},
    )
)

if (
    ground_truth_document.get("template_id")
    != "TPL-02"
):
    raise RuntimeError(
        "Ground truth bukan milik TPL-02."
    )

if (
    ground_truth_document.get("document_id")
    != qa_report.get("document_id")
):
    raise RuntimeError(
        "Document ID QA report dan ground truth "
        "tidak sama."
    )


# ================================================================
# Verifikasi checksum sebelum migrasi
# ================================================================

actual_checksums = {
    "prototype_pdf": migration_sha256(
        pdf_path
    ),
    "preview": migration_sha256(
        preview_path
    ),
    "ground_truth": migration_sha256(
        ground_truth_path
    ),
}

recorded_checksums = qa_report.get(
    "checksums_sha256",
    {},
)

if not isinstance(
    recorded_checksums,
    dict,
):
    raise TypeError(
        "checksums_sha256 wajib berupa dictionary."
    )

recorded_pdf_checksum = (
    recorded_checksums.get(
        "prototype_pdf"
    )
    or recorded_checksums.get(
        "pdf"
    )
)

checksum_mismatches = {}

if (
    recorded_pdf_checksum
    != actual_checksums["prototype_pdf"]
):
    checksum_mismatches[
        "prototype_pdf"
    ] = {
        "expected": recorded_pdf_checksum,
        "actual": (
            actual_checksums[
                "prototype_pdf"
            ]
        ),
    }

for checksum_name in (
    "preview",
    "ground_truth",
):
    if (
        recorded_checksums.get(
            checksum_name
        )
        != actual_checksums[
            checksum_name
        ]
    ):
        checksum_mismatches[
            checksum_name
        ] = {
            "expected": (
                recorded_checksums.get(
                    checksum_name
                )
            ),
            "actual": (
                actual_checksums[
                    checksum_name
                ]
            ),
        }

if checksum_mismatches:
    raise RuntimeError(
        "Checksum artifact berubah sebelum migrasi: "
        f"{checksum_mismatches}"
    )


# ================================================================
# Verifikasi jumlah anotasi
# ================================================================

annotations = ground_truth.get(
    "annotations",
    [],
)

if not isinstance(annotations, list):
    raise TypeError(
        "ground_truth.annotations wajib berupa list."
    )

actual_annotation_count = len(
    annotations
)

metrics = qa_report.get(
    "metrics",
    {},
)

if not isinstance(metrics, dict):
    raise TypeError(
        "metrics wajib berupa dictionary."
    )

reported_annotation_count = (
    metrics.get(
        "annotation_count"
    )
)

if (
    reported_annotation_count
    != actual_annotation_count
):
    raise RuntimeError(
        "Jumlah anotasi QA report dan ground truth "
        "tidak sama. "
        f"QA: {reported_annotation_count}; "
        f"ground truth: {actual_annotation_count}."
    )


# ================================================================
# Tambahkan field kompatibilitas
# ================================================================

metrics[
    "expected_annotation_count"
] = actual_annotation_count

recorded_checksums[
    "prototype_pdf"
] = actual_checksums[
    "prototype_pdf"
]

qa_report["metrics"] = metrics

qa_report[
    "checksums_sha256"
] = recorded_checksums


# ================================================================
# Simpan secara atomik
# ================================================================

write_json_atomically(
    qa_report_path,
    qa_report,
)


# ================================================================
# Verifikasi round-trip
# ================================================================

with qa_report_path.open(
    "r",
    encoding="utf-8",
) as input_file:
    persisted_report = json.load(
        input_file
    )

if (
    persisted_report
    .get("metrics", {})
    .get("expected_annotation_count")
    != actual_annotation_count
):
    raise RuntimeError(
        "expected_annotation_count gagal disimpan."
    )

if (
    persisted_report
    .get("checksums_sha256", {})
    .get("prototype_pdf")
    != actual_checksums["prototype_pdf"]
):
    raise RuntimeError(
        "Checksum prototype_pdf gagal disimpan."
    )

if (
    persisted_report.get("status")
    != "PASSED"
):
    raise RuntimeError(
        "Status teknis berubah setelah migrasi."
    )

if (
    persisted_report.get(
        "manual_visual_review"
    )
    != "PASSED"
):
    raise RuntimeError(
        "Status manual berubah setelah migrasi."
    )


# ================================================================
# Output
# ================================================================

print(
    f"QA report                : "
    f"{qa_report_path}"
)

print(
    f"Technical status         : "
    f"{persisted_report['status']}"
)

print(
    f"Manual visual review     : "
    f"{persisted_report['manual_visual_review']}"
)

print(
    f"Expected annotations     : "
    f"{actual_annotation_count}"
)

print(
    "Prototype PDF checksum  : "
    f"{actual_checksums['prototype_pdf']}"
)

print(
    "Legacy PDF key retained : "
    f"{'pdf' in persisted_report['checksums_sha256']}"
)

print()

print(
    "✅ Schema QA TPL-02 berhasil dimigrasikan "
    "tanpa mengubah PDF, preview, atau ground truth."
)

In [ ]:
# ================================================================
# CELL 74 — FINAL
# Batch rendering 200 synthetic invoices
# ================================================================

from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import hashlib
import json
import os
import time
import unicodedata

import pandas as pd
import pymupdf


# ================================================================
# Konfigurasi batch
# ================================================================

BATCH_EXPECTED_DOCUMENTS = 200
BATCH_EXPECTED_TEMPLATES = 10
BATCH_EXPECTED_PER_TEMPLATE = 20
BATCH_PREVIEW_DPI = 150
BATCH_MAX_FAILURES = 5

BATCH_DATASET_ROOT = (
    Path(BUILD_RUN_ROOT)
    / "rendered_dataset"
)

BATCH_PDF_ROOT = (
    BATCH_DATASET_ROOT
    / "pdf"
)

BATCH_PREVIEW_ROOT = (
    BATCH_DATASET_ROOT
    / "preview"
)

BATCH_GROUND_TRUTH_ROOT = (
    BATCH_DATASET_ROOT
    / "ground_truth"
)

BATCH_STAGING_ROOT = (
    BATCH_DATASET_ROOT
    / "_staging"
)

BATCH_RENDER_INDEX_PATH = (
    Path(BUILD_MANIFESTS_ROOT)
    / "batch_render_index.jsonl"
)

BATCH_RENDER_MANIFEST_PATH = (
    Path(BUILD_MANIFESTS_ROOT)
    / "batch_render_manifest.json"
)


# ================================================================
# Validasi dependency
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "BUILD_RUN_ROOT",
    "BUILD_MANIFESTS_ROOT",
    "CANONICAL_RECORDS_JSONL_PATH",
    "invoice_render_payloads",
    "BATCH_RENDERER_MAPPING",
    "BATCH_RENDER_CAPACITIES",
    "annotation_to_dict",
    "write_json_atomically",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Dependency Cell 74 belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali Cell 73."
    )

if not isinstance(
    invoice_render_payloads,
    dict,
):
    raise TypeError(
        "invoice_render_payloads wajib berupa dictionary."
    )

if not isinstance(
    BATCH_RENDERER_MAPPING,
    dict,
):
    raise TypeError(
        "BATCH_RENDERER_MAPPING wajib berupa dictionary."
    )

if not isinstance(
    BATCH_RENDER_CAPACITIES,
    dict,
):
    raise TypeError(
        "BATCH_RENDER_CAPACITIES wajib berupa dictionary."
    )

if (
    len(invoice_render_payloads)
    != BATCH_EXPECTED_DOCUMENTS
):
    raise RuntimeError(
        "Jumlah render payload tidak sesuai. "
        f"Expected: {BATCH_EXPECTED_DOCUMENTS}; "
        f"actual: {len(invoice_render_payloads)}."
    )

if (
    len(BATCH_RENDERER_MAPPING)
    != BATCH_EXPECTED_TEMPLATES
):
    raise RuntimeError(
        "Jumlah renderer tidak sesuai. "
        f"Expected: {BATCH_EXPECTED_TEMPLATES}; "
        f"actual: {len(BATCH_RENDERER_MAPPING)}."
    )

expected_template_ids = {
    f"TPL-{template_number:02d}"
    for template_number in range(1, 11)
}

if (
    set(BATCH_RENDERER_MAPPING)
    != expected_template_ids
):
    raise RuntimeError(
        "Renderer mapping harus berisi "
        "TPL-01 sampai TPL-10."
    )

canonical_jsonl_path = Path(
    CANONICAL_RECORDS_JSONL_PATH
)

if not canonical_jsonl_path.is_file():
    raise FileNotFoundError(
        "Canonical JSONL tidak ditemukan: "
        f"{canonical_jsonl_path}"
    )


# ================================================================
# Helper checksum
# ================================================================

def sha256_file(file_path):
    """Menghitung checksum SHA-256 file."""

    file_path = Path(file_path)
    digest = hashlib.sha256()

    with file_path.open("rb") as binary_file:
        for chunk in iter(
            lambda: binary_file.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ================================================================
# Helper normalisasi teks PDF
# ================================================================

def normalize_pdf_text(value):
    """
    Menormalisasi teks untuk validasi hasil ekstraksi PDF.

    Separator seperti —, –, -, ·, bullet, tanda baca, dan
    variasi spasi tidak menyebabkan perbedaan palsu.
    """

    normalized_value = unicodedata.normalize(
        "NFKC",
        str(value),
    ).casefold()

    normalized_value = "".join(
        character
        if character.isalnum()
        else " "
        for character in normalized_value
    )

    return " ".join(
        normalized_value.split()
    )


# ================================================================
# Helper penulisan JSONL atomik
# ================================================================

def write_jsonl_atomically(
    output_path,
    records,
):
    """Menulis JSONL melalui file sementara."""

    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        output_path.parent
        / f".{output_path.name}.tmp"
    )

    temporary_path.unlink(
        missing_ok=True
    )

    try:
        with temporary_path.open(
            mode="w",
            encoding="utf-8",
            newline="\n",
        ) as output_file:
            for record in records:
                serialized_record = json.dumps(
                    record,
                    ensure_ascii=False,
                    separators=(",", ":"),
                    allow_nan=False,
                )

                output_file.write(
                    serialized_record + "\n"
                )

        os.replace(
            temporary_path,
            output_path,
        )

    except Exception:
        temporary_path.unlink(
            missing_ok=True
        )
        raise


# ================================================================
# Helper staging
# ================================================================

def clean_staging_files(*file_paths):
    """Menghapus file sementara milik satu dokumen."""

    for file_path in file_paths:
        Path(file_path).unlink(
            missing_ok=True
        )


# ================================================================
# Helper preview PDF
# ================================================================

def create_pdf_preview(
    pdf_path,
    preview_path,
    dpi=150,
):
    """Membuat preview halaman pertama secara atomik."""

    pdf_path = Path(pdf_path)
    preview_path = Path(preview_path)

    preview_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_preview_path = (
        preview_path.parent
        / (
            f".{preview_path.stem}"
            ".tmp.png"
        )
    )

    temporary_preview_path.unlink(
        missing_ok=True
    )

    try:
        with pymupdf.open(
            str(pdf_path)
        ) as pdf_document:
            if pdf_document.page_count != 1:
                raise RuntimeError(
                    "PDF harus memiliki tepat satu halaman."
                )

            page = pdf_document[0]

            pixmap = page.get_pixmap(
                matrix=pymupdf.Matrix(
                    dpi / 72,
                    dpi / 72,
                ),
                alpha=False,
            )

            pixmap.save(
                str(temporary_preview_path)
            )

        if not temporary_preview_path.is_file():
            raise RuntimeError(
                "Preview sementara gagal dibuat."
            )

        if (
            temporary_preview_path.stat().st_size
            == 0
        ):
            raise RuntimeError(
                "Preview sementara berukuran nol byte."
            )

        os.replace(
            temporary_preview_path,
            preview_path,
        )

    except Exception:
        temporary_preview_path.unlink(
            missing_ok=True
        )
        raise


# ================================================================
# Muat canonical payload
# ================================================================

canonical_payloads = {}

with canonical_jsonl_path.open(
    mode="r",
    encoding="utf-8",
) as canonical_file:
    for line_number, line in enumerate(
        canonical_file,
        start=1,
    ):
        clean_line = line.strip()

        if not clean_line:
            continue

        try:
            canonical_payload = json.loads(
                clean_line
            )
        except json.JSONDecodeError as error:
            raise RuntimeError(
                "Canonical JSONL tidak valid "
                f"pada baris {line_number}."
            ) from error

        canonical_id = (
            canonical_payload.get(
                "canonical_invoice_id"
            )
        )

        if not isinstance(
            canonical_id,
            str,
        ) or not canonical_id.strip():
            raise RuntimeError(
                "canonical_invoice_id tidak valid "
                f"pada baris {line_number}."
            )

        if canonical_id in canonical_payloads:
            raise RuntimeError(
                "Canonical ID duplikat: "
                f"{canonical_id}"
            )

        canonical_payloads[
            canonical_id
        ] = canonical_payload


if (
    len(canonical_payloads)
    != BATCH_EXPECTED_DOCUMENTS
):
    raise RuntimeError(
        "Jumlah canonical payload tidak sesuai. "
        f"Expected: {BATCH_EXPECTED_DOCUMENTS}; "
        f"actual: {len(canonical_payloads)}."
    )

if (
    set(canonical_payloads)
    != set(invoice_render_payloads)
):
    missing_canonical_ids = sorted(
        set(invoice_render_payloads)
        - set(canonical_payloads)
    )

    missing_presentation_ids = sorted(
        set(canonical_payloads)
        - set(invoice_render_payloads)
    )

    raise RuntimeError(
        "Canonical dan presentation payload "
        "tidak sinkron. "
        f"Missing canonical: "
        f"{missing_canonical_ids[:10]}; "
        f"missing presentation: "
        f"{missing_presentation_ids[:10]}."
    )


# ================================================================
# Validasi urutan ID
# ================================================================

ordered_canonical_ids = sorted(
    invoice_render_payloads
)

expected_canonical_ids = [
    f"CANON-{record_number:06d}"
    for record_number in range(
        1,
        BATCH_EXPECTED_DOCUMENTS + 1,
    )
]

if (
    ordered_canonical_ids
    != expected_canonical_ids
):
    raise RuntimeError(
        "Canonical ID harus berurutan dari "
        "CANON-000001 sampai CANON-000200."
    )

document_ids = [
    invoice_render_payloads[
        canonical_id
    ]["document_id"]
    for canonical_id
    in ordered_canonical_ids
]

if (
    len(set(document_ids))
    != BATCH_EXPECTED_DOCUMENTS
):
    raise RuntimeError(
        "document_id tidak unik."
    )


# ================================================================
# Validasi distribusi template
# ================================================================

template_distribution = Counter(
    invoice_render_payloads[
        canonical_id
    ]["template_id"]
    for canonical_id
    in ordered_canonical_ids
)

if set(template_distribution) != expected_template_ids:
    raise RuntimeError(
        "Distribusi template tidak mencakup "
        "TPL-01 sampai TPL-10."
    )

invalid_template_distribution = {
    template_id: count
    for template_id, count
    in template_distribution.items()
    if count != BATCH_EXPECTED_PER_TEMPLATE
}

if invalid_template_distribution:
    raise RuntimeError(
        "Jumlah payload per template tidak sesuai: "
        f"{invalid_template_distribution}"
    )


# ================================================================
# Validasi kapasitas item
# ================================================================

for canonical_id in ordered_canonical_ids:
    render_payload = (
        invoice_render_payloads[
            canonical_id
        ]
    )

    template_id = (
        render_payload["template_id"]
    )

    items = render_payload.get(
        "items"
    )

    if not isinstance(items, list):
        raise TypeError(
            f"items untuk {canonical_id} "
            "wajib berupa list."
        )

    item_count = len(items)

    minimum_items, maximum_items = (
        BATCH_RENDER_CAPACITIES[
            template_id
        ]
    )

    if not (
        minimum_items
        <= item_count
        <= maximum_items
    ):
        raise RuntimeError(
            "Kapasitas item tidak valid untuk "
            f"{canonical_id}. "
            f"Actual: {item_count}; "
            f"allowed: {minimum_items}–"
            f"{maximum_items}."
        )


# ================================================================
# Siapkan direktori batch
# ================================================================

required_directories = [
    BATCH_PDF_ROOT,
    BATCH_PREVIEW_ROOT,
    BATCH_GROUND_TRUTH_ROOT,
    BATCH_STAGING_ROOT,
    Path(BUILD_MANIFESTS_ROOT),
]

for required_directory in required_directories:
    required_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ================================================================
# Mulai batch rendering
# ================================================================

batch_started_utc = datetime.now(
    timezone.utc
)

batch_started_monotonic = (
    time.monotonic()
)

batch_index_records = []
batch_failures = []

print(
    "Memulai batch rendering 200 invoice..."
)

print(
    f"Output root: {BATCH_DATASET_ROOT}"
)

print()


for sequence_number, canonical_id in enumerate(
    ordered_canonical_ids,
    start=1,
):
    render_payload = (
        invoice_render_payloads[
            canonical_id
        ]
    )

    canonical_payload = (
        canonical_payloads[
            canonical_id
        ]
    )

    document_id = (
        render_payload["document_id"]
    )

    template_id = (
        render_payload["template_id"]
    )

    split_name = (
        render_payload["split"]
    )

    language = (
        render_payload["language"]
    )

    currency = (
        render_payload["currency"]
    )

    item_count = len(
        render_payload["items"]
    )

    renderer = (
        BATCH_RENDERER_MAPPING[
            template_id
        ]
    )

    final_pdf_directory = (
        BATCH_PDF_ROOT
        / split_name
        / template_id
    )

    final_preview_directory = (
        BATCH_PREVIEW_ROOT
        / split_name
        / template_id
    )

    final_ground_truth_directory = (
        BATCH_GROUND_TRUTH_ROOT
        / split_name
        / template_id
    )

    for final_directory in [
        final_pdf_directory,
        final_preview_directory,
        final_ground_truth_directory,
    ]:
        final_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

    final_pdf_path = (
        final_pdf_directory
        / f"{document_id}.pdf"
    )

    final_preview_path = (
        final_preview_directory
        / f"{document_id}.png"
    )

    final_ground_truth_path = (
        final_ground_truth_directory
        / f"{document_id}.json"
    )

    staging_pdf_path = (
        BATCH_STAGING_ROOT
        / f"{document_id}.pdf"
    )

    staging_preview_path = (
        BATCH_STAGING_ROOT
        / f"{document_id}.png"
    )

    staging_ground_truth_path = (
        BATCH_STAGING_ROOT
        / f"{document_id}.json"
    )

    clean_staging_files(
        staging_pdf_path,
        staging_preview_path,
        staging_ground_truth_path,
    )

    try:
        annotations, rendering_metadata = (
            renderer(
                render_payload=render_payload,
                output_pdf_path=(
                    staging_pdf_path
                ),
            )
        )

        expected_annotation_count = (
            19 + 4 * item_count
        )

        actual_annotation_count = len(
            annotations
        )

        if (
            actual_annotation_count
            != expected_annotation_count
        ):
            raise RuntimeError(
                "Jumlah anotasi tidak sesuai. "
                f"Expected: "
                f"{expected_annotation_count}; "
                f"actual: "
                f"{actual_annotation_count}."
            )

        if not staging_pdf_path.is_file():
            raise RuntimeError(
                "Renderer tidak menghasilkan PDF."
            )

        if staging_pdf_path.stat().st_size == 0:
            raise RuntimeError(
                "PDF hasil render berukuran nol byte."
            )

        with pymupdf.open(
            str(staging_pdf_path)
        ) as rendered_document:
            if rendered_document.page_count != 1:
                raise RuntimeError(
                    "PDF harus memiliki satu halaman."
                )

            rendered_page = (
                rendered_document[0]
            )

            extracted_text = (
                rendered_page.get_text(
                    "text"
                )
            )

        # --------------------------------------------------------
        # Validasi teks dengan normalisasi
        # --------------------------------------------------------

        required_text_fields = {
            "invoice_number": (
                render_payload[
                    "metadata"
                ]["invoice_number"][
                    "display_value"
                ]
            ),
            "vendor_name": (
                render_payload[
                    "vendor"
                ]["name"]
            ),
            "buyer_name": (
                render_payload[
                    "buyer"
                ]["name"]
            ),
            "quantity_label": (
                render_payload[
                    "labels"
                ]["quantity"]
            ),
            "total_value": (
                render_payload[
                    "financials"
                ]["total"][
                    "display_value"
                ]
            ),
            "synthetic_notice": (
                render_payload[
                    "synthetic_notice"
                ]
            ),
        }

        normalized_extracted_text = (
            normalize_pdf_text(
                extracted_text
            )
        )

        missing_text_fields = []

        for (
            field_name,
            expected_text,
        ) in required_text_fields.items():
            normalized_expected_text = (
                normalize_pdf_text(
                    expected_text
                )
            )

            if (
                normalized_expected_text
                not in normalized_extracted_text
            ):
                missing_text_fields.append(
                    {
                        "field_name": field_name,
                        "expected_text": (
                            expected_text
                        ),
                        "normalized_text": (
                            normalized_expected_text
                        ),
                    }
                )

        if missing_text_fields:
            raise RuntimeError(
                "Teks penting tidak ditemukan "
                "pada PDF setelah normalisasi: "
                f"{missing_text_fields}"
            )

        # --------------------------------------------------------
        # Buat preview
        # --------------------------------------------------------

        create_pdf_preview(
            pdf_path=staging_pdf_path,
            preview_path=(
                staging_preview_path
            ),
            dpi=BATCH_PREVIEW_DPI,
        )

        # --------------------------------------------------------
        # Buat ground truth
        # --------------------------------------------------------

        ground_truth = {
            "schema_version": "1.0.0",
            "dataset_id": (
                "SYNTHETIC-INVOICE-V1"
            ),
            "document": {
                "document_id": document_id,
                "canonical_invoice_id": (
                    canonical_id
                ),
                "split": split_name,
                "template_id": template_id,
                "language": language,
                "currency": currency,
            },
            "canonical": canonical_payload,
            "rendering": (
                rendering_metadata
            ),
            "annotations": [
                annotation_to_dict(
                    annotation
                )
                for annotation
                in annotations
            ],
        }

        write_json_atomically(
            staging_ground_truth_path,
            ground_truth,
        )

        if (
            not staging_ground_truth_path
            .is_file()
        ):
            raise RuntimeError(
                "Ground truth gagal dibuat."
            )

        if (
            staging_ground_truth_path
            .stat()
            .st_size
            == 0
        ):
            raise RuntimeError(
                "Ground truth berukuran nol byte."
            )

        # --------------------------------------------------------
        # Round-trip ground truth
        # --------------------------------------------------------

        with staging_ground_truth_path.open(
            mode="r",
            encoding="utf-8",
        ) as ground_truth_file:
            reloaded_ground_truth = (
                json.load(
                    ground_truth_file
                )
            )

        if (
            reloaded_ground_truth[
                "document"
            ]["document_id"]
            != document_id
        ):
            raise RuntimeError(
                "Round-trip document_id "
                "ground truth gagal."
            )

        if (
            reloaded_ground_truth[
                "document"
            ]["canonical_invoice_id"]
            != canonical_id
        ):
            raise RuntimeError(
                "Round-trip canonical_invoice_id "
                "ground truth gagal."
            )

        if (
            len(
                reloaded_ground_truth[
                    "annotations"
                ]
            )
            != expected_annotation_count
        ):
            raise RuntimeError(
                "Jumlah anotasi ground truth "
                "berubah setelah dibuka ulang."
            )

        # --------------------------------------------------------
        # Commit artifact ke lokasi final
        # --------------------------------------------------------

        os.replace(
            staging_pdf_path,
            final_pdf_path,
        )

        os.replace(
            staging_preview_path,
            final_preview_path,
        )

        os.replace(
            staging_ground_truth_path,
            final_ground_truth_path,
        )

        # --------------------------------------------------------
        # Checksum artifact
        # --------------------------------------------------------

        pdf_checksum = sha256_file(
            final_pdf_path
        )

        preview_checksum = sha256_file(
            final_preview_path
        )

        ground_truth_checksum = (
            sha256_file(
                final_ground_truth_path
            )
        )

        batch_index_records.append(
            {
                "sequence_number": (
                    sequence_number
                ),
                "canonical_invoice_id": (
                    canonical_id
                ),
                "document_id": document_id,
                "template_id": template_id,
                "split": split_name,
                "language": language,
                "currency": currency,
                "item_count": item_count,
                "page_count": 1,
                "annotation_count": (
                    actual_annotation_count
                ),
                "minimum_field_font_size": (
                    rendering_metadata.get(
                        "minimum_field_font_size"
                    )
                ),
                "font_adjustment_count": (
                    rendering_metadata.get(
                        "font_adjustment_count"
                    )
                ),
                "artifacts": {
                    "pdf": {
                        "relative_path": str(
                            final_pdf_path
                            .relative_to(
                                BATCH_DATASET_ROOT
                            )
                        ),
                        "size_bytes": (
                            final_pdf_path
                            .stat()
                            .st_size
                        ),
                        "sha256": pdf_checksum,
                    },
                    "preview": {
                        "relative_path": str(
                            final_preview_path
                            .relative_to(
                                BATCH_DATASET_ROOT
                            )
                        ),
                        "size_bytes": (
                            final_preview_path
                            .stat()
                            .st_size
                        ),
                        "sha256": (
                            preview_checksum
                        ),
                    },
                    "ground_truth": {
                        "relative_path": str(
                            final_ground_truth_path
                            .relative_to(
                                BATCH_DATASET_ROOT
                            )
                        ),
                        "size_bytes": (
                            final_ground_truth_path
                            .stat()
                            .st_size
                        ),
                        "sha256": (
                            ground_truth_checksum
                        ),
                    },
                },
                "status": "RENDERED",
            }
        )

    except Exception as error:
        clean_staging_files(
            staging_pdf_path,
            staging_preview_path,
            staging_ground_truth_path,
        )

        batch_failures.append(
            {
                "sequence_number": (
                    sequence_number
                ),
                "canonical_invoice_id": (
                    canonical_id
                ),
                "document_id": document_id,
                "template_id": template_id,
                "error_type": (
                    type(error).__name__
                ),
                "error": str(error),
            }
        )

    if (
        sequence_number % 10 == 0
        or sequence_number
        == BATCH_EXPECTED_DOCUMENTS
    ):
        print(
            f"[{sequence_number:03d}/"
            f"{BATCH_EXPECTED_DOCUMENTS}] "
            f"berhasil="
            f"{len(batch_index_records)}, "
            f"gagal="
            f"{len(batch_failures)}"
        )

    # Fail-fast agar kesalahan sistemik
    # tidak diulang terhadap 200 dokumen.
    if (
        len(batch_failures)
        >= BATCH_MAX_FAILURES
    ):
        print()
        print(
            "Batch dihentikan lebih awal karena "
            f"mencapai {BATCH_MAX_FAILURES} kegagalan."
        )
        break


# ================================================================
# Hentikan jika ada kegagalan
# ================================================================

if batch_failures:
    failure_table = pd.DataFrame(
        batch_failures
    )

    display(
        failure_table
    )

    print()
    print(
        f"Dokumen berhasil : "
        f"{len(batch_index_records)}"
    )

    print(
        f"Dokumen gagal    : "
        f"{len(batch_failures)}"
    )

    print(
        "Manifest final belum ditulis karena "
        "batch belum lengkap."
    )

    raise RuntimeError(
        "Batch rendering belum berhasil sepenuhnya. "
        "Periksa tabel kegagalan di atas."
    )


# ================================================================
# Validasi kelengkapan hasil
# ================================================================

if (
    len(batch_index_records)
    != BATCH_EXPECTED_DOCUMENTS
):
    raise RuntimeError(
        "Jumlah dokumen hasil render tidak sesuai. "
        f"Expected: {BATCH_EXPECTED_DOCUMENTS}; "
        f"actual: {len(batch_index_records)}."
    )

rendered_document_ids = [
    record["document_id"]
    for record in batch_index_records
]

if (
    len(set(rendered_document_ids))
    != BATCH_EXPECTED_DOCUMENTS
):
    raise RuntimeError(
        "Document ID hasil render tidak unik."
    )

rendered_canonical_ids = [
    record["canonical_invoice_id"]
    for record in batch_index_records
]

if (
    rendered_canonical_ids
    != ordered_canonical_ids
):
    raise RuntimeError(
        "Urutan canonical ID hasil render "
        "tidak sesuai."
    )


# ================================================================
# Simpan batch index
# ================================================================

write_jsonl_atomically(
    BATCH_RENDER_INDEX_PATH,
    batch_index_records,
)

batch_index_checksum = sha256_file(
    BATCH_RENDER_INDEX_PATH
)


# ================================================================
# Buka ulang batch index
# ================================================================

reloaded_batch_index = []

with BATCH_RENDER_INDEX_PATH.open(
    mode="r",
    encoding="utf-8",
) as index_file:
    for line_number, line in enumerate(
        index_file,
        start=1,
    ):
        clean_line = line.strip()

        if not clean_line:
            continue

        try:
            record = json.loads(
                clean_line
            )
        except json.JSONDecodeError as error:
            raise RuntimeError(
                "Batch index tidak valid "
                f"pada baris {line_number}."
            ) from error

        reloaded_batch_index.append(
            record
        )

if (
    len(reloaded_batch_index)
    != BATCH_EXPECTED_DOCUMENTS
):
    raise RuntimeError(
        "Batch index harus memiliki "
        "tepat 200 record."
    )

if (
    reloaded_batch_index
    != batch_index_records
):
    raise RuntimeError(
        "Round-trip batch index tidak sama."
    )


# ================================================================
# Hitung artifact aktual
# ================================================================

actual_pdf_paths = sorted(
    BATCH_PDF_ROOT.rglob("*.pdf")
)

actual_preview_paths = sorted(
    BATCH_PREVIEW_ROOT.rglob("*.png")
)

actual_ground_truth_paths = sorted(
    BATCH_GROUND_TRUTH_ROOT.rglob(
        "*.json"
    )
)

artifact_count_controls = [
    {
        "artifact": "PDF",
        "expected": 200,
        "actual": len(
            actual_pdf_paths
        ),
    },
    {
        "artifact": "preview",
        "expected": 200,
        "actual": len(
            actual_preview_paths
        ),
    },
    {
        "artifact": "ground_truth",
        "expected": 200,
        "actual": len(
            actual_ground_truth_paths
        ),
    },
    {
        "artifact": "index_records",
        "expected": 200,
        "actual": len(
            reloaded_batch_index
        ),
    },
]

for control in artifact_count_controls:
    control["status"] = (
        "VALID"
        if control["actual"]
        == control["expected"]
        else "INVALID"
    )

artifact_count_summary = pd.DataFrame(
    artifact_count_controls
)

display(
    artifact_count_summary
)

if not all(
    artifact_count_summary["status"]
    == "VALID"
):
    raise RuntimeError(
        "Jumlah artifact batch tidak sesuai."
    )


# ================================================================
# Ringkasan per template
# ================================================================

batch_index_frame = pd.DataFrame(
    [
        {
            "template_id": (
                record["template_id"]
            ),
            "split": record["split"],
            "language": (
                record["language"]
            ),
            "currency": (
                record["currency"]
            ),
            "item_count": (
                record["item_count"]
            ),
            "annotation_count": (
                record[
                    "annotation_count"
                ]
            ),
            "minimum_field_font": (
                record[
                    "minimum_field_font_size"
                ]
            ),
            "font_adjustments": (
                record[
                    "font_adjustment_count"
                ]
            ),
        }
        for record in batch_index_records
    ]
)

template_batch_summary = (
    batch_index_frame
    .groupby(
        "template_id",
        as_index=False,
    )
    .agg(
        document_count=(
            "template_id",
            "size",
        ),
        minimum_items=(
            "item_count",
            "min",
        ),
        maximum_items=(
            "item_count",
            "max",
        ),
        minimum_font=(
            "minimum_field_font",
            "min",
        ),
        total_annotations=(
            "annotation_count",
            "sum",
        ),
        font_adjustments=(
            "font_adjustments",
            "sum",
        ),
    )
    .sort_values(
        "template_id"
    )
    .reset_index(drop=True)
)

template_batch_summary["status"] = (
    template_batch_summary[
        "document_count"
    ].apply(
        lambda count: (
            "VALID"
            if count
            == BATCH_EXPECTED_PER_TEMPLATE
            else "INVALID"
        )
    )
)

display(
    template_batch_summary
)

if not all(
    template_batch_summary["status"]
    == "VALID"
):
    raise RuntimeError(
        "Jumlah dokumen per template "
        "tidak sesuai."
    )


# ================================================================
# Distribusi batch
# ================================================================

split_distribution = {
    str(key): int(value)
    for key, value
    in batch_index_frame[
        "split"
    ].value_counts().sort_index().items()
}

language_distribution = {
    str(key): int(value)
    for key, value
    in batch_index_frame[
        "language"
    ].value_counts().sort_index().items()
}

currency_distribution = {
    str(key): int(value)
    for key, value
    in batch_index_frame[
        "currency"
    ].value_counts().sort_index().items()
}

template_distribution_final = {
    str(key): int(value)
    for key, value
    in batch_index_frame[
        "template_id"
    ].value_counts().sort_index().items()
}


# ================================================================
# Buat manifest
# ================================================================

batch_completed_utc = datetime.now(
    timezone.utc
)

batch_duration_seconds = (
    time.monotonic()
    - batch_started_monotonic
)

batch_render_manifest = {
    "schema_version": "1.0.0",
    "dataset_id": "SYNTHETIC-INVOICE-V1",
    "dataset_version": "1.0.0",
    "build_id": Path(
        BUILD_RUN_ROOT
    ).name,
    "artifact_type": (
        "rendered_invoice_collection"
    ),
    "render_status": "PASSED",
    "started_utc": (
        batch_started_utc.isoformat()
    ),
    "completed_utc": (
        batch_completed_utc.isoformat()
    ),
    "duration_seconds": round(
        batch_duration_seconds,
        3,
    ),
    "preview_dpi": BATCH_PREVIEW_DPI,
    "document_count": len(
        batch_index_records
    ),
    "template_count": int(
        batch_index_frame[
            "template_id"
        ].nunique()
    ),
    "artifact_counts": {
        "pdf": len(
            actual_pdf_paths
        ),
        "preview": len(
            actual_preview_paths
        ),
        "ground_truth": len(
            actual_ground_truth_paths
        ),
    },
    "distributions": {
        "template": (
            template_distribution_final
        ),
        "split": split_distribution,
        "language": (
            language_distribution
        ),
        "currency": (
            currency_distribution
        ),
    },
    "quality_controls": {
        "batch_readiness": "PASSED",
        "render_failures": 0,
        "missing_required_text": 0,
        "duplicate_document_ids": 0,
        "duplicate_canonical_ids": 0,
        "expected_annotation_formula": (
            "19 + 4 * item_count"
        ),
        "all_single_page": True,
        "text_comparison": (
            "unicode_nfkc_casefold_"
            "alphanumeric"
        ),
    },
    "artifacts": {
        "dataset_root": str(
            BATCH_DATASET_ROOT
        ),
        "pdf_root": str(
            BATCH_PDF_ROOT
        ),
        "preview_root": str(
            BATCH_PREVIEW_ROOT
        ),
        "ground_truth_root": str(
            BATCH_GROUND_TRUTH_ROOT
        ),
        "batch_index": {
            "path": str(
                BATCH_RENDER_INDEX_PATH
            ),
            "record_count": len(
                batch_index_records
            ),
            "size_bytes": (
                BATCH_RENDER_INDEX_PATH
                .stat()
                .st_size
            ),
            "sha256": (
                batch_index_checksum
            ),
        },
    },
}

write_json_atomically(
    BATCH_RENDER_MANIFEST_PATH,
    batch_render_manifest,
)


# ================================================================
# Verifikasi manifest
# ================================================================

with BATCH_RENDER_MANIFEST_PATH.open(
    mode="r",
    encoding="utf-8",
) as manifest_file:
    reloaded_batch_manifest = (
        json.load(
            manifest_file
        )
    )

if (
    reloaded_batch_manifest[
        "render_status"
    ]
    != "PASSED"
):
    raise RuntimeError(
        "Status manifest batch bukan PASSED."
    )

if (
    reloaded_batch_manifest[
        "document_count"
    ]
    != BATCH_EXPECTED_DOCUMENTS
):
    raise RuntimeError(
        "Document count manifest tidak sesuai."
    )

recorded_index_checksum = (
    reloaded_batch_manifest[
        "artifacts"
    ]["batch_index"]["sha256"]
)

current_index_checksum = sha256_file(
    BATCH_RENDER_INDEX_PATH
)

if (
    recorded_index_checksum
    != current_index_checksum
):
    raise RuntimeError(
        "Checksum batch index tidak sesuai."
    )


# ================================================================
# Variabel runtime untuk QA batch
# ================================================================

BATCH_RENDER_RECORDS = {
    record["canonical_invoice_id"]: record
    for record in reloaded_batch_index
}

BATCH_RENDER_STATUS = "PASSED"


# ================================================================
# Ringkasan akhir
# ================================================================

print(
    f"Batch dataset root : "
    f"{BATCH_DATASET_ROOT}"
)

print(
    f"PDF root           : "
    f"{BATCH_PDF_ROOT}"
)

print(
    f"Preview root       : "
    f"{BATCH_PREVIEW_ROOT}"
)

print(
    f"Ground truth root  : "
    f"{BATCH_GROUND_TRUTH_ROOT}"
)

print(
    f"Batch index        : "
    f"{BATCH_RENDER_INDEX_PATH}"
)

print(
    f"Batch manifest     : "
    f"{BATCH_RENDER_MANIFEST_PATH}"
)

print(
    f"Documents rendered : "
    f"{len(batch_index_records)}"
)

print(
    f"PDF files          : "
    f"{len(actual_pdf_paths)}"
)

print(
    f"Preview files      : "
    f"{len(actual_preview_paths)}"
)

print(
    f"Ground truths      : "
    f"{len(actual_ground_truth_paths)}"
)

print(
    f"Render failures    : "
    f"{len(batch_failures)}"
)

print(
    f"Index SHA-256      : "
    f"{batch_index_checksum}"
)

print(
    f"Duration           : "
    f"{batch_duration_seconds:.2f} seconds"
)

print(
    f"Batch status       : "
    f"{BATCH_RENDER_STATUS}"
)

print()

print(
    "✅ BATCH RENDERING PASSED — "
    "200 PDF, 200 preview, dan 200 ground truth "
    "berhasil dibuat serta diverifikasi."
)

print(
    "Audit QA teknis seluruh dokumen "
    "belum dijalankan."
)

In [ ]:
# ================================================================
# CELL 74C — READ-ONLY
# Pemulihan state batch setelah runtime terputus
# ================================================================

from pathlib import Path

import hashlib
import json

import pandas as pd


# ================================================================
# Pastikan Google Drive tersedia
# ================================================================

GOOGLE_DRIVE_ROOT = Path(
    "/content/drive/MyDrive"
)

if not GOOGLE_DRIVE_ROOT.is_dir():
    try:
        from google.colab import drive

        drive.mount(
            "/content/drive"
        )
    except ImportError as error:
        raise RuntimeError(
            "Cell ini dirancang untuk Google Colab."
        ) from error

if not GOOGLE_DRIVE_ROOT.is_dir():
    raise RuntimeError(
        "Google Drive belum berhasil dipasang."
    )


# ================================================================
# Lokasi build yang sudah selesai
# ================================================================

BUILD_RUN_ROOT = Path(
    "/content/drive/MyDrive/"
    "InvoiceFlow-AI-Data/"
    "interim/"
    "synthetic_v1_build/"
    "20260904T150025Z"
)

BUILD_MANIFESTS_ROOT = (
    BUILD_RUN_ROOT
    / "manifests"
)

BATCH_DATASET_ROOT = (
    BUILD_RUN_ROOT
    / "rendered_dataset"
)

BATCH_PDF_ROOT = (
    BATCH_DATASET_ROOT
    / "pdf"
)

BATCH_PREVIEW_ROOT = (
    BATCH_DATASET_ROOT
    / "preview"
)

BATCH_GROUND_TRUTH_ROOT = (
    BATCH_DATASET_ROOT
    / "ground_truth"
)

BATCH_RENDER_INDEX_PATH = (
    BUILD_MANIFESTS_ROOT
    / "batch_render_index.jsonl"
)

BATCH_RENDER_MANIFEST_PATH = (
    BUILD_MANIFESTS_ROOT
    / "batch_render_manifest.json"
)

CANONICAL_RECORDS_JSONL_PATH = (
    BUILD_MANIFESTS_ROOT
    / "canonical_invoices.jsonl"
)

PRESENTATION_PAYLOADS_JSONL_PATH = (
    BUILD_MANIFESTS_ROOT
    / "invoice_render_payloads.jsonl"
)


# ================================================================
# Helper checksum
# ================================================================

def recovery_sha256_file(file_path):
    """Menghitung SHA-256 sebuah file."""

    file_path = Path(file_path)
    digest = hashlib.sha256()

    with file_path.open("rb") as binary_file:
        for chunk in iter(
            lambda: binary_file.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ================================================================
# Validasi file checkpoint
# ================================================================

required_paths = [
    BUILD_RUN_ROOT,
    BUILD_MANIFESTS_ROOT,
    BATCH_DATASET_ROOT,
    BATCH_PDF_ROOT,
    BATCH_PREVIEW_ROOT,
    BATCH_GROUND_TRUTH_ROOT,
    BATCH_RENDER_INDEX_PATH,
    BATCH_RENDER_MANIFEST_PATH,
    CANONICAL_RECORDS_JSONL_PATH,
    PRESENTATION_PAYLOADS_JSONL_PATH,
]

missing_paths = [
    str(required_path)
    for required_path in required_paths
    if not required_path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Artifact recovery belum lengkap: "
        f"{missing_paths}"
    )


# ================================================================
# Muat manifest
# ================================================================

with BATCH_RENDER_MANIFEST_PATH.open(
    mode="r",
    encoding="utf-8",
) as manifest_file:
    batch_render_manifest = json.load(
        manifest_file
    )

if (
    batch_render_manifest.get(
        "render_status"
    )
    != "PASSED"
):
    raise RuntimeError(
        "Status batch manifest bukan PASSED."
    )

if (
    batch_render_manifest.get(
        "document_count"
    )
    != 200
):
    raise RuntimeError(
        "Document count pada manifest "
        "bukan 200."
    )


# ================================================================
# Muat batch index
# ================================================================

batch_index_records = []

with BATCH_RENDER_INDEX_PATH.open(
    mode="r",
    encoding="utf-8",
) as index_file:
    for line_number, line in enumerate(
        index_file,
        start=1,
    ):
        clean_line = line.strip()

        if not clean_line:
            continue

        try:
            record = json.loads(
                clean_line
            )
        except json.JSONDecodeError as error:
            raise RuntimeError(
                "Batch index tidak valid "
                f"pada baris {line_number}."
            ) from error

        batch_index_records.append(
            record
        )

if len(batch_index_records) != 200:
    raise RuntimeError(
        "Batch index harus berisi "
        "tepat 200 record."
    )


# ================================================================
# Validasi checksum index
# ================================================================

recorded_index_checksum = (
    batch_render_manifest[
        "artifacts"
    ]["batch_index"]["sha256"]
)

current_index_checksum = (
    recovery_sha256_file(
        BATCH_RENDER_INDEX_PATH
    )
)

if (
    recorded_index_checksum
    != current_index_checksum
):
    raise RuntimeError(
        "Checksum batch index tidak sesuai "
        "dengan manifest."
    )


# ================================================================
# Validasi keunikan record
# ================================================================

canonical_ids = [
    record[
        "canonical_invoice_id"
    ]
    for record in batch_index_records
]

document_ids = [
    record["document_id"]
    for record in batch_index_records
]

if len(set(canonical_ids)) != 200:
    raise RuntimeError(
        "Canonical ID pada batch index "
        "tidak unik."
    )

if len(set(document_ids)) != 200:
    raise RuntimeError(
        "Document ID pada batch index "
        "tidak unik."
    )


# ================================================================
# Hitung ulang artifact
# ================================================================

pdf_paths = sorted(
    BATCH_PDF_ROOT.rglob("*.pdf")
)

preview_paths = sorted(
    BATCH_PREVIEW_ROOT.rglob("*.png")
)

ground_truth_paths = sorted(
    BATCH_GROUND_TRUTH_ROOT.rglob(
        "*.json"
    )
)

recovery_controls = [
    {
        "control": "manifest_status",
        "expected": "PASSED",
        "actual": (
            batch_render_manifest[
                "render_status"
            ]
        ),
    },
    {
        "control": "index_checksum",
        "expected": "MATCH",
        "actual": (
            "MATCH"
            if recorded_index_checksum
            == current_index_checksum
            else "MISMATCH"
        ),
    },
    {
        "control": "index_records",
        "expected": 200,
        "actual": len(
            batch_index_records
        ),
    },
    {
        "control": "unique_canonical_ids",
        "expected": 200,
        "actual": len(
            set(canonical_ids)
        ),
    },
    {
        "control": "unique_document_ids",
        "expected": 200,
        "actual": len(
            set(document_ids)
        ),
    },
    {
        "control": "pdf_files",
        "expected": 200,
        "actual": len(pdf_paths),
    },
    {
        "control": "preview_files",
        "expected": 200,
        "actual": len(
            preview_paths
        ),
    },
    {
        "control": "ground_truth_files",
        "expected": 200,
        "actual": len(
            ground_truth_paths
        ),
    },
]

for control in recovery_controls:
    control["status"] = (
        "VALID"
        if control["actual"]
        == control["expected"]
        else "INVALID"
    )

recovery_summary = pd.DataFrame(
    recovery_controls
)

display(
    recovery_summary
)

if not all(
    recovery_summary["status"]
    == "VALID"
):
    raise RuntimeError(
        "Recovery checkpoint tidak valid."
    )


# ================================================================
# Pulihkan variabel runtime
# ================================================================

BATCH_RENDER_RECORDS = {
    record["canonical_invoice_id"]: record
    for record in batch_index_records
}

BATCH_RENDER_STATUS = "PASSED"

BATCH_RECOVERY_READY = True


# ================================================================
# Ringkasan
# ================================================================

print(
    f"Build root        : "
    f"{BUILD_RUN_ROOT}"
)

print(
    f"Batch manifest    : "
    f"{BATCH_RENDER_MANIFEST_PATH}"
)

print(
    f"Batch index       : "
    f"{BATCH_RENDER_INDEX_PATH}"
)

print(
    f"Recovered records : "
    f"{len(BATCH_RENDER_RECORDS)}"
)

print(
    f"PDF files         : "
    f"{len(pdf_paths)}"
)

print(
    f"Preview files     : "
    f"{len(preview_paths)}"
)

print(
    f"Ground truths     : "
    f"{len(ground_truth_paths)}"
)

print(
    f"Recovery status   : "
    f"{BATCH_RENDER_STATUS}"
)

print()

print(
    "✅ CHECKPOINT RECOVERY READY — "
    "state batch berhasil dipulihkan dari Google Drive."
)

print(
    "Jika runtime terputus, jalankan cell ini kembali "
    "lalu lanjutkan ke Cell 75."
)

In [ ]:
# ================================================================
# CELL 75A — FINAL
# Definisi mesin audit teknis batch
# Belum menulis atau mengubah artifact
# ================================================================

from pathlib import Path
from typing import Any

import hashlib
import json
import math
import unicodedata

import numpy as np
import pandas as pd
import pymupdf
from PIL import Image as PILImage


# ================================================================
# Konfigurasi QA
# ================================================================

BATCH_QA_SCHEMA_VERSION = "1.0.0"
BATCH_QA_AUDITOR_VERSION = "1.0.0"

MINIMUM_FIELD_FONT_SIZE = 6.5
MAXIMUM_FONT_ADJUSTMENT_RATIO = 0.20
MINIMUM_RASTER_CONTENT_RATIO = 0.01
MAXIMUM_RASTER_CONTENT_RATIO = 0.40
MINIMUM_RASTER_CONTRAST = 10.0
SEVERE_OVERLAP_THRESHOLD = 0.50

BATCH_EXPECTED_DOCUMENTS = 200

BATCH_QA_ROOT = (
    BATCH_DATASET_ROOT
    / "qa"
)

BATCH_QA_INDEX_PATH = (
    BUILD_MANIFESTS_ROOT
    / "batch_qa_index.jsonl"
)

BATCH_QA_SUMMARY_PATH = (
    BUILD_MANIFESTS_ROOT
    / "batch_qa_summary.csv"
)

BATCH_QA_MANIFEST_PATH = (
    BUILD_MANIFESTS_ROOT
    / "batch_qa_manifest.json"
)


# ================================================================
# Validasi checkpoint recovery
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "BUILD_RUN_ROOT",
    "BUILD_MANIFESTS_ROOT",
    "BATCH_DATASET_ROOT",
    "BATCH_PDF_ROOT",
    "BATCH_PREVIEW_ROOT",
    "BATCH_GROUND_TRUTH_ROOT",
    "BATCH_RENDER_INDEX_PATH",
    "BATCH_RENDER_MANIFEST_PATH",
    "PRESENTATION_PAYLOADS_JSONL_PATH",
    "BATCH_RENDER_RECORDS",
    "BATCH_RENDER_STATUS",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "State recovery belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali Cell 74C."
    )

if BATCH_RENDER_STATUS != "PASSED":
    raise RuntimeError(
        "Status batch rendering bukan PASSED."
    )

if len(BATCH_RENDER_RECORDS) != 200:
    raise RuntimeError(
        "BATCH_RENDER_RECORDS harus berisi "
        "tepat 200 record."
    )

required_paths = [
    Path(BATCH_DATASET_ROOT),
    Path(BATCH_PDF_ROOT),
    Path(BATCH_PREVIEW_ROOT),
    Path(BATCH_GROUND_TRUTH_ROOT),
    Path(BATCH_RENDER_INDEX_PATH),
    Path(BATCH_RENDER_MANIFEST_PATH),
    Path(PRESENTATION_PAYLOADS_JSONL_PATH),
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Artifact batch belum lengkap: "
        f"{missing_paths}"
    )


# ================================================================
# Helper checksum
# ================================================================

def batch_qa_sha256_file(
    file_path: Path,
) -> str:
    """Menghitung checksum SHA-256."""

    digest = hashlib.sha256()

    with Path(file_path).open(
        "rb"
    ) as binary_file:
        for chunk in iter(
            lambda: binary_file.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ================================================================
# Helper normalisasi teks
# ================================================================

def batch_qa_normalize_text(
    value: Any,
) -> str:
    """
    Normalisasi Unicode, kapitalisasi, tanda baca,
    separator, dan spasi untuk perbandingan PDF.
    """

    text = unicodedata.normalize(
        "NFKC",
        str(value or ""),
    ).casefold()

    text = "".join(
        character
        if character.isalnum()
        else " "
        for character in text
    )

    return " ".join(
        text.split()
    )


# ================================================================
# Helper schema anotasi
# ================================================================

def batch_qa_first_present(
    mapping: dict[str, Any],
    candidate_keys: tuple[str, ...],
) -> Any:
    for key in candidate_keys:
        if key in mapping:
            return mapping[key]

    return None


def batch_qa_annotation_field_name(
    annotation: dict[str, Any],
) -> str:
    value = batch_qa_first_present(
        annotation,
        (
            "field_name",
            "field_path",
            "field",
            "path",
            "name",
        ),
    )

    if value is None:
        raise KeyError(
            "Nama field anotasi tidak ditemukan. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return str(value)


def batch_qa_annotation_text(
    annotation: dict[str, Any],
) -> str:
    value = batch_qa_first_present(
        annotation,
        (
            "text",
            "value",
            "display_value",
            "field_value",
            "raw_value",
            "raw_text",
            "text_value",
            "content",
        ),
    )

    if isinstance(value, dict):
        value = batch_qa_first_present(
            value,
            (
                "text",
                "value",
                "display_value",
                "raw",
            ),
        )

    if value is None:
        raise KeyError(
            "Teks anotasi tidak ditemukan. "
            f"Keys: {sorted(annotation.keys())}"
        )

    return str(value)


def batch_qa_bbox_from_value(
    value: Any,
):
    """Mengubah beberapa format bbox menjadi x0,y0,x1,y1."""

    if isinstance(
        value,
        (list, tuple),
    ):
        if len(value) != 4:
            return None

        try:
            return tuple(
                float(number)
                for number in value
            )
        except (TypeError, ValueError):
            return None

    if not isinstance(value, dict):
        return None

    direct_key_sets = [
        ("x0", "y0", "x1", "y1"),
        ("left", "top", "right", "bottom"),
    ]

    for key_set in direct_key_sets:
        if all(
            key in value
            for key in key_set
        ):
            try:
                return tuple(
                    float(value[key])
                    for key in key_set
                )
            except (TypeError, ValueError):
                return None

    if all(
        key in value
        for key in (
            "x",
            "y",
            "width",
            "height",
        )
    ):
        try:
            x0 = float(value["x"])
            y0 = float(value["y"])
            width = float(
                value["width"]
            )
            height = float(
                value["height"]
            )

            return (
                x0,
                y0,
                x0 + width,
                y0 + height,
            )
        except (TypeError, ValueError):
            return None

    if all(
        key in value
        for key in (
            "x",
            "y",
            "w",
            "h",
        )
    ):
        try:
            x0 = float(value["x"])
            y0 = float(value["y"])
            width = float(value["w"])
            height = float(value["h"])

            return (
                x0,
                y0,
                x0 + width,
                y0 + height,
            )
        except (TypeError, ValueError):
            return None

    return None


def batch_qa_annotation_bbox(
    annotation: dict[str, Any],
):
    candidate = batch_qa_first_present(
        annotation,
        (
            "bbox",
            "bbox_points",
            "bbox_pt",
            "bbox_pdf",
            "bounding_box",
            "rectangle",
            "rect",
            "coordinates",
            "coordinates_points",
        ),
    )

    if candidate is None:
        geometry = annotation.get(
            "geometry"
        )

        if isinstance(geometry, dict):
            candidate = (
                batch_qa_first_present(
                    geometry,
                    (
                        "bbox",
                        "bounding_box",
                        "rectangle",
                    ),
                )
            )

    bbox = batch_qa_bbox_from_value(
        candidate
    )

    if bbox is None:
        raise KeyError(
            "Bounding box tidak ditemukan "
            "atau tidak valid."
        )

    return bbox


# ================================================================
# Helper geometri
# ================================================================

def batch_qa_rectangle_area(
    bbox,
) -> float:
    x0, y0, x1, y1 = bbox

    return (
        max(0.0, x1 - x0)
        * max(0.0, y1 - y0)
    )


def batch_qa_intersection_area(
    first_bbox,
    second_bbox,
) -> float:
    first_x0, first_y0, first_x1, first_y1 = (
        first_bbox
    )

    second_x0, second_y0, second_x1, second_y1 = (
        second_bbox
    )

    width = max(
        0.0,
        min(first_x1, second_x1)
        - max(first_x0, second_x0),
    )

    height = max(
        0.0,
        min(first_y1, second_y1)
        - max(first_y0, second_y0),
    )

    return width * height


def batch_qa_bbox_intersects_word(
    annotation_bbox,
    word_bbox,
) -> bool:
    return (
        batch_qa_intersection_area(
            annotation_bbox,
            word_bbox,
        )
        > 0.0
    )


# ================================================================
# Helper lokasi artifact
# ================================================================

def batch_qa_resolve_artifact_path(
    render_record: dict[str, Any],
    artifact_name: str,
) -> Path:
    artifact_record = (
        render_record[
            "artifacts"
        ][artifact_name]
    )

    relative_path = Path(
        artifact_record[
            "relative_path"
        ]
    )

    if relative_path.is_absolute():
        raise RuntimeError(
            "Path artifact pada index "
            "harus bersifat relatif."
        )

    dataset_root_resolved = (
        Path(BATCH_DATASET_ROOT)
        .resolve()
    )

    artifact_path = (
        Path(BATCH_DATASET_ROOT)
        / relative_path
    ).resolve()

    try:
        artifact_path.relative_to(
            dataset_root_resolved
        )
    except ValueError as error:
        raise RuntimeError(
            "Path artifact keluar dari "
            "BATCH_DATASET_ROOT."
        ) from error

    return artifact_path


# ================================================================
# Expected annotation mapping
# ================================================================

def batch_qa_expected_values(
    render_payload: dict[str, Any],
) -> dict[str, str]:
    expected_values = {
        "invoice_number": str(
            render_payload[
                "metadata"
            ]["invoice_number"][
                "display_value"
            ]
        ),
        "invoice_date": str(
            render_payload[
                "metadata"
            ]["invoice_date"][
                "display_value"
            ]
        ),
        "due_date": str(
            render_payload[
                "metadata"
            ]["due_date"][
                "display_value"
            ]
        ),
        "currency": str(
            render_payload["currency"]
        ),
    }

    for party_type in (
        "vendor",
        "buyer",
    ):
        party = render_payload[
            party_type
        ]

        expected_values.update(
            {
                f"{party_type}.name": str(
                    party["name"]
                ),
                (
                    f"{party_type}"
                    ".address_lines"
                ): "\n".join(
                    party["address_lines"]
                ),
                f"{party_type}.email": str(
                    party["email"]
                ),
                f"{party_type}.phone": str(
                    party["phone"]
                ),
                (
                    f"{party_type}"
                    ".tax_identifier"
                ): str(
                    party[
                        "tax_identifier_display"
                    ]
                ),
            }
        )

    for item_index, item in enumerate(
        render_payload["items"]
    ):
        for field_name in (
            "description",
            "quantity",
            "unit_price",
            "line_total",
        ):
            expected_values[
                (
                    f"items[{item_index}]"
                    f".{field_name}"
                )
            ] = str(
                item[field_name]
            )

    for field_name in (
        "subtotal",
        "tax",
        "discount",
        "total",
    ):
        expected_values[
            f"financials.{field_name}"
        ] = str(
            render_payload[
                "financials"
            ][field_name][
                "display_value"
            ]
        )

    expected_values[
        "document.synthetic_notice"
    ] = str(
        render_payload[
            "synthetic_notice"
        ]
    )

    return expected_values


# ================================================================
# Audit satu dokumen
# ================================================================

def batch_qa_audit_document(
    render_record: dict[str, Any],
    render_payload: dict[str, Any],
) -> dict[str, Any]:
    canonical_id = (
        render_record[
            "canonical_invoice_id"
        ]
    )

    document_id = (
        render_record["document_id"]
    )

    template_id = (
        render_record["template_id"]
    )

    pdf_path = (
        batch_qa_resolve_artifact_path(
            render_record,
            "pdf",
        )
    )

    preview_path = (
        batch_qa_resolve_artifact_path(
            render_record,
            "preview",
        )
    )

    ground_truth_path = (
        batch_qa_resolve_artifact_path(
            render_record,
            "ground_truth",
        )
    )

    artifact_paths = {
        "pdf": pdf_path,
        "preview": preview_path,
        "ground_truth": (
            ground_truth_path
        ),
    }

    missing_artifacts = [
        artifact_name
        for artifact_name, artifact_path
        in artifact_paths.items()
        if not artifact_path.is_file()
    ]

    if missing_artifacts:
        raise FileNotFoundError(
            "Artifact tidak ditemukan: "
            f"{missing_artifacts}"
        )

    empty_artifacts = [
        artifact_name
        for artifact_name, artifact_path
        in artifact_paths.items()
        if artifact_path.stat().st_size
        == 0
    ]

    if empty_artifacts:
        raise RuntimeError(
            "Artifact berukuran nol byte: "
            f"{empty_artifacts}"
        )

    current_checksums = {
        artifact_name: (
            batch_qa_sha256_file(
                artifact_path
            )
        )
        for artifact_name, artifact_path
        in artifact_paths.items()
    }

    expected_checksums = {
        artifact_name: (
            render_record[
                "artifacts"
            ][artifact_name]["sha256"]
        )
        for artifact_name
        in (
            "pdf",
            "preview",
            "ground_truth",
        )
    }

    checksum_mismatches = {
        artifact_name: {
            "expected": (
                expected_checksums[
                    artifact_name
                ]
            ),
            "actual": (
                current_checksums[
                    artifact_name
                ]
            ),
        }
        for artifact_name
        in expected_checksums
        if (
            expected_checksums[
                artifact_name
            ]
            != current_checksums[
                artifact_name
            ]
        )
    }

    with ground_truth_path.open(
        mode="r",
        encoding="utf-8",
    ) as ground_truth_file:
        ground_truth = json.load(
            ground_truth_file
        )

    ground_truth_document = (
        ground_truth.get(
            "document",
            {},
        )
    )

    ground_truth_identity_valid = all(
        [
            ground_truth_document.get(
                "document_id"
            )
            == document_id,
            ground_truth_document.get(
                "canonical_invoice_id"
            )
            == canonical_id,
            ground_truth_document.get(
                "template_id"
            )
            == template_id,
        ]
    )

    annotation_records = (
        ground_truth.get(
            "annotations",
            [],
        )
    )

    if not isinstance(
        annotation_records,
        list,
    ):
        raise TypeError(
            "ground_truth.annotations "
            "wajib berupa list."
        )

    rendering_metadata = (
        ground_truth.get(
            "rendering",
            {},
        )
    )

    expected_values = (
        batch_qa_expected_values(
            render_payload
        )
    )

    expected_annotation_count = len(
        expected_values
    )

    parsed_annotations = []
    annotation_parse_failures = []

    for annotation_index, annotation in enumerate(
        annotation_records
    ):
        try:
            parsed_annotations.append(
                {
                    "index": (
                        annotation_index
                    ),
                    "field_name": (
                        batch_qa_annotation_field_name(
                            annotation
                        )
                    ),
                    "text": (
                        batch_qa_annotation_text(
                            annotation
                        )
                    ),
                    "bbox": (
                        batch_qa_annotation_bbox(
                            annotation
                        )
                    ),
                }
            )

        except Exception as error:
            annotation_parse_failures.append(
                {
                    "index": (
                        annotation_index
                    ),
                    "error_type": (
                        type(error).__name__
                    ),
                    "error": str(error),
                }
            )

    annotation_names = [
        annotation["field_name"]
        for annotation
        in parsed_annotations
    ]

    duplicate_annotation_count = (
        len(annotation_names)
        - len(set(annotation_names))
    )

    annotation_by_name = {
        annotation["field_name"]: annotation
        for annotation
        in parsed_annotations
    }

    missing_required_annotations = sorted(
        set(expected_values)
        - set(annotation_names)
    )

    normalization_mismatches = []

    for (
        field_name,
        expected_value,
    ) in expected_values.items():
        annotation = annotation_by_name.get(
            field_name
        )

        if annotation is None:
            continue

        actual_value = annotation[
            "text"
        ]

        if (
            batch_qa_normalize_text(
                actual_value
            )
            != batch_qa_normalize_text(
                expected_value
            )
        ):
            normalization_mismatches.append(
                {
                    "field_name": (
                        field_name
                    ),
                    "expected": (
                        expected_value
                    ),
                    "actual": (
                        actual_value
                    ),
                }
            )

    with pymupdf.open(
        str(pdf_path)
    ) as pdf_document:
        pdf_page_count = (
            pdf_document.page_count
        )

        if pdf_page_count < 1:
            raise RuntimeError(
                "PDF tidak memiliki halaman."
            )

        first_page = pdf_document[0]
        page_rectangle = first_page.rect

        extracted_text = (
            first_page.get_text("text")
        )

        extracted_words = (
            first_page.get_text("words")
        )

    page_width = float(
        page_rectangle.width
    )

    page_height = float(
        page_rectangle.height
    )

    word_boxes = [
        (
            float(word[0]),
            float(word[1]),
            float(word[2]),
            float(word[3]),
        )
        for word in extracted_words
        if len(word) >= 5
        and str(word[4]).strip()
    ]

    invalid_bounding_boxes = []
    annotations_without_words = []
    valid_annotations = []

    for annotation in parsed_annotations:
        field_name = annotation[
            "field_name"
        ]

        x0, y0, x1, y1 = (
            annotation["bbox"]
        )

        coordinates_are_finite = all(
            math.isfinite(value)
            for value in (
                x0,
                y0,
                x1,
                y1,
            )
        )

        bbox_is_valid = all(
            [
                coordinates_are_finite,
                x0 >= 0.0,
                y0 >= 0.0,
                x1 <= page_width,
                y1 <= page_height,
                x1 > x0,
                y1 > y0,
            ]
        )

        if not bbox_is_valid:
            invalid_bounding_boxes.append(
                field_name
            )
            continue

        valid_annotations.append(
            annotation
        )

        contains_word = any(
            batch_qa_bbox_intersects_word(
                annotation["bbox"],
                word_bbox,
            )
            for word_bbox in word_boxes
        )

        if not contains_word:
            annotations_without_words.append(
                field_name
            )

    normalized_extracted_text = (
        batch_qa_normalize_text(
            extracted_text
        )
    )

    missing_extracted_values = []

    for annotation in parsed_annotations:
        normalized_annotation_text = (
            batch_qa_normalize_text(
                annotation["text"]
            )
        )

        if (
            normalized_annotation_text
            and normalized_annotation_text
            not in normalized_extracted_text
        ):
            missing_extracted_values.append(
                annotation["field_name"]
            )

    severe_annotation_overlaps = []

    for first_index in range(
        len(valid_annotations)
    ):
        first_annotation = (
            valid_annotations[
                first_index
            ]
        )

        first_area = (
            batch_qa_rectangle_area(
                first_annotation["bbox"]
            )
        )

        if first_area <= 0.0:
            continue

        for second_index in range(
            first_index + 1,
            len(valid_annotations),
        ):
            second_annotation = (
                valid_annotations[
                    second_index
                ]
            )

            second_area = (
                batch_qa_rectangle_area(
                    second_annotation["bbox"]
                )
            )

            if second_area <= 0.0:
                continue

            overlap_area = (
                batch_qa_intersection_area(
                    first_annotation["bbox"],
                    second_annotation["bbox"],
                )
            )

            overlap_ratio = (
                overlap_area
                / min(
                    first_area,
                    second_area,
                )
            )

            if (
                overlap_ratio
                >= SEVERE_OVERLAP_THRESHOLD
            ):
                severe_annotation_overlaps.append(
                    {
                        "first": (
                            first_annotation[
                                "field_name"
                            ]
                        ),
                        "second": (
                            second_annotation[
                                "field_name"
                            ]
                        ),
                        "overlap_ratio": round(
                            overlap_ratio,
                            6,
                        ),
                    }
                )

    minimum_field_font_size = float(
        rendering_metadata.get(
            "minimum_field_font_size",
            0.0,
        )
    )

    font_adjustment_count = int(
        rendering_metadata.get(
            "font_adjustment_count",
            0,
        )
    )

    font_adjustment_ratio = (
        font_adjustment_count
        / max(
            1,
            len(parsed_annotations),
        )
    )

    with PILImage.open(
        preview_path
    ) as preview_image:
        grayscale_array = np.asarray(
            preview_image.convert("L"),
            dtype=np.float32,
        ).copy()

        image_width_pixels = (
            preview_image.width
        )

        image_height_pixels = (
            preview_image.height
        )

    if grayscale_array.size == 0:
        raise RuntimeError(
            "Preview tidak memiliki piksel."
        )

    raster_content_ratio = float(
        np.mean(
            grayscale_array < 245.0
        )
    )

    raster_contrast = float(
        np.std(grayscale_array)
    )

    index_consistency_valid = all(
        [
            render_record.get(
                "page_count"
            )
            == pdf_page_count,
            render_record.get(
                "annotation_count"
            )
            == len(parsed_annotations),
            render_record.get(
                "item_count"
            )
            == len(
                render_payload["items"]
            ),
        ]
    )

    checks = [
        {
            "control": (
                "artifact_checksums"
            ),
            "expected": "MATCH",
            "actual": (
                "MATCH"
                if not checksum_mismatches
                else "MISMATCH"
            ),
            "valid": (
                not checksum_mismatches
            ),
        },
        {
            "control": (
                "ground_truth_identity"
            ),
            "expected": True,
            "actual": (
                ground_truth_identity_valid
            ),
            "valid": (
                ground_truth_identity_valid
            ),
        },
        {
            "control": (
                "index_consistency"
            ),
            "expected": True,
            "actual": (
                index_consistency_valid
            ),
            "valid": (
                index_consistency_valid
            ),
        },
        {
            "control": "pdf_page_count",
            "expected": 1,
            "actual": pdf_page_count,
            "valid": (
                pdf_page_count == 1
            ),
        },
        {
            "control": "annotation_count",
            "expected": (
                expected_annotation_count
            ),
            "actual": len(
                parsed_annotations
            ),
            "valid": (
                len(parsed_annotations)
                == expected_annotation_count
            ),
        },
        {
            "control": (
                "annotation_parse_failures"
            ),
            "expected": 0,
            "actual": len(
                annotation_parse_failures
            ),
            "valid": (
                not annotation_parse_failures
            ),
        },
        {
            "control": (
                "duplicate_annotations"
            ),
            "expected": 0,
            "actual": (
                duplicate_annotation_count
            ),
            "valid": (
                duplicate_annotation_count
                == 0
            ),
        },
        {
            "control": (
                "missing_required_annotations"
            ),
            "expected": 0,
            "actual": len(
                missing_required_annotations
            ),
            "valid": (
                not missing_required_annotations
            ),
        },
        {
            "control": (
                "invalid_bounding_boxes"
            ),
            "expected": 0,
            "actual": len(
                invalid_bounding_boxes
            ),
            "valid": (
                not invalid_bounding_boxes
            ),
        },
        {
            "control": (
                "normalization_mismatches"
            ),
            "expected": 0,
            "actual": len(
                normalization_mismatches
            ),
            "valid": (
                not normalization_mismatches
            ),
        },
        {
            "control": (
                "annotations_without_words"
            ),
            "expected": 0,
            "actual": len(
                annotations_without_words
            ),
            "valid": (
                not annotations_without_words
            ),
        },
        {
            "control": (
                "severe_annotation_overlaps"
            ),
            "expected": 0,
            "actual": len(
                severe_annotation_overlaps
            ),
            "valid": (
                not severe_annotation_overlaps
            ),
        },
        {
            "control": (
                "missing_extracted_values"
            ),
            "expected": 0,
            "actual": len(
                missing_extracted_values
            ),
            "valid": (
                not missing_extracted_values
            ),
        },
        {
            "control": (
                "minimum_field_font_size"
            ),
            "expected": ">=6.5",
            "actual": round(
                minimum_field_font_size,
                4,
            ),
            "valid": (
                minimum_field_font_size
                >= MINIMUM_FIELD_FONT_SIZE
            ),
        },
        {
            "control": (
                "font_adjustment_ratio"
            ),
            "expected": "<=0.20",
            "actual": round(
                font_adjustment_ratio,
                4,
            ),
            "valid": (
                font_adjustment_ratio
                <= MAXIMUM_FONT_ADJUSTMENT_RATIO
            ),
        },
        {
            "control": (
                "raster_content_ratio"
            ),
            "expected": "0.01–0.40",
            "actual": round(
                raster_content_ratio,
                4,
            ),
            "valid": (
                MINIMUM_RASTER_CONTENT_RATIO
                <= raster_content_ratio
                <= MAXIMUM_RASTER_CONTENT_RATIO
            ),
        },
        {
            "control": (
                "raster_contrast"
            ),
            "expected": ">=10.0",
            "actual": round(
                raster_contrast,
                4,
            ),
            "valid": (
                raster_contrast
                >= MINIMUM_RASTER_CONTRAST
            ),
        },
    ]

    for check in checks:
        check["status"] = (
            "VALID"
            if check["valid"]
            else "INVALID"
        )

    failed_checks = [
        check["control"]
        for check in checks
        if not check["valid"]
    ]

    qa_status = (
        "PASSED"
        if not failed_checks
        else "FAILED"
    )

    return {
        "schema_version": (
            BATCH_QA_SCHEMA_VERSION
        ),
        "dataset_id": (
            "SYNTHETIC-INVOICE-V1"
        ),
        "document_id": document_id,
        "canonical_invoice_id": (
            canonical_id
        ),
        "template_id": template_id,
        "split": render_record["split"],
        "language": (
            render_record["language"]
        ),
        "currency": (
            render_record["currency"]
        ),
        "status": qa_status,
        "manual_visual_review": (
            "PENDING"
        ),
        "metrics": {
            "auditor_version": (
                BATCH_QA_AUDITOR_VERSION
            ),
            "page_count": (
                pdf_page_count
            ),
            "annotation_count": len(
                parsed_annotations
            ),
            "expected_annotation_count": (
                expected_annotation_count
            ),
            "annotation_parse_failure_count": len(
                annotation_parse_failures
            ),
            "duplicate_annotation_count": (
                duplicate_annotation_count
            ),
            "missing_required_annotation_count": len(
                missing_required_annotations
            ),
            "invalid_bounding_box_count": len(
                invalid_bounding_boxes
            ),
            "normalization_mismatch_count": len(
                normalization_mismatches
            ),
            "annotation_without_words_count": len(
                annotations_without_words
            ),
            "severe_overlap_count": len(
                severe_annotation_overlaps
            ),
            "missing_extracted_value_count": len(
                missing_extracted_values
            ),
            "minimum_field_font_size": round(
                minimum_field_font_size,
                6,
            ),
            "font_adjustment_count": (
                font_adjustment_count
            ),
            "font_adjustment_ratio": round(
                font_adjustment_ratio,
                6,
            ),
            "raster_content_ratio": round(
                raster_content_ratio,
                6,
            ),
            "raster_contrast": round(
                raster_contrast,
                6,
            ),
            "image_width_pixels": (
                image_width_pixels
            ),
            "image_height_pixels": (
                image_height_pixels
            ),
        },
        "failures": {
            "failed_checks": (
                failed_checks
            ),
            "checksum_mismatches": (
                checksum_mismatches
            ),
            "annotation_parse_failures": (
                annotation_parse_failures
            ),
            "missing_required_annotations": (
                missing_required_annotations
            ),
            "invalid_bounding_boxes": (
                invalid_bounding_boxes
            ),
            "normalization_mismatches": (
                normalization_mismatches
            ),
            "annotations_without_words": (
                annotations_without_words
            ),
            "severe_annotation_overlaps": (
                severe_annotation_overlaps
            ),
            "missing_extracted_values": (
                missing_extracted_values
            ),
        },
        "checks": [
            {
                "control": check[
                    "control"
                ],
                "expected": check[
                    "expected"
                ],
                "actual": check[
                    "actual"
                ],
                "status": check[
                    "status"
                ],
            }
            for check in checks
        ],
        "checksums_sha256": (
            current_checksums
        ),
        "artifact_paths": {
            artifact_name: str(
                artifact_path
            )
            for artifact_name, artifact_path
            in artifact_paths.items()
        },
    }


# ================================================================
# Status definisi
# ================================================================

BATCH_QA_ENGINE_READY = True

print(
    "✅ Cell 75A siap — mesin audit teknis "
    "batch berhasil didefinisikan."
)

print(
    "Belum ada QA report yang dibuat atau diubah."
)

print(
    "Lanjutkan ke Cell 75B untuk menjalankan "
    "audit terhadap 200 dokumen."
)

In [ ]:
# ================================================================
# CELL 75B — FINAL
# Menjalankan dan menyimpan audit teknis 200 dokumen
# ================================================================

from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import hashlib
import json
import os
import time

import pandas as pd


# ================================================================
# Konfigurasi eksekusi
# ================================================================

BATCH_QA_EXPECTED_DOCUMENTS = 200
BATCH_QA_EXPECTED_PER_TEMPLATE = 20

# QA report PASSED dengan versi auditor dan checksum yang sama
# dapat digunakan kembali saat cell dijalankan ulang.
BATCH_QA_RESUME_EXISTING = True


# ================================================================
# Validasi mesin audit
# ================================================================

if (
    "BATCH_QA_ENGINE_READY"
    not in globals()
    or not BATCH_QA_ENGINE_READY
):
    raise RuntimeError(
        "Mesin audit belum tersedia. "
        "Jalankan Cell 75A terlebih dahulu."
    )

if len(BATCH_RENDER_RECORDS) != 200:
    raise RuntimeError(
        "BATCH_RENDER_RECORDS harus berisi "
        "tepat 200 record."
    )

presentation_jsonl_path = Path(
    PRESENTATION_PAYLOADS_JSONL_PATH
)

if not presentation_jsonl_path.is_file():
    raise FileNotFoundError(
        "Presentation payload JSONL "
        "tidak ditemukan: "
        f"{presentation_jsonl_path}"
    )


# ================================================================
# Helper penulisan atomik
# ================================================================

def batch_qa_write_json_atomically(
    output_path,
    payload,
):
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        output_path.parent
        / f".{output_path.name}.tmp"
    )

    temporary_path.unlink(
        missing_ok=True
    )

    try:
        with temporary_path.open(
            mode="w",
            encoding="utf-8",
        ) as output_file:
            json.dump(
                payload,
                output_file,
                ensure_ascii=False,
                indent=2,
                allow_nan=False,
            )

            output_file.write("\n")

        os.replace(
            temporary_path,
            output_path,
        )

    except Exception:
        temporary_path.unlink(
            missing_ok=True
        )
        raise


def batch_qa_write_jsonl_atomically(
    output_path,
    records,
):
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        output_path.parent
        / f".{output_path.name}.tmp"
    )

    temporary_path.unlink(
        missing_ok=True
    )

    try:
        with temporary_path.open(
            mode="w",
            encoding="utf-8",
            newline="\n",
        ) as output_file:
            for record in records:
                output_file.write(
                    json.dumps(
                        record,
                        ensure_ascii=False,
                        separators=(",", ":"),
                        allow_nan=False,
                    )
                    + "\n"
                )

        os.replace(
            temporary_path,
            output_path,
        )

    except Exception:
        temporary_path.unlink(
            missing_ok=True
        )
        raise


def batch_qa_write_csv_atomically(
    output_path,
    dataframe,
):
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        output_path.parent
        / f".{output_path.name}.tmp"
    )

    temporary_path.unlink(
        missing_ok=True
    )

    try:
        dataframe.to_csv(
            temporary_path,
            index=False,
            encoding="utf-8",
            lineterminator="\n",
        )

        os.replace(
            temporary_path,
            output_path,
        )

    except Exception:
        temporary_path.unlink(
            missing_ok=True
        )
        raise


# ================================================================
# Muat presentation payload
# ================================================================

presentation_payloads = {}

with presentation_jsonl_path.open(
    mode="r",
    encoding="utf-8",
) as presentation_file:
    for line_number, line in enumerate(
        presentation_file,
        start=1,
    ):
        clean_line = line.strip()

        if not clean_line:
            continue

        try:
            payload = json.loads(
                clean_line
            )
        except json.JSONDecodeError as error:
            raise RuntimeError(
                "Presentation JSONL tidak valid "
                f"pada baris {line_number}."
            ) from error

        canonical_id = payload.get(
            "canonical_invoice_id"
        )

        if not isinstance(
            canonical_id,
            str,
        ) or not canonical_id.strip():
            raise RuntimeError(
                "canonical_invoice_id tidak valid "
                "pada presentation payload "
                f"baris {line_number}."
            )

        if canonical_id in presentation_payloads:
            raise RuntimeError(
                "Canonical ID presentation duplikat: "
                f"{canonical_id}"
            )

        presentation_payloads[
            canonical_id
        ] = payload


if len(presentation_payloads) != 200:
    raise RuntimeError(
        "Presentation payload harus "
        "berjumlah 200. "
        f"Actual: {len(presentation_payloads)}."
    )

if (
    set(presentation_payloads)
    != set(BATCH_RENDER_RECORDS)
):
    raise RuntimeError(
        "Presentation payload dan batch render index "
        "tidak memiliki canonical ID yang sama."
    )


# ================================================================
# Urutan audit
# ================================================================

ordered_render_records = sorted(
    BATCH_RENDER_RECORDS.values(),
    key=lambda record: int(
        record["sequence_number"]
    ),
)

sequence_numbers = [
    int(record["sequence_number"])
    for record in ordered_render_records
]

if sequence_numbers != list(
    range(1, 201)
):
    raise RuntimeError(
        "Sequence number batch harus "
        "berurutan dari 1 sampai 200."
    )


# ================================================================
# Helper recovery QA report
# ================================================================

def existing_qa_report_is_reusable(
    qa_report,
    render_record,
):
    if not isinstance(
        qa_report,
        dict,
    ):
        return False

    if qa_report.get("status") != "PASSED":
        return False

    if (
        qa_report.get(
            "canonical_invoice_id"
        )
        != render_record.get(
            "canonical_invoice_id"
        )
    ):
        return False

    if (
        qa_report.get("document_id")
        != render_record.get("document_id")
    ):
        return False

    if (
        qa_report.get("template_id")
        != render_record.get("template_id")
    ):
        return False

    metrics = qa_report.get(
        "metrics",
        {},
    )

    if (
        metrics.get("auditor_version")
        != BATCH_QA_AUDITOR_VERSION
    ):
        return False

    recorded_checksums = (
        qa_report.get(
            "checksums_sha256",
            {},
        )
    )

    expected_checksums = {
        artifact_name: (
            render_record[
                "artifacts"
            ][artifact_name]["sha256"]
        )
        for artifact_name
        in (
            "pdf",
            "preview",
            "ground_truth",
        )
    }

    if recorded_checksums != expected_checksums:
        return False

    return True


# ================================================================
# Mulai audit
# ================================================================

BATCH_QA_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

qa_started_utc = datetime.now(
    timezone.utc
)

qa_started_monotonic = (
    time.monotonic()
)

qa_reports = []
qa_index_records = []
qa_summary_rows = []
execution_errors = []

resumed_document_count = 0
audited_document_count = 0

print(
    "Memulai audit teknis 200 dokumen..."
)

print(
    f"QA root: {BATCH_QA_ROOT}"
)

print()


for position, render_record in enumerate(
    ordered_render_records,
    start=1,
):
    canonical_id = (
        render_record[
            "canonical_invoice_id"
        ]
    )

    document_id = (
        render_record["document_id"]
    )

    template_id = (
        render_record["template_id"]
    )

    split_name = (
        render_record["split"]
    )

    render_payload = (
        presentation_payloads[
            canonical_id
        ]
    )

    qa_report_directory = (
        BATCH_QA_ROOT
        / split_name
        / template_id
    )

    qa_report_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    qa_report_path = (
        qa_report_directory
        / (
            f"{document_id}"
            "_qa_report.json"
        )
    )

    qa_report = None
    report_was_resumed = False

    if (
        BATCH_QA_RESUME_EXISTING
        and qa_report_path.is_file()
    ):
        try:
            with qa_report_path.open(
                mode="r",
                encoding="utf-8",
            ) as existing_report_file:
                existing_report = json.load(
                    existing_report_file
                )

            if existing_qa_report_is_reusable(
                existing_report,
                render_record,
            ):
                qa_report = existing_report
                report_was_resumed = True
                resumed_document_count += 1

        except Exception:
            qa_report = None
            report_was_resumed = False

    if qa_report is None:
        try:
            qa_report = (
                batch_qa_audit_document(
                    render_record=(
                        render_record
                    ),
                    render_payload=(
                        render_payload
                    ),
                )
            )

            batch_qa_write_json_atomically(
                qa_report_path,
                qa_report,
            )

            # Pastikan report dapat dibuka kembali.
            with qa_report_path.open(
                mode="r",
                encoding="utf-8",
            ) as saved_report_file:
                reloaded_qa_report = json.load(
                    saved_report_file
                )

            if (
                reloaded_qa_report
                != qa_report
            ):
                raise RuntimeError(
                    "Round-trip QA report "
                    "tidak sama."
                )

            audited_document_count += 1

        except Exception as error:
            qa_report = {
                "schema_version": (
                    BATCH_QA_SCHEMA_VERSION
                ),
                "dataset_id": (
                    "SYNTHETIC-INVOICE-V1"
                ),
                "document_id": (
                    document_id
                ),
                "canonical_invoice_id": (
                    canonical_id
                ),
                "template_id": (
                    template_id
                ),
                "split": split_name,
                "language": (
                    render_record[
                        "language"
                    ]
                ),
                "currency": (
                    render_record[
                        "currency"
                    ]
                ),
                "status": "ERROR",
                "manual_visual_review": (
                    "PENDING"
                ),
                "metrics": {
                    "auditor_version": (
                        BATCH_QA_AUDITOR_VERSION
                    ),
                },
                "failures": {
                    "execution_error": {
                        "error_type": (
                            type(error).__name__
                        ),
                        "error": str(error),
                    }
                },
                "checks": [],
                "checksums_sha256": {},
                "artifact_paths": {},
            }

            execution_errors.append(
                {
                    "canonical_invoice_id": (
                        canonical_id
                    ),
                    "document_id": (
                        document_id
                    ),
                    "template_id": (
                        template_id
                    ),
                    "error_type": (
                        type(error).__name__
                    ),
                    "error": str(error),
                }
            )

            batch_qa_write_json_atomically(
                qa_report_path,
                qa_report,
            )

    qa_reports.append(
        qa_report
    )

    metrics = qa_report.get(
        "metrics",
        {},
    )

    failures = qa_report.get(
        "failures",
        {},
    )

    failed_checks = failures.get(
        "failed_checks",
        [],
    )

    qa_report_checksum = (
        batch_qa_sha256_file(
            qa_report_path
        )
    )

    qa_index_records.append(
        {
            "sequence_number": (
                render_record[
                    "sequence_number"
                ]
            ),
            "canonical_invoice_id": (
                canonical_id
            ),
            "document_id": (
                document_id
            ),
            "template_id": (
                template_id
            ),
            "split": split_name,
            "language": (
                render_record[
                    "language"
                ]
            ),
            "currency": (
                render_record[
                    "currency"
                ]
            ),
            "technical_status": (
                qa_report["status"]
            ),
            "manual_visual_review": (
                qa_report[
                    "manual_visual_review"
                ]
            ),
            "qa_report_relative_path": str(
                qa_report_path.relative_to(
                    BATCH_DATASET_ROOT
                )
            ),
            "qa_report_size_bytes": (
                qa_report_path
                .stat()
                .st_size
            ),
            "qa_report_sha256": (
                qa_report_checksum
            ),
            "resumed": (
                report_was_resumed
            ),
        }
    )

    qa_summary_rows.append(
        {
            "sequence_number": (
                render_record[
                    "sequence_number"
                ]
            ),
            "canonical_invoice_id": (
                canonical_id
            ),
            "document_id": (
                document_id
            ),
            "template_id": (
                template_id
            ),
            "split": split_name,
            "language": (
                render_record[
                    "language"
                ]
            ),
            "currency": (
                render_record[
                    "currency"
                ]
            ),
            "technical_status": (
                qa_report["status"]
            ),
            "manual_visual_review": (
                qa_report[
                    "manual_visual_review"
                ]
            ),
            "page_count": (
                metrics.get(
                    "page_count"
                )
            ),
            "annotation_count": (
                metrics.get(
                    "annotation_count"
                )
            ),
            "expected_annotation_count": (
                metrics.get(
                    "expected_annotation_count"
                )
            ),
            "minimum_field_font_size": (
                metrics.get(
                    "minimum_field_font_size"
                )
            ),
            "font_adjustment_count": (
                metrics.get(
                    "font_adjustment_count"
                )
            ),
            "font_adjustment_ratio": (
                metrics.get(
                    "font_adjustment_ratio"
                )
            ),
            "raster_content_ratio": (
                metrics.get(
                    "raster_content_ratio"
                )
            ),
            "raster_contrast": (
                metrics.get(
                    "raster_contrast"
                )
            ),
            "failed_check_count": (
                len(failed_checks)
                if isinstance(
                    failed_checks,
                    list,
                )
                else None
            ),
            "resumed": (
                report_was_resumed
            ),
        }
    )

    if (
        position % 10 == 0
        or position
        == BATCH_QA_EXPECTED_DOCUMENTS
    ):
        passed_count = sum(
            report.get("status")
            == "PASSED"
            for report in qa_reports
        )

        failed_count = sum(
            report.get("status")
            == "FAILED"
            for report in qa_reports
        )

        error_count = sum(
            report.get("status")
            == "ERROR"
            for report in qa_reports
        )

        print(
            f"[{position:03d}/"
            f"{BATCH_QA_EXPECTED_DOCUMENTS}] "
            f"passed={passed_count}, "
            f"failed={failed_count}, "
            f"error={error_count}, "
            f"resumed="
            f"{resumed_document_count}"
        )


# ================================================================
# Validasi jumlah hasil audit
# ================================================================

if len(qa_reports) != 200:
    raise RuntimeError(
        "Jumlah QA report dalam memori "
        "bukan 200."
    )

if len(qa_index_records) != 200:
    raise RuntimeError(
        "Jumlah QA index record "
        "bukan 200."
    )

if len(qa_summary_rows) != 200:
    raise RuntimeError(
        "Jumlah QA summary row "
        "bukan 200."
    )


# ================================================================
# DataFrame ringkasan
# ================================================================

qa_summary_frame = pd.DataFrame(
    qa_summary_rows
).sort_values(
    "sequence_number"
).reset_index(
    drop=True
)

technical_status_counts = Counter(
    qa_summary_frame[
        "technical_status"
    ]
)

technical_passed_count = int(
    technical_status_counts.get(
        "PASSED",
        0,
    )
)

technical_failed_count = int(
    technical_status_counts.get(
        "FAILED",
        0,
    )
)

technical_error_count = int(
    technical_status_counts.get(
        "ERROR",
        0,
    )
)


# ================================================================
# Ringkasan per template
# ================================================================

template_summary_rows = []

for template_id in sorted(
    qa_summary_frame[
        "template_id"
    ].unique()
):
    template_frame = (
        qa_summary_frame[
            qa_summary_frame[
                "template_id"
            ]
            == template_id
        ]
    )

    template_passed_count = int(
        (
            template_frame[
                "technical_status"
            ]
            == "PASSED"
        ).sum()
    )

    template_failed_count = int(
        (
            template_frame[
                "technical_status"
            ]
            == "FAILED"
        ).sum()
    )

    template_error_count = int(
        (
            template_frame[
                "technical_status"
            ]
            == "ERROR"
        ).sum()
    )

    numeric_minimum_fonts = (
        pd.to_numeric(
            template_frame[
                "minimum_field_font_size"
            ],
            errors="coerce",
        )
    )

    numeric_adjustment_ratios = (
        pd.to_numeric(
            template_frame[
                "font_adjustment_ratio"
            ],
            errors="coerce",
        )
    )

    numeric_content_ratios = (
        pd.to_numeric(
            template_frame[
                "raster_content_ratio"
            ],
            errors="coerce",
        )
    )

    numeric_contrasts = (
        pd.to_numeric(
            template_frame[
                "raster_contrast"
            ],
            errors="coerce",
        )
    )

    document_count = len(
        template_frame
    )

    template_valid = all(
        [
            document_count
            == BATCH_QA_EXPECTED_PER_TEMPLATE,
            template_passed_count
            == BATCH_QA_EXPECTED_PER_TEMPLATE,
            template_failed_count == 0,
            template_error_count == 0,
        ]
    )

    template_summary_rows.append(
        {
            "template_id": template_id,
            "documents": document_count,
            "passed": (
                template_passed_count
            ),
            "failed": (
                template_failed_count
            ),
            "errors": (
                template_error_count
            ),
            "minimum_font": (
                round(
                    float(
                        numeric_minimum_fonts.min()
                    ),
                    4,
                )
                if numeric_minimum_fonts.notna().any()
                else None
            ),
            "maximum_font_adjustment_ratio": (
                round(
                    float(
                        numeric_adjustment_ratios.max()
                    ),
                    4,
                )
                if numeric_adjustment_ratios.notna().any()
                else None
            ),
            "minimum_raster_content": (
                round(
                    float(
                        numeric_content_ratios.min()
                    ),
                    4,
                )
                if numeric_content_ratios.notna().any()
                else None
            ),
            "maximum_raster_content": (
                round(
                    float(
                        numeric_content_ratios.max()
                    ),
                    4,
                )
                if numeric_content_ratios.notna().any()
                else None
            ),
            "minimum_raster_contrast": (
                round(
                    float(
                        numeric_contrasts.min()
                    ),
                    4,
                )
                if numeric_contrasts.notna().any()
                else None
            ),
            "status": (
                "VALID"
                if template_valid
                else "INVALID"
            ),
        }
    )

template_qa_summary = pd.DataFrame(
    template_summary_rows
)

display(
    template_qa_summary
)


# ================================================================
# Global controls
# ================================================================

actual_qa_report_paths = sorted(
    BATCH_QA_ROOT.rglob(
        "*_qa_report.json"
    )
)

duplicate_qa_document_ids = (
    len(
        qa_summary_frame[
            "document_id"
        ]
    )
    - qa_summary_frame[
        "document_id"
    ].nunique()
)

global_controls = [
    {
        "control": "qa_report_count",
        "expected": 200,
        "actual": len(
            actual_qa_report_paths
        ),
    },
    {
        "control": "qa_index_records",
        "expected": 200,
        "actual": len(
            qa_index_records
        ),
    },
    {
        "control": "qa_summary_rows",
        "expected": 200,
        "actual": len(
            qa_summary_frame
        ),
    },
    {
        "control": "technical_passed",
        "expected": 200,
        "actual": (
            technical_passed_count
        ),
    },
    {
        "control": "technical_failed",
        "expected": 0,
        "actual": (
            technical_failed_count
        ),
    },
    {
        "control": "technical_errors",
        "expected": 0,
        "actual": (
            technical_error_count
        ),
    },
    {
        "control": "valid_templates",
        "expected": 10,
        "actual": int(
            (
                template_qa_summary[
                    "status"
                ]
                == "VALID"
            ).sum()
        ),
    },
    {
        "control": "duplicate_qa_document_ids",
        "expected": 0,
        "actual": (
            duplicate_qa_document_ids
        ),
    },
]

for control in global_controls:
    control["status"] = (
        "VALID"
        if control["actual"]
        == control["expected"]
        else "INVALID"
    )

global_qa_summary = pd.DataFrame(
    global_controls
)

display(
    global_qa_summary
)


# ================================================================
# Simpan summary CSV
# ================================================================

batch_qa_write_csv_atomically(
    BATCH_QA_SUMMARY_PATH,
    qa_summary_frame,
)


# ================================================================
# Simpan QA index
# ================================================================

batch_qa_write_jsonl_atomically(
    BATCH_QA_INDEX_PATH,
    qa_index_records,
)

qa_index_checksum = (
    batch_qa_sha256_file(
        BATCH_QA_INDEX_PATH
    )
)

qa_summary_checksum = (
    batch_qa_sha256_file(
        BATCH_QA_SUMMARY_PATH
    )
)


# ================================================================
# Status akhir
# ================================================================

batch_technical_status = (
    "PASSED"
    if all(
        global_qa_summary[
            "status"
        ]
        == "VALID"
    )
    else "FAILED"
)


# ================================================================
# Manifest QA batch
# ================================================================

qa_completed_utc = datetime.now(
    timezone.utc
)

qa_duration_seconds = (
    time.monotonic()
    - qa_started_monotonic
)

batch_qa_manifest = {
    "schema_version": (
        BATCH_QA_SCHEMA_VERSION
    ),
    "dataset_id": (
        "SYNTHETIC-INVOICE-V1"
    ),
    "dataset_version": "1.0.0",
    "build_id": Path(
        BUILD_RUN_ROOT
    ).name,
    "artifact_type": (
        "batch_technical_qa"
    ),
    "auditor_version": (
        BATCH_QA_AUDITOR_VERSION
    ),
    "technical_status": (
        batch_technical_status
    ),
    "manual_visual_review": (
        "PENDING"
    ),
    "started_utc": (
        qa_started_utc.isoformat()
    ),
    "completed_utc": (
        qa_completed_utc.isoformat()
    ),
    "duration_seconds": round(
        qa_duration_seconds,
        3,
    ),
    "document_count": len(
        qa_reports
    ),
    "technical_status_counts": {
        "passed": (
            technical_passed_count
        ),
        "failed": (
            technical_failed_count
        ),
        "error": (
            technical_error_count
        ),
    },
    "execution": {
        "newly_audited_documents": (
            audited_document_count
        ),
        "resumed_documents": (
            resumed_document_count
        ),
        "resume_enabled": (
            BATCH_QA_RESUME_EXISTING
        ),
    },
    "thresholds": {
        "minimum_field_font_size": (
            MINIMUM_FIELD_FONT_SIZE
        ),
        "maximum_font_adjustment_ratio": (
            MAXIMUM_FONT_ADJUSTMENT_RATIO
        ),
        "minimum_raster_content_ratio": (
            MINIMUM_RASTER_CONTENT_RATIO
        ),
        "maximum_raster_content_ratio": (
            MAXIMUM_RASTER_CONTENT_RATIO
        ),
        "minimum_raster_contrast": (
            MINIMUM_RASTER_CONTRAST
        ),
        "severe_overlap_threshold": (
            SEVERE_OVERLAP_THRESHOLD
        ),
    },
    "artifacts": {
        "qa_root": str(
            BATCH_QA_ROOT
        ),
        "qa_report_count": len(
            actual_qa_report_paths
        ),
        "qa_index": {
            "path": str(
                BATCH_QA_INDEX_PATH
            ),
            "record_count": len(
                qa_index_records
            ),
            "size_bytes": (
                BATCH_QA_INDEX_PATH
                .stat()
                .st_size
            ),
            "sha256": (
                qa_index_checksum
            ),
        },
        "qa_summary_csv": {
            "path": str(
                BATCH_QA_SUMMARY_PATH
            ),
            "row_count": len(
                qa_summary_frame
            ),
            "size_bytes": (
                BATCH_QA_SUMMARY_PATH
                .stat()
                .st_size
            ),
            "sha256": (
                qa_summary_checksum
            ),
        },
    },
}

batch_qa_write_json_atomically(
    BATCH_QA_MANIFEST_PATH,
    batch_qa_manifest,
)


# ================================================================
# Verifikasi manifest
# ================================================================

with BATCH_QA_MANIFEST_PATH.open(
    mode="r",
    encoding="utf-8",
) as manifest_file:
    reloaded_qa_manifest = json.load(
        manifest_file
    )

if (
    reloaded_qa_manifest[
        "document_count"
    ]
    != 200
):
    raise RuntimeError(
        "Document count QA manifest "
        "tidak sesuai."
    )

if (
    reloaded_qa_manifest[
        "artifacts"
    ]["qa_index"]["sha256"]
    != batch_qa_sha256_file(
        BATCH_QA_INDEX_PATH
    )
):
    raise RuntimeError(
        "Checksum QA index tidak sesuai."
    )

if (
    reloaded_qa_manifest[
        "artifacts"
    ]["qa_summary_csv"]["sha256"]
    != batch_qa_sha256_file(
        BATCH_QA_SUMMARY_PATH
    )
):
    raise RuntimeError(
        "Checksum QA summary tidak sesuai."
    )


# ================================================================
# Tampilkan kegagalan jika ada
# ================================================================

failed_document_rows = (
    qa_summary_frame[
        qa_summary_frame[
            "technical_status"
        ]
        != "PASSED"
    ]
)

if not failed_document_rows.empty:
    print()
    print(
        "DOKUMEN YANG TIDAK LULUS"
    )

    display(
        failed_document_rows
    )


# ================================================================
# Variabel recovery
# ================================================================

BATCH_QA_REPORTS = {
    report[
        "canonical_invoice_id"
    ]: report
    for report in qa_reports
}

BATCH_QA_STATUS = (
    batch_technical_status
)

BATCH_QA_READY = (
    batch_technical_status
    == "PASSED"
)


# ================================================================
# Ringkasan akhir
# ================================================================

print(
    f"QA root             : "
    f"{BATCH_QA_ROOT}"
)

print(
    f"QA index            : "
    f"{BATCH_QA_INDEX_PATH}"
)

print(
    f"QA summary          : "
    f"{BATCH_QA_SUMMARY_PATH}"
)

print(
    f"QA manifest         : "
    f"{BATCH_QA_MANIFEST_PATH}"
)

print(
    f"Documents checked   : "
    f"{len(qa_reports)}"
)

print(
    f"Technical passed    : "
    f"{technical_passed_count}"
)

print(
    f"Technical failed    : "
    f"{technical_failed_count}"
)

print(
    f"Execution errors    : "
    f"{technical_error_count}"
)

print(
    f"Newly audited       : "
    f"{audited_document_count}"
)

print(
    f"Recovered reports   : "
    f"{resumed_document_count}"
)

print(
    f"QA duration         : "
    f"{qa_duration_seconds:.2f} seconds"
)

print(
    f"Technical status    : "
    f"{BATCH_QA_STATUS}"
)

print(
    "Manual visual review: PENDING"
)

print()


if BATCH_QA_STATUS != "PASSED":
    print(
        "❌ BATCH TECHNICAL QA FAILED — "
        "periksa dokumen yang tidak lulus."
    )

    raise RuntimeError(
        "Audit teknis batch belum lulus."
    )


print(
    "✅ BATCH TECHNICAL QA PASSED — "
    "seluruh 200 dokumen lulus audit teknis."
)

print(
    "Review visual batch masih PENDING "
    "dan akan dilakukan pada tahap berikutnya."
)

In [ ]:
# ================================================================
# CELL 76A — READ-ONLY
# Pemilihan sampel visual batch secara deterministik
# ================================================================

from collections import Counter
from pathlib import Path

import hashlib
import json

import pandas as pd
from IPython.display import display
from PIL import (
    Image as PILImage,
    ImageDraw,
    ImageFont,
)


# ================================================================
# Konfigurasi
# ================================================================

VISUAL_SAMPLES_PER_TEMPLATE = 3
EXPECTED_TEMPLATE_COUNT = 10
EXPECTED_SAMPLE_COUNT = 30

BATCH_QA_ROOT = (
    BATCH_DATASET_ROOT
    / "qa"
)

BATCH_QA_INDEX_PATH = (
    BUILD_MANIFESTS_ROOT
    / "batch_qa_index.jsonl"
)

BATCH_QA_SUMMARY_PATH = (
    BUILD_MANIFESTS_ROOT
    / "batch_qa_summary.csv"
)

BATCH_QA_MANIFEST_PATH = (
    BUILD_MANIFESTS_ROOT
    / "batch_qa_manifest.json"
)


# ================================================================
# Helper
# ================================================================

def visual_review_sha256(
    file_path,
):
    digest = hashlib.sha256()

    with Path(file_path).open(
        "rb"
    ) as binary_file:
        for chunk in iter(
            lambda: binary_file.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def longest_display_money_length(
    payload,
):
    money_values = []

    for item in payload.get(
        "items",
        [],
    ):
        money_values.extend(
            [
                str(
                    item.get(
                        "unit_price",
                        "",
                    )
                ),
                str(
                    item.get(
                        "line_total",
                        "",
                    )
                ),
            ]
        )

    for field_name in (
        "subtotal",
        "tax",
        "discount",
        "total",
    ):
        money_values.append(
            str(
                payload.get(
                    "financials",
                    {},
                ).get(
                    field_name,
                    {},
                ).get(
                    "display_value",
                    "",
                )
            )
        )

    return max(
        (
            len(value)
            for value in money_values
        ),
        default=0,
    )


def maximum_description_length(
    payload,
):
    return max(
        (
            len(
                str(
                    item.get(
                        "description",
                        "",
                    )
                )
            )
            for item in payload.get(
                "items",
                []
            )
        ),
        default=0,
    )


def combined_party_text_length(
    payload,
):
    total_length = 0

    for party_type in (
        "vendor",
        "buyer",
    ):
        party = payload.get(
            party_type,
            {},
        )

        party_values = [
            party.get("name", ""),
            " ".join(
                party.get(
                    "address_lines",
                    [],
                )
            ),
            party.get("email", ""),
            party.get("phone", ""),
            party.get(
                "tax_identifier_display",
                "",
            ),
        ]

        total_length += sum(
            len(str(value))
            for value in party_values
        )

    return total_length


def resolve_preview_path(
    render_record,
):
    relative_path = Path(
        render_record[
            "artifacts"
        ]["preview"][
            "relative_path"
        ]
    )

    if relative_path.is_absolute():
        raise RuntimeError(
            "Preview path pada index "
            "harus bersifat relatif."
        )

    dataset_root_resolved = (
        Path(BATCH_DATASET_ROOT)
        .resolve()
    )

    preview_path = (
        Path(BATCH_DATASET_ROOT)
        / relative_path
    ).resolve()

    try:
        preview_path.relative_to(
            dataset_root_resolved
        )
    except ValueError as error:
        raise RuntimeError(
            "Preview path keluar dari "
            "BATCH_DATASET_ROOT."
        ) from error

    return preview_path


# ================================================================
# Validasi artifact QA
# ================================================================

required_paths = [
    BATCH_QA_ROOT,
    BATCH_QA_INDEX_PATH,
    BATCH_QA_SUMMARY_PATH,
    BATCH_QA_MANIFEST_PATH,
    PRESENTATION_PAYLOADS_JSONL_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not Path(path).exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Artifact QA batch belum lengkap: "
        f"{missing_paths}"
    )


# ================================================================
# Muat dan validasi QA manifest
# ================================================================

with BATCH_QA_MANIFEST_PATH.open(
    mode="r",
    encoding="utf-8",
) as manifest_file:
    batch_qa_manifest = json.load(
        manifest_file
    )

if (
    batch_qa_manifest.get(
        "technical_status"
    )
    != "PASSED"
):
    raise RuntimeError(
        "Technical status batch bukan PASSED."
    )

if (
    batch_qa_manifest.get(
        "document_count"
    )
    != 200
):
    raise RuntimeError(
        "QA manifest tidak memiliki "
        "tepat 200 dokumen."
    )

recorded_qa_index_checksum = (
    batch_qa_manifest[
        "artifacts"
    ]["qa_index"]["sha256"]
)

current_qa_index_checksum = (
    visual_review_sha256(
        BATCH_QA_INDEX_PATH
    )
)

if (
    recorded_qa_index_checksum
    != current_qa_index_checksum
):
    raise RuntimeError(
        "Checksum QA index tidak sesuai."
    )

recorded_qa_summary_checksum = (
    batch_qa_manifest[
        "artifacts"
    ]["qa_summary_csv"][
        "sha256"
    ]
)

current_qa_summary_checksum = (
    visual_review_sha256(
        BATCH_QA_SUMMARY_PATH
    )
)

if (
    recorded_qa_summary_checksum
    != current_qa_summary_checksum
):
    raise RuntimeError(
        "Checksum QA summary tidak sesuai."
    )


# ================================================================
# Muat QA summary
# ================================================================

qa_summary = pd.read_csv(
    BATCH_QA_SUMMARY_PATH
)

if len(qa_summary) != 200:
    raise RuntimeError(
        "QA summary harus memiliki "
        "tepat 200 baris."
    )

if not (
    qa_summary[
        "technical_status"
    ]
    == "PASSED"
).all():
    raise RuntimeError(
        "Masih ada dokumen yang tidak "
        "lulus audit teknis."
    )


# ================================================================
# Muat presentation payload
# ================================================================

presentation_payloads = {}

with Path(
    PRESENTATION_PAYLOADS_JSONL_PATH
).open(
    mode="r",
    encoding="utf-8",
) as presentation_file:
    for line_number, line in enumerate(
        presentation_file,
        start=1,
    ):
        clean_line = line.strip()

        if not clean_line:
            continue

        try:
            payload = json.loads(
                clean_line
            )
        except json.JSONDecodeError as error:
            raise RuntimeError(
                "Presentation JSONL tidak valid "
                f"pada baris {line_number}."
            ) from error

        canonical_id = payload.get(
            "canonical_invoice_id"
        )

        if canonical_id in presentation_payloads:
            raise RuntimeError(
                "Canonical ID presentation duplikat: "
                f"{canonical_id}"
            )

        presentation_payloads[
            canonical_id
        ] = payload

if len(presentation_payloads) != 200:
    raise RuntimeError(
        "Presentation payload harus "
        "berjumlah 200."
    )

if (
    set(presentation_payloads)
    != set(BATCH_RENDER_RECORDS)
):
    raise RuntimeError(
        "Presentation payload tidak sinkron "
        "dengan batch render index."
    )


# ================================================================
# Bangun kandidat visual
# ================================================================

candidate_rows = []

qa_by_canonical_id = (
    qa_summary
    .set_index(
        "canonical_invoice_id"
    )
    .to_dict(
        orient="index"
    )
)

for canonical_id in sorted(
    presentation_payloads
):
    payload = presentation_payloads[
        canonical_id
    ]

    render_record = (
        BATCH_RENDER_RECORDS[
            canonical_id
        ]
    )

    qa_record = qa_by_canonical_id[
        canonical_id
    ]

    item_count = len(
        payload["items"]
    )

    description_length = (
        maximum_description_length(
            payload
        )
    )

    party_text_length = (
        combined_party_text_length(
            payload
        )
    )

    money_length = (
        longest_display_money_length(
            payload
        )
    )

    # Skor hanya untuk pemilihan sampel,
    # bukan skor kualitas dokumen.
    text_risk_score = (
        item_count * 10
        + description_length * 2
        + party_text_length
        + money_length * 3
    )

    preview_path = resolve_preview_path(
        render_record
    )

    if not preview_path.is_file():
        raise FileNotFoundError(
            "Preview tidak ditemukan: "
            f"{preview_path}"
        )

    candidate_rows.append(
        {
            "canonical_invoice_id": (
                canonical_id
            ),
            "document_id": (
                payload["document_id"]
            ),
            "template_id": (
                payload["template_id"]
            ),
            "split": payload["split"],
            "language": (
                payload["language"]
            ),
            "currency": (
                payload["currency"]
            ),
            "item_count": item_count,
            "maximum_description_length": (
                description_length
            ),
            "party_text_length": (
                party_text_length
            ),
            "maximum_money_length": (
                money_length
            ),
            "text_risk_score": (
                text_risk_score
            ),
            "minimum_field_font_size": (
                float(
                    qa_record[
                        "minimum_field_font_size"
                    ]
                )
            ),
            "font_adjustment_ratio": (
                float(
                    qa_record[
                        "font_adjustment_ratio"
                    ]
                )
            ),
            "raster_content_ratio": (
                float(
                    qa_record[
                        "raster_content_ratio"
                    ]
                )
            ),
            "raster_contrast": (
                float(
                    qa_record[
                        "raster_contrast"
                    ]
                )
            ),
            "preview_path": str(
                preview_path
            ),
        }
    )

candidate_frame = pd.DataFrame(
    candidate_rows
)

if (
    candidate_frame[
        "template_id"
    ].nunique()
    != EXPECTED_TEMPLATE_COUNT
):
    raise RuntimeError(
        "Kandidat tidak mencakup "
        "10 template."
    )


# ================================================================
# Pilih tiga sampel per template
# ================================================================

selected_samples = []

for template_id in sorted(
    candidate_frame[
        "template_id"
    ].unique()
):
    template_candidates = (
        candidate_frame[
            candidate_frame[
                "template_id"
            ]
            == template_id
        ]
        .copy()
    )

    if len(template_candidates) != 20:
        raise RuntimeError(
            f"{template_id} harus memiliki "
            "20 kandidat."
        )

    selected_ids = set()

    def add_selected_sample(
        selected_row,
        selection_reason,
    ):
        canonical_id = (
            selected_row[
                "canonical_invoice_id"
            ]
        )

        if canonical_id in selected_ids:
            return False

        selected_ids.add(
            canonical_id
        )

        selected_record = (
            selected_row.to_dict()
        )

        selected_record[
            "selection_reason"
        ] = selection_reason

        selected_samples.append(
            selected_record
        )

        return True

    # Sampel dengan jumlah item paling banyak.
    maximum_item_row = (
        template_candidates
        .sort_values(
            [
                "item_count",
                "text_risk_score",
                "canonical_invoice_id",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .iloc[0]
    )

    add_selected_sample(
        maximum_item_row,
        "maximum_items",
    )

    # Sampel dengan jumlah item paling sedikit.
    minimum_item_candidates = (
        template_candidates[
            ~template_candidates[
                "canonical_invoice_id"
            ].isin(selected_ids)
        ]
        .sort_values(
            [
                "item_count",
                "text_risk_score",
                "canonical_invoice_id",
            ],
            ascending=[
                True,
                False,
                True,
            ],
        )
    )

    add_selected_sample(
        minimum_item_candidates.iloc[0],
        "minimum_items",
    )

    # Sampel ketiga memastikan dua bahasa ikut diperiksa.
    selected_template_rows = [
        row
        for row in selected_samples
        if row["template_id"]
        == template_id
    ]

    selected_languages = {
        row["language"]
        for row in selected_template_rows
    }

    missing_languages = (
        {"id", "en"}
        - selected_languages
    )

    remaining_candidates = (
        template_candidates[
            ~template_candidates[
                "canonical_invoice_id"
            ].isin(selected_ids)
        ]
    )

    if missing_languages:
        language_candidates = (
            remaining_candidates[
                remaining_candidates[
                    "language"
                ].isin(
                    missing_languages
                )
            ]
        )

        if not language_candidates.empty:
            remaining_candidates = (
                language_candidates
            )

        third_reason = (
            "language_coverage_high_risk"
        )

    else:
        third_reason = (
            "highest_text_risk"
        )

    third_row = (
        remaining_candidates
        .sort_values(
            [
                "text_risk_score",
                "font_adjustment_ratio",
                "canonical_invoice_id",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .iloc[0]
    )

    add_selected_sample(
        third_row,
        third_reason,
    )


visual_sample_frame = (
    pd.DataFrame(
        selected_samples
    )
    .sort_values(
        [
            "template_id",
            "selection_reason",
        ]
    )
    .reset_index(drop=True)
)


# ================================================================
# Validasi sampel
# ================================================================

if len(visual_sample_frame) != 30:
    raise RuntimeError(
        "Jumlah sampel visual harus 30. "
        f"Actual: {len(visual_sample_frame)}."
    )

sample_counts = (
    visual_sample_frame[
        "template_id"
    ].value_counts()
)

if not (
    sample_counts == 3
).all():
    raise RuntimeError(
        "Setiap template harus memiliki "
        "tepat tiga sampel."
    )

language_counts_per_template = (
    visual_sample_frame
    .groupby("template_id")[
        "language"
    ]
    .nunique()
)

if not (
    language_counts_per_template
    >= 2
).all():
    raise RuntimeError(
        "Setiap template harus memiliki "
        "sampel bahasa Indonesia dan Inggris."
    )

if (
    visual_sample_frame[
        "canonical_invoice_id"
    ].nunique()
    != 30
):
    raise RuntimeError(
        "Sampel visual mengandung "
        "canonical ID duplikat."
    )


# ================================================================
# Tampilkan tabel pemilihan
# ================================================================

sample_display_columns = [
    "template_id",
    "document_id",
    "selection_reason",
    "language",
    "currency",
    "item_count",
    "maximum_description_length",
    "maximum_money_length",
    "minimum_field_font_size",
    "font_adjustment_ratio",
]

display(
    visual_sample_frame[
        sample_display_columns
    ]
)


# ================================================================
# Membuat contact sheet hanya di memori
# ================================================================

CONTACT_THUMBNAIL_WIDTH = 340
CONTACT_THUMBNAIL_HEIGHT = 440
CONTACT_CARD_WIDTH = 365
CONTACT_CARD_HEIGHT = 520
CONTACT_GAP = 18
CONTACT_MARGIN = 20
CONTACT_LABEL_HEIGHT = 64

try:
    contact_font = ImageFont.truetype(
        "/usr/share/fonts/truetype/"
        "dejavu/DejaVuSans.ttf",
        12,
    )

    contact_title_font = (
        ImageFont.truetype(
            "/usr/share/fonts/truetype/"
            "dejavu/DejaVuSans-Bold.ttf",
            16,
        )
    )

except OSError:
    contact_font = (
        ImageFont.load_default()
    )

    contact_title_font = (
        ImageFont.load_default()
    )

try:
    resampling_method = (
        PILImage.Resampling.LANCZOS
    )
except AttributeError:
    resampling_method = (
        PILImage.LANCZOS
    )


VISUAL_CONTACT_SHEETS = {}

for template_id in sorted(
    visual_sample_frame[
        "template_id"
    ].unique()
):
    template_samples = (
        visual_sample_frame[
            visual_sample_frame[
                "template_id"
            ]
            == template_id
        ]
        .reset_index(drop=True)
    )

    canvas_width = (
        CONTACT_MARGIN * 2
        + CONTACT_CARD_WIDTH * 3
        + CONTACT_GAP * 2
    )

    canvas_height = (
        CONTACT_MARGIN * 2
        + 35
        + CONTACT_CARD_HEIGHT
    )

    contact_sheet = PILImage.new(
        mode="RGB",
        size=(
            canvas_width,
            canvas_height,
        ),
        color=(238, 241, 245),
    )

    draw = ImageDraw.Draw(
        contact_sheet
    )

    draw.text(
        (
            CONTACT_MARGIN,
            CONTACT_MARGIN,
        ),
        (
            f"{template_id} - "
            "Manual Visual Review"
        ),
        fill=(25, 35, 50),
        font=contact_title_font,
    )

    for sample_index, sample in (
        template_samples.iterrows()
    ):
        card_x = (
            CONTACT_MARGIN
            + sample_index
            * (
                CONTACT_CARD_WIDTH
                + CONTACT_GAP
            )
        )

        card_y = (
            CONTACT_MARGIN + 35
        )

        draw.rectangle(
            (
                card_x,
                card_y,
                card_x
                + CONTACT_CARD_WIDTH,
                card_y
                + CONTACT_CARD_HEIGHT,
            ),
            fill=(255, 255, 255),
            outline=(185, 193, 204),
            width=1,
        )

        label_line_1 = (
            f"{sample['document_id']} | "
            f"{sample['selection_reason']}"
        )

        label_line_2 = (
            f"{sample['language']} | "
            f"{sample['currency']} | "
            f"items={sample['item_count']} | "
            f"font="
            f"{sample['minimum_field_font_size']:.2f}"
        )

        draw.text(
            (
                card_x + 10,
                card_y + 8,
            ),
            label_line_1,
            fill=(25, 35, 50),
            font=contact_font,
        )

        draw.text(
            (
                card_x + 10,
                card_y + 29,
            ),
            label_line_2,
            fill=(75, 84, 96),
            font=contact_font,
        )

        with PILImage.open(
            sample["preview_path"]
        ) as preview_image:
            preview_copy = (
                preview_image
                .convert("RGB")
            )

            preview_copy.thumbnail(
                (
                    CONTACT_THUMBNAIL_WIDTH,
                    CONTACT_THUMBNAIL_HEIGHT,
                ),
                resampling_method,
            )

        image_x = (
            card_x
            + (
                CONTACT_CARD_WIDTH
                - preview_copy.width
            )
            // 2
        )

        image_y = (
            card_y
            + CONTACT_LABEL_HEIGHT
        )

        contact_sheet.paste(
            preview_copy,
            (
                image_x,
                image_y,
            ),
        )

    VISUAL_CONTACT_SHEETS[
        template_id
    ] = contact_sheet

    display(
        contact_sheet
    )


# ================================================================
# Variabel runtime
# ================================================================

VISUAL_SAMPLE_SELECTION = (
    visual_sample_frame.copy()
)

VISUAL_SAMPLE_RECORDS = (
    visual_sample_frame.to_dict(
        orient="records"
    )
)

VISUAL_REVIEW_SAMPLE_COUNT = len(
    VISUAL_SAMPLE_RECORDS
)

VISUAL_REVIEW_SELECTION_READY = True


# ================================================================
# Ringkasan
# ================================================================

print(
    f"Technical QA status : "
    f"{batch_qa_manifest['technical_status']}"
)

print(
    f"Templates covered   : "
    f"{visual_sample_frame['template_id'].nunique()}"
)

print(
    f"Samples selected    : "
    f"{VISUAL_REVIEW_SAMPLE_COUNT}"
)

print(
    "Samples/template    : "
    f"{VISUAL_SAMPLES_PER_TEMPLATE}"
)

print(
    "Languages/template  : id + en"
)

print(
    "Selection method    : "
    "minimum items, maximum items, "
    "and high-risk language coverage"
)

print(
    "Artifact writes     : 0"
)

print()

print(
    "✅ SAMPEL VISUAL SIAP — "
    "30 preview representatif berhasil "
    "dipilih dan ditampilkan."
)

print(
    "Periksa seluruh 10 contact sheet "
    "sebelum status manual dicatat."
)

In [ ]:
# ================================================================
# CELL 76B — FINAL
# Mencatat hasil manual visual review batch
# ================================================================

from datetime import datetime, timezone
from pathlib import Path

import hashlib
import json
import os

import pandas as pd


# ================================================================
# Validasi state visual review
# ================================================================

REQUIRED_VISUAL_OBJECTS = [
    "VISUAL_REVIEW_SELECTION_READY",
    "VISUAL_SAMPLE_RECORDS",
    "VISUAL_SAMPLE_SELECTION",
    "BATCH_RENDER_RECORDS",
    "BATCH_DATASET_ROOT",
    "BUILD_MANIFESTS_ROOT",
    "BATCH_QA_MANIFEST_PATH",
]

missing_visual_objects = [
    object_name
    for object_name in REQUIRED_VISUAL_OBJECTS
    if object_name not in globals()
]

if missing_visual_objects:
    raise RuntimeError(
        "State Cell 76A belum lengkap: "
        f"{missing_visual_objects}. "
        "Jalankan kembali Cell 76A."
    )

if not VISUAL_REVIEW_SELECTION_READY:
    raise RuntimeError(
        "Pemilihan sampel visual belum siap."
    )

if len(VISUAL_SAMPLE_RECORDS) != 30:
    raise RuntimeError(
        "Manual review harus memiliki "
        "tepat 30 sampel."
    )


# ================================================================
# Lokasi artifact review
# ================================================================

VISUAL_REVIEW_SELECTION_PATH = (
    Path(BUILD_MANIFESTS_ROOT)
    / "batch_visual_review_selection.csv"
)

VISUAL_REVIEW_MANIFEST_PATH = (
    Path(BUILD_MANIFESTS_ROOT)
    / "batch_visual_review_manifest.json"
)

batch_qa_manifest_path = Path(
    BATCH_QA_MANIFEST_PATH
)


# ================================================================
# Helper
# ================================================================

def visual_record_sha256(
    file_path,
):
    digest = hashlib.sha256()

    with Path(file_path).open(
        "rb"
    ) as binary_file:
        for chunk in iter(
            lambda: binary_file.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def visual_record_write_json(
    output_path,
    payload,
):
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        output_path.parent
        / f".{output_path.name}.tmp"
    )

    temporary_path.unlink(
        missing_ok=True
    )

    try:
        with temporary_path.open(
            mode="w",
            encoding="utf-8",
        ) as output_file:
            json.dump(
                payload,
                output_file,
                ensure_ascii=False,
                indent=2,
                allow_nan=False,
            )

            output_file.write("\n")

        os.replace(
            temporary_path,
            output_path,
        )

    except Exception:
        temporary_path.unlink(
            missing_ok=True
        )
        raise


def visual_record_write_csv(
    output_path,
    dataframe,
):
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        output_path.parent
        / f".{output_path.name}.tmp"
    )

    temporary_path.unlink(
        missing_ok=True
    )

    try:
        dataframe.to_csv(
            temporary_path,
            index=False,
            encoding="utf-8",
            lineterminator="\n",
        )

        os.replace(
            temporary_path,
            output_path,
        )

    except Exception:
        temporary_path.unlink(
            missing_ok=True
        )
        raise


# ================================================================
# Muat QA manifest
# ================================================================

if not batch_qa_manifest_path.is_file():
    raise FileNotFoundError(
        "Batch QA manifest tidak ditemukan: "
        f"{batch_qa_manifest_path}"
    )

with batch_qa_manifest_path.open(
    mode="r",
    encoding="utf-8",
) as qa_manifest_file:
    batch_qa_manifest = json.load(
        qa_manifest_file
    )

if (
    batch_qa_manifest.get(
        "technical_status"
    )
    != "PASSED"
):
    raise RuntimeError(
        "Technical QA harus PASSED sebelum "
        "manual visual review dicatat."
    )

if (
    batch_qa_manifest.get(
        "document_count"
    )
    != 200
):
    raise RuntimeError(
        "Batch QA manifest harus memiliki "
        "200 dokumen."
    )

technical_qa_manifest_checksum_before = (
    visual_record_sha256(
        batch_qa_manifest_path
    )
)


# ================================================================
# Validasi dan ikat sampel ke checksum preview
# ================================================================

review_sample_records = []
preview_checksum_mismatches = []

for sample in VISUAL_SAMPLE_RECORDS:
    canonical_id = (
        sample[
            "canonical_invoice_id"
        ]
    )

    if canonical_id not in (
        BATCH_RENDER_RECORDS
    ):
        raise KeyError(
            "Canonical ID sampel tidak ditemukan "
            "pada batch render index: "
            f"{canonical_id}"
        )

    render_record = (
        BATCH_RENDER_RECORDS[
            canonical_id
        ]
    )

    expected_preview_checksum = (
        render_record[
            "artifacts"
        ]["preview"]["sha256"]
    )

    preview_relative_path = Path(
        render_record[
            "artifacts"
        ]["preview"][
            "relative_path"
        ]
    )

    if preview_relative_path.is_absolute():
        raise RuntimeError(
            "Preview path pada batch index "
            "harus relatif."
        )

    preview_path = (
        Path(BATCH_DATASET_ROOT)
        / preview_relative_path
    )

    if not preview_path.is_file():
        raise FileNotFoundError(
            "Preview sampel tidak ditemukan: "
            f"{preview_path}"
        )

    current_preview_checksum = (
        visual_record_sha256(
            preview_path
        )
    )

    if (
        current_preview_checksum
        != expected_preview_checksum
    ):
        preview_checksum_mismatches.append(
            {
                "canonical_invoice_id": (
                    canonical_id
                ),
                "document_id": (
                    sample["document_id"]
                ),
                "expected": (
                    expected_preview_checksum
                ),
                "actual": (
                    current_preview_checksum
                ),
            }
        )

    review_sample_records.append(
        {
            "template_id": (
                sample["template_id"]
            ),
            "canonical_invoice_id": (
                canonical_id
            ),
            "document_id": (
                sample["document_id"]
            ),
            "selection_reason": (
                sample[
                    "selection_reason"
                ]
            ),
            "language": (
                sample["language"]
            ),
            "currency": (
                sample["currency"]
            ),
            "item_count": int(
                sample["item_count"]
            ),
            "maximum_description_length": int(
                sample[
                    "maximum_description_length"
                ]
            ),
            "maximum_money_length": int(
                sample[
                    "maximum_money_length"
                ]
            ),
            "minimum_field_font_size": float(
                sample[
                    "minimum_field_font_size"
                ]
            ),
            "font_adjustment_ratio": float(
                sample[
                    "font_adjustment_ratio"
                ]
            ),
            "preview_relative_path": str(
                preview_relative_path
            ),
            "preview_sha256": (
                current_preview_checksum
            ),
            "visual_status": "PASSED",
        }
    )

if preview_checksum_mismatches:
    raise RuntimeError(
        "Checksum preview sampel berubah: "
        f"{preview_checksum_mismatches}"
    )


# ================================================================
# Validasi cakupan sampel
# ================================================================

review_sample_frame = (
    pd.DataFrame(
        review_sample_records
    )
    .sort_values(
        [
            "template_id",
            "document_id",
        ]
    )
    .reset_index(drop=True)
)

template_sample_counts = (
    review_sample_frame[
        "template_id"
    ].value_counts()
)

language_coverage = (
    review_sample_frame
    .groupby("template_id")[
        "language"
    ]
    .nunique()
)

review_controls = [
    {
        "control": "sample_count",
        "expected": 30,
        "actual": len(
            review_sample_frame
        ),
    },
    {
        "control": "template_count",
        "expected": 10,
        "actual": (
            review_sample_frame[
                "template_id"
            ].nunique()
        ),
    },
    {
        "control": "samples_per_template",
        "expected": 3,
        "actual": (
            int(
                template_sample_counts.min()
            )
            if not template_sample_counts.empty
            else 0
        ),
    },
    {
        "control": "languages_per_template",
        "expected": 2,
        "actual": (
            int(
                language_coverage.min()
            )
            if not language_coverage.empty
            else 0
        ),
    },
    {
        "control": "unique_documents",
        "expected": 30,
        "actual": (
            review_sample_frame[
                "document_id"
            ].nunique()
        ),
    },
    {
        "control": "preview_checksum_mismatches",
        "expected": 0,
        "actual": len(
            preview_checksum_mismatches
        ),
    },
]

for control in review_controls:
    control["status"] = (
        "VALID"
        if control["actual"]
        == control["expected"]
        else "INVALID"
    )

review_control_summary = pd.DataFrame(
    review_controls
)

display(
    review_control_summary
)

if not all(
    review_control_summary["status"]
    == "VALID"
):
    raise RuntimeError(
        "Cakupan manual visual review "
        "belum valid."
    )


# ================================================================
# Simpan selection CSV
# ================================================================

visual_record_write_csv(
    VISUAL_REVIEW_SELECTION_PATH,
    review_sample_frame,
)

selection_checksum = (
    visual_record_sha256(
        VISUAL_REVIEW_SELECTION_PATH
    )
)


# ================================================================
# Buat visual review manifest
# ================================================================

review_recorded_utc = datetime.now(
    timezone.utc
).isoformat()

visual_review_manifest = {
    "schema_version": "1.0.0",
    "dataset_id": (
        "SYNTHETIC-INVOICE-V1"
    ),
    "dataset_version": "1.0.0",
    "build_id": Path(
        BUILD_RUN_ROOT
    ).name,
    "artifact_type": (
        "batch_manual_visual_review"
    ),
    "status": "PASSED",
    "review_scope": (
        "STRATIFIED_SAMPLE"
    ),
    "reviewed_document_count": 30,
    "total_document_count": 200,
    "template_count": 10,
    "samples_per_template": 3,
    "language_coverage_per_template": [
        "id",
        "en",
    ],
    "reviewer_confirmation": (
        "30 sampel visual rapi dan terbaca"
    ),
    "review_recorded_utc": (
        review_recorded_utc
    ),
    "selection_method": [
        "minimum_items",
        "maximum_items",
        "high_risk_language_coverage",
    ],
    "review_criteria": [
        "no_clipped_text",
        "no_overlapping_text",
        "table_and_totals_do_not_collide",
        "currency_and_numbers_are_readable",
        "indonesian_and_english_labels_are_correct",
        "synthetic_notice_is_visible",
        "content_remains_inside_page",
    ],
    "review_result": {
        "passed_samples": 30,
        "failed_samples": 0,
        "preview_checksum_mismatches": 0,
    },
    "source_technical_qa": {
        "technical_status": (
            batch_qa_manifest[
                "technical_status"
            ]
        ),
        "document_count": (
            batch_qa_manifest[
                "document_count"
            ]
        ),
        "manifest_path": str(
            batch_qa_manifest_path
        ),
        "manifest_sha256_before_visual_review": (
            technical_qa_manifest_checksum_before
        ),
    },
    "artifacts": {
        "selection_csv": {
            "path": str(
                VISUAL_REVIEW_SELECTION_PATH
            ),
            "row_count": len(
                review_sample_frame
            ),
            "size_bytes": (
                VISUAL_REVIEW_SELECTION_PATH
                .stat()
                .st_size
            ),
            "sha256": (
                selection_checksum
            ),
        }
    },
    "samples": (
        review_sample_records
    ),
}

visual_record_write_json(
    VISUAL_REVIEW_MANIFEST_PATH,
    visual_review_manifest,
)

visual_review_manifest_checksum = (
    visual_record_sha256(
        VISUAL_REVIEW_MANIFEST_PATH
    )
)


# ================================================================
# Perbarui batch QA manifest
# ================================================================

batch_qa_manifest[
    "manual_visual_review"
] = "PASSED"

batch_qa_manifest[
    "manual_visual_review_details"
] = {
    "scope": "STRATIFIED_SAMPLE",
    "status": "PASSED",
    "reviewed_document_count": 30,
    "total_document_count": 200,
    "template_count": 10,
    "samples_per_template": 3,
    "review_recorded_utc": (
        review_recorded_utc
    ),
    "visual_review_manifest": {
        "path": str(
            VISUAL_REVIEW_MANIFEST_PATH
        ),
        "size_bytes": (
            VISUAL_REVIEW_MANIFEST_PATH
            .stat()
            .st_size
        ),
        "sha256": (
            visual_review_manifest_checksum
        ),
    },
    "selection_csv": {
        "path": str(
            VISUAL_REVIEW_SELECTION_PATH
        ),
        "row_count": 30,
        "size_bytes": (
            VISUAL_REVIEW_SELECTION_PATH
            .stat()
            .st_size
        ),
        "sha256": (
            selection_checksum
        ),
    },
}

visual_record_write_json(
    batch_qa_manifest_path,
    batch_qa_manifest,
)


# ================================================================
# Verifikasi round-trip
# ================================================================

with VISUAL_REVIEW_MANIFEST_PATH.open(
    mode="r",
    encoding="utf-8",
) as visual_manifest_file:
    verified_visual_manifest = json.load(
        visual_manifest_file
    )

with batch_qa_manifest_path.open(
    mode="r",
    encoding="utf-8",
) as qa_manifest_file:
    verified_qa_manifest = json.load(
        qa_manifest_file
    )

if (
    verified_visual_manifest.get(
        "status"
    )
    != "PASSED"
):
    raise RuntimeError(
        "Status visual review manifest "
        "bukan PASSED."
    )

if (
    verified_visual_manifest.get(
        "review_scope"
    )
    != "STRATIFIED_SAMPLE"
):
    raise RuntimeError(
        "Scope visual review tidak sesuai."
    )

if (
    verified_qa_manifest.get(
        "technical_status"
    )
    != "PASSED"
):
    raise RuntimeError(
        "Technical status berubah."
    )

if (
    verified_qa_manifest.get(
        "manual_visual_review"
    )
    != "PASSED"
):
    raise RuntimeError(
        "Manual visual review belum "
        "tersimpan sebagai PASSED."
    )

recorded_visual_manifest_checksum = (
    verified_qa_manifest[
        "manual_visual_review_details"
    ]["visual_review_manifest"][
        "sha256"
    ]
)

if (
    recorded_visual_manifest_checksum
    != visual_record_sha256(
        VISUAL_REVIEW_MANIFEST_PATH
    )
):
    raise RuntimeError(
        "Checksum visual review manifest "
        "tidak sesuai."
    )

recorded_selection_checksum = (
    verified_qa_manifest[
        "manual_visual_review_details"
    ]["selection_csv"]["sha256"]
)

if (
    recorded_selection_checksum
    != visual_record_sha256(
        VISUAL_REVIEW_SELECTION_PATH
    )
):
    raise RuntimeError(
        "Checksum selection CSV "
        "tidak sesuai."
    )


# ================================================================
# Variabel runtime
# ================================================================

BATCH_VISUAL_REVIEW_STATUS = "PASSED"
BATCH_VISUAL_REVIEW_SCOPE = (
    "STRATIFIED_SAMPLE"
)
BATCH_VISUAL_REVIEW_READY = True


# ================================================================
# Ringkasan
# ================================================================

print(
    f"Selection CSV       : "
    f"{VISUAL_REVIEW_SELECTION_PATH}"
)

print(
    f"Visual manifest     : "
    f"{VISUAL_REVIEW_MANIFEST_PATH}"
)

print(
    f"QA manifest updated : "
    f"{batch_qa_manifest_path}"
)

print(
    f"Technical status    : "
    f"{verified_qa_manifest['technical_status']}"
)

print(
    f"Manual review       : "
    f"{verified_qa_manifest['manual_visual_review']}"
)

print(
    "Review scope        : "
    f"{BATCH_VISUAL_REVIEW_SCOPE}"
)

print(
    f"Reviewed samples    : "
    f"{len(review_sample_frame)}/200"
)

print(
    f"Template coverage   : "
    f"{review_sample_frame['template_id'].nunique()}/10"
)

print(
    f"Selection SHA-256   : "
    f"{selection_checksum}"
)

print(
    f"Visual SHA-256      : "
    f"{visual_review_manifest_checksum}"
)

print()

print(
    "✅ BATCH VISUAL REVIEW PASSED — "
    "30 sampel terstratifikasi dari 10 template "
    "telah dikonfirmasi rapi dan terbaca."
)

In [ ]:
# ================================================================
# CELL 77 — FINAL
# Audit integritas final seluruh dataset
# ================================================================

from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import hashlib
import json
import os

import pandas as pd


# ================================================================
# Konfigurasi
# ================================================================

FINAL_EXPECTED_DOCUMENTS = 200
FINAL_EXPECTED_TEMPLATES = 10
FINAL_EXPECTED_PER_TEMPLATE = 20
FINAL_EXPECTED_VISUAL_SAMPLES = 30

EXPECTED_LANGUAGE_DISTRIBUTION = {
    "en": 100,
    "id": 100,
}

EXPECTED_CURRENCY_DISTRIBUTION = {
    "EUR": 30,
    "GBP": 20,
    "IDR": 80,
    "USD": 70,
}

BATCH_RENDER_INDEX_PATH = (
    BUILD_MANIFESTS_ROOT
    / "batch_render_index.jsonl"
)

BATCH_RENDER_MANIFEST_PATH = (
    BUILD_MANIFESTS_ROOT
    / "batch_render_manifest.json"
)

BATCH_QA_INDEX_PATH = (
    BUILD_MANIFESTS_ROOT
    / "batch_qa_index.jsonl"
)

BATCH_QA_SUMMARY_PATH = (
    BUILD_MANIFESTS_ROOT
    / "batch_qa_summary.csv"
)

BATCH_QA_MANIFEST_PATH = (
    BUILD_MANIFESTS_ROOT
    / "batch_qa_manifest.json"
)

VISUAL_REVIEW_SELECTION_PATH = (
    BUILD_MANIFESTS_ROOT
    / "batch_visual_review_selection.csv"
)

VISUAL_REVIEW_MANIFEST_PATH = (
    BUILD_MANIFESTS_ROOT
    / "batch_visual_review_manifest.json"
)

FINAL_INTEGRITY_SUMMARY_PATH = (
    BUILD_MANIFESTS_ROOT
    / "final_integrity_summary.csv"
)

FINAL_INTEGRITY_REPORT_PATH = (
    BUILD_MANIFESTS_ROOT
    / "final_integrity_report.json"
)

FINAL_INTEGRITY_CHECKSUM_PATH = (
    BUILD_MANIFESTS_ROOT
    / "final_integrity_report.sha256"
)

BATCH_QA_ROOT = (
    BATCH_DATASET_ROOT
    / "qa"
)


# ================================================================
# Validasi state recovery
# ================================================================

REQUIRED_RUNTIME_OBJECTS = [
    "BUILD_RUN_ROOT",
    "BUILD_MANIFESTS_ROOT",
    "BATCH_DATASET_ROOT",
    "BATCH_PDF_ROOT",
    "BATCH_PREVIEW_ROOT",
    "BATCH_GROUND_TRUTH_ROOT",
    "CANONICAL_RECORDS_JSONL_PATH",
    "PRESENTATION_PAYLOADS_JSONL_PATH",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "State recovery belum lengkap: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali Cell 74C."
    )


# ================================================================
# Helper
# ================================================================

def final_sha256_file(
    file_path,
):
    digest = hashlib.sha256()

    with Path(file_path).open(
        "rb"
    ) as binary_file:
        for chunk in iter(
            lambda: binary_file.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def final_load_json(
    file_path,
):
    with Path(file_path).open(
        mode="r",
        encoding="utf-8",
    ) as input_file:
        return json.load(
            input_file
        )


def final_load_jsonl(
    file_path,
):
    records = []

    with Path(file_path).open(
        mode="r",
        encoding="utf-8",
    ) as input_file:
        for line_number, line in enumerate(
            input_file,
            start=1,
        ):
            clean_line = line.strip()

            if not clean_line:
                continue

            try:
                record = json.loads(
                    clean_line
                )
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    "JSONL tidak valid pada "
                    f"{file_path}, "
                    f"baris {line_number}."
                ) from error

            records.append(
                record
            )

    return records


def final_write_json_atomically(
    output_path,
    payload,
):
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        output_path.parent
        / f".{output_path.name}.tmp"
    )

    temporary_path.unlink(
        missing_ok=True
    )

    try:
        with temporary_path.open(
            mode="w",
            encoding="utf-8",
        ) as output_file:
            json.dump(
                payload,
                output_file,
                ensure_ascii=False,
                indent=2,
                allow_nan=False,
            )

            output_file.write("\n")

        os.replace(
            temporary_path,
            output_path,
        )

    except Exception:
        temporary_path.unlink(
            missing_ok=True
        )
        raise


def final_write_csv_atomically(
    output_path,
    dataframe,
):
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        output_path.parent
        / f".{output_path.name}.tmp"
    )

    temporary_path.unlink(
        missing_ok=True
    )

    try:
        dataframe.to_csv(
            temporary_path,
            index=False,
            encoding="utf-8",
            lineterminator="\n",
        )

        os.replace(
            temporary_path,
            output_path,
        )

    except Exception:
        temporary_path.unlink(
            missing_ok=True
        )
        raise


def final_write_text_atomically(
    output_path,
    text,
):
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        output_path.parent
        / f".{output_path.name}.tmp"
    )

    temporary_path.unlink(
        missing_ok=True
    )

    try:
        with temporary_path.open(
            mode="w",
            encoding="utf-8",
            newline="\n",
        ) as output_file:
            output_file.write(text)

        os.replace(
            temporary_path,
            output_path,
        )

    except Exception:
        temporary_path.unlink(
            missing_ok=True
        )
        raise


def final_resolve_dataset_path(
    relative_path,
):
    relative_path = Path(
        relative_path
    )

    if relative_path.is_absolute():
        raise RuntimeError(
            "Artifact path harus relatif."
        )

    dataset_root_resolved = (
        Path(BATCH_DATASET_ROOT)
        .resolve()
    )

    artifact_path = (
        Path(BATCH_DATASET_ROOT)
        / relative_path
    ).resolve()

    try:
        artifact_path.relative_to(
            dataset_root_resolved
        )
    except ValueError as error:
        raise RuntimeError(
            "Artifact path keluar dari "
            "BATCH_DATASET_ROOT."
        ) from error

    return artifact_path


def final_id_set(
    records,
    key,
):
    return {
        record[key]
        for record in records
    }


# ================================================================
# Validasi file yang wajib tersedia
# ================================================================

required_files = [
    Path(CANONICAL_RECORDS_JSONL_PATH),
    Path(PRESENTATION_PAYLOADS_JSONL_PATH),
    BATCH_RENDER_INDEX_PATH,
    BATCH_RENDER_MANIFEST_PATH,
    BATCH_QA_INDEX_PATH,
    BATCH_QA_SUMMARY_PATH,
    BATCH_QA_MANIFEST_PATH,
    VISUAL_REVIEW_SELECTION_PATH,
    VISUAL_REVIEW_MANIFEST_PATH,
]

missing_required_files = [
    str(file_path)
    for file_path in required_files
    if not file_path.is_file()
]

if missing_required_files:
    raise FileNotFoundError(
        "Artifact final belum lengkap: "
        f"{missing_required_files}"
    )


# ================================================================
# Muat seluruh manifest dan index
# ================================================================

canonical_records = final_load_jsonl(
    CANONICAL_RECORDS_JSONL_PATH
)

presentation_records = final_load_jsonl(
    PRESENTATION_PAYLOADS_JSONL_PATH
)

render_index_records = final_load_jsonl(
    BATCH_RENDER_INDEX_PATH
)

qa_index_records = final_load_jsonl(
    BATCH_QA_INDEX_PATH
)

render_manifest = final_load_json(
    BATCH_RENDER_MANIFEST_PATH
)

qa_manifest = final_load_json(
    BATCH_QA_MANIFEST_PATH
)

visual_manifest = final_load_json(
    VISUAL_REVIEW_MANIFEST_PATH
)

qa_summary = pd.read_csv(
    BATCH_QA_SUMMARY_PATH
)

visual_selection = pd.read_csv(
    VISUAL_REVIEW_SELECTION_PATH
)


# ================================================================
# Pemeriksaan checksum manifest turunan
# ================================================================

manifest_checksum_mismatches = []

recorded_render_index_checksum = (
    render_manifest[
        "artifacts"
    ]["batch_index"]["sha256"]
)

actual_render_index_checksum = (
    final_sha256_file(
        BATCH_RENDER_INDEX_PATH
    )
)

if (
    recorded_render_index_checksum
    != actual_render_index_checksum
):
    manifest_checksum_mismatches.append(
        {
            "artifact": (
                "batch_render_index"
            ),
            "expected": (
                recorded_render_index_checksum
            ),
            "actual": (
                actual_render_index_checksum
            ),
        }
    )

recorded_qa_index_checksum = (
    qa_manifest[
        "artifacts"
    ]["qa_index"]["sha256"]
)

actual_qa_index_checksum = (
    final_sha256_file(
        BATCH_QA_INDEX_PATH
    )
)

if (
    recorded_qa_index_checksum
    != actual_qa_index_checksum
):
    manifest_checksum_mismatches.append(
        {
            "artifact": "batch_qa_index",
            "expected": (
                recorded_qa_index_checksum
            ),
            "actual": (
                actual_qa_index_checksum
            ),
        }
    )

recorded_qa_summary_checksum = (
    qa_manifest[
        "artifacts"
    ]["qa_summary_csv"]["sha256"]
)

actual_qa_summary_checksum = (
    final_sha256_file(
        BATCH_QA_SUMMARY_PATH
    )
)

if (
    recorded_qa_summary_checksum
    != actual_qa_summary_checksum
):
    manifest_checksum_mismatches.append(
        {
            "artifact": (
                "batch_qa_summary"
            ),
            "expected": (
                recorded_qa_summary_checksum
            ),
            "actual": (
                actual_qa_summary_checksum
            ),
        }
    )

visual_details = (
    qa_manifest[
        "manual_visual_review_details"
    ]
)

recorded_visual_manifest_checksum = (
    visual_details[
        "visual_review_manifest"
    ]["sha256"]
)

actual_visual_manifest_checksum = (
    final_sha256_file(
        VISUAL_REVIEW_MANIFEST_PATH
    )
)

if (
    recorded_visual_manifest_checksum
    != actual_visual_manifest_checksum
):
    manifest_checksum_mismatches.append(
        {
            "artifact": (
                "batch_visual_review_manifest"
            ),
            "expected": (
                recorded_visual_manifest_checksum
            ),
            "actual": (
                actual_visual_manifest_checksum
            ),
        }
    )

recorded_selection_checksum = (
    visual_details[
        "selection_csv"
    ]["sha256"]
)

actual_selection_checksum = (
    final_sha256_file(
        VISUAL_REVIEW_SELECTION_PATH
    )
)

if (
    recorded_selection_checksum
    != actual_selection_checksum
):
    manifest_checksum_mismatches.append(
        {
            "artifact": (
                "batch_visual_review_selection"
            ),
            "expected": (
                recorded_selection_checksum
            ),
            "actual": (
                actual_selection_checksum
            ),
        }
    )


# ================================================================
# Pemeriksaan jumlah dan ID
# ================================================================

canonical_ids = final_id_set(
    canonical_records,
    "canonical_invoice_id",
)

presentation_ids = final_id_set(
    presentation_records,
    "canonical_invoice_id",
)

render_index_ids = final_id_set(
    render_index_records,
    "canonical_invoice_id",
)

qa_index_ids = final_id_set(
    qa_index_records,
    "canonical_invoice_id",
)

visual_sample_ids = set(
    visual_selection[
        "canonical_invoice_id"
    ].tolist()
)

all_main_id_sets_match = (
    canonical_ids
    == presentation_ids
    == render_index_ids
    == qa_index_ids
)

visual_ids_are_subset = (
    visual_sample_ids
    .issubset(render_index_ids)
)

render_document_ids = [
    record["document_id"]
    for record in render_index_records
]

qa_document_ids = [
    record["document_id"]
    for record in qa_index_records
]


# ================================================================
# Distribusi dataset
# ================================================================

template_distribution = dict(
    sorted(
        Counter(
            record["template_id"]
            for record
            in render_index_records
        ).items()
    )
)

language_distribution = dict(
    sorted(
        Counter(
            record["language"]
            for record
            in render_index_records
        ).items()
    )
)

currency_distribution = dict(
    sorted(
        Counter(
            record["currency"]
            for record
            in render_index_records
        ).items()
    )
)

split_distribution = dict(
    sorted(
        Counter(
            record["split"]
            for record
            in render_index_records
        ).items()
    )
)

recorded_split_distribution = {
    str(key): int(value)
    for key, value
    in render_manifest[
        "distributions"
    ]["split"].items()
}

template_distribution_valid = all(
    template_distribution.get(
        f"TPL-{template_number:02d}"
    )
    == FINAL_EXPECTED_PER_TEMPLATE
    for template_number in range(
        1,
        FINAL_EXPECTED_TEMPLATES + 1,
    )
)


# ================================================================
# Audit setiap artifact dan QA report
# ================================================================

missing_artifacts = []
empty_artifacts = []
artifact_checksum_mismatches = []
qa_report_checksum_mismatches = []
qa_report_identity_mismatches = []
qa_report_status_failures = []
qa_artifact_checksum_mismatches = []
annotation_formula_mismatches = []

qa_index_by_canonical_id = {
    record["canonical_invoice_id"]: record
    for record in qa_index_records
}

print(
    "Memeriksa integritas 200 dokumen..."
)

print()


for position, render_record in enumerate(
    sorted(
        render_index_records,
        key=lambda record: int(
            record["sequence_number"]
        ),
    ),
    start=1,
):
    canonical_id = (
        render_record[
            "canonical_invoice_id"
        ]
    )

    document_id = (
        render_record["document_id"]
    )

    item_count = int(
        render_record["item_count"]
    )

    expected_annotation_count = (
        19 + 4 * item_count
    )

    actual_annotation_count = int(
        render_record[
            "annotation_count"
        ]
    )

    if (
        actual_annotation_count
        != expected_annotation_count
    ):
        annotation_formula_mismatches.append(
            {
                "canonical_invoice_id": (
                    canonical_id
                ),
                "document_id": (
                    document_id
                ),
                "item_count": item_count,
                "expected_annotations": (
                    expected_annotation_count
                ),
                "actual_annotations": (
                    actual_annotation_count
                ),
            }
        )

    current_artifact_checksums = {}

    for artifact_name in (
        "pdf",
        "preview",
        "ground_truth",
    ):
        artifact_record = (
            render_record[
                "artifacts"
            ][artifact_name]
        )

        artifact_path = (
            final_resolve_dataset_path(
                artifact_record[
                    "relative_path"
                ]
            )
        )

        if not artifact_path.is_file():
            missing_artifacts.append(
                {
                    "canonical_invoice_id": (
                        canonical_id
                    ),
                    "artifact": (
                        artifact_name
                    ),
                    "path": str(
                        artifact_path
                    ),
                }
            )
            continue

        if artifact_path.stat().st_size == 0:
            empty_artifacts.append(
                {
                    "canonical_invoice_id": (
                        canonical_id
                    ),
                    "artifact": (
                        artifact_name
                    ),
                    "path": str(
                        artifact_path
                    ),
                }
            )
            continue

        current_checksum = (
            final_sha256_file(
                artifact_path
            )
        )

        current_artifact_checksums[
            artifact_name
        ] = current_checksum

        expected_checksum = (
            artifact_record["sha256"]
        )

        if (
            current_checksum
            != expected_checksum
        ):
            artifact_checksum_mismatches.append(
                {
                    "canonical_invoice_id": (
                        canonical_id
                    ),
                    "document_id": (
                        document_id
                    ),
                    "artifact": (
                        artifact_name
                    ),
                    "expected": (
                        expected_checksum
                    ),
                    "actual": (
                        current_checksum
                    ),
                }
            )

    qa_index_record = (
        qa_index_by_canonical_id.get(
            canonical_id
        )
    )

    if qa_index_record is None:
        qa_report_status_failures.append(
            {
                "canonical_invoice_id": (
                    canonical_id
                ),
                "reason": (
                    "missing_qa_index_record"
                ),
            }
        )
        continue

    qa_report_path = (
        final_resolve_dataset_path(
            qa_index_record[
                "qa_report_relative_path"
            ]
        )
    )

    if not qa_report_path.is_file():
        missing_artifacts.append(
            {
                "canonical_invoice_id": (
                    canonical_id
                ),
                "artifact": "qa_report",
                "path": str(
                    qa_report_path
                ),
            }
        )
        continue

    current_qa_report_checksum = (
        final_sha256_file(
            qa_report_path
        )
    )

    expected_qa_report_checksum = (
        qa_index_record[
            "qa_report_sha256"
        ]
    )

    if (
        current_qa_report_checksum
        != expected_qa_report_checksum
    ):
        qa_report_checksum_mismatches.append(
            {
                "canonical_invoice_id": (
                    canonical_id
                ),
                "expected": (
                    expected_qa_report_checksum
                ),
                "actual": (
                    current_qa_report_checksum
                ),
            }
        )

    try:
        qa_report = final_load_json(
            qa_report_path
        )
    except Exception as error:
        qa_report_status_failures.append(
            {
                "canonical_invoice_id": (
                    canonical_id
                ),
                "reason": (
                    "unreadable_qa_report"
                ),
                "error": str(error),
            }
        )
        continue

    qa_identity_valid = all(
        [
            qa_report.get(
                "canonical_invoice_id"
            )
            == canonical_id,
            qa_report.get(
                "document_id"
            )
            == document_id,
            qa_report.get(
                "template_id"
            )
            == render_record[
                "template_id"
            ],
        ]
    )

    if not qa_identity_valid:
        qa_report_identity_mismatches.append(
            {
                "canonical_invoice_id": (
                    canonical_id
                ),
                "qa_document_id": (
                    qa_report.get(
                        "document_id"
                    )
                ),
                "expected_document_id": (
                    document_id
                ),
            }
        )

    if qa_report.get("status") != "PASSED":
        qa_report_status_failures.append(
            {
                "canonical_invoice_id": (
                    canonical_id
                ),
                "status": (
                    qa_report.get("status")
                ),
            }
        )

    qa_recorded_artifact_checksums = (
        qa_report.get(
            "checksums_sha256",
            {},
        )
    )

    if (
        qa_recorded_artifact_checksums
        != current_artifact_checksums
    ):
        qa_artifact_checksum_mismatches.append(
            {
                "canonical_invoice_id": (
                    canonical_id
                ),
                "expected": (
                    current_artifact_checksums
                ),
                "qa_report": (
                    qa_recorded_artifact_checksums
                ),
            }
        )

    if position % 25 == 0:
        print(
            f"[{position:03d}/200] "
            f"artifact_mismatch="
            f"{len(artifact_checksum_mismatches)}, "
            f"qa_mismatch="
            f"{len(qa_report_checksum_mismatches)}, "
            f"missing="
            f"{len(missing_artifacts)}"
        )


# ================================================================
# Validasi QA summary dan visual selection
# ================================================================

qa_summary_all_passed = bool(
    (
        qa_summary[
            "technical_status"
        ]
        == "PASSED"
    ).all()
)

qa_index_all_passed = all(
    record.get(
        "technical_status"
    )
    == "PASSED"
    for record in qa_index_records
)

visual_selection_all_passed = bool(
    (
        visual_selection[
            "visual_status"
        ]
        == "PASSED"
    ).all()
)

visual_template_count = int(
    visual_selection[
        "template_id"
    ].nunique()
)

visual_languages_per_template = (
    visual_selection
    .groupby("template_id")[
        "language"
    ]
    .nunique()
)

visual_language_coverage_valid = bool(
    (
        visual_languages_per_template
        == 2
    ).all()
)


# ================================================================
# Kontrol final
# ================================================================

final_controls = [
    {
        "control": (
            "render_manifest_status"
        ),
        "expected": "PASSED",
        "actual": (
            render_manifest.get(
                "render_status"
            )
        ),
    },
    {
        "control": (
            "qa_manifest_technical_status"
        ),
        "expected": "PASSED",
        "actual": (
            qa_manifest.get(
                "technical_status"
            )
        ),
    },
    {
        "control": (
            "qa_manifest_manual_review"
        ),
        "expected": "PASSED",
        "actual": (
            qa_manifest.get(
                "manual_visual_review"
            )
        ),
    },
    {
        "control": (
            "visual_manifest_status"
        ),
        "expected": "PASSED",
        "actual": (
            visual_manifest.get(
                "status"
            )
        ),
    },
    {
        "control": (
            "visual_review_scope"
        ),
        "expected": (
            "STRATIFIED_SAMPLE"
        ),
        "actual": (
            visual_manifest.get(
                "review_scope"
            )
        ),
    },
    {
        "control": "canonical_records",
        "expected": 200,
        "actual": len(
            canonical_records
        ),
    },
    {
        "control": (
            "presentation_records"
        ),
        "expected": 200,
        "actual": len(
            presentation_records
        ),
    },
    {
        "control": (
            "render_index_records"
        ),
        "expected": 200,
        "actual": len(
            render_index_records
        ),
    },
    {
        "control": "qa_index_records",
        "expected": 200,
        "actual": len(
            qa_index_records
        ),
    },
    {
        "control": "qa_summary_rows",
        "expected": 200,
        "actual": len(
            qa_summary
        ),
    },
    {
        "control": (
            "visual_sample_count"
        ),
        "expected": 30,
        "actual": len(
            visual_selection
        ),
    },
    {
        "control": (
            "visual_template_count"
        ),
        "expected": 10,
        "actual": (
            visual_template_count
        ),
    },
    {
        "control": (
            "visual_language_coverage"
        ),
        "expected": True,
        "actual": (
            visual_language_coverage_valid
        ),
    },
    {
        "control": (
            "main_id_sets_match"
        ),
        "expected": True,
        "actual": (
            all_main_id_sets_match
        ),
    },
    {
        "control": (
            "visual_ids_are_subset"
        ),
        "expected": True,
        "actual": (
            visual_ids_are_subset
        ),
    },
    {
        "control": (
            "unique_render_document_ids"
        ),
        "expected": 200,
        "actual": len(
            set(render_document_ids)
        ),
    },
    {
        "control": (
            "unique_qa_document_ids"
        ),
        "expected": 200,
        "actual": len(
            set(qa_document_ids)
        ),
    },
    {
        "control": (
            "valid_template_distribution"
        ),
        "expected": True,
        "actual": (
            template_distribution_valid
        ),
    },
    {
        "control": (
            "language_distribution"
        ),
        "expected": (
            EXPECTED_LANGUAGE_DISTRIBUTION
        ),
        "actual": (
            language_distribution
        ),
    },
    {
        "control": (
            "currency_distribution"
        ),
        "expected": (
            EXPECTED_CURRENCY_DISTRIBUTION
        ),
        "actual": (
            currency_distribution
        ),
    },
    {
        "control": (
            "split_distribution_matches_manifest"
        ),
        "expected": (
            recorded_split_distribution
        ),
        "actual": (
            split_distribution
        ),
    },
    {
        "control": (
            "manifest_checksum_mismatches"
        ),
        "expected": 0,
        "actual": len(
            manifest_checksum_mismatches
        ),
    },
    {
        "control": (
            "missing_artifacts"
        ),
        "expected": 0,
        "actual": len(
            missing_artifacts
        ),
    },
    {
        "control": "empty_artifacts",
        "expected": 0,
        "actual": len(
            empty_artifacts
        ),
    },
    {
        "control": (
            "artifact_checksum_mismatches"
        ),
        "expected": 0,
        "actual": len(
            artifact_checksum_mismatches
        ),
    },
    {
        "control": (
            "qa_report_checksum_mismatches"
        ),
        "expected": 0,
        "actual": len(
            qa_report_checksum_mismatches
        ),
    },
    {
        "control": (
            "qa_report_identity_mismatches"
        ),
        "expected": 0,
        "actual": len(
            qa_report_identity_mismatches
        ),
    },
    {
        "control": (
            "qa_report_status_failures"
        ),
        "expected": 0,
        "actual": len(
            qa_report_status_failures
        ),
    },
    {
        "control": (
            "qa_artifact_checksum_mismatches"
        ),
        "expected": 0,
        "actual": len(
            qa_artifact_checksum_mismatches
        ),
    },
    {
        "control": (
            "annotation_formula_mismatches"
        ),
        "expected": 0,
        "actual": len(
            annotation_formula_mismatches
        ),
    },
    {
        "control": (
            "qa_summary_all_passed"
        ),
        "expected": True,
        "actual": (
            qa_summary_all_passed
        ),
    },
    {
        "control": (
            "qa_index_all_passed"
        ),
        "expected": True,
        "actual": (
            qa_index_all_passed
        ),
    },
    {
        "control": (
            "visual_selection_all_passed"
        ),
        "expected": True,
        "actual": (
            visual_selection_all_passed
        ),
    },
]

for control in final_controls:
    control["status"] = (
        "VALID"
        if control["actual"]
        == control["expected"]
        else "INVALID"
    )

final_integrity_summary = pd.DataFrame(
    final_controls
)

display(
    final_integrity_summary
)


# ================================================================
# Ringkasan per template
# ================================================================

template_summary_rows = []

for template_id in sorted(
    template_distribution
):
    render_count = int(
        template_distribution.get(
            template_id,
            0,
        )
    )

    qa_template_frame = (
        qa_summary[
            qa_summary[
                "template_id"
            ]
            == template_id
        ]
    )

    qa_passed = int(
        (
            qa_template_frame[
                "technical_status"
            ]
            == "PASSED"
        ).sum()
    )

    visual_count = int(
        (
            visual_selection[
                "template_id"
            ]
            == template_id
        ).sum()
    )

    template_valid = all(
        [
            render_count == 20,
            len(
                qa_template_frame
            )
            == 20,
            qa_passed == 20,
            visual_count == 3,
        ]
    )

    template_summary_rows.append(
        {
            "template_id": template_id,
            "rendered_documents": (
                render_count
            ),
            "technical_qa_records": int(
                len(qa_template_frame)
            ),
            "technical_passed": (
                qa_passed
            ),
            "visual_samples": (
                visual_count
            ),
            "status": (
                "VALID"
                if template_valid
                else "INVALID"
            ),
        }
    )

final_template_summary = pd.DataFrame(
    template_summary_rows
)

display(
    final_template_summary
)


# ================================================================
# Tentukan status final
# ================================================================

final_integrity_status = (
    "PASSED"
    if all(
        final_integrity_summary[
            "status"
        ]
        == "VALID"
    )
    and all(
        final_template_summary[
            "status"
        ]
        == "VALID"
    )
    else "FAILED"
)


# ================================================================
# Simpan summary final
# ================================================================

final_write_csv_atomically(
    FINAL_INTEGRITY_SUMMARY_PATH,
    final_integrity_summary,
)

final_summary_checksum = (
    final_sha256_file(
        FINAL_INTEGRITY_SUMMARY_PATH
    )
)


# ================================================================
# Susun laporan final
# ================================================================

final_integrity_report = {
    "schema_version": "1.0.0",
    "dataset_id": (
        "SYNTHETIC-INVOICE-V1"
    ),
    "dataset_version": "1.0.0",
    "build_id": Path(
        BUILD_RUN_ROOT
    ).name,
    "artifact_type": (
        "final_dataset_integrity_audit"
    ),
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "status": (
        final_integrity_status
    ),
    "document_count": len(
        render_index_records
    ),
    "template_count": len(
        template_distribution
    ),
    "artifact_counts": {
        "pdf": len(
            list(
                Path(
                    BATCH_PDF_ROOT
                ).rglob("*.pdf")
            )
        ),
        "preview": len(
            list(
                Path(
                    BATCH_PREVIEW_ROOT
                ).rglob("*.png")
            )
        ),
        "ground_truth": len(
            list(
                Path(
                    BATCH_GROUND_TRUTH_ROOT
                ).rglob("*.json")
            )
        ),
        "qa_reports": len(
            list(
                Path(
                    BATCH_QA_ROOT
                ).rglob(
                    "*_qa_report.json"
                )
            )
        ),
    },
    "quality_status": {
        "batch_render": (
            render_manifest.get(
                "render_status"
            )
        ),
        "technical_qa": (
            qa_manifest.get(
                "technical_status"
            )
        ),
        "manual_visual_review": (
            qa_manifest.get(
                "manual_visual_review"
            )
        ),
        "manual_review_scope": (
            visual_manifest.get(
                "review_scope"
            )
        ),
        "reviewed_visual_samples": len(
            visual_selection
        ),
    },
    "distributions": {
        "template": (
            template_distribution
        ),
        "split": split_distribution,
        "language": (
            language_distribution
        ),
        "currency": (
            currency_distribution
        ),
    },
    "controls": [
        {
            "control": row["control"],
            "expected": row["expected"],
            "actual": row["actual"],
            "status": row["status"],
        }
        for row in final_controls
    ],
    "template_summary": (
        template_summary_rows
    ),
    "failure_details": {
        "manifest_checksum_mismatches": (
            manifest_checksum_mismatches
        ),
        "missing_artifacts": (
            missing_artifacts
        ),
        "empty_artifacts": (
            empty_artifacts
        ),
        "artifact_checksum_mismatches": (
            artifact_checksum_mismatches
        ),
        "qa_report_checksum_mismatches": (
            qa_report_checksum_mismatches
        ),
        "qa_report_identity_mismatches": (
            qa_report_identity_mismatches
        ),
        "qa_report_status_failures": (
            qa_report_status_failures
        ),
        "qa_artifact_checksum_mismatches": (
            qa_artifact_checksum_mismatches
        ),
        "annotation_formula_mismatches": (
            annotation_formula_mismatches
        ),
    },
    "source_artifacts": {
        "canonical_jsonl": str(
            CANONICAL_RECORDS_JSONL_PATH
        ),
        "presentation_jsonl": str(
            PRESENTATION_PAYLOADS_JSONL_PATH
        ),
        "render_manifest": str(
            BATCH_RENDER_MANIFEST_PATH
        ),
        "render_index": str(
            BATCH_RENDER_INDEX_PATH
        ),
        "qa_manifest": str(
            BATCH_QA_MANIFEST_PATH
        ),
        "qa_index": str(
            BATCH_QA_INDEX_PATH
        ),
        "qa_summary": str(
            BATCH_QA_SUMMARY_PATH
        ),
        "visual_review_manifest": str(
            VISUAL_REVIEW_MANIFEST_PATH
        ),
        "visual_selection": str(
            VISUAL_REVIEW_SELECTION_PATH
        ),
    },
    "final_summary": {
        "path": str(
            FINAL_INTEGRITY_SUMMARY_PATH
        ),
        "size_bytes": (
            FINAL_INTEGRITY_SUMMARY_PATH
            .stat()
            .st_size
        ),
        "sha256": (
            final_summary_checksum
        ),
    },
}

final_write_json_atomically(
    FINAL_INTEGRITY_REPORT_PATH,
    final_integrity_report,
)

final_report_checksum = (
    final_sha256_file(
        FINAL_INTEGRITY_REPORT_PATH
    )
)

final_write_text_atomically(
    FINAL_INTEGRITY_CHECKSUM_PATH,
    (
        f"{final_report_checksum}  "
        f"{FINAL_INTEGRITY_REPORT_PATH.name}\n"
    ),
)


# ================================================================
# Round-trip final report
# ================================================================

verified_final_report = final_load_json(
    FINAL_INTEGRITY_REPORT_PATH
)

if (
    verified_final_report
    != final_integrity_report
):
    raise RuntimeError(
        "Final integrity report berubah "
        "setelah dibuka ulang."
    )

with FINAL_INTEGRITY_CHECKSUM_PATH.open(
    mode="r",
    encoding="utf-8",
) as checksum_file:
    checksum_line = (
        checksum_file.read().strip()
    )

if not checksum_line.startswith(
    final_report_checksum
):
    raise RuntimeError(
        "Checksum sidecar final "
        "tidak sesuai."
    )


# ================================================================
# Variabel runtime
# ================================================================

FINAL_DATASET_INTEGRITY_STATUS = (
    final_integrity_status
)

FINAL_DATASET_READY = (
    final_integrity_status
    == "PASSED"
)


# ================================================================
# Ringkasan akhir
# ================================================================

print(
    f"Build root          : "
    f"{BUILD_RUN_ROOT}"
)

print(
    f"Documents           : "
    f"{len(render_index_records)}"
)

print(
    f"Templates           : "
    f"{len(template_distribution)}"
)

print(
    f"Technical QA        : "
    f"{qa_manifest.get('technical_status')}"
)

print(
    f"Manual review       : "
    f"{qa_manifest.get('manual_visual_review')}"
)

print(
    f"Review scope        : "
    f"{visual_manifest.get('review_scope')}"
)

print(
    f"Visual samples      : "
    f"{len(visual_selection)}"
)

print(
    f"Integrity report    : "
    f"{FINAL_INTEGRITY_REPORT_PATH}"
)

print(
    f"Integrity summary   : "
    f"{FINAL_INTEGRITY_SUMMARY_PATH}"
)

print(
    f"Report SHA-256      : "
    f"{final_report_checksum}"
)

print(
    f"Final status        : "
    f"{FINAL_DATASET_INTEGRITY_STATUS}"
)

print()


if (
    FINAL_DATASET_INTEGRITY_STATUS
    != "PASSED"
):
    print(
        "❌ FINAL DATASET INTEGRITY FAILED — "
        "periksa kontrol INVALID dan "
        "failure_details."
    )

    raise RuntimeError(
        "Audit integritas final dataset gagal."
    )


print(
    "✅ FINAL DATASET INTEGRITY PASSED — "
    "200 dokumen, 10 template, seluruh artifact, "
    "checksum, technical QA, dan bukti visual review "
    "konsisten."
)

In [ ]:
# ================================================================
# CELL 78A — FINAL
# Preflight freeze release
# Belum membuat atau mengubah artifact
# ================================================================

from collections import Counter
from pathlib import Path

import hashlib
import json

import pandas as pd


# ================================================================
# Identitas release
# ================================================================

RELEASE_DATASET_ID = (
    "SYNTHETIC-INVOICE-V1"
)

RELEASE_VERSION = "1.0.0"

RELEASE_EXPECTED_DOCUMENTS = 200
RELEASE_EXPECTED_TEMPLATES = 10
RELEASE_EXPECTED_PER_TEMPLATE = 20

PROJECT_DATA_ROOT = (
    Path(BUILD_RUN_ROOT)
    .parents[2]
)

RELEASES_ROOT = (
    PROJECT_DATA_ROOT
    / "releases"
)

FINAL_RELEASE_ROOT = (
    RELEASES_ROOT
    / RELEASE_DATASET_ID
    / RELEASE_VERSION
)

FINAL_RELEASE_INDEX_PATH = (
    FINAL_RELEASE_ROOT
    / "release_index.jsonl"
)

FINAL_RELEASE_SUMMARY_PATH = (
    FINAL_RELEASE_ROOT
    / "release_summary.csv"
)

FINAL_DATASET_CARD_PATH = (
    FINAL_RELEASE_ROOT
    / "DATASET_CARD.md"
)

FINAL_SPLIT_GUIDE_PATH = (
    FINAL_RELEASE_ROOT
    / "SPLIT_USAGE.md"
)

FINAL_RELEASE_MANIFEST_PATH = (
    FINAL_RELEASE_ROOT
    / "release_manifest.json"
)

FINAL_RELEASE_CHECKSUM_PATH = (
    FINAL_RELEASE_ROOT
    / "release_manifest.sha256"
)

CURRENT_RELEASE_POINTER_PATH = (
    RELEASES_ROOT
    / "current_release.json"
)


# ================================================================
# Sumber release
# ================================================================

FINAL_INTEGRITY_REPORT_PATH = (
    BUILD_MANIFESTS_ROOT
    / "final_integrity_report.json"
)

FINAL_INTEGRITY_CHECKSUM_PATH = (
    BUILD_MANIFESTS_ROOT
    / "final_integrity_report.sha256"
)

BATCH_RENDER_INDEX_PATH = (
    BUILD_MANIFESTS_ROOT
    / "batch_render_index.jsonl"
)

BATCH_QA_INDEX_PATH = (
    BUILD_MANIFESTS_ROOT
    / "batch_qa_index.jsonl"
)

BATCH_QA_MANIFEST_PATH = (
    BUILD_MANIFESTS_ROOT
    / "batch_qa_manifest.json"
)

VISUAL_REVIEW_MANIFEST_PATH = (
    BUILD_MANIFESTS_ROOT
    / "batch_visual_review_manifest.json"
)


# ================================================================
# Validasi runtime
# ================================================================

required_runtime_objects = [
    "BUILD_RUN_ROOT",
    "BUILD_MANIFESTS_ROOT",
    "BATCH_DATASET_ROOT",
]

missing_runtime_objects = [
    object_name
    for object_name
    in required_runtime_objects
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "State recovery belum tersedia: "
        f"{missing_runtime_objects}. "
        "Jalankan kembali Cell 74C."
    )


# ================================================================
# Helper
# ================================================================

def release_sha256_file(
    file_path,
):
    digest = hashlib.sha256()

    with Path(file_path).open(
        "rb"
    ) as binary_file:
        for chunk in iter(
            lambda: binary_file.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def release_load_json(
    file_path,
):
    with Path(file_path).open(
        mode="r",
        encoding="utf-8",
    ) as input_file:
        return json.load(
            input_file
        )


def release_load_jsonl(
    file_path,
):
    records = []

    with Path(file_path).open(
        mode="r",
        encoding="utf-8",
    ) as input_file:
        for line_number, line in enumerate(
            input_file,
            start=1,
        ):
            clean_line = line.strip()

            if not clean_line:
                continue

            try:
                record = json.loads(
                    clean_line
                )
            except json.JSONDecodeError as error:
                raise RuntimeError(
                    "JSONL tidak valid pada "
                    f"baris {line_number}: "
                    f"{file_path}"
                ) from error

            records.append(record)

    return records


def release_relative_to_project(
    file_path,
):
    resolved_path = Path(
        file_path
    ).resolve()

    project_root_resolved = (
        PROJECT_DATA_ROOT.resolve()
    )

    try:
        relative_path = (
            resolved_path.relative_to(
                project_root_resolved
            )
        )
    except ValueError as error:
        raise RuntimeError(
            "Artifact berada di luar "
            "PROJECT_DATA_ROOT: "
            f"{resolved_path}"
        ) from error

    return str(relative_path)


# ================================================================
# Jangan menimpa release yang sudah frozen
# ================================================================

if FINAL_RELEASE_MANIFEST_PATH.is_file():
    existing_release_manifest = (
        release_load_json(
            FINAL_RELEASE_MANIFEST_PATH
        )
    )

    if (
        existing_release_manifest.get(
            "release_status"
        )
        == "FROZEN"
    ):
        raise RuntimeError(
            "Release v1.0.0 sudah dibekukan. "
            "Cell 78 tidak perlu dijalankan ulang."
        )


# ================================================================
# Validasi sumber
# ================================================================

required_source_files = [
    FINAL_INTEGRITY_REPORT_PATH,
    FINAL_INTEGRITY_CHECKSUM_PATH,
    BATCH_RENDER_INDEX_PATH,
    BATCH_QA_INDEX_PATH,
    BATCH_QA_MANIFEST_PATH,
    VISUAL_REVIEW_MANIFEST_PATH,
]

missing_source_files = [
    str(file_path)
    for file_path
    in required_source_files
    if not file_path.is_file()
]

if missing_source_files:
    raise FileNotFoundError(
        "Sumber release belum lengkap: "
        f"{missing_source_files}"
    )


# ================================================================
# Final integrity report
# ================================================================

final_integrity_report = (
    release_load_json(
        FINAL_INTEGRITY_REPORT_PATH
    )
)

if (
    final_integrity_report.get(
        "status"
    )
    != "PASSED"
):
    raise RuntimeError(
        "Final integrity report bukan PASSED."
    )

with FINAL_INTEGRITY_CHECKSUM_PATH.open(
    mode="r",
    encoding="utf-8",
) as checksum_file:
    recorded_integrity_checksum = (
        checksum_file.read()
        .strip()
        .split()[0]
    )

current_integrity_checksum = (
    release_sha256_file(
        FINAL_INTEGRITY_REPORT_PATH
    )
)

if (
    recorded_integrity_checksum
    != current_integrity_checksum
):
    raise RuntimeError(
        "Checksum final integrity report "
        "tidak sesuai."
    )

integrity_controls = (
    final_integrity_report.get(
        "controls",
        [],
    )
)

if not integrity_controls:
    raise RuntimeError(
        "Final integrity report tidak "
        "memiliki controls."
    )

if not all(
    control.get("status") == "VALID"
    for control in integrity_controls
):
    raise RuntimeError(
        "Masih ada integrity control "
        "yang tidak VALID."
    )


# ================================================================
# QA dan visual review
# ================================================================

batch_qa_manifest = (
    release_load_json(
        BATCH_QA_MANIFEST_PATH
    )
)

visual_review_manifest = (
    release_load_json(
        VISUAL_REVIEW_MANIFEST_PATH
    )
)

if (
    batch_qa_manifest.get(
        "technical_status"
    )
    != "PASSED"
):
    raise RuntimeError(
        "Technical QA bukan PASSED."
    )

if (
    batch_qa_manifest.get(
        "manual_visual_review"
    )
    != "PASSED"
):
    raise RuntimeError(
        "Manual visual review bukan PASSED."
    )

if (
    visual_review_manifest.get(
        "status"
    )
    != "PASSED"
):
    raise RuntimeError(
        "Visual review manifest bukan PASSED."
    )

if (
    visual_review_manifest.get(
        "review_scope"
    )
    != "STRATIFIED_SAMPLE"
):
    raise RuntimeError(
        "Visual review scope tidak sesuai."
    )

if (
    visual_review_manifest.get(
        "reviewed_document_count"
    )
    != 30
):
    raise RuntimeError(
        "Visual review harus berisi "
        "30 sampel."
    )


# ================================================================
# Muat render dan QA index
# ================================================================

render_index_records = (
    release_load_jsonl(
        BATCH_RENDER_INDEX_PATH
    )
)

qa_index_records = (
    release_load_jsonl(
        BATCH_QA_INDEX_PATH
    )
)

if len(render_index_records) != 200:
    raise RuntimeError(
        "Render index harus memiliki "
        "200 record."
    )

if len(qa_index_records) != 200:
    raise RuntimeError(
        "QA index harus memiliki "
        "200 record."
    )

qa_index_by_id = {
    record[
        "canonical_invoice_id"
    ]: record
    for record in qa_index_records
}

if len(qa_index_by_id) != 200:
    raise RuntimeError(
        "Canonical ID QA index tidak unik."
    )


# ================================================================
# Susun release index di memori
# ================================================================

release_index_records = []
missing_referenced_artifacts = []

for render_record in sorted(
    render_index_records,
    key=lambda record: int(
        record["sequence_number"]
    ),
):
    canonical_id = (
        render_record[
            "canonical_invoice_id"
        ]
    )

    qa_record = qa_index_by_id.get(
        canonical_id
    )

    if qa_record is None:
        raise RuntimeError(
            "QA index tidak ditemukan: "
            f"{canonical_id}"
        )

    referenced_artifacts = {}

    for artifact_name in (
        "pdf",
        "preview",
        "ground_truth",
    ):
        artifact_metadata = (
            render_record[
                "artifacts"
            ][artifact_name]
        )

        artifact_absolute_path = (
            Path(BATCH_DATASET_ROOT)
            / artifact_metadata[
                "relative_path"
            ]
        )

        if not artifact_absolute_path.is_file():
            missing_referenced_artifacts.append(
                str(
                    artifact_absolute_path
                )
            )

        referenced_artifacts[
            artifact_name
        ] = {
            "project_relative_path": (
                release_relative_to_project(
                    artifact_absolute_path
                )
            ),
            "size_bytes": int(
                artifact_metadata[
                    "size_bytes"
                ]
            ),
            "sha256": (
                artifact_metadata[
                    "sha256"
                ]
            ),
        }

    qa_report_absolute_path = (
        Path(BATCH_DATASET_ROOT)
        / qa_record[
            "qa_report_relative_path"
        ]
    )

    if not qa_report_absolute_path.is_file():
        missing_referenced_artifacts.append(
            str(
                qa_report_absolute_path
            )
        )

    referenced_artifacts[
        "qa_report"
    ] = {
        "project_relative_path": (
            release_relative_to_project(
                qa_report_absolute_path
            )
        ),
        "size_bytes": int(
            qa_record[
                "qa_report_size_bytes"
            ]
        ),
        "sha256": (
            qa_record[
                "qa_report_sha256"
            ]
        ),
    }

    release_index_records.append(
        {
            "sequence_number": int(
                render_record[
                    "sequence_number"
                ]
            ),
            "canonical_invoice_id": (
                canonical_id
            ),
            "document_id": (
                render_record[
                    "document_id"
                ]
            ),
            "template_id": (
                render_record[
                    "template_id"
                ]
            ),
            "split": (
                render_record[
                    "split"
                ]
            ),
            "language": (
                render_record[
                    "language"
                ]
            ),
            "currency": (
                render_record[
                    "currency"
                ]
            ),
            "item_count": int(
                render_record[
                    "item_count"
                ]
            ),
            "page_count": int(
                render_record[
                    "page_count"
                ]
            ),
            "annotation_count": int(
                render_record[
                    "annotation_count"
                ]
            ),
            "technical_status": (
                qa_record[
                    "technical_status"
                ]
            ),
            "manual_visual_review": (
                "COVERED_BY_STRATIFIED_"
                "BATCH_REVIEW"
            ),
            "artifacts": (
                referenced_artifacts
            ),
        }
    )

if missing_referenced_artifacts:
    raise FileNotFoundError(
        "Artifact referensi release hilang: "
        f"{missing_referenced_artifacts[:10]}"
    )

if len(release_index_records) != 200:
    raise RuntimeError(
        "Release index harus berisi "
        "200 record."
    )


# ================================================================
# Distribusi
# ================================================================

template_distribution = dict(
    sorted(
        Counter(
            record["template_id"]
            for record
            in release_index_records
        ).items()
    )
)

split_distribution = dict(
    sorted(
        Counter(
            record["split"]
            for record
            in release_index_records
        ).items()
    )
)

language_distribution = dict(
    sorted(
        Counter(
            record["language"]
            for record
            in release_index_records
        ).items()
    )
)

currency_distribution = dict(
    sorted(
        Counter(
            record["currency"]
            for record
            in release_index_records
        ).items()
    )
)

template_distribution_valid = all(
    template_distribution.get(
        f"TPL-{template_number:02d}"
    )
    == RELEASE_EXPECTED_PER_TEMPLATE
    for template_number in range(
        1,
        11,
    )
)


# ================================================================
# Preflight controls
# ================================================================

release_preflight_controls = [
    {
        "control": "integrity_status",
        "expected": "PASSED",
        "actual": (
            final_integrity_report[
                "status"
            ]
        ),
    },
    {
        "control": "integrity_checksum",
        "expected": "MATCH",
        "actual": (
            "MATCH"
            if recorded_integrity_checksum
            == current_integrity_checksum
            else "MISMATCH"
        ),
    },
    {
        "control": "technical_qa",
        "expected": "PASSED",
        "actual": (
            batch_qa_manifest[
                "technical_status"
            ]
        ),
    },
    {
        "control": "manual_review",
        "expected": "PASSED",
        "actual": (
            batch_qa_manifest[
                "manual_visual_review"
            ]
        ),
    },
    {
        "control": "visual_scope",
        "expected": (
            "STRATIFIED_SAMPLE"
        ),
        "actual": (
            visual_review_manifest[
                "review_scope"
            ]
        ),
    },
    {
        "control": "release_records",
        "expected": 200,
        "actual": len(
            release_index_records
        ),
    },
    {
        "control": "template_count",
        "expected": 10,
        "actual": len(
            template_distribution
        ),
    },
    {
        "control": (
            "template_distribution"
        ),
        "expected": True,
        "actual": (
            template_distribution_valid
        ),
    },
    {
        "control": (
            "missing_referenced_artifacts"
        ),
        "expected": 0,
        "actual": len(
            missing_referenced_artifacts
        ),
    },
]

for control in release_preflight_controls:
    control["status"] = (
        "VALID"
        if control["actual"]
        == control["expected"]
        else "INVALID"
    )

release_preflight_summary = (
    pd.DataFrame(
        release_preflight_controls
    )
)

display(
    release_preflight_summary
)

print(
    "Template distribution:"
)

print(
    json.dumps(
        template_distribution,
        indent=2,
        ensure_ascii=False,
    )
)

print()

print(
    "Split distribution:"
)

print(
    json.dumps(
        split_distribution,
        indent=2,
        ensure_ascii=False,
    )
)

print()

if not all(
    release_preflight_summary[
        "status"
    ]
    == "VALID"
):
    raise RuntimeError(
        "Release preflight gagal."
    )

RELEASE_PREFLIGHT_READY = True

print(
    "✅ CELL 78A PASSED — "
    "release index berhasil disusun di memori "
    "dan seluruh sumber release valid."
)

print(
    "Belum ada release artifact yang dibuat."
)

print(
    "Lanjutkan ke Cell 78B untuk menulis "
    "release metadata final."
)

In [ ]:
# ================================================================
# CELL 78B — FINAL
# Menulis release metadata tanpa menduplikasi artifact utama
# ================================================================

from datetime import datetime, timezone
from pathlib import Path

import hashlib
import json
import os

import pandas as pd


# ================================================================
# Validasi preflight
# ================================================================

if (
    "RELEASE_PREFLIGHT_READY"
    not in globals()
    or not RELEASE_PREFLIGHT_READY
):
    raise RuntimeError(
        "Release preflight belum siap. "
        "Jalankan Cell 78A terlebih dahulu."
    )

if len(release_index_records) != 200:
    raise RuntimeError(
        "release_index_records harus "
        "berjumlah 200."
    )

if FINAL_RELEASE_MANIFEST_PATH.exists():
    raise RuntimeError(
        "Release manifest sudah tersedia. "
        "Jangan menimpa release yang telah dibuat: "
        f"{FINAL_RELEASE_MANIFEST_PATH}"
    )


# ================================================================
# Helper penulisan atomik
# ================================================================

def freeze_write_text(
    output_path,
    text,
):
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        output_path.parent
        / f".{output_path.name}.tmp"
    )

    temporary_path.unlink(
        missing_ok=True
    )

    try:
        with temporary_path.open(
            mode="w",
            encoding="utf-8",
            newline="\n",
        ) as output_file:
            output_file.write(text)

        os.replace(
            temporary_path,
            output_path,
        )

    except Exception:
        temporary_path.unlink(
            missing_ok=True
        )
        raise


def freeze_write_json(
    output_path,
    payload,
):
    serialized_payload = json.dumps(
        payload,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )

    freeze_write_text(
        output_path,
        serialized_payload + "\n",
    )


def freeze_write_jsonl(
    output_path,
    records,
):
    serialized_lines = [
        json.dumps(
            record,
            ensure_ascii=False,
            separators=(",", ":"),
            allow_nan=False,
        )
        for record in records
    ]

    freeze_write_text(
        output_path,
        "\n".join(
            serialized_lines
        )
        + "\n",
    )


def freeze_write_csv(
    output_path,
    dataframe,
):
    output_path = Path(output_path)

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        output_path.parent
        / f".{output_path.name}.tmp"
    )

    temporary_path.unlink(
        missing_ok=True
    )

    try:
        dataframe.to_csv(
            temporary_path,
            index=False,
            encoding="utf-8",
            lineterminator="\n",
        )

        os.replace(
            temporary_path,
            output_path,
        )

    except Exception:
        temporary_path.unlink(
            missing_ok=True
        )
        raise


def freeze_sha256(
    file_path,
):
    digest = hashlib.sha256()

    with Path(file_path).open(
        "rb"
    ) as binary_file:
        for chunk in iter(
            lambda: binary_file.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


# ================================================================
# Buat release root
# ================================================================

FINAL_RELEASE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ================================================================
# Tulis release index
# ================================================================

freeze_write_jsonl(
    FINAL_RELEASE_INDEX_PATH,
    release_index_records,
)

release_index_checksum = freeze_sha256(
    FINAL_RELEASE_INDEX_PATH
)


# ================================================================
# Buat release summary
# ================================================================

release_summary_rows = []

for template_id in sorted(
    template_distribution
):
    template_records = [
        record
        for record in release_index_records
        if record["template_id"]
        == template_id
    ]

    release_summary_rows.append(
        {
            "template_id": template_id,
            "document_count": len(
                template_records
            ),
            "minimum_items": min(
                record["item_count"]
                for record
                in template_records
            ),
            "maximum_items": max(
                record["item_count"]
                for record
                in template_records
            ),
            "languages": ",".join(
                sorted(
                    {
                        record["language"]
                        for record
                        in template_records
                    }
                )
            ),
            "currencies": ",".join(
                sorted(
                    {
                        record["currency"]
                        for record
                        in template_records
                    }
                )
            ),
            "technical_passed": sum(
                record["technical_status"]
                == "PASSED"
                for record
                in template_records
            ),
            "status": (
                "VALID"
                if len(template_records) == 20
                and all(
                    record[
                        "technical_status"
                    ]
                    == "PASSED"
                    for record
                    in template_records
                )
                else "INVALID"
            ),
        }
    )

release_summary_frame = pd.DataFrame(
    release_summary_rows
)

if not all(
    release_summary_frame["status"]
    == "VALID"
):
    raise RuntimeError(
        "Release summary belum valid."
    )

freeze_write_csv(
    FINAL_RELEASE_SUMMARY_PATH,
    release_summary_frame,
)

release_summary_checksum = freeze_sha256(
    FINAL_RELEASE_SUMMARY_PATH
)


# ================================================================
# Susun teks distribusi
# ================================================================

split_distribution_lines = [
    (
        f"- `{split_name}`: "
        f"{document_count} documents"
    )
    for split_name, document_count
    in split_distribution.items()
]

template_distribution_lines = [
    (
        f"- `{template_id}`: "
        f"{document_count} documents"
    )
    for template_id, document_count
    in template_distribution.items()
]


# ================================================================
# Dataset card
# ================================================================

dataset_card_lines = [
    f"# {RELEASE_DATASET_ID}",
    "",
    f"Version: `{RELEASE_VERSION}`  ",
    "Release status: `FROZEN`  ",
    "Dataset type: Synthetic invoice document dataset",
    "",
    "## Overview",
    "",
    (
        "This release contains 200 synthetic invoice documents "
        "generated from 10 layout templates."
    ),
    (
        "Every document includes a PDF, preview PNG, structured "
        "ground truth, field-level annotations, and technical QA."
    ),
    "",
    (
        "The dataset contains no real transaction records and "
        "must not be represented as real financial data."
    ),
    "",
    "## Dataset Size",
    "",
    "- Documents: 200",
    "- Templates: 10",
    "- PDF files: 200",
    "- Preview images: 200",
    "- Ground-truth files: 200",
    "- Technical QA reports: 200",
    "- Manual visual samples reviewed: 30",
    "- Languages: Indonesian and English",
    "- Currencies: IDR, USD, EUR, and GBP",
    "",
    "## Splits",
    "",
    *split_distribution_lines,
    "",
    "## Templates",
    "",
    *template_distribution_lines,
    "",
    "## Quality Assurance",
    "",
    "- Batch rendering: PASSED",
    "- Technical QA: 200/200 PASSED",
    "- Manual visual review: PASSED",
    "- Manual review scope: stratified sample",
    "- Visual samples reviewed: 30",
    "- Template coverage: 10/10",
    "- Missing artifacts: 0",
    "- Checksum mismatches: 0",
    "- Final integrity status: PASSED",
    "",
    "Final integrity report SHA-256:",
    "",
    f"`{current_integrity_checksum}`",
    "",
    "## Storage Model",
    "",
    (
        "The release uses reference-only storage. PDF, preview, "
        "ground-truth, and QA files are not duplicated."
    ),
    (
        "The release index stores project-relative paths and "
        "SHA-256 checksums for each artifact."
    ),
    "",
    "## Annotation Scope",
    "",
    "- Invoice number, invoice date, and due date",
    "- Currency",
    "- Vendor and buyer information",
    "- Item descriptions and quantities",
    "- Unit prices and line totals",
    "- Subtotal, tax, discount, and total",
    "- Synthetic-data notice",
    "",
    "## Intended Uses",
    "",
    "- Invoice field-extraction experiments",
    "- Document AI pipeline development",
    "- OCR and layout-understanding evaluation",
    "- Synthetic-data research prototypes",
    "- Portfolio and software demonstrations",
    "",
    "## Limitations",
    "",
    "- All documents are synthetic.",
    "- The release contains 200 documents.",
    "- Layout diversity is limited to 10 templates.",
    (
        "- Manual visual review covers a stratified sample of "
        "30 documents, not all documents individually."
    ),
    (
        "- Technical success does not guarantee performance "
        "on real-world invoices."
    ),
    "",
    "## License Status",
    "",
    "No public-distribution license has been assigned.",
    (
        "Assign explicit license and usage terms before "
        "publishing, sharing, or selling the dataset."
    ),
    "",
    "## Safety Notice",
    "",
    "DATA SINTETIS — BUKAN DOKUMEN TRANSAKSI NYATA.",
    "",
]

dataset_card_text = "\n".join(
    dataset_card_lines
)

freeze_write_text(
    FINAL_DATASET_CARD_PATH,
    dataset_card_text,
)

dataset_card_checksum = freeze_sha256(
    FINAL_DATASET_CARD_PATH
)


# ================================================================
# Split usage guide
# ================================================================

split_guide_lines = [
    "# Train, Validation, and Test Split Usage",
    "",
    f"Dataset: `{RELEASE_DATASET_ID}`  ",
    f"Version: `{RELEASE_VERSION}`",
    "",
    "## Split Distribution",
    "",
    *split_distribution_lines,
    "",
    "## Usage Policy",
    "",
    (
        "- `development`: model training and "
        "development experiments"
    ),
    (
        "- `validation`: hyperparameter selection "
        "and model selection"
    ),
    "- `test`: final evaluation only",
    "",
    "Do not use the test split for:",
    "",
    "- training",
    "- prompt tuning",
    "- threshold selection",
    "- hyperparameter selection",
    "- repeated model-selection decisions",
    "",
    "## Recommended Workflow",
    "",
    "1. Train only with the development split.",
    "2. Evaluate during development using validation.",
    (
        "3. Select preprocessing, parameters, and thresholds "
        "using validation."
    ),
    "4. Freeze the selected pipeline.",
    "5. Run the test split once for final reporting.",
    "",
    "## Release Index Location",
    "",
    (
        "`releases/SYNTHETIC-INVOICE-V1/"
        "1.0.0/release_index.jsonl`"
    ),
    "",
    "## Artifact Resolution",
    "",
    (
        "Each index record contains project-relative paths "
        "under the `artifacts` field."
    ),
    "",
    "Resolve a PDF by joining:",
    "",
    (
        "`InvoiceFlow-AI-Data/` + "
        "`record['artifacts']['pdf']['project_relative_path']`"
    ),
    "",
    "Always verify the stored SHA-256 checksum before using an "
    "artifact for training or evaluation.",
    "",
]

split_guide_text = "\n".join(
    split_guide_lines
)

freeze_write_text(
    FINAL_SPLIT_GUIDE_PATH,
    split_guide_text,
)

split_guide_checksum = freeze_sha256(
    FINAL_SPLIT_GUIDE_PATH
)


# ================================================================
# Release manifest
# ================================================================

release_created_utc = datetime.now(
    timezone.utc
).isoformat()

release_manifest = {
    "schema_version": "1.0.0",
    "dataset_id": RELEASE_DATASET_ID,
    "dataset_version": RELEASE_VERSION,
    "release_status": "FROZEN",
    "release_created_utc": (
        release_created_utc
    ),
    "storage_mode": (
        "REFERENCE_ONLY_NO_DUPLICATION"
    ),
    "source_build": {
        "build_id": Path(
            BUILD_RUN_ROOT
        ).name,
        "project_relative_path": (
            release_relative_to_project(
                BUILD_RUN_ROOT
            )
        ),
        "final_integrity_report": {
            "project_relative_path": (
                release_relative_to_project(
                    FINAL_INTEGRITY_REPORT_PATH
                )
            ),
            "sha256": (
                current_integrity_checksum
            ),
            "status": "PASSED",
        },
    },
    "release_counts": {
        "documents": 200,
        "templates": 10,
        "pdf": 200,
        "preview": 200,
        "ground_truth": 200,
        "qa_reports": 200,
        "visual_review_samples": 30,
    },
    "quality_status": {
        "final_integrity": "PASSED",
        "technical_qa": "PASSED",
        "technical_passed_documents": 200,
        "manual_visual_review": "PASSED",
        "manual_review_scope": (
            "STRATIFIED_SAMPLE"
        ),
    },
    "distributions": {
        "template": (
            template_distribution
        ),
        "split": split_distribution,
        "language": (
            language_distribution
        ),
        "currency": (
            currency_distribution
        ),
    },
    "license": {
        "status": "NOT_ASSIGNED",
        "public_distribution_ready": False,
        "note": (
            "Assign explicit license and usage terms "
            "before public distribution or sale."
        ),
    },
    "release_artifacts": {
        "release_index": {
            "path": str(
                FINAL_RELEASE_INDEX_PATH
            ),
            "record_count": 200,
            "size_bytes": (
                FINAL_RELEASE_INDEX_PATH
                .stat()
                .st_size
            ),
            "sha256": (
                release_index_checksum
            ),
        },
        "release_summary": {
            "path": str(
                FINAL_RELEASE_SUMMARY_PATH
            ),
            "row_count": 10,
            "size_bytes": (
                FINAL_RELEASE_SUMMARY_PATH
                .stat()
                .st_size
            ),
            "sha256": (
                release_summary_checksum
            ),
        },
        "dataset_card": {
            "path": str(
                FINAL_DATASET_CARD_PATH
            ),
            "size_bytes": (
                FINAL_DATASET_CARD_PATH
                .stat()
                .st_size
            ),
            "sha256": (
                dataset_card_checksum
            ),
        },
        "split_usage_guide": {
            "path": str(
                FINAL_SPLIT_GUIDE_PATH
            ),
            "size_bytes": (
                FINAL_SPLIT_GUIDE_PATH
                .stat()
                .st_size
            ),
            "sha256": (
                split_guide_checksum
            ),
        },
    },
}

freeze_write_json(
    FINAL_RELEASE_MANIFEST_PATH,
    release_manifest,
)

release_manifest_checksum = freeze_sha256(
    FINAL_RELEASE_MANIFEST_PATH
)

freeze_write_text(
    FINAL_RELEASE_CHECKSUM_PATH,
    (
        f"{release_manifest_checksum}  "
        f"{FINAL_RELEASE_MANIFEST_PATH.name}\n"
    ),
)


# ================================================================
# Recovery pointer
# ================================================================

current_release_pointer = {
    "schema_version": "1.0.0",
    "dataset_id": RELEASE_DATASET_ID,
    "dataset_version": RELEASE_VERSION,
    "release_status": "FROZEN",
    "release_root": str(
        FINAL_RELEASE_ROOT
    ),
    "release_manifest": str(
        FINAL_RELEASE_MANIFEST_PATH
    ),
    "release_manifest_sha256": (
        release_manifest_checksum
    ),
    "source_build_root": str(
        BUILD_RUN_ROOT
    ),
    "storage_mode": (
        "REFERENCE_ONLY_NO_DUPLICATION"
    ),
}

freeze_write_json(
    CURRENT_RELEASE_POINTER_PATH,
    current_release_pointer,
)


# ================================================================
# Verifikasi release
# ================================================================

verified_release_manifest = (
    release_load_json(
        FINAL_RELEASE_MANIFEST_PATH
    )
)

verified_release_pointer = (
    release_load_json(
        CURRENT_RELEASE_POINTER_PATH
    )
)

verified_release_index = (
    release_load_jsonl(
        FINAL_RELEASE_INDEX_PATH
    )
)

release_controls = [
    {
        "control": "release_status",
        "expected": "FROZEN",
        "actual": (
            verified_release_manifest[
                "release_status"
            ]
        ),
    },
    {
        "control": "storage_mode",
        "expected": (
            "REFERENCE_ONLY_NO_DUPLICATION"
        ),
        "actual": (
            verified_release_manifest[
                "storage_mode"
            ]
        ),
    },
    {
        "control": "release_documents",
        "expected": 200,
        "actual": (
            verified_release_manifest[
                "release_counts"
            ]["documents"]
        ),
    },
    {
        "control": "release_index_records",
        "expected": 200,
        "actual": len(
            verified_release_index
        ),
    },
    {
        "control": "technical_status",
        "expected": "PASSED",
        "actual": (
            verified_release_manifest[
                "quality_status"
            ]["technical_qa"]
        ),
    },
    {
        "control": "manual_review",
        "expected": "PASSED",
        "actual": (
            verified_release_manifest[
                "quality_status"
            ]["manual_visual_review"]
        ),
    },
    {
        "control": "integrity_status",
        "expected": "PASSED",
        "actual": (
            verified_release_manifest[
                "quality_status"
            ]["final_integrity"]
        ),
    },
    {
        "control": "manifest_checksum",
        "expected": (
            release_manifest_checksum
        ),
        "actual": (
            freeze_sha256(
                FINAL_RELEASE_MANIFEST_PATH
            )
        ),
    },
    {
        "control": "pointer_checksum",
        "expected": (
            release_manifest_checksum
        ),
        "actual": (
            verified_release_pointer[
                "release_manifest_sha256"
            ]
        ),
    },
    {
        "control": (
            "public_distribution_ready"
        ),
        "expected": False,
        "actual": (
            verified_release_manifest[
                "license"
            ]["public_distribution_ready"]
        ),
    },
]

for control in release_controls:
    control["status"] = (
        "VALID"
        if control["actual"]
        == control["expected"]
        else "INVALID"
    )

release_control_summary = (
    pd.DataFrame(
        release_controls
    )
)

display(
    release_control_summary
)

display(
    release_summary_frame
)

if not all(
    release_control_summary["status"]
    == "VALID"
):
    raise RuntimeError(
        "Release freeze tidak valid."
    )


# ================================================================
# Runtime state
# ================================================================

FINAL_RELEASE_STATUS = "FROZEN"
FINAL_RELEASE_READY = True
FINAL_RELEASE_PUBLIC_READY = False


# ================================================================
# Ringkasan akhir
# ================================================================

print(
    f"Release root       : "
    f"{FINAL_RELEASE_ROOT}"
)

print(
    f"Release index      : "
    f"{FINAL_RELEASE_INDEX_PATH}"
)

print(
    f"Release summary    : "
    f"{FINAL_RELEASE_SUMMARY_PATH}"
)

print(
    f"Dataset card       : "
    f"{FINAL_DATASET_CARD_PATH}"
)

print(
    f"Split guide        : "
    f"{FINAL_SPLIT_GUIDE_PATH}"
)

print(
    f"Release manifest   : "
    f"{FINAL_RELEASE_MANIFEST_PATH}"
)

print(
    f"Current pointer    : "
    f"{CURRENT_RELEASE_POINTER_PATH}"
)

print(
    f"Manifest SHA-256   : "
    f"{release_manifest_checksum}"
)

print(
    f"Documents          : "
    f"{len(verified_release_index)}"
)

print(
    "Storage mode       : "
    "REFERENCE_ONLY_NO_DUPLICATION"
)

print(
    f"Release status     : "
    f"{FINAL_RELEASE_STATUS}"
)

print(
    "Public distribution: "
    "BLOCKED — LICENSE NOT ASSIGNED"
)

print()

print(
    "✅ DATASET RELEASE v1.0.0 FROZEN — "
    "release index, summary, dataset card, "
    "split guide, manifest, checksum, dan "
    "recovery pointer berhasil dibuat."
)